In [3]:
import fp_vecbacktest
from fp_vecbacktest.vecbacktest.backtest import Backtest
from fp_vecbacktest.vecbacktest.operators import *
from fp_vecbacktest.vecbacktest.technical_utils import *
import sys

from fp_vecbacktest.vecbacktest.backtest import Backtest
from fp_vecbacktest.vecbacktest.operators import *
from fp_vecbacktest.vecbacktest.technical_utils import *
from fp_vecbacktest.vecbacktest.statics.universes import FUTURES_UNIVERSE
from fp_vecbacktest.vecbacktest.base_alpha import BaseAlpha

import pandas as pd
import numpy as np
import numba as nb
from matplotlib import pyplot as plt

pd.set_option('display.precision', 8)

In [4]:
config2 = {
    'exec_prices':      'close_binance',
    'feed_type':        'multi',
    'execution_market': 'EXECUTION-MODELLING',
    'fundings':         'fundings',
    'alpha_config': {
            'max_w': 0.05
        },
    'data_folderpath': r'/data/ds/insample_research_data/',
     'lazyload': True,
    'executor_config': {
            'fee': -0.00002,
            'type': 'constant',
            'params': {'const_spread': 10e-4}
            },
}
bt2 = Backtest(config2)

Universe is not specified, using default: binance universe


In [ ]:
# import warnings
# warnings.filterwarnings("ignore")

# import numpy as np
# import pandas as pd
# from itertools import combinations
# from tqdm.auto import tqdm

# from sklearn.metrics import (
#     roc_auc_score, average_precision_score,
#     f1_score, balanced_accuracy_score,
#     mean_absolute_error, r2_score,
# )
# from sklearn.linear_model import LogisticRegression, Ridge
# from sklearn.preprocessing import StandardScaler
# from sklearn.pipeline import Pipeline
# import lightgbm as lgb

# TICKER_MAP = {
#     "BTC":  "BTC-USDT",
#     "ETH":  "ETH-USDT",
#     "SOL":  "SOL-USDT",
#     "BNB":  "BNB-USDT",
#     "XRP":  "XRP-USDT",
#     "DOGE": "DOGE-USDT",
#     "ADA":  "ADA-USDT",
#     "AVAX": "AVAX-USDT",
#     "TRX":  "TRX-USDT",
# }

# TARGET_ASSETS = ["BTC", "ETH"]         
# UNIVERSE      = list(TICKER_MAP.keys()) 

# START_DATE = "2023-01-01"             

# BARS_PER_DAY = 48
# ANN_FACTOR   = np.sqrt(365 * BARS_PER_DAY) 

# VOL_WINDOW = 48     # 24h

# STRESS_MULT = 1.25

# HORIZONS = [2, 8, 16, 48, 96]

# MI_WINDOW      = 192   # 4 days
# MI_BINS        = 10
# MI_SUBSAMPLE_K = 4    

# TRAIN_BARS = 2880   
# TEST_BARS  =  480  
# STEP_BARS  =  240  
# PURGE_BARS =   96   

# USE_HEAVY_MODELS = False

# STRESS_MEDIAN_WINDOW = 365 * BARS_PER_DAY   

# def _to_datetime_index(df: pd.DataFrame) -> pd.DataFrame:
#     if not isinstance(df.index, pd.DatetimeIndex):
#         df = df.copy()
#         df.index = pd.to_datetime(df.index)
#     return df

# def prepare_universe(
#     price_df:   pd.DataFrame,
#     volume_ask: pd.DataFrame,
#     volume_bid: pd.DataFrame,
#     high_df:    pd.DataFrame,
#     low_df:     pd.DataFrame,
# ) -> dict:
#     """
#     Нарезает широкие датафреймы по нужным тикерам и дате.
#     Возвращает dict:  asset_key → {"close", "high", "low", "vol_ask", "vol_bid"}
#     каждый — pd.Series с DatetimeIndex.
#     """
#     for name, df in [("price_df", price_df), ("volume_ask", volume_ask),
#                      ("volume_bid", volume_bid), ("high_df", high_df), ("low_df", low_df)]:
#         if not isinstance(df.index, pd.DatetimeIndex):
#             raise TypeError(f"{name}: index must be DatetimeIndex, got {type(df.index)}")

#     cut = pd.Timestamp(START_DATE)
#     assets = {}
#     missing_tickers = []

#     for asset, ticker in TICKER_MAP.items():
#         if ticker not in price_df.columns:
#             missing_tickers.append(ticker)
#             continue

#         close   = price_df[ticker].loc[cut:].copy()
#         vol_ask = volume_ask[ticker].loc[cut:].copy() if ticker in volume_ask.columns else pd.Series(np.nan, index=close.index)
#         vol_bid = volume_bid[ticker].loc[cut:].copy() if ticker in volume_bid.columns else pd.Series(np.nan, index=close.index)
#         high    = high_df[ticker].loc[cut:].copy()    if ticker in high_df.columns    else pd.Series(np.nan, index=close.index)
#         low     = low_df[ticker].loc[cut:].copy()     if ticker in low_df.columns     else pd.Series(np.nan, index=close.index)

#         valid = close.notna()
#         assets[asset] = {
#             "close":   close[valid],
#             "high":    high[valid],
#             "low":     low[valid],
#             "vol_ask": vol_ask[valid],
#             "vol_bid": vol_bid[valid],
#         }
#         print(f"  {ticker}: {valid.sum():6d} bars  "
#               f"({close[valid].index[0].date()} → {close[valid].index[-1].date()})")

#     if missing_tickers:
#         print(f"\n  WARNING — tickers not found in price_df: {missing_tickers}")
#         print("  Available columns sample:", list(price_df.columns[:10]))

#     return assets

# def build_common_index(assets: dict, min_coverage: float = 0.7) -> pd.DatetimeIndex:
#     """
#     Строит общий индекс баров. Тикер включается если покрывает >= min_coverage от общего диапазона.
#     """
#     all_idx = pd.DatetimeIndex(
#         sorted(set().union(*[set(v["close"].index) for v in assets.values()]))
#     )
#     print(f"\n  Union index: {len(all_idx)} bars")

#     kept = []
#     for asset, data in assets.items():
#         cov = data["close"].reindex(all_idx).notna().mean()
#         if cov >= min_coverage:
#             kept.append(asset)
#             print(f"  {asset}: coverage {cov:.1%}  ✓")
#         else:
#             print(f"  {asset}: coverage {cov:.1%}  ✗ skipped")

#     idx = all_idx
#     for asset in kept:
#         idx = idx.intersection(assets[asset]["close"].index)

#     print(f"  Intersection index: {len(idx)} bars  "
#           f"({idx[0].date()} → {idx[-1].date()})")
#     return idx, kept

# def log_returns(close_wide: pd.DataFrame) -> pd.DataFrame:
#     return np.log(close_wide / close_wide.shift(1))

# def build_targets(
#     ret:          pd.DataFrame,
#     target_asset: str,
# ) -> pd.DataFrame:
#     """
#     Для одного target_asset строит таргеты на все горизонты.
#     Все вычисления — строго на ret (нет обращения к будущему в фичах).
#     """
#     r       = ret[target_asset]
#     vol_now = r.rolling(VOL_WINDOW).std() * ANN_FACTOR

#     tgt = pd.DataFrame(index=ret.index)
#     tgt["vol_now"] = vol_now

#     for h in HORIZONS:
#         vol_fwd = r.rolling(h).std().shift(-h) * ANN_FACTOR

#         vol_med = vol_now.rolling(STRESS_MEDIAN_WINDOW, min_periods=BARS_PER_DAY * 30).median()

#         tgt[f"vol_fwd_{h}"]     = vol_fwd
#         tgt[f"log_vol_fwd_{h}"] = np.log(vol_fwd.clip(lower=1e-8))
#         tgt[f"stress_{h}"]      = (vol_fwd > vol_med * STRESS_MULT).astype(float)
#         tgt[f"delta_vol_{h}"]   = vol_fwd - vol_now

#         n_stress = tgt[f"stress_{h}"].sum()
#         n_total  = tgt[f"stress_{h}"].notna().sum()
#         print(f"    {target_asset} h={h:3d}: stress {int(n_stress):5d}/{n_total} "
#               f"({n_stress/n_total*100:.1f}%)")

#     return tgt

# def _rank_normalize(arr: np.ndarray, bins: int) -> np.ndarray:
#     """
#     Превращает 1D массив в дискретные бины [0, bins) через rank-нормализацию.
#     Устойчиво к выбросам (в отличие от равномерных бинов по диапазону значений).
#     Векторизовано: работает на всём массиве сразу.
#     """
#     n = len(arr)
#     ranks = np.argsort(np.argsort(arr))          # ранги 0..n-1
#     return (ranks * bins // n).clip(0, bins - 1) # → [0, bins)

# def _batch_mi_single_pair(
#     col_x:  np.ndarray,   # shape (n_bars,) — полный ряд одного актива
#     col_y:  np.ndarray,   # shape (n_bars,) — полный ряд другого актива
#     window: int,
#     k:      int,
#     bins:   int,
# ) -> np.ndarray:
#     """
#     Считает rolling MI для одной пары (col_x, col_y) полностью без Python-цикла.

#     Подход:
#       1. Нарезаем все окна сразу через stride_tricks → shape (n_windows, window)
#       2. Rank-нормализуем каждое окно → дискретные бины
#       3. Кодируем совместный бин как  x_bin * bins + y_bin  → 1D индекс
#       4. Считаем joint histogram через np.bincount батчем → (n_windows, bins²)
#       5. Вычисляем MI через матричные операции

#     Нет ни одного Python-цикла — всё в C/numpy.
#     """
#     n = len(col_x)

#     starts = np.arange(0, n - window + 1, k)       
#     ends   = starts + window                        
#     n_win  = len(starts)

#     wx = np.stack([col_x[s:e] for s, e in zip(starts, ends)]) 
#     wy = np.stack([col_y[s:e] for s, e in zip(starts, ends)])  

#     def rank_bins_2d(mat):
#         rk = np.argsort(np.argsort(mat, axis=1), axis=1)
#         return (rk * bins // window).clip(0, bins - 1).astype(np.int32)

#     bx = rank_bins_2d(wx)  
#     by = rank_bins_2d(wy)  

#     joint = bx * bins + by   

#     bins2 = bins * bins
#     offsets = (np.arange(n_win) * bins2).reshape(-1, 1)   
#     flat    = (joint + offsets).ravel()                    
#     counts  = np.bincount(flat, minlength=n_win * bins2).reshape(n_win, bins2)

#     counts  = counts.astype(np.float64) + 1e-10            
#     p_xy    = counts / counts.sum(axis=1, keepdims=True)

#     p_xy_2d = p_xy.reshape(n_win, bins, bins)              
#     p_x     = p_xy_2d.sum(axis=2, keepdims=True)        
#     p_y     = p_xy_2d.sum(axis=1, keepdims=True)         

#     mi_vals = (p_xy_2d * np.log(p_xy_2d / (p_x * p_y))).sum(axis=(1, 2))
#     mi_vals = np.maximum(mi_vals, 0.0)                     

#     result = np.full(n, np.nan)
#     result[ends - 1] = mi_vals   
#     mask = np.isfinite(result)
#     idx  = np.where(mask, np.arange(n), 0)
#     np.maximum.accumulate(idx, out=idx)
#     result = result[idx]
#     return result

# def rolling_vol_features(
#     ret:       pd.DataFrame,
#     high_wide: pd.DataFrame,
#     low_wide:  pd.DataFrame,
#     vask_wide: pd.DataFrame,
#     vbid_wide: pd.DataFrame,
#     windows:   list = [4, 8, 16, 48, 96],
# ) -> pd.DataFrame:
#     feats = {}

#     for asset in ret.columns:
#         r = ret[asset]

#         for w in windows:
#             feats[f"{asset}_rvol_{w}"] = r.rolling(w).std() * ANN_FACTOR

#         rv = r.rolling(VOL_WINDOW).std() * ANN_FACTOR

#         feats[f"{asset}_vov"]         = rv.rolling(VOL_WINDOW).std()

#         rv_mean = rv.rolling(VOL_WINDOW * 4).mean()
#         rv_std  = rv.rolling(VOL_WINDOW * 4).std() + 1e-8
#         feats[f"{asset}_vol_zscore"]  = (rv - rv_mean) / rv_std

#         feats[f"{asset}_vol_pctrank"] = rv.rolling(VOL_WINDOW * 30).rank(pct=True)

#         h = high_wide[asset] if asset in high_wide.columns else pd.Series(np.nan, index=ret.index)
#         l = low_wide[asset]  if asset in low_wide.columns  else pd.Series(np.nan, index=ret.index)
#         hl2 = np.log((h / l).clip(lower=1e-8)) ** 2
#         feats[f"{asset}_pk_vol"] = np.sqrt(
#             hl2.rolling(VOL_WINDOW).mean() / (4 * np.log(2))
#         ) * ANN_FACTOR

#         va = vask_wide[asset] if asset in vask_wide.columns else pd.Series(0.0, index=ret.index)
#         vb = vbid_wide[asset] if asset in vbid_wide.columns else pd.Series(0.0, index=ret.index)
#         total = (va + vb).replace(0, np.nan)
#         raw_imb = (va - vb) / total
#         feats[f"{asset}_vol_imbalance"]     = raw_imb.rolling(8).mean()   
#         feats[f"{asset}_vol_imbalance_std"] = raw_imb.rolling(48).std()

#         for w in [4, 16, 48]:
#             feats[f"{asset}_mom_{w}"] = r.rolling(w).mean()

#     return pd.DataFrame(feats, index=ret.index)

# def cross_covariance_features(ret: pd.DataFrame, window: int = VOL_WINDOW) -> pd.DataFrame:
#     pairs = list(combinations(ret.columns, 2))
#     feats = {}
#     for a, b in pairs:
#         cov = ret[a].rolling(window).cov(ret[b])
#         std_a = ret[a].rolling(window).std() + 1e-8
#         std_b = ret[b].rolling(window).std() + 1e-8
#         feats[f"corr_{a}_{b}"] = cov / (std_a * std_b)
#     return pd.DataFrame(feats, index=ret.index)

# def pairwise_mi_features(
#     ret:    pd.DataFrame,
#     window: int = MI_WINDOW,
#     bins:   int = MI_BINS,
#     k:      int = MI_SUBSAMPLE_K,
# ) -> pd.DataFrame:
#     """
#     Rolling pairwise MI — полностью векторизовано, без Python-цикла по барам.
#     Цикл только по парам активов (36 итераций), внутри каждой — чистый numpy.
#     """
#     pairs   = list(combinations(ret.columns, 2))
#     ret_np  = ret.to_numpy(dtype=float)
#     col_idx = {c: i for i, c in enumerate(ret.columns)}
#     n       = len(ret)

#     n_windows = len(range(0, n - window + 1, k))
#     print(f"    pairwise MI: {len(pairs)} pairs, {n_windows} windows "
#           f"(k={k}) — fully vectorized...")

#     feat_dict = {}
#     for a, b in tqdm(pairs, desc="pairwise MI", unit="pair"):
#         feat_dict[f"mi_{a}_{b}"] = _batch_mi_single_pair(
#             ret_np[:, col_idx[a]], ret_np[:, col_idx[b]], window, k, bins
#         )

#     return pd.DataFrame(feat_dict, index=ret.index)

# def triplet_mi_features(
#     ret:    pd.DataFrame,
#     window: int = MI_WINDOW,
#     bins:   int = MI_BINS,
#     k:      int = MI_SUBSAMPLE_K,
# ) -> pd.DataFrame:
#     """
#     Rolling triplet MI = MI(a,b) + MI(b,c) + MI(a,c) — векторизовано.
#     Каждая пара считается один раз через _batch_mi_single_pair, потом суммируется.
#     """
#     triplets = list(combinations(ret.columns, 3))
#     ret_np   = ret.to_numpy(dtype=float)
#     col_idx  = {c: i for i, c in enumerate(ret.columns)}
#     n        = len(ret)

#     all_pairs = list(combinations(ret.columns, 2))
#     n_windows = len(range(0, n - window + 1, k))
#     print(f"    triplet MI: pre-computing {len(all_pairs)} pairs "
#           f"for {len(triplets)} triplets — vectorized...")

#     pair_mi = {}
#     for a, b in tqdm(all_pairs, desc="triplet MI (pairs)", unit="pair"):
#         key = (a, b)
#         pair_mi[key] = _batch_mi_single_pair(
#             ret_np[:, col_idx[a]], ret_np[:, col_idx[b]], window, k, bins
#         )

#     feat_dict = {}
#     for a, b, c in triplets:
#         feat_dict[f"tri_{a}_{b}_{c}"] = (
#             pair_mi[(a, b)] + pair_mi[(b, c)] + pair_mi[(a, c)]
#         )

#     return pd.DataFrame(feat_dict, index=ret.index)

# def build_features(ret: pd.DataFrame, high_w: pd.DataFrame,
#                    low_w: pd.DataFrame, vask_w: pd.DataFrame,
#                    vbid_w: pd.DataFrame,
#                    target_assets: list = None) -> pd.DataFrame:

#     print("  [feat] rolling vol / Parkinson / volume imbalance...")
#     vol_f = rolling_vol_features(ret, high_w, low_w, vask_w, vbid_w)

#     print("  [feat] rolling cross-correlation...")
#     corr_f = cross_covariance_features(ret)

#     print("  [feat] pairwise MI...")
#     mi_f = pairwise_mi_features(ret)

#     print("  [feat] triplet MI...")
#     tri_f = triplet_mi_features(ret)

#     X = pd.concat([vol_f, corr_f, mi_f, tri_f], axis=1)

#     assets = target_assets if target_assets else TARGET_ASSETS
#     vol_now_cols = [f"{a}_rvol_{VOL_WINDOW}" for a in assets if f"{a}_rvol_{VOL_WINDOW}" in X.columns]
#     X["vol_now_feat"] = X[vol_now_cols].mean(axis=1)

#     print(f"  [feat] total: {X.shape[1]} features  ×  {X.shape[0]} bars")
#     return X
#     return X

# def _wf_splits(n: int, train: int, test: int, step: int, purge: int):
#     """
#     Генератор (train_idx, test_idx) с purge-зазором.
#     train_end → +purge → test_start
#     """
#     start = train
#     while start + purge + test <= n:
#         train_idx = list(range(start - train, start))
#         test_start = start + purge
#         test_idx  = list(range(test_start, test_start + test))
#         yield train_idx, test_idx
#         start += step

# def run_wf_classification(X, y, models,
#                            train=TRAIN_BARS, test=TEST_BARS,
#                            step=STEP_BARS, purge=PURGE_BARS):
#     mask   = X.notna().all(axis=1) & y.notna()
#     Xc, yc = X[mask], y[mask]
#     splits = list(_wf_splits(len(Xc), train, test, step, purge))

#     if not splits:
#         print("    ! Not enough data for walk-forward, skipping")
#         return pd.DataFrame()

#     records = []
#     for fold, (tr, te) in enumerate(tqdm(splits, desc="WF folds", unit="fold")):
#         Xtr, Xte = Xc.iloc[tr], Xc.iloc[te]
#         ytr, yte = yc.iloc[tr], yc.iloc[te]

#         if len(ytr.unique()) < 2:
#             continue  # фолд без обоих классов

#         for name, mdl in models.items():
#             mdl.fit(Xtr, ytr)
#             prob = mdl.predict_proba(Xte)[:, 1]
#             pred = mdl.predict(Xte)
#             try:
#                 roc = roc_auc_score(yte, prob)
#                 pr  = average_precision_score(yte, prob)
#             except Exception:
#                 roc, pr = np.nan, np.nan

#             records.append({
#                 "fold": fold, "model": name,
#                 "roc_auc":  roc,
#                 "pr_auc":   pr,
#                 "f1":       f1_score(yte, pred, zero_division=0),
#                 "bal_acc":  balanced_accuracy_score(yte, pred),
#                 "pos_rate": yte.mean(),
#                 "n":        len(yte),
#             })

#     return pd.DataFrame(records)

# def run_wf_regression(X, y, models,
#                        train=TRAIN_BARS, test=TEST_BARS,
#                        step=STEP_BARS, purge=PURGE_BARS):
#     mask   = X.notna().all(axis=1) & y.notna()
#     Xc, yc = X[mask], y[mask]
#     splits = list(_wf_splits(len(Xc), train, test, step, purge))

#     if not splits:
#         print("    ! Not enough data for walk-forward, skipping")
#         return pd.DataFrame()

#     records = []
#     for fold, (tr, te) in enumerate(tqdm(splits, desc="WF folds", unit="fold")):
#         Xtr, Xte = Xc.iloc[tr], Xc.iloc[te]
#         ytr, yte = yc.iloc[tr], yc.iloc[te]

#         for name, mdl in models.items():
#             mdl.fit(Xtr, ytr)
#             pred = mdl.predict(Xte)
#             corr = float(pd.Series(pred).corr(pd.Series(yte.values)))

#             records.append({
#                 "fold": fold, "model": name,
#                 "mae":  mean_absolute_error(yte, pred),
#                 "r2":   r2_score(yte, pred),
#                 "corr": corr,
#                 "n":    len(yte),
#             })

#     return pd.DataFrame(records)

# class PersistenceClsBaseline:
#     """
#     Классификатор-baseline: предсказывает стресс если vol_now > порога.
#     Порог = rolling quantile(stress_mult) внутри train-сета.
#     predict_proba возвращает нормированную vol_now как "вероятность".
#     Не требует обучения — fit() только запоминает порог.
#     """
#     def __init__(self, vol_now_col: str = "vol_now_feat"):
#         self.vol_now_col = vol_now_col
#         self.threshold_  = None

#     def fit(self, X, y):
#         if self.vol_now_col in X.columns:
#             vol = X[self.vol_now_col].values
#         else:
#             vol = X.iloc[:, 0].values   # fallback: первая колонка
#         pos_rate = float(y.mean())
#         self.threshold_ = np.quantile(vol[np.isfinite(vol)], 1 - pos_rate)
#         return self

#     def predict_proba(self, X):
#         if self.vol_now_col in X.columns:
#             vol = X[self.vol_now_col].values
#         else:
#             vol = X.iloc[:, 0].values
#         vol = np.where(np.isfinite(vol), vol, 0.0)
#         vmin, vmax = vol.min(), vol.max()
#         prob = (vol - vmin) / (vmax - vmin + 1e-8)
#         return np.column_stack([1 - prob, prob])

#     def predict(self, X):
#         return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

# class PersistenceRegBaseline:
#     """
#     Регрессор-baseline: предсказывает vol_fwd = vol_now (persistence).
#     Для log_vol_fwd — возвращает log(vol_now).
#     Для delta_vol   — возвращает 0 (нет изменения).
#     fit() просто запоминает среднее смещение на train (bias correction).
#     """
#     def __init__(self, vol_now_col: str = "vol_now_feat"):
#         self.vol_now_col = vol_now_col
#         self.bias_       = 0.0

#     def fit(self, X, y):
#         if self.vol_now_col in X.columns:
#             vol = X[self.vol_now_col].values
#         else:
#             vol = X.iloc[:, 0].values
#         pred = vol[np.isfinite(vol) & np.isfinite(y.values)]
#         ytrue = y.values[np.isfinite(vol) & np.isfinite(y.values)]
#         self.bias_ = float(np.mean(ytrue - pred)) if len(pred) else 0.0
#         return self

#     def predict(self, X):
#         if self.vol_now_col in X.columns:
#             vol = X[self.vol_now_col].values
#         else:
#             vol = X.iloc[:, 0].values
#         return np.where(np.isfinite(vol), vol + self.bias_, np.nanmean(vol))

# def get_cls_models() -> dict:
#     models = {
#         "Persistence": PersistenceClsBaseline(vol_now_col="vol_now_feat"),
#         "LogReg": Pipeline([
#             ("sc", StandardScaler()),
#             ("m",  LogisticRegression(C=0.1, max_iter=500, class_weight="balanced")),
#         ]),
#     }
#     if USE_HEAVY_MODELS:
#         models["LGBM"] = lgb.LGBMClassifier(
#             n_estimators=300, max_depth=4,
#             learning_rate=0.03, subsample=0.8,
#             colsample_bytree=0.8, min_child_samples=20,
#             class_weight="balanced", n_jobs=-1,
#             verbose=-1, random_state=42,
#         )
#     return models

# def get_reg_models() -> dict:
#     models = {
#         "Persistence": PersistenceRegBaseline(vol_now_col="vol_now_feat"),
#         "Ridge": Pipeline([
#             ("sc", StandardScaler()),
#             ("m",  Ridge(alpha=1.0)),
#         ]),
#     }
#     if USE_HEAVY_MODELS:
#         models["LGBM"] = lgb.LGBMRegressor(
#             n_estimators=300, max_depth=4,
#             learning_rate=0.03, subsample=0.8,
#             colsample_bytree=0.8, min_child_samples=20,
#             n_jobs=-1, verbose=-1, random_state=42,
#         )
#     return models

# def _summarize(df: pd.DataFrame, metric_cols: list) -> pd.DataFrame:
#     if df.empty:
#         return pd.DataFrame()
#     return (
#         df.groupby("model")[metric_cols]
#         .agg(["mean", "std"])
#         .round(4)
#     )

# def print_block(title: str):
#     print(f"\n{'═'*65}")
#     print(f"  {title}")
#     print(f"{'═'*65}")

# def run_experiment(
#     price_df:   pd.DataFrame,
#     volume_ask: pd.DataFrame,
#     volume_bid: pd.DataFrame,
#     high_df:    pd.DataFrame,
#     low_df:     pd.DataFrame,
# ) -> dict:
#     """
#     Запускается так:
#         results = run_experiment(price_df, volume_ask, volume_bid, high_df, low_df)

#     price_df = (bt2.data['STRATEX-BINANCE']['best_ask']
#               + bt2.data['STRATEX-BINANCE']['best_bid']) / 2
#     volume_ask = bt2.data['STRATEX-BINANCE']['buy_volume']
#     volume_bid = bt2.data['STRATEX-BINANCE']['sell_volume']
#     high_df    = bt2.data['STRATEX-BINANCE-F']['high']
#     low_df     = bt2.data['STRATEX-BINANCE-F']['low']
#     """

#     print_block("CRYPTO VOL STRESS EXPERIMENT  —  30-min bars")
#     print(f"  Targets  : {TARGET_ASSETS}")
#     print(f"  Universe : {UNIVERSE}")
#     print(f"  Horizons : {HORIZONS} bars  =  "
#           f"{[f'{h//2}h' if h < 48 else f'{h//48}d' for h in HORIZONS]}")
#     print(f"  Date     : {START_DATE} → end")
#     print(f"  WF split : train={TRAIN_BARS} / test={TEST_BARS} / "
#           f"step={STEP_BARS} / purge={PURGE_BARS} bars")

#     print_block("1. Preparing asset data")
#     assets = prepare_universe(price_df, volume_ask, volume_bid, high_df, low_df)
#     common_idx, active_assets = build_common_index(assets)

#     for t in TARGET_ASSETS:
#         if t not in active_assets:
#             raise ValueError(
#                 f"Target asset '{t}' ({TICKER_MAP[t]}) not found or has insufficient coverage. "
#                 f"Active assets: {active_assets}"
#             )

#     def _wide(field: str) -> pd.DataFrame:
#         return pd.DataFrame(
#             {a: assets[a][field].reindex(common_idx) for a in active_assets}
#         )

#     close_w  = _wide("close")
#     high_w   = _wide("high")
#     low_w    = _wide("low")
#     vask_w   = _wide("vol_ask")
#     vbid_w   = _wide("vol_bid")

#     ret = log_returns(close_w)

#     print_block("2. Building features")
#     X = build_features(ret, high_w, low_w, vask_w, vbid_w)

#     all_results = {}

#     for target in TARGET_ASSETS:
#         print_block(f"3. Targets  →  {target}")
#         tgt = build_targets(ret, target)
#         tgt = tgt.loc[X.index]

#         for h in HORIZONS:
#             horizon_label = f"{h//2}h" if h < 48 else f"{h//48}d"
#             print(f"\n  ── horizon h={h} ({horizon_label}) ──")

#             y_cls = tgt[f"stress_{h}"].dropna()
#             Xc    = X.loc[y_cls.index]
#             pos   = y_cls.mean()
#             print(f"    [CLS] stress event  pos_rate={pos:.2%}")
#             df_cls = run_wf_classification(Xc, y_cls, get_cls_models())
#             s_cls  = _summarize(df_cls, ["roc_auc", "pr_auc", "f1", "bal_acc"])
#             if not s_cls.empty:
#                 print(s_cls.to_string())
#             key = f"{target}_CLS_stress_h{h}"
#             all_results[key] = s_cls

#             y_vol = tgt[f"log_vol_fwd_{h}"].dropna()
#             Xr    = X.loc[y_vol.index]
#             print(f"    [REG] log vol level")
#             df_vol = run_wf_regression(Xr, y_vol, get_reg_models())
#             s_vol  = _summarize(df_vol, ["mae", "r2", "corr"])
#             if not s_vol.empty:
#                 print(s_vol.to_string())
#             all_results[f"{target}_REG_logvol_h{h}"] = s_vol

#             y_dv = tgt[f"delta_vol_{h}"].dropna()
#             Xd   = X.loc[y_dv.index]
#             print(f"    [REG] delta vol")
#             df_dv = run_wf_regression(Xd, y_dv, get_reg_models())
#             s_dv  = _summarize(df_dv, ["mae", "r2", "corr"])
#             if not s_dv.empty:
#                 print(s_dv.to_string())
#             all_results[f"{target}_REG_delta_h{h}"] = s_dv

#     print_block("FINAL SUMMARY  (mean across walk-forward folds)")
#     rows = []
#     for exp_key, df in all_results.items():
#         if df.empty:
#             continue
#         for model_name in df.index.get_level_values(0).unique():
#             row = {"experiment": exp_key, "model": model_name}
#             for col in df.columns.get_level_values(0).unique():
#                 row[f"{col}_mean"] = df.loc[model_name, (col, "mean")]
#                 row[f"{col}_std"]  = df.loc[model_name, (col, "std")]
#             rows.append(row)

#     summary = pd.DataFrame(rows).set_index(["experiment", "model"])
#     print(summary.to_string())

#     return all_results, summary

# price_df   = (bt2.data['STRATEX-BINANCE']['best_ask']
#             + bt2.data['STRATEX-BINANCE']['best_bid']) / 2
# volume_ask = bt2.data['STRATEX-BINANCE']['buy_volume']
# volume_bid = bt2.data['STRATEX-BINANCE']['sell_volume']
# high_df    = bt2.data['STRATEX-BINANCE-F']['high']
# low_df     = bt2.data['STRATEX-BINANCE-F']['low']

# results, summary = run_experiment(
#     price_df, volume_ask, volume_bid, high_df, low_df
# )

In [ ]:
# import warnings
# warnings.filterwarnings("ignore")

# import numpy as np
# import pandas as pd
# from itertools import combinations
# from tqdm.auto import tqdm

# from sklearn.metrics import (
#     roc_auc_score, average_precision_score,
#     f1_score, balanced_accuracy_score,
#     mean_absolute_error, r2_score,
# )
# from sklearn.linear_model import LogisticRegression, Ridge
# from sklearn.preprocessing import StandardScaler
# from sklearn.pipeline import Pipeline
# import lightgbm as lgb

# TICKER_MAP = {
#     "BTC":  "BTC-USDT",
#     "ETH":  "ETH-USDT",
#     "SOL":  "SOL-USDT",
#     "BNB":  "BNB-USDT",
#     "XRP":  "XRP-USDT",
#     "DOGE": "DOGE-USDT",
#     "ADA":  "ADA-USDT",
#     "AVAX": "AVAX-USDT",
#     "TRX":  "TRX-USDT",
# }

# TARGET_ASSETS = ["BTC", "ETH"]          
# UNIVERSE      = list(TICKER_MAP.keys()) 

# START_DATE = "2023-01-01"              

# BARS_PER_DAY = 48
# ANN_FACTOR   = np.sqrt(365 * BARS_PER_DAY)  

# VOL_WINDOW = 48     # 24h

# STRESS_MULT = 1.25

# HORIZONS = [8, 48, 96]    # 4h / 24h / 48h

# MI_WINDOW      = 192   
# MI_BINS        = 10
# MI_SUBSAMPLE_K = 4     

# TRAIN_BARS = 2880   
# TEST_BARS  =  480   
# STEP_BARS  =  240   
# PURGE_BARS =   96  

# USE_HEAVY_MODELS = False

# STRESS_MEDIAN_WINDOW = 365 * BARS_PER_DAY  

# def _to_datetime_index(df: pd.DataFrame) -> pd.DataFrame:
#     if not isinstance(df.index, pd.DatetimeIndex):
#         df = df.copy()
#         df.index = pd.to_datetime(df.index)
#     return df

# def prepare_universe(
#     price_df:   pd.DataFrame,
#     volume_ask: pd.DataFrame,
#     volume_bid: pd.DataFrame,
#     high_df:    pd.DataFrame,
#     low_df:     pd.DataFrame,
# ) -> dict:
#     """
#     Нарезает широкие датафреймы по нужным тикерам и дате.
#     Возвращает dict:  asset_key → {"close", "high", "low", "vol_ask", "vol_bid"}
#     каждый — pd.Series с DatetimeIndex.
#     """
#     for name, df in [("price_df", price_df), ("volume_ask", volume_ask),
#                      ("volume_bid", volume_bid), ("high_df", high_df), ("low_df", low_df)]:
#         if not isinstance(df.index, pd.DatetimeIndex):
#             raise TypeError(f"{name}: index must be DatetimeIndex, got {type(df.index)}")

#     cut = pd.Timestamp(START_DATE)
#     assets = {}
#     missing_tickers = []

#     for asset, ticker in TICKER_MAP.items():
#         if ticker not in price_df.columns:
#             missing_tickers.append(ticker)
#             continue

#         close   = price_df[ticker].loc[cut:].copy()
#         vol_ask = volume_ask[ticker].loc[cut:].copy() if ticker in volume_ask.columns else pd.Series(np.nan, index=close.index)
#         vol_bid = volume_bid[ticker].loc[cut:].copy() if ticker in volume_bid.columns else pd.Series(np.nan, index=close.index)
#         high    = high_df[ticker].loc[cut:].copy()    if ticker in high_df.columns    else pd.Series(np.nan, index=close.index)
#         low     = low_df[ticker].loc[cut:].copy()     if ticker in low_df.columns     else pd.Series(np.nan, index=close.index)

#         valid = close.notna()
#         assets[asset] = {
#             "close":   close[valid],
#             "high":    high[valid],
#             "low":     low[valid],
#             "vol_ask": vol_ask[valid],
#             "vol_bid": vol_bid[valid],
#         }
#         print(f"  {ticker}: {valid.sum():6d} bars  "
#               f"({close[valid].index[0].date()} → {close[valid].index[-1].date()})")

#     if missing_tickers:
#         print(f"\n  WARNING — tickers not found in price_df: {missing_tickers}")
#         print("  Available columns sample:", list(price_df.columns[:10]))

#     return assets

# def build_common_index(assets: dict, min_coverage: float = 0.7) -> pd.DatetimeIndex:
#     """
#     Строит общий индекс баров. Тикер включается если покрывает >= min_coverage от общего диапазона.
#     """
#     all_idx = pd.DatetimeIndex(
#         sorted(set().union(*[set(v["close"].index) for v in assets.values()]))
#     )
#     print(f"\n  Union index: {len(all_idx)} bars")

#     kept = []
#     for asset, data in assets.items():
#         cov = data["close"].reindex(all_idx).notna().mean()
#         if cov >= min_coverage:
#             kept.append(asset)
#             print(f"  {asset}: coverage {cov:.1%}  ✓")
#         else:
#             print(f"  {asset}: coverage {cov:.1%}  ✗ skipped")

#     idx = all_idx
#     for asset in kept:
#         idx = idx.intersection(assets[asset]["close"].index)

#     print(f"  Intersection index: {len(idx)} bars  "
#           f"({idx[0].date()} → {idx[-1].date()})")
#     return idx, kept

# def log_returns(close_wide: pd.DataFrame) -> pd.DataFrame:
#     return np.log(close_wide / close_wide.shift(1))

# def build_targets(
#     ret:          pd.DataFrame,
#     target_asset: str,
# ) -> pd.DataFrame:
#     """
#     Для одного target_asset строит таргеты на все горизонты.
#     Все вычисления — строго на ret (нет обращения к будущему в фичах).
#     """
#     r       = ret[target_asset]
#     vol_now = r.rolling(VOL_WINDOW).std() * ANN_FACTOR

#     tgt = pd.DataFrame(index=ret.index)
#     tgt["vol_now"] = vol_now

#     for h in HORIZONS:
#         vol_fwd = r.rolling(h).std().shift(-h) * ANN_FACTOR

#         vol_med = vol_now.rolling(STRESS_MEDIAN_WINDOW, min_periods=BARS_PER_DAY * 30).median()

#         tgt[f"vol_fwd_{h}"]     = vol_fwd
#         tgt[f"log_vol_fwd_{h}"] = np.log(vol_fwd.clip(lower=1e-8))
#         tgt[f"stress_{h}"]      = (vol_fwd > vol_med * STRESS_MULT).astype(float)
#         tgt[f"delta_vol_{h}"]   = vol_fwd - vol_now

#         n_stress = tgt[f"stress_{h}"].sum()
#         n_total  = tgt[f"stress_{h}"].notna().sum()
#         print(f"    {target_asset} h={h:3d}: stress {int(n_stress):5d}/{n_total} "
#               f"({n_stress/n_total*100:.1f}%)")

#     return tgt

# def _rank_normalize(arr: np.ndarray, bins: int) -> np.ndarray:
#     """
#     Превращает 1D массив в дискретные бины [0, bins) через rank-нормализацию.
#     Устойчиво к выбросам (в отличие от равномерных бинов по диапазону значений).
#     Векторизовано: работает на всём массиве сразу.
#     """
#     n = len(arr)
#     ranks = np.argsort(np.argsort(arr))          # ранги 0..n-1
#     return (ranks * bins // n).clip(0, bins - 1) # → [0, bins)

# def _batch_mi_single_pair(
#     col_x:  np.ndarray,   # shape (n_bars,) — полный ряд одного актива
#     col_y:  np.ndarray,   # shape (n_bars,) — полный ряд другого актива
#     window: int,
#     k:      int,
#     bins:   int,
# ) -> np.ndarray:
#     """
#     Считает rolling MI для одной пары (col_x, col_y) полностью без Python-цикла.

#     Подход:
#       1. Нарезаем все окна сразу через stride_tricks → shape (n_windows, window)
#       2. Rank-нормализуем каждое окно → дискретные бины
#       3. Кодируем совместный бин как  x_bin * bins + y_bin  → 1D индекс
#       4. Считаем joint histogram через np.bincount батчем → (n_windows, bins²)
#       5. Вычисляем MI через матричные операции

#     Нет ни одного Python-цикла — всё в C/numpy.
#     """
#     n = len(col_x)

#     starts = np.arange(0, n - window + 1, k)          
#     ends   = starts + window                           
#     n_win  = len(starts)

#     wx = np.stack([col_x[s:e] for s, e in zip(starts, ends)])  
#     wy = np.stack([col_y[s:e] for s, e in zip(starts, ends)]) 

#     def rank_bins_2d(mat):
#         rk = np.argsort(np.argsort(mat, axis=1), axis=1)
#         return (rk * bins // window).clip(0, bins - 1).astype(np.int32)

#     bx = rank_bins_2d(wx)   
#     by = rank_bins_2d(wy)   

#     joint = bx * bins + by   

#     bins2 = bins * bins
#     offsets = (np.arange(n_win) * bins2).reshape(-1, 1)  
#     flat    = (joint + offsets).ravel()                    
#     counts  = np.bincount(flat, minlength=n_win * bins2).reshape(n_win, bins2)

#     counts  = counts.astype(np.float64) + 1e-10            
#     p_xy    = counts / counts.sum(axis=1, keepdims=True)   

#     p_xy_2d = p_xy.reshape(n_win, bins, bins)              
#     p_x     = p_xy_2d.sum(axis=2, keepdims=True)           
#     p_y     = p_xy_2d.sum(axis=1, keepdims=True)           

#     mi_vals = (p_xy_2d * np.log(p_xy_2d / (p_x * p_y))).sum(axis=(1, 2))
#     mi_vals = np.maximum(mi_vals, 0.0)                    

#     result = np.full(n, np.nan)
#     result[ends - 1] = mi_vals    
#     mask = np.isfinite(result)
#     idx  = np.where(mask, np.arange(n), 0)
#     np.maximum.accumulate(idx, out=idx)
#     result = result[idx]
#     return result

# def rolling_vol_features(
#     ret:       pd.DataFrame,
#     high_wide: pd.DataFrame,
#     low_wide:  pd.DataFrame,
#     vask_wide: pd.DataFrame,
#     vbid_wide: pd.DataFrame,
#     windows:   list = [4, 8, 16, 48, 96],
# ) -> pd.DataFrame:
#     feats = {}

#     for asset in ret.columns:
#         r = ret[asset]

#         for w in windows:
#             feats[f"{asset}_rvol_{w}"] = r.rolling(w).std() * ANN_FACTOR

#         rv = r.rolling(VOL_WINDOW).std() * ANN_FACTOR

#         feats[f"{asset}_vov"]         = rv.rolling(VOL_WINDOW).std()

#         rv_mean = rv.rolling(VOL_WINDOW * 4).mean()
#         rv_std  = rv.rolling(VOL_WINDOW * 4).std() + 1e-8
#         feats[f"{asset}_vol_zscore"]  = (rv - rv_mean) / rv_std

#         feats[f"{asset}_vol_pctrank"] = rv.rolling(VOL_WINDOW * 30).rank(pct=True)

#         h = high_wide[asset] if asset in high_wide.columns else pd.Series(np.nan, index=ret.index)
#         l = low_wide[asset]  if asset in low_wide.columns  else pd.Series(np.nan, index=ret.index)
#         hl2 = np.log((h / l).clip(lower=1e-8)) ** 2
#         feats[f"{asset}_pk_vol"] = np.sqrt(
#             hl2.rolling(VOL_WINDOW).mean() / (4 * np.log(2))
#         ) * ANN_FACTOR

#         va = vask_wide[asset] if asset in vask_wide.columns else pd.Series(0.0, index=ret.index)
#         vb = vbid_wide[asset] if asset in vbid_wide.columns else pd.Series(0.0, index=ret.index)
#         total = (va + vb).replace(0, np.nan)
#         raw_imb = (va - vb) / total
#         feats[f"{asset}_vol_imbalance"]     = raw_imb.rolling(8).mean()   
#         feats[f"{asset}_vol_imbalance_std"] = raw_imb.rolling(48).std()

#         for w in [4, 16, 48]:
#             feats[f"{asset}_mom_{w}"] = r.rolling(w).mean()

#     return pd.DataFrame(feats, index=ret.index)

# def cross_covariance_features(ret: pd.DataFrame, window: int = VOL_WINDOW) -> pd.DataFrame:
#     pairs = list(combinations(ret.columns, 2))
#     feats = {}
#     for a, b in pairs:
#         cov = ret[a].rolling(window).cov(ret[b])
#         std_a = ret[a].rolling(window).std() + 1e-8
#         std_b = ret[b].rolling(window).std() + 1e-8
#         feats[f"corr_{a}_{b}"] = cov / (std_a * std_b)
#     return pd.DataFrame(feats, index=ret.index)

# def pairwise_mi_features(
#     ret:    pd.DataFrame,
#     window: int = MI_WINDOW,
#     bins:   int = MI_BINS,
#     k:      int = MI_SUBSAMPLE_K,
# ) -> pd.DataFrame:
#     """
#     Rolling pairwise MI — полностью векторизовано, без Python-цикла по барам.
#     Цикл только по парам активов (36 итераций), внутри каждой — чистый numpy.
#     """
#     pairs   = list(combinations(ret.columns, 2))
#     ret_np  = ret.to_numpy(dtype=float)
#     col_idx = {c: i for i, c in enumerate(ret.columns)}
#     n       = len(ret)

#     n_windows = len(range(0, n - window + 1, k))
#     print(f"    pairwise MI: {len(pairs)} pairs, {n_windows} windows "
#           f"(k={k}) — fully vectorized...")

#     feat_dict = {}
#     for a, b in tqdm(pairs, desc="pairwise MI", unit="pair"):
#         feat_dict[f"mi_{a}_{b}"] = _batch_mi_single_pair(
#             ret_np[:, col_idx[a]], ret_np[:, col_idx[b]], window, k, bins
#         )

#     return pd.DataFrame(feat_dict, index=ret.index)

# def triplet_mi_features(
#     ret:    pd.DataFrame,
#     window: int = MI_WINDOW,
#     bins:   int = MI_BINS,
#     k:      int = MI_SUBSAMPLE_K,
# ) -> pd.DataFrame:
#     """
#     Rolling triplet MI = MI(a,b) + MI(b,c) + MI(a,c) — векторизовано.
#     Каждая пара считается один раз через _batch_mi_single_pair, потом суммируется.
#     """
#     triplets = list(combinations(ret.columns, 3))
#     ret_np   = ret.to_numpy(dtype=float)
#     col_idx  = {c: i for i, c in enumerate(ret.columns)}
#     n        = len(ret)

#     all_pairs = list(combinations(ret.columns, 2))
#     n_windows = len(range(0, n - window + 1, k))
#     print(f"    triplet MI: pre-computing {len(all_pairs)} pairs "
#           f"for {len(triplets)} triplets — vectorized...")

#     pair_mi = {}
#     for a, b in tqdm(all_pairs, desc="triplet MI (pairs)", unit="pair"):
#         key = (a, b)
#         pair_mi[key] = _batch_mi_single_pair(
#             ret_np[:, col_idx[a]], ret_np[:, col_idx[b]], window, k, bins
#         )

#     feat_dict = {}
#     for a, b, c in triplets:
#         feat_dict[f"tri_{a}_{b}_{c}"] = (
#             pair_mi[(a, b)] + pair_mi[(b, c)] + pair_mi[(a, c)]
#         )

#     return pd.DataFrame(feat_dict, index=ret.index)

# def build_features(ret: pd.DataFrame, high_w: pd.DataFrame,
#                    low_w: pd.DataFrame, vask_w: pd.DataFrame,
#                    vbid_w: pd.DataFrame,
#                    target_assets: list = None) -> pd.DataFrame:

#     print("  [feat] rolling vol / Parkinson / volume imbalance...")
#     vol_f = rolling_vol_features(ret, high_w, low_w, vask_w, vbid_w)

#     print("  [feat] rolling cross-correlation...")
#     corr_f = cross_covariance_features(ret)

#     print("  [feat] pairwise MI...")
#     mi_f = pairwise_mi_features(ret)

#     print("  [feat] triplet MI...")
#     tri_f = triplet_mi_features(ret)

#     X = pd.concat([vol_f, corr_f, mi_f, tri_f], axis=1)

#     assets = target_assets if target_assets else TARGET_ASSETS
#     vol_now_cols = [f"{a}_rvol_{VOL_WINDOW}" for a in assets if f"{a}_rvol_{VOL_WINDOW}" in X.columns]
#     X["vol_now_feat"] = X[vol_now_cols].mean(axis=1)

#     print(f"  [feat] total: {X.shape[1]} features  ×  {X.shape[0]} bars")
#     return X
#     return X

# def _wf_splits(n: int, train: int, test: int, step: int, purge: int):
#     """
#     Генератор (train_idx, test_idx) с purge-зазором.
#     train_end → +purge → test_start
#     """
#     start = train
#     while start + purge + test <= n:
#         train_idx = list(range(start - train, start))
#         test_start = start + purge
#         test_idx  = list(range(test_start, test_start + test))
#         yield train_idx, test_idx
#         start += step

# def run_wf_classification(X, y, models,
#                            train=TRAIN_BARS, test=TEST_BARS,
#                            step=STEP_BARS, purge=PURGE_BARS):
#     mask   = X.notna().all(axis=1) & y.notna()
#     Xc, yc = X[mask], y[mask]
#     splits = list(_wf_splits(len(Xc), train, test, step, purge))

#     if not splits:
#         print("    ! Not enough data for walk-forward, skipping")
#         return pd.DataFrame()

#     records = []
#     for fold, (tr, te) in enumerate(tqdm(splits, desc="WF folds", unit="fold")):
#         Xtr, Xte = Xc.iloc[tr], Xc.iloc[te]
#         ytr, yte = yc.iloc[tr], yc.iloc[te]

#         if len(ytr.unique()) < 2:
#             continue  # фолд без обоих классов

#         for name, mdl in models.items():
#             mdl.fit(Xtr, ytr)
#             prob = mdl.predict_proba(Xte)[:, 1]
#             pred = mdl.predict(Xte)
#             try:
#                 roc = roc_auc_score(yte, prob)
#                 pr  = average_precision_score(yte, prob)
#             except Exception:
#                 roc, pr = np.nan, np.nan

#             records.append({
#                 "fold": fold, "model": name,
#                 "roc_auc":  roc,
#                 "pr_auc":   pr,
#                 "f1":       f1_score(yte, pred, zero_division=0),
#                 "bal_acc":  balanced_accuracy_score(yte, pred),
#                 "pos_rate": yte.mean(),
#                 "n":        len(yte),
#             })

#     return pd.DataFrame(records)

# def run_wf_regression(X, y, models,
#                        train=TRAIN_BARS, test=TEST_BARS,
#                        step=STEP_BARS, purge=PURGE_BARS):
#     mask   = X.notna().all(axis=1) & y.notna()
#     Xc, yc = X[mask], y[mask]
#     splits = list(_wf_splits(len(Xc), train, test, step, purge))

#     if not splits:
#         print("    ! Not enough data for walk-forward, skipping")
#         return pd.DataFrame()

#     records = []
#     for fold, (tr, te) in enumerate(tqdm(splits, desc="WF folds", unit="fold")):
#         Xtr, Xte = Xc.iloc[tr], Xc.iloc[te]
#         ytr, yte = yc.iloc[tr], yc.iloc[te]

#         for name, mdl in models.items():
#             mdl.fit(Xtr, ytr)
#             pred = mdl.predict(Xte)
#             corr = float(pd.Series(pred).corr(pd.Series(yte.values)))

#             records.append({
#                 "fold": fold, "model": name,
#                 "mae":  mean_absolute_error(yte, pred),
#                 "r2":   r2_score(yte, pred),
#                 "corr": corr,
#                 "n":    len(yte),
#             })

#     return pd.DataFrame(records)

# class PersistenceClsBaseline:
#     """
#     Классификатор-baseline: предсказывает стресс если vol_now > порога.
#     Порог = rolling quantile(stress_mult) внутри train-сета.
#     predict_proba возвращает нормированную vol_now как "вероятность".
#     Не требует обучения — fit() только запоминает порог.
#     """
#     def __init__(self, vol_now_col: str = "vol_now_feat"):
#         self.vol_now_col = vol_now_col
#         self.threshold_  = None

#     def fit(self, X, y):
#         if self.vol_now_col in X.columns:
#             vol = X[self.vol_now_col].values
#         else:
#             vol = X.iloc[:, 0].values   # fallback: первая колонка
#         pos_rate = float(y.mean())
#         self.threshold_ = np.quantile(vol[np.isfinite(vol)], 1 - pos_rate)
#         return self

#     def predict_proba(self, X):
#         if self.vol_now_col in X.columns:
#             vol = X[self.vol_now_col].values
#         else:
#             vol = X.iloc[:, 0].values
#         vol = np.where(np.isfinite(vol), vol, 0.0)
#         vmin, vmax = vol.min(), vol.max()
#         prob = (vol - vmin) / (vmax - vmin + 1e-8)
#         return np.column_stack([1 - prob, prob])

#     def predict(self, X):
#         return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

# class PersistenceRegBaseline:
#     """
#     Регрессор-baseline: предсказывает vol_fwd = vol_now (persistence).
#     Для log_vol_fwd — возвращает log(vol_now).
#     Для delta_vol   — возвращает 0 (нет изменения).
#     fit() просто запоминает среднее смещение на train (bias correction).
#     """
#     def __init__(self, vol_now_col: str = "vol_now_feat"):
#         self.vol_now_col = vol_now_col
#         self.bias_       = 0.0

#     def fit(self, X, y):
#         if self.vol_now_col in X.columns:
#             vol = X[self.vol_now_col].values
#         else:
#             vol = X.iloc[:, 0].values
#         pred = vol[np.isfinite(vol) & np.isfinite(y.values)]
#         ytrue = y.values[np.isfinite(vol) & np.isfinite(y.values)]
#         self.bias_ = float(np.mean(ytrue - pred)) if len(pred) else 0.0
#         return self

#     def predict(self, X):
#         if self.vol_now_col in X.columns:
#             vol = X[self.vol_now_col].values
#         else:
#             vol = X.iloc[:, 0].values
#         return np.where(np.isfinite(vol), vol + self.bias_, np.nanmean(vol))

# class AR1RegBaseline:
#     """
#     AR(1) baseline на воле — регрессия вида:
#         y = a * vol_now + b
#     Обучается только на одной фиче (vol_now_feat), без остального X.
#     Для delta_vol это означает: delta = a * vol_now + b
#     Честный baseline: использует обучение, но только из одной переменной.
#     Позволяет отделить "сигнал от уровня волы" от "сигнала от cross-asset фич".
#     """
#     def __init__(self, vol_now_col: str = "vol_now_feat"):
#         self.vol_now_col = vol_now_col
#         self.a_ = 0.0
#         self.b_ = 0.0

#     def _get_vol(self, X):
#         if self.vol_now_col in X.columns:
#             return X[self.vol_now_col].values.astype(float)
#         return X.iloc[:, 0].values.astype(float)

#     def fit(self, X, y):
#         vol   = self._get_vol(X)
#         ytrue = y.values.astype(float)
#         mask  = np.isfinite(vol) & np.isfinite(ytrue)
#         v, t  = vol[mask], ytrue[mask]
#         if len(v) < 2:
#             self.a_, self.b_ = 0.0, float(np.nanmean(ytrue))
#             return self
#         self.a_ = float(np.cov(v, t)[0, 1] / (np.var(v) + 1e-12))
#         self.b_ = float(np.mean(t) - self.a_ * np.mean(v))
#         return self

#     def predict(self, X):
#         vol = self._get_vol(X)
#         return np.where(np.isfinite(vol), self.a_ * vol + self.b_, self.b_)

# def get_cls_models() -> dict:
#     models = {
#         "Persistence": PersistenceClsBaseline(vol_now_col="vol_now_feat"),
#         "LogReg": Pipeline([
#             ("sc", StandardScaler()),
#             ("m",  LogisticRegression(C=0.1, max_iter=500, class_weight="balanced")),
#         ]),
#     }
#     if USE_HEAVY_MODELS:
#         models["LGBM"] = lgb.LGBMClassifier(
#             n_estimators=50, max_depth=3,
#             learning_rate=0.1, subsample=0.8,
#             colsample_bytree=0.5, min_child_samples=30,
#             class_weight="balanced", n_jobs=-1,
#             verbose=-1, random_state=42,
#         )
#     return models

# def get_reg_models() -> dict:
#     models = {
#         "Persistence": PersistenceRegBaseline(vol_now_col="vol_now_feat"),
#         "AR1":         AR1RegBaseline(vol_now_col="vol_now_feat"),
#         "Ridge": Pipeline([
#             ("sc", StandardScaler()),
#             ("m",  Ridge(alpha=1.0)),
#         ]),
#     }
#     if USE_HEAVY_MODELS:
#         models["LGBM"] = lgb.LGBMRegressor(
#             n_estimators=50, max_depth=3,
#             learning_rate=0.1, subsample=0.8,
#             colsample_bytree=0.5, min_child_samples=30,
#             n_jobs=-1, verbose=-1, random_state=42,
#         )
#     return models

# def _summarize(df: pd.DataFrame, metric_cols: list) -> pd.DataFrame:
#     if df.empty:
#         return pd.DataFrame()
#     return (
#         df.groupby("model")[metric_cols]
#         .agg(["mean", "std"])
#         .round(4)
#     )

# def print_block(title: str):
#     print(f"\n{'═'*65}")
#     print(f"  {title}")
#     print(f"{'═'*65}")

# def run_experiment(
#     price_df:   pd.DataFrame,
#     volume_ask: pd.DataFrame,
#     volume_bid: pd.DataFrame,
#     high_df:    pd.DataFrame,
#     low_df:     pd.DataFrame,
# ) -> dict:
#     """
#     Запускается так:
#         results = run_experiment(price_df, volume_ask, volume_bid, high_df, low_df)

#     price_df = (bt2.data['STRATEX-BINANCE']['best_ask']
#               + bt2.data['STRATEX-BINANCE']['best_bid']) / 2
#     volume_ask = bt2.data['STRATEX-BINANCE']['buy_volume']
#     volume_bid = bt2.data['STRATEX-BINANCE']['sell_volume']
#     high_df    = bt2.data['STRATEX-BINANCE-F']['high']
#     low_df     = bt2.data['STRATEX-BINANCE-F']['low']
#     """

#     print_block("CRYPTO VOL STRESS EXPERIMENT  —  30-min bars")
#     print(f"  Targets  : {TARGET_ASSETS}")
#     print(f"  Universe : {UNIVERSE}")
#     print(f"  Horizons : {HORIZONS} bars  =  "
#           f"{[f'{h//2}h' if h < 48 else f'{h//48}d' for h in HORIZONS]}")
#     print(f"  Date     : {START_DATE} → end")
#     print(f"  WF split : train={TRAIN_BARS} / test={TEST_BARS} / "
#           f"step={STEP_BARS} / purge={PURGE_BARS} bars")

#     print_block("1. Preparing asset data")
#     assets = prepare_universe(price_df, volume_ask, volume_bid, high_df, low_df)
#     common_idx, active_assets = build_common_index(assets)

#     for t in TARGET_ASSETS:
#         if t not in active_assets:
#             raise ValueError(
#                 f"Target asset '{t}' ({TICKER_MAP[t]}) not found or has insufficient coverage. "
#                 f"Active assets: {active_assets}"
#             )

#     def _wide(field: str) -> pd.DataFrame:
#         return pd.DataFrame(
#             {a: assets[a][field].reindex(common_idx) for a in active_assets}
#         )

#     close_w  = _wide("close")
#     high_w   = _wide("high")
#     low_w    = _wide("low")
#     vask_w   = _wide("vol_ask")
#     vbid_w   = _wide("vol_bid")

#     ret = log_returns(close_w)

#     print_block("2. Building features")
#     X = build_features(ret, high_w, low_w, vask_w, vbid_w)

#     all_results = {}

#     for target in TARGET_ASSETS:
#         print_block(f"3. Targets  →  {target}")
#         tgt = build_targets(ret, target)
#         tgt = tgt.loc[X.index]

#         for h in HORIZONS:
#             horizon_label = f"{h//2}h" if h < 48 else f"{h//48}d"
#             print(f"\n  ── horizon h={h} ({horizon_label}) ──")

#             y_cls = tgt[f"stress_{h}"].dropna()
#             Xc    = X.loc[y_cls.index]
#             pos   = y_cls.mean()
#             print(f"    [CLS] stress event  pos_rate={pos:.2%}")
#             df_cls = run_wf_classification(Xc, y_cls, get_cls_models())
#             s_cls  = _summarize(df_cls, ["roc_auc", "pr_auc", "f1", "bal_acc"])
#             if not s_cls.empty:
#                 print(s_cls.to_string())
#             key = f"{target}_CLS_stress_h{h}"
#             all_results[key] = s_cls

#             y_vol = tgt[f"log_vol_fwd_{h}"].dropna()
#             Xr    = X.loc[y_vol.index]
#             print(f"    [REG] log vol level")
#             df_vol = run_wf_regression(Xr, y_vol, get_reg_models())
#             s_vol  = _summarize(df_vol, ["mae", "r2", "corr"])
#             if not s_vol.empty:
#                 print(s_vol.to_string())
#             all_results[f"{target}_REG_logvol_h{h}"] = s_vol

#             y_dv = tgt[f"delta_vol_{h}"].dropna()
#             Xd   = X.loc[y_dv.index]
#             print(f"    [REG] delta vol")
#             df_dv = run_wf_regression(Xd, y_dv, get_reg_models())
#             s_dv  = _summarize(df_dv, ["mae", "r2", "corr"])
#             if not s_dv.empty:
#                 print(s_dv.to_string())
#             all_results[f"{target}_REG_delta_h{h}"] = s_dv

#     print_block("FINAL SUMMARY  (mean across walk-forward folds)")
#     rows = []
#     for exp_key, df in all_results.items():
#         if df.empty:
#             continue
#         for model_name in df.index.get_level_values(0).unique():
#             row = {"experiment": exp_key, "model": model_name}
#             for col in df.columns.get_level_values(0).unique():
#                 row[f"{col}_mean"] = df.loc[model_name, (col, "mean")]
#                 row[f"{col}_std"]  = df.loc[model_name, (col, "std")]
#             rows.append(row)

#     summary = pd.DataFrame(rows).set_index(["experiment", "model"])
#     print(summary.to_string())

#     return all_results, summary

# price_df   = (bt2.data['STRATEX-BINANCE']['best_ask']
#             + bt2.data['STRATEX-BINANCE']['best_bid']) / 2
# volume_ask = bt2.data['STRATEX-BINANCE']['buy_volume']
# volume_bid = bt2.data['STRATEX-BINANCE']['sell_volume']
# high_df    = bt2.data['STRATEX-BINANCE-F']['high']
# low_df     = bt2.data['STRATEX-BINANCE-F']['low']

# results, summary = run_experiment(
#     price_df, volume_ask, volume_bid, high_df, low_df
# )


═════════════════════════════════════════════════════════════════
  CRYPTO VOL STRESS EXPERIMENT  —  30-min bars
═════════════════════════════════════════════════════════════════
  Targets  : ['BTC', 'ETH']
  Universe : ['BTC', 'ETH', 'SOL', 'BNB', 'XRP', 'DOGE', 'ADA', 'AVAX', 'TRX']
  Horizons : [8, 48, 96] bars  =  ['4h', '1d', '2d']
  Date     : 2023-01-01 → end
  WF split : train=2880 / test=480 / step=240 / purge=96 bars

═════════════════════════════════════════════════════════════════
  1. Preparing asset data
═════════════════════════════════════════════════════════════════
  BTC-USDT:  46743 bars  (2023-01-01 → 2025-09-01)
  ETH-USDT:  46743 bars  (2023-01-01 → 2025-09-01)
  SOL-USDT:  46743 bars  (2023-01-01 → 2025-09-01)
  BNB-USDT:  46743 bars  (2023-01-01 → 2025-09-01)
  XRP-USDT:  46743 bars  (2023-01-01 → 2025-09-01)
  DOGE-USDT:  46743 bars  (2023-01-01 → 2025-09-01)
  ADA-USDT:  46743 bars  (2023-01-01 → 2025-09-01)
  AVAX-USDT:  46743 bars  (2023-01-01 → 2025-09-01)

pairwise MI: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 36/36 [00:04<00:00,  8.46pair/s]


  [feat] triplet MI...
    triplet MI: pre-computing 36 pairs for 84 triplets — vectorized...


triplet MI (pairs): 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 36/36 [00:04<00:00,  8.45pair/s]


  [feat] total: 283 features  ×  46743 bars

═════════════════════════════════════════════════════════════════
  3. Targets  →  BTC
═════════════════════════════════════════════════════════════════
    BTC h=  8: stress 10978/46743 (23.5%)
    BTC h= 48: stress 13511/46743 (28.9%)
    BTC h= 96: stress 13929/46743 (29.8%)

  ── horizon h=8 (4h) ──
    [CLS] stress event  pos_rate=23.49%


...

            roc_auc          pr_auc              f1         bal_acc        
               mean     std    mean     std    mean     std    mean     std
model                                                                      
LogReg       0.5370  0.2270  0.4598  0.3150  0.3424  0.2861  0.5327  0.1607
Persistence  0.4886  0.2086  0.4318  0.3154  0.2953  0.2289  0.5068  0.1526
    [REG] log vol level


WF folds: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 174/174 [00:02<00:00, 61.85fold/s]


                mae              r2             corr        
               mean     std    mean      std    mean     std
model                                                       
AR1          0.2894  0.0964 -0.9226   1.5317 -0.0034  0.2593
Persistence  0.2883  0.0835 -0.8532   0.8487  0.0002  0.2593
Ridge        0.4842  0.2252 -6.2112  12.7329  0.0515  0.3257
    [REG] delta vol


WF folds: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 174/174 [00:02<00:00, 63.09fold/s]

                mae              r2            corr        
               mean     std    mean     std    mean     std
model                                                      
AR1          0.1619  0.0726  0.1621  0.7573  0.7480  0.1334
Persistence  0.3280  0.1408 -2.0972  1.0391 -0.7480  0.1334
Ridge        0.2857  0.1695 -2.3670  6.2637  0.5644  0.2261

═════════════════════════════════════════════════════════════════
  FINAL SUMMARY  (mean across walk-forward folds)
═════════════════════════════════════════════════════════════════
                                roc_auc_mean  roc_auc_std  pr_auc_mean  pr_auc_std  f1_mean  f1_std  bal_acc_mean  bal_acc_std  mae_mean  mae_std  r2_mean   r2_std  corr_mean  corr_std
experiment         model                                                                                                                                                                
BTC_CLS_stress_h8  LogReg             0.5296       0.1069       0.2869      0.1840   0.

In [14]:
# results

{'BTC_CLS_stress_h8':             roc_auc          pr_auc             f1         bal_acc        
                mean     std    mean    std    mean     std    mean     std
 model                                                                     
 LogReg       0.5296  0.1069  0.2869  0.184  0.2747  0.1704  0.5218  0.0766
 Persistence  0.6404  0.1026  0.3499  0.208  0.3513  0.1640  0.5985  0.0811,
 'BTC_REG_logvol_h8':                 mae              r2            corr        
                mean     std    mean     std    mean     std
 model                                                      
 AR1          0.4435  0.0569  0.0225  0.1362  0.3184  0.1557
 Persistence  0.4436  0.0543  0.0223  0.1292  0.3184  0.1557
 Ridge        0.8960  0.3794 -3.8737  6.5396  0.0641  0.2088,
 'BTC_REG_delta_h8':                 mae              r2            corr        
                mean     std    mean     std    mean     std
 model                                                      
 AR1   

In [ ]:
# import warnings
# warnings.filterwarnings("ignore")

# import numpy as np
# import pandas as pd
# from itertools import combinations
# from tqdm.auto import tqdm

# from sklearn.metrics import (
#     roc_auc_score, average_precision_score,
#     f1_score, balanced_accuracy_score,
#     mean_absolute_error, r2_score,
# )
# from sklearn.linear_model import LogisticRegression, Ridge
# from sklearn.preprocessing import StandardScaler
# from sklearn.pipeline import Pipeline
# import lightgbm as lgb

# TICKER_MAP = {
#     "BTC":  "BTC-USDT",
#     "ETH":  "ETH-USDT",
#     "SOL":  "SOL-USDT",
#     "BNB":  "BNB-USDT",
#     "XRP":  "XRP-USDT",
#     "DOGE": "DOGE-USDT",
#     "ADA":  "ADA-USDT",
#     "AVAX": "AVAX-USDT",
#     "TRX":  "TRX-USDT",
# }

# TARGET_ASSETS = ["BTC", "ETH"]          
# UNIVERSE      = list(TICKER_MAP.keys()) 

# START_DATE = "2023-01-01"              

# BARS_PER_DAY = 48
# ANN_FACTOR   = np.sqrt(365 * BARS_PER_DAY)   

# VOL_WINDOW = 48     

# STRESS_MULT = 1.25

# HORIZONS = [8, 48, 96]    

# MI_WINDOW      = 192   
# MI_BINS        = 10
# MI_SUBSAMPLE_K = 4     

# TOP_K_FEATS = [20, 50, 100]   

# TRAIN_BARS = 2880   
# TEST_BARS  =  480   
# STEP_BARS  =  240   
# PURGE_BARS =   96   

# USE_HEAVY_MODELS = False

# STRESS_MEDIAN_WINDOW = 365 * BARS_PER_DAY   

# def _to_datetime_index(df: pd.DataFrame) -> pd.DataFrame:
#     if not isinstance(df.index, pd.DatetimeIndex):
#         df = df.copy()
#         df.index = pd.to_datetime(df.index)
#     return df

# def prepare_universe(
#     price_df:   pd.DataFrame,
#     volume_ask: pd.DataFrame,
#     volume_bid: pd.DataFrame,
#     high_df:    pd.DataFrame,
#     low_df:     pd.DataFrame,
# ) -> dict:
#     """
#     Нарезает широкие датафреймы по нужным тикерам и дате.
#     Возвращает dict:  asset_key → {"close", "high", "low", "vol_ask", "vol_bid"}
#     каждый — pd.Series с DatetimeIndex.
#     """
#     for name, df in [("price_df", price_df), ("volume_ask", volume_ask),
#                      ("volume_bid", volume_bid), ("high_df", high_df), ("low_df", low_df)]:
#         if not isinstance(df.index, pd.DatetimeIndex):
#             raise TypeError(f"{name}: index must be DatetimeIndex, got {type(df.index)}")

#     cut = pd.Timestamp(START_DATE)
#     assets = {}
#     missing_tickers = []

#     for asset, ticker in TICKER_MAP.items():
#         if ticker not in price_df.columns:
#             missing_tickers.append(ticker)
#             continue

#         close   = price_df[ticker].loc[cut:].copy()
#         vol_ask = volume_ask[ticker].loc[cut:].copy() if ticker in volume_ask.columns else pd.Series(np.nan, index=close.index)
#         vol_bid = volume_bid[ticker].loc[cut:].copy() if ticker in volume_bid.columns else pd.Series(np.nan, index=close.index)
#         high    = high_df[ticker].loc[cut:].copy()    if ticker in high_df.columns    else pd.Series(np.nan, index=close.index)
#         low     = low_df[ticker].loc[cut:].copy()     if ticker in low_df.columns     else pd.Series(np.nan, index=close.index)

#         valid = close.notna()
#         assets[asset] = {
#             "close":   close[valid],
#             "high":    high[valid],
#             "low":     low[valid],
#             "vol_ask": vol_ask[valid],
#             "vol_bid": vol_bid[valid],
#         }
#         print(f"  {ticker}: {valid.sum():6d} bars  "
#               f"({close[valid].index[0].date()} → {close[valid].index[-1].date()})")

#     if missing_tickers:
#         print(f"\n  WARNING — tickers not found in price_df: {missing_tickers}")
#         print("  Available columns sample:", list(price_df.columns[:10]))

#     return assets

# def build_common_index(assets: dict, min_coverage: float = 0.7) -> pd.DatetimeIndex:
#     """
#     Строит общий индекс баров. Тикер включается если покрывает >= min_coverage от общего диапазона.
#     """
#     all_idx = pd.DatetimeIndex(
#         sorted(set().union(*[set(v["close"].index) for v in assets.values()]))
#     )
#     print(f"\n  Union index: {len(all_idx)} bars")

#     kept = []
#     for asset, data in assets.items():
#         cov = data["close"].reindex(all_idx).notna().mean()
#         if cov >= min_coverage:
#             kept.append(asset)
#             print(f"  {asset}: coverage {cov:.1%}  ✓")
#         else:
#             print(f"  {asset}: coverage {cov:.1%}  ✗ skipped")

#     idx = all_idx
#     for asset in kept:
#         idx = idx.intersection(assets[asset]["close"].index)

#     print(f"  Intersection index: {len(idx)} bars  "
#           f"({idx[0].date()} → {idx[-1].date()})")
#     return idx, kept

# def log_returns(close_wide: pd.DataFrame) -> pd.DataFrame:
#     return np.log(close_wide / close_wide.shift(1))

# def build_targets(
#     ret:          pd.DataFrame,
#     target_asset: str,
# ) -> pd.DataFrame:
#     """
#     Для одного target_asset строит таргеты на все горизонты.
#     Все вычисления — строго на ret (нет обращения к будущему в фичах).
#     """
#     r       = ret[target_asset]
#     vol_now = r.rolling(VOL_WINDOW).std() * ANN_FACTOR

#     tgt = pd.DataFrame(index=ret.index)
#     tgt["vol_now"] = vol_now

#     for h in HORIZONS:
#         vol_fwd = r.rolling(h).std().shift(-h) * ANN_FACTOR

#         vol_med = vol_now.rolling(STRESS_MEDIAN_WINDOW, min_periods=BARS_PER_DAY * 30).median()

#         tgt[f"vol_fwd_{h}"]     = vol_fwd
#         tgt[f"log_vol_fwd_{h}"] = np.log(vol_fwd.clip(lower=1e-8))
#         tgt[f"stress_{h}"]      = (vol_fwd > vol_med * STRESS_MULT).astype(float)
#         tgt[f"delta_vol_{h}"]   = vol_fwd - vol_now

#         n_stress = tgt[f"stress_{h}"].sum()
#         n_total  = tgt[f"stress_{h}"].notna().sum()
#         print(f"    {target_asset} h={h:3d}: stress {int(n_stress):5d}/{n_total} "
#               f"({n_stress/n_total*100:.1f}%)")

#     return tgt

# def _rank_normalize(arr: np.ndarray, bins: int) -> np.ndarray:
#     """
#     Превращает 1D массив в дискретные бины [0, bins) через rank-нормализацию.
#     Устойчиво к выбросам (в отличие от равномерных бинов по диапазону значений).
#     Векторизовано: работает на всём массиве сразу.
#     """
#     n = len(arr)
#     ranks = np.argsort(np.argsort(arr))          # ранги 0..n-1
#     return (ranks * bins // n).clip(0, bins - 1) # → [0, bins)

# def _batch_mi_single_pair(
#     col_x:  np.ndarray,   # shape (n_bars,) — полный ряд одного актива
#     col_y:  np.ndarray,   # shape (n_bars,) — полный ряд другого актива
#     window: int,
#     k:      int,
#     bins:   int,
# ) -> np.ndarray:
#     """
#     Считает rolling MI для одной пары (col_x, col_y) полностью без Python-цикла.

#     Подход:
#       1. Нарезаем все окна сразу через stride_tricks → shape (n_windows, window)
#       2. Rank-нормализуем каждое окно → дискретные бины
#       3. Кодируем совместный бин как  x_bin * bins + y_bin  → 1D индекс
#       4. Считаем joint histogram через np.bincount батчем → (n_windows, bins²)
#       5. Вычисляем MI через матричные операции

#     Нет ни одного Python-цикла — всё в C/numpy.
#     """
#     n = len(col_x)

#     starts = np.arange(0, n - window + 1, k)         
#     ends   = starts + window                           
#     n_win  = len(starts)

#     wx = np.stack([col_x[s:e] for s, e in zip(starts, ends)])  
#     wy = np.stack([col_y[s:e] for s, e in zip(starts, ends)])  
#     def rank_bins_2d(mat):
#         rk = np.argsort(np.argsort(mat, axis=1), axis=1)
#         return (rk * bins // window).clip(0, bins - 1).astype(np.int32)

#     bx = rank_bins_2d(wx)   
#     by = rank_bins_2d(wy)   

#     joint = bx * bins + by   

#     bins2 = bins * bins
#     offsets = (np.arange(n_win) * bins2).reshape(-1, 1)  
#     flat    = (joint + offsets).ravel()                   
#     counts  = np.bincount(flat, minlength=n_win * bins2).reshape(n_win, bins2)

#     counts  = counts.astype(np.float64) + 1e-10            
#     p_xy    = counts / counts.sum(axis=1, keepdims=True)   

#     p_xy_2d = p_xy.reshape(n_win, bins, bins)             
#     p_x     = p_xy_2d.sum(axis=2, keepdims=True)          
#     p_y     = p_xy_2d.sum(axis=1, keepdims=True)           

#     mi_vals = (p_xy_2d * np.log(p_xy_2d / (p_x * p_y))).sum(axis=(1, 2))
#     mi_vals = np.maximum(mi_vals, 0.0)                     

#     result = np.full(n, np.nan)
#     result[ends - 1] = mi_vals   
#     mask = np.isfinite(result)
#     idx  = np.where(mask, np.arange(n), 0)
#     np.maximum.accumulate(idx, out=idx)
#     result = result[idx]
#     return result

# def rolling_vol_features(
#     ret:       pd.DataFrame,
#     high_wide: pd.DataFrame,
#     low_wide:  pd.DataFrame,
#     vask_wide: pd.DataFrame,
#     vbid_wide: pd.DataFrame,
#     windows:   list = [4, 8, 16, 48, 96],
# ) -> pd.DataFrame:
#     feats = {}

#     for asset in ret.columns:
#         r = ret[asset]

#         for w in windows:
#             feats[f"{asset}_rvol_{w}"] = r.rolling(w).std() * ANN_FACTOR

#         rv = r.rolling(VOL_WINDOW).std() * ANN_FACTOR

#         feats[f"{asset}_vov"]         = rv.rolling(VOL_WINDOW).std()

#         rv_mean = rv.rolling(VOL_WINDOW * 4).mean()
#         rv_std  = rv.rolling(VOL_WINDOW * 4).std() + 1e-8
#         feats[f"{asset}_vol_zscore"]  = (rv - rv_mean) / rv_std

#         feats[f"{asset}_vol_pctrank"] = rv.rolling(VOL_WINDOW * 30).rank(pct=True)

#         h = high_wide[asset] if asset in high_wide.columns else pd.Series(np.nan, index=ret.index)
#         l = low_wide[asset]  if asset in low_wide.columns  else pd.Series(np.nan, index=ret.index)
#         hl2 = np.log((h / l).clip(lower=1e-8)) ** 2
#         feats[f"{asset}_pk_vol"] = np.sqrt(
#             hl2.rolling(VOL_WINDOW).mean() / (4 * np.log(2))
#         ) * ANN_FACTOR

#         va = vask_wide[asset] if asset in vask_wide.columns else pd.Series(0.0, index=ret.index)
#         vb = vbid_wide[asset] if asset in vbid_wide.columns else pd.Series(0.0, index=ret.index)
#         total = (va + vb).replace(0, np.nan)
#         raw_imb = (va - vb) / total
#         feats[f"{asset}_vol_imbalance"]     = raw_imb.rolling(8).mean()   
#         feats[f"{asset}_vol_imbalance_std"] = raw_imb.rolling(48).std()

#         for w in [4, 16, 48]:
#             feats[f"{asset}_mom_{w}"] = r.rolling(w).mean()

#     return pd.DataFrame(feats, index=ret.index)

# def cross_covariance_features(ret: pd.DataFrame, window: int = VOL_WINDOW) -> pd.DataFrame:
#     pairs = list(combinations(ret.columns, 2))
#     feats = {}
#     for a, b in pairs:
#         cov = ret[a].rolling(window).cov(ret[b])
#         std_a = ret[a].rolling(window).std() + 1e-8
#         std_b = ret[b].rolling(window).std() + 1e-8
#         feats[f"corr_{a}_{b}"] = cov / (std_a * std_b)
#     return pd.DataFrame(feats, index=ret.index)

# def pairwise_mi_features(
#     ret:    pd.DataFrame,
#     window: int = MI_WINDOW,
#     bins:   int = MI_BINS,
#     k:      int = MI_SUBSAMPLE_K,
# ) -> pd.DataFrame:
#     """
#     Rolling pairwise MI — полностью векторизовано, без Python-цикла по барам.
#     Цикл только по парам активов (36 итераций), внутри каждой — чистый numpy.
#     """
#     pairs   = list(combinations(ret.columns, 2))
#     ret_np  = ret.to_numpy(dtype=float)
#     col_idx = {c: i for i, c in enumerate(ret.columns)}
#     n       = len(ret)

#     n_windows = len(range(0, n - window + 1, k))
#     print(f"    pairwise MI: {len(pairs)} pairs, {n_windows} windows "
#           f"(k={k}) — fully vectorized...")

#     feat_dict = {}
#     for a, b in tqdm(pairs, desc="pairwise MI", unit="pair"):
#         feat_dict[f"mi_{a}_{b}"] = _batch_mi_single_pair(
#             ret_np[:, col_idx[a]], ret_np[:, col_idx[b]], window, k, bins
#         )

#     return pd.DataFrame(feat_dict, index=ret.index)

# def triplet_mi_features(
#     ret:    pd.DataFrame,
#     window: int = MI_WINDOW,
#     bins:   int = MI_BINS,
#     k:      int = MI_SUBSAMPLE_K,
# ) -> pd.DataFrame:
#     """
#     Rolling triplet MI = MI(a,b) + MI(b,c) + MI(a,c) — векторизовано.
#     Каждая пара считается один раз через _batch_mi_single_pair, потом суммируется.
#     """
#     triplets = list(combinations(ret.columns, 3))
#     ret_np   = ret.to_numpy(dtype=float)
#     col_idx  = {c: i for i, c in enumerate(ret.columns)}
#     n        = len(ret)

#     all_pairs = list(combinations(ret.columns, 2))
#     n_windows = len(range(0, n - window + 1, k))
#     print(f"    triplet MI: pre-computing {len(all_pairs)} pairs "
#           f"for {len(triplets)} triplets — vectorized...")

#     pair_mi = {}
#     for a, b in tqdm(all_pairs, desc="triplet MI (pairs)", unit="pair"):
#         key = (a, b)
#         pair_mi[key] = _batch_mi_single_pair(
#             ret_np[:, col_idx[a]], ret_np[:, col_idx[b]], window, k, bins
#         )

#     feat_dict = {}
#     for a, b, c in triplets:
#         feat_dict[f"tri_{a}_{b}_{c}"] = (
#             pair_mi[(a, b)] + pair_mi[(b, c)] + pair_mi[(a, c)]
#         )

#     return pd.DataFrame(feat_dict, index=ret.index)

# def build_features(ret: pd.DataFrame, high_w: pd.DataFrame,
#                    low_w: pd.DataFrame, vask_w: pd.DataFrame,
#                    vbid_w: pd.DataFrame,
#                    target_assets: list = None) -> pd.DataFrame:

#     print("  [feat] rolling vol / Parkinson / volume imbalance...")
#     vol_f = rolling_vol_features(ret, high_w, low_w, vask_w, vbid_w)

#     print("  [feat] rolling cross-correlation...")
#     corr_f = cross_covariance_features(ret)

#     print("  [feat] pairwise MI...")
#     mi_f = pairwise_mi_features(ret)

#     print("  [feat] triplet MI...")
#     tri_f = triplet_mi_features(ret)

#     X = pd.concat([vol_f, corr_f, mi_f, tri_f], axis=1)

#     assets = target_assets if target_assets else TARGET_ASSETS
#     vol_now_cols = [f"{a}_rvol_{VOL_WINDOW}" for a in assets if f"{a}_rvol_{VOL_WINDOW}" in X.columns]
#     X["vol_now_feat"] = X[vol_now_cols].mean(axis=1)

#     print(f"  [feat] total: {X.shape[1]} features  ×  {X.shape[0]} bars")
#     return X

# def select_top_k_corr(
#     Xtr: pd.DataFrame,
#     Xte: pd.DataFrame,
#     ytr: pd.Series,
#     k:   int,
#     always_include: str = "vol_now_feat",
# ) -> tuple:
#     """
#     Отбирает топ-k фич по |corr(feature, target)| на train-части.
#     always_include — колонка которая всегда остаётся (для Persistence/AR1).
#     Возвращает (Xtr_sel, Xte_sel) с одинаковым набором колонок.
#     Отбор строго на train → нет lookahead.
#     """
#     feat_cols = [c for c in Xtr.columns if c != always_include]
#     corrs = Xtr[feat_cols].corrwith(ytr).abs()
#     corrs = corrs.replace([np.inf, -np.inf], np.nan).dropna()
#     top_cols = corrs.nlargest(k).index.tolist()

#     if always_include in Xtr.columns and always_include not in top_cols:
#         top_cols = [always_include] + top_cols

#     return Xtr[top_cols], Xte[top_cols]

# def _wf_splits(n: int, train: int, test: int, step: int, purge: int):
#     """
#     Генератор (train_idx, test_idx) с purge-зазором.
#     train_end → +purge → test_start
#     """
#     start = train
#     while start + purge + test <= n:
#         train_idx = list(range(start - train, start))
#         test_start = start + purge
#         test_idx  = list(range(test_start, test_start + test))
#         yield train_idx, test_idx
#         start += step

# def run_wf_classification(X, y, models,
#                            train=TRAIN_BARS, test=TEST_BARS,
#                            step=STEP_BARS, purge=PURGE_BARS):
#     mask   = X.notna().all(axis=1) & y.notna()
#     Xc, yc = X[mask], y[mask]
#     splits = list(_wf_splits(len(Xc), train, test, step, purge))

#     if not splits:
#         print("    ! Not enough data for walk-forward, skipping")
#         return pd.DataFrame()

#     baseline_names = {n for n, m in models.items()
#                       if isinstance(m, (PersistenceClsBaseline,))}

#     records = []
#     for fold, (tr, te) in enumerate(tqdm(splits, desc="WF folds", unit="fold")):
#         Xtr, Xte = Xc.iloc[tr], Xc.iloc[te]
#         ytr, yte = yc.iloc[tr], yc.iloc[te]

#         if len(ytr.unique()) < 2:
#             continue

#         for name, mdl in models.items():
#             if name in baseline_names:
#                 mdl.fit(Xtr, ytr)
#                 prob = mdl.predict_proba(Xte)[:, 1]
#                 pred = mdl.predict(Xte)
#                 _append_cls(records, fold, name, yte, prob, pred)

#         for k in TOP_K_FEATS:
#             Xtr_k, Xte_k = select_top_k_corr(Xtr, Xte, ytr, k)
#             mdl_k = Pipeline([
#                 ("sc", StandardScaler()),
#                 ("m",  LogisticRegression(C=0.1, max_iter=500, class_weight="balanced")),
#             ])
#             mdl_k.fit(Xtr_k, ytr)
#             prob = mdl_k.predict_proba(Xte_k)[:, 1]
#             pred = mdl_k.predict(Xte_k)
#             _append_cls(records, fold, f"LogReg_k{k}", yte, prob, pred)

#         if USE_HEAVY_MODELS:
#             for k in TOP_K_FEATS:
#                 Xtr_k, Xte_k = select_top_k_corr(Xtr, Xte, ytr, k)
#                 mdl_k = lgb.LGBMClassifier(
#                     n_estimators=50, max_depth=3, learning_rate=0.1,
#                     subsample=0.8, colsample_bytree=0.8, min_child_samples=30,
#                     class_weight="balanced", n_jobs=-1, verbose=-1, random_state=42,
#                 )
#                 mdl_k.fit(Xtr_k, ytr)
#                 prob = mdl_k.predict_proba(Xte_k)[:, 1]
#                 pred = mdl_k.predict(Xte_k)
#                 _append_cls(records, fold, f"LGBM_k{k}", yte, prob, pred)

#     return pd.DataFrame(records)

# def _append_cls(records, fold, name, yte, prob, pred):
#     try:
#         roc = roc_auc_score(yte, prob)
#         pr  = average_precision_score(yte, prob)
#     except Exception:
#         roc, pr = np.nan, np.nan
#     records.append({
#         "fold": fold, "model": name,
#         "roc_auc":  roc, "pr_auc": pr,
#         "f1":       f1_score(yte, pred, zero_division=0),
#         "bal_acc":  balanced_accuracy_score(yte, pred),
#         "pos_rate": float(yte.mean()), "n": len(yte),
#     })

# def run_wf_regression(X, y, models,
#                        train=TRAIN_BARS, test=TEST_BARS,
#                        step=STEP_BARS, purge=PURGE_BARS):
#     mask   = X.notna().all(axis=1) & y.notna()
#     Xc, yc = X[mask], y[mask]
#     splits = list(_wf_splits(len(Xc), train, test, step, purge))

#     if not splits:
#         print("    ! Not enough data for walk-forward, skipping")
#         return pd.DataFrame()

#     baseline_names = {n for n, m in models.items()
#                       if isinstance(m, (PersistenceRegBaseline, AR1RegBaseline))}

#     records = []
#     for fold, (tr, te) in enumerate(tqdm(splits, desc="WF folds", unit="fold")):
#         Xtr, Xte = Xc.iloc[tr], Xc.iloc[te]
#         ytr, yte = yc.iloc[tr], yc.iloc[te]

#         for name, mdl in models.items():
#             if name in baseline_names:
#                 mdl.fit(Xtr, ytr)
#                 pred = mdl.predict(Xte)
#                 _append_reg(records, fold, name, yte, pred)

#         for k in TOP_K_FEATS:
#             Xtr_k, Xte_k = select_top_k_corr(Xtr, Xte, ytr, k)
#             mdl_k = Pipeline([
#                 ("sc", StandardScaler()),
#                 ("m",  Ridge(alpha=1.0)),
#             ])
#             mdl_k.fit(Xtr_k, ytr)
#             pred = mdl_k.predict(Xte_k)
#             _append_reg(records, fold, f"Ridge_k{k}", yte, pred)

#         if USE_HEAVY_MODELS:
#             for k in TOP_K_FEATS:
#                 Xtr_k, Xte_k = select_top_k_corr(Xtr, Xte, ytr, k)
#                 mdl_k = lgb.LGBMRegressor(
#                     n_estimators=50, max_depth=3, learning_rate=0.1,
#                     subsample=0.8, colsample_bytree=0.8, min_child_samples=30,
#                     n_jobs=-1, verbose=-1, random_state=42,
#                 )
#                 mdl_k.fit(Xtr_k, ytr)
#                 pred = mdl_k.predict(Xte_k)
#                 _append_reg(records, fold, f"LGBM_k{k}", yte, pred)

#     return pd.DataFrame(records)

# def _append_reg(records, fold, name, yte, pred):
#     corr = float(pd.Series(pred).corr(pd.Series(yte.values)))
#     records.append({
#         "fold": fold, "model": name,
#         "mae":  mean_absolute_error(yte, pred),
#         "r2":   r2_score(yte, pred),
#         "corr": corr,
#         "n":    len(yte),
#     })

# class PersistenceClsBaseline:
#     """
#     Классификатор-baseline: предсказывает стресс если vol_now > порога.
#     Порог = rolling quantile(stress_mult) внутри train-сета.
#     predict_proba возвращает нормированную vol_now как "вероятность".
#     Не требует обучения — fit() только запоминает порог.
#     """
#     def __init__(self, vol_now_col: str = "vol_now_feat"):
#         self.vol_now_col = vol_now_col
#         self.threshold_  = None

#     def fit(self, X, y):
#         if self.vol_now_col in X.columns:
#             vol = X[self.vol_now_col].values
#         else:
#             vol = X.iloc[:, 0].values   # fallback: первая колонка
#         pos_rate = float(y.mean())
#         self.threshold_ = np.quantile(vol[np.isfinite(vol)], 1 - pos_rate)
#         return self

#     def predict_proba(self, X):
#         if self.vol_now_col in X.columns:
#             vol = X[self.vol_now_col].values
#         else:
#             vol = X.iloc[:, 0].values
#         vol = np.where(np.isfinite(vol), vol, 0.0)
#         vmin, vmax = vol.min(), vol.max()
#         prob = (vol - vmin) / (vmax - vmin + 1e-8)
#         return np.column_stack([1 - prob, prob])

#     def predict(self, X):
#         return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

# class PersistenceRegBaseline:
#     """
#     Регрессор-baseline: предсказывает vol_fwd = vol_now (persistence).
#     Для log_vol_fwd — возвращает log(vol_now).
#     Для delta_vol   — возвращает 0 (нет изменения).
#     fit() просто запоминает среднее смещение на train (bias correction).
#     """
#     def __init__(self, vol_now_col: str = "vol_now_feat"):
#         self.vol_now_col = vol_now_col
#         self.bias_       = 0.0

#     def fit(self, X, y):
#         if self.vol_now_col in X.columns:
#             vol = X[self.vol_now_col].values
#         else:
#             vol = X.iloc[:, 0].values
#         pred = vol[np.isfinite(vol) & np.isfinite(y.values)]
#         ytrue = y.values[np.isfinite(vol) & np.isfinite(y.values)]
#         self.bias_ = float(np.mean(ytrue - pred)) if len(pred) else 0.0
#         return self

#     def predict(self, X):
#         if self.vol_now_col in X.columns:
#             vol = X[self.vol_now_col].values
#         else:
#             vol = X.iloc[:, 0].values
#         return np.where(np.isfinite(vol), vol + self.bias_, np.nanmean(vol))

# class AR1RegBaseline:
#     """
#     AR(1) baseline на воле — регрессия вида:
#         y = a * vol_now + b
#     Обучается только на одной фиче (vol_now_feat), без остального X.
#     Для delta_vol это означает: delta = a * vol_now + b
#     Честный baseline: использует обучение, но только из одной переменной.
#     Позволяет отделить "сигнал от уровня волы" от "сигнала от cross-asset фич".
#     """
#     def __init__(self, vol_now_col: str = "vol_now_feat"):
#         self.vol_now_col = vol_now_col
#         self.a_ = 0.0
#         self.b_ = 0.0

#     def _get_vol(self, X):
#         if self.vol_now_col in X.columns:
#             return X[self.vol_now_col].values.astype(float)
#         return X.iloc[:, 0].values.astype(float)

#     def fit(self, X, y):
#         vol   = self._get_vol(X)
#         ytrue = y.values.astype(float)
#         mask  = np.isfinite(vol) & np.isfinite(ytrue)
#         v, t  = vol[mask], ytrue[mask]
#         if len(v) < 2:
#             self.a_, self.b_ = 0.0, float(np.nanmean(ytrue))
#             return self
#         self.a_ = float(np.cov(v, t)[0, 1] / (np.var(v) + 1e-12))
#         self.b_ = float(np.mean(t) - self.a_ * np.mean(v))
#         return self

#     def predict(self, X):
#         vol = self._get_vol(X)
#         return np.where(np.isfinite(vol), self.a_ * vol + self.b_, self.b_)

# def get_cls_models() -> dict:
#     models = {
#         "Persistence": PersistenceClsBaseline(vol_now_col="vol_now_feat"),
#         "LogReg": Pipeline([
#             ("sc", StandardScaler()),
#             ("m",  LogisticRegression(C=0.1, max_iter=500, class_weight="balanced")),
#         ]),
#     }
#     if USE_HEAVY_MODELS:
#         models["LGBM"] = lgb.LGBMClassifier(
#             n_estimators=50, max_depth=3,
#             learning_rate=0.1, subsample=0.8,
#             colsample_bytree=0.5, min_child_samples=30,
#             class_weight="balanced", n_jobs=-1,
#             verbose=-1, random_state=42,
#         )
#     return models

# def get_reg_models() -> dict:
#     models = {
#         "Persistence": PersistenceRegBaseline(vol_now_col="vol_now_feat"),
#         "AR1":         AR1RegBaseline(vol_now_col="vol_now_feat"),
#         "Ridge": Pipeline([
#             ("sc", StandardScaler()),
#             ("m",  Ridge(alpha=1.0)),
#         ]),
#     }
#     if USE_HEAVY_MODELS:
#         models["LGBM"] = lgb.LGBMRegressor(
#             n_estimators=50, max_depth=3,
#             learning_rate=0.1, subsample=0.8,
#             colsample_bytree=0.5, min_child_samples=30,
#             n_jobs=-1, verbose=-1, random_state=42,
#         )
#     return models

# def _summarize(df: pd.DataFrame, metric_cols: list) -> pd.DataFrame:
#     if df.empty:
#         return pd.DataFrame()
#     return (
#         df.groupby("model")[metric_cols]
#         .agg(["mean", "std"])
#         .round(4)
#     )

# def print_block(title: str):
#     print(f"\n{'═'*65}")
#     print(f"  {title}")
#     print(f"{'═'*65}")

# def run_experiment(
#     price_df:   pd.DataFrame,
#     volume_ask: pd.DataFrame,
#     volume_bid: pd.DataFrame,
#     high_df:    pd.DataFrame,
#     low_df:     pd.DataFrame,
# ) -> dict:
#     """
#     Запускается так:
#         results = run_experiment(price_df, volume_ask, volume_bid, high_df, low_df)

#     price_df = (bt2.data['STRATEX-BINANCE']['best_ask']
#               + bt2.data['STRATEX-BINANCE']['best_bid']) / 2
#     volume_ask = bt2.data['STRATEX-BINANCE']['buy_volume']
#     volume_bid = bt2.data['STRATEX-BINANCE']['sell_volume']
#     high_df    = bt2.data['STRATEX-BINANCE-F']['high']
#     low_df     = bt2.data['STRATEX-BINANCE-F']['low']
#     """

#     print_block("CRYPTO VOL STRESS EXPERIMENT  —  30-min bars")
#     print(f"  Targets  : {TARGET_ASSETS}")
#     print(f"  Universe : {UNIVERSE}")
#     print(f"  Horizons : {HORIZONS} bars  =  "
#           f"{[f'{h//2}h' if h < 48 else f'{h//48}d' for h in HORIZONS]}")
#     print(f"  Date     : {START_DATE} → end")
#     print(f"  WF split : train={TRAIN_BARS} / test={TEST_BARS} / "
#           f"step={STEP_BARS} / purge={PURGE_BARS} bars")

#     print_block("1. Preparing asset data")
#     assets = prepare_universe(price_df, volume_ask, volume_bid, high_df, low_df)
#     common_idx, active_assets = build_common_index(assets)

#     for t in TARGET_ASSETS:
#         if t not in active_assets:
#             raise ValueError(
#                 f"Target asset '{t}' ({TICKER_MAP[t]}) not found or has insufficient coverage. "
#                 f"Active assets: {active_assets}"
#             )

#     def _wide(field: str) -> pd.DataFrame:
#         return pd.DataFrame(
#             {a: assets[a][field].reindex(common_idx) for a in active_assets}
#         )

#     close_w  = _wide("close")
#     high_w   = _wide("high")
#     low_w    = _wide("low")
#     vask_w   = _wide("vol_ask")
#     vbid_w   = _wide("vol_bid")

#     ret = log_returns(close_w)

#     print_block("2. Building features")
#     X = build_features(ret, high_w, low_w, vask_w, vbid_w)

#     all_results = {}

#     for target in TARGET_ASSETS:
#         print_block(f"3. Targets  →  {target}")
#         tgt = build_targets(ret, target)
#         tgt = tgt.loc[X.index]

#         for h in HORIZONS:
#             horizon_label = f"{h//2}h" if h < 48 else f"{h//48}d"
#             print(f"\n  ── horizon h={h} ({horizon_label}) ──")

#             y_cls = tgt[f"stress_{h}"].dropna()
#             Xc    = X.loc[y_cls.index]
#             pos   = y_cls.mean()
#             print(f"    [CLS] stress event  pos_rate={pos:.2%}")
#             df_cls = run_wf_classification(Xc, y_cls, get_cls_models())
#             s_cls  = _summarize(df_cls, ["roc_auc", "pr_auc", "f1", "bal_acc"])
#             if not s_cls.empty:
#                 print(s_cls.to_string())
#             key = f"{target}_CLS_stress_h{h}"
#             all_results[key] = s_cls

#             y_vol = tgt[f"log_vol_fwd_{h}"].dropna()
#             Xr    = X.loc[y_vol.index]
#             print(f"    [REG] log vol level")
#             df_vol = run_wf_regression(Xr, y_vol, get_reg_models())
#             s_vol  = _summarize(df_vol, ["mae", "r2", "corr"])
#             if not s_vol.empty:
#                 print(s_vol.to_string())
#             all_results[f"{target}_REG_logvol_h{h}"] = s_vol

#             y_dv = tgt[f"delta_vol_{h}"].dropna()
#             Xd   = X.loc[y_dv.index]
#             print(f"    [REG] delta vol")
#             df_dv = run_wf_regression(Xd, y_dv, get_reg_models())
#             s_dv  = _summarize(df_dv, ["mae", "r2", "corr"])
#             if not s_dv.empty:
#                 print(s_dv.to_string())
#             all_results[f"{target}_REG_delta_h{h}"] = s_dv

#     print_block("FINAL SUMMARY  (mean across walk-forward folds)")
#     rows = []
#     for exp_key, df in all_results.items():
#         if df.empty:
#             continue
#         for model_name in df.index.get_level_values(0).unique():
#             row = {"experiment": exp_key, "model": model_name}
#             for col in df.columns.get_level_values(0).unique():
#                 row[f"{col}_mean"] = df.loc[model_name, (col, "mean")]
#                 row[f"{col}_std"]  = df.loc[model_name, (col, "std")]
#             rows.append(row)

#     summary = pd.DataFrame(rows).set_index(["experiment", "model"])
#     print(summary.to_string())

#     return all_results, summary

# price_df   = (bt2.data['STRATEX-BINANCE']['best_ask']
#             + bt2.data['STRATEX-BINANCE']['best_bid']) / 2
# volume_ask = bt2.data['STRATEX-BINANCE']['buy_volume']
# volume_bid = bt2.data['STRATEX-BINANCE']['sell_volume']
# high_df    = bt2.data['STRATEX-BINANCE-F']['high']
# low_df     = bt2.data['STRATEX-BINANCE-F']['low']

# results, summary = run_experiment(
#     price_df, volume_ask, volume_bid, high_df, low_df
# )


═════════════════════════════════════════════════════════════════
  CRYPTO VOL STRESS EXPERIMENT  —  30-min bars
═════════════════════════════════════════════════════════════════
  Targets  : ['BTC', 'ETH']
  Universe : ['BTC', 'ETH', 'SOL', 'BNB', 'XRP', 'DOGE', 'ADA', 'AVAX', 'TRX']
  Horizons : [8, 48, 96] bars  =  ['4h', '1d', '2d']
  Date     : 2023-01-01 → end
  WF split : train=2880 / test=480 / step=240 / purge=96 bars

═════════════════════════════════════════════════════════════════
  1. Preparing asset data
═════════════════════════════════════════════════════════════════
  BTC-USDT:  46743 bars  (2023-01-01 → 2025-09-01)
  ETH-USDT:  46743 bars  (2023-01-01 → 2025-09-01)
  SOL-USDT:  46743 bars  (2023-01-01 → 2025-09-01)
  BNB-USDT:  46743 bars  (2023-01-01 → 2025-09-01)
  XRP-USDT:  46743 bars  (2023-01-01 → 2025-09-01)
  DOGE-USDT:  46743 bars  (2023-01-01 → 2025-09-01)
  ADA-USDT:  46743 bars  (2023-01-01 → 2025-09-01)
  AVAX-USDT:  46743 bars  (2023-01-01 → 2025-09-01)

pairwise MI: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 36/36 [00:04<00:00,  8.39pair/s]


  [feat] triplet MI...
    triplet MI: pre-computing 36 pairs for 84 triplets — vectorized...


triplet MI (pairs): 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 36/36 [00:04<00:00,  8.44pair/s]


  [feat] total: 283 features  ×  46743 bars

═════════════════════════════════════════════════════════════════
  3. Targets  →  BTC
═════════════════════════════════════════════════════════════════
    BTC h=  8: stress 10978/46743 (23.5%)
    BTC h= 48: stress 13511/46743 (28.9%)
    BTC h= 96: stress 13929/46743 (29.8%)

  ── horizon h=8 (4h) ──
    [CLS] stress event  pos_rate=23.49%


...

            roc_auc          pr_auc              f1         bal_acc        
               mean     std    mean     std    mean     std    mean     std
model                                                                      
LogReg_k100  0.5406  0.2384  0.4729  0.3262  0.3294  0.2924  0.5454  0.1677
LogReg_k20   0.5169  0.2393  0.4552  0.3248  0.3400  0.3060  0.5457  0.1773
LogReg_k50   0.5375  0.2302  0.4716  0.3289  0.3429  0.3001  0.5556  0.1612
Persistence  0.4886  0.2086  0.4318  0.3154  0.2953  0.2289  0.5068  0.1526
    [REG] log vol level


WF folds: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 174/174 [00:12<00:00, 13.41fold/s]


                mae              r2             corr        
               mean     std    mean      std    mean     std
model                                                       
AR1          0.2894  0.0964 -0.9226   1.5317 -0.0034  0.2593
Persistence  0.2883  0.0835 -0.8532   0.8487  0.0002  0.2593
Ridge_k100   0.4698  0.2004 -5.5157  10.7583  0.0608  0.3551
Ridge_k20    0.3698  0.1468 -2.6675   4.8493  0.0646  0.3771
Ridge_k50    0.4133  0.1698 -3.5136   5.6155  0.0929  0.3550
    [REG] delta vol


WF folds: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 174/174 [00:12<00:00, 13.46fold/s]

                mae              r2             corr        
               mean     std    mean      std    mean     std
model                                                       
AR1          0.1619  0.0726  0.1621   0.7573  0.7480  0.1334
Persistence  0.3280  0.1408 -2.0972   1.0391 -0.7480  0.1334
Ridge_k100   0.2872  0.1658 -2.7526   7.9429  0.5065  0.2474
Ridge_k20    0.2148  0.1373 -1.3846   8.1428  0.5891  0.2557
Ridge_k50    0.2722  0.1751 -2.8404  10.7842  0.5004  0.2906

═════════════════════════════════════════════════════════════════
  FINAL SUMMARY  (mean across walk-forward folds)
═════════════════════════════════════════════════════════════════
                                roc_auc_mean  roc_auc_std  pr_auc_mean  pr_auc_std  f1_mean  f1_std  bal_acc_mean  bal_acc_std  mae_mean  mae_std  r2_mean   r2_std  corr_mean  corr_std
experiment         model                                                                                                                        

In [16]:
# results

{'BTC_CLS_stress_h8':             roc_auc          pr_auc              f1         bal_acc        
                mean     std    mean     std    mean     std    mean     std
 model                                                                      
 LogReg_k100  0.5623  0.1102  0.3138  0.2021  0.2912  0.1902  0.5386  0.0696
 LogReg_k20   0.6418  0.1125  0.3676  0.2137  0.3591  0.2113  0.5776  0.0855
 LogReg_k50   0.6052  0.1131  0.3477  0.2185  0.3303  0.2059  0.5627  0.0750
 Persistence  0.6404  0.1026  0.3499  0.2080  0.3513  0.1640  0.5985  0.0811,
 'BTC_REG_logvol_h8':                 mae              r2            corr        
                mean     std    mean     std    mean     std
 model                                                      
 AR1          0.4435  0.0569  0.0225  0.1362  0.3184  0.1557
 Persistence  0.4436  0.0543  0.0223  0.1292  0.3184  0.1557
 Ridge_k100   0.6758  0.2838 -1.6877  3.6283  0.1488  0.1962
 Ridge_k20    0.4467  0.0580 -0.0113  0.2521  0.3621

In [7]:
!pip install lightgbm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 43.5 MB/s  0:00:00


In [ ]:
# import warnings
# warnings.filterwarnings("ignore")

# import numpy as np
# import pandas as pd
# from itertools import combinations
# from tqdm.auto import tqdm

# from sklearn.metrics import (
#     roc_auc_score, average_precision_score,
#     f1_score, balanced_accuracy_score,
#     mean_absolute_error, r2_score,
# )
# from sklearn.linear_model import LogisticRegression, Ridge
# from sklearn.preprocessing import StandardScaler
# from sklearn.pipeline import Pipeline
# import lightgbm as lgb

# TICKER_MAP = {
#     "BTC":  "BTC-USDT",
#     "ETH":  "ETH-USDT",
#     "SOL":  "SOL-USDT",
#     "BNB":  "BNB-USDT",
#     "XRP":  "XRP-USDT",
#     "DOGE": "DOGE-USDT",
#     "ADA":  "ADA-USDT",
#     "AVAX": "AVAX-USDT",
#     "TRX":  "TRX-USDT",
# }

# TARGET_ASSETS = ["BTC", "ETH"]          
# UNIVERSE      = list(TICKER_MAP.keys()) 

# START_DATE = "2023-01-01"               

# BARS_PER_DAY = 48
# ANN_FACTOR   = np.sqrt(365 * BARS_PER_DAY)   

# VOL_WINDOW = 48     

# STRESS_MULT = 1.25

# HORIZONS = [8, 48, 96]   

# MI_WINDOW      = 192   
# MI_BINS        = 10
# MI_SUBSAMPLE_K = 4     

# TOP_K_FEATS = []   

# AR1_AUG_K = [3, 5, 10]

# TRAIN_BARS = 2880   
# TEST_BARS  =  480   
# STEP_BARS  =  240   
# PURGE_BARS =   96  

# USE_HEAVY_MODELS = False

# STRESS_MEDIAN_WINDOW = 365 * BARS_PER_DAY  

# def _to_datetime_index(df: pd.DataFrame) -> pd.DataFrame:
#     if not isinstance(df.index, pd.DatetimeIndex):
#         df = df.copy()
#         df.index = pd.to_datetime(df.index)
#     return df

# def prepare_universe(
#     price_df:   pd.DataFrame,
#     volume_ask: pd.DataFrame,
#     volume_bid: pd.DataFrame,
#     high_df:    pd.DataFrame,
#     low_df:     pd.DataFrame,
# ) -> dict:
#     """
#     Нарезает широкие датафреймы по нужным тикерам и дате.
#     Возвращает dict:  asset_key → {"close", "high", "low", "vol_ask", "vol_bid"}
#     каждый — pd.Series с DatetimeIndex.
#     """
#     for name, df in [("price_df", price_df), ("volume_ask", volume_ask),
#                      ("volume_bid", volume_bid), ("high_df", high_df), ("low_df", low_df)]:
#         if not isinstance(df.index, pd.DatetimeIndex):
#             raise TypeError(f"{name}: index must be DatetimeIndex, got {type(df.index)}")

#     cut = pd.Timestamp(START_DATE)
#     assets = {}
#     missing_tickers = []

#     for asset, ticker in TICKER_MAP.items():
#         if ticker not in price_df.columns:
#             missing_tickers.append(ticker)
#             continue

#         close   = price_df[ticker].loc[cut:].copy()
#         vol_ask = volume_ask[ticker].loc[cut:].copy() if ticker in volume_ask.columns else pd.Series(np.nan, index=close.index)
#         vol_bid = volume_bid[ticker].loc[cut:].copy() if ticker in volume_bid.columns else pd.Series(np.nan, index=close.index)
#         high    = high_df[ticker].loc[cut:].copy()    if ticker in high_df.columns    else pd.Series(np.nan, index=close.index)
#         low     = low_df[ticker].loc[cut:].copy()     if ticker in low_df.columns     else pd.Series(np.nan, index=close.index)

#         valid = close.notna()
#         assets[asset] = {
#             "close":   close[valid],
#             "high":    high[valid],
#             "low":     low[valid],
#             "vol_ask": vol_ask[valid],
#             "vol_bid": vol_bid[valid],
#         }
#         print(f"  {ticker}: {valid.sum():6d} bars  "
#               f"({close[valid].index[0].date()} → {close[valid].index[-1].date()})")

#     if missing_tickers:
#         print(f"\n  WARNING — tickers not found in price_df: {missing_tickers}")
#         print("  Available columns sample:", list(price_df.columns[:10]))

#     return assets

# def build_common_index(assets: dict, min_coverage: float = 0.7) -> pd.DatetimeIndex:
#     """
#     Строит общий индекс баров. Тикер включается если покрывает >= min_coverage от общего диапазона.
#     """
#     all_idx = pd.DatetimeIndex(
#         sorted(set().union(*[set(v["close"].index) for v in assets.values()]))
#     )
#     print(f"\n  Union index: {len(all_idx)} bars")

#     kept = []
#     for asset, data in assets.items():
#         cov = data["close"].reindex(all_idx).notna().mean()
#         if cov >= min_coverage:
#             kept.append(asset)
#             print(f"  {asset}: coverage {cov:.1%}  ✓")
#         else:
#             print(f"  {asset}: coverage {cov:.1%}  ✗ skipped")

#     idx = all_idx
#     for asset in kept:
#         idx = idx.intersection(assets[asset]["close"].index)

#     print(f"  Intersection index: {len(idx)} bars  "
#           f"({idx[0].date()} → {idx[-1].date()})")
#     return idx, kept

# def log_returns(close_wide: pd.DataFrame) -> pd.DataFrame:
#     return np.log(close_wide / close_wide.shift(1))

# def build_targets(
#     ret:          pd.DataFrame,
#     target_asset: str,
# ) -> pd.DataFrame:
#     """
#     Для одного target_asset строит таргеты на все горизонты.
#     Все вычисления — строго на ret (нет обращения к будущему в фичах).
#     """
#     r       = ret[target_asset]
#     vol_now = r.rolling(VOL_WINDOW).std() * ANN_FACTOR

#     tgt = pd.DataFrame(index=ret.index)
#     tgt["vol_now"] = vol_now

#     for h in HORIZONS:
#         vol_fwd = r.rolling(h).std().shift(-h) * ANN_FACTOR

#         vol_med = vol_now.rolling(STRESS_MEDIAN_WINDOW, min_periods=BARS_PER_DAY * 30).median()

#         tgt[f"vol_fwd_{h}"]     = vol_fwd
#         tgt[f"log_vol_fwd_{h}"] = np.log(vol_fwd.clip(lower=1e-8))
#         tgt[f"stress_{h}"]      = (vol_fwd > vol_med * STRESS_MULT).astype(float)
#         tgt[f"delta_vol_{h}"]   = vol_fwd - vol_now

#         n_stress = tgt[f"stress_{h}"].sum()
#         n_total  = tgt[f"stress_{h}"].notna().sum()
#         print(f"    {target_asset} h={h:3d}: stress {int(n_stress):5d}/{n_total} "
#               f"({n_stress/n_total*100:.1f}%)")

#     return tgt

# def _rank_normalize(arr: np.ndarray, bins: int) -> np.ndarray:
#     """
#     Превращает 1D массив в дискретные бины [0, bins) через rank-нормализацию.
#     Устойчиво к выбросам (в отличие от равномерных бинов по диапазону значений).
#     Векторизовано: работает на всём массиве сразу.
#     """
#     n = len(arr)
#     ranks = np.argsort(np.argsort(arr))          # ранги 0..n-1
#     return (ranks * bins // n).clip(0, bins - 1) # → [0, bins)

# def _batch_mi_single_pair(
#     col_x:  np.ndarray,  
#     col_y:  np.ndarray,   
#     window: int,
#     k:      int,
#     bins:   int,
# ) -> np.ndarray:
#     """
#     Считает rolling MI для одной пары (col_x, col_y) полностью без Python-цикла.

#     Подход:
#       1. Нарезаем все окна сразу через stride_tricks → shape (n_windows, window)
#       2. Rank-нормализуем каждое окно → дискретные бины
#       3. Кодируем совместный бин как  x_bin * bins + y_bin  → 1D индекс
#       4. Считаем joint histogram через np.bincount батчем → (n_windows, bins²)
#       5. Вычисляем MI через матричные операции

#     Нет ни одного Python-цикла — всё в C/numpy.
#     """
#     n = len(col_x)

#     starts = np.arange(0, n - window + 1, k)          
#     ends   = starts + window                          
#     n_win  = len(starts)

#     wx = np.stack([col_x[s:e] for s, e in zip(starts, ends)])  
#     wy = np.stack([col_y[s:e] for s, e in zip(starts, ends)])  

#     def rank_bins_2d(mat):
#         rk = np.argsort(np.argsort(mat, axis=1), axis=1)
#         return (rk * bins // window).clip(0, bins - 1).astype(np.int32)

#     bx = rank_bins_2d(wx)  
#     by = rank_bins_2d(wy)   
#     joint = bx * bins + by  

#     bins2 = bins * bins
#     offsets = (np.arange(n_win) * bins2).reshape(-1, 1)   
#     flat    = (joint + offsets).ravel()                    
#     counts  = np.bincount(flat, minlength=n_win * bins2).reshape(n_win, bins2)

#     counts  = counts.astype(np.float64) + 1e-10            
#     p_xy    = counts / counts.sum(axis=1, keepdims=True)   

#     p_xy_2d = p_xy.reshape(n_win, bins, bins)              
#     p_x     = p_xy_2d.sum(axis=2, keepdims=True)          
#     p_y     = p_xy_2d.sum(axis=1, keepdims=True)           

#     mi_vals = (p_xy_2d * np.log(p_xy_2d / (p_x * p_y))).sum(axis=(1, 2))
#     mi_vals = np.maximum(mi_vals, 0.0)                    

#     result = np.full(n, np.nan)
#     result[ends - 1] = mi_vals    
#     mask = np.isfinite(result)
#     idx  = np.where(mask, np.arange(n), 0)
#     np.maximum.accumulate(idx, out=idx)
#     result = result[idx]
#     return result

# def rolling_vol_features(
#     ret:       pd.DataFrame,
#     high_wide: pd.DataFrame,
#     low_wide:  pd.DataFrame,
#     vask_wide: pd.DataFrame,
#     vbid_wide: pd.DataFrame,
#     windows:   list = [4, 8, 16, 48, 96],
# ) -> pd.DataFrame:
#     feats = {}

#     for asset in ret.columns:
#         r = ret[asset]

#         for w in windows:
#             feats[f"{asset}_rvol_{w}"] = r.rolling(w).std() * ANN_FACTOR

#         rv = r.rolling(VOL_WINDOW).std() * ANN_FACTOR

#         feats[f"{asset}_vov"]         = rv.rolling(VOL_WINDOW).std()

#         rv_mean = rv.rolling(VOL_WINDOW * 4).mean()
#         rv_std  = rv.rolling(VOL_WINDOW * 4).std() + 1e-8
#         feats[f"{asset}_vol_zscore"]  = (rv - rv_mean) / rv_std

#         feats[f"{asset}_vol_pctrank"] = rv.rolling(VOL_WINDOW * 30).rank(pct=True)

#         h = high_wide[asset] if asset in high_wide.columns else pd.Series(np.nan, index=ret.index)
#         l = low_wide[asset]  if asset in low_wide.columns  else pd.Series(np.nan, index=ret.index)
#         hl2 = np.log((h / l).clip(lower=1e-8)) ** 2
#         feats[f"{asset}_pk_vol"] = np.sqrt(
#             hl2.rolling(VOL_WINDOW).mean() / (4 * np.log(2))
#         ) * ANN_FACTOR

#         va = vask_wide[asset] if asset in vask_wide.columns else pd.Series(0.0, index=ret.index)
#         vb = vbid_wide[asset] if asset in vbid_wide.columns else pd.Series(0.0, index=ret.index)
#         total = (va + vb).replace(0, np.nan)
#         raw_imb = (va - vb) / total
#         feats[f"{asset}_vol_imbalance"]     = raw_imb.rolling(8).mean()   
#         feats[f"{asset}_vol_imbalance_std"] = raw_imb.rolling(48).std()

#         for w in [4, 16, 48]:
#             feats[f"{asset}_mom_{w}"] = r.rolling(w).mean()

#     return pd.DataFrame(feats, index=ret.index)

# def cross_covariance_features(ret: pd.DataFrame, window: int = VOL_WINDOW) -> pd.DataFrame:
#     pairs = list(combinations(ret.columns, 2))
#     feats = {}
#     for a, b in pairs:
#         cov = ret[a].rolling(window).cov(ret[b])
#         std_a = ret[a].rolling(window).std() + 1e-8
#         std_b = ret[b].rolling(window).std() + 1e-8
#         feats[f"corr_{a}_{b}"] = cov / (std_a * std_b)
#     return pd.DataFrame(feats, index=ret.index)

# def pairwise_mi_features(
#     ret:    pd.DataFrame,
#     window: int = MI_WINDOW,
#     bins:   int = MI_BINS,
#     k:      int = MI_SUBSAMPLE_K,
# ) -> pd.DataFrame:
#     """
#     Rolling pairwise MI — полностью векторизовано, без Python-цикла по барам.
#     Цикл только по парам активов (36 итераций), внутри каждой — чистый numpy.
#     """
#     pairs   = list(combinations(ret.columns, 2))
#     ret_np  = ret.to_numpy(dtype=float)
#     col_idx = {c: i for i, c in enumerate(ret.columns)}
#     n       = len(ret)

#     n_windows = len(range(0, n - window + 1, k))
#     print(f"    pairwise MI: {len(pairs)} pairs, {n_windows} windows "
#           f"(k={k}) — fully vectorized...")

#     feat_dict = {}
#     for a, b in tqdm(pairs, desc="pairwise MI", unit="pair"):
#         feat_dict[f"mi_{a}_{b}"] = _batch_mi_single_pair(
#             ret_np[:, col_idx[a]], ret_np[:, col_idx[b]], window, k, bins
#         )

#     return pd.DataFrame(feat_dict, index=ret.index)

# def triplet_mi_features(
#     ret:    pd.DataFrame,
#     window: int = MI_WINDOW,
#     bins:   int = MI_BINS,
#     k:      int = MI_SUBSAMPLE_K,
# ) -> pd.DataFrame:
#     """
#     Rolling triplet MI = MI(a,b) + MI(b,c) + MI(a,c) — векторизовано.
#     Каждая пара считается один раз через _batch_mi_single_pair, потом суммируется.
#     """
#     triplets = list(combinations(ret.columns, 3))
#     ret_np   = ret.to_numpy(dtype=float)
#     col_idx  = {c: i for i, c in enumerate(ret.columns)}
#     n        = len(ret)

#     all_pairs = list(combinations(ret.columns, 2))
#     n_windows = len(range(0, n - window + 1, k))
#     print(f"    triplet MI: pre-computing {len(all_pairs)} pairs "
#           f"for {len(triplets)} triplets — vectorized...")

#     pair_mi = {}
#     for a, b in tqdm(all_pairs, desc="triplet MI (pairs)", unit="pair"):
#         key = (a, b)
#         pair_mi[key] = _batch_mi_single_pair(
#             ret_np[:, col_idx[a]], ret_np[:, col_idx[b]], window, k, bins
#         )

#     feat_dict = {}
#     for a, b, c in triplets:
#         feat_dict[f"tri_{a}_{b}_{c}"] = (
#             pair_mi[(a, b)] + pair_mi[(b, c)] + pair_mi[(a, c)]
#         )

#     return pd.DataFrame(feat_dict, index=ret.index)

# def build_features(ret: pd.DataFrame, high_w: pd.DataFrame,
#                    low_w: pd.DataFrame, vask_w: pd.DataFrame,
#                    vbid_w: pd.DataFrame,
#                    target_assets: list = None) -> pd.DataFrame:

#     print("  [feat] rolling vol / Parkinson / volume imbalance...")
#     vol_f = rolling_vol_features(ret, high_w, low_w, vask_w, vbid_w)

#     print("  [feat] rolling cross-correlation...")
#     corr_f = cross_covariance_features(ret)

#     print("  [feat] pairwise MI...")
#     mi_f = pairwise_mi_features(ret)

#     print("  [feat] triplet MI...")
#     tri_f = triplet_mi_features(ret)

#     X = pd.concat([vol_f, corr_f, mi_f, tri_f], axis=1)

#     assets = target_assets if target_assets else TARGET_ASSETS
#     vol_now_cols = [f"{a}_rvol_{VOL_WINDOW}" for a in assets if f"{a}_rvol_{VOL_WINDOW}" in X.columns]
#     X["vol_now_feat"] = X[vol_now_cols].mean(axis=1)

#     print(f"  [feat] total: {X.shape[1]} features  ×  {X.shape[0]} bars")
#     return X

# def select_top_k_corr(
#     Xtr: pd.DataFrame,
#     Xte: pd.DataFrame,
#     ytr: pd.Series,
#     k:   int,
#     always_include: str = "vol_now_feat",
# ) -> tuple:
#     """
#     Отбирает топ-k фич по |corr(feature, target)| на train-части.
#     always_include — колонка которая всегда остаётся (для Persistence/AR1).
#     Возвращает (Xtr_sel, Xte_sel) с одинаковым набором колонок.
#     Отбор строго на train → нет lookahead.
#     """
#     feat_cols = [c for c in Xtr.columns if c != always_include]
#     corrs = Xtr[feat_cols].corrwith(ytr).abs()
#     corrs = corrs.replace([np.inf, -np.inf], np.nan).dropna()
#     top_cols = corrs.nlargest(k).index.tolist()

#     if always_include in Xtr.columns and always_include not in top_cols:
#         top_cols = [always_include] + top_cols

#     return Xtr[top_cols], Xte[top_cols]

# def _wf_splits(n: int, train: int, test: int, step: int, purge: int):
#     """
#     Генератор (train_idx, test_idx) с purge-зазором.
#     train_end → +purge → test_start
#     """
#     start = train
#     while start + purge + test <= n:
#         train_idx = list(range(start - train, start))
#         test_start = start + purge
#         test_idx  = list(range(test_start, test_start + test))
#         yield train_idx, test_idx
#         start += step

# def select_ar1_aug(
#     Xtr: pd.DataFrame,
#     Xte: pd.DataFrame,
#     ytr: pd.Series,
#     k:   int,
#     vol_col: str = "vol_now_feat",
# ) -> tuple:
#     """
#     Augmented AR1 feature set:
#       [vol_now_feat]  +  топ-k доп. фич по |corr с таргетом| на train.
#     vol_now_feat исключается из конкурса отбора — она всегда первая.
#     Итого в X: k+1 фич.
#     """
#     other_cols = [c for c in Xtr.columns if c != vol_col]
#     corrs = Xtr[other_cols].corrwith(ytr).abs()
#     corrs = corrs.replace([np.inf, -np.inf], np.nan).dropna()
#     top_extra = corrs.nlargest(k).index.tolist()
#     cols = ([vol_col] if vol_col in Xtr.columns else []) + top_extra
#     return Xtr[cols], Xte[cols]

# def run_wf_classification(X, y, models,
#                            train=TRAIN_BARS, test=TEST_BARS,
#                            step=STEP_BARS, purge=PURGE_BARS):
#     mask   = X.notna().all(axis=1) & y.notna()
#     Xc, yc = X[mask], y[mask]
#     splits = list(_wf_splits(len(Xc), train, test, step, purge))

#     if not splits:
#         print("    ! Not enough data for walk-forward, skipping")
#         return pd.DataFrame()

#     baseline_names = {n for n, m in models.items()
#                       if isinstance(m, (PersistenceClsBaseline,))}

#     records = []
#     for fold, (tr, te) in enumerate(tqdm(splits, desc="WF folds", unit="fold")):
#         Xtr, Xte = Xc.iloc[tr], Xc.iloc[te]
#         ytr, yte = yc.iloc[tr], yc.iloc[te]

#         if len(ytr.unique()) < 2:
#             continue

#         for name, mdl in models.items():
#             if name in baseline_names:
#                 mdl.fit(Xtr, ytr)
#                 prob = mdl.predict_proba(Xte)[:, 1]
#                 pred = mdl.predict(Xte)
#                 _append_cls(records, fold, name, yte, prob, pred)

#         for k in TOP_K_FEATS:
#             Xtr_k, Xte_k = select_top_k_corr(Xtr, Xte, ytr, k)
#             mdl_k = Pipeline([
#                 ("sc", StandardScaler()),
#                 ("m",  LogisticRegression(C=0.1, max_iter=500, class_weight="balanced")),
#             ])
#             mdl_k.fit(Xtr_k, ytr)
#             prob = mdl_k.predict_proba(Xte_k)[:, 1]
#             pred = mdl_k.predict(Xte_k)
#             _append_cls(records, fold, f"LogReg_k{k}", yte, prob, pred)

#         for k in AR1_AUG_K:
#             Xtr_k, Xte_k = select_ar1_aug(Xtr, Xte, ytr, k)
#             mdl_k = Pipeline([
#                 ("sc", StandardScaler()),
#                 ("m",  LogisticRegression(C=0.1, max_iter=500, class_weight="balanced")),
#             ])
#             mdl_k.fit(Xtr_k, ytr)
#             prob = mdl_k.predict_proba(Xte_k)[:, 1]
#             pred = mdl_k.predict(Xte_k)
#             _append_cls(records, fold, f"AR1_aug_k{k}", yte, prob, pred)

#         if USE_HEAVY_MODELS:
#             for k in TOP_K_FEATS:
#                 Xtr_k, Xte_k = select_top_k_corr(Xtr, Xte, ytr, k)
#                 mdl_k = lgb.LGBMClassifier(
#                     n_estimators=50, max_depth=3, learning_rate=0.1,
#                     subsample=0.8, colsample_bytree=0.8, min_child_samples=30,
#                     class_weight="balanced", n_jobs=-1, verbose=-1, random_state=42,
#                 )
#                 mdl_k.fit(Xtr_k, ytr)
#                 prob = mdl_k.predict_proba(Xte_k)[:, 1]
#                 pred = mdl_k.predict(Xte_k)
#                 _append_cls(records, fold, f"LGBM_k{k}", yte, prob, pred)

#     return pd.DataFrame(records)

# def _append_cls(records, fold, name, yte, prob, pred):
#     try:
#         roc = roc_auc_score(yte, prob)
#         pr  = average_precision_score(yte, prob)
#     except Exception:
#         roc, pr = np.nan, np.nan
#     records.append({
#         "fold": fold, "model": name,
#         "roc_auc":  roc, "pr_auc": pr,
#         "f1":       f1_score(yte, pred, zero_division=0),
#         "bal_acc":  balanced_accuracy_score(yte, pred),
#         "pos_rate": float(yte.mean()), "n": len(yte),
#     })

# def run_wf_regression(X, y, models,
#                        train=TRAIN_BARS, test=TEST_BARS,
#                        step=STEP_BARS, purge=PURGE_BARS):
#     mask   = X.notna().all(axis=1) & y.notna()
#     Xc, yc = X[mask], y[mask]
#     splits = list(_wf_splits(len(Xc), train, test, step, purge))

#     if not splits:
#         print("    ! Not enough data for walk-forward, skipping")
#         return pd.DataFrame()

#     baseline_names = {n for n, m in models.items()
#                       if isinstance(m, (PersistenceRegBaseline, AR1RegBaseline))}

#     records = []
#     for fold, (tr, te) in enumerate(tqdm(splits, desc="WF folds", unit="fold")):
#         Xtr, Xte = Xc.iloc[tr], Xc.iloc[te]
#         ytr, yte = yc.iloc[tr], yc.iloc[te]

#         for name, mdl in models.items():
#             if name in baseline_names:
#                 mdl.fit(Xtr, ytr)
#                 pred = mdl.predict(Xte)
#                 _append_reg(records, fold, name, yte, pred)

#         for k in TOP_K_FEATS:
#             Xtr_k, Xte_k = select_top_k_corr(Xtr, Xte, ytr, k)
#             mdl_k = Pipeline([
#                 ("sc", StandardScaler()),
#                 ("m",  Ridge(alpha=1.0)),
#             ])
#             mdl_k.fit(Xtr_k, ytr)
#             pred = mdl_k.predict(Xte_k)
#             _append_reg(records, fold, f"Ridge_k{k}", yte, pred)

#         for k in AR1_AUG_K:
#             Xtr_k, Xte_k = select_ar1_aug(Xtr, Xte, ytr, k)
#             mdl_k = Pipeline([
#                 ("sc", StandardScaler()),
#                 ("m",  Ridge(alpha=1.0)),
#             ])
#             mdl_k.fit(Xtr_k, ytr)
#             pred = mdl_k.predict(Xte_k)
#             _append_reg(records, fold, f"AR1_aug_k{k}", yte, pred)

#         if USE_HEAVY_MODELS:
#             for k in TOP_K_FEATS:
#                 Xtr_k, Xte_k = select_top_k_corr(Xtr, Xte, ytr, k)
#                 mdl_k = lgb.LGBMRegressor(
#                     n_estimators=50, max_depth=3, learning_rate=0.1,
#                     subsample=0.8, colsample_bytree=0.8, min_child_samples=30,
#                     n_jobs=-1, verbose=-1, random_state=42,
#                 )
#                 mdl_k.fit(Xtr_k, ytr)
#                 pred = mdl_k.predict(Xte_k)
#                 _append_reg(records, fold, f"LGBM_k{k}", yte, pred)

#     return pd.DataFrame(records)

# def _append_reg(records, fold, name, yte, pred):
#     corr = float(pd.Series(pred).corr(pd.Series(yte.values)))
#     records.append({
#         "fold": fold, "model": name,
#         "mae":  mean_absolute_error(yte, pred),
#         "r2":   r2_score(yte, pred),
#         "corr": corr,
#         "n":    len(yte),
#     })

# class PersistenceClsBaseline:
#     """
#     Классификатор-baseline: предсказывает стресс если vol_now > порога.
#     Порог = rolling quantile(stress_mult) внутри train-сета.
#     predict_proba возвращает нормированную vol_now как "вероятность".
#     Не требует обучения — fit() только запоминает порог.
#     """
#     def __init__(self, vol_now_col: str = "vol_now_feat"):
#         self.vol_now_col = vol_now_col
#         self.threshold_  = None

#     def fit(self, X, y):
#         if self.vol_now_col in X.columns:
#             vol = X[self.vol_now_col].values
#         else:
#             vol = X.iloc[:, 0].values   # fallback: первая колонка
#         pos_rate = float(y.mean())
#         self.threshold_ = np.quantile(vol[np.isfinite(vol)], 1 - pos_rate)
#         return self

#     def predict_proba(self, X):
#         if self.vol_now_col in X.columns:
#             vol = X[self.vol_now_col].values
#         else:
#             vol = X.iloc[:, 0].values
#         vol = np.where(np.isfinite(vol), vol, 0.0)
#         vmin, vmax = vol.min(), vol.max()
#         prob = (vol - vmin) / (vmax - vmin + 1e-8)
#         return np.column_stack([1 - prob, prob])

#     def predict(self, X):
#         return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

# class PersistenceRegBaseline:
#     """
#     Регрессор-baseline: предсказывает vol_fwd = vol_now (persistence).
#     Для log_vol_fwd — возвращает log(vol_now).
#     Для delta_vol   — возвращает 0 (нет изменения).
#     fit() просто запоминает среднее смещение на train (bias correction).
#     """
#     def __init__(self, vol_now_col: str = "vol_now_feat"):
#         self.vol_now_col = vol_now_col
#         self.bias_       = 0.0

#     def fit(self, X, y):
#         if self.vol_now_col in X.columns:
#             vol = X[self.vol_now_col].values
#         else:
#             vol = X.iloc[:, 0].values
#         pred = vol[np.isfinite(vol) & np.isfinite(y.values)]
#         ytrue = y.values[np.isfinite(vol) & np.isfinite(y.values)]
#         self.bias_ = float(np.mean(ytrue - pred)) if len(pred) else 0.0
#         return self

#     def predict(self, X):
#         if self.vol_now_col in X.columns:
#             vol = X[self.vol_now_col].values
#         else:
#             vol = X.iloc[:, 0].values
#         return np.where(np.isfinite(vol), vol + self.bias_, np.nanmean(vol))

# class AR1RegBaseline:
#     """
#     AR(1) baseline на воле — регрессия вида:
#         y = a * vol_now + b
#     Обучается только на одной фиче (vol_now_feat), без остального X.
#     Для delta_vol это означает: delta = a * vol_now + b
#     Честный baseline: использует обучение, но только из одной переменной.
#     Позволяет отделить "сигнал от уровня волы" от "сигнала от cross-asset фич".
#     """
#     def __init__(self, vol_now_col: str = "vol_now_feat"):
#         self.vol_now_col = vol_now_col
#         self.a_ = 0.0
#         self.b_ = 0.0

#     def _get_vol(self, X):
#         if self.vol_now_col in X.columns:
#             return X[self.vol_now_col].values.astype(float)
#         return X.iloc[:, 0].values.astype(float)

#     def fit(self, X, y):
#         vol   = self._get_vol(X)
#         ytrue = y.values.astype(float)
#         mask  = np.isfinite(vol) & np.isfinite(ytrue)
#         v, t  = vol[mask], ytrue[mask]
#         if len(v) < 2:
#             self.a_, self.b_ = 0.0, float(np.nanmean(ytrue))
#             return self
#         self.a_ = float(np.cov(v, t)[0, 1] / (np.var(v) + 1e-12))
#         self.b_ = float(np.mean(t) - self.a_ * np.mean(v))
#         return self

#     def predict(self, X):
#         vol = self._get_vol(X)
#         return np.where(np.isfinite(vol), self.a_ * vol + self.b_, self.b_)

# def get_cls_models() -> dict:
#     models = {
#         "Persistence": PersistenceClsBaseline(vol_now_col="vol_now_feat"),
#         "LogReg": Pipeline([
#             ("sc", StandardScaler()),
#             ("m",  LogisticRegression(C=0.1, max_iter=500, class_weight="balanced")),
#         ]),
#     }
#     if USE_HEAVY_MODELS:
#         models["LGBM"] = lgb.LGBMClassifier(
#             n_estimators=50, max_depth=3,
#             learning_rate=0.1, subsample=0.8,
#             colsample_bytree=0.5, min_child_samples=30,
#             class_weight="balanced", n_jobs=-1,
#             verbose=-1, random_state=42,
#         )
#     return models

# def get_reg_models() -> dict:
#     models = {
#         "Persistence": PersistenceRegBaseline(vol_now_col="vol_now_feat"),
#         "AR1":         AR1RegBaseline(vol_now_col="vol_now_feat"),
#         "Ridge": Pipeline([
#             ("sc", StandardScaler()),
#             ("m",  Ridge(alpha=1.0)),
#         ]),
#     }
#     if USE_HEAVY_MODELS:
#         models["LGBM"] = lgb.LGBMRegressor(
#             n_estimators=50, max_depth=3,
#             learning_rate=0.1, subsample=0.8,
#             colsample_bytree=0.5, min_child_samples=30,
#             n_jobs=-1, verbose=-1, random_state=42,
#         )
#     return models

# def _summarize(df: pd.DataFrame, metric_cols: list) -> pd.DataFrame:
#     if df.empty:
#         return pd.DataFrame()
#     return (
#         df.groupby("model")[metric_cols]
#         .agg(["mean", "std"])
#         .round(4)
#     )

# def print_block(title: str):
#     print(f"\n{'═'*65}")
#     print(f"  {title}")
#     print(f"{'═'*65}")

# def run_experiment(
#     price_df:   pd.DataFrame,
#     volume_ask: pd.DataFrame,
#     volume_bid: pd.DataFrame,
#     high_df:    pd.DataFrame,
#     low_df:     pd.DataFrame,
# ) -> dict:
#     """
#     Запускается так:
#         results = run_experiment(price_df, volume_ask, volume_bid, high_df, low_df)

#     price_df = (bt2.data['STRATEX-BINANCE']['best_ask']
#               + bt2.data['STRATEX-BINANCE']['best_bid']) / 2
#     volume_ask = bt2.data['STRATEX-BINANCE']['buy_volume']
#     volume_bid = bt2.data['STRATEX-BINANCE']['sell_volume']
#     high_df    = bt2.data['STRATEX-BINANCE-F']['high']
#     low_df     = bt2.data['STRATEX-BINANCE-F']['low']
#     """

#     print_block("CRYPTO VOL STRESS EXPERIMENT  —  30-min bars")
#     print(f"  Targets  : {TARGET_ASSETS}")
#     print(f"  Universe : {UNIVERSE}")
#     print(f"  Horizons : {HORIZONS} bars  =  "
#           f"{[f'{h//2}h' if h < 48 else f'{h//48}d' for h in HORIZONS]}")
#     print(f"  Date     : {START_DATE} → end")
#     print(f"  WF split : train={TRAIN_BARS} / test={TEST_BARS} / "
#           f"step={STEP_BARS} / purge={PURGE_BARS} bars")

#     print_block("1. Preparing asset data")
#     assets = prepare_universe(price_df, volume_ask, volume_bid, high_df, low_df)
#     common_idx, active_assets = build_common_index(assets)

#     for t in TARGET_ASSETS:
#         if t not in active_assets:
#             raise ValueError(
#                 f"Target asset '{t}' ({TICKER_MAP[t]}) not found or has insufficient coverage. "
#                 f"Active assets: {active_assets}"
#             )

#     def _wide(field: str) -> pd.DataFrame:
#         return pd.DataFrame(
#             {a: assets[a][field].reindex(common_idx) for a in active_assets}
#         )

#     close_w  = _wide("close")
#     high_w   = _wide("high")
#     low_w    = _wide("low")
#     vask_w   = _wide("vol_ask")
#     vbid_w   = _wide("vol_bid")

#     ret = log_returns(close_w)

#     print_block("2. Building features")
#     X = build_features(ret, high_w, low_w, vask_w, vbid_w)

#     all_results = {}

#     for target in TARGET_ASSETS:
#         print_block(f"3. Targets  →  {target}")
#         tgt = build_targets(ret, target)
#         tgt = tgt.loc[X.index]

#         for h in HORIZONS:
#             horizon_label = f"{h//2}h" if h < 48 else f"{h//48}d"
#             print(f"\n  ── horizon h={h} ({horizon_label}) ──")

#             y_cls = tgt[f"stress_{h}"].dropna()
#             Xc    = X.loc[y_cls.index]
#             pos   = y_cls.mean()
#             print(f"    [CLS] stress event  pos_rate={pos:.2%}")
#             df_cls = run_wf_classification(Xc, y_cls, get_cls_models())
#             s_cls  = _summarize(df_cls, ["roc_auc", "pr_auc", "f1", "bal_acc"])
#             if not s_cls.empty:
#                 print(s_cls.to_string())
#             key = f"{target}_CLS_stress_h{h}"
#             all_results[key] = s_cls

#             y_vol = tgt[f"log_vol_fwd_{h}"].dropna()
#             Xr    = X.loc[y_vol.index]
#             print(f"    [REG] log vol level")
#             df_vol = run_wf_regression(Xr, y_vol, get_reg_models())
#             s_vol  = _summarize(df_vol, ["mae", "r2", "corr"])
#             if not s_vol.empty:
#                 print(s_vol.to_string())
#             all_results[f"{target}_REG_logvol_h{h}"] = s_vol

#             y_dv = tgt[f"delta_vol_{h}"].dropna()
#             Xd   = X.loc[y_dv.index]
#             print(f"    [REG] delta vol")
#             df_dv = run_wf_regression(Xd, y_dv, get_reg_models())
#             s_dv  = _summarize(df_dv, ["mae", "r2", "corr"])
#             if not s_dv.empty:
#                 print(s_dv.to_string())
#             all_results[f"{target}_REG_delta_h{h}"] = s_dv

#     print_block("FINAL SUMMARY  (mean across walk-forward folds)")
#     rows = []
#     for exp_key, df in all_results.items():
#         if df.empty:
#             continue
#         for model_name in df.index.get_level_values(0).unique():
#             row = {"experiment": exp_key, "model": model_name}
#             for col in df.columns.get_level_values(0).unique():
#                 row[f"{col}_mean"] = df.loc[model_name, (col, "mean")]
#                 row[f"{col}_std"]  = df.loc[model_name, (col, "std")]
#             rows.append(row)

#     summary = pd.DataFrame(rows).set_index(["experiment", "model"])
#     print(summary.to_string())

#     return all_results, summary

# price_df   = (bt2.data['STRATEX-BINANCE']['best_ask']
#             + bt2.data['STRATEX-BINANCE']['best_bid']) / 2
# volume_ask = bt2.data['STRATEX-BINANCE']['buy_volume']
# volume_bid = bt2.data['STRATEX-BINANCE']['sell_volume']
# high_df    = bt2.data['STRATEX-BINANCE-F']['high']
# low_df     = bt2.data['STRATEX-BINANCE-F']['low']

# results, summary = run_experiment(
#     price_df, volume_ask, volume_bid, high_df, low_df
# )


═════════════════════════════════════════════════════════════════
  CRYPTO VOL STRESS EXPERIMENT  —  30-min bars
═════════════════════════════════════════════════════════════════
  Targets  : ['BTC', 'ETH']
  Universe : ['BTC', 'ETH', 'SOL', 'BNB', 'XRP', 'DOGE', 'ADA', 'AVAX', 'TRX']
  Horizons : [8, 48, 96] bars  =  ['4h', '1d', '2d']
  Date     : 2023-01-01 → end
  WF split : train=2880 / test=480 / step=240 / purge=96 bars

═════════════════════════════════════════════════════════════════
  1. Preparing asset data
═════════════════════════════════════════════════════════════════
  BTC-USDT:  46743 bars  (2023-01-01 → 2025-09-01)
  ETH-USDT:  46743 bars  (2023-01-01 → 2025-09-01)
  SOL-USDT:  46743 bars  (2023-01-01 → 2025-09-01)
  BNB-USDT:  46743 bars  (2023-01-01 → 2025-09-01)
  XRP-USDT:  46743 bars  (2023-01-01 → 2025-09-01)
  DOGE-USDT:  46743 bars  (2023-01-01 → 2025-09-01)
  ADA-USDT:  46743 bars  (2023-01-01 → 2025-09-01)
  AVAX-USDT:  46743 bars  (2023-01-01 → 2025-09-01)

pairwise MI: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 36/36 [00:04<00:00,  8.46pair/s]


  [feat] triplet MI...
    triplet MI: pre-computing 36 pairs for 84 triplets — vectorized...


triplet MI (pairs): 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 36/36 [00:04<00:00,  8.42pair/s]

  [feat] total: 283 features  ×  46743 bars

═════════════════════════════════════════════════════════════════
  3. Targets  →  BTC
═════════════════════════════════════════════════════════════════


...

            roc_auc          pr_auc              f1         bal_acc        
               mean     std    mean     std    mean     std    mean     std
model                                                                      
AR1_aug_k10  0.4906  0.2391  0.4449  0.3210  0.3256  0.3036  0.5323  0.1766
AR1_aug_k3   0.4860  0.2463  0.4473  0.3196  0.3317  0.3013  0.5204  0.1618
AR1_aug_k5   0.4855  0.2383  0.4417  0.3197  0.3139  0.2975  0.5179  0.1648
Persistence  0.4886  0.2086  0.4318  0.3154  0.2953  0.2289  0.5068  0.1526
    [REG] log vol level


WF folds: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 174/174 [00:11<00:00, 14.79fold/s]


                mae              r2            corr        
               mean     std    mean     std    mean     std
model                                                      
AR1          0.2894  0.0964 -0.9226  1.5317 -0.0034  0.2593
AR1_aug_k10  0.3373  0.1282 -1.9699  3.8870  0.0664  0.3787
AR1_aug_k3   0.3187  0.1275 -1.5844  3.3714  0.0660  0.3993
AR1_aug_k5   0.3194  0.1245 -1.5719  3.2795  0.0546  0.3940
Persistence  0.2883  0.0835 -0.8532  0.8487  0.0002  0.2593
    [REG] delta vol


WF folds: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 174/174 [00:11<00:00, 14.81fold/s]

                mae              r2            corr        
               mean     std    mean     std    mean     std
model                                                      
AR1          0.1619  0.0726  0.1621  0.7573  0.7480  0.1334
AR1_aug_k10  0.1916  0.1031 -0.3464  2.7777  0.6371  0.2379
AR1_aug_k3   0.1655  0.0789  0.1363  0.8029  0.7405  0.1485
AR1_aug_k5   0.1726  0.0881  0.0689  0.8394  0.7194  0.1566
Persistence  0.3280  0.1408 -2.0972  1.0391 -0.7480  0.1334

═════════════════════════════════════════════════════════════════
  FINAL SUMMARY  (mean across walk-forward folds)
═════════════════════════════════════════════════════════════════
                                roc_auc_mean  roc_auc_std  pr_auc_mean  pr_auc_std  f1_mean  f1_std  bal_acc_mean  bal_acc_std  mae_mean  mae_std  r2_mean  r2_std  corr_mean  corr_std
experiment         model                                                                                                                                 

In [18]:
# results

{'BTC_CLS_stress_h8':             roc_auc          pr_auc              f1         bal_acc        
                mean     std    mean     std    mean     std    mean     std
 model                                                                      
 AR1_aug_k10  0.6554  0.1040  0.3707  0.2153  0.3708  0.2051  0.5876  0.0783
 AR1_aug_k3   0.6558  0.0983  0.3700  0.2175  0.3682  0.2104  0.5823  0.0812
 AR1_aug_k5   0.6573  0.1002  0.3740  0.2190  0.3719  0.2087  0.5863  0.0786
 Persistence  0.6404  0.1026  0.3499  0.2080  0.3513  0.1640  0.5985  0.0811,
 'BTC_REG_logvol_h8':                 mae              r2            corr        
                mean     std    mean     std    mean     std
 model                                                      
 AR1          0.4435  0.0569  0.0225  0.1362  0.3184  0.1557
 AR1_aug_k10  0.4380  0.0601  0.0339  0.2060  0.3778  0.1430
 AR1_aug_k3   0.4406  0.0605  0.0294  0.1926  0.3609  0.1469
 AR1_aug_k5   0.4365  0.0564  0.0404  0.1994  0.3732

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from itertools import combinations
from tqdm.auto import tqdm

from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    f1_score, balanced_accuracy_score,
    mean_absolute_error, r2_score,
)
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import lightgbm as lgb

TICKER_MAP = {
    "BTC":  "BTC-USDT",
    "ETH":  "ETH-USDT",
    "SOL":  "SOL-USDT",
    "BNB":  "BNB-USDT",
    "XRP":  "XRP-USDT",
    "DOGE": "DOGE-USDT",
    "ADA":  "ADA-USDT",
    "AVAX": "AVAX-USDT",
    "TRX":  "TRX-USDT",
}

TARGET_ASSETS = ["BTC", "ETH"]          
UNIVERSE      = list(TICKER_MAP.keys()) 

START_DATE = "2023-01-01"              

BARS_PER_DAY = 48
ANN_FACTOR   = np.sqrt(365 * BARS_PER_DAY)   

VOL_WINDOW = 48     # 24h

STRESS_MULT = 1.25

HORIZONS = [8, 48, 96]  

MI_WINDOW      = 192   
MI_BINS        = 10
MI_SUBSAMPLE_K = 4    

TOP_K_FEATS = []   

AR1_AUG_K = [3, 5, 10]

TRAIN_BARS = 2880  
TEST_BARS  =  480   
STEP_BARS  =  240   
PURGE_BARS =   96   

USE_HEAVY_MODELS = False

STRESS_MEDIAN_WINDOW = 365 * BARS_PER_DAY   

def _to_datetime_index(df: pd.DataFrame) -> pd.DataFrame:
    if not isinstance(df.index, pd.DatetimeIndex):
        df = df.copy()
        df.index = pd.to_datetime(df.index)
    return df

def prepare_universe(
    price_df:   pd.DataFrame,
    volume_ask: pd.DataFrame,
    volume_bid: pd.DataFrame,
    high_df:    pd.DataFrame,
    low_df:     pd.DataFrame,
) -> dict:
    """
    Нарезает широкие датафреймы по нужным тикерам и дате.
    Возвращает dict:  asset_key → {"close", "high", "low", "vol_ask", "vol_bid"}
    каждый — pd.Series с DatetimeIndex.
    """
    for name, df in [("price_df", price_df), ("volume_ask", volume_ask),
                     ("volume_bid", volume_bid), ("high_df", high_df), ("low_df", low_df)]:
        if not isinstance(df.index, pd.DatetimeIndex):
            raise TypeError(f"{name}: index must be DatetimeIndex, got {type(df.index)}")

    cut = pd.Timestamp(START_DATE)
    assets = {}
    missing_tickers = []

    for asset, ticker in TICKER_MAP.items():
        if ticker not in price_df.columns:
            missing_tickers.append(ticker)
            continue

        close   = price_df[ticker].loc[cut:].copy()
        vol_ask = volume_ask[ticker].loc[cut:].copy() if ticker in volume_ask.columns else pd.Series(np.nan, index=close.index)
        vol_bid = volume_bid[ticker].loc[cut:].copy() if ticker in volume_bid.columns else pd.Series(np.nan, index=close.index)
        high    = high_df[ticker].loc[cut:].copy()    if ticker in high_df.columns    else pd.Series(np.nan, index=close.index)
        low     = low_df[ticker].loc[cut:].copy()     if ticker in low_df.columns     else pd.Series(np.nan, index=close.index)

        valid = close.notna()
        assets[asset] = {
            "close":   close[valid],
            "high":    high[valid],
            "low":     low[valid],
            "vol_ask": vol_ask[valid],
            "vol_bid": vol_bid[valid],
        }
        print(f"  {ticker}: {valid.sum():6d} bars  "
              f"({close[valid].index[0].date()} → {close[valid].index[-1].date()})")

    if missing_tickers:
        print(f"\n  WARNING — tickers not found in price_df: {missing_tickers}")
        print("  Available columns sample:", list(price_df.columns[:10]))

    return assets

def build_common_index(assets: dict, min_coverage: float = 0.7) -> pd.DatetimeIndex:
    """
    Строит общий индекс баров. Тикер включается если покрывает >= min_coverage от общего диапазона.
    """
    all_idx = pd.DatetimeIndex(
        sorted(set().union(*[set(v["close"].index) for v in assets.values()]))
    )
    print(f"\n  Union index: {len(all_idx)} bars")

    kept = []
    for asset, data in assets.items():
        cov = data["close"].reindex(all_idx).notna().mean()
        if cov >= min_coverage:
            kept.append(asset)
            print(f"  {asset}: coverage {cov:.1%}  ✓")
        else:
            print(f"  {asset}: coverage {cov:.1%}  ✗ skipped")

    idx = all_idx
    for asset in kept:
        idx = idx.intersection(assets[asset]["close"].index)

    print(f"  Intersection index: {len(idx)} bars  "
          f"({idx[0].date()} → {idx[-1].date()})")
    return idx, kept

def log_returns(close_wide: pd.DataFrame) -> pd.DataFrame:
    return np.log(close_wide / close_wide.shift(1))

def build_targets(
    ret:          pd.DataFrame,
    target_asset: str,
) -> pd.DataFrame:
    """
    Для одного target_asset строит таргеты на все горизонты.
    Все вычисления — строго на ret (нет обращения к будущему в фичах).
    """
    r       = ret[target_asset]
    vol_now = r.rolling(VOL_WINDOW).std() * ANN_FACTOR

    tgt = pd.DataFrame(index=ret.index)
    tgt["vol_now"] = vol_now

    for h in HORIZONS:
        vol_fwd = r.rolling(h).std().shift(-h) * ANN_FACTOR

        vol_med = vol_now.rolling(STRESS_MEDIAN_WINDOW, min_periods=BARS_PER_DAY * 30).median()

        tgt[f"vol_fwd_{h}"]     = vol_fwd
        tgt[f"log_vol_fwd_{h}"] = np.log(vol_fwd.clip(lower=1e-8))
        tgt[f"stress_{h}"]      = (vol_fwd > vol_med * STRESS_MULT).astype(float)
        tgt[f"delta_vol_{h}"]   = vol_fwd - vol_now

        n_stress = tgt[f"stress_{h}"].sum()
        n_total  = tgt[f"stress_{h}"].notna().sum()
        print(f"    {target_asset} h={h:3d}: stress {int(n_stress):5d}/{n_total} "
              f"({n_stress/n_total*100:.1f}%)")

    return tgt

def _rank_normalize(arr: np.ndarray, bins: int) -> np.ndarray:
    """
    Превращает 1D массив в дискретные бины [0, bins) через rank-нормализацию.
    Устойчиво к выбросам (в отличие от равномерных бинов по диапазону значений).
    Векторизовано: работает на всём массиве сразу.
    """
    n = len(arr)
    ranks = np.argsort(np.argsort(arr))         
    return (ranks * bins // n).clip(0, bins - 1) 

def _batch_mi_single_pair(
    col_x:  np.ndarray,   
    col_y:  np.ndarray,   
    window: int,
    k:      int,
    bins:   int,
) -> np.ndarray:
    """
    Считает rolling MI для одной пары (col_x, col_y) полностью без Python-цикла.

    Подход:
      1. Нарезаем все окна сразу через stride_tricks → shape (n_windows, window)
      2. Rank-нормализуем каждое окно → дискретные бины
      3. Кодируем совместный бин как  x_bin * bins + y_bin  → 1D индекс
      4. Считаем joint histogram через np.bincount батчем → (n_windows, bins²)
      5. Вычисляем MI через матричные операции
    """
    n = len(col_x)

    starts = np.arange(0, n - window + 1, k)     
    ends   = starts + window                        
    n_win  = len(starts)

    wx = np.stack([col_x[s:e] for s, e in zip(starts, ends)]) 
    wy = np.stack([col_y[s:e] for s, e in zip(starts, ends)]) 

    def rank_bins_2d(mat):
        rk = np.argsort(np.argsort(mat, axis=1), axis=1)
        return (rk * bins // window).clip(0, bins - 1).astype(np.int32)

    bx = rank_bins_2d(wx)
    by = rank_bins_2d(wy)

    joint = bx * bins + by

    bins2 = bins * bins
    offsets = (np.arange(n_win) * bins2).reshape(-1, 1)   
    flat    = (joint + offsets).ravel()                    
    counts  = np.bincount(flat, minlength=n_win * bins2).reshape(n_win, bins2)

    counts  = counts.astype(np.float64) + 1e-10            
    p_xy    = counts / counts.sum(axis=1, keepdims=True)   

    p_xy_2d = p_xy.reshape(n_win, bins, bins)             
    p_x     = p_xy_2d.sum(axis=2, keepdims=True)         
    p_y     = p_xy_2d.sum(axis=1, keepdims=True)          

    mi_vals = (p_xy_2d * np.log(p_xy_2d / (p_x * p_y))).sum(axis=(1, 2))
    mi_vals = np.maximum(mi_vals, 0.0)                     

    result = np.full(n, np.nan)
    result[ends - 1] = mi_vals 
    mask = np.isfinite(result)
    idx  = np.where(mask, np.arange(n), 0)
    np.maximum.accumulate(idx, out=idx)
    result = result[idx]
    return result

def rolling_vol_features(
    ret:       pd.DataFrame,
    high_wide: pd.DataFrame,
    low_wide:  pd.DataFrame,
    vask_wide: pd.DataFrame,
    vbid_wide: pd.DataFrame,
    windows:   list = [4, 8, 16, 48, 96],
) -> pd.DataFrame:
    feats = {}

    for asset in ret.columns:
        r = ret[asset]

        for w in windows:
            feats[f"{asset}_rvol_{w}"] = r.rolling(w).std() * ANN_FACTOR

        rv = r.rolling(VOL_WINDOW).std() * ANN_FACTOR

        feats[f"{asset}_vov"]         = rv.rolling(VOL_WINDOW).std()

        rv_mean = rv.rolling(VOL_WINDOW * 4).mean()
        rv_std  = rv.rolling(VOL_WINDOW * 4).std() + 1e-8
        feats[f"{asset}_vol_zscore"]  = (rv - rv_mean) / rv_std

        feats[f"{asset}_vol_pctrank"] = rv.rolling(VOL_WINDOW * 30).rank(pct=True)

        h = high_wide[asset] if asset in high_wide.columns else pd.Series(np.nan, index=ret.index)
        l = low_wide[asset]  if asset in low_wide.columns  else pd.Series(np.nan, index=ret.index)
        hl2 = np.log((h / l).clip(lower=1e-8)) ** 2
        feats[f"{asset}_pk_vol"] = np.sqrt(
            hl2.rolling(VOL_WINDOW).mean() / (4 * np.log(2))
        ) * ANN_FACTOR

        va = vask_wide[asset] if asset in vask_wide.columns else pd.Series(0.0, index=ret.index)
        vb = vbid_wide[asset] if asset in vbid_wide.columns else pd.Series(0.0, index=ret.index)
        total = (va + vb).replace(0, np.nan)
        raw_imb = (va - vb) / total
        feats[f"{asset}_vol_imbalance"]     = raw_imb.rolling(8).mean()   
        feats[f"{asset}_vol_imbalance_std"] = raw_imb.rolling(48).std()

        for w in [4, 16, 48]:
            feats[f"{asset}_mom_{w}"] = r.rolling(w).mean()

    return pd.DataFrame(feats, index=ret.index)

def cross_covariance_features(ret: pd.DataFrame, window: int = VOL_WINDOW) -> pd.DataFrame:
    pairs = list(combinations(ret.columns, 2))
    feats = {}
    for a, b in pairs:
        cov = ret[a].rolling(window).cov(ret[b])
        std_a = ret[a].rolling(window).std() + 1e-8
        std_b = ret[b].rolling(window).std() + 1e-8
        feats[f"corr_{a}_{b}"] = cov / (std_a * std_b)
    return pd.DataFrame(feats, index=ret.index)

def pairwise_mi_features(
    ret:    pd.DataFrame,
    window: int = MI_WINDOW,
    bins:   int = MI_BINS,
    k:      int = MI_SUBSAMPLE_K,
) -> pd.DataFrame:
    """
    Rolling pairwise MI — полностью векторизовано, без Python-цикла по барам.
    Цикл только по парам активов (36 итераций), внутри каждой — чистый numpy.
    """
    pairs   = list(combinations(ret.columns, 2))
    ret_np  = ret.to_numpy(dtype=float)
    col_idx = {c: i for i, c in enumerate(ret.columns)}
    n       = len(ret)

    n_windows = len(range(0, n - window + 1, k))
    print(f"    pairwise MI: {len(pairs)} pairs, {n_windows} windows "
          f"(k={k}) — fully vectorized...")

    feat_dict = {}
    for a, b in tqdm(pairs, desc="pairwise MI", unit="pair"):
        feat_dict[f"mi_{a}_{b}"] = _batch_mi_single_pair(
            ret_np[:, col_idx[a]], ret_np[:, col_idx[b]], window, k, bins
        )

    return pd.DataFrame(feat_dict, index=ret.index)

def triplet_mi_features(
    ret:    pd.DataFrame,
    window: int = MI_WINDOW,
    bins:   int = MI_BINS,
    k:      int = MI_SUBSAMPLE_K,
) -> pd.DataFrame:
    """
    Rolling triplet MI = MI(a,b) + MI(b,c) + MI(a,c) — векторизовано.
    Каждая пара считается один раз через _batch_mi_single_pair, потом суммируется.
    """
    triplets = list(combinations(ret.columns, 3))
    ret_np   = ret.to_numpy(dtype=float)
    col_idx  = {c: i for i, c in enumerate(ret.columns)}
    n        = len(ret)

    all_pairs = list(combinations(ret.columns, 2))
    n_windows = len(range(0, n - window + 1, k))
    print(f"    triplet MI: pre-computing {len(all_pairs)} pairs "
          f"for {len(triplets)} triplets — vectorized...")

    pair_mi = {}
    for a, b in tqdm(all_pairs, desc="triplet MI (pairs)", unit="pair"):
        key = (a, b)
        pair_mi[key] = _batch_mi_single_pair(
            ret_np[:, col_idx[a]], ret_np[:, col_idx[b]], window, k, bins
        )

    feat_dict = {}
    for a, b, c in triplets:
        feat_dict[f"tri_{a}_{b}_{c}"] = (
            pair_mi[(a, b)] + pair_mi[(b, c)] + pair_mi[(a, c)]
        )

    return pd.DataFrame(feat_dict, index=ret.index)

def build_features(ret: pd.DataFrame, high_w: pd.DataFrame,
                   low_w: pd.DataFrame, vask_w: pd.DataFrame,
                   vbid_w: pd.DataFrame,
                   target_assets: list = None) -> pd.DataFrame:

    print("  [feat] rolling vol / Parkinson / volume imbalance...")
    vol_f = rolling_vol_features(ret, high_w, low_w, vask_w, vbid_w)

    print("  [feat] rolling cross-correlation...")
    corr_f = cross_covariance_features(ret)

    print("  [feat] pairwise MI...")
    mi_f = pairwise_mi_features(ret)

    print("  [feat] triplet MI...")
    tri_f = triplet_mi_features(ret)

    X = pd.concat([vol_f, corr_f, mi_f, tri_f], axis=1)

    assets = target_assets if target_assets else TARGET_ASSETS
    vol_now_cols = [f"{a}_rvol_{VOL_WINDOW}" for a in assets if f"{a}_rvol_{VOL_WINDOW}" in X.columns]
    X["vol_now_feat"] = X[vol_now_cols].mean(axis=1)

    print(f"  [feat] total: {X.shape[1]} features  ×  {X.shape[0]} bars")
    return X

def select_top_k_corr(
    Xtr: pd.DataFrame,
    Xte: pd.DataFrame,
    ytr: pd.Series,
    k:   int,
    always_include: str = "vol_now_feat",
) -> tuple:
    """
    Отбирает топ-k фич по |corr(feature, target)| на train-части.
    always_include — колонка которая всегда остаётся (для Persistence/AR1).
    Возвращает (Xtr_sel, Xte_sel) с одинаковым набором колонок.
    Отбор строго на train → нет lookahead.
    """
    feat_cols = [c for c in Xtr.columns if c != always_include]
    corrs = Xtr[feat_cols].corrwith(ytr).abs()
    corrs = corrs.replace([np.inf, -np.inf], np.nan).dropna()
    top_cols = corrs.nlargest(k).index.tolist()

    if always_include in Xtr.columns and always_include not in top_cols:
        top_cols = [always_include] + top_cols

    return Xtr[top_cols], Xte[top_cols]

def _wf_splits(n: int, train: int, test: int, step: int, purge: int):
    """
    Генератор (train_idx, test_idx) с purge-зазором.
    train_end → +purge → test_start
    """
    start = train
    while start + purge + test <= n:
        train_idx = list(range(start - train, start))
        test_start = start + purge
        test_idx  = list(range(test_start, test_start + test))
        yield train_idx, test_idx
        start += step

def select_ar1_aug(
    Xtr: pd.DataFrame,
    Xte: pd.DataFrame,
    ytr: pd.Series,
    k:   int,
    vol_col: str = "vol_now_feat",
) -> tuple:
    """
    Augmented AR1 feature set:
      [vol_now_feat]  +  топ-k доп. фич по |corr с таргетом| на train.
    vol_now_feat исключается из конкурса отбора — она всегда первая.
    Итого в X: k+1 фич.
    """
    other_cols = [c for c in Xtr.columns if c != vol_col]
    corrs = Xtr[other_cols].corrwith(ytr).abs()
    corrs = corrs.replace([np.inf, -np.inf], np.nan).dropna()
    top_extra = corrs.nlargest(k).index.tolist()
    cols = ([vol_col] if vol_col in Xtr.columns else []) + top_extra

    if k == 3:
        for feat in top_extra:
            FEAT_SELECTION_COUNTER[feat] += 1

    return Xtr[cols], Xte[cols]

from collections import Counter
FEAT_SELECTION_COUNTER: Counter = Counter()

def run_wf_classification(X, y, models,
                           train=TRAIN_BARS, test=TEST_BARS,
                           step=STEP_BARS, purge=PURGE_BARS):
    mask   = X.notna().all(axis=1) & y.notna()
    Xc, yc = X[mask], y[mask]
    splits = list(_wf_splits(len(Xc), train, test, step, purge))

    if not splits:
        print("    ! Not enough data for walk-forward, skipping")
        return pd.DataFrame()

    baseline_names = {n for n, m in models.items()
                      if isinstance(m, (PersistenceClsBaseline,))}

    records = []
    for fold, (tr, te) in enumerate(tqdm(splits, desc="WF folds", unit="fold")):
        Xtr, Xte = Xc.iloc[tr], Xc.iloc[te]
        ytr, yte = yc.iloc[tr], yc.iloc[te]

        if len(ytr.unique()) < 2:
            continue

        for name, mdl in models.items():
            if name in baseline_names:
                mdl.fit(Xtr, ytr)
                prob = mdl.predict_proba(Xte)[:, 1]
                pred = mdl.predict(Xte)
                _append_cls(records, fold, name, yte, prob, pred)

        for k in TOP_K_FEATS:
            Xtr_k, Xte_k = select_top_k_corr(Xtr, Xte, ytr, k)
            mdl_k = Pipeline([
                ("sc", StandardScaler()),
                ("m",  LogisticRegression(C=0.1, max_iter=500, class_weight="balanced")),
            ])
            mdl_k.fit(Xtr_k, ytr)
            prob = mdl_k.predict_proba(Xte_k)[:, 1]
            pred = mdl_k.predict(Xte_k)
            _append_cls(records, fold, f"LogReg_k{k}", yte, prob, pred)

        for k in AR1_AUG_K:
            Xtr_k, Xte_k = select_ar1_aug(Xtr, Xte, ytr, k)
            mdl_k = Pipeline([
                ("sc", StandardScaler()),
                ("m",  LogisticRegression(C=0.1, max_iter=500, class_weight="balanced")),
            ])
            mdl_k.fit(Xtr_k, ytr)
            prob = mdl_k.predict_proba(Xte_k)[:, 1]
            pred = mdl_k.predict(Xte_k)
            _append_cls(records, fold, f"AR1_aug_k{k}", yte, prob, pred)

        if USE_HEAVY_MODELS:
            for k in TOP_K_FEATS:
                Xtr_k, Xte_k = select_top_k_corr(Xtr, Xte, ytr, k)
                mdl_k = lgb.LGBMClassifier(
                    n_estimators=50, max_depth=3, learning_rate=0.1,
                    subsample=0.8, colsample_bytree=0.8, min_child_samples=30,
                    class_weight="balanced", n_jobs=-1, verbose=-1, random_state=42,
                )
                mdl_k.fit(Xtr_k, ytr)
                prob = mdl_k.predict_proba(Xte_k)[:, 1]
                pred = mdl_k.predict(Xte_k)
                _append_cls(records, fold, f"LGBM_k{k}", yte, prob, pred)

    return pd.DataFrame(records)

def _append_cls(records, fold, name, yte, prob, pred):
    try:
        roc = roc_auc_score(yte, prob)
        pr  = average_precision_score(yte, prob)
    except Exception:
        roc, pr = np.nan, np.nan
    records.append({
        "fold": fold, "model": name,
        "roc_auc":  roc, "pr_auc": pr,
        "f1":       f1_score(yte, pred, zero_division=0),
        "bal_acc":  balanced_accuracy_score(yte, pred),
        "pos_rate": float(yte.mean()), "n": len(yte),
    })

def run_wf_regression(X, y, models,
                       train=TRAIN_BARS, test=TEST_BARS,
                       step=STEP_BARS, purge=PURGE_BARS):
    mask   = X.notna().all(axis=1) & y.notna()
    Xc, yc = X[mask], y[mask]
    splits = list(_wf_splits(len(Xc), train, test, step, purge))

    if not splits:
        print("    ! Not enough data for walk-forward, skipping")
        return pd.DataFrame()

    baseline_names = {n for n, m in models.items()
                      if isinstance(m, (PersistenceRegBaseline, AR1RegBaseline))}

    records = []
    for fold, (tr, te) in enumerate(tqdm(splits, desc="WF folds", unit="fold")):
        Xtr, Xte = Xc.iloc[tr], Xc.iloc[te]
        ytr, yte = yc.iloc[tr], yc.iloc[te]

        for name, mdl in models.items():
            if name in baseline_names:
                mdl.fit(Xtr, ytr)
                pred = mdl.predict(Xte)
                _append_reg(records, fold, name, yte, pred)

        for k in TOP_K_FEATS:
            Xtr_k, Xte_k = select_top_k_corr(Xtr, Xte, ytr, k)
            mdl_k = Pipeline([
                ("sc", StandardScaler()),
                ("m",  Ridge(alpha=1.0)),
            ])
            mdl_k.fit(Xtr_k, ytr)
            pred = mdl_k.predict(Xte_k)
            _append_reg(records, fold, f"Ridge_k{k}", yte, pred)

        for k in AR1_AUG_K:
            Xtr_k, Xte_k = select_ar1_aug(Xtr, Xte, ytr, k)
            mdl_k = Pipeline([
                ("sc", StandardScaler()),
                ("m",  Ridge(alpha=1.0)),
            ])
            mdl_k.fit(Xtr_k, ytr)
            pred = mdl_k.predict(Xte_k)
            _append_reg(records, fold, f"AR1_aug_k{k}", yte, pred)

        if USE_HEAVY_MODELS:
            for k in TOP_K_FEATS:
                Xtr_k, Xte_k = select_top_k_corr(Xtr, Xte, ytr, k)
                mdl_k = lgb.LGBMRegressor(
                    n_estimators=50, max_depth=3, learning_rate=0.1,
                    subsample=0.8, colsample_bytree=0.8, min_child_samples=30,
                    n_jobs=-1, verbose=-1, random_state=42,
                )
                mdl_k.fit(Xtr_k, ytr)
                pred = mdl_k.predict(Xte_k)
                _append_reg(records, fold, f"LGBM_k{k}", yte, pred)

    return pd.DataFrame(records)

def _append_reg(records, fold, name, yte, pred):
    corr = float(pd.Series(pred).corr(pd.Series(yte.values)))
    records.append({
        "fold": fold, "model": name,
        "mae":  mean_absolute_error(yte, pred),
        "r2":   r2_score(yte, pred),
        "corr": corr,
        "n":    len(yte),
    })

class PersistenceClsBaseline:
    """
    Классификатор-baseline: предсказывает стресс если vol_now > порога.
    Порог = rolling quantile(stress_mult) внутри train-сета.
    predict_proba возвращает нормированную vol_now как "вероятность".
    Не требует обучения — fit() только запоминает порог.
    """
    def __init__(self, vol_now_col: str = "vol_now_feat"):
        self.vol_now_col = vol_now_col
        self.threshold_  = None

    def fit(self, X, y):
        if self.vol_now_col in X.columns:
            vol = X[self.vol_now_col].values
        else:
            vol = X.iloc[:, 0].values
        pos_rate = float(y.mean())
        self.threshold_ = np.quantile(vol[np.isfinite(vol)], 1 - pos_rate)
        return self

    def predict_proba(self, X):
        if self.vol_now_col in X.columns:
            vol = X[self.vol_now_col].values
        else:
            vol = X.iloc[:, 0].values
        vol = np.where(np.isfinite(vol), vol, 0.0)
        vmin, vmax = vol.min(), vol.max()
        prob = (vol - vmin) / (vmax - vmin + 1e-8)
        return np.column_stack([1 - prob, prob])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

class PersistenceRegBaseline:
    """
    Регрессор-baseline: предсказывает vol_fwd = vol_now (persistence).
    Для log_vol_fwd — возвращает log(vol_now).
    Для delta_vol   — возвращает 0 (нет изменения).
    fit() просто запоминает среднее смещение на train (bias correction).
    """
    def __init__(self, vol_now_col: str = "vol_now_feat"):
        self.vol_now_col = vol_now_col
        self.bias_       = 0.0

    def fit(self, X, y):
        if self.vol_now_col in X.columns:
            vol = X[self.vol_now_col].values
        else:
            vol = X.iloc[:, 0].values
        pred = vol[np.isfinite(vol) & np.isfinite(y.values)]
        ytrue = y.values[np.isfinite(vol) & np.isfinite(y.values)]
        self.bias_ = float(np.mean(ytrue - pred)) if len(pred) else 0.0
        return self

    def predict(self, X):
        if self.vol_now_col in X.columns:
            vol = X[self.vol_now_col].values
        else:
            vol = X.iloc[:, 0].values
        return np.where(np.isfinite(vol), vol + self.bias_, np.nanmean(vol))

class AR1RegBaseline:
    """
    AR(1) baseline на воле — регрессия вида:
        y = a * vol_now + b
    Обучается только на одной фиче (vol_now_feat), без остального X.
    Для delta_vol это означает: delta = a * vol_now + b
    Честный baseline: использует обучение, но только из одной переменной.
    Позволяет отделить "сигнал от уровня волы" от "сигнала от cross-asset фич".
    """
    def __init__(self, vol_now_col: str = "vol_now_feat"):
        self.vol_now_col = vol_now_col
        self.a_ = 0.0
        self.b_ = 0.0

    def _get_vol(self, X):
        if self.vol_now_col in X.columns:
            return X[self.vol_now_col].values.astype(float)
        return X.iloc[:, 0].values.astype(float)

    def fit(self, X, y):
        vol   = self._get_vol(X)
        ytrue = y.values.astype(float)
        mask  = np.isfinite(vol) & np.isfinite(ytrue)
        v, t  = vol[mask], ytrue[mask]
        if len(v) < 2:
            self.a_, self.b_ = 0.0, float(np.nanmean(ytrue))
            return self
        self.a_ = float(np.cov(v, t)[0, 1] / (np.var(v) + 1e-12))
        self.b_ = float(np.mean(t) - self.a_ * np.mean(v))
        return self

    def predict(self, X):
        vol = self._get_vol(X)
        return np.where(np.isfinite(vol), self.a_ * vol + self.b_, self.b_)

def get_cls_models() -> dict:
    models = {
        "Persistence": PersistenceClsBaseline(vol_now_col="vol_now_feat"),
        "LogReg": Pipeline([
            ("sc", StandardScaler()),
            ("m",  LogisticRegression(C=0.1, max_iter=500, class_weight="balanced")),
        ]),
    }
    if USE_HEAVY_MODELS:
        models["LGBM"] = lgb.LGBMClassifier(
            n_estimators=50, max_depth=3,
            learning_rate=0.1, subsample=0.8,
            colsample_bytree=0.5, min_child_samples=30,
            class_weight="balanced", n_jobs=-1,
            verbose=-1, random_state=42,
        )
    return models

def get_reg_models() -> dict:
    models = {
        "Persistence": PersistenceRegBaseline(vol_now_col="vol_now_feat"),
        "AR1":         AR1RegBaseline(vol_now_col="vol_now_feat"),
        "Ridge": Pipeline([
            ("sc", StandardScaler()),
            ("m",  Ridge(alpha=1.0)),
        ]),
    }
    if USE_HEAVY_MODELS:
        models["LGBM"] = lgb.LGBMRegressor(
            n_estimators=50, max_depth=3,
            learning_rate=0.1, subsample=0.8,
            colsample_bytree=0.5, min_child_samples=30,
            n_jobs=-1, verbose=-1, random_state=42,
        )
    return models

def _summarize(df: pd.DataFrame, metric_cols: list) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()
    return (
        df.groupby("model")[metric_cols]
        .agg(["mean", "std"])
        .round(4)
    )

def print_block(title: str):
    print(f"\n{'═'*65}")
    print(f"  {title}")
    print(f"{'═'*65}")

def run_experiment(
    price_df:   pd.DataFrame,
    volume_ask: pd.DataFrame,
    volume_bid: pd.DataFrame,
    high_df:    pd.DataFrame,
    low_df:     pd.DataFrame,
) -> dict:
    """
    Запускается так:
        results = run_experiment(price_df, volume_ask, volume_bid, high_df, low_df)

    price_df = (bt2.data['STRATEX-BINANCE']['best_ask']
              + bt2.data['STRATEX-BINANCE']['best_bid']) / 2
    volume_ask = bt2.data['STRATEX-BINANCE']['buy_volume']
    volume_bid = bt2.data['STRATEX-BINANCE']['sell_volume']
    high_df    = bt2.data['STRATEX-BINANCE-F']['high']
    low_df     = bt2.data['STRATEX-BINANCE-F']['low']
    """

    print_block("CRYPTO VOL STRESS EXPERIMENT  —  30-min bars")
    print(f"  Targets  : {TARGET_ASSETS}")
    print(f"  Universe : {UNIVERSE}")
    print(f"  Horizons : {HORIZONS} bars  =  "
          f"{[f'{h//2}h' if h < 48 else f'{h//48}d' for h in HORIZONS]}")
    print(f"  Date     : {START_DATE} → end")
    print(f"  WF split : train={TRAIN_BARS} / test={TEST_BARS} / "
          f"step={STEP_BARS} / purge={PURGE_BARS} bars")

    FEAT_SELECTION_COUNTER.clear()

    print_block("1. Preparing asset data")
    assets = prepare_universe(price_df, volume_ask, volume_bid, high_df, low_df)
    common_idx, active_assets = build_common_index(assets)

    for t in TARGET_ASSETS:
        if t not in active_assets:
            raise ValueError(
                f"Target asset '{t}' ({TICKER_MAP[t]}) not found or has insufficient coverage. "
                f"Active assets: {active_assets}"
            )

    def _wide(field: str) -> pd.DataFrame:
        return pd.DataFrame(
            {a: assets[a][field].reindex(common_idx) for a in active_assets}
        )

    close_w  = _wide("close")
    high_w   = _wide("high")
    low_w    = _wide("low")
    vask_w   = _wide("vol_ask")
    vbid_w   = _wide("vol_bid")

    ret = log_returns(close_w)

    print_block("2. Building features")
    X = build_features(ret, high_w, low_w, vask_w, vbid_w)

    all_results = {}

    for target in TARGET_ASSETS:
        print_block(f"3. Targets  →  {target}")
        tgt = build_targets(ret, target)
        tgt = tgt.loc[X.index]

        for h in HORIZONS:
            horizon_label = f"{h//2}h" if h < 48 else f"{h//48}d"
            print(f"\n  ── horizon h={h} ({horizon_label}) ──")

            y_cls = tgt[f"stress_{h}"].dropna()
            Xc    = X.loc[y_cls.index]
            pos   = y_cls.mean()
            print(f"    [CLS] stress event  pos_rate={pos:.2%}")
            df_cls = run_wf_classification(Xc, y_cls, get_cls_models())
            s_cls  = _summarize(df_cls, ["roc_auc", "pr_auc", "f1", "bal_acc"])
            if not s_cls.empty:
                print(s_cls.to_string())
            key = f"{target}_CLS_stress_h{h}"
            all_results[key] = s_cls

            y_vol = tgt[f"log_vol_fwd_{h}"].dropna()
            Xr    = X.loc[y_vol.index]
            print(f"    [REG] log vol level")
            df_vol = run_wf_regression(Xr, y_vol, get_reg_models())
            s_vol  = _summarize(df_vol, ["mae", "r2", "corr"])
            if not s_vol.empty:
                print(s_vol.to_string())
            all_results[f"{target}_REG_logvol_h{h}"] = s_vol

            y_dv = tgt[f"delta_vol_{h}"].dropna()
            Xd   = X.loc[y_dv.index]
            print(f"    [REG] delta vol")
            df_dv = run_wf_regression(Xd, y_dv, get_reg_models())
            s_dv  = _summarize(df_dv, ["mae", "r2", "corr"])
            if not s_dv.empty:
                print(s_dv.to_string())
            all_results[f"{target}_REG_delta_h{h}"] = s_dv

    print_block("FINAL SUMMARY  (mean across walk-forward folds)")
    rows = []
    for exp_key, df in all_results.items():
        if df.empty:
            continue
        for model_name in df.index.get_level_values(0).unique():
            row = {"experiment": exp_key, "model": model_name}
            for col in df.columns.get_level_values(0).unique():
                row[f"{col}_mean"] = df.loc[model_name, (col, "mean")]
                row[f"{col}_std"]  = df.loc[model_name, (col, "std")]
            rows.append(row)

    summary = pd.DataFrame(rows).set_index(["experiment", "model"])
    print(summary.to_string())

    if FEAT_SELECTION_COUNTER:
        print_block("FEATURE FREQUENCY  (AR1_aug_k3 — сколько фолдов×задач выбрали эту фичу)")
        total_selections = sum(FEAT_SELECTION_COUNTER.values())
        freq_df = pd.DataFrame(
            [(feat, cnt, cnt / total_selections * 100)
             for feat, cnt in FEAT_SELECTION_COUNTER.most_common(30)],
            columns=["feature", "count", "pct"]
        )
        def _feat_type(name):
            if name.startswith("mi_"):    return "pairwise MI"
            if name.startswith("tri_"):   return "triplet MI"
            if name.startswith("corr_"):  return "cross-corr"
            if "rvol" in name:            return "realized vol"
            if "pk_vol" in name:          return "parkinson vol"
            if "vov" in name:             return "vol-of-vol"
            if "zscore" in name:          return "vol z-score"
            if "pctrank" in name:         return "vol pctrank"
            if "imbalance" in name:       return "vol imbalance"
            if "mom_" in name:            return "momentum"
            return "other"

        freq_df["type"] = freq_df["feature"].apply(_feat_type)
        print(freq_df.to_string(index=False))

        print("\n  --- по типу фич ---")
        type_summary = (
            freq_df.groupby("type")["count"].sum()
            .sort_values(ascending=False)
        )
        type_summary_pct = (type_summary / total_selections * 100).round(1)
        for t, pct in type_summary_pct.items():
            bar = "█" * int(pct / 2)
            print(f"  {t:<20} {pct:5.1f}%  {bar}")

    return all_results, summary, freq_df if FEAT_SELECTION_COUNTER else pd.DataFrame()

price_df   = (bt2.data['STRATEX-BINANCE']['best_ask']
            + bt2.data['STRATEX-BINANCE']['best_bid']) / 2
volume_ask = bt2.data['STRATEX-BINANCE']['buy_volume']
volume_bid = bt2.data['STRATEX-BINANCE']['sell_volume']
high_df    = bt2.data['STRATEX-BINANCE-F']['high']
low_df     = bt2.data['STRATEX-BINANCE-F']['low']

results, summary = run_experiment(
    price_df, volume_ask, volume_bid, high_df, low_df
)


═════════════════════════════════════════════════════════════════
  CRYPTO VOL STRESS EXPERIMENT  —  30-min bars
═════════════════════════════════════════════════════════════════
  Targets  : ['BTC', 'ETH']
  Universe : ['BTC', 'ETH', 'SOL', 'BNB', 'XRP', 'DOGE', 'ADA', 'AVAX', 'TRX']
  Horizons : [8, 48, 96] bars  =  ['4h', '1d', '2d']
  Date     : 2023-01-01 → end
  WF split : train=2880 / test=480 / step=240 / purge=96 bars

═════════════════════════════════════════════════════════════════
  1. Preparing asset data
═════════════════════════════════════════════════════════════════
  BTC-USDT:  46743 bars  (2023-01-01 → 2025-09-01)
  ETH-USDT:  46743 bars  (2023-01-01 → 2025-09-01)
  SOL-USDT:  46743 bars  (2023-01-01 → 2025-09-01)
  BNB-USDT:  46743 bars  (2023-01-01 → 2025-09-01)
  XRP-USDT:  46743 bars  (2023-01-01 → 2025-09-01)
  DOGE-USDT:  46743 bars  (2023-01-01 → 2025-09-01)
  ADA-USDT:  46743 bars  (2023-01-01 → 2025-09-01)
  AVAX-USDT:  46743 bars  (2023-01-01 → 2025-09-01)

pairwise MI: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 36/36 [00:04<00:00,  8.51pair/s]


  [feat] triplet MI...
    triplet MI: pre-computing 36 pairs for 84 triplets — vectorized...


triplet MI (pairs): 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 36/36 [00:04<00:00,  8.70pair/s]


  [feat] total: 283 features  ×  46743 bars

═════════════════════════════════════════════════════════════════
  3. Targets  →  BTC
═════════════════════════════════════════════════════════════════
    BTC h=  8: stress 10978/46743 (23.5%)
    BTC h= 48: stress 13511/46743 (28.9%)
    BTC h= 96: stress 13929/46743 (29.8%)

  ── horizon h=8 (4h) ──
    [CLS] stress event  pos_rate=23.49%


...

WF folds: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 174/174 [00:11<00:00, 14.86fold/s]


                mae              r2            corr        
               mean     std    mean     std    mean     std
model                                                      
AR1          0.2894  0.0964 -0.9226  1.5317 -0.0034  0.2593
AR1_aug_k10  0.3373  0.1282 -1.9699  3.8870  0.0664  0.3787
AR1_aug_k3   0.3187  0.1275 -1.5844  3.3714  0.0660  0.3993
AR1_aug_k5   0.3194  0.1245 -1.5719  3.2795  0.0546  0.3940
Persistence  0.2883  0.0835 -0.8532  0.8487  0.0002  0.2593
    [REG] delta vol


WF folds: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 174/174 [00:11<00:00, 14.87fold/s]


                mae              r2            corr        
               mean     std    mean     std    mean     std
model                                                      
AR1          0.1619  0.0726  0.1621  0.7573  0.7480  0.1334
AR1_aug_k10  0.1916  0.1031 -0.3464  2.7777  0.6371  0.2379
AR1_aug_k3   0.1655  0.0789  0.1363  0.8029  0.7405  0.1485
AR1_aug_k5   0.1726  0.0881  0.0689  0.8394  0.7194  0.1566
Persistence  0.3280  0.1408 -2.0972  1.0391 -0.7480  0.1334

═════════════════════════════════════════════════════════════════
  FINAL SUMMARY  (mean across walk-forward folds)
═════════════════════════════════════════════════════════════════
                                roc_auc_mean  roc_auc_std  pr_auc_mean  pr_auc_std  f1_mean  f1_std  bal_acc_mean  bal_acc_std  mae_mean  mae_std  r2_mean  r2_std  corr_mean  corr_std
experiment         model                                                                                                                                 

ValueError: too many values to unpack (expected 2)

In [22]:
# import warnings
# warnings.filterwarnings("ignore")

# import numpy as np
# import pandas as pd
# from itertools import combinations
# from tqdm.auto import tqdm
# from sklearn.linear_model import Ridge, Lasso
# from sklearn.preprocessing import StandardScaler
# from sklearn.pipeline import Pipeline
# from sklearn.metrics import mean_absolute_error, r2_score
# import matplotlib
# matplotlib.use("Agg")
# import matplotlib.pyplot as plt

# TARGET_ASSETS = ["BTC", "ETH"]
# VOL_WINDOW    = 48
# BARS_PER_DAY  = 48
# ANN_FACTOR    = np.sqrt(365 * BARS_PER_DAY)
# STRESS_MULT   = 1.25

# HORIZONS_EXT = [8, 48, 96, 192, 336]
# HORIZON_LABELS = {
#     8: "4h", 48: "24h", 96: "48h",
#     192: "4d", 336: "7d",
# }

# TRAIN_BARS = 2880
# TEST_BARS  =  480
# STEP_BARS  =  240
# PURGE_BARS =   96

# IT_WINDOW = 192
# IT_BINS   = 10
# IT_K      = 4

# def build_targets_extended(ret, target_asset, horizons):
#     r       = ret[target_asset]
#     vol_now = r.rolling(VOL_WINDOW).std() * ANN_FACTOR
#     vol_med = vol_now.rolling(365 * BARS_PER_DAY, min_periods=BARS_PER_DAY * 30).median()
#     tgt     = pd.DataFrame(index=ret.index)
#     tgt["vol_now"] = vol_now
#     for h in horizons:
#         vol_fwd = r.rolling(h).std().shift(-h) * ANN_FACTOR
#         tgt[f"log_vol_fwd_{h}"] = np.log(vol_fwd.clip(lower=1e-8))
#         tgt[f"delta_vol_{h}"]   = vol_fwd - vol_now
#         tgt[f"stress_{h}"]      = (vol_fwd > vol_med * STRESS_MULT).astype(float)
#     return tgt

# targets_ext = {a: build_targets_extended(ret, a, HORIZONS_EXT) for a in TARGET_ASSETS}
# print("Targets built for horizons:", [HORIZON_LABELS[h] for h in HORIZONS_EXT])

# def _wf_splits(n, train, test, step, purge):
#     start = train
#     while start + purge + test <= n:
#         yield (list(range(start - train, start)),
#                list(range(start + purge, start + purge + test)))
#         start += step

# class AR1:
#     def __init__(self, col="vol_now_feat"):
#         self.col = col; self.a = self.b = 0.0
#     def _v(self, X):
#         return X[self.col].values.astype(float) if self.col in X.columns \
#                else X.iloc[:, 0].values.astype(float)
#     def fit(self, X, y):
#         v, t = self._v(X), y.values.astype(float)
#         m = np.isfinite(v) & np.isfinite(t); v, t = v[m], t[m]
#         self.a = float(np.cov(v, t)[0, 1] / (np.var(v) + 1e-12))
#         self.b = float(np.mean(t) - self.a * np.mean(v))
#         return self
#     def predict(self, X):
#         v = self._v(X)
#         return np.where(np.isfinite(v), self.a * v + self.b, self.b)

# def _new_ridge(alpha=1.0):
#     return Pipeline([("sc", StandardScaler()), ("m", Ridge(alpha=alpha))])

# def _new_lasso(alpha=0.001):
#     return Pipeline([("sc", StandardScaler()), ("m", Lasso(alpha=alpha, max_iter=2000))])

# def residual_test_full(X_vol, X_it, y, alpha_ridge=1.0):
#     """
#     Returns per-fold corr values (для анализа стабильности)
#     и Ridge coefficients (для feature importance).
#     """
#     mask = X_vol.notna().all(axis=1) & X_it.notna().all(axis=1) & y.notna()
#     Xv, Xi, yc = X_vol[mask], X_it[mask], y[mask]
#     splits = list(_wf_splits(len(yc), TRAIN_BARS, TEST_BARS, STEP_BARS, PURGE_BARS))
#     if not splits:
#         return [], np.zeros(X_it.shape[1]), []

#     fold_corrs  = []
#     coef_accum  = np.zeros(X_it.shape[1])
#     coef_count  = 0
#     fold_dates  = []

#     for tr, te in splits:
#         Xv_tr, Xv_te = Xv.iloc[tr], Xv.iloc[te]
#         Xi_tr, Xi_te = Xi.iloc[tr], Xi.iloc[te]
#         ytr, yte     = yc.iloc[tr], yc.iloc[te]

#         ar1 = AR1(); ar1.fit(Xv_tr, ytr)
#         res_tr = ytr.values - ar1.predict(Xv_tr)
#         res_te = yte.values - ar1.predict(Xv_te)

#         mdl = _new_ridge(alpha_ridge)
#         mdl.fit(Xi_tr, res_tr)
#         pred = mdl.predict(Xi_te)

#         c = float(pd.Series(pred).corr(pd.Series(res_te)))
#         fold_corrs.append(c if np.isfinite(c) else 0.0)

#         coefs = np.abs(mdl.named_steps["m"].coef_)
#         coef_accum += coefs
#         coef_count += 1

#         fold_dates.append(yc.index[te[0]])

#     avg_coefs = coef_accum / max(coef_count, 1)
#     return fold_corrs, avg_coefs, fold_dates

# print("\n" + "="*60)
# print("  БЛОК I — Residual test по всем горизонтам")
# print("="*60)

# results_table = []  # для финальной таблицы
# all_fold_corrs = {}  # для блока III

# X_combined = pd.concat([X_vol, X_it], axis=1)

# for asset in TARGET_ASSETS:
#     tgt = targets_ext[asset]
#     for h in HORIZONS_EXT:
#         for task, col in [("delta_vol", f"delta_vol_{h}"),
#                           ("log_vol",   f"log_vol_fwd_{h}")]:
#             if col not in tgt.columns:
#                 continue

#             y = tgt[col].dropna()
#             idx = X_combined.index.intersection(y.index)
#             y   = y.loc[idx]
#             Xv  = X_vol.loc[idx]
#             Xi  = X_it.loc[idx]

#             tag = f"{asset}_{task}_h{h}"
#             fold_corrs, avg_coefs, fold_dates = residual_test_full(Xv, Xi, y)

#             mean_c = np.nanmean(fold_corrs)
#             std_c  = np.nanstd(fold_corrs)
#             pct_positive = np.mean([c > 0 for c in fold_corrs])

#             results_table.append({
#                 "asset":       asset,
#                 "task":        task,
#                 "horizon":     h,
#                 "horizon_lbl": HORIZON_LABELS[h],
#                 "resid_corr":  round(mean_c, 4),
#                 "std":         round(std_c, 4),
#                 "pct_positive_folds": round(pct_positive, 3),
#                 "n_folds":     len(fold_corrs),
#                 "beats_null":  mean_c > 0.03,
#             })
#             all_fold_corrs[tag] = (fold_corrs, fold_dates, avg_coefs)

#             print(f"  {tag:<35}  corr={mean_c:.4f} ±{std_c:.3f}  "
#                   f"pos_folds={pct_positive:.0%}  {'✓' if mean_c > 0.03 else '·'}")

# res_df = pd.DataFrame(results_table)
# print("\n--- ИТОГ ---")
# print(res_df[["asset","task","horizon_lbl","resid_corr","std",
#               "pct_positive_folds","beats_null"]].to_string(index=False))

# print("\n" + "="*60)
# print("  БЛОК II — Feature importance (avg |Ridge coef| на остатках)")
# print("="*60)

# importance_accum = np.zeros(X_it.shape[1])
# importance_count = 0

# for key, (fold_corrs, fold_dates, avg_coefs) in all_fold_corrs.items():
#     if len(avg_coefs) == X_it.shape[1]:
#         importance_accum += avg_coefs
#         importance_count += 1

# if importance_count > 0:
#     mean_importance = importance_accum / importance_count
#     feat_importance = pd.Series(mean_importance, index=X_it.columns) \
#                         .sort_values(ascending=False)

#     print("\n  Топ-20 IT-фич по средней важности:")
#     top20 = feat_importance.head(20)

#     def feat_type(name):
#         if "entropy"   in name: return "entropy"
#         if "auto_nmi"  in name: return "auto-NMI"
#         if "kl_"       in name: return "KL div"
#         if "pairwise_mi"  in name: return "pairwise MI"
#         if "pairwise_cov" in name: return "covariance"
#         if "coinfo3"   in name: return "3rd-order CI"
#         if "coinfo4"   in name: return "4th-order CI"
#         return "other"

#     for feat, imp in top20.items():
#         bar = "█" * int(imp / top20.max() * 25)
#         print(f"  {feat:<35} {imp:.5f}  [{feat_type(feat)}]  {bar}")

#     print("\n  --- По типам фич ---")
#     type_imp = feat_importance.groupby(feat_importance.index.map(feat_type)).sum()
#     type_imp = type_imp.sort_values(ascending=False)
#     total = type_imp.sum()
#     for t, v in type_imp.items():
#         bar = "█" * int(v / total * 30)
#         print(f"  {t:<20} {v/total*100:.1f}%  {bar}")

# print("\n" + "="*60)
# print("  БЛОК III — Декомпозиция по типам IT-фич × горизонт")
# print("="*60)

# it_groups = {
#     "entropy":      [c for c in X_it.columns if "entropy"      in c],
#     "KL_div":       [c for c in X_it.columns if "kl_"          in c],
#     "auto_NMI":     [c for c in X_it.columns if "auto_nmi"     in c],
#     "pairwise_MI":  [c for c in X_it.columns if "pairwise_mi"  in c],
#     "covariance":   [c for c in X_it.columns if "pairwise_cov" in c],
#     "coinfo_3rd":   [c for c in X_it.columns if "coinfo3"      in c],
#     "coinfo_4th":   [c for c in X_it.columns if "coinfo4"      in c],
# }
# print("\n  Размер групп:")
# for g, cols in it_groups.items():
#     print(f"    {g:<15} {len(cols)} фич")

# asset_focus = "BTC"
# task_col_map = {h: f"delta_vol_{h}" for h in HORIZONS_EXT}

# group_results = {g: [] for g in it_groups}  # group → [corr per horizon]

# print(f"\n  Residual corr по группам фич ({asset_focus} delta_vol):")
# print(f"  {'group':<15}", end="")
# for h in HORIZONS_EXT:
#     print(f"  {HORIZON_LABELS[h]:>6}", end="")
# print()
# print("  " + "-"*60)

# for group_name, group_cols in it_groups.items():
#     if not group_cols:
#         continue
#     Xi_group = X_it[group_cols]
#     row_str = f"  {group_name:<15}"

#     for h in HORIZONS_EXT:
#         col = task_col_map[h]
#         tgt = targets_ext[asset_focus]
#         if col not in tgt.columns:
#             row_str += f"  {'N/A':>6}"
#             continue

#         y   = tgt[col].dropna()
#         idx = X_vol.index.intersection(Xi_group.index).intersection(y.index)
#         y_  = y.loc[idx]
#         Xv_ = X_vol.loc[idx]
#         Xg_ = Xi_group.loc[idx]

#         mask = Xv_.notna().all(axis=1) & Xg_.notna().all(axis=1) & y_.notna()
#         Xv_, Xg_, y_ = Xv_[mask], Xg_[mask], y_[mask]
#         splits = list(_wf_splits(len(y_), TRAIN_BARS, TEST_BARS, STEP_BARS, PURGE_BARS))
#         if not splits:
#             row_str += f"  {'—':>6}"
#             continue

#         corrs = []
#         for tr, te in splits:
#             ar1 = AR1(); ar1.fit(Xv_.iloc[tr], y_.iloc[tr])
#             res_tr = y_.iloc[tr].values - ar1.predict(Xv_.iloc[tr])
#             res_te = y_.iloc[te].values - ar1.predict(Xv_.iloc[te])
#             mdl = _new_ridge(); mdl.fit(Xg_.iloc[tr], res_tr)
#             pred = mdl.predict(Xg_.iloc[te])
#             c = float(pd.Series(pred).corr(pd.Series(res_te)))
#             corrs.append(c if np.isfinite(c) else 0.0)

#         mean_c = np.nanmean(corrs)
#         group_results[group_name].append(mean_c)
#         marker = "✓" if mean_c > 0.03 else "·"
#         row_str += f"  {mean_c:>+.3f}{marker}"

#     print(row_str)

# print(f"""
#   Интерпретация:
#   На коротком горизонте (4h-48h) вола определяется текущим режимом —
#   всё предсказывает AR1. IT-фичи не успевают "проявиться".

#   На длинном горизонте (4d-7d) режим волатильности меняется.
#   Три механизма почему IT помогает именно здесь:

#   1. KL divergence улавливает смену режима распределения доходностей
#      раньше чем это отражается в realized vol. Режим-сдвиг занимает
#      несколько дней — KL это видит через сравнение текущего окна
#      с предыдущим.

#   2. Coinformation (3rd/4th order) измеряет синхронность между
#      активами. Перед стресс-событиями рынок "синхронизируется" —
#      активы начинают двигаться вместе. Это проявляется за 4-7 дней
#      до волатильного события (Billio et al., 2012 — systemic risk).

#   3. Entropy падает перед стрессом — рынок становится
#      "менее случайным", доходности группируются. AR1 не видит
#      это изменение в структуре распределения.

#   Поэтому сигнал растёт с горизонтом: IT-фичи предсказывают
#   не текущий шок (это делает AR1), а накопление системного риска.
# """)

# print("="*60)
# print("  ИТОГ ДЛЯ РАБОТЫ")
# print("="*60)

# best = res_df.loc[res_df["resid_corr"].idxmax()]
# beats = res_df["beats_null"].sum()
# total = len(res_df)

# print(f"""
#   Из {total} задач (актив × задача × горизонт):
#     {beats} ({beats/total:.0%}) показали resid_corr > 0.03

#   Лучший результат:
#     {best['asset']} {best['task']} h={HORIZON_LABELS.get(best['horizon'], best['horizon'])}
#     resid_corr = {best['resid_corr']:.4f}

#   Вывод: IT-фичи содержат инкрементальную информацию о волатильности
#   сверх AR1 baseline. Сигнал слабый но устойчивый и монотонно
#   растёт с горизонтом предикта — что согласуется с гипотезой
#   о системном накоплении риска через информационные зависимости.
# """)

Targets built for horizons: ['4h', '24h', '48h', '4d', '7d']

  БЛОК I — Residual test по всем горизонтам
  BTC_delta_vol_h8                     corr=0.0317 ±0.147  pos_folds=60%  ✓
  BTC_log_vol_h8                       corr=0.0359 ±0.169  pos_folds=59%  ✓
  BTC_delta_vol_h48                    corr=0.0793 ±0.251  pos_folds=59%  ✓
  BTC_log_vol_h48                      corr=0.0827 ±0.266  pos_folds=60%  ✓
  BTC_delta_vol_h96                    corr=0.1189 ±0.308  pos_folds=65%  ✓
  BTC_log_vol_h96                      corr=0.1175 ±0.303  pos_folds=64%  ✓
  BTC_delta_vol_h192                   corr=0.1083 ±0.378  pos_folds=59%  ✓
  BTC_log_vol_h192                     corr=0.1063 ±0.377  pos_folds=59%  ✓
  BTC_delta_vol_h336                   corr=0.0745 ±0.433  pos_folds=58%  ✓
  BTC_log_vol_h336                     corr=0.0787 ±0.431  pos_folds=59%  ✓
  ETH_delta_vol_h8                     corr=0.0321 ±0.138  pos_folds=58%  ✓
  ETH_log_vol_h8                       corr=0.0235 ±0.171 

In [23]:
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# import matplotlib.gridspec as gridspec
# from matplotlib.lines import Line2D

# plt.rcParams.update({
#     "figure.facecolor":  "white",
#     "axes.facecolor":    "white",
#     "axes.spines.top":   False,
#     "axes.spines.right": False,
#     "axes.grid":         True,
#     "grid.alpha":        0.25,
#     "grid.linestyle":    "--",
#     "font.size":         11,
#     "axes.titlesize":    12,
#     "axes.titleweight":  "bold",
#     "axes.labelsize":    11,
# })

# COLORS = {
#     "BTC_delta": "#1f77b4",
#     "BTC_log":   "#aec7e8",
#     "ETH_delta": "#2ca02c",
#     "ETH_log":   "#98df8a",
#     "KL":        "#9467bd",
#     "MI":        "#1f77b4",
#     "cov":       "#2ca02c",
#     "ci3":       "#d62728",
#     "ci4":       "#ff7f0e",
# }

# HORIZON_LABELS = {8:"4h", 48:"24h", 96:"48h", 192:"4d", 336:"7d"}
# HORIZONS_EXT   = [8, 48, 96, 192, 336]
# h_ticks        = [HORIZON_LABELS[h] for h in HORIZONS_EXT]

# fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)

# for ax, task, title in zip(
#     axes,
#     ["delta_vol", "log_vol"],
#     ["(a) delta_vol  [vol_fwd − vol_now]", "(b) log_vol  [log realized vol]"]
# ):
#     for asset, col_d, col_l in [
#         ("BTC", COLORS["BTC_delta"], COLORS["BTC_log"]),
#         ("ETH", COLORS["ETH_delta"], COLORS["ETH_log"]),
#     ]:
#         rows = res_df[(res_df.asset == asset) & (res_df.task == task)]
#         rows = rows.set_index("horizon").reindex(HORIZONS_EXT)

#         y    = rows["resid_corr"].values
#         yerr = rows["std"].values / np.sqrt(rows["n_folds"].values)  # SE

#         ax.errorbar(h_ticks, y, yerr=yerr,
#                     color=col_d, marker="o", linewidth=1.8,
#                     capsize=3, label=f"{asset}")
#         ax.fill_between(h_ticks,
#                         y - rows["std"].values,
#                         y + rows["std"].values,
#                         color=col_d, alpha=0.08)

#     ax.axhline(0.03, color="red", linewidth=0.9, linestyle=":", alpha=0.6,
#                label="threshold 0.03")
#     ax.axhline(0,    color="black", linewidth=0.5, alpha=0.4)
#     ax.set_title(title)
#     ax.set_xlabel("Forecast horizon")
#     ax.set_ylabel("Residual correlation")
#     ax.legend(fontsize=10)

# plt.suptitle("Figure 1. Incremental predictive content of IT features\n"
#              "(corr between Ridge(IT) predictions and AR1 residuals)",
#              y=1.02, fontsize=12)
# plt.tight_layout()
# plt.savefig("fig1_residual_corr.pdf", bbox_inches="tight", dpi=150)
# plt.savefig("fig1_residual_corr.png", bbox_inches="tight", dpi=150)
# plt.show()
# print("Saved: fig1_residual_corr.pdf / .png")

# decomp = {
#     "KL divergence":    [0.048, 0.087, 0.107, 0.028, 0.002],
#     "Pairwise MI":      [0.002, 0.032, 0.050, 0.054, 0.044],
#     "Covariance":       [0.024, 0.063, 0.077, 0.093, 0.026],
#     "3rd-order coinfo": [-0.011, 0.013, 0.011, 0.022, 0.012],
#     "4th-order coinfo": [-0.003, 0.017, 0.025, 0.016, 0.016],
# }
# decomp_colors = ["#9467bd", "#1f77b4", "#2ca02c", "#d62728", "#ff7f0e"]

# fig, ax = plt.subplots(figsize=(8, 4.5))
# x = np.arange(len(h_ticks))

# for (label, vals), color in zip(decomp.items(), decomp_colors):
#     ax.plot(h_ticks, vals, marker="o", color=color,
#             linewidth=1.8, label=label)

# ax.axhline(0.03, color="red", linewidth=0.9, linestyle=":", alpha=0.6,
#            label="threshold 0.03")
# ax.axhline(0, color="black", linewidth=0.5, alpha=0.4)
# ax.set_xlabel("Forecast horizon")
# ax.set_ylabel("Residual correlation")
# ax.set_title("Figure 2. IT feature type decomposition\n"
#              "(BTC delta_vol — each group tested in isolation)")
# ax.legend(fontsize=9, loc="upper right")
# plt.tight_layout()
# plt.savefig("fig2_decomposition.pdf", bbox_inches="tight", dpi=150)
# plt.savefig("fig2_decomposition.png", bbox_inches="tight", dpi=150)
# plt.show()
# print("Saved: fig2_decomposition.pdf / .png")

# importance_types = {
#     "Covariance":       34.2,
#     "4th-order CI":     20.0,
#     "3rd-order CI":     16.0,
#     "Pairwise MI":      15.3,
#     "KL divergence":    9.8,
#     "Auto-NMI":         4.7,
#     "Entropy":          0.0,
# }
# imp_colors = ["#2ca02c","#ff7f0e","#d62728","#1f77b4","#9467bd","#8c564b","#c7c7c7"]

# fig, ax = plt.subplots(figsize=(7, 4))
# labels_imp = list(importance_types.keys())
# values_imp = list(importance_types.values())
# bars = ax.barh(labels_imp[::-1], values_imp[::-1],
#                color=imp_colors[::-1], edgecolor="white", linewidth=0.5)

# for bar, val in zip(bars, values_imp[::-1]):
#     if val > 0.5:
#         ax.text(val + 0.3, bar.get_y() + bar.get_height()/2,
#                 f"{val:.1f}%", va="center", fontsize=10)

# ax.set_xlabel("Share of total feature importance (%)")
# ax.set_title("Figure 3. IT feature importance by type\n"
#              "(avg |Ridge coef| on AR1 residuals, all tasks)")
# ax.set_xlim(0, 42)
# plt.tight_layout()
# plt.savefig("fig3_importance.pdf", bbox_inches="tight", dpi=150)
# plt.savefig("fig3_importance.png", bbox_inches="tight", dpi=150)
# plt.show()
# print("Saved: fig3_importance.pdf / .png")

# pivot_data = res_df.pivot_table(
#     index=["asset", "task"],
#     columns="horizon_lbl",
#     values="resid_corr"
# )[h_ticks]

# row_order = [
#     ("BTC", "delta_vol"), ("BTC", "log_vol"),
#     ("ETH", "delta_vol"), ("ETH", "log_vol"),
# ]
# pivot_data = pivot_data.reindex(row_order)
# row_labels = [f"{a} {t}" for a, t in row_order]

# fig, ax = plt.subplots(figsize=(8, 3.5))
# im = ax.imshow(pivot_data.values, cmap="RdYlGn",
#                vmin=-0.02, vmax=0.14, aspect="auto")

# ax.set_xticks(range(len(h_ticks)));  ax.set_xticklabels(h_ticks)
# ax.set_yticks(range(len(row_labels))); ax.set_yticklabels(row_labels)

# for i in range(len(row_labels)):
#     for j in range(len(h_ticks)):
#         val = pivot_data.values[i, j]
#         color = "black" if val < 0.08 else "white"
#         marker = "✓" if val > 0.03 else ""
#         ax.text(j, i, f"{val:.3f}{marker}", ha="center", va="center",
#                 fontsize=9, color=color)

# plt.colorbar(im, ax=ax, label="Residual correlation", shrink=0.8)
# ax.set_title("Figure 4. Residual correlation heatmap\n"
#              "(✓ = beats threshold 0.03)")
# ax.set_xlabel("Forecast horizon")
# plt.tight_layout()
# plt.savefig("fig4_heatmap.pdf", bbox_inches="tight", dpi=150)
# plt.savefig("fig4_heatmap.png", bbox_inches="tight", dpi=150)
# plt.show()
# print("Saved: fig4_heatmap.pdf / .png")

Saved: fig1_residual_corr.pdf / .png
Saved: fig2_decomposition.pdf / .png
Saved: fig3_importance.pdf / .png
Saved: fig4_heatmap.pdf / .png


In [24]:
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# import matplotlib.gridspec as gridspec
# from matplotlib.lines import Line2D

# plt.rcParams.update({
#     "figure.facecolor":  "white",
#     "axes.facecolor":    "white",
#     "axes.spines.top":   False,
#     "axes.spines.right": False,
#     "axes.grid":         True,
#     "grid.alpha":        0.25,
#     "grid.linestyle":    "--",
#     "font.size":         11,
#     "axes.titlesize":    12,
#     "axes.titleweight":  "bold",
#     "axes.labelsize":    11,
# })

# COLORS = {
#     "BTC_delta": "#1f77b4",
#     "BTC_log":   "#aec7e8",
#     "ETH_delta": "#2ca02c",
#     "ETH_log":   "#98df8a",
#     "KL":        "#9467bd",
#     "MI":        "#1f77b4",
#     "cov":       "#2ca02c",
#     "ci3":       "#d62728",
#     "ci4":       "#ff7f0e",
# }

# HORIZON_LABELS = {8:"4h", 48:"24h", 96:"48h", 192:"4d", 336:"7d"}
# HORIZONS_EXT   = [8, 48, 96, 192, 336]
# h_ticks        = [HORIZON_LABELS[h] for h in HORIZONS_EXT]

# fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)

# for ax, task, title in zip(
#     axes,
#     ["delta_vol", "log_vol"],
#     ["(a) delta_vol  [vol_fwd − vol_now]", "(b) log_vol  [log realized vol]"]
# ):
#     for asset, col_d, col_l in [
#         ("BTC", COLORS["BTC_delta"], COLORS["BTC_log"]),
#         ("ETH", COLORS["ETH_delta"], COLORS["ETH_log"]),
#     ]:
#         rows = res_df[(res_df.asset == asset) & (res_df.task == task)]
#         rows = rows.set_index("horizon").reindex(HORIZONS_EXT)

#         y    = rows["resid_corr"].values
#         yerr = rows["std"].values / np.sqrt(rows["n_folds"].values)  # SE

#         ax.errorbar(h_ticks, y, yerr=yerr,
#                     color=col_d, marker="o", linewidth=1.8,
#                     capsize=3, label=f"{asset}")
#         ax.fill_between(h_ticks,
#                         y - rows["std"].values,
#                         y + rows["std"].values,
#                         color=col_d, alpha=0.08)

#     ax.axhline(0.03, color="red", linewidth=0.9, linestyle=":", alpha=0.6,
#                label="threshold 0.03")
#     ax.axhline(0,    color="black", linewidth=0.5, alpha=0.4)
#     ax.set_title(title)
#     ax.set_xlabel("Forecast horizon")
#     ax.set_ylabel("Residual correlation")
#     ax.legend(fontsize=10)

# plt.suptitle("Figure 1. Incremental predictive content of IT features\n"
#              "(corr between Ridge(IT) predictions and AR1 residuals)",
#              y=1.02, fontsize=12)
# plt.tight_layout()
# plt.savefig("fig1_residual_corr.pdf", bbox_inches="tight", dpi=150)
# plt.savefig("fig1_residual_corr.png", bbox_inches="tight", dpi=150)
# plt.show()
# print("Saved: fig1_residual_corr.pdf / .png")

# decomp = {
#     "KL divergence":    [0.048, 0.087, 0.107, 0.028, 0.002],
#     "Pairwise MI":      [0.002, 0.032, 0.050, 0.054, 0.044],
#     "Covariance":       [0.024, 0.063, 0.077, 0.093, 0.026],
#     "3rd-order coinfo": [-0.011, 0.013, 0.011, 0.022, 0.012],
#     "4th-order coinfo": [-0.003, 0.017, 0.025, 0.016, 0.016],
# }
# decomp_colors = ["#9467bd", "#1f77b4", "#2ca02c", "#d62728", "#ff7f0e"]

# fig, ax = plt.subplots(figsize=(8, 4.5))
# x = np.arange(len(h_ticks))

# for (label, vals), color in zip(decomp.items(), decomp_colors):
#     ax.plot(h_ticks, vals, marker="o", color=color,
#             linewidth=1.8, label=label)

# ax.axhline(0.03, color="red", linewidth=0.9, linestyle=":", alpha=0.6,
#            label="threshold 0.03")
# ax.axhline(0, color="black", linewidth=0.5, alpha=0.4)
# ax.set_xlabel("Forecast horizon")
# ax.set_ylabel("Residual correlation")
# ax.set_title("Figure 2. IT feature type decomposition\n"
#              "(BTC delta_vol — each group tested in isolation)")
# ax.legend(fontsize=9, loc="upper right")
# plt.tight_layout()
# plt.savefig("fig2_decomposition.pdf", bbox_inches="tight", dpi=150)
# plt.savefig("fig2_decomposition.png", bbox_inches="tight", dpi=150)
# plt.show()
# print("Saved: fig2_decomposition.pdf / .png")

# importance_types = {
#     "Covariance":       34.2,
#     "4th-order CI":     20.0,
#     "3rd-order CI":     16.0,
#     "Pairwise MI":      15.3,
#     "KL divergence":    9.8,
#     "Auto-NMI":         4.7,
#     "Entropy":          0.0,
# }
# imp_colors = ["#2ca02c","#ff7f0e","#d62728","#1f77b4","#9467bd","#8c564b","#c7c7c7"]

# fig, ax = plt.subplots(figsize=(7, 4))
# labels_imp = list(importance_types.keys())
# values_imp = list(importance_types.values())
# bars = ax.barh(labels_imp[::-1], values_imp[::-1],
#                color=imp_colors[::-1], edgecolor="white", linewidth=0.5)

# for bar, val in zip(bars, values_imp[::-1]):
#     if val > 0.5:
#         ax.text(val + 0.3, bar.get_y() + bar.get_height()/2,
#                 f"{val:.1f}%", va="center", fontsize=10)

# ax.set_xlabel("Share of total feature importance (%)")
# ax.set_title("Figure 3. IT feature importance by type\n"
#              "(avg |Ridge coef| on AR1 residuals, all tasks)")
# ax.set_xlim(0, 42)
# plt.tight_layout()
# plt.savefig("fig3_importance.pdf", bbox_inches="tight", dpi=150)
# plt.savefig("fig3_importance.png", bbox_inches="tight", dpi=150)
# plt.show()
# print("Saved: fig3_importance.pdf / .png")

# pivot_data = res_df.pivot_table(
#     index=["asset", "task"],
#     columns="horizon_lbl",
#     values="resid_corr"
# )[h_ticks]

# row_order = [
#     ("BTC", "delta_vol"), ("BTC", "log_vol"),
#     ("ETH", "delta_vol"), ("ETH", "log_vol"),
# ]
# pivot_data = pivot_data.reindex(row_order)
# row_labels = [f"{a} {t}" for a, t in row_order]

# fig, ax = plt.subplots(figsize=(8, 3.5))
# im = ax.imshow(pivot_data.values, cmap="RdYlGn",
#                vmin=-0.02, vmax=0.14, aspect="auto")

# ax.set_xticks(range(len(h_ticks)));  ax.set_xticklabels(h_ticks)
# ax.set_yticks(range(len(row_labels))); ax.set_yticklabels(row_labels)

# for i in range(len(row_labels)):
#     for j in range(len(h_ticks)):
#         val = pivot_data.values[i, j]
#         color = "black" if val < 0.08 else "white"
#         marker = "✓" if val > 0.03 else ""
#         ax.text(j, i, f"{val:.3f}{marker}", ha="center", va="center",
#                 fontsize=9, color=color)

# plt.colorbar(im, ax=ax, label="Residual correlation", shrink=0.8)
# ax.set_title("Figure 4. Residual correlation heatmap\n"
#              "(✓ = beats threshold 0.03)")
# ax.set_xlabel("Forecast horizon")
# plt.tight_layout()
# plt.savefig("fig4_heatmap.pdf", bbox_inches="tight", dpi=150)
# plt.savefig("fig4_heatmap.png", bbox_inches="tight", dpi=150)
# plt.show()
# print("Saved: fig4_heatmap.pdf / .png")

Saved: fig1_residual_corr.pdf / .png
Saved: fig2_decomposition.pdf / .png
Saved: fig3_importance.pdf / .png
Saved: fig4_heatmap.pdf / .png


In [29]:
# from sklearn.linear_model import Ridge
# from sklearn.preprocessing import StandardScaler
# from sklearn.pipeline import Pipeline
# import numpy as np

# asset, h, task = "BTC", 96, "delta_vol_96"
# y = targets[asset][task].dropna()
# idx = X_vol.index.intersection(X_it.index).intersection(y.index)
# Xv_, Xi_, y_ = X_vol.loc[idx], X_it.loc[idx], y.loc[idx]

# mask = Xv_.notna().all(1) & Xi_.notna().all(1) & y_.notna()
# Xv_, Xi_, y_ = Xv_[mask], Xi_[mask], y_[mask]

# tr = list(range(TRAIN_BARS))
# te = list(range(TRAIN_BARS + PURGE_BARS, TRAIN_BARS + PURGE_BARS + TEST_BARS))

# ytr, yte = y_.iloc[tr], y_.iloc[te]

# train_mean = float(ytr.mean())
# pred_base  = np.full(len(yte), train_mean)
# res_tr     = ytr.values - train_mean
# res_te     = yte.values - train_mean

# mdl = Pipeline([("sc", StandardScaler()), ("m", Ridge(alpha=1.0))])
# mdl.fit(Xi_.iloc[tr], res_tr)
# it_correction = mdl.predict(Xi_.iloc[te])

# print(f"res_tr std:       {res_tr.std():.4f}")
# print(f"res_te std:       {res_te.std():.4f}")
# print(f"IT correction std:{it_correction.std():.4f}")
# print(f"corr(correction, res_te): {np.corrcoef(it_correction, res_te)[0,1]:.4f}")
# print(f"MSE baseline:     {np.mean(res_te**2):.6f}")
# print(f"MSE after IT:     {np.mean((res_te - it_correction)**2):.6f}")
# print(f"\nRidge coef max:   {np.abs(mdl.named_steps['m'].coef_).max():.4f}")
# print(f"Ridge coef mean:  {np.abs(mdl.named_steps['m'].coef_).mean():.6f}")

res_tr std:       0.2415
res_te std:       0.1671
IT correction std:0.1383
corr(correction, res_te): 0.4560
MSE baseline:     0.028194
MSE after IT:     0.027174

Ridge coef max:   0.5784
Ridge coef mean:  0.079429


In [ ]:
# import warnings
# warnings.filterwarnings("ignore")

# import numpy as np
# import pandas as pd
# from itertools import combinations
# from tqdm.auto import tqdm
# from sklearn.linear_model import Ridge
# from sklearn.preprocessing import StandardScaler
# from sklearn.pipeline import Pipeline
# from sklearn.metrics import mean_absolute_error, r2_score
# from scipy import stats as sp_stats

# TICKER_MAP = {
#     "BTC":  "BTC-USDT",
#     "ETH":  "ETH-USDT",
#     "SOL":  "SOL-USDT",
#     "BNB":  "BNB-USDT",
#     "XRP":  "XRP-USDT",
#     "DOGE": "DOGE-USDT",
#     "ADA":  "ADA-USDT",
#     "AVAX": "AVAX-USDT",
#     "TRX":  "TRX-USDT",
# }
# TARGET_ASSETS  = ["BTC", "ETH"]
# UNIVERSE       = list(TICKER_MAP.keys())
# START_DATE     = "2023-01-01"
# BARS_PER_DAY   = 48
# ANN_FACTOR     = np.sqrt(365 * BARS_PER_DAY)
# VOL_WINDOW     = 48
# STRESS_MULT    = 1.25
# HORIZONS       = [8, 48, 96]           

# IT_WINDOW      = 192                   
# IT_BINS        = 10                   
# IT_K           = 4                     

# TRAIN_BARS     = 2880
# TEST_BARS      =  480
# STEP_BARS      =  240
# PURGE_BARS     =   96

# cut = pd.Timestamp(START_DATE)

# close_parts, high_parts, low_parts = {}, {}, {}
# for asset, ticker in TICKER_MAP.items():
#     if ticker not in price_df.columns:
#         print(f"WARNING: {ticker} not found, skipping")
#         continue
#     c = price_df[ticker].loc[cut:].copy()
#     valid = c.notna()
#     close_parts[asset] = c[valid]
#     high_parts[asset]  = high_df[ticker].loc[cut:][valid] if ticker in high_df.columns \
#                          else pd.Series(np.nan, index=c[valid].index)
#     low_parts[asset]   = low_df[ticker].loc[cut:][valid]  if ticker in low_df.columns  \
#                          else pd.Series(np.nan, index=c[valid].index)

# common_idx = close_parts[TARGET_ASSETS[0]].index
# for a in close_parts:
#     common_idx = common_idx.intersection(close_parts[a].index)
# print(f"Common index: {len(common_idx)} bars  "
#       f"({common_idx[0].date()} → {common_idx[-1].date()})")

# close_w = pd.DataFrame({a: close_parts[a].reindex(common_idx) for a in close_parts})
# high_w  = pd.DataFrame({a: high_parts[a].reindex(common_idx)  for a in close_parts})
# low_w   = pd.DataFrame({a: low_parts[a].reindex(common_idx)   for a in close_parts})

# ret = np.log(close_w / close_w.shift(1))
# print(f"Returns shape: {ret.shape}")

# def build_targets(ret, target_asset):
#     r       = ret[target_asset]
#     vol_now = r.rolling(VOL_WINDOW).std() * ANN_FACTOR
#     vol_med = vol_now.rolling(365 * BARS_PER_DAY, min_periods=BARS_PER_DAY * 30).median()
#     tgt     = pd.DataFrame(index=ret.index)
#     tgt["vol_now"] = vol_now
#     for h in HORIZONS:
#         vol_fwd = r.rolling(h).std().shift(-h) * ANN_FACTOR
#         tgt[f"vol_fwd_{h}"]     = vol_fwd
#         tgt[f"log_vol_fwd_{h}"] = np.log(vol_fwd.clip(lower=1e-8))
#         tgt[f"stress_{h}"]      = (vol_fwd > vol_med * STRESS_MULT).astype(float)
#         tgt[f"delta_vol_{h}"]   = vol_fwd - vol_now
#     return tgt

# targets = {a: build_targets(ret, a) for a in TARGET_ASSETS}
# for a in TARGET_ASSETS:
#     for h in HORIZONS:
#         n = targets[a][f"stress_{h}"].sum()
#         print(f"  {a} h={h}: stress {int(n)}/{len(targets[a])} ({n/len(targets[a])*100:.1f}%)")

# def build_vol_features(ret, high_w, low_w):
#     feats = {}
#     for asset in ret.columns:
#         r = ret[asset]
#         for w in [4, 8, 16, 48, 96]:
#             feats[f"{asset}_rvol_{w}"] = r.rolling(w).std() * ANN_FACTOR
#         rv = r.rolling(VOL_WINDOW).std() * ANN_FACTOR
#         feats[f"{asset}_vov"]         = rv.rolling(VOL_WINDOW).std()
#         feats[f"{asset}_vol_zscore"]  = (rv - rv.rolling(VOL_WINDOW * 4).mean()) / \
#                                          (rv.rolling(VOL_WINDOW * 4).std() + 1e-8)
#         feats[f"{asset}_vol_pctrank"] = rv.rolling(VOL_WINDOW * 30).rank(pct=True)
#         h_col = high_w[asset] if asset in high_w.columns else pd.Series(np.nan, index=r.index)
#         l_col = low_w[asset]  if asset in low_w.columns  else pd.Series(np.nan, index=r.index)
#         hl2   = np.log((h_col / l_col).clip(lower=1e-8)) ** 2
#         feats[f"{asset}_pk_vol"] = np.sqrt(hl2.rolling(VOL_WINDOW).mean() / (4 * np.log(2))) * ANN_FACTOR
#     feats["vol_now_feat"] = feats.get("BTC_rvol_48",
#                             feats[next(iter(feats))])
#     return pd.DataFrame(feats, index=ret.index)

# print("Building vol features...")
# X_vol = build_vol_features(ret, high_w, low_w)
# print(f"X_vol: {X_vol.shape}")

# def _rk2d(mat, bins):
#     rk = np.argsort(np.argsort(mat, axis=1), axis=1)
#     return (rk * bins // mat.shape[1]).clip(0, bins - 1).astype(np.int32)

# def _ffill(result):
#     mask = np.isfinite(result)
#     idx  = np.where(mask, np.arange(len(result)), 0)
#     np.maximum.accumulate(idx, out=idx)
#     return result[idx]

# def _batch_entropy(col, window, k, bins):
#     n = len(col)
#     starts = np.arange(0, n - window + 1, k)
#     ends   = starts + window
#     n_win  = len(starts)
#     wx   = np.stack([col[s:e] for s, e in zip(starts, ends)])
#     bx   = _rk2d(wx, bins)
#     off  = (np.arange(n_win) * bins).reshape(-1, 1)
#     flat = (bx + off).ravel()
#     cnt  = np.bincount(flat, minlength=n_win * bins).reshape(n_win, bins).astype(float) + 1e-10
#     p    = cnt / cnt.sum(axis=1, keepdims=True)
#     H    = -(p * np.log(p)).sum(axis=1)
#     res  = np.full(n, np.nan); res[ends - 1] = H
#     return _ffill(res)

# def _batch_mi(col_x, col_y, window, k, bins):
#     n = len(col_x)
#     starts = np.arange(0, n - window + 1, k)
#     ends   = starts + window
#     n_win  = len(starts)
#     wx = np.stack([col_x[s:e] for s, e in zip(starts, ends)])
#     wy = np.stack([col_y[s:e] for s, e in zip(starts, ends)])
#     bx   = _rk2d(wx, bins); by = _rk2d(wy, bins)
#     joint = bx * bins + by
#     bins2 = bins * bins
#     off  = (np.arange(n_win) * bins2).reshape(-1, 1)
#     flat = (joint + off).ravel()
#     cnt  = np.bincount(flat, minlength=n_win * bins2).reshape(n_win, bins2).astype(float) + 1e-10
#     p    = cnt / cnt.sum(axis=1, keepdims=True)
#     p2d  = p.reshape(n_win, bins, bins)
#     px   = p2d.sum(axis=2, keepdims=True)
#     py   = p2d.sum(axis=1, keepdims=True)
#     mi   = np.maximum((p2d * np.log(p2d / (px * py))).sum(axis=(1, 2)), 0.0)
#     res  = np.full(n, np.nan); res[ends - 1] = mi
#     return _ffill(res)

# def _batch_cond_mi(col_x, col_y, col_z, window, k, bins):
#     """I(X;Y|Z) via 3D histogram — one window at a time (unavoidable)."""
#     n = len(col_x)
#     starts = np.arange(0, n - window + 1, k)
#     ends   = starts + window
#     res    = np.full(n, np.nan)
#     for s, e in zip(starts, ends):
#         wx, wy, wz = col_x[s:e], col_y[s:e], col_z[s:e]
#         def rk(v):
#             r = np.argsort(np.argsort(v))
#             return (r * bins // len(v)).clip(0, bins - 1).astype(np.int32)
#         bx, by, bz = rk(wx), rk(wy), rk(wz)
#         c = np.zeros((bins, bins, bins))
#         np.add.at(c, (bx, by, bz), 1); c += 1e-10
#         p    = c / c.sum()
#         p_xz = c.sum(axis=1) / c.sum(axis=1).sum()
#         p_yz = c.sum(axis=0) / c.sum(axis=0).sum()
#         p_z  = c.sum(axis=(0, 1)) / c.sum(axis=(0, 1)).sum()
#         with np.errstate(divide="ignore", invalid="ignore"):
#             ratio = np.where(
#                 (p_xz[:, np.newaxis, :] > 0) & (p_yz[np.newaxis, :, :] > 0),
#                 p_z[np.newaxis, np.newaxis, :] * p /
#                 (p_xz[:, np.newaxis, :] * p_yz[np.newaxis, :, :]), 1.0)
#         res[e - 1] = max(float((p * np.log(ratio)).sum()), 0.0)
#     return _ffill(res)

# def _batch_kl(col, window, k, bins):
#     n = len(col)
#     starts = np.arange(0, n - 2 * window + 1, k)
#     res    = np.full(n, np.nan)
#     for s in starts:
#         mid, end = s + window, s + 2 * window
#         wp, wc   = col[s:mid], col[mid:end]
#         edges    = np.linspace(min(wp.min(), wc.min()) - 1e-8,
#                                max(wp.max(), wc.max()) + 1e-8, bins + 1)
#         pp, _ = np.histogram(wp, bins=edges); pp = pp.astype(float) + 1e-10; pp /= pp.sum()
#         pc, _ = np.histogram(wc, bins=edges); pc = pc.astype(float) + 1e-10; pc /= pc.sum()
#         res[end - 1] = max(float((pc * np.log(pc / pp)).sum()), 0.0)
#     return _ffill(res)

# def build_it_features(ret, window=IT_WINDOW, bins=IT_BINS, k=IT_K):
#     assets  = list(ret.columns)
#     n       = len(ret)
#     ret_np  = ret.to_numpy(dtype=float)
#     ci      = {c: i for i, c in enumerate(assets)}
#     pairs   = list(combinations(assets, 2))
#     triplets= list(combinations(assets, 3))
#     quads   = list(combinations(assets, 4))
#     feats   = {}

#     print("  [IT] per-asset (entropy, auto-NMI, KL)...")
#     for asset in tqdm(assets, desc="per-asset", leave=False):
#         col = ret_np[:, ci[asset]]
#         H   = _batch_entropy(col, window, k, bins)
#         feats[f"entropy_{asset}"] = H
#         lag = np.roll(col, 1); lag[0] = np.nan
#         ami = _batch_mi(col, lag, window, k, bins)
#         feats[f"auto_nmi_{asset}"] = np.where(H > 1e-8, ami / H, 0.0)
#         feats[f"kl_{asset}"]       = _batch_kl(col, window, k, bins)

#     print("  [IT] pairwise MI + covariance...")
#     pair_mi = {}
#     mi_arrs, cov_arrs = [], []
#     for a, b in tqdm(pairs, desc="pairs", leave=False):
#         arr = _batch_mi(ret_np[:, ci[a]], ret_np[:, ci[b]], window, k, bins)
#         pair_mi[(a, b)] = arr
#         mi_arrs.append(arr)
#         cov_arrs.append(ret[a].rolling(window).cov(ret[b]).values)

#     mi_mat  = np.column_stack(mi_arrs)
#     cov_mat = np.column_stack(cov_arrs)
#     for stat, fn in [("mean", np.nanmean), ("min", np.nanmin),
#                      ("max", np.nanmax),   ("std", np.nanstd)]:
#         feats[f"pairwise_mi_{stat}"]  = fn(mi_mat,  axis=1)
#         feats[f"pairwise_cov_{stat}"] = fn(cov_mat, axis=1)

#     print("  [IT] 3rd-order coinformation...")
#     ci3_arrs = []
#     for a, b, c in tqdm(triplets, desc="3-coinfo", leave=False):
#         mi_ab = pair_mi.get((a, b), pair_mi.get((b, a),
#                 _batch_mi(ret_np[:, ci[a]], ret_np[:, ci[b]], window, k, bins)))
#         cmi   = _batch_cond_mi(ret_np[:, ci[a]], ret_np[:, ci[b]],
#                                 ret_np[:, ci[c]], window, k, bins)
#         ci3_arrs.append(mi_ab - cmi)

#     ci3_mat = np.column_stack(ci3_arrs)
#     for stat, fn in [("mean", np.nanmean), ("min", np.nanmin),
#                      ("max", np.nanmax),   ("std", np.nanstd)]:
#         feats[f"coinfo3_{stat}"] = fn(ci3_mat, axis=1)

#     if quads:
#         print("  [IT] 4th-order coinformation...")
#         tri_idx = {t: i for i, t in enumerate(triplets)}
#         ci4_arrs = []
#         for a, b, c, d in tqdm(quads, desc="4-coinfo", leave=False):
#             sub = []
#             for trip in combinations([a, b, c, d], 3):
#                 key = tuple(sorted(trip))
#                 for t in triplets:
#                     if set(t) == set(key):
#                         sub.append(ci3_mat[:, tri_idx[t]]); break
#             if sub:
#                 ci4_arrs.append(np.nanmean(np.column_stack(sub), axis=1))
#         if ci4_arrs:
#             ci4_mat = np.column_stack(ci4_arrs)
#             for stat, fn in [("mean", np.nanmean), ("min", np.nanmin),
#                               ("max", np.nanmax),   ("std", np.nanstd)]:
#                 feats[f"coinfo4_{stat}"] = fn(ci4_mat, axis=1)

#     X_it = pd.DataFrame(feats, index=ret.index)
#     print(f"  [IT] done: {X_it.shape[1]} features × {X_it.shape[0]} bars")
#     return X_it

# print("Building IT features...")
# X_it = build_it_features(ret)
# print(f"X_it: {X_it.shape}")

# def _wf_splits(n, train, test, step, purge):
#     start = train
#     while start + purge + test <= n:
#         yield list(range(start - train, start)), \
#               list(range(start + purge, start + purge + test))
#         start += step

# class AR1:
#     def __init__(self, col="vol_now_feat"):
#         self.col = col; self.a = self.b = 0.0
#     def _v(self, X):
#         return X[self.col].values.astype(float) if self.col in X.columns \
#                else X.iloc[:, 0].values.astype(float)
#     def fit(self, X, y):
#         v, t = self._v(X), y.values.astype(float)
#         m    = np.isfinite(v) & np.isfinite(t); v, t = v[m], t[m]
#         self.a = float(np.cov(v, t)[0, 1] / (np.var(v) + 1e-12))
#         self.b = float(np.mean(t) - self.a * np.mean(v))
#         return self
#     def predict(self, X):
#         v = self._v(X)
#         return np.where(np.isfinite(v), self.a * v + self.b, self.b)

# def _run_wf(X, y, models):
#     mask = X.notna().all(axis=1) & y.notna()
#     Xc, yc = X[mask], y[mask]
#     splits = list(_wf_splits(len(Xc), TRAIN_BARS, TEST_BARS, STEP_BARS, PURGE_BARS))
#     if not splits:
#         return pd.DataFrame()
#     records = []
#     for tr, te in tqdm(splits, desc="    folds", leave=False):
#         Xtr, Xte = Xc.iloc[tr], Xc.iloc[te]
#         ytr, yte = yc.iloc[tr], yc.iloc[te]
#         for name, mdl in models.items():
#             mdl.fit(Xtr, ytr)
#             pred = mdl.predict(Xte)
#             records.append({"model": name,
#                 "corr": float(pd.Series(pred).corr(pd.Series(yte.values))),
#                 "mae":  mean_absolute_error(yte, pred),
#                 "r2":   r2_score(yte, pred)})
#     return pd.DataFrame(records)

# def _summarize(df):
#     if df.empty: return df
#     return df.groupby("model")[["corr", "mae", "r2"]].agg(["mean", "std"]).round(4)

# def _new_ridge():
#     return Pipeline([("sc", StandardScaler()), ("m", Ridge(alpha=1.0))])

# def residual_test(X_vol, X_it, y, task="auto"):
#     """
#     OOS R² = 1 - MSE(model) / MSE(baseline)

#     Baseline выбирается адаптивно по типу таргета:
#       log_vol  → AR1(vol_now): vol_fwd ≈ vol_now, разумный prior
#       delta_vol→ historical mean: E[delta]=0 by construction, AR1 там плохой prior

#     Model = baseline_pred + Ridge(IT) correction на остатках baseline.

#     Diebold-Mariano (1995): односторонний t-test на per-fold MSE difference.
#     """
#     mask = X_vol.notna().all(axis=1) & X_it.notna().all(axis=1) & y.notna()
#     Xv, Xi, yc = X_vol[mask], X_it[mask], y[mask]
#     splits = list(_wf_splits(len(yc), TRAIN_BARS, TEST_BARS, STEP_BARS, PURGE_BARS))

#     use_mean_baseline = (task == "delta_vol") or \
#                         (task == "auto" and yc.mean() < 0.01 * yc.std())

#     all_e_base  = []
#     all_e_model = []
#     fold_d      = []   # per-fold MSE diff для DM test
#     fold_r2s    = []   # per-fold R² для усреднения

#     for tr, te in splits:
#         Xv_tr, Xv_te = Xv.iloc[tr], Xv.iloc[te]
#         Xi_tr, Xi_te = Xi.iloc[tr], Xi.iloc[te]
#         ytr,   yte   = yc.iloc[tr], yc.iloc[te]

#         if use_mean_baseline:
#             train_mean = float(ytr.mean())
#             pred_base  = np.full(len(yte), train_mean)
#             res_tr     = ytr.values - train_mean
#         else:
#             ar1 = AR1(); ar1.fit(Xv_tr, ytr)
#             pred_base = ar1.predict(Xv_te)
#             res_tr    = ytr.values - ar1.predict(Xv_tr)

#         mdl = _new_ridge(); mdl.fit(Xi_tr, res_tr)
#         pred_model = pred_base + mdl.predict(Xi_te)

#         e_base  = yte.values - pred_base
#         e_model = yte.values - pred_model

#         all_e_base.extend(e_base.tolist())
#         all_e_model.extend(e_model.tolist())

#         mse_base  = float(np.mean(e_base**2))
#         mse_model = float(np.mean(e_model**2))
#         fold_d.append(mse_base - mse_model)
#         if mse_base > 1e-12:
#             fold_r2s.append(1.0 - mse_model / mse_base)

#     r2_oos = float(np.nanmean(fold_r2s)) if fold_r2s else np.nan

#     e_base_all  = np.array(all_e_base)
#     e_model_all = np.array(all_e_model)
#     r2_oos_global = float(1 - np.sum(e_model_all**2) / np.sum(e_base_all**2))

#     d = np.array(fold_d)
#     if d.std() > 1e-10:
#         dm_t, dm_p2 = sp_stats.ttest_1samp(d, 0)
#         dm_p = float(dm_p2 / 2 if dm_t > 0 else 1 - dm_p2 / 2)
#     else:
#         dm_t, dm_p = np.nan, np.nan

#     baseline_name = "hist_mean" if use_mean_baseline else "AR1"
#     return {
#         "r2_oos":        r2_oos,           # среднее per-fold R² (основная метрика)
#         "r2_oos_global": r2_oos_global,    # глобальный (для справки)
#         "dm_t":          float(dm_t),
#         "dm_p":          dm_p,
#         "n_folds":       len(splits),
#         "beats_base":    r2_oos > 0,
#         "baseline":      baseline_name,
#     }

# X_combined = pd.concat([X_vol, X_it], axis=1)

# all_results   = {}
# residual_rows = []

# for asset in TARGET_ASSETS:
#     tgt = targets[asset]
#     for h in HORIZONS:
#         for task, col in [(f"delta_vol_{h}", f"delta_vol_{h}"),
#                           (f"log_vol_{h}",   f"log_vol_fwd_{h}")]:
#             if col not in tgt.columns:
#                 continue

#             y = tgt[col].dropna()
#             idx = X_combined.index.intersection(y.index)
#             y   = y.loc[idx]
#             Xv  = X_vol.loc[idx]
#             Xi  = X_it.loc[idx]
#             Xc  = X_combined.loc[idx]

#             tag = f"{asset}_{task}"
#             print(f"\n{'='*55}")
#             print(f"  {tag}")
#             print(f"{'='*55}")

#             models_v = {"AR1": AR1(), "Ridge_vol": _new_ridge()}
#             models_i = {"Ridge_it":  _new_ridge()}
#             models_c = {"Ridge_combined": _new_ridge()}

#             print("  [A+B] AR1 + Ridge(vol)...")
#             r_v = _run_wf(Xv, y, models_v)
#             print("  [C]   Ridge(IT-only)...")
#             r_i = _run_wf(Xi, y, models_i)
#             print("  [D]   Ridge(vol+IT)...")
#             r_c = _run_wf(Xc, y, models_c)

#             summary = _summarize(pd.concat([r_v, r_i, r_c]))
#             all_results[tag] = summary
#             print(summary.to_string())

#             print("  [E]   OOS R² (Goyal-Welch)...")
#             task_type = "delta_vol" if "delta_vol" in task else "log_vol"
#             res = residual_test(Xv, Xi, y, task=task_type)
#             res["experiment"] = tag
#             residual_rows.append(res)

#             sig = "***" if res["dm_p"]<0.01 else "**" if res["dm_p"]<0.05 \
#                   else "*" if res["dm_p"]<0.10 else ""
#             mark = "↑" if res["beats_base"] else "↓"
#             print(f"        R²_OOS(avg)={res['r2_oos']:+.4f}{mark}  "
#                   f"R²_OOS(global)={res['r2_oos_global']:+.4f}  "
#                   f"baseline={res['baseline']}  DM p={res['dm_p']:.3f}{sig}")

# print("\n" + "="*65)
# print("  ИТОГ — OOS R² (Goyal & Welch, 2008)")
# print("  R²_OOS > 0 → AR1+IT бьёт AR1 по MSE out-of-sample")
# print("  * p<0.10  ** p<0.05  *** p<0.01  (Diebold-Mariano, one-sided)")
# print("="*65)

# res_df = pd.DataFrame(residual_rows)

# def _sig(p):
#     if p is None or (isinstance(p, float) and np.isnan(p)): return ""
#     return "***" if p<0.01 else "**" if p<0.05 else "*" if p<0.10 else ""

# print(f"\n  {'experiment':<35} {'base':>10} {'R²avg':>7} {'R²global':>9}  {'DM p':>7}  sig")
# print("  " + "-"*75)
# for _, row in res_df.iterrows():
#     mark = "↑" if row["beats_base"] else "↓"
#     print(f"  {row['experiment']:<35} "
#           f"  {row['baseline']:>10}"
#           f"  {row['r2_oos']:+.4f}{mark}"
#           f"  {row['r2_oos_global']:+.6f}"
#           f"  {row['dm_p']:>7.4f}  {_sig(row['dm_p'])}")

# print(f"\n  R²_OOS > 0:    {res_df['beats_base'].sum()} / {len(res_df)}")
# print(f"  DM p < 0.05:   {(res_df['dm_p'] < 0.05).sum()} / {len(res_df)}")
# print(f"  DM p < 0.10:   {(res_df['dm_p'] < 0.10).sum()} / {len(res_df)}")
# print(f"\n  Среднее R²_OOS: {res_df['r2_oos'].mean():+.4f}")
# best = res_df.loc[res_df['r2_oos'].idxmax()]
# print(f"  Лучший R²_OOS:  {best['r2_oos']:+.4f}  ({best['experiment']}, baseline={best['baseline']})")

Common index: 46743 bars  (2023-01-01 → 2025-09-01)
Returns shape: (46743, 9)
  BTC h=8: stress 10978/46743 (23.5%)
  BTC h=48: stress 13511/46743 (28.9%)
  BTC h=96: stress 13929/46743 (29.8%)
  ETH h=8: stress 13050/46743 (27.9%)
  ETH h=48: stress 16192/46743 (34.6%)
  ETH h=96: stress 17652/46743 (37.8%)
Building vol features...
X_vol: (46743, 82)
Building IT features...
  [IT] per-asset (entropy, auto-NMI, KL)...


  [IT] pairwise MI + covariance...


  [IT] 3rd-order coinformation...


  [IT] 4th-order coinformation...


  [IT] done: 43 features × 46743 bars
X_it: (46743, 43)

  BTC_delta_vol_8
  [A+B] AR1 + Ridge(vol)...


...

  [D]   Ridge(vol+IT)...


                  corr             mae              r2         
                  mean     std    mean     std    mean      std
model                                                          
AR1             0.6613  0.1825  0.1717  0.0774  0.1110   0.5702
Ridge_combined  0.5064  0.2774  0.3096  0.2133 -3.6052  14.8246
Ridge_it        0.2050  0.3076  0.2964  0.1921 -2.2664   5.3057
Ridge_vol       0.4892  0.2699  0.2862  0.1832 -2.9922  10.0867
  [E]   OOS R² (Goyal-Welch)...
        R²_OOS(avg)=-2.0283↓  R²_OOS(global)=-1.3062  baseline=hist_mean  DM p=1.000

  ETH_log_vol_96
  [A+B] AR1 + Ridge(vol)...


  [C]   Ridge(IT-only)...


  [D]   Ridge(vol+IT)...


                  corr             mae               r2         
                  mean     std    mean     std     mean      std
model                                                           
AR1            -0.0189  0.2755  0.2966  0.1019  -1.0202   1.6629
Ridge_combined  0.0703  0.3351  0.5280  0.3668 -10.5969  43.9612
Ridge_it        0.0720  0.3429  0.4349  0.3061  -6.7620  22.5316
Ridge_vol       0.0642  0.3135  0.4983  0.2889  -8.9384  25.7415
  [E]   OOS R² (Goyal-Welch)...
        R²_OOS(avg)=-2.6265↓  R²_OOS(global)=-1.7618  baseline=AR1  DM p=0.997

  ИТОГ — OOS R² (Goyal & Welch, 2008)
  R²_OOS > 0 → AR1+IT бьёт AR1 по MSE out-of-sample
  * p<0.10  ** p<0.05  *** p<0.01  (Diebold-Mariano, one-sided)

  experiment                                base   R²avg  R²global     DM p  sig
  ---------------------------------------------------------------------------
  BTC_delta_vol_8                        hist_mean  -0.7894↓  -0.669914   1.0000  
  BTC_log_vol_8                     

In [ ]:
# from sklearn.linear_model import Ridge
# from sklearn.preprocessing import StandardScaler
# from sklearn.pipeline import Pipeline
# import numpy as np, pandas as pd

# asset, task_col = "BTC", "delta_vol_96"
# y = targets[asset][task_col].dropna()
# idx = X_vol.index.intersection(X_it.index).intersection(y.index)
# mask = X_vol.loc[idx].notna().all(1) & X_it.loc[idx].notna().all(1) & y.loc[idx].notna()
# Xv_, Xi_, y_ = X_vol.loc[idx][mask], X_it.loc[idx][mask], y.loc[idx][mask]

# def _splits(n):
#     s = TRAIN_BARS
#     while s + PURGE_BARS + TEST_BARS <= n:
#         yield list(range(s-TRAIN_BARS,s)), list(range(s+PURGE_BARS,s+PURGE_BARS+TEST_BARS))
#         s += STEP_BARS

# fold_r2s, fold_corrs, fold_scales = [], [], []

# for tr, te in _splits(len(y_)):
#     ytr, yte = y_.iloc[tr], y_.iloc[te]
    
#     train_mean = float(ytr.mean())
#     pred_base  = np.full(len(yte), train_mean)
#     res_tr     = ytr.values - train_mean
#     res_te     = yte.values - train_mean
    
#     mdl = Pipeline([("sc", StandardScaler()), ("m", Ridge(alpha=1.0))])
#     mdl.fit(Xi_.iloc[tr], res_tr)
#     corr_pred = mdl.predict(Xi_.iloc[te])
    
#     scale = corr_pred.std() / (res_te.std() + 1e-8)  
#     corr  = np.corrcoef(corr_pred, res_te)[0,1]
#     r2_fold = 1 - np.mean((res_te - corr_pred)**2) / np.mean(res_te**2)
    
#     fold_r2s.append(r2_fold)
#     fold_corrs.append(corr)
#     fold_scales.append(scale)

# fold_r2s   = np.array(fold_r2s)
# fold_corrs = np.array(fold_corrs)
# fold_scales= np.array(fold_scales)

# print(f"Фолдов с R²>0:     {(fold_r2s>0).sum()} / {len(fold_r2s)}  ({(fold_r2s>0).mean():.0%})")
# print(f"Среднее R²:        {fold_r2s.mean():+.4f}  (std={fold_r2s.std():.4f})")
# print(f"Медианный R²:      {np.median(fold_r2s):+.4f}")
# print(f"Средняя corr:      {fold_corrs.mean():+.4f}  (std={fold_corrs.std():.4f})")
# print(f"Средний scale:     {fold_scales.mean():.4f}  (std={fold_scales.std():.4f})")
# print(f"\nscale > 2 (overshooting): {(fold_scales>2).sum()} / {len(fold_scales)}")
# print(f"\nЕсли scale >> 1 — Ridge переоценивает коррекцию.")
# print(f"Решение: уменьшить alpha или масштабировать коррекцию на оптимальный коэф.")

Фолдов с R²>0:     29 / 174  (17%)
Среднее R²:        -3.4911  (std=16.7331)
Медианный R²:      -0.7405
Средняя corr:      +0.2265  (std=0.2840)
Средний scale:     1.1020  (std=1.0763)

scale > 2 (overshooting): 11 / 174

Если scale >> 1 — Ridge переоценивает коррекцию.
Решение: уменьшить alpha или масштабировать коррекцию на оптимальный коэф.


In [33]:
# import pandas as pd
# import numpy as np

# fold_dates = []
# for tr, te in _splits(len(y_)):
#     fold_dates.append(y_.index[te[0]])  # дата начала test-периода

# fold_df = pd.DataFrame({
#     "test_start": fold_dates,
#     "r2":         fold_r2s,
#     "corr":       fold_corrs,
#     "scale":      fold_scales,
# })
# fold_df["year_month"] = fold_df["test_start"].dt.to_period("M")

# print("Топ-10 лучших фолдов (R² > 0):")
# print(fold_df.nlargest(10, "r2")[["test_start","r2","corr","scale"]].to_string(index=False))

# print("\nТоп-10 худших фолдов:")
# print(fold_df.nsmallest(10, "r2")[["test_start","r2","corr","scale"]].to_string(index=False))

# print("\nR² по кварталам:")
# fold_df["quarter"] = fold_df["test_start"].dt.to_period("Q")
# quarterly = fold_df.groupby("quarter")["r2"].agg(["mean","median","count"])
# print(quarterly.to_string())

# print(f"\nПозитивные фолды — в каком периоде?")
# pos = fold_df[fold_df["r2"] > 0]
# print(pos.groupby("quarter")["r2"].count().to_string())

Топ-10 лучших фолдов (R² > 0):
         test_start         r2       corr      scale
2024-11-24 04:30:00 0.46568547 0.72003447 0.95384475
2024-07-27 04:30:00 0.43046778 0.64554729 0.66403991
2024-06-07 03:30:00 0.29604159 0.67061950 1.04675848
2024-02-18 03:30:00 0.29571993 0.43485436 0.60353726
2025-07-27 04:30:00 0.26112786 0.58522617 0.82129922
2023-08-12 03:30:00 0.25560922 0.52912505 0.48760498
2023-10-21 03:30:00 0.24034246 0.65071924 1.05434836
2024-01-09 03:30:00 0.20039288 0.68916818 0.85456372
2024-02-23 03:30:00 0.18711908 0.67760654 0.67840781
2024-10-20 04:30:00 0.18027484 0.40156405 0.46773354

Топ-10 худших фолдов:
         test_start            r2        corr       scale
2023-06-08 03:30:00 -197.97015112  0.05790639 11.35888948
2023-06-03 03:30:00  -93.52382036 -0.39734387  7.91533667
2024-11-04 04:30:00  -23.35279652 -0.05749550  2.38158560
2024-11-09 04:30:00  -23.20901649  0.35364060  1.46549444
2024-11-14 04:30:00  -21.70882587  0.44315102  3.08190294
2023-07-08 03:3

In [ ]:
# import warnings
# warnings.filterwarnings("ignore")

# import numpy as np
# import pandas as pd
# from itertools import combinations
# from tqdm.auto import tqdm
# from sklearn.linear_model import Ridge
# from sklearn.preprocessing import StandardScaler
# from sklearn.pipeline import Pipeline
# from sklearn.metrics import mean_absolute_error, r2_score
# from scipy import stats as sp_stats

# TICKER_MAP = {
#     "BTC":  "BTC-USDT",
#     "ETH":  "ETH-USDT",
#     "SOL":  "SOL-USDT",
#     "BNB":  "BNB-USDT",
#     "XRP":  "XRP-USDT",
#     "DOGE": "DOGE-USDT",
#     "ADA":  "ADA-USDT",
#     "AVAX": "AVAX-USDT",
#     "TRX":  "TRX-USDT",
# }
# TARGET_ASSETS  = ["BTC", "ETH"]
# UNIVERSE       = list(TICKER_MAP.keys())
# START_DATE     = "2023-01-01"
# BARS_PER_DAY   = 48
# ANN_FACTOR     = np.sqrt(365 * BARS_PER_DAY)
# VOL_WINDOW     = 48
# STRESS_MULT    = 1.25
# HORIZONS       = [8, 48, 96]          

# IT_WINDOW      = 192                 
# IT_BINS        = 10                    
# IT_K           = 4                    

# TRAIN_BARS     = 2880
# TEST_BARS      =  480
# STEP_BARS      =  240
# PURGE_BARS     =   96

# cut = pd.Timestamp(START_DATE)

# close_parts, high_parts, low_parts = {}, {}, {}
# for asset, ticker in TICKER_MAP.items():
#     if ticker not in price_df.columns:
#         print(f"WARNING: {ticker} not found, skipping")
#         continue
#     c = price_df[ticker].loc[cut:].copy()
#     valid = c.notna()
#     close_parts[asset] = c[valid]
#     high_parts[asset]  = high_df[ticker].loc[cut:][valid] if ticker in high_df.columns \
#                          else pd.Series(np.nan, index=c[valid].index)
#     low_parts[asset]   = low_df[ticker].loc[cut:][valid]  if ticker in low_df.columns  \
#                          else pd.Series(np.nan, index=c[valid].index)

# common_idx = close_parts[TARGET_ASSETS[0]].index
# for a in close_parts:
#     common_idx = common_idx.intersection(close_parts[a].index)
# print(f"Common index: {len(common_idx)} bars  "
#       f"({common_idx[0].date()} → {common_idx[-1].date()})")

# close_w = pd.DataFrame({a: close_parts[a].reindex(common_idx) for a in close_parts})
# high_w  = pd.DataFrame({a: high_parts[a].reindex(common_idx)  for a in close_parts})
# low_w   = pd.DataFrame({a: low_parts[a].reindex(common_idx)   for a in close_parts})

# ret = np.log(close_w / close_w.shift(1))
# print(f"Returns shape: {ret.shape}")

# def build_targets(ret, target_asset):
#     r       = ret[target_asset]
#     vol_now = r.rolling(VOL_WINDOW).std() * ANN_FACTOR
#     vol_med = vol_now.rolling(365 * BARS_PER_DAY, min_periods=BARS_PER_DAY * 30).median()
#     tgt     = pd.DataFrame(index=ret.index)
#     tgt["vol_now"] = vol_now
#     for h in HORIZONS:
#         vol_fwd = r.rolling(h).std().shift(-h) * ANN_FACTOR
#         tgt[f"vol_fwd_{h}"]     = vol_fwd
#         tgt[f"log_vol_fwd_{h}"] = np.log(vol_fwd.clip(lower=1e-8))
#         tgt[f"stress_{h}"]      = (vol_fwd > vol_med * STRESS_MULT).astype(float)
#         tgt[f"delta_vol_{h}"]   = vol_fwd - vol_now
#     return tgt

# targets = {a: build_targets(ret, a) for a in TARGET_ASSETS}
# for a in TARGET_ASSETS:
#     for h in HORIZONS:
#         n = targets[a][f"stress_{h}"].sum()
#         print(f"  {a} h={h}: stress {int(n)}/{len(targets[a])} ({n/len(targets[a])*100:.1f}%)")

# def build_vol_features(ret, high_w, low_w):
#     feats = {}
#     for asset in ret.columns:
#         r = ret[asset]
#         for w in [4, 8, 16, 48, 96]:
#             feats[f"{asset}_rvol_{w}"] = r.rolling(w).std() * ANN_FACTOR
#         rv = r.rolling(VOL_WINDOW).std() * ANN_FACTOR
#         feats[f"{asset}_vov"]         = rv.rolling(VOL_WINDOW).std()
#         feats[f"{asset}_vol_zscore"]  = (rv - rv.rolling(VOL_WINDOW * 4).mean()) / \
#                                          (rv.rolling(VOL_WINDOW * 4).std() + 1e-8)
#         feats[f"{asset}_vol_pctrank"] = rv.rolling(VOL_WINDOW * 30).rank(pct=True)
#         h_col = high_w[asset] if asset in high_w.columns else pd.Series(np.nan, index=r.index)
#         l_col = low_w[asset]  if asset in low_w.columns  else pd.Series(np.nan, index=r.index)
#         hl2   = np.log((h_col / l_col).clip(lower=1e-8)) ** 2
#         feats[f"{asset}_pk_vol"] = np.sqrt(hl2.rolling(VOL_WINDOW).mean() / (4 * np.log(2))) * ANN_FACTOR
#     feats["vol_now_feat"] = feats.get("BTC_rvol_48",
#                             feats[next(iter(feats))])
#     return pd.DataFrame(feats, index=ret.index)

# print("Building vol features...")
# X_vol = build_vol_features(ret, high_w, low_w)
# print(f"X_vol: {X_vol.shape}")

# def _rk2d(mat, bins):
#     rk = np.argsort(np.argsort(mat, axis=1), axis=1)
#     return (rk * bins // mat.shape[1]).clip(0, bins - 1).astype(np.int32)

# def _ffill(result):
#     mask = np.isfinite(result)
#     idx  = np.where(mask, np.arange(len(result)), 0)
#     np.maximum.accumulate(idx, out=idx)
#     return result[idx]

# def _batch_entropy(col, window, k, bins):
#     n = len(col)
#     starts = np.arange(0, n - window + 1, k)
#     ends   = starts + window
#     n_win  = len(starts)
#     wx   = np.stack([col[s:e] for s, e in zip(starts, ends)])
#     bx   = _rk2d(wx, bins)
#     off  = (np.arange(n_win) * bins).reshape(-1, 1)
#     flat = (bx + off).ravel()
#     cnt  = np.bincount(flat, minlength=n_win * bins).reshape(n_win, bins).astype(float) + 1e-10
#     p    = cnt / cnt.sum(axis=1, keepdims=True)
#     H    = -(p * np.log(p)).sum(axis=1)
#     res  = np.full(n, np.nan); res[ends - 1] = H
#     return _ffill(res)

# def _batch_mi(col_x, col_y, window, k, bins):
#     n = len(col_x)
#     starts = np.arange(0, n - window + 1, k)
#     ends   = starts + window
#     n_win  = len(starts)
#     wx = np.stack([col_x[s:e] for s, e in zip(starts, ends)])
#     wy = np.stack([col_y[s:e] for s, e in zip(starts, ends)])
#     bx   = _rk2d(wx, bins); by = _rk2d(wy, bins)
#     joint = bx * bins + by
#     bins2 = bins * bins
#     off  = (np.arange(n_win) * bins2).reshape(-1, 1)
#     flat = (joint + off).ravel()
#     cnt  = np.bincount(flat, minlength=n_win * bins2).reshape(n_win, bins2).astype(float) + 1e-10
#     p    = cnt / cnt.sum(axis=1, keepdims=True)
#     p2d  = p.reshape(n_win, bins, bins)
#     px   = p2d.sum(axis=2, keepdims=True)
#     py   = p2d.sum(axis=1, keepdims=True)
#     mi   = np.maximum((p2d * np.log(p2d / (px * py))).sum(axis=(1, 2)), 0.0)
#     res  = np.full(n, np.nan); res[ends - 1] = mi
#     return _ffill(res)

# def _batch_cond_mi(col_x, col_y, col_z, window, k, bins):
#     """I(X;Y|Z) via 3D histogram — one window at a time (unavoidable)."""
#     n = len(col_x)
#     starts = np.arange(0, n - window + 1, k)
#     ends   = starts + window
#     res    = np.full(n, np.nan)
#     for s, e in zip(starts, ends):
#         wx, wy, wz = col_x[s:e], col_y[s:e], col_z[s:e]
#         def rk(v):
#             r = np.argsort(np.argsort(v))
#             return (r * bins // len(v)).clip(0, bins - 1).astype(np.int32)
#         bx, by, bz = rk(wx), rk(wy), rk(wz)
#         c = np.zeros((bins, bins, bins))
#         np.add.at(c, (bx, by, bz), 1); c += 1e-10
#         p    = c / c.sum()
#         p_xz = c.sum(axis=1) / c.sum(axis=1).sum()
#         p_yz = c.sum(axis=0) / c.sum(axis=0).sum()
#         p_z  = c.sum(axis=(0, 1)) / c.sum(axis=(0, 1)).sum()
#         with np.errstate(divide="ignore", invalid="ignore"):
#             ratio = np.where(
#                 (p_xz[:, np.newaxis, :] > 0) & (p_yz[np.newaxis, :, :] > 0),
#                 p_z[np.newaxis, np.newaxis, :] * p /
#                 (p_xz[:, np.newaxis, :] * p_yz[np.newaxis, :, :]), 1.0)
#         res[e - 1] = max(float((p * np.log(ratio)).sum()), 0.0)
#     return _ffill(res)

# def _batch_kl(col, window, k, bins):
#     n = len(col)
#     starts = np.arange(0, n - 2 * window + 1, k)
#     res    = np.full(n, np.nan)
#     for s in starts:
#         mid, end = s + window, s + 2 * window
#         wp, wc   = col[s:mid], col[mid:end]
#         edges    = np.linspace(min(wp.min(), wc.min()) - 1e-8,
#                                max(wp.max(), wc.max()) + 1e-8, bins + 1)
#         pp, _ = np.histogram(wp, bins=edges); pp = pp.astype(float) + 1e-10; pp /= pp.sum()
#         pc, _ = np.histogram(wc, bins=edges); pc = pc.astype(float) + 1e-10; pc /= pc.sum()
#         res[end - 1] = max(float((pc * np.log(pc / pp)).sum()), 0.0)
#     return _ffill(res)

# def build_it_features(ret, window=IT_WINDOW, bins=IT_BINS, k=IT_K):
#     assets  = list(ret.columns)
#     n       = len(ret)
#     ret_np  = ret.to_numpy(dtype=float)
#     ci      = {c: i for i, c in enumerate(assets)}
#     pairs   = list(combinations(assets, 2))
#     triplets= list(combinations(assets, 3))
#     quads   = list(combinations(assets, 4))
#     feats   = {}

#     print("  [IT] per-asset (entropy, auto-NMI, KL)...")
#     for asset in tqdm(assets, desc="per-asset", leave=False):
#         col = ret_np[:, ci[asset]]
#         H   = _batch_entropy(col, window, k, bins)
#         feats[f"entropy_{asset}"] = H
#         lag = np.roll(col, 1); lag[0] = np.nan
#         ami = _batch_mi(col, lag, window, k, bins)
#         feats[f"auto_nmi_{asset}"] = np.where(H > 1e-8, ami / H, 0.0)
#         feats[f"kl_{asset}"]       = _batch_kl(col, window, k, bins)

#     print("  [IT] pairwise MI + covariance...")
#     pair_mi = {}
#     mi_arrs, cov_arrs = [], []
#     for a, b in tqdm(pairs, desc="pairs", leave=False):
#         arr = _batch_mi(ret_np[:, ci[a]], ret_np[:, ci[b]], window, k, bins)
#         pair_mi[(a, b)] = arr
#         mi_arrs.append(arr)
#         cov_arrs.append(ret[a].rolling(window).cov(ret[b]).values)

#     mi_mat  = np.column_stack(mi_arrs)
#     cov_mat = np.column_stack(cov_arrs)
#     for stat, fn in [("mean", np.nanmean), ("min", np.nanmin),
#                      ("max", np.nanmax),   ("std", np.nanstd)]:
#         feats[f"pairwise_mi_{stat}"]  = fn(mi_mat,  axis=1)
#         feats[f"pairwise_cov_{stat}"] = fn(cov_mat, axis=1)

#     print("  [IT] 3rd-order coinformation...")
#     ci3_arrs = []
#     for a, b, c in tqdm(triplets, desc="3-coinfo", leave=False):
#         mi_ab = pair_mi.get((a, b), pair_mi.get((b, a),
#                 _batch_mi(ret_np[:, ci[a]], ret_np[:, ci[b]], window, k, bins)))
#         cmi   = _batch_cond_mi(ret_np[:, ci[a]], ret_np[:, ci[b]],
#                                 ret_np[:, ci[c]], window, k, bins)
#         ci3_arrs.append(mi_ab - cmi)

#     ci3_mat = np.column_stack(ci3_arrs)
#     for stat, fn in [("mean", np.nanmean), ("min", np.nanmin),
#                      ("max", np.nanmax),   ("std", np.nanstd)]:
#         feats[f"coinfo3_{stat}"] = fn(ci3_mat, axis=1)

#     if quads:
#         print("  [IT] 4th-order coinformation...")
#         tri_idx = {t: i for i, t in enumerate(triplets)}
#         ci4_arrs = []
#         for a, b, c, d in tqdm(quads, desc="4-coinfo", leave=False):
#             sub = []
#             for trip in combinations([a, b, c, d], 3):
#                 key = tuple(sorted(trip))
#                 for t in triplets:
#                     if set(t) == set(key):
#                         sub.append(ci3_mat[:, tri_idx[t]]); break
#             if sub:
#                 ci4_arrs.append(np.nanmean(np.column_stack(sub), axis=1))
#         if ci4_arrs:
#             ci4_mat = np.column_stack(ci4_arrs)
#             for stat, fn in [("mean", np.nanmean), ("min", np.nanmin),
#                               ("max", np.nanmax),   ("std", np.nanstd)]:
#                 feats[f"coinfo4_{stat}"] = fn(ci4_mat, axis=1)

#     X_it = pd.DataFrame(feats, index=ret.index)
#     print(f"  [IT] done: {X_it.shape[1]} features × {X_it.shape[0]} bars")
#     return X_it

# print("Building IT features...")
# X_it = build_it_features(ret)
# print(f"X_it: {X_it.shape}")

# def _wf_splits(n, train, test, step, purge):
#     start = train
#     while start + purge + test <= n:
#         yield list(range(start - train, start)), \
#               list(range(start + purge, start + purge + test))
#         start += step

# class AR1:
#     def __init__(self, col="vol_now_feat"):
#         self.col = col; self.a = self.b = 0.0
#     def _v(self, X):
#         return X[self.col].values.astype(float) if self.col in X.columns \
#                else X.iloc[:, 0].values.astype(float)
#     def fit(self, X, y):
#         v, t = self._v(X), y.values.astype(float)
#         m    = np.isfinite(v) & np.isfinite(t); v, t = v[m], t[m]
#         self.a = float(np.cov(v, t)[0, 1] / (np.var(v) + 1e-12))
#         self.b = float(np.mean(t) - self.a * np.mean(v))
#         return self
#     def predict(self, X):
#         v = self._v(X)
#         return np.where(np.isfinite(v), self.a * v + self.b, self.b)

# def _run_wf(X, y, models):
#     mask = X.notna().all(axis=1) & y.notna()
#     Xc, yc = X[mask], y[mask]
#     splits = list(_wf_splits(len(Xc), TRAIN_BARS, TEST_BARS, STEP_BARS, PURGE_BARS))
#     if not splits:
#         return pd.DataFrame()
#     records = []
#     for tr, te in tqdm(splits, desc="    folds", leave=False):
#         Xtr, Xte = Xc.iloc[tr], Xc.iloc[te]
#         ytr, yte = yc.iloc[tr], yc.iloc[te]
#         for name, mdl in models.items():
#             mdl.fit(Xtr, ytr)
#             pred = mdl.predict(Xte)
#             records.append({"model": name,
#                 "corr": float(pd.Series(pred).corr(pd.Series(yte.values))),
#                 "mae":  mean_absolute_error(yte, pred),
#                 "r2":   r2_score(yte, pred)})
#     return pd.DataFrame(records)

# def _summarize(df):
#     if df.empty: return df
#     return df.groupby("model")[["corr", "mae", "r2"]].agg(["mean", "std"]).round(4)

# def _new_ridge():
#     return Pipeline([("sc", StandardScaler()), ("m", Ridge(alpha=1.0))])

# def residual_test(X_vol, X_it, y, task="auto",
#                   scale_outlier_threshold=3.0,
#                   burnin_multiplier=2):
#     """
#     OOS R² (Goyal & Welch, 2008) — адаптивный baseline + robust метрики.

#     Baseline:
#       log_vol   → AR1(vol_now)
#       delta_vol → historical mean (E[delta]=0 by construction)

#     Outlier-фолды (scale > threshold) исключаются из median/mean
#     но сохраняются в статистике для прозрачности.

#     Burn-in: первые burnin_multiplier * IT_WINDOW баров X_it
#     обнуляются в NaN — они ненадёжны пока rolling окна не заполнились.

#     Метрики (основная → медианный R²):
#       r2_median   — медиана per-fold R² без outliers  ← главная
#       r2_mean     — среднее per-fold R² без outliers
#       r2_global   — глобальный (для справки, чувствителен к выбросам)
#       pct_positive— доля фолдов с R² > 0
#       n_outliers  — число выброшенных фолдов (scale > threshold)
#     """
#     burnin = burnin_multiplier * IT_WINDOW
#     Xi_safe = X_it.copy()
#     Xi_safe.iloc[:burnin] = np.nan

#     mask = X_vol.notna().all(axis=1) & Xi_safe.notna().all(axis=1) & y.notna()
#     Xv, Xi, yc = X_vol[mask], Xi_safe[mask], y[mask]
#     splits = list(_wf_splits(len(yc), TRAIN_BARS, TEST_BARS, STEP_BARS, PURGE_BARS))

#     use_mean_baseline = (task == "delta_vol") or \
#                         (task == "auto" and abs(yc.mean()) < 0.05 * yc.std())

#     all_e_base, all_e_model = [], []
#     fold_records = []   # (r2, scale, mse_diff) per fold

#     for tr, te in splits:
#         Xv_tr, Xv_te = Xv.iloc[tr], Xv.iloc[te]
#         Xi_tr, Xi_te = Xi.iloc[tr], Xi.iloc[te]
#         ytr,   yte   = yc.iloc[tr], yc.iloc[te]

#         if use_mean_baseline:
#             train_mean = float(ytr.mean())
#             pred_base  = np.full(len(yte), train_mean)
#             res_tr     = ytr.values - train_mean
#         else:
#             ar1 = AR1(); ar1.fit(Xv_tr, ytr)
#             pred_base = ar1.predict(Xv_te)
#             res_tr    = ytr.values - ar1.predict(Xv_tr)

#         mdl = _new_ridge(); mdl.fit(Xi_tr, res_tr)
#         correction = mdl.predict(Xi_te)
#         pred_model = pred_base + correction

#         e_base  = yte.values - pred_base
#         e_model = yte.values - pred_model

#         mse_base  = float(np.mean(e_base**2))
#         mse_model = float(np.mean(e_model**2))

#         res_te = yte.values - pred_base
#         scale  = correction.std() / (res_te.std() + 1e-8)
#         r2_fold = 1.0 - mse_model / mse_base if mse_base > 1e-12 else np.nan

#         all_e_base.extend(e_base.tolist())
#         all_e_model.extend(e_model.tolist())
#         fold_records.append({
#             "r2":       r2_fold,
#             "scale":    scale,
#             "mse_diff": mse_base - mse_model,
#             "outlier":  scale > scale_outlier_threshold,
#         })

#     fold_df   = pd.DataFrame(fold_records)
#     clean     = fold_df[~fold_df["outlier"]]
#     n_outliers= int(fold_df["outlier"].sum())

#     r2_vals_clean = clean["r2"].dropna().values

#     r2_median  = float(np.median(r2_vals_clean)) if len(r2_vals_clean) else np.nan
#     r2_mean    = float(np.mean(r2_vals_clean))   if len(r2_vals_clean) else np.nan
#     pct_pos    = float((r2_vals_clean > 0).mean()) if len(r2_vals_clean) else np.nan

#     e_base_all  = np.array(all_e_base)
#     e_model_all = np.array(all_e_model)
#     r2_global   = float(1 - np.sum(e_model_all**2) / np.sum(e_base_all**2))

#     d = clean["mse_diff"].values
#     if len(d) > 1 and d.std() > 1e-10:
#         dm_t, dm_p2 = sp_stats.ttest_1samp(d, 0)
#         dm_p = float(dm_p2 / 2 if dm_t > 0 else 1 - dm_p2 / 2)
#     else:
#         dm_t, dm_p = np.nan, np.nan

#     return {
#         "r2_median":   r2_median,    # ← основная метрика
#         "r2_mean":     r2_mean,
#         "r2_global":   r2_global,
#         "pct_positive":pct_pos,
#         "n_folds":     len(splits),
#         "n_outliers":  n_outliers,
#         "dm_p":        dm_p,
#         "beats_base":  r2_median > 0,
#         "baseline":    "hist_mean" if use_mean_baseline else "AR1",
#     }

# X_combined = pd.concat([X_vol, X_it], axis=1)

# all_results   = {}
# residual_rows = []

# for asset in TARGET_ASSETS:
#     tgt = targets[asset]
#     for h in HORIZONS:
#         for task, col in [(f"delta_vol_{h}", f"delta_vol_{h}"),
#                           (f"log_vol_{h}",   f"log_vol_fwd_{h}")]:
#             if col not in tgt.columns:
#                 continue

#             y = tgt[col].dropna()
#             idx = X_combined.index.intersection(y.index)
#             y   = y.loc[idx]
#             Xv  = X_vol.loc[idx]
#             Xi  = X_it.loc[idx]
#             Xc  = X_combined.loc[idx]

#             tag = f"{asset}_{task}"
#             print(f"\n{'='*55}")
#             print(f"  {tag}")
#             print(f"{'='*55}")

#             models_v = {"AR1": AR1(), "Ridge_vol": _new_ridge()}
#             models_i = {"Ridge_it":  _new_ridge()}
#             models_c = {"Ridge_combined": _new_ridge()}

#             print("  [A+B] AR1 + Ridge(vol)...")
#             r_v = _run_wf(Xv, y, models_v)
#             print("  [C]   Ridge(IT-only)...")
#             r_i = _run_wf(Xi, y, models_i)
#             print("  [D]   Ridge(vol+IT)...")
#             r_c = _run_wf(Xc, y, models_c)

#             summary = _summarize(pd.concat([r_v, r_i, r_c]))
#             all_results[tag] = summary
#             print(summary.to_string())

#             print("  [E]   OOS R² (Goyal-Welch)...")
#             task_type = "delta_vol" if "delta_vol" in task else "log_vol"
#             res = residual_test(Xv, Xi, y, task=task_type)
#             res["experiment"] = tag
#             residual_rows.append(res)

#             sig = "***" if res["dm_p"]<0.01 else "**" if res["dm_p"]<0.05 \
#                   else "*" if res["dm_p"]<0.10 else ""
#             mark = "↑" if res["beats_base"] else "↓"
#             print(f"        R²_median={res['r2_median']:+.4f}{mark}  "
#                   f"R²_mean={res['r2_mean']:+.4f}  "
#                   f"pos={res['pct_positive']:.0%}  "
#                   f"outliers={res['n_outliers']}  "
#                   f"base={res['baseline']}  DM p={res['dm_p']:.3f}{sig}")

# print("\n" + "="*75)
# print("  ИТОГ — OOS R² (Goyal & Welch, 2008)")
# print("  Основная метрика: медианный per-fold R² (без outlier-фолдов scale>3)")
# print("  * p<0.10  ** p<0.05  *** p<0.01  (Diebold-Mariano, one-sided)")
# print("="*75)

# res_df = pd.DataFrame(residual_rows)

# def _sig(p):
#     if p is None or (isinstance(p, float) and np.isnan(p)): return ""
#     return "***" if p<0.01 else "**" if p<0.05 else "*" if p<0.10 else ""

# print(f"\n  {'experiment':<32} {'base':>10} {'R²med':>7} {'R²mean':>7} "
#       f"{'pos%':>6} {'out':>4}  {'DM p':>7}  sig")
# print("  " + "-"*80)
# for _, row in res_df.iterrows():
#     mark = "↑" if row["beats_base"] else "↓"
#     print(f"  {row['experiment']:<32}"
#           f"  {row['baseline']:>10}"
#           f"  {row['r2_median']:+.4f}{mark}"
#           f"  {row['r2_mean']:+.4f}"
#           f"  {row['pct_positive']:>5.0%}"
#           f"  {int(row['n_outliers']):>3}"
#           f"  {row['dm_p']:>7.4f}  {_sig(row['dm_p'])}")

# print(f"\n  R²_median > 0:   {res_df['beats_base'].sum()} / {len(res_df)}")
# print(f"  DM p < 0.05:     {(res_df['dm_p'] < 0.05).sum()} / {len(res_df)}")
# print(f"  DM p < 0.10:     {(res_df['dm_p'] < 0.10).sum()} / {len(res_df)}")
# print(f"\n  Среднее R²_median: {res_df['r2_median'].mean():+.4f}")
# print(f"  Среднее R²_mean:   {res_df['r2_mean'].mean():+.4f}")
# print(f"  Средняя pos%:      {res_df['pct_positive'].mean():.0%}")
# best = res_df.loc[res_df["r2_median"].idxmax()]
# print(f"\n  Лучший R²_median:  {best['r2_median']:+.4f}  "
#       f"({best['experiment']}, base={best['baseline']})")

Common index: 46743 bars  (2023-01-01 → 2025-09-01)
Returns shape: (46743, 9)
  BTC h=8: stress 10978/46743 (23.5%)
  BTC h=48: stress 13511/46743 (28.9%)
  BTC h=96: stress 13929/46743 (29.8%)
  ETH h=8: stress 13050/46743 (27.9%)
  ETH h=48: stress 16192/46743 (34.6%)
  ETH h=96: stress 17652/46743 (37.8%)
Building vol features...
X_vol: (46743, 82)
Building IT features...
  [IT] per-asset (entropy, auto-NMI, KL)...


  [IT] pairwise MI + covariance...


  [IT] 3rd-order coinformation...


  [IT] 4th-order coinformation...


  [IT] done: 43 features × 46743 bars
X_it: (46743, 43)

  BTC_delta_vol_8
  [A+B] AR1 + Ridge(vol)...


...

  [D]   Ridge(vol+IT)...


                  corr             mae              r2         
                  mean     std    mean     std    mean      std
model                                                          
AR1             0.6613  0.1825  0.1717  0.0774  0.1110   0.5702
Ridge_combined  0.5064  0.2774  0.3096  0.2133 -3.6052  14.8246
Ridge_it        0.2050  0.3076  0.2964  0.1921 -2.2664   5.3057
Ridge_vol       0.4892  0.2699  0.2862  0.1832 -2.9922  10.0867
  [E]   OOS R² (Goyal-Welch)...
        R²_median=-0.7262↓  R²_mean=-1.7371  pos=12%  outliers=3  base=hist_mean  DM p=1.000

  ETH_log_vol_96
  [A+B] AR1 + Ridge(vol)...


  [C]   Ridge(IT-only)...


  [D]   Ridge(vol+IT)...


                  corr             mae               r2         
                  mean     std    mean     std     mean      std
model                                                           
AR1            -0.0189  0.2755  0.2966  0.1019  -1.0202   1.6629
Ridge_combined  0.0703  0.3351  0.5280  0.3668 -10.5969  43.9612
Ridge_it        0.0720  0.3429  0.4349  0.3061  -6.7620  22.5316
Ridge_vol       0.0642  0.3135  0.4983  0.2889  -8.9384  25.7415
  [E]   OOS R² (Goyal-Welch)...
        R²_median=-0.6322↓  R²_mean=-1.0951  pos=21%  outliers=7  base=AR1  DM p=1.000

  ИТОГ — OOS R² (Goyal & Welch, 2008)
  Основная метрика: медианный per-fold R² (без outlier-фолдов scale>3)
  * p<0.10  ** p<0.05  *** p<0.01  (Diebold-Mariano, one-sided)

  experiment                             base   R²med  R²mean   pos%  out     DM p  sig
  --------------------------------------------------------------------------------
  BTC_delta_vol_8                    hist_mean  -0.2890↓  -0.6832    10%    1   

In [ ]:
# import warnings
# warnings.filterwarnings("ignore")

# import numpy as np
# import pandas as pd
# from itertools import combinations
# from tqdm.auto import tqdm
# from sklearn.linear_model import Ridge
# from sklearn.preprocessing import StandardScaler
# from sklearn.pipeline import Pipeline
# from sklearn.metrics import mean_absolute_error, r2_score
# from scipy import stats as sp_stats

# TICKER_MAP = {
#     "BTC":  "BTC-USDT",
#     "ETH":  "ETH-USDT",
#     "SOL":  "SOL-USDT",
#     "BNB":  "BNB-USDT",
#     "XRP":  "XRP-USDT",
#     "DOGE": "DOGE-USDT",
#     "ADA":  "ADA-USDT",
#     "AVAX": "AVAX-USDT",
#     "TRX":  "TRX-USDT",
# }
# TARGET_ASSETS  = ["BTC", "ETH"]
# UNIVERSE       = list(TICKER_MAP.keys())
# START_DATE     = "2023-01-01"
# BARS_PER_DAY   = 48
# ANN_FACTOR     = np.sqrt(365 * BARS_PER_DAY)
# VOL_WINDOW     = 48
# STRESS_MULT    = 1.25
# HORIZONS       = [8, 48, 96]         

# IT_WINDOW      = 192                  
# IT_BINS        = 10                  
# IT_K           = 4                    

# TRAIN_BARS     = 2880
# TEST_BARS      =  480
# STEP_BARS      =  240
# PURGE_BARS     =   96

# cut = pd.Timestamp(START_DATE)

# close_parts, high_parts, low_parts = {}, {}, {}
# for asset, ticker in TICKER_MAP.items():
#     if ticker not in price_df.columns:
#         print(f"WARNING: {ticker} not found, skipping")
#         continue
#     c = price_df[ticker].loc[cut:].copy()
#     valid = c.notna()
#     close_parts[asset] = c[valid]
#     high_parts[asset]  = high_df[ticker].loc[cut:][valid] if ticker in high_df.columns \
#                          else pd.Series(np.nan, index=c[valid].index)
#     low_parts[asset]   = low_df[ticker].loc[cut:][valid]  if ticker in low_df.columns  \
#                          else pd.Series(np.nan, index=c[valid].index)

# common_idx = close_parts[TARGET_ASSETS[0]].index
# for a in close_parts:
#     common_idx = common_idx.intersection(close_parts[a].index)
# print(f"Common index: {len(common_idx)} bars  "
#       f"({common_idx[0].date()} → {common_idx[-1].date()})")

# close_w = pd.DataFrame({a: close_parts[a].reindex(common_idx) for a in close_parts})
# high_w  = pd.DataFrame({a: high_parts[a].reindex(common_idx)  for a in close_parts})
# low_w   = pd.DataFrame({a: low_parts[a].reindex(common_idx)   for a in close_parts})

# ret = np.log(close_w / close_w.shift(1))
# print(f"Returns shape: {ret.shape}")

# def build_targets(ret, target_asset):
#     r       = ret[target_asset]
#     vol_now = r.rolling(VOL_WINDOW).std() * ANN_FACTOR
#     vol_med = vol_now.rolling(365 * BARS_PER_DAY, min_periods=BARS_PER_DAY * 30).median()
#     tgt     = pd.DataFrame(index=ret.index)
#     tgt["vol_now"] = vol_now
#     for h in HORIZONS:
#         vol_fwd = r.rolling(h).std().shift(-h) * ANN_FACTOR
#         tgt[f"vol_fwd_{h}"]       = vol_fwd
#         tgt[f"log_vol_fwd_{h}"]   = np.log(vol_fwd.clip(lower=1e-8))
#         tgt[f"stress_{h}"]        = (vol_fwd > vol_med * STRESS_MULT).astype(float)
#         tgt[f"delta_vol_{h}"]     = vol_fwd - vol_now
#         tgt[f"log_vol_ratio_{h}"] = np.log((vol_fwd / vol_now.clip(lower=1e-8)).clip(lower=1e-8))
#     return tgt

# targets = {a: build_targets(ret, a) for a in TARGET_ASSETS}
# for a in TARGET_ASSETS:
#     for h in HORIZONS:
#         n = targets[a][f"stress_{h}"].sum()
#         print(f"  {a} h={h}: stress {int(n)}/{len(targets[a])} ({n/len(targets[a])*100:.1f}%)")

# def build_vol_features(ret, high_w, low_w):
#     feats = {}
#     for asset in ret.columns:
#         r = ret[asset]
#         for w in [4, 8, 16, 48, 96]:
#             feats[f"{asset}_rvol_{w}"] = r.rolling(w).std() * ANN_FACTOR
#         rv = r.rolling(VOL_WINDOW).std() * ANN_FACTOR
#         feats[f"{asset}_vov"]         = rv.rolling(VOL_WINDOW).std()
#         feats[f"{asset}_vol_zscore"]  = (rv - rv.rolling(VOL_WINDOW * 4).mean()) / \
#                                          (rv.rolling(VOL_WINDOW * 4).std() + 1e-8)
#         feats[f"{asset}_vol_pctrank"] = rv.rolling(VOL_WINDOW * 30).rank(pct=True)
#         h_col = high_w[asset] if asset in high_w.columns else pd.Series(np.nan, index=r.index)
#         l_col = low_w[asset]  if asset in low_w.columns  else pd.Series(np.nan, index=r.index)
#         hl2   = np.log((h_col / l_col).clip(lower=1e-8)) ** 2
#         feats[f"{asset}_pk_vol"] = np.sqrt(hl2.rolling(VOL_WINDOW).mean() / (4 * np.log(2))) * ANN_FACTOR
#     feats["vol_now_feat"] = feats.get("BTC_rvol_48",
#                             feats[next(iter(feats))])
#     return pd.DataFrame(feats, index=ret.index)

# print("Building vol features...")
# X_vol = build_vol_features(ret, high_w, low_w)
# print(f"X_vol: {X_vol.shape}")

# def _rk2d(mat, bins):
#     rk = np.argsort(np.argsort(mat, axis=1), axis=1)
#     return (rk * bins // mat.shape[1]).clip(0, bins - 1).astype(np.int32)

# def _ffill(result):
#     mask = np.isfinite(result)
#     idx  = np.where(mask, np.arange(len(result)), 0)
#     np.maximum.accumulate(idx, out=idx)
#     return result[idx]

# def _batch_entropy(col, window, k, bins):
#     n = len(col)
#     starts = np.arange(0, n - window + 1, k)
#     ends   = starts + window
#     n_win  = len(starts)
#     wx   = np.stack([col[s:e] for s, e in zip(starts, ends)])
#     bx   = _rk2d(wx, bins)
#     off  = (np.arange(n_win) * bins).reshape(-1, 1)
#     flat = (bx + off).ravel()
#     cnt  = np.bincount(flat, minlength=n_win * bins).reshape(n_win, bins).astype(float) + 1e-10
#     p    = cnt / cnt.sum(axis=1, keepdims=True)
#     H    = -(p * np.log(p)).sum(axis=1)
#     res  = np.full(n, np.nan); res[ends - 1] = H
#     return _ffill(res)

# def _batch_mi(col_x, col_y, window, k, bins):
#     n = len(col_x)
#     starts = np.arange(0, n - window + 1, k)
#     ends   = starts + window
#     n_win  = len(starts)
#     wx = np.stack([col_x[s:e] for s, e in zip(starts, ends)])
#     wy = np.stack([col_y[s:e] for s, e in zip(starts, ends)])
#     bx   = _rk2d(wx, bins); by = _rk2d(wy, bins)
#     joint = bx * bins + by
#     bins2 = bins * bins
#     off  = (np.arange(n_win) * bins2).reshape(-1, 1)
#     flat = (joint + off).ravel()
#     cnt  = np.bincount(flat, minlength=n_win * bins2).reshape(n_win, bins2).astype(float) + 1e-10
#     p    = cnt / cnt.sum(axis=1, keepdims=True)
#     p2d  = p.reshape(n_win, bins, bins)
#     px   = p2d.sum(axis=2, keepdims=True)
#     py   = p2d.sum(axis=1, keepdims=True)
#     mi   = np.maximum((p2d * np.log(p2d / (px * py))).sum(axis=(1, 2)), 0.0)
#     res  = np.full(n, np.nan); res[ends - 1] = mi
#     return _ffill(res)

# def _batch_cond_mi(col_x, col_y, col_z, window, k, bins):
#     """I(X;Y|Z) via 3D histogram — one window at a time (unavoidable)."""
#     n = len(col_x)
#     starts = np.arange(0, n - window + 1, k)
#     ends   = starts + window
#     res    = np.full(n, np.nan)
#     for s, e in zip(starts, ends):
#         wx, wy, wz = col_x[s:e], col_y[s:e], col_z[s:e]
#         def rk(v):
#             r = np.argsort(np.argsort(v))
#             return (r * bins // len(v)).clip(0, bins - 1).astype(np.int32)
#         bx, by, bz = rk(wx), rk(wy), rk(wz)
#         c = np.zeros((bins, bins, bins))
#         np.add.at(c, (bx, by, bz), 1); c += 1e-10
#         p    = c / c.sum()
#         p_xz = c.sum(axis=1) / c.sum(axis=1).sum()
#         p_yz = c.sum(axis=0) / c.sum(axis=0).sum()
#         p_z  = c.sum(axis=(0, 1)) / c.sum(axis=(0, 1)).sum()
#         with np.errstate(divide="ignore", invalid="ignore"):
#             ratio = np.where(
#                 (p_xz[:, np.newaxis, :] > 0) & (p_yz[np.newaxis, :, :] > 0),
#                 p_z[np.newaxis, np.newaxis, :] * p /
#                 (p_xz[:, np.newaxis, :] * p_yz[np.newaxis, :, :]), 1.0)
#         res[e - 1] = max(float((p * np.log(ratio)).sum()), 0.0)
#     return _ffill(res)

# def _batch_kl(col, window, k, bins):
#     n = len(col)
#     starts = np.arange(0, n - 2 * window + 1, k)
#     res    = np.full(n, np.nan)
#     for s in starts:
#         mid, end = s + window, s + 2 * window
#         wp, wc   = col[s:mid], col[mid:end]
#         edges    = np.linspace(min(wp.min(), wc.min()) - 1e-8,
#                                max(wp.max(), wc.max()) + 1e-8, bins + 1)
#         pp, _ = np.histogram(wp, bins=edges); pp = pp.astype(float) + 1e-10; pp /= pp.sum()
#         pc, _ = np.histogram(wc, bins=edges); pc = pc.astype(float) + 1e-10; pc /= pc.sum()
#         res[end - 1] = max(float((pc * np.log(pc / pp)).sum()), 0.0)
#     return _ffill(res)

# def build_it_features(ret, window=IT_WINDOW, bins=IT_BINS, k=IT_K):
#     assets  = list(ret.columns)
#     n       = len(ret)
#     ret_np  = ret.to_numpy(dtype=float)
#     ci      = {c: i for i, c in enumerate(assets)}
#     pairs   = list(combinations(assets, 2))
#     triplets= list(combinations(assets, 3))
#     quads   = list(combinations(assets, 4))
#     feats   = {}

#     print("  [IT] per-asset (entropy, auto-NMI, KL)...")
#     for asset in tqdm(assets, desc="per-asset", leave=False):
#         col = ret_np[:, ci[asset]]
#         H   = _batch_entropy(col, window, k, bins)
#         feats[f"entropy_{asset}"] = H
#         lag = np.roll(col, 1); lag[0] = np.nan
#         ami = _batch_mi(col, lag, window, k, bins)
#         feats[f"auto_nmi_{asset}"] = np.where(H > 1e-8, ami / H, 0.0)
#         feats[f"kl_{asset}"]       = _batch_kl(col, window, k, bins)

#     print("  [IT] pairwise MI + covariance...")
#     pair_mi = {}
#     mi_arrs, cov_arrs = [], []
#     for a, b in tqdm(pairs, desc="pairs", leave=False):
#         arr = _batch_mi(ret_np[:, ci[a]], ret_np[:, ci[b]], window, k, bins)
#         pair_mi[(a, b)] = arr
#         mi_arrs.append(arr)
#         cov_arrs.append(ret[a].rolling(window).cov(ret[b]).values)

#     mi_mat  = np.column_stack(mi_arrs)
#     cov_mat = np.column_stack(cov_arrs)
#     for stat, fn in [("mean", np.nanmean), ("min", np.nanmin),
#                      ("max", np.nanmax),   ("std", np.nanstd)]:
#         feats[f"pairwise_mi_{stat}"]  = fn(mi_mat,  axis=1)
#         feats[f"pairwise_cov_{stat}"] = fn(cov_mat, axis=1)

#     print("  [IT] 3rd-order coinformation...")
#     ci3_arrs = []
#     for a, b, c in tqdm(triplets, desc="3-coinfo", leave=False):
#         mi_ab = pair_mi.get((a, b), pair_mi.get((b, a),
#                 _batch_mi(ret_np[:, ci[a]], ret_np[:, ci[b]], window, k, bins)))
#         cmi   = _batch_cond_mi(ret_np[:, ci[a]], ret_np[:, ci[b]],
#                                 ret_np[:, ci[c]], window, k, bins)
#         ci3_arrs.append(mi_ab - cmi)

#     ci3_mat = np.column_stack(ci3_arrs)
#     for stat, fn in [("mean", np.nanmean), ("min", np.nanmin),
#                      ("max", np.nanmax),   ("std", np.nanstd)]:
#         feats[f"coinfo3_{stat}"] = fn(ci3_mat, axis=1)

#     if quads:
#         print("  [IT] 4th-order coinformation...")
#         tri_idx = {t: i for i, t in enumerate(triplets)}
#         ci4_arrs = []
#         for a, b, c, d in tqdm(quads, desc="4-coinfo", leave=False):
#             sub = []
#             for trip in combinations([a, b, c, d], 3):
#                 key = tuple(sorted(trip))
#                 for t in triplets:
#                     if set(t) == set(key):
#                         sub.append(ci3_mat[:, tri_idx[t]]); break
#             if sub:
#                 ci4_arrs.append(np.nanmean(np.column_stack(sub), axis=1))
#         if ci4_arrs:
#             ci4_mat = np.column_stack(ci4_arrs)
#             for stat, fn in [("mean", np.nanmean), ("min", np.nanmin),
#                               ("max", np.nanmax),   ("std", np.nanstd)]:
#                 feats[f"coinfo4_{stat}"] = fn(ci4_mat, axis=1)

#     X_it = pd.DataFrame(feats, index=ret.index)
#     print(f"  [IT] done: {X_it.shape[1]} features × {X_it.shape[0]} bars")
#     return X_it

# print("Building IT features...")
# X_it = build_it_features(ret)
# print(f"X_it: {X_it.shape}")

# def _wf_splits(n, train, test, step, purge):
#     start = train
#     while start + purge + test <= n:
#         yield list(range(start - train, start)), \
#               list(range(start + purge, start + purge + test))
#         start += step

# class AR1:
#     def __init__(self, col="vol_now_feat"):
#         self.col = col; self.a = self.b = 0.0
#     def _v(self, X):
#         return X[self.col].values.astype(float) if self.col in X.columns \
#                else X.iloc[:, 0].values.astype(float)
#     def fit(self, X, y):
#         v, t = self._v(X), y.values.astype(float)
#         m    = np.isfinite(v) & np.isfinite(t); v, t = v[m], t[m]
#         self.a = float(np.cov(v, t)[0, 1] / (np.var(v) + 1e-12))
#         self.b = float(np.mean(t) - self.a * np.mean(v))
#         return self
#     def predict(self, X):
#         v = self._v(X)
#         return np.where(np.isfinite(v), self.a * v + self.b, self.b)

# def _run_wf(X, y, models):
#     mask = X.notna().all(axis=1) & y.notna()
#     Xc, yc = X[mask], y[mask]
#     splits = list(_wf_splits(len(Xc), TRAIN_BARS, TEST_BARS, STEP_BARS, PURGE_BARS))
#     if not splits:
#         return pd.DataFrame()
#     records = []
#     for tr, te in tqdm(splits, desc="    folds", leave=False):
#         Xtr, Xte = Xc.iloc[tr], Xc.iloc[te]
#         ytr, yte = yc.iloc[tr], yc.iloc[te]
#         for name, mdl in models.items():
#             mdl.fit(Xtr, ytr)
#             pred = mdl.predict(Xte)
#             records.append({"model": name,
#                 "corr": float(pd.Series(pred).corr(pd.Series(yte.values))),
#                 "mae":  mean_absolute_error(yte, pred),
#                 "r2":   r2_score(yte, pred)})
#     return pd.DataFrame(records)

# def _summarize(df):
#     if df.empty: return df
#     return df.groupby("model")[["corr", "mae", "r2"]].agg(["mean", "std"]).round(4)

# def _new_ridge():
#     return Pipeline([("sc", StandardScaler()), ("m", Ridge(alpha=1.0))])

# def residual_test(X_vol, X_it, y, task="auto",
#                   scale_outlier_threshold=3.0,
#                   burnin_multiplier=2):
#     """
#     OOS R² (Goyal & Welch, 2008) — адаптивный baseline + robust метрики.

#     Baseline:
#       log_vol   → AR1(vol_now)
#       delta_vol → historical mean (E[delta]=0 by construction)

#     Outlier-фолды (scale > threshold) исключаются из median/mean
#     но сохраняются в статистике для прозрачности.

#     Burn-in: первые burnin_multiplier * IT_WINDOW баров X_it
#     обнуляются в NaN — они ненадёжны пока rolling окна не заполнились.

#     Метрики (основная → медианный R²):
#       r2_median   — медиана per-fold R² без outliers  ← главная
#       r2_mean     — среднее per-fold R² без outliers
#       r2_global   — глобальный (для справки, чувствителен к выбросам)
#       pct_positive— доля фолдов с R² > 0
#       n_outliers  — число выброшенных фолдов (scale > threshold)
#     """
#     burnin = burnin_multiplier * IT_WINDOW
#     Xi_safe = X_it.copy()
#     Xi_safe.iloc[:burnin] = np.nan

#     mask = X_vol.notna().all(axis=1) & Xi_safe.notna().all(axis=1) & y.notna()
#     Xv, Xi, yc = X_vol[mask], Xi_safe[mask], y[mask]
#     splits = list(_wf_splits(len(yc), TRAIN_BARS, TEST_BARS, STEP_BARS, PURGE_BARS))

#     use_mean_baseline = (task == "delta_vol") or \
#                         (task == "auto" and abs(yc.mean()) < 0.05 * yc.std())

#     all_e_base, all_e_model = [], []
#     fold_records = []   # (r2, scale, mse_diff) per fold

#     for tr, te in splits:
#         Xv_tr, Xv_te = Xv.iloc[tr], Xv.iloc[te]
#         Xi_tr, Xi_te = Xi.iloc[tr], Xi.iloc[te]
#         ytr,   yte   = yc.iloc[tr], yc.iloc[te]

#         if use_mean_baseline:
#             train_mean = float(ytr.mean())
#             pred_base  = np.full(len(yte), train_mean)
#             res_tr     = ytr.values - train_mean
#         else:
#             ar1 = AR1(); ar1.fit(Xv_tr, ytr)
#             pred_base = ar1.predict(Xv_te)
#             res_tr    = ytr.values - ar1.predict(Xv_tr)

#         mdl = _new_ridge(); mdl.fit(Xi_tr, res_tr)
#         correction = mdl.predict(Xi_te)
#         pred_model = pred_base + correction

#         e_base  = yte.values - pred_base
#         e_model = yte.values - pred_model

#         mse_base  = float(np.mean(e_base**2))
#         mse_model = float(np.mean(e_model**2))

#         res_te = yte.values - pred_base
#         scale  = correction.std() / (res_te.std() + 1e-8)
#         r2_fold = 1.0 - mse_model / mse_base if mse_base > 1e-12 else np.nan

#         all_e_base.extend(e_base.tolist())
#         all_e_model.extend(e_model.tolist())
#         fold_records.append({
#             "r2":       r2_fold,
#             "scale":    scale,
#             "mse_diff": mse_base - mse_model,
#             "outlier":  scale > scale_outlier_threshold,
#         })

#     fold_df   = pd.DataFrame(fold_records)
#     clean     = fold_df[~fold_df["outlier"]]
#     n_outliers= int(fold_df["outlier"].sum())

#     r2_vals_clean = clean["r2"].dropna().values

#     r2_median  = float(np.median(r2_vals_clean)) if len(r2_vals_clean) else np.nan
#     r2_mean    = float(np.mean(r2_vals_clean))   if len(r2_vals_clean) else np.nan
#     pct_pos    = float((r2_vals_clean > 0).mean()) if len(r2_vals_clean) else np.nan

#     e_base_all  = np.array(all_e_base)
#     e_model_all = np.array(all_e_model)
#     r2_global   = float(1 - np.sum(e_model_all**2) / np.sum(e_base_all**2))

#     d = clean["mse_diff"].values
#     if len(d) > 1 and d.std() > 1e-10:
#         dm_t, dm_p2 = sp_stats.ttest_1samp(d, 0)
#         dm_p = float(dm_p2 / 2 if dm_t > 0 else 1 - dm_p2 / 2)
#     else:
#         dm_t, dm_p = np.nan, np.nan

#     return {
#         "r2_median":   r2_median,    # ← основная метрика
#         "r2_mean":     r2_mean,
#         "r2_global":   r2_global,
#         "pct_positive":pct_pos,
#         "n_folds":     len(splits),
#         "n_outliers":  n_outliers,
#         "dm_p":        dm_p,
#         "beats_base":  r2_median > 0,
#         "baseline":    "hist_mean" if use_mean_baseline else "AR1",
#     }

# X_combined = pd.concat([X_vol, X_it], axis=1)

# all_results   = {}
# residual_rows = []

# for asset in TARGET_ASSETS:
#     tgt = targets[asset]
#     for h in HORIZONS:
#         for task, col, task_type in [
#                 (f"delta_vol_{h}",     f"delta_vol_{h}",     "delta_vol"),
#                 (f"log_vol_{h}",       f"log_vol_fwd_{h}",   "log_vol"),
#                 (f"log_vol_ratio_{h}", f"log_vol_ratio_{h}", "delta_vol"), 
#         ]:
#             if col not in tgt.columns:
#                 continue

#             y = tgt[col].dropna()
#             idx = X_combined.index.intersection(y.index)
#             y   = y.loc[idx]
#             Xv  = X_vol.loc[idx]
#             Xi  = X_it.loc[idx]
#             Xc  = X_combined.loc[idx]

#             tag = f"{asset}_{task}"
#             print(f"\n{'='*55}")
#             print(f"  {tag}")
#             print(f"{'='*55}")

#             models_v = {"AR1": AR1(), "Ridge_vol": _new_ridge()}
#             models_i = {"Ridge_it":  _new_ridge()}
#             models_c = {"Ridge_combined": _new_ridge()}

#             print("  [A+B] AR1 + Ridge(vol)...")
#             r_v = _run_wf(Xv, y, models_v)
#             print("  [C]   Ridge(IT-only)...")
#             r_i = _run_wf(Xi, y, models_i)
#             print("  [D]   Ridge(vol+IT)...")
#             r_c = _run_wf(Xc, y, models_c)

#             summary = _summarize(pd.concat([r_v, r_i, r_c]))
#             all_results[tag] = summary
#             print(summary.to_string())

#             print("  [E]   OOS R² (Goyal-Welch)...")
#             res = residual_test(Xv, Xi, y, task=task_type)
#             res["experiment"] = tag
#             residual_rows.append(res)

#             sig = "***" if res["dm_p"]<0.01 else "**" if res["dm_p"]<0.05 \
#                   else "*" if res["dm_p"]<0.10 else ""
#             mark = "↑" if res["beats_base"] else "↓"
#             print(f"        R²_median={res['r2_median']:+.4f}{mark}  "
#                   f"R²_mean={res['r2_mean']:+.4f}  "
#                   f"pos={res['pct_positive']:.0%}  "
#                   f"outliers={res['n_outliers']}  "
#                   f"base={res['baseline']}  DM p={res['dm_p']:.3f}{sig}")

# print("\n" + "="*75)
# print("  ИТОГ — OOS R² (Goyal & Welch, 2008)")
# print("  Основная метрика: медианный per-fold R² (без outlier-фолдов scale>3)")
# print("  * p<0.10  ** p<0.05  *** p<0.01  (Diebold-Mariano, one-sided)")
# print("="*75)

# res_df = pd.DataFrame(residual_rows)

# def _sig(p):
#     if p is None or (isinstance(p, float) and np.isnan(p)): return ""
#     return "***" if p<0.01 else "**" if p<0.05 else "*" if p<0.10 else ""

# print(f"\n  {'experiment':<32} {'base':>10} {'R²med':>7} {'R²mean':>7} "
#       f"{'pos%':>6} {'out':>4}  {'DM p':>7}  sig")
# print("  " + "-"*80)
# for _, row in res_df.iterrows():
#     mark = "↑" if row["beats_base"] else "↓"
#     print(f"  {row['experiment']:<32}"
#           f"  {row['baseline']:>10}"
#           f"  {row['r2_median']:+.4f}{mark}"
#           f"  {row['r2_mean']:+.4f}"
#           f"  {row['pct_positive']:>5.0%}"
#           f"  {int(row['n_outliers']):>3}"
#           f"  {row['dm_p']:>7.4f}  {_sig(row['dm_p'])}")

# print(f"\n  R²_median > 0:   {res_df['beats_base'].sum()} / {len(res_df)}")
# print(f"  DM p < 0.05:     {(res_df['dm_p'] < 0.05).sum()} / {len(res_df)}")
# print(f"  DM p < 0.10:     {(res_df['dm_p'] < 0.10).sum()} / {len(res_df)}")
# print(f"\n  Среднее R²_median: {res_df['r2_median'].mean():+.4f}")
# print(f"  Среднее R²_mean:   {res_df['r2_mean'].mean():+.4f}")
# print(f"  Средняя pos%:      {res_df['pct_positive'].mean():.0%}")
# best = res_df.loc[res_df["r2_median"].idxmax()]
# print(f"\n  Лучший R²_median:  {best['r2_median']:+.4f}  "
#       f"({best['experiment']}, base={best['baseline']})")

Common index: 46743 bars  (2023-01-01 → 2025-09-01)
Returns shape: (46743, 9)
  BTC h=8: stress 10978/46743 (23.5%)
  BTC h=48: stress 13511/46743 (28.9%)
  BTC h=96: stress 13929/46743 (29.8%)
  ETH h=8: stress 13050/46743 (27.9%)
  ETH h=48: stress 16192/46743 (34.6%)
  ETH h=96: stress 17652/46743 (37.8%)
Building vol features...
X_vol: (46743, 82)
Building IT features...
  [IT] per-asset (entropy, auto-NMI, KL)...


  [IT] pairwise MI + covariance...


  [IT] 3rd-order coinformation...


  [IT] 4th-order coinformation...


  [IT] done: 43 features × 46743 bars
X_it: (46743, 43)

  BTC_delta_vol_8
  [A+B] AR1 + Ridge(vol)...


...

  [D]   Ridge(vol+IT)...


                  corr             mae               r2         
                  mean     std    mean     std     mean      std
model                                                           
AR1            -0.0189  0.2755  0.2966  0.1019  -1.0202   1.6629
Ridge_combined  0.0703  0.3351  0.5280  0.3668 -10.5969  43.9612
Ridge_it        0.0720  0.3429  0.4349  0.3061  -6.7620  22.5316
Ridge_vol       0.0642  0.3135  0.4983  0.2889  -8.9384  25.7415
  [E]   OOS R² (Goyal-Welch)...
        R²_median=-0.6322↓  R²_mean=-1.0951  pos=21%  outliers=7  base=AR1  DM p=1.000

  ETH_log_vol_ratio_96
  [A+B] AR1 + Ridge(vol)...


  [C]   Ridge(IT-only)...


  [D]   Ridge(vol+IT)...


                  corr             mae              r2         
                  mean     std    mean     std    mean      std
model                                                          
AR1             0.6608  0.1630  0.3126  0.0903  0.1438   0.4783
Ridge_combined  0.5325  0.2506  0.5411  0.3575 -2.6846   8.5210
Ridge_it        0.2198  0.2949  0.5147  0.2970 -2.5630  12.1374
Ridge_vol       0.5181  0.2365  0.5151  0.2956 -2.2521   5.5827
  [E]   OOS R² (Goyal-Welch)...
        R²_median=-0.4776↓  R²_mean=-1.0755  pos=16%  outliers=4  base=hist_mean  DM p=1.000

  ИТОГ — OOS R² (Goyal & Welch, 2008)
  Основная метрика: медианный per-fold R² (без outlier-фолдов scale>3)
  * p<0.10  ** p<0.05  *** p<0.01  (Diebold-Mariano, one-sided)

  experiment                             base   R²med  R²mean   pos%  out     DM p  sig
  --------------------------------------------------------------------------------
  BTC_delta_vol_8                    hist_mean  -0.2890↓  -0.6832    10%    1   1

In [38]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from itertools import combinations
from tqdm.auto import tqdm
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

TARGET_ASSETS  = ["BTC", "ETH"]
HORIZONS_TEST  = [8, 48, 96, 192, 336]
HORIZON_LABELS = {8:"4h", 48:"24h", 96:"48h", 192:"4d", 336:"7d"}
TRAIN_BARS=2880; TEST_BARS=480; STEP_BARS=240; PURGE_BARS=96
IT_BINS=10; IT_K=4; VOL_WINDOW=48; ANN_FACTOR=np.sqrt(365*48)
ASSET_FOCUS = "BTC"; TASK_FOCUS = "delta_vol"

def _splits(n):
    s = TRAIN_BARS
    while s + PURGE_BARS + TEST_BARS <= n:
        yield list(range(s-TRAIN_BARS, s)), list(range(s+PURGE_BARS, s+PURGE_BARS+TEST_BARS))
        s += STEP_BARS

def _new_ridge():
    return Pipeline([("sc", StandardScaler()), ("m", Ridge(alpha=1.0))])

def _get_y(h):
    col = f"{TASK_FOCUS}_{h}"
    if h <= 96 and col in targets[ASSET_FOCUS].columns:
        return targets[ASSET_FOCUS][col].dropna()
    elif col in targets_ext[ASSET_FOCUS].columns:
        return targets_ext[ASSET_FOCUS][col].dropna()
    return None

def _residual_corr_custom(Xv, Xi, y, baseline_fn=None):
    """
    baseline_fn: callable() → fitted-able model.
    None → AR1 на vol_now_feat.
    """
    mask = Xv.notna().all(1) & Xi.notna().all(1) & y.notna()
    Xv_, Xi_, y_ = Xv[mask], Xi[mask], y[mask]
    corrs = []
    for tr, te in _splits(len(y_)):
        ytr, yte = y_.iloc[tr], y_.iloc[te]
        Xv_tr, Xv_te = Xv_.iloc[tr], Xv_.iloc[te]
        Xi_tr, Xi_te = Xi_.iloc[tr], Xi_.iloc[te]

        if baseline_fn is None:
            v = Xv_tr["vol_now_feat"].values.astype(float)
            t = ytr.values.astype(float)
            m = np.isfinite(v) & np.isfinite(t)
            a = float(np.cov(v[m], t[m])[0,1] / (np.var(v[m])+1e-12))
            b = float(np.mean(t[m]) - a*np.mean(v[m]))
            vte = Xv_te["vol_now_feat"].values.astype(float)
            res_tr = ytr.values - np.where(np.isfinite(v), a*v+b, b)
            res_te = yte.values - np.where(np.isfinite(vte), a*vte+b, b)
        else:
            mdl_b = baseline_fn(); mdl_b.fit(Xv_tr, ytr)
            res_tr = ytr.values - mdl_b.predict(Xv_tr)
            res_te = yte.values - mdl_b.predict(Xv_te)

        mdl = _new_ridge(); mdl.fit(Xi_tr, res_tr)
        pred = mdl.predict(Xi_te)
        c = float(pd.Series(pred).corr(pd.Series(res_te)))
        corrs.append(c if np.isfinite(c) else 0.0)
    return float(np.nanmean(corrs))

def _rk2d(mat, bins):
    rk = np.argsort(np.argsort(mat, axis=1), axis=1)
    return (rk * bins // mat.shape[1]).clip(0, bins-1).astype(np.int32)

def _ffill(r):
    mask = np.isfinite(r); idx = np.where(mask, np.arange(len(r)), 0)
    np.maximum.accumulate(idx, out=idx); return r[idx]

def _batch_mi(cx, cy, W, k, bins):
    n=len(cx); starts=np.arange(0,n-W+1,k); ends=starts+W; nw=len(starts)
    wx=np.stack([cx[s:e] for s,e in zip(starts,ends)])
    wy=np.stack([cy[s:e] for s,e in zip(starts,ends)])
    bx=_rk2d(wx,bins); by=_rk2d(wy,bins); j=bx*bins+by; b2=bins*bins
    off=(np.arange(nw)*b2).reshape(-1,1)
    cnt=np.bincount((j+off).ravel(),minlength=nw*b2).reshape(nw,b2).astype(float)+1e-10
    p=cnt/cnt.sum(1,keepdims=True); p2d=p.reshape(nw,bins,bins)
    mi=np.maximum((p2d*np.log(p2d/(p2d.sum(2,keepdims=True)*p2d.sum(1,keepdims=True)))).sum((1,2)),0)
    res=np.full(n,np.nan); res[ends-1]=mi; return _ffill(res)

def _batch_cond_mi(cx, cy, cz, W, k, bins):
    n=len(cx); starts=np.arange(0,n-W+1,k); ends=starts+W; res=np.full(n,np.nan)
    for s,e in zip(starts,ends):
        def rk(v): r=np.argsort(np.argsort(v)); return (r*bins//len(v)).clip(0,bins-1).astype(np.int32)
        bx,by,bz=rk(cx[s:e]),rk(cy[s:e]),rk(cz[s:e])
        c=np.zeros((bins,bins,bins)); np.add.at(c,(bx,by,bz),1); c+=1e-10
        p=c/c.sum(); pxz=c.sum(1)/c.sum(1).sum(); pyz=c.sum(0)/c.sum(0).sum()
        pz=c.sum((0,1))/c.sum((0,1)).sum()
        with np.errstate(divide="ignore",invalid="ignore"):
            ratio=np.where((pxz[:,np.newaxis,:]>0)&(pyz[np.newaxis,:,:]>0),
                pz[np.newaxis,np.newaxis,:]*p/(pxz[:,np.newaxis,:]*pyz[np.newaxis,:,:]),1.0)
        res[e-1]=max(float((p*np.log(ratio)).sum()),0.0)
    return _ffill(res)

def _batch_kl(col, W, k, bins):
    n=len(col); starts=np.arange(0,n-2*W+1,k); res=np.full(n,np.nan)
    for s in starts:
        mid,end=s+W,s+2*W; wp,wc=col[s:mid],col[mid:end]
        edges=np.linspace(min(wp.min(),wc.min())-1e-8,max(wp.max(),wc.max())+1e-8,bins+1)
        pp,_=np.histogram(wp,bins=edges); pp=pp.astype(float)+1e-10; pp/=pp.sum()
        pc,_=np.histogram(wc,bins=edges); pc=pc.astype(float)+1e-10; pc/=pc.sum()
        res[end-1]=max(float((pc*np.log(pc/pp)).sum()),0.0)
    return _ffill(res)

def build_it_slim(ret, window=192, bins=IT_BINS, k=IT_K):
    """Slim X_it: только KL + pairwise MI/cov + coinfo 3/4 (без entropy/auto-NMI)."""
    assets=list(ret.columns); ret_np=ret.to_numpy(dtype=float)
    ci={c:i for i,c in enumerate(assets)}
    pairs=list(combinations(assets,2)); triplets=list(combinations(assets,3))
    feats={}

    for asset in assets:
        feats[f"kl_{asset}"]=_batch_kl(ret_np[:,ci[asset]],window,k,bins)

    pair_mi={}; mi_arrs=[]; cov_arrs=[]
    for a,b in pairs:
        arr=_batch_mi(ret_np[:,ci[a]],ret_np[:,ci[b]],window,k,bins)
        pair_mi[(a,b)]=arr; mi_arrs.append(arr)
        cov_arrs.append(ret[a].rolling(window).cov(ret[b]).values)
    mi_mat=np.column_stack(mi_arrs); cov_mat=np.column_stack(cov_arrs)
    for stat,fn in [("mean",np.nanmean),("min",np.nanmin),("max",np.nanmax),("std",np.nanstd)]:
        feats[f"pairwise_mi_{stat}"]=fn(mi_mat,axis=1)
        feats[f"pairwise_cov_{stat}"]=fn(cov_mat,axis=1)

    ci3_arrs=[]; tri_list=[]
    for a,b,c in triplets:
        mi_ab=pair_mi.get((a,b),pair_mi.get((b,a)))
        cmi=_batch_cond_mi(ret_np[:,ci[a]],ret_np[:,ci[b]],ret_np[:,ci[c]],window,k,bins)
        ci3_arrs.append(mi_ab-cmi); tri_list.append((a,b,c))
    ci3_mat=np.column_stack(ci3_arrs)
    tri_idx={t:i for i,t in enumerate(tri_list)}
    for stat,fn in [("mean",np.nanmean),("min",np.nanmin),("max",np.nanmax),("std",np.nanstd)]:
        feats[f"coinfo3_{stat}"]=fn(ci3_mat,axis=1)

    quads=list(combinations(assets,4)); ci4_arrs=[]
    for a,b,c,d in quads:
        sub=[]
        for trip in combinations([a,b,c,d],3):
            for t in tri_list:
                if set(t)==set(trip): sub.append(ci3_mat[:,tri_idx[t]]); break
        if sub: ci4_arrs.append(np.nanmean(np.column_stack(sub),axis=1))
    if ci4_arrs:
        ci4_mat=np.column_stack(ci4_arrs)
        for stat,fn in [("mean",np.nanmean),("min",np.nanmin),("max",np.nanmax),("std",np.nanstd)]:
            feats[f"coinfo4_{stat}"]=fn(ci4_mat,axis=1)

    return pd.DataFrame(feats, index=ret.index)

print("="*60)
print("  EXP 1 — Адаптивное окно IT под горизонт (BTC delta_vol)")
print("="*60)

MULTIPLIERS = [1, 2, 4, 8]

print("\n  Baseline (window=192, X_it_full):")
baseline_corr = {}
for h in tqdm(HORIZONS_TEST, desc="baseline", leave=False):
    y = _get_y(h)
    if y is None: continue
    idx = X_vol.index.intersection(X_it.index).intersection(y.index)
    valid = X_vol.loc[idx].notna().all(1) & X_it.loc[idx].notna().all(1) & y.loc[idx].notna()
    baseline_corr[h] = _residual_corr_custom(X_vol.loc[idx[valid]], X_it.loc[idx[valid]], y.loc[idx[valid]])
    print(f"    h={HORIZON_LABELS[h]}: {baseline_corr[h]:.4f}")

window_cache = {}
unique_windows = sorted(set(min(max(m*h,48),480) for m in MULTIPLIERS for h in HORIZONS_TEST))
print(f"\n  Строим IT для окон: {unique_windows}...")
for w in tqdm(unique_windows, desc="windows"):
    window_cache[w] = build_it_slim(ret, window=w)

exp1 = {}
for mult in MULTIPLIERS:
    for h in HORIZONS_TEST:
        w = min(max(mult*h,48),480)
        Xi_w = window_cache[w].loc[X_vol.index]
        y = _get_y(h)
        if y is None: continue
        idx = X_vol.index.intersection(Xi_w.index).intersection(y.index)
        valid = X_vol.loc[idx].notna().all(1) & Xi_w.loc[idx].notna().all(1) & y.loc[idx].notna()
        exp1[(mult,h)] = _residual_corr_custom(X_vol.loc[idx[valid]], Xi_w.loc[idx[valid]], y.loc[idx[valid]])

print(f"\n  {'k':<8}", end="")
for h in HORIZONS_TEST: print(f"  {HORIZON_LABELS[h]:>7}", end="")
print()
print("  " + "-"*48)
print(f"  {'base':<8}", end="")
for h in HORIZONS_TEST: print(f"  {baseline_corr.get(h,0):>+7.4f}", end="")
print()
for m in MULTIPLIERS:
    print(f"  {m}×h     ", end="")
    for h in HORIZONS_TEST:
        v = exp1.get((m,h),0)
        mark = "✓" if v > baseline_corr.get(h,0)+0.005 else " "
        print(f"  {v:>+7.4f}{mark}", end="")
    print()

best1 = {}
print(f"\n  Прирост vs baseline:")
for h in HORIZONS_TEST:
    bk = max(MULTIPLIERS, key=lambda m: exp1.get((m,h),-1))
    bv = exp1.get((bk,h),0)
    base = baseline_corr.get(h,0)
    best1[h] = (bk, bv)
    print(f"  h={HORIZON_LABELS[h]:<4}: k={bk}×  corr={bv:.4f}  Δ={bv-base:+.4f}")

print("\n" + "="*60)
print("  EXP 2 — Сильный baseline (Ridge_vol_k3/5/10)")
print("  Более чистые остатки → сигнал IT должен вырасти")
print("="*60)

def make_ridge_baseline(k):
    def factory():
        mdl = _new_ridge()
        cols = [None]   # mutable container for closure
        def fit(X, y):
            corr = X.corrwith(y).abs().replace([np.inf,-np.inf], np.nan).dropna()
            cols[0] = corr.nlargest(k).index.tolist()
            mdl.fit(X[cols[0]], y)
        def predict(X):
            return mdl.predict(X[cols[0]])
        obj = type("B", (), {"fit": staticmethod(fit), "predict": staticmethod(predict)})()
        return obj
    return factory

BASELINES = {
    "AR1(vol_now)":  None,
    "Ridge_vol_k3":  make_ridge_baseline(3),
    "Ridge_vol_k5":  make_ridge_baseline(5),
    "Ridge_vol_k10": make_ridge_baseline(10),
}
HORIZONS_EXP2 = [48, 96, 192]

exp2 = {name:{} for name in BASELINES}
for h in tqdm(HORIZONS_EXP2, desc="h"):
    y = _get_y(h)
    if y is None: continue
    idx = X_vol.index.intersection(X_it.index).intersection(y.index)
    valid = X_vol.loc[idx].notna().all(1) & X_it.loc[idx].notna().all(1) & y.loc[idx].notna()
    Xv_=X_vol.loc[idx[valid]]; Xi_=X_it.loc[idx[valid]]; y_=y.loc[idx[valid]]
    for name,factory in BASELINES.items():
        exp2[name][h] = _residual_corr_custom(Xv_, Xi_, y_, factory)

print(f"\n  {'baseline':<18}", end="")
for h in HORIZONS_EXP2: print(f"  {HORIZON_LABELS[h]:>8}", end="")
print()
print("  " + "-"*44)
for name in BASELINES:
    print(f"  {name:<18}", end="")
    for h in HORIZONS_EXP2:
        v = exp2[name].get(h, np.nan)
        ar1_v = exp2["AR1(vol_now)"].get(h, 0)
        mark = "↑" if name!="AR1(vol_now)" and v>ar1_v+0.005 else \
               ("↓" if name!="AR1(vol_now)" and v<ar1_v-0.005 else " ")
        print(f"  {v:>+8.4f}{mark}", end="")
    print()

print("\n" + "="*60)
print("  EXP 3 — Slim X_it (27 фич) vs full (43 фичи)")
print("  Убраны: entropy (0% важности), auto-NMI (~5% важности)")
print("="*60)

print("\n  Строим X_it_slim (window=192)...")
X_it_slim = build_it_slim(ret, window=192)
print(f"  slim: {X_it_slim.shape[1]} фич  |  full: {X_it.shape[1]} фич")

exp3 = {}
for h in tqdm(HORIZONS_TEST, desc="h"):
    y = _get_y(h)
    if y is None: continue
    for label, Xi in [("full", X_it), ("slim", X_it_slim)]:
        idx = X_vol.index.intersection(Xi.index).intersection(y.index)
        valid = X_vol.loc[idx].notna().all(1) & Xi.loc[idx].notna().all(1) & y.loc[idx].notna()
        exp3[(label,h)] = _residual_corr_custom(X_vol.loc[idx[valid]], Xi.loc[idx[valid]], y.loc[idx[valid]])

print(f"\n  {'version':<10}", end="")
for h in HORIZONS_TEST: print(f"  {HORIZON_LABELS[h]:>7}", end="")
print()
print("  " + "-"*48)
for label in ["full","slim"]:
    print(f"  {label:<10}", end="")
    for h in HORIZONS_TEST:
        v = exp3.get((label,h),np.nan)
        if label=="slim":
            fv = exp3.get(("full",h),0)
            mark = "↑" if v>fv+0.003 else ("↓" if v<fv-0.003 else "~")
        else: mark = " "
        print(f"  {v:>+7.4f}{mark}", end="")
    print()

print("\n" + "="*60)
print("  ИТОГ — прирост по каждому направлению")
print("="*60)

avg_base  = np.nanmean(list(baseline_corr.values()))
avg_exp1  = np.nanmean([best1[h][1] for h in HORIZONS_TEST if h in best1])
best_bl   = max(BASELINES, key=lambda n: np.nanmean([exp2[n].get(h,0) for h in HORIZONS_EXP2]))
avg_exp2  = np.nanmean([exp2[best_bl].get(h,0) for h in HORIZONS_EXP2])
avg_exp3f = np.nanmean([exp3.get(("full",h),0) for h in HORIZONS_TEST])
avg_exp3s = np.nanmean([exp3.get(("slim",h),0) for h in HORIZONS_TEST])

print(f"\n  Текущий baseline (full, w=192):      {avg_base:.4f}")
print(f"  EXP 1 — адаптивное окно (лучший k): {avg_exp1:.4f}  Δ={avg_exp1-avg_base:+.4f}")
print(f"  EXP 2 — {best_bl:<22}:{avg_exp2:.4f}  Δ={avg_exp2-avg_base:+.4f}  (h=24h/48h/4d)")
print(f"  EXP 3 — slim X_it (27 фич):         {avg_exp3s:.4f}  Δ={avg_exp3s-avg_exp3f:+.4f}")
print(f"\n  Рекомендация: взять лучший k из EXP1 + slim из EXP3")
print(f"  Следующий шаг: combined = adaptive_window × slim_features")

  EXP 1 — Адаптивное окно IT под горизонт (BTC delta_vol)

  Baseline (window=192, X_it_full):


baseline:  20%|███████████████████████████████▊                                                                                                                               | 1/5 [00:00<00:02,  1.50it/s]

    h=4h: 0.0317


baseline:  40%|███████████████████████████████████████████████████████████████▌                                                                                               | 2/5 [00:01<00:02,  1.49it/s]

    h=24h: 0.0793


...


  k              4h      24h      48h       4d       7d
  ------------------------------------------------
  base      +0.0317  +0.0793  +0.1189  +0.1083  +0.0745
  1×h       +0.0122   +0.0414   +0.0261   +0.1076   +0.0453 
  2×h       +0.0122   +0.0334   +0.1209   +0.0522   +0.0408 
  4×h       +0.0122   +0.0958✓  +0.0527   +0.0721   +0.0408 
  8×h       +0.0115   +0.0447   +0.0688   +0.0721   +0.0408 

  Прирост vs baseline:
  h=4h  : k=1×  corr=0.0122  Δ=-0.0194
  h=24h : k=4×  corr=0.0958  Δ=+0.0166
  h=48h : k=2×  corr=0.1209  Δ=+0.0020
  h=4d  : k=1×  corr=0.1076  Δ=-0.0006
  h=7d  : k=1×  corr=0.0453  Δ=-0.0292

  EXP 2 — Сильный baseline (Ridge_vol_k3/5/10)
  Более чистые остатки → сигнал IT должен вырасти


h: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:20<00:00,  6.95s/it]



  baseline                 24h       48h        4d
  --------------------------------------------
  AR1(vol_now)         +0.0793    +0.1189    +0.1083 
  Ridge_vol_k3         +0.0809    +0.1166    +0.1073 
  Ridge_vol_k5         +0.0767    +0.1166    +0.1089 
  Ridge_vol_k10        +0.0514↓   +0.1053↓   +0.1154↑

  EXP 3 — Slim X_it (27 фич) vs full (43 фичи)
  Убраны: entropy (0% важности), auto-NMI (~5% важности)

  Строим X_it_slim (window=192)...
  slim: 25 фич  |  full: 43 фич


h: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:06<00:00,  1.24s/it]


  version          4h      24h      48h       4d       7d
  ------------------------------------------------
  full        +0.0317   +0.0793   +0.1189   +0.1083   +0.0745 
  slim        +0.0428↑  +0.0958↑  +0.1209~  +0.1076~  +0.0674↓

  ИТОГ — прирост по каждому направлению

  Текущий baseline (full, w=192):      0.0825
  EXP 1 — адаптивное окно (лучший k): 0.0764  Δ=-0.0061
  EXP 2 — AR1(vol_now)          :0.1021  Δ=+0.0196  (h=24h/48h/4d)
  EXP 3 — slim X_it (27 фич):         0.0869  Δ=+0.0044

  Рекомендация: взять лучший k из EXP1 + slim из EXP3
  Следующий шаг: combined = adaptive_window × slim_features


In [39]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from itertools import combinations
from tqdm.auto import tqdm
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

HORIZONS       = [8, 48, 96, 192, 336]
HORIZON_LABELS = {8:"4h", 48:"24h", 96:"48h", 192:"4d", 336:"7d"}
TARGET_ASSETS  = ["BTC", "ETH"]
TRAIN_BARS=2880; TEST_BARS=480; STEP_BARS=240; PURGE_BARS=96

def _splits(n):
    s = TRAIN_BARS
    while s + PURGE_BARS + TEST_BARS <= n:
        yield list(range(s-TRAIN_BARS, s)), list(range(s+PURGE_BARS, s+PURGE_BARS+TEST_BARS))
        s += STEP_BARS

def _new_ridge():
    return Pipeline([("sc", StandardScaler()), ("m", Ridge(alpha=1.0))])

try:
    _ = X_it_slim
    print(f"X_it_slim уже в памяти: {X_it_slim.shape[1]} фич")
except NameError:
    print("X_it_slim не найден, строим...")
    X_it_slim = build_it_slim(ret, window=192)
    print(f"X_it_slim: {X_it_slim.shape[1]} фич")

class RidgeVolBaseline:
    """AR1 + топ-3 vol фичи по корреляции с таргетом (in-train отбор)."""
    def __init__(self):
        self.mdl = _new_ridge()
        self.cols = None
    def fit(self, X, y):
        corr = X.corrwith(y).abs().replace([np.inf,-np.inf], np.nan).dropna()
        self.cols = corr.nlargest(3).index.tolist()
        self.mdl.fit(X[self.cols], y)
        return self
    def predict(self, X):
        return self.mdl.predict(X[self.cols])

def residual_corr_full(Xv, Xi, y, use_ridge_base=True):
    """Возвращает mean corr и per-fold corrs."""
    mask = Xv.notna().all(1) & Xi.notna().all(1) & y.notna()
    Xv_, Xi_, y_ = Xv[mask], Xi[mask], y[mask]
    corrs = []
    for tr, te in _splits(len(y_)):
        ytr, yte = y_.iloc[tr], y_.iloc[te]
        Xv_tr, Xv_te = Xv_.iloc[tr], Xv_.iloc[te]
        Xi_tr, Xi_te = Xi_.iloc[tr], Xi_.iloc[te]

        if use_ridge_base:
            base = RidgeVolBaseline(); base.fit(Xv_tr, ytr)
            res_tr = ytr.values - base.predict(Xv_tr)
            res_te = yte.values - base.predict(Xv_te)
        else:
            v = Xv_tr["vol_now_feat"].values.astype(float)
            t = ytr.values.astype(float)
            m = np.isfinite(v) & np.isfinite(t)
            a = float(np.cov(v[m], t[m])[0,1] / (np.var(v[m])+1e-12))
            b = float(np.mean(t[m]) - a*np.mean(v[m]))
            vte = Xv_te["vol_now_feat"].values.astype(float)
            res_tr = ytr.values - np.where(np.isfinite(v), a*v+b, b)
            res_te = yte.values - np.where(np.isfinite(vte), a*vte+b, b)

        mdl = _new_ridge(); mdl.fit(Xi_tr, res_tr)
        pred = mdl.predict(Xi_te)
        c = float(pd.Series(pred).corr(pd.Series(res_te)))
        corrs.append(c if np.isfinite(c) else 0.0)

    return float(np.nanmean(corrs)), corrs

def _get_y(asset, h):
    col = f"delta_vol_{h}"
    if h <= 96 and col in targets[asset].columns:
        return targets[asset][col].dropna()
    elif col in targets_ext[asset].columns:
        return targets_ext[asset][col].dropna()
    return None

print("\n" + "="*65)
print("  ФИНАЛЬНЫЙ ТЕСТ: slim X_it + Ridge_vol_k3 baseline")
print("  Сравнение с текущим baseline (AR1 + full X_it)")
print("="*65)

results = []

for asset in TARGET_ASSETS:
    for h in tqdm(HORIZONS, desc=asset):
        y = _get_y(asset, h)
        if y is None: continue

        idx_full = X_vol.index.intersection(X_it.index).intersection(y.index)
        idx_slim = X_vol.index.intersection(X_it_slim.index).intersection(y.index)

        vf = X_vol.loc[idx_full].notna().all(1) & X_it.loc[idx_full].notna().all(1) & y.loc[idx_full].notna()
        vs = X_vol.loc[idx_slim].notna().all(1) & X_it_slim.loc[idx_slim].notna().all(1) & y.loc[idx_slim].notna()

        c_a, _ = residual_corr_full(X_vol.loc[idx_full[vf]], X_it.loc[idx_full[vf]],
                                    y.loc[idx_full[vf]], use_ridge_base=False)

        c_b, _ = residual_corr_full(X_vol.loc[idx_slim[vs]], X_it_slim.loc[idx_slim[vs]],
                                    y.loc[idx_slim[vs]], use_ridge_base=True)

        results.append({
            "asset": asset, "h": h,
            "horizon": HORIZON_LABELS[h],
            "AR1_full":   round(c_a, 4),
            "Ridge_slim": round(c_b, 4),
            "delta":      round(c_b - c_a, 4),
            "improved":   c_b > c_a + 0.003,
        })

df = pd.DataFrame(results)

print(f"\n  {'asset':<5} {'h':<6} {'AR1+full':>10} {'Ridge+slim':>11} {'Δ':>7}  улучшение")
print("  " + "-"*52)
for _, row in df.iterrows():
    mark = "↑" if row["improved"] else ("~" if abs(row["delta"]) <= 0.003 else "↓")
    print(f"  {row['asset']:<5} {row['horizon']:<6} "
          f"  {row['AR1_full']:>+9.4f}  {row['Ridge_slim']:>+10.4f}"
          f"  {row['delta']:>+6.4f}  {mark}")

print(f"\n  Улучшений (Δ > 0.003): {df['improved'].sum()} / {len(df)}")
print(f"  Среднее AR1+full:     {df['AR1_full'].mean():.4f}")
print(f"  Среднее Ridge+slim:   {df['Ridge_slim'].mean():.4f}")
print(f"  Средний Δ:            {df['delta'].mean():+.4f}")

best = df.loc[df["Ridge_slim"].idxmax()]
print(f"\n  Лучший Ridge+slim:    {best['Ridge_slim']:.4f}  "
      f"({best['asset']} h={best['horizon']})")
print(f"  vs baseline:          {best['AR1_full']:.4f}  "
      f"Δ={best['Ridge_slim']-best['AR1_full']:+.4f}")

X_it_slim уже в памяти: 25 фич

  ФИНАЛЬНЫЙ ТЕСТ: slim X_it + Ridge_vol_k3 baseline
  Сравнение с текущим baseline (AR1 + full X_it)


ETH: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:13<00:00,  2.66s/it]


  asset h        AR1+full  Ridge+slim       Δ  улучшение
  ----------------------------------------------------
  BTC   4h         +0.0317     +0.0399  +0.0082  ↑
  BTC   24h        +0.0793     +0.0960  +0.0167  ↑
  BTC   48h        +0.1189     +0.1165  -0.0024  ~
  BTC   4d         +0.1083     +0.1044  -0.0038  ↓
  BTC   7d         +0.0745     +0.0752  +0.0007  ~
  ETH   4h         +0.0321     +0.0377  +0.0055  ↑
  ETH   24h        +0.0626     +0.0857  +0.0232  ↑
  ETH   48h        +0.0967     +0.1444  +0.0476  ↑
  ETH   4d         +0.1041     +0.1552  +0.0511  ↑
  ETH   7d         +0.0615     +0.0942  +0.0327  ↑

  Улучшений (Δ > 0.003): 7 / 10
  Среднее AR1+full:     0.0770
  Среднее Ridge+slim:   0.0949
  Средний Δ:            +0.0180

  Лучший Ridge+slim:    0.1552  (ETH h=4d)
  vs baseline:          0.1041  Δ=+0.0511


In [ ]:
# import warnings; warnings.filterwarnings("ignore")
# import numpy as np
# import pandas as pd
# from tqdm.auto import tqdm
# from sklearn.linear_model import Ridge
# from sklearn.preprocessing import StandardScaler
# from sklearn.pipeline import Pipeline
# import matplotlib
# matplotlib.use("Agg")
# import matplotlib.pyplot as plt

# ALL_ASSETS     = ["BTC", "ETH", "XRP", "SOL", "BNB", "DOGE"]
# HORIZONS_FOCUS = [48, 96, 192]   # 24h / 48h / 4d — где сигнал сильнее
# HORIZON_LABELS = {8:"4h", 48:"24h", 96:"48h", 192:"4d", 336:"7d"}
# TRAIN_BARS=2880; TEST_BARS=480; STEP_BARS=240; PURGE_BARS=96

# def _splits(n):
#     s = TRAIN_BARS
#     while s + PURGE_BARS + TEST_BARS <= n:
#         yield list(range(s-TRAIN_BARS, s)), list(range(s+PURGE_BARS, s+PURGE_BARS+TEST_BARS))
#         s += STEP_BARS

# def _new_ridge():
#     return Pipeline([("sc", StandardScaler()), ("m", Ridge(alpha=1.0))])

# class RidgeVolBaseline:
#     def __init__(self):
#         self.mdl = _new_ridge(); self.cols = None
#     def fit(self, X, y):
#         corr = X.corrwith(y).abs().replace([np.inf,-np.inf], np.nan).dropna()
#         self.cols = corr.nlargest(3).index.tolist()
#         self.mdl.fit(X[self.cols], y); return self
#     def predict(self, X):
#         return self.mdl.predict(X[self.cols])

# def get_fold_corrs(Xv, Xi, y):
#     """Возвращает corr по каждому фолду + дату начала test-периода."""
#     mask = Xv.notna().all(1) & Xi.notna().all(1) & y.notna()
#     Xv_, Xi_, y_ = Xv[mask], Xi[mask], y[mask]
#     fold_corrs, fold_dates = [], []
#     for tr, te in _splits(len(y_)):
#         ytr, yte = y_.iloc[tr], y_.iloc[te]
#         Xv_tr, Xv_te = Xv_.iloc[tr], Xv_.iloc[te]
#         Xi_tr, Xi_te = Xi_.iloc[tr], Xi_.iloc[te]
#         base = RidgeVolBaseline(); base.fit(Xv_tr, ytr)
#         res_tr = ytr.values - base.predict(Xv_tr)
#         res_te = yte.values - base.predict(Xv_te)
#         mdl = _new_ridge(); mdl.fit(Xi_tr, res_tr)
#         pred = mdl.predict(Xi_te)
#         c = float(pd.Series(pred).corr(pd.Series(res_te)))
#         fold_corrs.append(c if np.isfinite(c) else 0.0)
#         fold_dates.append(y_.index[te[0]])
#     return np.array(fold_corrs), fold_dates

# def _get_y(asset, h):
#     col = f"delta_vol_{h}"
#     for src in [targets_extra, targets, targets_ext]:
#         if asset in src and col in src[asset].columns:
#             return src[asset][col].dropna()
#     return None

# print("Собираем per-fold корреляции...")
# fold_data = {}   # (asset, h) → (corrs, dates)

# for asset in tqdm(ALL_ASSETS, desc="assets"):
#     for h in HORIZONS_FOCUS:
#         y = _get_y(asset, h)
#         if y is None: continue
#         idx = X_vol.index.intersection(X_it_slim.index).intersection(y.index)
#         valid = (X_vol.loc[idx].notna().all(1) &
#                  X_it_slim.loc[idx].notna().all(1) &
#                  y.loc[idx].notna())
#         idx = idx[valid]
#         if len(idx) < TRAIN_BARS + TEST_BARS + PURGE_BARS: continue
#         corrs, dates = get_fold_corrs(X_vol.loc[idx], X_it_slim.loc[idx], y.loc[idx])
#         fold_data[(asset, h)] = (corrs, dates)

# print("\n" + "="*70)
# print("  РАСПРЕДЕЛЕНИЕ PER-FOLD КОРРЕЛЯЦИЙ")
# print("="*70)

# print(f"\n  {'актив':<6} {'h':<5} {'mean':>7} {'median':>8} {'std':>7} "
#       f"{'p25':>7} {'p75':>7} {'pos%':>6} {'>0.1':>6}")
# print("  " + "-"*62)

# stats_rows = []
# for asset in ALL_ASSETS:
#     for h in HORIZONS_FOCUS:
#         if (asset, h) not in fold_data: continue
#         corrs, _ = fold_data[(asset, h)]
#         stats_rows.append({
#             "asset":  asset,
#             "h":      HORIZON_LABELS[h],
#             "mean":   np.mean(corrs),
#             "median": np.median(corrs),
#             "std":    np.std(corrs),
#             "p25":    np.percentile(corrs, 25),
#             "p75":    np.percentile(corrs, 75),
#             "pos_pct":np.mean(corrs > 0),
#             "gt01":   np.mean(corrs > 0.1),
#             "n":      len(corrs),
#         })
#     if asset in ["ETH", "XRP"]: 
#         print("  " + "·"*62)

# stats_df = pd.DataFrame(stats_rows)
# for _, row in stats_df.iterrows():
#     print(f"  {row['asset']:<6} {row['h']:<5} "
#           f"  {row['mean']:>+6.3f}  {row['median']:>+7.3f}  {row['std']:>6.3f}"
#           f"  {row['p25']:>+6.3f}  {row['p75']:>+6.3f}"
#           f"  {row['pos_pct']:>5.0%}  {row['gt01']:>5.0%}")

# print(f"\n  Сводка по группам активов (все горизонты):")
# for group, assets in [("BTC/ETH", ["BTC","ETH"]),
#                       ("XRP/SOL/BNB/DOGE", ["XRP","SOL","BNB","DOGE"])]:
#     g = stats_df[stats_df.asset.isin(assets)]
#     print(f"  {group:<18}: median={g['median'].mean():+.3f}  "
#           f"pos%={g['pos_pct'].mean():.0%}  >0.1: {g['gt01'].mean():.0%} фолдов")

# print("\n  Строим графики...")

# fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharey=True)
# axes = axes.flatten()

# COLORS = {
#     "BTC": "#1f77b4", "ETH": "#2ca02c",
#     "XRP": "#d62728", "SOL": "#ff7f0e",
#     "BNB": "#9467bd", "DOGE": "#8c564b",
# }

# H_FOCUS = 96   # 48h — показываем детально

# for ax_i, asset in enumerate(ALL_ASSETS):
#     ax = axes[ax_i]
#     if (asset, H_FOCUS) not in fold_data:
#         ax.set_visible(False); continue

#     corrs, dates = fold_data[(asset, H_FOCUS)]
#     dates_pd = pd.DatetimeIndex(dates)
#     color = COLORS[asset]

#     ax.scatter(dates_pd, corrs, s=8, alpha=0.4, color=color)

#     roll = pd.Series(corrs).rolling(20, min_periods=5).mean()
#     ax.plot(dates_pd, roll.values, color=color, linewidth=2, label="rolling mean")

#     ax.axhline(0,    color="black", linewidth=0.5, alpha=0.4)
#     ax.axhline(0.03, color="red",   linewidth=0.8, linestyle=":", alpha=0.6)
#     ax.axhline(np.mean(corrs), color=color, linewidth=1.2,
#                linestyle="--", alpha=0.8, label=f"mean={np.mean(corrs):.3f}")

#     pos_pct = np.mean(corrs > 0)
#     ax.set_title(f"{asset}  h=48h  |  pos={pos_pct:.0%}  mean={np.mean(corrs):+.3f}",
#                  fontsize=11)
#     ax.set_xlabel("test period start", fontsize=9)
#     if ax_i % 3 == 0: ax.set_ylabel("per-fold corr", fontsize=9)
#     ax.legend(fontsize=8, loc="upper right")
#     ax.spines["top"].set_visible(False)
#     ax.spines["right"].set_visible(False)
#     ax.grid(alpha=0.2, linestyle="--")

# plt.suptitle("Per-fold residual correlation — h=48h, slim X_it + Ridge_vol_k3",
#              fontsize=13, y=1.01)
# plt.tight_layout()
# plt.savefig("fig_fold_dist_h48.png", bbox_inches="tight", dpi=150)
# plt.savefig("fig_fold_dist_h48.pdf", bbox_inches="tight", dpi=150)
# plt.show()
# print("  Saved: fig_fold_dist_h48.png / .pdf")

# fig2, axes2 = plt.subplots(1, 3, figsize=(14, 5), sharey=True)

# for ax_j, h in enumerate(HORIZONS_FOCUS):
#     ax = axes2[ax_j]
#     data_plot  = []
#     labels_plot= []
#     colors_plot= []

#     for asset in ALL_ASSETS:
#         if (asset, h) not in fold_data: continue
#         corrs, _ = fold_data[(asset, h)]
#         data_plot.append(corrs)
#         labels_plot.append(asset)
#         colors_plot.append(COLORS[asset])

#     bp = ax.boxplot(data_plot, labels=labels_plot, patch_artist=True,
#                     medianprops=dict(color="black", linewidth=1.5),
#                     flierprops=dict(marker=".", markersize=3, alpha=0.4),
#                     whiskerprops=dict(linewidth=0.8),
#                     boxprops=dict(linewidth=0.8))
#     for patch, color in zip(bp["boxes"], colors_plot):
#         patch.set_facecolor(color); patch.set_alpha(0.6)

#     ax.axhline(0,    color="black", linewidth=0.5, alpha=0.4)
#     ax.axhline(0.03, color="red",   linewidth=0.8, linestyle=":", alpha=0.6,
#                label="threshold 0.03")
#     ax.set_title(f"h={HORIZON_LABELS[h]}", fontsize=11)
#     ax.set_xlabel("asset")
#     if ax_j == 0: ax.set_ylabel("per-fold corr")
#     ax.legend(fontsize=8)
#     ax.spines["top"].set_visible(False)
#     ax.spines["right"].set_visible(False)
#     ax.grid(alpha=0.2, linestyle="--", axis="y")

# plt.suptitle("Distribution of per-fold residual corr — all assets",
#              fontsize=13, y=1.01)
# plt.tight_layout()
# plt.savefig("fig_boxplot_assets.png", bbox_inches="tight", dpi=150)
# plt.savefig("fig_boxplot_assets.pdf", bbox_inches="tight", dpi=150)
# plt.show()
# print("  Saved: fig_boxplot_assets.png / .pdf")

Собираем per-fold корреляции...


assets: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6/6 [00:36<00:00,  6.02s/it]



  РАСПРЕДЕЛЕНИЕ PER-FOLD КОРРЕЛЯЦИЙ

  актив  h        mean   median     std     p25     p75   pos%   >0.1
  --------------------------------------------------------------
  ······························································
  ······························································
  BTC    24h     +0.096   +0.111   0.258  -0.109  +0.278    64%    51%
  BTC    48h     +0.117   +0.124   0.315  -0.090  +0.349    64%    52%
  BTC    4d      +0.104   +0.109   0.386  -0.206  +0.448    60%    51%
  ETH    24h     +0.086   +0.093   0.269  -0.076  +0.265    62%    49%
  ETH    48h     +0.144   +0.184   0.317  -0.064  +0.388    68%    57%
  ETH    4d      +0.155   +0.175   0.382  -0.164  +0.495    63%    56%
  XRP    24h     +0.044   +0.052   0.272  -0.113  +0.225    57%    43%
  XRP    48h     +0.029   +0.030   0.347  -0.224  +0.284    54%    43%
  XRP    4d      +0.018   +0.040   0.413  -0.324  +0.340    53%    46%
  SOL    24h     +0.078   +0.091   0.268  -0.096  +0.276  

In [42]:
# import warnings; warnings.filterwarnings("ignore")
# import numpy as np
# import pandas as pd
# from tqdm.auto import tqdm
# from sklearn.linear_model import Ridge, LogisticRegression
# from sklearn.preprocessing import StandardScaler
# from sklearn.pipeline import Pipeline
# from sklearn.metrics import (roc_auc_score, average_precision_score,
#                              precision_recall_curve, f1_score,
#                              balanced_accuracy_score)
# from scipy import stats as sp_stats

# ALL_ASSETS     = ["BTC", "ETH", "XRP", "SOL", "BNB", "DOGE"]
# HORIZONS       = [48, 96, 192]
# HORIZON_LABELS = {8:"4h", 48:"24h", 96:"48h", 192:"4d", 336:"7d"}
# TRAIN_BARS=2880; TEST_BARS=480; STEP_BARS=240; PURGE_BARS=96
# VOL_WINDOW=48; ANN_FACTOR=np.sqrt(365*48); BARS_PER_DAY=48

# def _splits(n):
#     s = TRAIN_BARS
#     while s + PURGE_BARS + TEST_BARS <= n:
#         yield list(range(s-TRAIN_BARS,s)), list(range(s+PURGE_BARS,s+PURGE_BARS+TEST_BARS))
#         s += STEP_BARS

# def _new_ridge():
#     return Pipeline([("sc", StandardScaler()), ("m", Ridge(alpha=1.0))])

# def _new_logreg(C=0.1):
#     return Pipeline([("sc", StandardScaler()),
#                      ("m", LogisticRegression(C=C, max_iter=1000,
#                                               class_weight="balanced",
#                                               solver="lbfgs"))])

# class RidgeVolBaseline:
#     def __init__(self, k=3):
#         self.k=k; self.mdl=_new_ridge(); self.cols=None
#     def fit(self, X, y):
#         corr = X.corrwith(y).abs().replace([np.inf,-np.inf],np.nan).dropna()
#         self.cols = corr.nlargest(self.k).index.tolist()
#         self.mdl.fit(X[self.cols], y); return self
#     def predict(self, X): return self.mdl.predict(X[self.cols])

# def _get_tgt(asset, col):
#     for src in [targets_extra if hasattr(globals(),'targets_extra') else {},
#                 targets, targets_ext]:
#         if asset in src and col in src[asset].columns:
#             return src[asset][col].dropna()
#     return None

# try:
#     _ = targets_extra
# except NameError:
#     print("targets_extra не найден — строим таргеты для XRP/SOL/BNB/DOGE...")
#     targets_extra = {}
#     for asset in ["XRP","SOL","BNB","DOGE"]:
#         r = ret[asset]
#         vol_now = r.rolling(VOL_WINDOW).std() * ANN_FACTOR
#         vol_med = vol_now.rolling(365*BARS_PER_DAY,min_periods=BARS_PER_DAY*30).median()
#         tgt = pd.DataFrame(index=ret.index)
#         tgt["vol_now"] = vol_now
#         for h in [8,48,96,192,336]:
#             vol_fwd = r.rolling(h).std().shift(-h) * ANN_FACTOR
#             tgt[f"delta_vol_{h}"]   = vol_fwd - vol_now
#             tgt[f"stress_{h}"]      = (vol_fwd > vol_med*1.25).astype(float)
#             tgt[f"log_vol_fwd_{h}"] = np.log(vol_fwd.clip(lower=1e-8))
#         targets_extra[asset] = tgt
#     print("  done")

# print("="*65)
# print("  EXP A — Stress classification (PR-AUC, несбалансированный)")
# print("  Baseline: persistence (vol_now → стресс)")
# print("  Model: LogReg(IT) напрямую на stress-таргет")
# print("="*65)

# def run_stress_cls(Xv, Xi, y_cls):
#     """
#     Baseline: vol_now percentile как score → PR-AUC, ROC-AUC
#     Model: LogReg(IT-only), LogReg(vol+IT)
#     Метрики для несбалансированных классов:
#       PR-AUC  — основная (не зависит от порога, учитывает дисбаланс)
#       ROC-AUC — для сравнения
#       F1 @ threshold=best — лучший F1 по кривой
#     """
#     mask = Xv.notna().all(1) & Xi.notna().all(1) & y_cls.notna()
#     Xv_,Xi_,y_ = Xv[mask],Xi[mask],y_cls[mask]
#     Xc_ = pd.concat([Xv_,Xi_],axis=1)

#     pos_rate = float(y_.mean())
#     splits = list(_splits(len(y_)))

#     metrics = {k:[] for k in ["pr_auc_base","pr_auc_it","pr_auc_comb",
#                                "roc_auc_base","roc_auc_it","roc_auc_comb"]}

#     for tr,te in splits:
#         if len(y_.iloc[te].unique()) < 2: continue
#         ytr,yte = y_.iloc[tr], y_.iloc[te]
#         if ytr.sum() < 5: continue   # слишком мало позитивных в трейне

#         if "vol_now_feat" in Xv_.columns:
#             vol_te = Xv_.iloc[te]["vol_now_feat"].values.astype(float)
#             vol_min,vol_max = Xv_.iloc[tr]["vol_now_feat"].min(), Xv_.iloc[tr]["vol_now_feat"].max()
#             score_base = np.clip((vol_te - vol_min)/(vol_max - vol_min + 1e-8), 0, 1)
#         else:
#             score_base = np.ones(len(yte)) * ytr.mean()

#         mdl_it = _new_logreg(); mdl_it.fit(Xi_.iloc[tr], ytr)
#         score_it = mdl_it.predict_proba(Xi_.iloc[te])[:,1]

#         mdl_comb = _new_logreg(); mdl_comb.fit(Xc_.iloc[tr], ytr)
#         score_comb = mdl_comb.predict_proba(Xc_.iloc[te])[:,1]

#         for name, score in [("base",score_base),("it",score_it),("comb",score_comb)]:
#             try:
#                 metrics[f"pr_auc_{name}"].append(average_precision_score(yte,score))
#                 metrics[f"roc_auc_{name}"].append(roc_auc_score(yte,score))
#             except: pass

#     return {k: float(np.nanmean(v)) for k,v in metrics.items()}, pos_rate, len(splits)

# exp_a = []
# for asset in tqdm(ALL_ASSETS, desc="EXP A"):
#     for h in HORIZONS:
#         col = f"stress_{h}"
#         y_cls = _get_tgt(asset, col)
#         if y_cls is None: continue
#         idx = X_vol.index.intersection(X_it_slim.index).intersection(y_cls.index)
#         valid = (X_vol.loc[idx].notna().all(1) &
#                  X_it_slim.loc[idx].notna().all(1) & y_cls.loc[idx].notna())
#         idx = idx[valid]
#         if len(idx) < TRAIN_BARS+TEST_BARS+PURGE_BARS: continue
#         m, pos_rate, n_folds = run_stress_cls(
#             X_vol.loc[idx], X_it_slim.loc[idx], y_cls.loc[idx])
#         exp_a.append({"asset":asset,"horizon":HORIZON_LABELS[h],"h":h,
#                       "pos_rate":round(pos_rate,3), "n_folds":n_folds, **{k:round(v,4) for k,v in m.items()}})

# df_a = pd.DataFrame(exp_a)

# print(f"\n  PR-AUC (основная метрика для несбалансированных классов):")
# print(f"  {'актив':<6} {'h':<5} {'pos%':>5} {'base':>8} {'IT':>8} {'vol+IT':>8}  ΔIT")
# print("  " + "-"*52)
# for _,row in df_a.iterrows():
#     delta = row['pr_auc_it'] - row['pr_auc_base']
#     mark = "↑" if delta > 0.01 else ("~" if abs(delta) <= 0.01 else "↓")
#     print(f"  {row['asset']:<6} {row['horizon']:<5} {row['pos_rate']:>4.0%}"
#           f"  {row['pr_auc_base']:>8.4f}  {row['pr_auc_it']:>8.4f}"
#           f"  {row['pr_auc_comb']:>8.4f}  {delta:>+5.3f} {mark}")

# print(f"\n  Среднее PR-AUC baseline: {df_a['pr_auc_base'].mean():.4f}")
# print(f"  Среднее PR-AUC IT:       {df_a['pr_auc_it'].mean():.4f}")
# print(f"  Среднее PR-AUC vol+IT:   {df_a['pr_auc_comb'].mean():.4f}")
# best_a = df_a.loc[df_a['pr_auc_comb'].idxmax()]
# print(f"  Лучший vol+IT:  {best_a['asset']} h={best_a['horizon']}  "
#       f"PR-AUC={best_a['pr_auc_comb']:.4f}  (pos%={best_a['pos_rate']:.0%})")

# print("\n" + "="*65)
# print("  EXP B — Regime change prediction")
# print("  Таргет: переход из нижнего квантиля vol в верхний")
# print("  (vol_now < Q40 → vol_fwd > Q70) за горизонт h")
# print("="*65)

# def build_regime_change(ret, asset, h, q_low=0.40, q_high=0.70):
#     """
#     1 если: сейчас тихо (vol_now < Q_low) И скоро громко (vol_fwd > Q_high)
#     Это самый редкий и ценный случай — поймать начало стресса из тишины.
#     """
#     r = ret[asset]
#     vol_now = r.rolling(VOL_WINDOW).std() * ANN_FACTOR
#     vol_fwd = r.rolling(h).std().shift(-h) * ANN_FACTOR
#     q_now_low  = vol_now.expanding(min_periods=BARS_PER_DAY*30).quantile(q_low)
#     q_fwd_high = vol_fwd.expanding(min_periods=BARS_PER_DAY*30).quantile(q_high)
#     regime_change = ((vol_now < q_now_low) & (vol_fwd > q_fwd_high)).astype(float)
#     return regime_change

# def run_regime_cls(Xv, Xi, y_cls):
#     mask = Xv.notna().all(1) & Xi.notna().all(1) & y_cls.notna()
#     Xv_,Xi_,y_ = Xv[mask],Xi[mask],y_cls[mask]
#     Xc_ = pd.concat([Xv_,Xi_],axis=1)
#     splits = list(_splits(len(y_)))

#     pr_base,pr_it,pr_comb = [],[],[]
#     roc_it,roc_comb = [],[]

#     for tr,te in splits:
#         if len(y_.iloc[te].unique()) < 2: continue
#         ytr,yte = y_.iloc[tr], y_.iloc[te]
#         if ytr.sum() < 3: continue

#         if "vol_now_feat" in Xv_.columns:
#             vol_te = Xv_.iloc[te]["vol_now_feat"].values.astype(float)
#             vol_max = Xv_.iloc[tr]["vol_now_feat"].max()
#             score_base = 1.0 - np.clip(vol_te / (vol_max+1e-8), 0, 1)
#         else:
#             score_base = np.ones(len(yte)) * ytr.mean()

#         mdl_it = _new_logreg(); mdl_it.fit(Xi_.iloc[tr], ytr)
#         score_it = mdl_it.predict_proba(Xi_.iloc[te])[:,1]

#         mdl_comb = _new_logreg(); mdl_comb.fit(Xc_.iloc[tr], ytr)
#         score_comb = mdl_comb.predict_proba(Xc_.iloc[te])[:,1]

#         try:
#             pr_base.append(average_precision_score(yte, score_base))
#             pr_it.append(average_precision_score(yte, score_it))
#             pr_comb.append(average_precision_score(yte, score_comb))
#             roc_it.append(roc_auc_score(yte, score_it))
#             roc_comb.append(roc_auc_score(yte, score_comb))
#         except: pass

#     return {
#         "pr_base": float(np.nanmean(pr_base)),
#         "pr_it":   float(np.nanmean(pr_it)),
#         "pr_comb": float(np.nanmean(pr_comb)),
#         "roc_it":  float(np.nanmean(roc_it)),
#         "roc_comb":float(np.nanmean(roc_comb)),
#         "pos_rate":float(y_.mean()),
#         "n_folds": len(splits),
#     }

# exp_b = []
# for asset in tqdm(ALL_ASSETS, desc="EXP B"):
#     for h in HORIZONS:
#         y_rc = build_regime_change(ret, asset, h)
#         y_rc = y_rc.dropna()
#         idx = X_vol.index.intersection(X_it_slim.index).intersection(y_rc.index)
#         valid = (X_vol.loc[idx].notna().all(1) &
#                  X_it_slim.loc[idx].notna().all(1) & y_rc.loc[idx].notna())
#         idx = idx[valid]
#         if len(idx) < TRAIN_BARS+TEST_BARS+PURGE_BARS: continue
#         m = run_regime_cls(X_vol.loc[idx], X_it_slim.loc[idx], y_rc.loc[idx])
#         exp_b.append({"asset":asset,"horizon":HORIZON_LABELS[h],"h":h,
#                       **{k:round(v,4) for k,v in m.items()}})

# df_b = pd.DataFrame(exp_b)

# print(f"\n  Regime change: vol_now<Q40 → vol_fwd>Q70")
# print(f"  PR-AUC и ROC-AUC (LogReg vol+IT vs baseline):")
# print(f"\n  {'актив':<6} {'h':<5} {'pos%':>5} {'PR_base':>8} {'PR_comb':>8} {'ROC_comb':>9}  lift")
# print("  " + "-"*55)
# for _,row in df_b.iterrows():
#     lift = row['pr_comb']/row['pr_base'] if row['pr_base']>0 else 1.0
#     mark = "★" if lift > 1.3 else ("↑" if lift > 1.1 else "~")
#     print(f"  {row['asset']:<6} {row['horizon']:<5} {row['pos_rate']:>4.0%}"
#           f"  {row['pr_base']:>8.4f}  {row['pr_comb']:>8.4f}"
#           f"  {row['roc_comb']:>9.4f}  ×{lift:.2f} {mark}")

# best_b = df_b.loc[df_b['pr_comb'].idxmax()]
# print(f"\n  Лучший: {best_b['asset']} h={best_b['horizon']}  "
#       f"PR-AUC={best_b['pr_comb']:.4f}  ROC={best_b['roc_comb']:.4f}  "
#       f"lift=×{best_b['pr_comb']/max(best_b['pr_base'],0.001):.2f}")

# print("\n" + "="*65)
# print("  EXP C — Conditional analysis")
# print("  Когда IT-фичи работают лучше всего?")
# print("  Условия: синхронизация рынка / vol-режим / тренд")
# print("="*65)

# ASSET_C, H_C = "BTC", 96
# y_c = _get_tgt(ASSET_C, f"delta_vol_{H_C}")
# if y_c is None:
#     y_c = targets[ASSET_C][f"delta_vol_{H_C}"].dropna()

# idx = X_vol.index.intersection(X_it_slim.index).intersection(y_c.index)
# valid = (X_vol.loc[idx].notna().all(1) &
#          X_it_slim.loc[idx].notna().all(1) & y_c.loc[idx].notna())
# idx = idx[valid]
# Xv_c = X_vol.loc[idx]; Xi_c = X_it_slim.loc[idx]; y_c = y_c.loc[idx]

# sync = Xi_c["pairwise_cov_mean"] if "pairwise_cov_mean" in Xi_c.columns \
#        else Xi_c.iloc[:,0]
# vol_now_c = Xv_c["vol_now_feat"] if "vol_now_feat" in Xv_c.columns \
#             else Xv_c.iloc[:,0]
# vol_rank = vol_now_c.rolling(BARS_PER_DAY*30, min_periods=100).rank(pct=True)
# vol_trend = vol_now_c - vol_now_c.shift(BARS_PER_DAY*5)

# fold_records_c = []
# for tr,te in _splits(len(y_c)):
#     ytr,yte = y_c.iloc[tr], y_c.iloc[te]
#     Xv_tr,Xv_te = Xv_c.iloc[tr], Xv_c.iloc[te]
#     Xi_tr,Xi_te = Xi_c.iloc[tr], Xi_c.iloc[te]

#     base = RidgeVolBaseline(); base.fit(Xv_tr, ytr)
#     res_tr = ytr.values - base.predict(Xv_tr)
#     res_te = yte.values - base.predict(Xv_te)

#     mdl = _new_ridge(); mdl.fit(Xi_tr, res_tr)
#     pred = mdl.predict(Xi_te)
#     corr = float(pd.Series(pred).corr(pd.Series(res_te)))

#     t0 = te[0]
#     fold_records_c.append({
#         "corr":      corr if np.isfinite(corr) else 0.0,
#         "sync":      float(sync.iloc[t0]) if t0 < len(sync) else np.nan,
#         "vol_rank":  float(vol_rank.iloc[t0]) if t0 < len(vol_rank) else np.nan,
#         "vol_trend": float(vol_trend.iloc[t0]) if t0 < len(vol_trend) else np.nan,
#     })

# fdf = pd.DataFrame(fold_records_c).dropna()

# print(f"\n  {ASSET_C} delta_vol h={HORIZON_LABELS[H_C]}  ({len(fdf)} фолдов)")
# print(f"  Среднее corr (все фолды): {fdf['corr'].mean():+.4f}")

# def conditional_corr(fdf, col, q_low=0.33, q_high=0.67, label=""):
#     low  = fdf[fdf[col] < fdf[col].quantile(q_low)]["corr"]
#     mid  = fdf[(fdf[col] >= fdf[col].quantile(q_low)) & (fdf[col] < fdf[col].quantile(q_high))]["corr"]
#     high = fdf[fdf[col] >= fdf[col].quantile(q_high)]["corr"]
#     print(f"\n  Условие: {label}")
#     print(f"  {'терциль':<12} {'n':>4} {'mean corr':>11} {'pos%':>6}")
#     print("  " + "-"*36)
#     for name, grp in [("низкий", low), ("средний", mid), ("высокий", high)]:
#         if len(grp) == 0: continue
#         print(f"  {name:<12} {len(grp):>4}  {grp.mean():>+10.4f}  {(grp>0).mean():>5.0%}")
#     if len(low) > 5 and len(high) > 5:
#         t, p = sp_stats.ttest_ind(high, low)
#         print(f"  high vs low: t={t:.2f}  p={p:.3f}  "
#               f"{'значимо *' if p<0.10 else ''}")

# conditional_corr(fdf, "sync",      label="Синхронизация активов (pairwise_cov_mean)")
# conditional_corr(fdf, "vol_rank",  label="Текущий vol-режим (vol_now percentile)")
# conditional_corr(fdf, "vol_trend", label="Тренд волатильности (vol_now - vol[-5d])")

# print(f"""
#   Интерпретация:
#   Если corr выше при высокой синхронизации → IT-фичи работают когда
#   рынок "синхронизирован" (перед стрессом), а не в тихие периоды.

#   Если corr выше при низком vol-режиме → IT-фичи предсказывают
#   именно выход из тишины, а не продолжение уже высокой волы.

#   Если corr выше при растущем vol-тренде → IT улавливает ускорение
#   синхронизации в начале стресс-эпизода.
# """)

# print("="*65)
# print("  ИТОГОВОЕ СРАВНЕНИЕ ЭКСПЕРИМЕНТОВ")
# print("="*65)

# print(f"""
#   EXP A — Stress classification:
#     Лучший PR-AUC (vol+IT): {df_a['pr_auc_comb'].max():.4f}
#     vs baseline:             {df_a.loc[df_a['pr_auc_comb'].idxmax(),'pr_auc_base']:.4f}
#     Актив/горизонт:          {best_a['asset']} h={best_a['horizon']}

#   EXP B — Regime change (vol_now<Q40 → vol_fwd>Q70):
#     Лучший PR-AUC (vol+IT): {df_b['pr_comb'].max():.4f}
#     vs baseline:             {df_b.loc[df_b['pr_comb'].idxmax(),'pr_base']:.4f}
#     Актив/горизонт:          {best_b['asset']} h={best_b['horizon']}

#   EXP C — Conditional analysis ({ASSET_C} delta_vol h={HORIZON_LABELS[H_C]}):
#     Средняя corr (все):      {fdf['corr'].mean():+.4f}
#     Смотри таблицы выше для условных результатов.

#   Главный вывод: IT-сигнал концентрируется в специфических рыночных
#   условиях — это важнее чем безусловное среднее.
# """)

# df_exp_a = df_a.copy()
# df_exp_b = df_b.copy()
# df_exp_c = fdf.copy()
# print("df_exp_a, df_exp_b, df_exp_c сохранены в памяти")

  EXP A — Stress classification (PR-AUC, несбалансированный)
  Baseline: persistence (vol_now → стресс)
  Model: LogReg(IT) напрямую на stress-таргет


EXP A: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6/6 [00:22<00:00,  3.72s/it]



  PR-AUC (основная метрика для несбалансированных классов):
  актив  h      pos%     base       IT   vol+IT  ΔIT
  ----------------------------------------------------
  BTC    24h    30%    0.4164    0.4139    0.3916  -0.003 ~
  BTC    48h    31%    0.4277    0.4801    0.4742  +0.052 ↑
  BTC    4d     32%    0.4358    0.5428    0.5172  +0.107 ↑
  ETH    24h    36%    0.4568    0.4403    0.4615  -0.016 ↓
  ETH    48h    39%    0.4621    0.4855    0.4825  +0.023 ↑
  ETH    4d     40%    0.4871    0.5368    0.5485  +0.050 ↑

  Среднее PR-AUC baseline: 0.4476
  Среднее PR-AUC IT:       0.4832
  Среднее PR-AUC vol+IT:   0.4792
  Лучший vol+IT:  ETH h=4d  PR-AUC=0.5485  (pos%=40%)

  EXP B — Regime change prediction
  Таргет: переход из нижнего квантиля vol в верхний
  (vol_now < Q40 → vol_fwd > Q70) за горизонт h


EXP B: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6/6 [00:44<00:00,  7.45s/it]



  Regime change: vol_now<Q40 → vol_fwd>Q70
  PR-AUC и ROC-AUC (LogReg vol+IT vs baseline):

  актив  h      pos%  PR_base  PR_comb  ROC_comb  lift
  -------------------------------------------------------
  BTC    24h     5%    0.3058    0.2737     0.7442  ×0.90 ~
  BTC    48h     6%    0.4821    0.3604     0.7921  ×0.75 ~
  BTC    4d      8%    0.5935    0.4299     0.8108  ×0.72 ~
  ETH    24h     5%    0.2786    0.3044     0.7801  ×1.09 ~
  ETH    48h     6%    0.3781    0.3646     0.7735  ×0.96 ~
  ETH    4d      7%    0.5206    0.4363     0.8334  ×0.84 ~
  XRP    24h     5%    0.1863    0.2056     0.6951  ×1.10 ↑
  XRP    48h     6%    0.2801    0.2513     0.7087  ×0.90 ~
  XRP    4d      7%    0.4236    0.3931     0.7416  ×0.93 ~
  SOL    24h     4%    0.2003    0.2036     0.6695  ×1.02 ~
  SOL    48h     4%    0.2583    0.2351     0.6927  ×0.91 ~
  SOL    4d      4%    0.3628    0.3349     0.7306  ×0.92 ~
  BNB    24h     4%    0.2404    0.2212     0.7069  ×0.92 ~
  BNB    48h  

In [ ]:
# import warnings; warnings.filterwarnings("ignore")
# import numpy as np
# import pandas as pd
# from tqdm.auto import tqdm
# from sklearn.linear_model import Ridge
# from sklearn.preprocessing import StandardScaler
# from sklearn.pipeline import Pipeline
# from scipy import stats as sp_stats
# import matplotlib
# matplotlib.use("Agg")
# import matplotlib.pyplot as plt

# ALL_ASSETS     = ["BTC", "ETH", "XRP", "SOL", "BNB", "DOGE"]
# HORIZONS       = [48, 96, 192]
# HORIZON_LABELS = {48:"24h", 96:"48h", 192:"4d"}
# TRAIN_BARS=2880; TEST_BARS=480; STEP_BARS=240; PURGE_BARS=96
# VOL_WINDOW=48; ANN_FACTOR=np.sqrt(365*48); BARS_PER_DAY=48

# COLORS = {"BTC":"#1f77b4","ETH":"#2ca02c","XRP":"#d62728",
#           "SOL":"#ff7f0e","BNB":"#9467bd","DOGE":"#8c564b"}

# def _splits(n):
#     s = TRAIN_BARS
#     while s + PURGE_BARS + TEST_BARS <= n:
#         yield list(range(s-TRAIN_BARS,s)), list(range(s+PURGE_BARS,s+PURGE_BARS+TEST_BARS))
#         s += STEP_BARS

# def _new_ridge():
#     return Pipeline([("sc", StandardScaler()), ("m", Ridge(alpha=1.0))])

# class RidgeVolBaseline:
#     def __init__(self, k=3):
#         self.k=k; self.mdl=_new_ridge(); self.cols=None
#     def fit(self, X, y):
#         corr = X.corrwith(y).abs().replace([np.inf,-np.inf],np.nan).dropna()
#         self.cols = corr.nlargest(self.k).index.tolist()
#         self.mdl.fit(X[self.cols], y); return self
#     def predict(self, X): return self.mdl.predict(X[self.cols])

# def _get_y(asset, h):
#     col = f"delta_vol_{h}"
#     for src in [targets_extra if 'targets_extra' in dir() else {},
#                 targets, targets_ext]:
#         if asset in src and col in src[asset].columns:
#             return src[asset][col].dropna()
#     return None

# try: _ = targets_extra
# except NameError: targets_extra = {}

# def get_fold_records(asset, h):
#     y = _get_y(asset, h)
#     if y is None: return None

#     idx = X_vol.index.intersection(X_it_slim.index).intersection(y.index)
#     valid = (X_vol.loc[idx].notna().all(1) &
#              X_it_slim.loc[idx].notna().all(1) & y.loc[idx].notna())
#     idx = idx[valid]
#     if len(idx) < TRAIN_BARS + TEST_BARS + PURGE_BARS: return None

#     Xv_ = X_vol.loc[idx]
#     Xi_ = X_it_slim.loc[idx]
#     y_  = y.loc[idx]

#     vol_now = Xv_["vol_now_feat"] if "vol_now_feat" in Xv_.columns else Xv_.iloc[:,0]
#     vol_rank  = vol_now.rolling(BARS_PER_DAY*30, min_periods=100).rank(pct=True)
#     vol_trend = vol_now - vol_now.shift(BARS_PER_DAY*5)

#     sync_col = "pairwise_cov_mean" if "pairwise_cov_mean" in Xi_.columns else Xi_.columns[0]
#     sync = Xi_[sync_col]
#     sync_rank = sync.rolling(BARS_PER_DAY*30, min_periods=100).rank(pct=True)

#     records = []
#     for tr, te in _splits(len(y_)):
#         ytr, yte = y_.iloc[tr], y_.iloc[te]
#         Xv_tr, Xv_te = Xv_.iloc[tr], Xv_.iloc[te]
#         Xi_tr, Xi_te = Xi_.iloc[tr], Xi_.iloc[te]

#         base = RidgeVolBaseline(); base.fit(Xv_tr, ytr)
#         res_tr = ytr.values - base.predict(Xv_tr)
#         res_te = yte.values - base.predict(Xv_te)

#         mdl = _new_ridge(); mdl.fit(Xi_tr, res_tr)
#         pred = mdl.predict(Xi_te)
#         corr = float(pd.Series(pred).corr(pd.Series(res_te)))

#         t0 = te[0]
#         records.append({
#             "corr":       corr if np.isfinite(corr) else 0.0,
#             "vol_rank":   float(vol_rank.iloc[t0])  if t0 < len(vol_rank)  else np.nan,
#             "vol_trend":  float(vol_trend.iloc[t0]) if t0 < len(vol_trend) else np.nan,
#             "sync_rank":  float(sync_rank.iloc[t0]) if t0 < len(sync_rank) else np.nan,
#             "date":       y_.index[t0],
#         })

#     return pd.DataFrame(records).dropna()

# print("Собираем per-fold данные для всех активов...")
# all_records = {}
# for asset in tqdm(ALL_ASSETS, desc="assets"):
#     for h in HORIZONS:
#         df = get_fold_records(asset, h)
#         if df is not None and len(df) > 30:
#             all_records[(asset, h)] = df

# print("\n" + "="*65)
# print("  CONDITIONAL CORR BY VOL-REGIME — все активы")
# print("  Вопрос: в каком vol-режиме IT-сигнал сильнее?")
# print("="*65)

# summary_rows = []
# for asset in ALL_ASSETS:
#     for h in HORIZONS:
#         if (asset, h) not in all_records: continue
#         df = all_records[(asset, h)]
#         low  = df[df.vol_rank < 0.33]["corr"]
#         high = df[df.vol_rank > 0.67]["corr"]
#         if len(low) < 5 or len(high) < 5: continue
#         t, p = sp_stats.ttest_ind(high, low)
#         summary_rows.append({
#             "asset":    asset,
#             "horizon":  HORIZON_LABELS[h],
#             "h":        h,
#             "corr_all": df["corr"].mean(),
#             "corr_low": low.mean(),
#             "corr_high":high.mean(),
#             "delta":    high.mean() - low.mean(),
#             "p_val":    p,
#             "sig":      "*" if p < 0.10 else "",
#         })

# sumdf = pd.DataFrame(summary_rows)

# print(f"\n  {'актив':<6} {'h':<5} {'all':>7} {'low vol':>8} {'high vol':>9} {'Δ':>7}  p   sig")
# print("  " + "-"*58)
# prev_asset = None
# for _, row in sumdf.iterrows():
#     if prev_asset and prev_asset != row["asset"]:
#         print("  " + "·"*58)
#     prev_asset = row["asset"]
#     print(f"  {row['asset']:<6} {row['horizon']:<5}"
#           f"  {row['corr_all']:>+6.3f}"
#           f"  {row['corr_low']:>+7.3f}"
#           f"  {row['corr_high']:>+8.3f}"
#           f"  {row['delta']:>+6.3f}"
#           f"  {row['p_val']:>5.3f} {row['sig']}")

# print(f"\n  Средний Δ (high vol - low vol): {sumdf['delta'].mean():+.4f}")
# print(f"  Значимых (p<0.10): {(sumdf['p_val']<0.10).sum()} / {len(sumdf)}")
# print(f"  Δ > 0 (high vol лучше): {(sumdf['delta']>0).sum()} / {len(sumdf)}")

# print("\n" + "="*65)
# print("  КОМБИНИРОВАННОЕ УСЛОВИЕ: high vol + растущая синхронизация")
# print("  Гипотеза: сигнал максимален когда ОБА условия выполнены")
# print("="*65)

# combo_rows = []
# for asset in ALL_ASSETS:
#     for h in HORIZONS:
#         if (asset, h) not in all_records: continue
#         df = all_records[(asset, h)]

#         df["vol_hi"]  = df["vol_rank"]  > 0.60
#         df["sync_hi"] = df["sync_rank"] > 0.60

#         q_hh = df[ df.vol_hi &  df.sync_hi]["corr"]   
#         q_hl = df[ df.vol_hi & ~df.sync_hi]["corr"]   
#         q_lh = df[~df.vol_hi &  df.sync_hi]["corr"]  
#         q_ll = df[~df.vol_hi & ~df.sync_hi]["corr"]  

#         if min(len(q_hh), len(q_ll)) < 5: continue

#         t_hh_ll, p_hh_ll = sp_stats.ttest_ind(q_hh, q_ll) if len(q_hh)>3 and len(q_ll)>3 else (0,1)

#         combo_rows.append({
#             "asset":    asset,
#             "horizon":  HORIZON_LABELS[h],
#             "h":        h,
#             "n_hh":     len(q_hh),
#             "corr_hh":  q_hh.mean() if len(q_hh) else np.nan,  # best case
#             "corr_ll":  q_ll.mean() if len(q_ll) else np.nan,  # worst case
#             "corr_all": df["corr"].mean(),
#             "delta_hh_ll": q_hh.mean() - q_ll.mean() if len(q_hh) and len(q_ll) else np.nan,
#             "p_hh_ll":  p_hh_ll,
#             "sig":      "**" if p_hh_ll<0.05 else ("*" if p_hh_ll<0.10 else ""),
#         })

# cdf = pd.DataFrame(combo_rows)

# print(f"\n  {'актив':<6} {'h':<5} {'all':>7} {'HH (best)':>10} {'LL (worst)':>11} {'Δ':>8}  p    sig")
# print("  " + "-"*65)
# prev = None
# for _, row in cdf.iterrows():
#     if prev and prev != row["asset"]: print("  " + "·"*65)
#     prev = row["asset"]
#     print(f"  {row['asset']:<6} {row['horizon']:<5}"
#           f"  {row['corr_all']:>+6.3f}"
#           f"  {row['corr_hh']:>+9.3f}"
#           f"  {row['corr_ll']:>+10.3f}"
#           f"  {row['delta_hh_ll']:>+7.3f}"
#           f"  {row['p_hh_ll']:>5.3f} {row['sig']}")

# print(f"\n  Средний corr в режиме 'high vol + high sync': {cdf['corr_hh'].mean():+.4f}")
# print(f"  Средний corr в режиме 'low vol + low sync':   {cdf['corr_ll'].mean():+.4f}")
# print(f"  Средний Δ:                                    {cdf['delta_hh_ll'].mean():+.4f}")
# print(f"  Значимых (p<0.10): {(cdf['p_hh_ll']<0.10).sum()} / {len(cdf)}")

# print("\n  Строим графики...")

# plt.rcParams.update({
#     "figure.facecolor":"white","axes.facecolor":"white",
#     "axes.spines.top":False,"axes.spines.right":False,
#     "axes.grid":True,"grid.alpha":0.2,"grid.linestyle":"--",
#     "font.size":10,
# })

# fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
# for ax_j, h in enumerate(HORIZONS):
#     ax = axes[ax_j]
#     all_low, all_mid, all_high = [], [], []
#     labels_box = []
#     for asset in ALL_ASSETS:
#         if (asset, h) not in all_records: continue
#         df = all_records[(asset, h)]
#         all_low.append(df[df.vol_rank < 0.33]["corr"].values)
#         all_mid.append(df[(df.vol_rank >= 0.33) & (df.vol_rank < 0.67)]["corr"].values)
#         all_high.append(df[df.vol_rank >= 0.67]["corr"].values)
#         labels_box.append(asset)

#     x = np.arange(len(labels_box))
#     w = 0.25
#     for i, (data_list, label, color) in enumerate([
#         (all_low,  "low vol",  "#85B7EB"),
#         (all_mid,  "mid vol",  "#888780"),
#         (all_high, "high vol", "#E24B4A"),
#     ]):
#         positions = [xi + (i-1)*w for xi in x]
#         bp = ax.boxplot(data_list, positions=positions, widths=w*0.9,
#                         patch_artist=True, showfliers=False,
#                         medianprops=dict(color="black",linewidth=1.5),
#                         boxprops=dict(facecolor=color, alpha=0.7, linewidth=0.8),
#                         whiskerprops=dict(linewidth=0.8),
#                         capprops=dict(linewidth=0.8))

#     ax.axhline(0, color="black", linewidth=0.5, alpha=0.5)
#     ax.axhline(0.03, color="red", linewidth=0.8, linestyle=":", alpha=0.6)
#     ax.set_xticks(x); ax.set_xticklabels(labels_box, fontsize=9)
#     ax.set_title(f"h={HORIZON_LABELS[h]}", fontsize=11)
#     if ax_j == 0: ax.set_ylabel("per-fold corr")

# from matplotlib.patches import Patch
# legend_elements = [Patch(facecolor="#85B7EB", alpha=0.7, label="low vol"),
#                    Patch(facecolor="#888780", alpha=0.7, label="mid vol"),
#                    Patch(facecolor="#E24B4A", alpha=0.7, label="high vol")]
# axes[1].legend(handles=legend_elements, fontsize=9, loc="upper right")
# plt.suptitle("Per-fold corr by vol-regime — all assets\n(red = low vol, gray = mid, blue = high vol)",
#              fontsize=12, y=1.01)
# plt.tight_layout()
# plt.savefig("fig_conditional_vol_regime.png", bbox_inches="tight", dpi=150)
# plt.savefig("fig_conditional_vol_regime.pdf", bbox_inches="tight", dpi=150)
# plt.show()
# print("  Saved: fig_conditional_vol_regime.png")

# fig2, ax2 = plt.subplots(figsize=(8, 4.5))
# pivot = sumdf.pivot(index="asset", columns="horizon", values="delta")
# pivot = pivot.reindex(ALL_ASSETS)

# im = ax2.imshow(pivot.values, cmap="RdYlGn", vmin=-0.15, vmax=0.15, aspect="auto")
# ax2.set_xticks(range(len(pivot.columns))); ax2.set_xticklabels(pivot.columns)
# ax2.set_yticks(range(len(pivot.index)));   ax2.set_yticklabels(pivot.index)

# for i in range(len(pivot.index)):
#     for j in range(len(pivot.columns)):
#         val = pivot.values[i, j]
#         sig_row = sumdf[(sumdf.asset==pivot.index[i]) & (sumdf.horizon==pivot.columns[j])]
#         sig_mark = sig_row["sig"].values[0] if len(sig_row) else ""
#         color = "white" if abs(val) > 0.08 else "black"
#         ax2.text(j, i, f"{val:+.3f}{sig_mark}", ha="center", va="center",
#                  fontsize=10, color=color, fontweight="bold" if sig_mark else "normal")

# plt.colorbar(im, ax=ax2, label="Δ corr (high vol - low vol)", shrink=0.8)
# ax2.set_title("Δ residual corr: high vol-regime minus low vol-regime\n(* p<0.10)")
# ax2.set_xlabel("Forecast horizon")
# plt.tight_layout()
# plt.savefig("fig_conditional_heatmap.png", bbox_inches="tight", dpi=150)
# plt.savefig("fig_conditional_heatmap.pdf", bbox_inches="tight", dpi=150)
# plt.show()
# print("  Saved: fig_conditional_heatmap.png")

# fig3, axes3 = plt.subplots(1, 2, figsize=(12, 4.5))
# for ax_i, asset in enumerate(["BTC", "ETH"]):
#     ax = axes3[ax_i]
#     h = 96
#     if (asset, h) not in all_records: continue
#     df = all_records[(asset, h)]
#     sc = ax.scatter(df["vol_rank"], df["corr"],
#                     c=df["sync_rank"], cmap="coolwarm",
#                     s=20, alpha=0.6, vmin=0, vmax=1)
#     df_sorted = df.sort_values("vol_rank")
#     roll = df_sorted["corr"].rolling(30, min_periods=10).mean()
#     ax.plot(df_sorted["vol_rank"], roll.values, "k-", linewidth=2, label="rolling mean")
#     ax.axhline(0,    color="black", linewidth=0.5, alpha=0.4)
#     ax.axhline(0.03, color="red",   linewidth=0.8, linestyle=":", alpha=0.6)
#     ax.set_xlabel("vol_now percentile rank")
#     ax.set_ylabel("per-fold corr")
#     ax.set_title(f"{asset}  h=48h | color = sync rank")
#     ax.legend(fontsize=9)
#     plt.colorbar(sc, ax=ax, label="sync rank\n(high = synchronized)")

# plt.suptitle("Residual corr vs vol-regime\n(color = market synchronization)",
#              fontsize=12, y=1.01)
# plt.tight_layout()
# plt.savefig("fig_scatter_corr_vol.png", bbox_inches="tight", dpi=150)
# plt.savefig("fig_scatter_corr_vol.pdf", bbox_inches="tight", dpi=150)
# plt.show()
# print("  Saved: fig_scatter_corr_vol.png")

# print("\n" + "="*65)
# print("  ИТОГ: CONDITIONAL IT SIGNAL")
# print("="*65)
# print(f"""
#   Ключевой результат:
#   IT-сигнал систематически сильнее в периоды высокой волатильности.

#   Средний corr по vol-режиму (все активы, h=48h):
#     low vol:   {sumdf[sumdf.h==96]['corr_low'].mean():+.4f}
#     high vol:  {sumdf[sumdf.h==96]['corr_high'].mean():+.4f}
#     Δ:         {sumdf[sumdf.h==96]['delta'].mean():+.4f}

#   Комбинированное условие (high vol + high sync):
#     corr_HH:   {cdf['corr_hh'].mean():+.4f}  (лучший квадрант)
#     corr_LL:   {cdf['corr_ll'].mean():+.4f}  (худший квадрант)
#     Δ:         {cdf['delta_hh_ll'].mean():+.4f}

#   Интерпретация для работы:
#   IT-фичи (особенно pairwise covariance и MI) наиболее информативны
#   когда рынок уже "разогрет" — именно тогда кросс-ассетная синхронизация
#   несёт предсказательный сигнал о дальнейшем развитии vol-режима.
#   В тихие периоды каждый актив движется независимо и IT-сигнал слабее.
#   Это согласуется с теорией системного риска (Billio et al., 2012).
# """)

Собираем per-fold данные для всех активов...


assets: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6/6 [00:12<00:00,  2.04s/it]



  CONDITIONAL CORR BY VOL-REGIME — все активы
  Вопрос: в каком vol-режиме IT-сигнал сильнее?

  актив  h         all  low vol  high vol       Δ  p   sig
  ----------------------------------------------------------
  BTC    24h    +0.096   +0.079    +0.113  +0.035  0.474 
  BTC    48h    +0.117   +0.098    +0.179  +0.081  0.170 
  BTC    4d     +0.104   +0.067    +0.163  +0.096  0.196 
  ··························································
  ETH    24h    +0.086   +0.041    +0.176  +0.135  0.004 *
  ETH    48h    +0.144   +0.110    +0.233  +0.123  0.026 *
  ETH    4d     +0.155   +0.145    +0.199  +0.054  0.469 

  Средний Δ (high vol - low vol): +0.0873
  Значимых (p<0.10): 2 / 6
  Δ > 0 (high vol лучше): 6 / 6

  КОМБИНИРОВАННОЕ УСЛОВИЕ: high vol + растущая синхронизация
  Гипотеза: сигнал максимален когда ОБА условия выполнены

  актив  h         all  HH (best)  LL (worst)        Δ  p    sig
  -----------------------------------------------------------------
  BTC    24h    +

In [44]:
# import warnings; warnings.filterwarnings("ignore")
# import numpy as np
# import pandas as pd
# from tqdm.auto import tqdm
# from sklearn.linear_model import Ridge
# from sklearn.preprocessing import StandardScaler
# from sklearn.pipeline import Pipeline
# from scipy import stats as sp_stats
# import matplotlib
# matplotlib.use("Agg")
# import matplotlib.pyplot as plt

# ALL_ASSETS     = ["BTC", "ETH", "XRP", "SOL", "BNB", "DOGE"]
# HORIZONS       = [48, 96, 192]
# HORIZON_LABELS = {48:"24h", 96:"48h", 192:"4d"}
# TRAIN_BARS=2880; TEST_BARS=480; STEP_BARS=240; PURGE_BARS=96
# VOL_WINDOW=48; ANN_FACTOR=np.sqrt(365*48); BARS_PER_DAY=48

# COLORS = {"BTC":"#1f77b4","ETH":"#2ca02c","XRP":"#d62728",
#           "SOL":"#ff7f0e","BNB":"#9467bd","DOGE":"#8c564b"}

# def _splits(n):
#     s = TRAIN_BARS
#     while s + PURGE_BARS + TEST_BARS <= n:
#         yield list(range(s-TRAIN_BARS,s)), list(range(s+PURGE_BARS,s+PURGE_BARS+TEST_BARS))
#         s += STEP_BARS

# def _new_ridge():
#     return Pipeline([("sc", StandardScaler()), ("m", Ridge(alpha=1.0))])

# class RidgeVolBaseline:
#     def __init__(self, k=3):
#         self.k=k; self.mdl=_new_ridge(); self.cols=None
#     def fit(self, X, y):
#         corr = X.corrwith(y).abs().replace([np.inf,-np.inf],np.nan).dropna()
#         self.cols = corr.nlargest(self.k).index.tolist()
#         self.mdl.fit(X[self.cols], y); return self
#     def predict(self, X): return self.mdl.predict(X[self.cols])

# def _get_y(asset, h):
#     col = f"delta_vol_{h}"
#     for src in [targets_extra if 'targets_extra' in dir() else {},
#                 targets, targets_ext]:
#         if asset in src and col in src[asset].columns:
#             return src[asset][col].dropna()
#     return None

# try: _ = targets_extra
# except NameError: targets_extra = {}

# def get_fold_records(asset, h):
#     y = _get_y(asset, h)
#     if y is None: return None

#     idx = X_vol.index.intersection(X_it_slim.index).intersection(y.index)
#     valid = (X_vol.loc[idx].notna().all(1) &
#              X_it_slim.loc[idx].notna().all(1) & y.loc[idx].notna())
#     idx = idx[valid]
#     if len(idx) < TRAIN_BARS + TEST_BARS + PURGE_BARS: return None

#     Xv_ = X_vol.loc[idx]
#     Xi_ = X_it_slim.loc[idx]
#     y_  = y.loc[idx]

#     vol_now = Xv_["vol_now_feat"] if "vol_now_feat" in Xv_.columns else Xv_.iloc[:,0]
#     vol_rank  = vol_now.rolling(BARS_PER_DAY*30, min_periods=100).rank(pct=True)
#     vol_trend = vol_now - vol_now.shift(BARS_PER_DAY*5)

#     sync_col = "pairwise_cov_mean" if "pairwise_cov_mean" in Xi_.columns else Xi_.columns[0]
#     sync = Xi_[sync_col]
#     sync_rank = sync.rolling(BARS_PER_DAY*30, min_periods=100).rank(pct=True)

#     records = []
#     for tr, te in _splits(len(y_)):
#         ytr, yte = y_.iloc[tr], y_.iloc[te]
#         Xv_tr, Xv_te = Xv_.iloc[tr], Xv_.iloc[te]
#         Xi_tr, Xi_te = Xi_.iloc[tr], Xi_.iloc[te]

#         base = RidgeVolBaseline(); base.fit(Xv_tr, ytr)
#         res_tr = ytr.values - base.predict(Xv_tr)
#         res_te = yte.values - base.predict(Xv_te)

#         mdl = _new_ridge(); mdl.fit(Xi_tr, res_tr)
#         pred = mdl.predict(Xi_te)
#         corr = float(pd.Series(pred).corr(pd.Series(res_te)))

#         t0 = te[0]
#         records.append({
#             "corr":       corr if np.isfinite(corr) else 0.0,
#             "vol_rank":   float(vol_rank.iloc[t0])  if t0 < len(vol_rank)  else np.nan,
#             "vol_trend":  float(vol_trend.iloc[t0]) if t0 < len(vol_trend) else np.nan,
#             "sync_rank":  float(sync_rank.iloc[t0]) if t0 < len(sync_rank) else np.nan,
#             "date":       y_.index[t0],
#         })

#     return pd.DataFrame(records).dropna()

# print("Собираем per-fold данные для всех активов...")
# all_records = {}
# for asset in tqdm(ALL_ASSETS, desc="assets"):
#     for h in HORIZONS:
#         df = get_fold_records(asset, h)
#         if df is not None and len(df) > 30:
#             all_records[(asset, h)] = df

# print("\n" + "="*65)
# print("  CONDITIONAL CORR BY VOL-REGIME — все активы")
# print("  Вопрос: в каком vol-режиме IT-сигнал сильнее?")
# print("="*65)

# summary_rows = []
# for asset in ALL_ASSETS:
#     for h in HORIZONS:
#         if (asset, h) not in all_records: continue
#         df = all_records[(asset, h)]
#         median_vol = df["vol_rank"].median()
#         low  = df[df.vol_rank <= median_vol]["corr"]
#         high = df[df.vol_rank >  median_vol]["corr"]
#         if len(low) < 5 or len(high) < 5: continue
#         t, p = sp_stats.ttest_ind(high, low)
#         summary_rows.append({
#             "asset":    asset,
#             "horizon":  HORIZON_LABELS[h],
#             "h":        h,
#             "corr_all": df["corr"].mean(),
#             "corr_low": low.mean(),
#             "corr_high":high.mean(),
#             "delta":    high.mean() - low.mean(),
#             "p_val":    p,
#             "sig":      "**" if p < 0.05 else ("*" if p < 0.10 else ""),
#         })

# sumdf = pd.DataFrame(summary_rows)

# print(f"\n  {'актив':<6} {'h':<5} {'all':>7} {'low vol':>8} {'high vol':>9} {'Δ':>7}  p   sig")
# print("  " + "-"*58)
# prev_asset = None
# for _, row in sumdf.iterrows():
#     if prev_asset and prev_asset != row["asset"]:
#         print("  " + "·"*58)
#     prev_asset = row["asset"]
#     print(f"  {row['asset']:<6} {row['horizon']:<5}"
#           f"  {row['corr_all']:>+6.3f}"
#           f"  {row['corr_low']:>+7.3f}"
#           f"  {row['corr_high']:>+8.3f}"
#           f"  {row['delta']:>+6.3f}"
#           f"  {row['p_val']:>5.3f} {row['sig']}")

# print(f"\n  Средний Δ (high vol - low vol): {sumdf['delta'].mean():+.4f}")
# print(f"  Значимых (p<0.10): {(sumdf['p_val']<0.10).sum()} / {len(sumdf)}")
# print(f"  Δ > 0 (high vol лучше): {(sumdf['delta']>0).sum()} / {len(sumdf)}")

# print("\n" + "="*65)
# print("  КОМБИНИРОВАННОЕ УСЛОВИЕ: high vol + растущая синхронизация")
# print("  Гипотеза: сигнал максимален когда ОБА условия выполнены")
# print("="*65)

# combo_rows = []
# for asset in ALL_ASSETS:
#     for h in HORIZONS:
#         if (asset, h) not in all_records: continue
#         df = all_records[(asset, h)]

#         vol_med  = df["vol_rank"].median()
#         sync_med = df["sync_rank"].median()
#         df["vol_hi"]  = df["vol_rank"]  > vol_med
#         df["sync_hi"] = df["sync_rank"] > sync_med

#         q_hh = df[ df.vol_hi &  df.sync_hi]["corr"]   # high vol + high sync
#         q_hl = df[ df.vol_hi & ~df.sync_hi]["corr"]   # high vol + low sync
#         q_lh = df[~df.vol_hi &  df.sync_hi]["corr"]   # low vol + high sync
#         q_ll = df[~df.vol_hi & ~df.sync_hi]["corr"]   # low vol + low sync

#         if min(len(q_hh), len(q_ll)) < 3: continue

#         t_hh_ll, p_hh_ll = sp_stats.ttest_ind(q_hh, q_ll) if len(q_hh)>3 and len(q_ll)>3 else (0,1)

#         combo_rows.append({
#             "asset":    asset,
#             "horizon":  HORIZON_LABELS[h],
#             "h":        h,
#             "n_hh":     len(q_hh),
#             "corr_hh":  q_hh.mean() if len(q_hh) else np.nan,  # best case
#             "corr_ll":  q_ll.mean() if len(q_ll) else np.nan,  # worst case
#             "corr_all": df["corr"].mean(),
#             "delta_hh_ll": q_hh.mean() - q_ll.mean() if len(q_hh) and len(q_ll) else np.nan,
#             "p_hh_ll":  p_hh_ll,
#             "sig":      "**" if p_hh_ll<0.05 else ("*" if p_hh_ll<0.10 else ""),
#         })

# cdf = pd.DataFrame(combo_rows)

# print(f"\n  {'актив':<6} {'h':<5} {'all':>7} {'HH (best)':>10} {'LL (worst)':>11} {'Δ':>8}  p    sig")
# print("  " + "-"*65)
# prev = None
# for _, row in cdf.iterrows():
#     if prev and prev != row["asset"]: print("  " + "·"*65)
#     prev = row["asset"]
#     print(f"  {row['asset']:<6} {row['horizon']:<5}"
#           f"  {row['corr_all']:>+6.3f}"
#           f"  {row['corr_hh']:>+9.3f}"
#           f"  {row['corr_ll']:>+10.3f}"
#           f"  {row['delta_hh_ll']:>+7.3f}"
#           f"  {row['p_hh_ll']:>5.3f} {row['sig']}")

# print(f"\n  Средний corr в режиме 'high vol + high sync': {cdf['corr_hh'].mean():+.4f}")
# print(f"  Средний corr в режиме 'low vol + low sync':   {cdf['corr_ll'].mean():+.4f}")
# print(f"  Средний Δ:                                    {cdf['delta_hh_ll'].mean():+.4f}")
# print(f"  Значимых (p<0.10): {(cdf['p_hh_ll']<0.10).sum()} / {len(cdf)}")

# print("\n  Строим графики...")

# plt.rcParams.update({
#     "figure.facecolor":"white","axes.facecolor":"white",
#     "axes.spines.top":False,"axes.spines.right":False,
#     "axes.grid":True,"grid.alpha":0.2,"grid.linestyle":"--",
#     "font.size":10,
# })

# fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
# for ax_j, h in enumerate(HORIZONS):
#     ax = axes[ax_j]
#     data_low, data_high = [], []
#     labels_box = []
#     for asset in ALL_ASSETS:
#         if (asset, h) not in all_records: continue
#         df = all_records[(asset, h)]
#         med = df["vol_rank"].median()
#         data_low.append(df[df.vol_rank <= med]["corr"].values)
#         data_high.append(df[df.vol_rank >  med]["corr"].values)
#         labels_box.append(asset)

#     x = np.arange(len(labels_box))
#     w = 0.35
#     for i, (data_list, label, color) in enumerate([
#         (data_low,  "low vol",  "#85B7EB"),
#         (data_high, "high vol", "#E24B4A"),
#     ]):
#         positions = [xi + (i-0.5)*w for xi in x]
#         bp = ax.boxplot(data_list, positions=positions, widths=w*0.85,
#                         patch_artist=True, showfliers=False,
#                         medianprops=dict(color="black", linewidth=1.5),
#                         boxprops=dict(facecolor=color, alpha=0.7, linewidth=0.8),
#                         whiskerprops=dict(linewidth=0.8),
#                         capprops=dict(linewidth=0.8))

#     ax.axhline(0,    color="black", linewidth=0.5, alpha=0.5)
#     ax.axhline(0.03, color="red",   linewidth=0.8, linestyle=":", alpha=0.6)
#     ax.set_xticks(x); ax.set_xticklabels(labels_box, fontsize=9)
#     ax.set_title(f"h={HORIZON_LABELS[h]}", fontsize=11)
#     if ax_j == 0: ax.set_ylabel("per-fold corr")

# from matplotlib.patches import Patch
# legend_elements = [Patch(facecolor="#85B7EB", alpha=0.7, label="low vol (below median)"),
#                    Patch(facecolor="#E24B4A", alpha=0.7, label="high vol (above median)")]
# axes[1].legend(handles=legend_elements, fontsize=9, loc="upper right")
# plt.suptitle("Per-fold corr: high vol vs low vol regime — all 6 assets",
#              fontsize=12, y=1.01)
# plt.tight_layout()
# plt.savefig("fig_conditional_vol_regime.png", bbox_inches="tight", dpi=150)
# plt.savefig("fig_conditional_vol_regime.pdf", bbox_inches="tight", dpi=150)
# plt.show()
# print("  Saved: fig_conditional_vol_regime.png")

# fig2, ax2 = plt.subplots(figsize=(8, 4.5))
# pivot = sumdf.pivot(index="asset", columns="horizon", values="delta")
# pivot = pivot.reindex(ALL_ASSETS)

# im = ax2.imshow(pivot.values, cmap="RdYlGn", vmin=-0.15, vmax=0.15, aspect="auto")
# ax2.set_xticks(range(len(pivot.columns))); ax2.set_xticklabels(pivot.columns)
# ax2.set_yticks(range(len(pivot.index)));   ax2.set_yticklabels(pivot.index)

# for i in range(len(pivot.index)):
#     for j in range(len(pivot.columns)):
#         val = pivot.values[i, j]
#         sig_row = sumdf[(sumdf.asset==pivot.index[i]) & (sumdf.horizon==pivot.columns[j])]
#         sig_mark = sig_row["sig"].values[0] if len(sig_row) else ""
#         color = "white" if abs(val) > 0.08 else "black"
#         ax2.text(j, i, f"{val:+.3f}{sig_mark}", ha="center", va="center",
#                  fontsize=10, color=color, fontweight="bold" if sig_mark else "normal")

# plt.colorbar(im, ax=ax2, label="Δ corr (high vol - low vol)", shrink=0.8)
# ax2.set_title("Δ residual corr: high vol-regime minus low vol-regime\n(* p<0.10)")
# ax2.set_xlabel("Forecast horizon")
# plt.tight_layout()
# plt.savefig("fig_conditional_heatmap.png", bbox_inches="tight", dpi=150)
# plt.savefig("fig_conditional_heatmap.pdf", bbox_inches="tight", dpi=150)
# plt.show()
# print("  Saved: fig_conditional_heatmap.png")

# fig3, axes3 = plt.subplots(1, 2, figsize=(12, 4.5))
# for ax_i, asset in enumerate(["BTC", "ETH"]):
#     ax = axes3[ax_i]
#     h = 96
#     if (asset, h) not in all_records: continue
#     df = all_records[(asset, h)]
#     sc = ax.scatter(df["vol_rank"], df["corr"],
#                     c=df["sync_rank"], cmap="coolwarm",
#                     s=20, alpha=0.6, vmin=0, vmax=1)
#     df_sorted = df.sort_values("vol_rank")
#     roll = df_sorted["corr"].rolling(30, min_periods=10).mean()
#     ax.plot(df_sorted["vol_rank"], roll.values, "k-", linewidth=2, label="rolling mean")
#     ax.axhline(0,    color="black", linewidth=0.5, alpha=0.4)
#     ax.axhline(0.03, color="red",   linewidth=0.8, linestyle=":", alpha=0.6)
#     ax.set_xlabel("vol_now percentile rank")
#     ax.set_ylabel("per-fold corr")
#     ax.set_title(f"{asset}  h=48h | color = sync rank")
#     ax.legend(fontsize=9)
#     plt.colorbar(sc, ax=ax, label="sync rank\n(high = synchronized)")

# plt.suptitle("Residual corr vs vol-regime\n(color = market synchronization)",
#              fontsize=12, y=1.01)
# plt.tight_layout()
# plt.savefig("fig_scatter_corr_vol.png", bbox_inches="tight", dpi=150)
# plt.savefig("fig_scatter_corr_vol.pdf", bbox_inches="tight", dpi=150)
# plt.show()
# print("  Saved: fig_scatter_corr_vol.png")

# print("\n" + "="*65)
# print("  ИТОГ: CONDITIONAL IT SIGNAL")
# print("="*65)
# print(f"""
#   Ключевой результат:
#   IT-сигнал систематически сильнее в периоды высокой волатильности.

#   Средний corr по vol-режиму (все активы, h=48h):
#     low vol:   {sumdf[sumdf.h==96]['corr_low'].mean():+.4f}
#     high vol:  {sumdf[sumdf.h==96]['corr_high'].mean():+.4f}
#     Δ:         {sumdf[sumdf.h==96]['delta'].mean():+.4f}

#   Комбинированное условие (high vol + high sync):
#     corr_HH:   {cdf['corr_hh'].mean():+.4f}  (лучший квадрант)
#     corr_LL:   {cdf['corr_ll'].mean():+.4f}  (худший квадрант)
#     Δ:         {cdf['delta_hh_ll'].mean():+.4f}

#   Интерпретация для работы:
#   IT-фичи (особенно pairwise covariance и MI) наиболее информативны
#   когда рынок уже "разогрет" — именно тогда кросс-ассетная синхронизация
#   несёт предсказательный сигнал о дальнейшем развитии vol-режима.
#   В тихие периоды каждый актив движется независимо и IT-сигнал слабее.
#   Это согласуется с теорией системного риска (Billio et al., 2012).
# """)

Собираем per-fold данные для всех активов...


assets: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6/6 [00:12<00:00,  2.08s/it]


  CONDITIONAL CORR BY VOL-REGIME — все активы
  Вопрос: в каком vol-режиме IT-сигнал сильнее?

  актив  h         all  low vol  high vol       Δ  p   sig
  ----------------------------------------------------------
  BTC    24h    +0.096   +0.070    +0.122  +0.053  0.181 
  BTC    48h    +0.117   +0.064    +0.169  +0.105  0.028 **
  BTC    4d     +0.104   +0.057    +0.152  +0.095  0.105 
  ··························································
  ETH    24h    +0.086   +0.028    +0.144  +0.116  0.004 **
  ETH    48h    +0.144   +0.070    +0.218  +0.148  0.002 **
  ETH    4d     +0.155   +0.101    +0.210  +0.109  0.060 *

  Средний Δ (high vol - low vol): +0.1044
  Значимых (p<0.10): 4 / 6
  Δ > 0 (high vol лучше): 6 / 6

  КОМБИНИРОВАННОЕ УСЛОВИЕ: high vol + растущая синхронизация
  Гипотеза: сигнал максимален когда ОБА условия выполнены

  актив  h         all  HH (best)  LL (worst)        Δ  p    sig
  -----------------------------------------------------------------
  BTC    24h

  Saved: fig_conditional_vol_regime.png
  Saved: fig_conditional_heatmap.png
  Saved: fig_scatter_corr_vol.png

  ИТОГ: CONDITIONAL IT SIGNAL

  Ключевой результат:
  IT-сигнал систематически сильнее в периоды высокой волатильности.

  Средний corr по vol-режиму (все активы, h=48h):
    low vol:   +0.0672
    high vol:  +0.1937
    Δ:         +0.1264

  Комбинированное условие (high vol + high sync):
    corr_HH:   +0.1849  (лучший квадрант)
    corr_LL:   +0.0839  (худший квадрант)
    Δ:         +0.1009

  Интерпретация для работы:
  IT-фичи (особенно pairwise covariance и MI) наиболее информативны
  когда рынок уже "разогрет" — именно тогда кросс-ассетная синхронизация
  несёт предсказательный сигнал о дальнейшем развитии vol-режима.
  В тихие периоды каждый актив движется независимо и IT-сигнал слабее.
  Это согласуется с теорией системного риска (Billio et al., 2012).



In [45]:
# import warnings; warnings.filterwarnings("ignore")
# import numpy as np
# import pandas as pd
# from tqdm.auto import tqdm
# from sklearn.linear_model import Ridge
# from sklearn.preprocessing import StandardScaler
# from sklearn.pipeline import Pipeline
# from scipy import stats as sp_stats

# ALL_ASSETS     = ["BTC", "ETH", "XRP", "SOL", "BNB", "DOGE"]
# HORIZONS       = [48, 96, 192]
# HORIZON_LABELS = {8:"4h", 48:"24h", 96:"48h", 192:"4d", 336:"7d"}
# TRAIN_BARS=2880; TEST_BARS=480; STEP_BARS=240; PURGE_BARS=96
# N_PERMUTATIONS = 10   # сколько раз перемешиваем каждую фичу (усредняем)
# RNG = np.random.default_rng(42)

# def _splits(n):
#     s = TRAIN_BARS
#     while s + PURGE_BARS + TEST_BARS <= n:
#         yield list(range(s-TRAIN_BARS,s)), list(range(s+PURGE_BARS,s+PURGE_BARS+TEST_BARS))
#         s += STEP_BARS

# def _new_ridge():
#     return Pipeline([("sc", StandardScaler()), ("m", Ridge(alpha=1.0))])

# class RidgeVolBaseline:
#     def __init__(self, k=3):
#         self.k=k; self.mdl=_new_ridge(); self.cols=None
#     def fit(self, X, y):
#         c = X.corrwith(y).abs().replace([np.inf,-np.inf],np.nan).dropna()
#         self.cols = c.nlargest(self.k).index.tolist()
#         self.mdl.fit(X[self.cols], y); return self
#     def predict(self, X): return self.mdl.predict(X[self.cols])

# def _get_y(asset, h):
#     col = f"delta_vol_{h}"
#     for src in [targets_extra if 'targets_extra' in dir() else {},
#                 targets, targets_ext]:
#         if asset in src and col in src[asset].columns:
#             return src[asset][col].dropna()
#     return None

# try: _ = targets_extra
# except NameError: targets_extra = {}

# def get_fold_corrs(Xv, Xi, y, Xi_override=None):
#     """Считает per-fold corr. Xi_override позволяет подставить изменённый Xi."""
#     Xi_use = Xi_override if Xi_override is not None else Xi
#     mask = Xv.notna().all(1) & Xi_use.notna().all(1) & y.notna()
#     Xv_,Xi_,y_ = Xv[mask], Xi_use[mask], y[mask]
#     corrs = []
#     for tr,te in _splits(len(y_)):
#         ytr,yte = y_.iloc[tr], y_.iloc[te]
#         Xv_tr,Xv_te = Xv_.iloc[tr], Xv_.iloc[te]
#         Xi_tr,Xi_te = Xi_.iloc[tr], Xi_.iloc[te]
#         base = RidgeVolBaseline(); base.fit(Xv_tr, ytr)
#         res_tr = ytr.values - base.predict(Xv_tr)
#         res_te = yte.values - base.predict(Xv_te)
#         mdl = _new_ridge(); mdl.fit(Xi_tr, res_tr)
#         pred = mdl.predict(Xi_te)
#         c = float(pd.Series(pred).corr(pd.Series(res_te)))
#         corrs.append(c if np.isfinite(c) else 0.0)
#     return np.array(corrs)

# print("="*65)
# print("  ЧАСТЬ 1 — Permutation feature importance")
# print("  Δcorr = corr_permuted - corr_baseline")
# print("  Δ > 0 → фича ВРЕДИТ (убираем)")
# print("  Δ < 0 → фича ПОМОГАЕТ (оставляем)")
# print("="*65)

# PILOT_ASSETS = ["BTC", "ETH"]
# PILOT_H = 96

# feat_scores = {col: [] for col in X_it_slim.columns}

# for asset in PILOT_ASSETS:
#     y = _get_y(asset, PILOT_H)
#     if y is None: continue
#     idx = X_vol.index.intersection(X_it_slim.index).intersection(y.index)
#     valid = (X_vol.loc[idx].notna().all(1) &
#              X_it_slim.loc[idx].notna().all(1) & y.loc[idx].notna())
#     idx = idx[valid]
#     Xv_ = X_vol.loc[idx]; Xi_ = X_it_slim.loc[idx]; y_ = y.loc[idx]

#     baseline_corrs = get_fold_corrs(Xv_, Xi_, y_)
#     baseline_mean  = float(np.nanmean(baseline_corrs))

#     print(f"\n  {asset} h={PILOT_H}  baseline corr={baseline_mean:.4f}")
#     print(f"  {'фича':<28} {'Δcorr':>8}  влияние")
#     print("  " + "-"*50)

#     for feat in tqdm(X_it_slim.columns, desc=f"{asset}", leave=False):
#         deltas = []
#         for _ in range(N_PERMUTATIONS):
#             Xi_perm = Xi_.copy()
#             Xi_perm[feat] = RNG.permutation(Xi_perm[feat].values)
#             perm_corrs = get_fold_corrs(Xv_, Xi_perm, y_, Xi_override=Xi_perm)
#             deltas.append(float(np.nanmean(perm_corrs)) - baseline_mean)
#         mean_delta = float(np.mean(deltas))
#         feat_scores[feat].append(mean_delta)

#     asset_scores = {f: np.mean([feat_scores[f][-1]]) for f in X_it_slim.columns}
#     for feat, delta in sorted(asset_scores.items(), key=lambda x: -x[1])[:10]:
#         mark = "✗ вредит" if delta > 0.005 else ("~ нейтрально" if abs(delta) <= 0.005 else "✓ помогает")
#         print(f"  {feat:<28} {delta:>+8.4f}  {mark}")

# print(f"\n  {'='*65}")
# print(f"  ИТОГОВЫЙ РЕЙТИНГ ФИЧЕЙ (среднее по BTC + ETH, h=96)")
# print(f"  {'='*65}")
# print(f"  {'фича':<28} {'Δcorr':>8}  решение")
# print("  " + "-"*50)

# avg_scores = {f: float(np.mean(v)) for f,v in feat_scores.items() if v}
# harmful  = []
# neutral  = []
# helpful  = []

# for feat, delta in sorted(avg_scores.items(), key=lambda x: -x[1]):
#     if delta > 0.005:
#         decision = "УДАЛИТЬ"; harmful.append(feat)
#     elif delta < -0.005:
#         decision = "оставить"; helpful.append(feat)
#     else:
#         decision = "нейтрально"; neutral.append(feat)
#     print(f"  {feat:<28} {delta:>+8.4f}  {decision}")

# print(f"\n  Вредных фич (Δ>0.005):    {len(harmful)}")
# print(f"  Нейтральных:              {len(neutral)}")
# print(f"  Полезных (Δ<-0.005):     {len(helpful)}")
# print(f"\n  Кандидаты на удаление: {harmful}")

# keep_cols = [f for f in X_it_slim.columns if f not in harmful]
# X_it_optimal = X_it_slim[keep_cols].copy()
# print(f"\n  X_it_optimal: {len(keep_cols)} фич (было {X_it_slim.shape[1]})")

# print(f"\n  Проверка X_it_optimal на всех активах × горизонтах:")
# print(f"  {'актив':<6} {'h':<5} {'baseline':>10} {'optimal':>9} {'Δ':>7}")
# print("  " + "-"*40)

# improvement_rows = []
# for asset in tqdm(ALL_ASSETS, desc="validation"):
#     for h in HORIZONS:
#         y = _get_y(asset, h)
#         if y is None: continue
#         idx = X_vol.index.intersection(X_it_slim.index).intersection(y.index)
#         valid = (X_vol.loc[idx].notna().all(1) &
#                  X_it_slim.loc[idx].notna().all(1) & y.loc[idx].notna())
#         idx = idx[valid]
#         if len(idx) < TRAIN_BARS + TEST_BARS + PURGE_BARS: continue
#         Xv_ = X_vol.loc[idx]; y_ = y.loc[idx]

#         c_base = float(np.nanmean(get_fold_corrs(Xv_, X_it_slim.loc[idx], y_)))
#         c_opt  = float(np.nanmean(get_fold_corrs(Xv_, X_it_optimal.loc[idx], y_,
#                                                   Xi_override=X_it_optimal.loc[idx])))
#         delta = c_opt - c_base
#         mark = "↑" if delta > 0.003 else ("↓" if delta < -0.003 else "~")
#         print(f"  {asset:<6} {HORIZON_LABELS[h]:<5} {c_base:>+10.4f} {c_opt:>+9.4f} {delta:>+6.4f} {mark}")
#         improvement_rows.append({"asset":asset,"h":h,"baseline":c_base,"optimal":c_opt,"delta":delta})

# imp_df = pd.DataFrame(improvement_rows)
# print(f"\n  Среднее baseline: {imp_df['baseline'].mean():.4f}")
# print(f"  Среднее optimal:  {imp_df['optimal'].mean():.4f}")
# print(f"  Средний Δ:        {imp_df['delta'].mean():+.4f}")
# print(f"  Улучшений (Δ>0.003): {(imp_df['delta']>0.003).sum()} / {len(imp_df)}")

# print("\n" + "="*65)
# print("  ЧАСТЬ 2 — Diebold-Mariano тест")
# print("  H0: mean(per-fold corr) = 0")
# print("  H1: mean(per-fold corr) > 0  (one-sided)")
# print("="*65)

# Xi_final = X_it_optimal if imp_df["delta"].mean() > 0 else X_it_slim
# print(f"\n  Используем: {'X_it_optimal' if imp_df['delta'].mean() > 0 else 'X_it_slim'}")

# dm_rows = []
# for asset in tqdm(ALL_ASSETS, desc="DM test"):
#     for h in HORIZONS:
#         y = _get_y(asset, h)
#         if y is None: continue
#         idx = X_vol.index.intersection(Xi_final.index).intersection(y.index)
#         valid = (X_vol.loc[idx].notna().all(1) &
#                  Xi_final.loc[idx].notna().all(1) & y.loc[idx].notna())
#         idx = idx[valid]
#         if len(idx) < TRAIN_BARS + TEST_BARS + PURGE_BARS: continue

#         fold_corrs = get_fold_corrs(X_vol.loc[idx], Xi_final.loc[idx], y.loc[idx])

#         n = len(fold_corrs)
#         mean_c = float(np.nanmean(fold_corrs))
#         std_c  = float(np.nanstd(fold_corrs, ddof=1))
#         se     = std_c / np.sqrt(n)

#         if se > 1e-10:
#             t_stat = mean_c / se
#             p_val  = float(sp_stats.t.sf(t_stat, df=n-1))
#         else:
#             t_stat, p_val = np.nan, np.nan

#         sig = "***" if p_val < 0.01 else ("**" if p_val < 0.05 else
#               ("*"  if p_val < 0.10 else ""))

#         dm_rows.append({
#             "asset":   asset,
#             "horizon": HORIZON_LABELS[h],
#             "h":       h,
#             "n_folds": n,
#             "mean":    round(mean_c, 4),
#             "std":     round(std_c, 4),
#             "t_stat":  round(t_stat, 3) if np.isfinite(t_stat) else np.nan,
#             "p_val":   round(p_val,  4) if np.isfinite(p_val)  else np.nan,
#             "sig":     sig,
#             "pos_pct": round(float(np.mean(fold_corrs > 0)), 3),
#         })

# dm_df = pd.DataFrame(dm_rows)

# print(f"\n  {'актив':<6} {'h':<5} {'n':>4} {'mean':>8} {'std':>7} {'t':>7} {'p':>8}  sig  pos%")
# print("  " + "-"*65)
# prev = None
# for _, row in dm_df.iterrows():
#     if prev and prev != row["asset"]: print("  " + "·"*65)
#     prev = row["asset"]
#     print(f"  {row['asset']:<6} {row['horizon']:<5}"
#           f" {row['n_folds']:>4}"
#           f" {row['mean']:>+8.4f}"
#           f" {row['std']:>7.4f}"
#           f" {row['t_stat']:>7.3f}"
#           f" {row['p_val']:>8.4f}"
#           f"  {row['sig']:<3}"
#           f"  {row['pos_pct']:.0%}")

# sig_05 = (dm_df["p_val"] < 0.05).sum()
# sig_10 = (dm_df["p_val"] < 0.10).sum()
# sig_01 = (dm_df["p_val"] < 0.01).sum()
# print(f"\n  ИТОГ:")
# print(f"  p < 0.01 (***): {sig_01} / {len(dm_df)}")
# print(f"  p < 0.05 (**):  {sig_05} / {len(dm_df)}")
# print(f"  p < 0.10 (*):   {sig_10} / {len(dm_df)}")
# print(f"\n  Среднее corr:   {dm_df['mean'].mean():.4f}")
# print(f"  Средняя pos%:   {dm_df['pos_pct'].mean():.0%}")

# best = dm_df.loc[dm_df["mean"].idxmax()]
# print(f"\n  Лучший результат: {best['asset']} h={best['horizon']}"
#       f"  corr={best['mean']:.4f}  p={best['p_val']:.4f} {best['sig']}")

# dm_results = dm_df.copy()
# X_it_final = Xi_final.copy()
# print(f"\n  dm_results, X_it_final сохранены в памяти")
# print(f"  X_it_final: {X_it_final.shape[1]} фич")

  ЧАСТЬ 1 — Permutation feature importance
  Δcorr = corr_permuted - corr_baseline
  Δ > 0 → фича ВРЕДИТ (убираем)
  Δ < 0 → фича ПОМОГАЕТ (оставляем)

  BTC h=96  baseline corr=0.1165
  фича                            Δcorr  влияние
  --------------------------------------------------


  pairwise_cov_std              +0.0161  ✗ вредит
  pairwise_cov_max              +0.0132  ✗ вредит
  pairwise_cov_min              +0.0122  ✗ вредит
  kl_XRP                        +0.0082  ✗ вредит
  kl_TRX                        +0.0051  ✗ вредит
  pairwise_mi_max               +0.0042  ~ нейтрально
  kl_ETH                        +0.0027  ~ нейтрально
  pairwise_mi_mean              +0.0027  ~ нейтрально
  coinfo3_std                   +0.0018  ~ нейтрально
  pairwise_cov_mean             +0.0016  ~ нейтрально



  ETH h=96  baseline corr=0.1444
  фича                            Δcorr  влияние
  --------------------------------------------------


  pairwise_cov_std              +0.0086  ✗ вредит
  pairwise_cov_max              +0.0071  ✗ вредит
  pairwise_cov_min              +0.0067  ✗ вредит
  pairwise_mi_mean              +0.0061  ✗ вредит
  pairwise_mi_max               +0.0042  ~ нейтрально
  kl_TRX                        +0.0039  ~ нейтрально
  kl_ETH                        +0.0012  ~ нейтрально
  coinfo3_min                   +0.0012  ~ нейтрально
  kl_BNB                        +0.0007  ~ нейтрально
  coinfo3_mean                  +0.0005  ~ нейтрально

  ИТОГОВЫЙ РЕЙТИНГ ФИЧЕЙ (среднее по BTC + ETH, h=96)
  фича                            Δcorr  решение
  --------------------------------------------------
  pairwise_cov_std              +0.0123  УДАЛИТЬ
  pairwise_cov_max              +0.0101  УДАЛИТЬ
  pairwise_cov_min              +0.0095  УДАЛИТЬ
  kl_TRX                        +0.0045  нейтрально
  pairwise_mi_mean              +0.0044  нейтрально
  pairwise_mi_max               +0.0042  нейтрально
  kl_XRP        

validation:   0%|                                                                                                                                                                     | 0/6 [00:00<?, ?it/s]

...

  BTC    4d       +0.1044   +0.0852 -0.0192 ↓
  ETH    24h      +0.0857   +0.0969 +0.0112 ↑
  ETH    48h      +0.1444   +0.1285 -0.0159 ↓


validation: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6/6 [00:24<00:00,  4.15s/it]


  ETH    4d       +0.1552   +0.1035 -0.0516 ↓

  Среднее baseline: 0.1170
  Среднее optimal:  0.1103
  Средний Δ:        -0.0067
  Улучшений (Δ>0.003): 3 / 6

  ЧАСТЬ 2 — Diebold-Mariano тест
  H0: mean(per-fold corr) = 0
  H1: mean(per-fold corr) > 0  (one-sided)

  Используем: X_it_slim


DM test: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6/6 [00:12<00:00,  2.06s/it]


  актив  h        n     mean     std       t        p  sig  pos%
  -----------------------------------------------------------------
  BTC    24h    174  +0.0960  0.2586   4.896   0.0000  ***  64%
  BTC    48h    174  +0.1165  0.3159   4.865   0.0000  ***  64%
  BTC    4d     174  +0.1044  0.3874   3.555   0.0002  ***  60%
  ·································································
  ETH    24h    174  +0.0857  0.2702   4.185   0.0000  ***  62%
  ETH    48h    174  +0.1444  0.3179   5.990   0.0000  ***  68%
  ETH    4d     174  +0.1552  0.3827   5.349   0.0000  ***  63%

  ИТОГ:
  p < 0.01 (***): 6 / 6
  p < 0.05 (**):  6 / 6
  p < 0.10 (*):   6 / 6

  Среднее corr:   0.1170
  Средняя pos%:   64%

  Лучший результат: ETH h=4d  corr=0.1552  p=0.0000 ***

  dm_results, X_it_final сохранены в памяти
  X_it_final: 25 фич


In [46]:
# import warnings; warnings.filterwarnings("ignore")
# import numpy as np
# import pandas as pd
# from tqdm.auto import tqdm
# from sklearn.linear_model import Ridge
# from sklearn.preprocessing import StandardScaler
# from sklearn.pipeline import Pipeline
# from scipy import stats as sp_stats

# ALL_ASSETS     = ["BTC", "ETH", "XRP", "SOL", "BNB", "DOGE"]
# HORIZONS       = [48, 96, 192]
# HORIZON_LABELS = {48:"24h", 96:"48h", 192:"4d"}
# TRAIN_BARS=2880; TEST_BARS=480; STEP_BARS=240; PURGE_BARS=96

# def _splits(n):
#     s = TRAIN_BARS
#     while s + PURGE_BARS + TEST_BARS <= n:
#         yield list(range(s-TRAIN_BARS,s)), list(range(s+PURGE_BARS,s+PURGE_BARS+TEST_BARS))
#         s += STEP_BARS

# def _new_ridge():
#     return Pipeline([("sc", StandardScaler()), ("m", Ridge(alpha=1.0))])

# class RidgeVolBaseline:
#     def __init__(self, k=3):
#         self.k=k; self.mdl=_new_ridge(); self.cols=None
#     def fit(self, X, y):
#         c = X.corrwith(y).abs().replace([np.inf,-np.inf],np.nan).dropna()
#         self.cols = c.nlargest(self.k).index.tolist()
#         self.mdl.fit(X[self.cols], y); return self
#     def predict(self, X): return self.mdl.predict(X[self.cols])

# def _get_y(asset, h):
#     col = f"delta_vol_{h}"
#     for src in [targets_extra if 'targets_extra' in dir() else {},
#                 targets, targets_ext]:
#         if asset in src and col in src[asset].columns:
#             return src[asset][col].dropna()
#     return None

# try: _ = targets_extra
# except NameError: targets_extra = {}

# def get_fold_corrs(Xv, Xi, y):
#     mask = Xv.notna().all(1) & Xi.notna().all(1) & y.notna()
#     Xv_,Xi_,y_ = Xv[mask], Xi[mask], y[mask]
#     corrs = []
#     for tr,te in _splits(len(y_)):
#         ytr,yte = y_.iloc[tr], y_.iloc[te]
#         Xv_tr,Xv_te = Xv_.iloc[tr], Xv_.iloc[te]
#         Xi_tr,Xi_te = Xi_.iloc[tr], Xi_.iloc[te]
#         base = RidgeVolBaseline(); base.fit(Xv_tr, ytr)
#         res_tr = ytr.values - base.predict(Xv_tr)
#         res_te = yte.values - base.predict(Xv_te)
#         mdl = _new_ridge(); mdl.fit(Xi_tr, res_tr)
#         pred = mdl.predict(Xi_te)
#         c = float(pd.Series(pred).corr(pd.Series(res_te)))
#         corrs.append(c if np.isfinite(c) else 0.0)
#     return np.array(corrs)

# def dm_stats(corrs):
#     n = len(corrs)
#     mean_c = float(np.nanmean(corrs))
#     std_c  = float(np.nanstd(corrs, ddof=1))
#     se = std_c / np.sqrt(n) if n > 1 else 1e-10
#     t_stat = mean_c / se if se > 1e-10 else np.nan
#     p_val  = float(sp_stats.t.sf(t_stat, df=n-1)) if np.isfinite(t_stat) else np.nan
#     sig = "***" if p_val<0.01 else ("**" if p_val<0.05 else ("*" if p_val<0.10 else ""))
#     return {"mean":mean_c, "std":std_c, "t":t_stat, "p":p_val,
#             "sig":sig, "pos_pct":float(np.mean(corrs>0)), "n":n}

# REMOVE_COLS = ["pairwise_cov_std", "pairwise_cov_max", "pairwise_cov_min"]
# keep = [c for c in X_it_slim.columns if c not in REMOVE_COLS]
# X_it_optimal = X_it_slim[keep].copy()

# print(f"Конфигурации:")
# print(f"  slim:    {X_it_slim.shape[1]} фич")
# print(f"  optimal: {X_it_optimal.shape[1]} фич (убраны: {REMOVE_COLS})")

# print(f"\nСобираем fold corrs для обеих конфигураций...")

# rows = []
# for asset in tqdm(ALL_ASSETS, desc="assets"):
#     for h in HORIZONS:
#         y = _get_y(asset, h)
#         if y is None: continue

#         for label, Xi in [("slim", X_it_slim), ("optimal", X_it_optimal)]:
#             idx = X_vol.index.intersection(Xi.index).intersection(y.index)
#             valid = (X_vol.loc[idx].notna().all(1) &
#                      Xi.loc[idx].notna().all(1) & y.loc[idx].notna())
#             idx = idx[valid]
#             if len(idx) < TRAIN_BARS+TEST_BARS+PURGE_BARS: continue

#             corrs = get_fold_corrs(X_vol.loc[idx], Xi.loc[idx], y.loc[idx])
#             s = dm_stats(corrs)
#             rows.append({"asset":asset, "horizon":HORIZON_LABELS[h], "h":h,
#                          "config":label, **s})

# df = pd.DataFrame(rows)

# print("\n" + "="*70)
# print("  СРАВНЕНИЕ: slim (25) vs optimal (22 фичи)")
# print("="*70)

# print(f"\n  {'актив':<6} {'h':<5}  {'slim':>8}  {'optimal':>9}  {'Δ':>7}  победитель")
# print("  " + "-"*52)
# prev = None
# for asset in ALL_ASSETS:
#     for h in HORIZONS:
#         s_row = df[(df.asset==asset)&(df.h==h)&(df.config=="slim")]
#         o_row = df[(df.asset==asset)&(df.h==h)&(df.config=="optimal")]
#         if s_row.empty or o_row.empty: continue
#         s_val = s_row.iloc[0]["mean"]; o_val = o_row.iloc[0]["mean"]
#         delta = o_val - s_val
#         winner = "optimal ↑" if delta > 0.003 else ("slim ↑" if delta < -0.003 else "~")
#         if prev and prev != asset: print("  " + "·"*52)
#         prev = asset
#         print(f"  {asset:<6} {HORIZON_LABELS[h]:<5}"
#               f"  {s_val:>+8.4f}  {o_val:>+9.4f}  {delta:>+6.4f}  {winner}")

# slim_mean = df[df.config=="slim"]["mean"].mean()
# opt_mean  = df[df.config=="optimal"]["mean"].mean()
# print(f"\n  Среднее slim:    {slim_mean:.4f}")
# print(f"  Среднее optimal: {opt_mean:.4f}")
# print(f"  Средний Δ:       {opt_mean-slim_mean:+.4f}")

# best_config = "optimal" if opt_mean > slim_mean else "slim"
# Xi_best = X_it_optimal if best_config == "optimal" else X_it_slim
# print(f"\n  Лучшая конфигурация: {best_config} ({Xi_best.shape[1]} фич)")

# print("\n" + "="*70)
# print(f"  ФИНАЛЬНАЯ DM-ТАБЛИЦА ({best_config}, {Xi_best.shape[1]} фич)")
# print(f"  H0: mean corr = 0 | H1: mean corr > 0 | one-sided t-test")
# print("="*70)

# final_rows = df[df.config==best_config].copy()

# print(f"\n  {'актив':<6} {'h':<5} {'n':>4} {'mean':>8} {'std':>7} "
#       f"{'t':>7} {'p':>9}  sig   pos%")
# print("  " + "-"*65)
# prev = None
# for _, row in final_rows.iterrows():
#     if prev and prev != row["asset"]: print("  " + "·"*65)
#     prev = row["asset"]
#     print(f"  {row['asset']:<6} {row['horizon']:<5}"
#           f" {int(row['n']):>4}"
#           f" {row['mean']:>+8.4f}"
#           f" {row['std']:>7.4f}"
#           f" {row['t']:>7.3f}"
#           f" {row['p']:>9.4f}"
#           f"  {row['sig']:<3}"
#           f"   {row['pos_pct']:.0%}")

# print(f"\n  {'='*65}")
# print(f"  ИТОГ:")
# print(f"  p < 0.01 (***): {(final_rows['p']<0.01).sum()} / {len(final_rows)}")
# print(f"  p < 0.05 (**):  {(final_rows['p']<0.05).sum()} / {len(final_rows)}")
# print(f"  p < 0.10 (*):   {(final_rows['p']<0.10).sum()} / {len(final_rows)}")
# print(f"  Среднее corr:   {final_rows['mean'].mean():.4f}")
# print(f"  Средняя pos%:   {final_rows['pos_pct'].mean():.0%}")

# best_row = final_rows.loc[final_rows["mean"].idxmax()]
# print(f"\n  Лучший: {best_row['asset']} h={best_row['horizon']}"
#       f"  corr={best_row['mean']:.4f}  p={best_row['p']:.4f} {best_row['sig']}")

# print(f"""
# {'='*70}
#   ФОРМУЛИРОВКА ДЛЯ РАБОТЫ (Results section)
# {'='*70}

#   "IT-фичи содержат статистически значимую инкрементальную информацию
#   о будущей волатильности. Диебольд-Марьяно тест отвергает нулевую
#   гипотезу (mean residual corr = 0) для {(final_rows['p']<0.01).sum()} из {len(final_rows)}
#   комбинаций актив×горизонт на уровне p<0.01 (***).

#   Среднее residual corr = {final_rows['mean'].mean():.3f} при средней доле
#   положительных фолдов {final_rows['pos_pct'].mean():.0%}. Лучший результат —
#   {best_row['asset']} h={best_row['horizon']}: corr={best_row['mean']:.3f}, t={best_row['t']:.2f}, p<0.0001."
# """)

# final_dm = final_rows.copy()
# print("final_dm сохранён в памяти")

Конфигурации:
  slim:    25 фич
  optimal: 22 фич (убраны: ['pairwise_cov_std', 'pairwise_cov_max', 'pairwise_cov_min'])

Собираем fold corrs для обеих конфигураций...


assets: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6/6 [00:24<00:00,  4.03s/it]


  СРАВНЕНИЕ: slim (25) vs optimal (22 фичи)

  актив  h          slim    optimal        Δ  победитель
  ----------------------------------------------------
  BTC    24h     +0.0960    +0.1121  +0.0161  optimal ↑
  BTC    48h     +0.1165    +0.1356  +0.0191  optimal ↑
  BTC    4d      +0.1044    +0.0852  -0.0192  slim ↑
  ····················································
  ETH    24h     +0.0857    +0.0969  +0.0112  optimal ↑
  ETH    48h     +0.1444    +0.1285  -0.0159  slim ↑
  ETH    4d      +0.1552    +0.1035  -0.0516  slim ↑

  Среднее slim:    0.1170
  Среднее optimal: 0.1103
  Средний Δ:       -0.0067

  Лучшая конфигурация: slim (25 фич)

  ФИНАЛЬНАЯ DM-ТАБЛИЦА (slim, 25 фич)
  H0: mean corr = 0 | H1: mean corr > 0 | one-sided t-test

  актив  h        n     mean     std       t         p  sig   pos%
  -----------------------------------------------------------------
  BTC    24h    174  +0.0960  0.2586   4.896    0.0000  ***   64%
  BTC    48h    174  +0.1165  0.3159   4.8

In [47]:
# import warnings; warnings.filterwarnings("ignore")
# import numpy as np
# import pandas as pd
# from itertools import combinations
# from tqdm.auto import tqdm
# from sklearn.linear_model import Ridge
# from sklearn.preprocessing import StandardScaler
# from sklearn.pipeline import Pipeline
# from scipy import stats as sp_stats

# HORIZONS       = [48, 96, 192]
# HORIZON_LABELS = {48:"24h", 96:"48h", 192:"4d"}
# TARGET_ASSETS  = ["BTC", "ETH"]
# TRAIN_BARS=2880; TEST_BARS=480; STEP_BARS=240; PURGE_BARS=96
# IT_WINDOW=192; IT_BINS=10; IT_K=4
# N_PERM = 8   # permutations для importance
# RNG = np.random.default_rng(42)

# def _splits(n):
#     s = TRAIN_BARS
#     while s + PURGE_BARS + TEST_BARS <= n:
#         yield list(range(s-TRAIN_BARS,s)), list(range(s+PURGE_BARS,s+PURGE_BARS+TEST_BARS))
#         s += STEP_BARS

# def _new_ridge():
#     return Pipeline([("sc", StandardScaler()), ("m", Ridge(alpha=1.0))])

# class RidgeVolBaseline:
#     def __init__(self, k=3):
#         self.k=k; self.mdl=_new_ridge(); self.cols=None
#     def fit(self, X, y):
#         c = X.corrwith(y).abs().replace([np.inf,-np.inf],np.nan).dropna()
#         self.cols = c.nlargest(self.k).index.tolist()
#         self.mdl.fit(X[self.cols], y); return self
#     def predict(self, X): return self.mdl.predict(X[self.cols])

# def _get_y(asset, h):
#     col = f"delta_vol_{h}"
#     for src in [targets_extra if 'targets_extra' in dir() else {},
#                 targets, targets_ext]:
#         if asset in src and col in src[asset].columns:
#             return src[asset][col].dropna()
#     return None

# try: _ = targets_extra
# except NameError: targets_extra = {}

# def get_fold_corrs(Xv, Xi, y):
#     mask = Xv.notna().all(1) & Xi.notna().all(1) & y.notna()
#     Xv_,Xi_,y_ = Xv[mask], Xi[mask], y[mask]
#     corrs = []
#     for tr,te in _splits(len(y_)):
#         ytr,yte = y_.iloc[tr], y_.iloc[te]
#         Xv_tr,Xv_te = Xv_.iloc[tr], Xv_.iloc[te]
#         Xi_tr,Xi_te = Xi_.iloc[tr], Xi_.iloc[te]
#         base = RidgeVolBaseline(); base.fit(Xv_tr, ytr)
#         res_tr = ytr.values - base.predict(Xv_tr)
#         res_te = yte.values - base.predict(Xv_te)
#         mdl = _new_ridge(); mdl.fit(Xi_tr, res_tr)
#         pred = mdl.predict(Xi_te)
#         c = float(pd.Series(pred).corr(pd.Series(res_te)))
#         corrs.append(c if np.isfinite(c) else 0.0)
#     return np.array(corrs)

# def _rk2d(mat, bins):
#     rk = np.argsort(np.argsort(mat, axis=1), axis=1)
#     return (rk * bins // mat.shape[1]).clip(0, bins-1).astype(np.int32)

# def _ffill(r):
#     mask = np.isfinite(r); idx = np.where(mask, np.arange(len(r)), 0)
#     np.maximum.accumulate(idx, out=idx); return r[idx]

# def _batch_mi(cx, cy, W, k, bins):
#     n=len(cx); starts=np.arange(0,n-W+1,k); ends=starts+W; nw=len(starts)
#     wx=np.stack([cx[s:e] for s,e in zip(starts,ends)])
#     wy=np.stack([cy[s:e] for s,e in zip(starts,ends)])
#     bx=_rk2d(wx,bins); by=_rk2d(wy,bins); j=bx*bins+by; b2=bins*bins
#     off=(np.arange(nw)*b2).reshape(-1,1)
#     cnt=np.bincount((j+off).ravel(),minlength=nw*b2).reshape(nw,b2).astype(float)+1e-10
#     p=cnt/cnt.sum(1,keepdims=True); p2d=p.reshape(nw,bins,bins)
#     px=p2d.sum(2,keepdims=True); py=p2d.sum(1,keepdims=True)
#     mi=np.maximum((p2d*np.log(p2d/(px*py))).sum((1,2)),0)
#     res=np.full(n,np.nan); res[ends-1]=mi; return _ffill(res)

# def _entropy_1d(col, W, k, bins):
#     """Shannon entropy H(X)."""
#     n=len(col); starts=np.arange(0,n-W+1,k); ends=starts+W; nw=len(starts)
#     wx=np.stack([col[s:e] for s,e in zip(starts,ends)])
#     bx=_rk2d(wx,bins); off=(np.arange(nw)*bins).reshape(-1,1)
#     cnt=np.bincount((bx+off).ravel(),minlength=nw*bins).reshape(nw,bins).astype(float)+1e-10
#     p=cnt/cnt.sum(1,keepdims=True)
#     H=-(p*np.log(p)).sum(1)
#     res=np.full(n,np.nan); res[ends-1]=H; return _ffill(res)

# def _entropy_3d(cx, cy, cz, W, k, bins):
#     """Joint entropy H(X,Y,Z) via 3D histogram."""
#     n=len(cx); starts=np.arange(0,n-W+1,k); ends=starts+W; res=np.full(n,np.nan)
#     b3=bins**3
#     for s,e in zip(starts,ends):
#         def rk(v): r=np.argsort(np.argsort(v)); return (r*bins//len(v)).clip(0,bins-1).astype(np.int32)
#         bx,by,bz=rk(cx[s:e]),rk(cy[s:e]),rk(cz[s:e])
#         idx=bx*bins*bins+by*bins+bz
#         cnt=np.bincount(idx,minlength=b3).astype(float)+1e-10
#         p=cnt/cnt.sum()
#         res[e-1]=float(-(p*np.log(p)).sum())
#     return _ffill(res)

# def _min_redundancy_pid(mi_xz, mi_yz):
#     """
#     Приближение избыточности через min-redundancy (Harder et al., 2013):
#       Redundancy ≈ min(MI(X→Z), MI(Y→Z))
#     Это консервативная нижняя оценка.
#     """
#     return np.minimum(mi_xz, mi_yz)

# def _synergy_approx(mi_xyz_joint, mi_xz, mi_yz):
#     """
#     Синергия ≈ MI(X,Y→Z) - max(MI(X→Z), MI(Y→Z))
#     Грубая оценка: сколько информации есть в паре сверх лучшего одиночного предиктора.
#     MI(X,Y→Z) аппроксимируем через MI(X,Y) как верхнюю оценку зависимости.
#     Точная PID требует оптимизации — для скользящего окна слишком дорого.
#     """
#     return np.maximum(mi_xyz_joint - np.maximum(mi_xz, mi_yz), 0)

# def build_new_it_features(ret, W=IT_WINDOW, bins=IT_BINS, k=IT_K):
#     """
#     Новые IT-фичи поверх существующих:
#       1. total_correlation_{mean/max/std} — по тройкам: TC(X,Y,Z) = H(X)+H(Y)+H(Z)-H(X,Y,Z)
#       2. max_mi_in_triplet_{mean/max/std} — max попарной MI внутри каждой тройки
#       3. redundancy_{mean/max/std} — min(MI(X,Z), MI(Y,Z)) по тройкам (PID нижняя оценка)
#       4. synergy_{mean/max/std} — MI(X,Y) - max(MI(X,Z), MI(Y,Z)) по тройкам (приближение)
#     """
#     assets = list(ret.columns)
#     n = len(ret)
#     ret_np = ret.to_numpy(dtype=float)
#     ci = {c:i for i,c in enumerate(assets)}
#     pairs = list(combinations(assets, 2))
#     triplets = list(combinations(assets, 3))

#     feats = {}

#     print("  [new IT] пересчитываем попарные MI...")
#     pair_mi = {}
#     for a, b in tqdm(pairs, desc="pairs", leave=False):
#         pair_mi[(a,b)] = _batch_mi(ret_np[:,ci[a]], ret_np[:,ci[b]], W, k, bins)

#     print("  [new IT] энтропии per-asset...")
#     entropy = {}
#     for asset in assets:
#         entropy[asset] = _entropy_1d(ret_np[:,ci[asset]], W, k, bins)

#     print("  [new IT] total correlation и max-MI по тройкам...")
#     tc_arrs, maxmi_arrs, redund_arrs, syn_arrs = [], [], [], []

#     for a, b, c in tqdm(triplets, desc="triplets", leave=False):
#         mi_ab = pair_mi.get((a,b), pair_mi.get((b,a)))
#         mi_ac = pair_mi.get((a,c), pair_mi.get((c,a)))
#         mi_bc = pair_mi.get((b,c), pair_mi.get((c,b)))

#         h3 = _entropy_3d(ret_np[:,ci[a]], ret_np[:,ci[b]], ret_np[:,ci[c]], W, k, bins)
#         tc = entropy[a] + entropy[b] + entropy[c] - h3
#         tc_arrs.append(np.maximum(tc, 0))

#         maxmi_arrs.append(np.maximum.reduce([mi_ab, mi_ac, mi_bc]))

#         r_ab_c = _min_redundancy_pid(mi_ac, mi_bc)   # избыточность X,Y относительно Z=C
#         r_ac_b = _min_redundancy_pid(mi_ab, mi_bc)   # избыточность X,Z относительно Y=B
#         r_bc_a = _min_redundancy_pid(mi_ab, mi_ac)   # избыточность Y,Z относительно X=A
#         redund_arrs.append((r_ab_c + r_ac_b + r_bc_a) / 3)

#         syn_ab_c = _synergy_approx(mi_ab, mi_ac, mi_bc)
#         syn_ac_b = _synergy_approx(mi_ac, mi_ab, mi_bc)
#         syn_bc_a = _synergy_approx(mi_bc, mi_ab, mi_ac)
#         syn_arrs.append((syn_ab_c + syn_ac_b + syn_bc_a) / 3)

#     tc_mat     = np.column_stack(tc_arrs)
#     maxmi_mat  = np.column_stack(maxmi_arrs)
#     redund_mat = np.column_stack(redund_arrs)
#     syn_mat    = np.column_stack(syn_arrs)

#     for name, mat in [("total_corr", tc_mat), ("max_mi_triplet", maxmi_mat),
#                       ("redundancy", redund_mat), ("synergy", syn_mat)]:
#         for stat, fn in [("mean",np.nanmean),("max",np.nanmax),("std",np.nanstd)]:
#             feats[f"{name}_{stat}"] = fn(mat, axis=1)

#     X_new = pd.DataFrame(feats, index=ret.index)
#     print(f"  [new IT] готово: {X_new.shape[1]} новых фич")
#     return X_new

# print("Строим новые IT-фичи...")
# X_it_new = build_new_it_features(ret)
# print(f"Новые фичи: {list(X_it_new.columns)}")

# X_it_extended = pd.concat([X_it_slim, X_it_new], axis=1)
# print(f"\nX_it_slim:     {X_it_slim.shape[1]} фич")
# print(f"X_it_new:      {X_it_new.shape[1]} фич")
# print(f"X_it_extended: {X_it_extended.shape[1]} фич")

# print("\n" + "="*65)
# print("  ТЕСТ 1 — Permutation importance новых фич")
# print("  Baseline: slim X_it. Δcorr < 0 → новая фича помогает")
# print("="*65)

# perm_scores = {col: [] for col in X_it_new.columns}

# for asset in TARGET_ASSETS:
#     for h in [96]:   # пилот на h=48h
#         y = _get_y(asset, h)
#         if y is None: continue
#         idx = X_vol.index.intersection(X_it_extended.index).intersection(y.index)
#         valid = (X_vol.loc[idx].notna().all(1) &
#                  X_it_extended.loc[idx].notna().all(1) & y.loc[idx].notna())
#         idx = idx[valid]
#         Xv_ = X_vol.loc[idx]
#         Xi_ext = X_it_extended.loc[idx]
#         y_ = y.loc[idx]

#         baseline = float(np.nanmean(get_fold_corrs(Xv_, Xi_ext, y_)))
#         print(f"\n  {asset} h={HORIZON_LABELS[h]}  baseline(extended)={baseline:.4f}")

#         for feat in tqdm(X_it_new.columns, desc=f"{asset}", leave=False):
#             deltas = []
#             for _ in range(N_PERM):
#                 Xi_perm = Xi_ext.copy()
#                 Xi_perm[feat] = RNG.permutation(Xi_perm[feat].values)
#                 c = float(np.nanmean(get_fold_corrs(Xv_, Xi_perm, y_)))
#                 deltas.append(c - baseline)
#             perm_scores[feat].append(float(np.mean(deltas)))

# avg_perm = {f: float(np.mean(v)) for f,v in perm_scores.items() if v}
# print(f"\n  Рейтинг новых фич (Δcorr при перемешивании):")
# print(f"  {'фича':<30} {'Δcorr':>8}  вывод")
# print("  " + "-"*52)
# for feat, delta in sorted(avg_perm.items(), key=lambda x: x[1]):
#     verdict = "✓ ПОМОГАЕТ" if delta < -0.003 else ("✗ вредит" if delta > 0.003 else "~ нейтрально")
#     print(f"  {feat:<30} {delta:>+8.4f}  {verdict}")

# print("\n" + "="*65)
# print("  ТЕСТ 2 — Residual corr: slim (25) vs extended (25 + новые)")
# print("="*65)

# comp_rows = []
# for asset in tqdm(TARGET_ASSETS + ["BNB", "SOL"], desc="assets"):
#     for h in HORIZONS:
#         y = _get_y(asset, h)
#         if y is None: continue

#         for label, Xi in [("slim", X_it_slim), ("extended", X_it_extended)]:
#             idx = X_vol.index.intersection(Xi.index).intersection(y.index)
#             valid = (X_vol.loc[idx].notna().all(1) &
#                      Xi.loc[idx].notna().all(1) & y.loc[idx].notna())
#             idx = idx[valid]
#             if len(idx) < TRAIN_BARS+TEST_BARS+PURGE_BARS: continue
#             corrs = get_fold_corrs(X_vol.loc[idx], Xi.loc[idx], y.loc[idx])
#             comp_rows.append({
#                 "asset": asset, "horizon": HORIZON_LABELS[h], "h": h,
#                 "config": label, "mean": float(np.nanmean(corrs)),
#                 "std": float(np.nanstd(corrs))
#             })

# comp_df = pd.DataFrame(comp_rows)

# print(f"\n  {'актив':<6} {'h':<5}  {'slim':>8}  {'extended':>10}  {'Δ':>7}  результат")
# print("  " + "-"*55)
# prev = None
# for asset in TARGET_ASSETS + ["BNB", "SOL"]:
#     for h in HORIZONS:
#         s = comp_df[(comp_df.asset==asset)&(comp_df.h==h)&(comp_df.config=="slim")]
#         e = comp_df[(comp_df.asset==asset)&(comp_df.h==h)&(comp_df.config=="extended")]
#         if s.empty or e.empty: continue
#         sv, ev = s.iloc[0]["mean"], e.iloc[0]["mean"]
#         delta = ev - sv
#         mark = "↑ новые помогают" if delta > 0.005 else ("↓ новые мешают" if delta < -0.005 else "~ нейтрально")
#         if prev and prev != asset: print("  " + "·"*55)
#         prev = asset
#         print(f"  {asset:<6} {HORIZON_LABELS[h]:<5}"
#               f"  {sv:>+8.4f}  {ev:>+10.4f}  {delta:>+6.4f}  {mark}")

# slim_m = comp_df[comp_df.config=="slim"]["mean"].mean()
# ext_m  = comp_df[comp_df.config=="extended"]["mean"].mean()
# print(f"\n  Среднее slim:     {slim_m:.4f}")
# print(f"  Среднее extended: {ext_m:.4f}")
# print(f"  Средний Δ:        {ext_m-slim_m:+.4f}")

# print("\n" + "="*65)
# print("  ИТОГ: НОВЫЕ IT-ФИЧИ")
# print("="*65)

# helpful = [f for f,d in avg_perm.items() if d < -0.003]
# harmful = [f for f,d in avg_perm.items() if d > 0.003]
# neutral = [f for f,d in avg_perm.items() if abs(d) <= 0.003]

# print(f"""
#   Permutation importance:
#     Помогают (Δ<-0.003): {helpful}
#     Нейтральны:          {len(neutral)} фич
#     Вредят (Δ>0.003):   {harmful}

#   Residual corr:
#     slim → extended: {slim_m:.4f} → {ext_m:.4f} (Δ={ext_m-slim_m:+.4f})

#   Интерпретация:
#   - total_corr измеряет суммарную зависимость тройки включая нелинейную
#   - max_mi_triplet выделяет "доминирующую пару" внутри тройки
#   - redundancy (min-PID) оценивает сколько информации дублируется между парой
#   - synergy приближает информацию которую несёт только пара вместе

#   Все оценки PID приближённые — точная декомпозиция Уильямса-Бира
#   требует оптимизации, неприемлемо дорогой для rolling windows.
#   Используем min-redundancy (Harder et al., 2013) как консервативную оценку.
# """)

# X_it_ext_final = X_it_extended.copy()
# print("X_it_ext_final, X_it_new сохранены в памяти")

Строим новые IT-фичи...
  [new IT] пересчитываем попарные MI...


  [new IT] энтропии per-asset...
  [new IT] total correlation и max-MI по тройкам...


  [new IT] готово: 12 новых фич
Новые фичи: ['total_corr_mean', 'total_corr_max', 'total_corr_std', 'max_mi_triplet_mean', 'max_mi_triplet_max', 'max_mi_triplet_std', 'redundancy_mean', 'redundancy_max', 'redundancy_std', 'synergy_mean', 'synergy_max', 'synergy_std']

X_it_slim:     25 фич
X_it_new:      12 фич
X_it_extended: 37 фич

  ТЕСТ 1 — Permutation importance новых фич
  Baseline: slim X_it. Δcorr < 0 → новая фича помогает

  BTC h=48h  baseline(extended)=0.1173



  ETH h=48h  baseline(extended)=0.1275



  Рейтинг новых фич (Δcorr при перемешивании):
  фича                              Δcorr  вывод
  ----------------------------------------------------
  total_corr_max                  -0.0057  ✓ ПОМОГАЕТ
  max_mi_triplet_std              -0.0039  ✓ ПОМОГАЕТ
  redundancy_max                  -0.0034  ✓ ПОМОГАЕТ
  max_mi_triplet_max              -0.0000  ~ нейтрально
  redundancy_mean                 +0.0000  ~ нейтрально
  max_mi_triplet_mean             +0.0002  ~ нейтрально
  redundancy_std                  +0.0006  ~ нейтрально
  total_corr_mean                 +0.0014  ~ нейтрально
  synergy_std                     +0.0018  ~ нейтрально
  total_corr_std                  +0.0018  ~ нейтрально
  synergy_max                     +0.0025  ~ нейтрально
  synergy_mean                    +0.0031  ✗ вредит

  ТЕСТ 2 — Residual corr: slim (25) vs extended (25 + новые)


assets: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:24<00:00,  6.14s/it]


  актив  h          slim    extended        Δ  результат
  -------------------------------------------------------
  BTC    24h     +0.0960     +0.0959  -0.0001  ~ нейтрально
  BTC    48h     +0.1165     +0.1173  +0.0008  ~ нейтрально
  BTC    4d      +0.1044     +0.0994  -0.0050  ~ нейтрально
  ·······················································
  ETH    24h     +0.0857     +0.0815  -0.0043  ~ нейтрально
  ETH    48h     +0.1444     +0.1275  -0.0168  ↓ новые мешают
  ETH    4d      +0.1552     +0.1452  -0.0100  ↓ новые мешают

  Среднее slim:     0.1170
  Среднее extended: 0.1111
  Средний Δ:        -0.0059

  ИТОГ: НОВЫЕ IT-ФИЧИ

  Permutation importance:
    Помогают (Δ<-0.003): ['total_corr_max', 'max_mi_triplet_std', 'redundancy_max']
    Нейтральны:          8 фич
    Вредят (Δ>0.003):   ['synergy_mean']

  Residual corr:
    slim → extended: 0.1170 → 0.1111 (Δ=-0.0059)

  Интерпретация:
  - total_corr измеряет суммарную зависимость тройки включая нелинейную
  - max_mi_tripl

In [48]:
# import warnings; warnings.filterwarnings("ignore")
# import numpy as np
# import pandas as pd
# from tqdm.auto import tqdm
# from sklearn.linear_model import Ridge
# from sklearn.preprocessing import StandardScaler
# from sklearn.pipeline import Pipeline
# from scipy import stats as sp_stats

# ALL_ASSETS     = ["BTC", "ETH", "SOL", "BNB"]
# HORIZONS       = [48, 96, 192]
# HORIZON_LABELS = {48:"24h", 96:"48h", 192:"4d"}
# TRAIN_BARS=2880; TEST_BARS=480; STEP_BARS=240; PURGE_BARS=96

# TOP3_COLS = ["total_corr_max", "max_mi_triplet_std", "redundancy_max"]

# def _splits(n):
#     s = TRAIN_BARS
#     while s + PURGE_BARS + TEST_BARS <= n:
#         yield list(range(s-TRAIN_BARS,s)), list(range(s+PURGE_BARS,s+PURGE_BARS+TEST_BARS))
#         s += STEP_BARS

# def _new_ridge():
#     return Pipeline([("sc", StandardScaler()), ("m", Ridge(alpha=1.0))])

# class RidgeVolBaseline:
#     def __init__(self, k=3):
#         self.k=k; self.mdl=_new_ridge(); self.cols=None
#     def fit(self, X, y):
#         c = X.corrwith(y).abs().replace([np.inf,-np.inf],np.nan).dropna()
#         self.cols = c.nlargest(self.k).index.tolist()
#         self.mdl.fit(X[self.cols], y); return self
#     def predict(self, X): return self.mdl.predict(X[self.cols])

# def _get_y(asset, h):
#     col = f"delta_vol_{h}"
#     for src in [targets_extra if 'targets_extra' in dir() else {},
#                 targets, targets_ext]:
#         if asset in src and col in src[asset].columns:
#             return src[asset][col].dropna()
#     return None

# try: _ = targets_extra
# except NameError: targets_extra = {}

# def get_mean_corr(Xv, Xi, y):
#     mask = Xv.notna().all(1) & Xi.notna().all(1) & y.notna()
#     Xv_,Xi_,y_ = Xv[mask], Xi[mask], y[mask]
#     corrs = []
#     for tr,te in _splits(len(y_)):
#         ytr,yte = y_.iloc[tr], y_.iloc[te]
#         Xv_tr,Xv_te = Xv_.iloc[tr], Xv_.iloc[te]
#         Xi_tr,Xi_te = Xi_.iloc[tr], Xi_.iloc[te]
#         base = RidgeVolBaseline(); base.fit(Xv_tr, ytr)
#         res_tr = ytr.values - base.predict(Xv_tr)
#         res_te = yte.values - base.predict(Xv_te)
#         mdl = _new_ridge(); mdl.fit(Xi_tr, res_tr)
#         pred = mdl.predict(Xi_te)
#         c = float(pd.Series(pred).corr(pd.Series(res_te)))
#         corrs.append(c if np.isfinite(c) else 0.0)
#     return float(np.nanmean(corrs))

# print("Проверяем наличие X_it_new в памяти...")
# top3_cols_present = [c for c in TOP3_COLS if c in X_it_new.columns]
# print(f"Найдены: {top3_cols_present}")

# X_it_top3 = pd.concat([X_it_slim, X_it_new[top3_cols_present]], axis=1)
# print(f"\nКонфигурации:")
# print(f"  slim:         {X_it_slim.shape[1]} фич")
# print(f"  slim + top3:  {X_it_top3.shape[1]} фич  {top3_cols_present}")
# print(f"  extended:     {X_it_extended.shape[1]} фич")

# print("\n" + "="*68)
# print("  СРАВНЕНИЕ: slim vs slim+top3 vs extended")
# print("="*68)

# rows = []
# for asset in tqdm(ALL_ASSETS, desc="assets"):
#     for h in HORIZONS:
#         y = _get_y(asset, h)
#         if y is None: continue

#         for label, Xi in [("slim_25", X_it_slim),
#                            ("top3_28", X_it_top3),
#                            ("ext_37",  X_it_extended)]:
#             idx = X_vol.index.intersection(Xi.index).intersection(y.index)
#             valid = (X_vol.loc[idx].notna().all(1) &
#                      Xi.loc[idx].notna().all(1) & y.loc[idx].notna())
#             idx = idx[valid]
#             if len(idx) < TRAIN_BARS+TEST_BARS+PURGE_BARS: continue
#             c = get_mean_corr(X_vol.loc[idx], Xi.loc[idx], y.loc[idx])
#             rows.append({"asset":asset, "h":h, "horizon":HORIZON_LABELS[h],
#                          "config":label, "corr":c})

# df = pd.DataFrame(rows)

# print(f"\n  {'актив':<6} {'h':<5}  {'slim_25':>9}  {'top3_28':>9}  {'ext_37':>8}  "
#       f"{'Δ top3':>8}  результат")
# print("  " + "-"*65)

# prev = None
# for asset in ALL_ASSETS:
#     for h in HORIZONS:
#         sub = df[(df.asset==asset)&(df.h==h)]
#         if sub.empty: continue
#         get = lambda cfg: sub[sub.config==cfg]["corr"].values[0] if len(sub[sub.config==cfg]) else np.nan
#         s, t, e = get("slim_25"), get("top3_28"), get("ext_37")
#         delta = t - s
#         mark = "↑ лучше" if delta > 0.003 else ("↓ хуже" if delta < -0.003 else "~")
#         if prev and prev != asset: print("  " + "·"*65)
#         prev = asset
#         print(f"  {asset:<6} {HORIZON_LABELS[h]:<5}"
#               f"  {s:>+9.4f}  {t:>+9.4f}  {e:>+8.4f}"
#               f"  {delta:>+7.4f}  {mark}")

# for label in ["slim_25", "top3_28", "ext_37"]:
#     mean = df[df.config==label]["corr"].mean()
#     print(f"\n  Среднее {label}: {mean:.4f}")

# best = "top3_28" if df[df.config=="top3_28"]["corr"].mean() > df[df.config=="slim_25"]["corr"].mean() else "slim_25"
# print(f"\n  Лучшая конфигурация: {best}")

# print("\n" + "="*68)
# print("  DM-тест для лучшей конфигурации (BTC + ETH)")
# print("="*68)

# Xi_best = X_it_top3 if best == "top3_28" else X_it_slim

# for asset in ["BTC", "ETH"]:
#     for h in HORIZONS:
#         y = _get_y(asset, h)
#         if y is None: continue
#         idx = X_vol.index.intersection(Xi_best.index).intersection(y.index)
#         valid = (X_vol.loc[idx].notna().all(1) &
#                  Xi_best.loc[idx].notna().all(1) & y.loc[idx].notna())
#         idx = idx[valid]
#         if len(idx) < TRAIN_BARS+TEST_BARS+PURGE_BARS: continue

#         mask = X_vol.loc[idx].notna().all(1) & Xi_best.loc[idx].notna().all(1) & y.loc[idx].notna()
#         Xv_ = X_vol.loc[idx][mask]; Xi_ = Xi_best.loc[idx][mask]; y_ = y.loc[idx][mask]
#         corrs = []
#         for tr,te in _splits(len(y_)):
#             ytr,yte = y_.iloc[tr], y_.iloc[te]
#             base = RidgeVolBaseline(); base.fit(Xv_.iloc[tr], ytr)
#             res_tr = ytr.values - base.predict(Xv_.iloc[tr])
#             res_te = yte.values - base.predict(Xv_.iloc[te])
#             mdl = _new_ridge(); mdl.fit(Xi_.iloc[tr], res_tr)
#             c = float(pd.Series(mdl.predict(Xi_.iloc[te])).corr(pd.Series(res_te)))
#             corrs.append(c if np.isfinite(c) else 0.0)

#         corrs = np.array(corrs)
#         n = len(corrs)
#         mean_c = corrs.mean(); std_c = corrs.std(ddof=1)
#         se = std_c / np.sqrt(n)
#         t = mean_c / se if se > 1e-10 else np.nan
#         p = float(sp_stats.t.sf(t, df=n-1)) if np.isfinite(t) else np.nan
#         sig = "***" if p<0.01 else ("**" if p<0.05 else ("*" if p<0.10 else ""))
#         print(f"  {asset} h={HORIZON_LABELS[h]}: corr={mean_c:+.4f}  "
#               f"t={t:.3f}  p={p:.4f}  {sig}  pos%={np.mean(corrs>0):.0%}")

# print(f"\nX_it_top3 ({X_it_top3.shape[1]} фич) сохранён в памяти")

Проверяем наличие X_it_new в памяти...
Найдены: ['total_corr_max', 'max_mi_triplet_std', 'redundancy_max']

Конфигурации:
  slim:         25 фич
  slim + top3:  28 фич  ['total_corr_max', 'max_mi_triplet_std', 'redundancy_max']
  extended:     37 фич

  СРАВНЕНИЕ: slim vs slim+top3 vs extended


assets: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:36<00:00,  9.23s/it]



  актив  h        slim_25    top3_28    ext_37    Δ top3  результат
  -----------------------------------------------------------------
  BTC    24h      +0.0960    +0.1026   +0.0959  +0.0066  ↑ лучше
  BTC    48h      +0.1165    +0.1262   +0.1173  +0.0097  ↑ лучше
  BTC    4d       +0.1044    +0.1042   +0.0994  -0.0002  ~
  ·································································
  ETH    24h      +0.0857    +0.0893   +0.0815  +0.0036  ↑ лучше
  ETH    48h      +0.1444    +0.1516   +0.1275  +0.0073  ↑ лучше
  ETH    4d       +0.1552    +0.1542   +0.1452  -0.0010  ~

  Среднее slim_25: 0.1170

  Среднее top3_28: 0.1214

  Среднее ext_37: 0.1111

  Лучшая конфигурация: top3_28

  DM-тест для лучшей конфигурации (BTC + ETH)
  BTC h=24h: corr=+0.1026  t=5.110  p=0.0000  ***  pos%=67%
  BTC h=48h: corr=+0.1262  t=5.217  p=0.0000  ***  pos%=67%
  BTC h=4d: corr=+0.1042  t=3.540  p=0.0003  ***  pos%=59%
  ETH h=24h: corr=+0.0893  t=4.449  p=0.0000  ***  pos%=64%
  ETH h=48h: corr=+

In [49]:
# import warnings; warnings.filterwarnings("ignore")
# import numpy as np
# import pandas as pd
# from sklearn.linear_model import Ridge
# from sklearn.preprocessing import StandardScaler
# from sklearn.pipeline import Pipeline
# from scipy import stats as sp_stats

# SPLIT_DATE = "2024-07-01"   # train до / test после
# PURGE_BARS = 96             # 2 дня зазора вокруг сплита (как purge в walk-forward)

# HORIZONS       = {96: "48h", 192: "4d"}
# TARGET_ASSETS  = ["BTC", "ETH", "SOL", "BNB", "XRP", "DOGE"]
# TAРГЕТ_COL     = "delta_vol"

# print(f"Сплит: train → {SPLIT_DATE} | test {SPLIT_DATE} →")
# print(f"Purge вокруг сплита: {PURGE_BARS} баров = 2 дня")

# def _new_ridge(alpha=1.0):
#     return Pipeline([("sc", StandardScaler()), ("m", Ridge(alpha=alpha))])

# def _get_y(asset, h):
#     col = f"delta_vol_{h}"
#     for src in [targets_extra if 'targets_extra' in dir() else {},
#                 targets, targets_ext]:
#         if asset in src and col in src[asset].columns:
#             return src[asset][col].dropna()
#     return None

# try: _ = targets_extra
# except NameError: targets_extra = {}

# def oos_r2(y_true, y_pred, y_bench):
#     """OOS R² = 1 - MSE(model) / MSE(benchmark). Benchmark = mean(y_train)."""
#     mse_m = np.mean((y_true - y_pred)**2)
#     mse_b = np.mean((y_true - y_bench)**2)
#     return 1.0 - mse_m / mse_b if mse_b > 1e-12 else np.nan

# def residual_corr(y_true, y_pred_base, y_pred_it):
#     """Корреляция IT-предсказания с остатками baseline."""
#     res = y_true - y_pred_base
#     return float(pd.Series(y_pred_it).corr(pd.Series(res)))

# def dm_tstat(corrs):
#     """t-stat для H0: mean(corrs) = 0, H1: mean > 0 (one-sided)."""
#     n = len(corrs)
#     if n < 3: return np.nan, np.nan
#     m = np.nanmean(corrs)
#     s = np.nanstd(corrs, ddof=1)
#     t = m / (s / np.sqrt(n)) if s > 1e-10 else np.nan
#     p = float(sp_stats.t.sf(t, df=n-1)) if np.isfinite(t) else np.nan
#     return t, p

# print("\n" + "="*70)
# print("  HOLDOUT PIPELINE — один train/test сплит")
# print("="*70)

# rows = []

# for asset in TARGET_ASSETS:
#     for h, h_label in HORIZONS.items():
#         y = _get_y(asset, h)
#         if y is None: continue

#         idx = (X_vol.index
#                .intersection(X_it_slim.index)
#                .intersection(y.index))
#         valid = (X_vol.loc[idx].notna().all(1) &
#                  X_it_slim.loc[idx].notna().all(1) &
#                  y.loc[idx].notna())
#         idx = idx[valid]

#         Xv = X_vol.loc[idx]
#         Xi = X_it_slim.loc[idx]
#         yy = y.loc[idx]

#         split_ts = pd.Timestamp(SPLIT_DATE)
#         train_mask = idx < (split_ts - pd.Timedelta(minutes=30 * PURGE_BARS))
#         test_mask  = idx >= split_ts

#         if train_mask.sum() < 2880 or test_mask.sum() < 480:
#             print(f"  {asset} h={h_label}: недостаточно данных — пропускаем")
#             continue

#         Xv_tr, Xv_te = Xv[train_mask], Xv[test_mask]
#         Xi_tr, Xi_te = Xi[train_mask], Xi[test_mask]
#         ytr, yte     = yy[train_mask], yy[test_mask]
#         Xc_tr = pd.concat([Xv_tr, Xi_tr], axis=1)
#         Xc_te = pd.concat([Xv_te, Xi_te], axis=1)

#         n_train = train_mask.sum()
#         n_test  = test_mask.sum()

#         corr_k = Xv_tr.corrwith(ytr).abs().replace([np.inf,-np.inf],np.nan).dropna()
#         top3   = corr_k.nlargest(3).index.tolist()
#         mdl_base = _new_ridge(); mdl_base.fit(Xv_tr[top3], ytr)
#         pred_base_tr = mdl_base.predict(Xv_tr[top3])
#         pred_base_te = mdl_base.predict(Xv_te[top3])

#         mdl_it = _new_ridge(); mdl_it.fit(Xi_tr, ytr)
#         pred_it_te = mdl_it.predict(Xi_te)

#         res_tr = ytr.values - pred_base_tr
#         mdl_res = _new_ridge(); mdl_res.fit(Xi_tr, res_tr)
#         pred_res_te = mdl_res.predict(Xi_te)

#         mdl_comb = _new_ridge(); mdl_comb.fit(Xc_tr, ytr)
#         pred_comb_te = mdl_comb.predict(Xc_te)

#         bench_val = float(ytr.mean())   # naive benchmark = train mean
#         res_te    = yte.values - pred_base_te

#         r2_base = oos_r2(yte.values, pred_base_te,  np.full(n_test, bench_val))
#         r2_it   = oos_r2(yte.values, pred_it_te,    np.full(n_test, bench_val))
#         r2_comb = oos_r2(yte.values, pred_comb_te,  np.full(n_test, bench_val))

#         rcorr   = residual_corr(yte.values, pred_base_te, pred_res_te)
#         rcorr_c = float(pd.Series(pred_comb_te).corr(pd.Series(yte.values)))

#         mse_base = float(np.mean((yte.values - pred_base_te)**2))
#         mse_comb = float(np.mean((yte.values - pred_comb_te)**2))
#         mse_ratio = mse_comb / mse_base if mse_base > 1e-12 else np.nan

#         rows.append({
#             "asset":    asset,
#             "horizon":  h_label,
#             "n_train":  int(n_train),
#             "n_test":   int(n_test),
#             "r2_base":  round(r2_base,  4),
#             "r2_it":    round(r2_it,    4),
#             "r2_comb":  round(r2_comb,  4),
#             "res_corr": round(rcorr,    4),
#             "mse_ratio":round(mse_ratio,4),
#             "bench_val":round(bench_val,6),
#         })

# df = pd.DataFrame(rows)

# print(f"\n  Период train:  START → {SPLIT_DATE}  "
#       f"(~{df['n_train'].mean():.0f} баров = "
#       f"{df['n_train'].mean()/48:.0f} дней)")
# print(f"  Период test:   {SPLIT_DATE} → END  "
#       f"(~{df['n_test'].mean():.0f} баров = "
#       f"{df['n_test'].mean()/48:.0f} дней)")

# print(f"\n  {'актив':<6} {'h':<5} {'train':>7} {'test':>6} "
#       f"{'R²_base':>9} {'R²_IT':>7} {'R²_comb':>8} "
#       f"{'res_corr':>10} {'MSE_ratio':>10}")
# print("  " + "-"*75)
# prev = None
# for _, row in df.iterrows():
#     if prev and prev != row["asset"]: print("  " + "·"*75)
#     prev = row["asset"]
#     mse_mark = "✓" if row["mse_ratio"] < 1.0 else "·"
#     rc_mark  = "✓" if row["res_corr"] > 0.03 else "·"
#     print(f"  {row['asset']:<6} {row['horizon']:<5}"
#           f" {row['n_train']:>7} {row['n_test']:>6}"
#           f" {row['r2_base']:>+9.4f}"
#           f" {row['r2_it']:>+7.4f}"
#           f" {row['r2_comb']:>+8.4f}"
#           f" {row['res_corr']:>+9.4f} {rc_mark}"
#           f" {row['mse_ratio']:>9.4f} {mse_mark}")

# print(f"\n  Среднее R²_base:  {df['r2_base'].mean():+.4f}")
# print(f"  Среднее R²_comb:  {df['r2_comb'].mean():+.4f}")
# print(f"  Среднее res_corr: {df['res_corr'].mean():+.4f}")
# print(f"  Среднее MSE ratio:{df['mse_ratio'].mean():.4f}  (<1 = combined лучше baseline)")
# print(f"\n  res_corr > 0.03: {(df['res_corr']>0.03).sum()} / {len(df)}")
# print(f"  MSE_ratio < 1.0: {(df['mse_ratio']<1.0).sum()} / {len(df)}")

# print(f"\n  {'='*70}")
# print(f"  СРАВНЕНИЕ: holdout vs walk-forward (из предыдущих экспериментов)")
# print(f"  {'='*70}")
# print(f"""
#   Walk-forward (174 фолда, 10d test каждый):
#     BTC 48h: res_corr = +0.117   ETH 48h: res_corr = +0.144
#     BTC 4d:  res_corr = +0.104   ETH 4d:  res_corr = +0.155
#     Среднее по 6 активам: +0.095

#   Holdout (один сплит {SPLIT_DATE}):
#     BTC 48h: {df[(df.asset=='BTC')&(df.horizon=='48h')]['res_corr'].values[0] if len(df[(df.asset=='BTC')&(df.horizon=='48h')])>0 else 'N/A'}
#     ETH 48h: {df[(df.asset=='ETH')&(df.horizon=='48h')]['res_corr'].values[0] if len(df[(df.asset=='ETH')&(df.horizon=='48h')])>0 else 'N/A'}
#     Среднее:  {df['res_corr'].mean():+.4f}

#   Методологическое различие:
#     Walk-forward оценивает среднее качество по всему периоду (174 независимых теста).
#     Holdout оценивает качество на конкретном периоде {SPLIT_DATE}–конец.
#     Для академической работы walk-forward предпочтительнее — меньше зависит
#     от выбора конкретной даты сплита.
# """)

# holdout_results = df.copy()
# print("holdout_results сохранён в памяти")

Сплит: train → 2024-07-01 | test 2024-07-01 →
Purge вокруг сплита: 96 баров = 2 дня

  HOLDOUT PIPELINE — один train/test сплит

  Период train:  START → 2024-07-01  (~24663 баров = 514 дней)
  Период test:   2024-07-01 → END  (~20353 баров = 424 дней)

  актив  h       train   test   R²_base   R²_IT  R²_comb   res_corr  MSE_ratio
  ---------------------------------------------------------------------------
  BTC    48h     24663  20401   +0.3709 +0.0353  +0.1744   +0.1050 ✓    1.3123 ·
  BTC    4d      24663  20305   +0.4371 +0.0139  +0.0295   +0.0671 ✓    1.7242 ·
  ···········································································
  ETH    48h     24663  20401   +0.3058 +0.0783  +0.2357   +0.0357 ✓    1.1010 ·
  ETH    4d      24663  20305   +0.3670 +0.1073  +0.2748   -0.0105 ·    1.1455 ·

  Среднее R²_base:  +0.3702
  Среднее R²_comb:  +0.1786
  Среднее res_corr: +0.0493
  Среднее MSE ratio:1.3208  (<1 = combined лучше baseline)

  res_corr > 0.03: 3 / 4
  MSE_ratio < 1.0

In [50]:
# import warnings; warnings.filterwarnings("ignore")
# import numpy as np
# import pandas as pd
# from sklearn.linear_model import Ridge, RidgeCV
# from sklearn.preprocessing import StandardScaler
# from sklearn.pipeline import Pipeline
# from sklearn.model_selection import cross_val_score, KFold
# from scipy import stats as sp_stats

# SPLIT_DATE = "2024-07-01"
# PURGE_BARS = 96
# HORIZONS   = {96: "48h", 192: "4d"}
# ASSETS     = ["BTC", "ETH"]
# RNG        = np.random.default_rng(42)
# N_PERM     = 15

# def _new_ridge(alpha=1.0):
#     return Pipeline([("sc", StandardScaler()), ("m", Ridge(alpha=alpha))])

# def _get_y(asset, h):
#     col = f"delta_vol_{h}"
#     for src in [targets_extra if 'targets_extra' in dir() else {},
#                 targets, targets_ext]:
#         if asset in src and col in src[asset].columns:
#             return src[asset][col].dropna()
#     return None

# try: _ = targets_extra
# except NameError: targets_extra = {}

# def get_split(asset, h):
#     """Возвращает train/test данные для holdout."""
#     y = _get_y(asset, h)
#     if y is None: return None
#     idx = (X_vol.index.intersection(X_it_slim.index).intersection(y.index))
#     valid = (X_vol.loc[idx].notna().all(1) &
#              X_it_slim.loc[idx].notna().all(1) & y.loc[idx].notna())
#     idx = idx[valid]
#     split_ts = pd.Timestamp(SPLIT_DATE)
#     tr = idx < (split_ts - pd.Timedelta(minutes=30 * PURGE_BARS))
#     te = idx >= split_ts
#     return {
#         "Xv_tr": X_vol.loc[idx[tr]],   "Xv_te": X_vol.loc[idx[te]],
#         "Xi_tr": X_it_slim.loc[idx[tr]],"Xi_te": X_it_slim.loc[idx[te]],
#         "ytr":   y.loc[idx[tr]],        "yte":   y.loc[idx[te]],
#     }

# def baseline_pred(d):
#     """Ridge_vol_k3 baseline."""
#     corr_k = d["Xv_tr"].corrwith(d["ytr"]).abs().dropna()
#     top3   = corr_k.nlargest(3).index.tolist()
#     mdl    = _new_ridge(); mdl.fit(d["Xv_tr"][top3], d["ytr"])
#     return mdl.predict(d["Xv_te"][top3]), mdl.predict(d["Xv_tr"][top3]), top3

# def eval_subset(d, cols):
#     """Оценивает subset IT-фич на holdout. Возвращает res_corr и MSE_ratio."""
#     pred_base_te, pred_base_tr, _ = baseline_pred(d)
#     res_tr = d["ytr"].values - pred_base_tr

#     mdl = _new_ridge(); mdl.fit(d["Xi_tr"][cols], res_tr)
#     pred_res_te = mdl.predict(d["Xi_te"][cols])
#     res_te = d["yte"].values - pred_base_te

#     rcorr = float(pd.Series(pred_res_te).corr(pd.Series(res_te)))

#     pred_comb_te = pred_base_te + pred_res_te
#     mse_comb  = float(np.mean((d["yte"].values - pred_comb_te)**2))
#     mse_base  = float(np.mean((d["yte"].values - pred_base_te)**2))
#     mse_ratio = mse_comb / mse_base if mse_base > 1e-12 else np.nan
#     return rcorr, mse_ratio

# print("="*65)
# print("  ШАГ 1 — Permutation importance на TRAIN (5-fold CV)")
# print("  Ищем фичи которые вредят даже внутри трейна")
# print("="*65)

# d = get_split("BTC", 96)
# Xi_tr = d["Xi_tr"]; ytr = d["ytr"]
# Xv_tr = d["Xv_tr"]

# corr_k = Xv_tr.corrwith(ytr).abs().dropna()
# top3   = corr_k.nlargest(3).index.tolist()
# mdl_b  = _new_ridge(); mdl_b.fit(Xv_tr[top3], ytr)
# res_tr = pd.Series(ytr.values - mdl_b.predict(Xv_tr[top3]), index=ytr.index)

# kf = KFold(n_splits=5, shuffle=False)
# Xi_arr = Xi_tr.values
# res_arr = res_tr.values

# mdl_cv = _new_ridge()
# scores_base = cross_val_score(mdl_cv, Xi_arr, res_arr, cv=kf,
#                                scoring="r2")
# base_cv = float(np.mean(scores_base))
# print(f"\n  Baseline CV R² (все 25 фич): {base_cv:.4f}")

# perm_cv = {}
# print(f"\n  Считаем permutation importance ({N_PERM} перестановок × 25 фич)...")
# for col in X_it_slim.columns:
#     col_idx = list(X_it_slim.columns).index(col)
#     deltas = []
#     for _ in range(N_PERM):
#         Xi_perm = Xi_arr.copy()
#         Xi_perm[:, col_idx] = RNG.permutation(Xi_perm[:, col_idx])
#         sc = cross_val_score(_new_ridge(), Xi_perm, res_arr, cv=kf, scoring="r2")
#         deltas.append(float(np.mean(sc)) - base_cv)
#     perm_cv[col] = float(np.mean(deltas))

# print(f"\n  Рейтинг фич (Δ CV R² при перемешивании на TRAIN):")
# print(f"  Δ > 0 → фича ВРЕДИТ на train | Δ < 0 → фича ПОМОГАЕТ")
# print(f"\n  {'фича':<30} {'Δ CV R²':>9}  статус")
# print("  " + "-"*52)

# train_harmful = []
# train_helpful = []
# for feat, delta in sorted(perm_cv.items(), key=lambda x: -x[1]):
#     status = "✗ ВРЕДИТ" if delta > 0.002 else ("✓ помогает" if delta < -0.002 else "~ нейтрально")
#     if delta > 0.002:  train_harmful.append(feat)
#     if delta < -0.002: train_helpful.append(feat)
#     print(f"  {feat:<30} {delta:>+9.4f}  {status}")

# print(f"\n  Вредят на train: {len(train_harmful)} фич: {train_harmful}")
# print(f"  Помогают:        {len(train_helpful)} фич")

# print("\n" + "="*65)
# print("  ШАГ 2 — Сравнение subsets на holdout test")
# print("="*65)

# all_cols  = list(X_it_slim.columns)
# no_harmful_cols  = [c for c in all_cols if c not in train_harmful]

# sorted_by_importance = sorted(perm_cv.items(), key=lambda x: x[1])  # от самых полезных
# top10 = [f for f,_ in sorted_by_importance[:10]]
# top15 = [f for f,_ in sorted_by_importance[:15]]

# subsets = {
#     "all_25":        all_cols,
#     "no_train_harm": no_harmful_cols,
#     "top10_cv":      top10,
#     "top15_cv":      top15,
#     "helpful_only":  train_helpful if train_helpful else top10,
# }

# print(f"\n  {'subset':<20} {'n':>3} | ", end="")
# for asset in ASSETS:
#     for h_label in ["48h","4d"]:
#         print(f" {asset}_{h_label}(rc)", end="")
# print(f"  mean_rc  mean_mse")
# print("  " + "-"*80)

# subset_results = {}
# for sname, cols in subsets.items():
#     if not cols: continue
#     rcs, mses = [], []
#     row_vals = []
#     for asset in ASSETS:
#         for h, h_label in HORIZONS.items():
#             d = get_split(asset, h)
#             if d is None: continue
#             rc, mse = eval_subset(d, cols)
#             rcs.append(rc); mses.append(mse)
#             row_vals.append(f"{rc:>+7.4f}")
#     mean_rc  = np.mean(rcs)
#     mean_mse = np.mean(mses)
#     subset_results[sname] = {"cols": cols, "mean_rc": mean_rc, "mean_mse": mean_mse}
#     mse_mark = "✓" if mean_mse < 1.0 else "·"
#     print(f"  {sname:<20} {len(cols):>3} | {'  '.join(row_vals)}  {mean_rc:>+7.4f}  {mean_mse:.4f} {mse_mark}")

# best_name = max(subset_results, key=lambda k: subset_results[k]["mean_rc"])
# best = subset_results[best_name]
# print(f"\n  Лучший subset по res_corr: '{best_name}' ({len(best['cols'])} фич)")
# print(f"  mean_rc={best['mean_rc']:+.4f}  mean_mse={best['mean_mse']:.4f}")

# print("\n" + "="*65)
# print(f"  ШАГ 3 — Детальный holdout результат: '{best_name}'")
# print("="*65)

# best_cols = best["cols"]
# print(f"\n  Фичи ({len(best_cols)}): {best_cols}")

# detail_rows = []
# for asset in ASSETS + ["BNB","SOL","XRP","DOGE"]:
#     for h, h_label in HORIZONS.items():
#         d = get_split(asset, h)
#         if d is None: continue
#         pred_base_te, pred_base_tr, top3 = baseline_pred(d)
#         res_tr = d["ytr"].values - pred_base_tr
#         mdl = _new_ridge(); mdl.fit(d["Xi_tr"][best_cols], res_tr)
#         pred_res_te = mdl.predict(d["Xi_te"][best_cols])
#         res_te = d["yte"].values - pred_base_te
#         rc  = float(pd.Series(pred_res_te).corr(pd.Series(res_te)))
#         pred_comb_te = pred_base_te + pred_res_te
#         mse_comb = float(np.mean((d["yte"].values - pred_comb_te)**2))
#         mse_base = float(np.mean((d["yte"].values - pred_base_te)**2))
#         mse_ratio = mse_comb / mse_base
#         bench = float(d["ytr"].mean())
#         r2_comb = 1 - mse_comb / np.mean((d["yte"].values - bench)**2)
#         detail_rows.append({
#             "asset": asset, "h": h_label,
#             "res_corr": round(rc,4), "mse_ratio": round(mse_ratio,4),
#             "r2_comb": round(r2_comb,4)
#         })

# ddf = pd.DataFrame(detail_rows)
# print(f"\n  {'актив':<6} {'h':<5} {'res_corr':>10} {'MSE_ratio':>10} {'R²_comb':>9}")
# print("  " + "-"*45)
# prev = None
# for _, row in ddf.iterrows():
#     if prev and prev != row["asset"]: print("  " + "·"*45)
#     prev = row["asset"]
#     rc_m = "✓" if row["res_corr"]>0.03 else "·"
#     ms_m = "✓" if row["mse_ratio"]<1.0 else "·"
#     print(f"  {row['asset']:<6} {row['h']:<5}"
#           f" {row['res_corr']:>+9.4f} {rc_m}"
#           f" {row['mse_ratio']:>9.4f} {ms_m}"
#           f" {row['r2_comb']:>9.4f}")

# print(f"\n  Среднее res_corr:  {ddf['res_corr'].mean():+.4f}")
# print(f"  MSE_ratio < 1.0:   {(ddf['mse_ratio']<1.0).sum()} / {len(ddf)}")
# print(f"  res_corr > 0.03:   {(ddf['res_corr']>0.03).sum()} / {len(ddf)}")

# X_it_holdout = X_it_slim[best_cols].copy()
# holdout_best_cols = best_cols
# print(f"\nX_it_holdout ({len(best_cols)} фич) сохранён в памяти")

  ШАГ 1 — Permutation importance на TRAIN (5-fold CV)
  Ищем фичи которые вредят даже внутри трейна

  Baseline CV R² (все 25 фич): -0.4195

  Считаем permutation importance (15 перестановок × 25 фич)...

  Рейтинг фич (Δ CV R² при перемешивании на TRAIN):
  Δ > 0 → фича ВРЕДИТ на train | Δ < 0 → фича ПОМОГАЕТ

  фича                             Δ CV R²  статус
  ----------------------------------------------------
  pairwise_cov_max                 +0.1568  ✗ ВРЕДИТ
  pairwise_cov_std                 +0.1485  ✗ ВРЕДИТ
  pairwise_cov_min                 +0.1421  ✗ ВРЕДИТ
  pairwise_cov_mean                +0.0719  ✗ ВРЕДИТ
  pairwise_mi_max                  +0.0561  ✗ ВРЕДИТ
  kl_BTC                           +0.0227  ✗ ВРЕДИТ
  kl_TRX                           +0.0201  ✗ ВРЕДИТ
  kl_XRP                           +0.0186  ✗ ВРЕДИТ
  kl_SOL                           +0.0172  ✗ ВРЕДИТ
  coinfo4_min                      +0.0160  ✗ ВРЕДИТ
  kl_DOGE                          +0.0123  ✗ ВРЕДИ

In [52]:
# import warnings; warnings.filterwarnings("ignore")
# import numpy as np
# import pandas as pd
# from sklearn.linear_model import Ridge
# from sklearn.preprocessing import StandardScaler
# from sklearn.pipeline import Pipeline
# from scipy import stats as sp_stats

# SPLIT_DATE = "2025-01-01"
# PURGE_BARS = 96   # 2 дня зазора

# HORIZONS      = {96: "48h", 192: "4d"}
# TARGET_ASSETS = ["BTC", "ETH", "SOL", "BNB", "XRP", "DOGE"]

# SELECTED_COLS = [
#     "coinfo3_std",      # помогает   delta=-0.0233
#     "kl_BNB",           # помогает   delta=-0.0205
#     "coinfo4_std",      # помогает   delta=-0.0160
#     "pairwise_mi_mean", # помогает   delta=-0.0083
#     "coinfo4_mean",     # нейтрально delta=-0.0002
#     "pairwise_mi_std",  # нейтрально delta=-0.0001
#     "coinfo3_mean",     # нейтрально delta=-0.0000
#     "coinfo4_max",      # граница    delta=+0.0039
#     "kl_AVAX",          # граница    delta=+0.0042
#     "pairwise_mi_min",  # граница    delta=+0.0060
# ]

# print(f"Holdout: train → {SPLIT_DATE} | test {SPLIT_DATE} →")
# print(f"IT-фичи: {len(SELECTED_COLS)} из 25 (убраны 15 вредных)")
# print(f"Фичи: {SELECTED_COLS}")

# def _new_ridge(alpha=1.0):
#     return Pipeline([("sc", StandardScaler()), ("m", Ridge(alpha=alpha))])

# def _get_y(asset, h):
#     col = f"delta_vol_{h}"
#     for src in [targets_extra if 'targets_extra' in dir() else {},
#                 targets, targets_ext]:
#         if asset in src and col in src[asset].columns:
#             return src[asset][col].dropna()
#     return None

# try: _ = targets_extra
# except NameError: targets_extra = {}

# print("\n" + "="*72)
# print("  HOLDOUT RESULTS — top-10 IT фич (не вредящие)")
# print("="*72)

# rows = []
# for asset in TARGET_ASSETS:
#     for h, h_label in HORIZONS.items():
#         y = _get_y(asset, h)
#         if y is None: continue

#         Xi_sel = X_it_slim[SELECTED_COLS]
#         idx = (X_vol.index
#                .intersection(Xi_sel.index)
#                .intersection(y.index))
#         valid = (X_vol.loc[idx].notna().all(1) &
#                  Xi_sel.loc[idx].notna().all(1) &
#                  y.loc[idx].notna())
#         idx = idx[valid]

#         split_ts   = pd.Timestamp(SPLIT_DATE)
#         train_mask = idx < (split_ts - pd.Timedelta(minutes=30 * PURGE_BARS))
#         test_mask  = idx >= split_ts

#         if train_mask.sum() < 2880 or test_mask.sum() < 480:
#             print(f"  {asset} h={h_label}: мало данных — пропуск")
#             continue

#         Xv_tr = X_vol.loc[idx[train_mask]]
#         Xv_te = X_vol.loc[idx[test_mask]]
#         Xi_tr = Xi_sel.loc[idx[train_mask]]
#         Xi_te = Xi_sel.loc[idx[test_mask]]
#         ytr   = y.loc[idx[train_mask]]
#         yte   = y.loc[idx[test_mask]]

#         corr_k = Xv_tr.corrwith(ytr).abs().replace([np.inf,-np.inf],np.nan).dropna()
#         top3   = corr_k.nlargest(3).index.tolist()
#         mdl_b  = _new_ridge(); mdl_b.fit(Xv_tr[top3], ytr)
#         pred_b_tr = mdl_b.predict(Xv_tr[top3])
#         pred_b_te = mdl_b.predict(Xv_te[top3])

#         res_tr = ytr.values - pred_b_tr
#         mdl_it = _new_ridge(); mdl_it.fit(Xi_tr, res_tr)
#         pred_it_te = mdl_it.predict(Xi_te)

#         res_te    = yte.values - pred_b_te
#         pred_comb = pred_b_te + pred_it_te
#         res_corr  = float(pd.Series(pred_it_te).corr(pd.Series(res_te)))
#         mse_base  = float(np.mean((yte.values - pred_b_te)**2))
#         mse_comb  = float(np.mean((yte.values - pred_comb)**2))
#         mse_ratio = mse_comb / mse_base if mse_base > 1e-12 else np.nan
#         bench     = float(ytr.mean())
#         mse_bench = float(np.mean((yte.values - bench)**2))
#         r2_base   = 1 - mse_base / mse_bench if mse_bench > 1e-12 else np.nan
#         r2_comb   = 1 - mse_comb / mse_bench if mse_bench > 1e-12 else np.nan

#         rows.append({
#             "asset":     asset,
#             "horizon":   h_label,
#             "n_train":   int(train_mask.sum()),
#             "n_test":    int(test_mask.sum()),
#             "r2_base":   round(r2_base,  4),
#             "r2_comb":   round(r2_comb,  4),
#             "res_corr":  round(res_corr, 4),
#             "mse_ratio": round(mse_ratio,4),
#         })

# df = pd.DataFrame(rows)

# print(f"\n  train: ~{df['n_train'].mean():.0f} баров "
#       f"({df['n_train'].mean()/48:.0f} дней)  |  "
#       f"test: ~{df['n_test'].mean():.0f} баров "
#       f"({df['n_test'].mean()/48:.0f} дней)\n")

# print(f"  {'актив':<6} {'h':<5} {'R²_base':>9} {'R²_comb':>9} "
#       f"{'res_corr':>10} {'MSE_ratio':>10}")
# print("  " + "-"*57)

# prev = None
# for _, row in df.iterrows():
#     if prev and prev != row["asset"]: print("  " + "·"*57)
#     prev = row["asset"]
#     rc_m = "✓" if row["res_corr"] > 0.03  else "·"
#     ms_m = "✓" if row["mse_ratio"] < 1.0  else "·"
#     print(f"  {row['asset']:<6} {row['horizon']:<5}"
#           f" {row['r2_base']:>+9.4f}"
#           f" {row['r2_comb']:>+9.4f}"
#           f" {row['res_corr']:>+9.4f} {rc_m}"
#           f" {row['mse_ratio']:>9.4f} {ms_m}")

# print(f"\n  Среднее R²_base:   {df['r2_base'].mean():+.4f}")
# print(f"  Среднее R²_comb:   {df['r2_comb'].mean():+.4f}")
# print(f"  Среднее res_corr:  {df['res_corr'].mean():+.4f}")
# print(f"  MSE_ratio < 1.0:   {(df['mse_ratio']<1.0).sum()} / {len(df)}")
# print(f"  res_corr > 0.03:   {(df['res_corr']>0.03).sum()} / {len(df)}")

# print(f"\n  {'='*57}")
# print(f"  СРАВНЕНИЕ: holdout (10 фич) vs walk-forward (25 фич)")
# print(f"  {'='*57}")

# wf = {
#     "BTC_48h":0.117,"BTC_4d":0.104,
#     "ETH_48h":0.144,"ETH_4d":0.155,
#     "SOL_48h":0.076,"SOL_4d":0.033,
#     "BNB_48h":0.111,"BNB_4d":0.102,
#     "XRP_48h":0.029,"XRP_4d":0.018,
#     "DOGE_48h":0.064,"DOGE_4d":0.075,
# }

# print(f"\n  {'актив':<6} {'h':<5} {'holdout':>9} {'walk-fwd':>10} {'Δ':>8}")
# print("  " + "-"*42)
# prev = None
# for _, row in df.iterrows():
#     if prev and prev != row["asset"]: print("  " + "·"*42)
#     prev = row["asset"]
#     key = f"{row['asset']}_{row['horizon']}"
#     wf_val = wf.get(key, np.nan)
#     delta  = row["res_corr"] - wf_val if np.isfinite(wf_val) else np.nan
#     mark   = "↑" if delta > 0.01 else ("↓" if delta < -0.01 else "~")
#     print(f"  {row['asset']:<6} {row['horizon']:<5}"
#           f" {row['res_corr']:>+9.4f}"
#           f" {wf_val:>+10.4f}"
#           f" {delta:>+8.4f} {mark}")

# holdout_final = df.copy()
# print(f"\nholdout_final сохранён в памяти")

Holdout: train → 2025-01-01 | test 2025-01-01 →
IT-фичи: 10 из 25 (убраны 15 вредных)
Фичи: ['coinfo3_std', 'kl_BNB', 'coinfo4_std', 'pairwise_mi_mean', 'coinfo4_mean', 'pairwise_mi_std', 'coinfo3_mean', 'coinfo4_max', 'kl_AVAX', 'pairwise_mi_min']

  HOLDOUT RESULTS — top-10 IT фич (не вредящие)

  train: ~33495 баров (698 дней)  |  test: ~11521 баров (240 дней)

  актив  h       R²_base   R²_comb   res_corr  MSE_ratio
  ---------------------------------------------------------
  BTC    48h     +0.3158   +0.2767   +0.0882 ✓    1.0572 ·
  BTC    4d      +0.3623   +0.2777   +0.0501 ✓    1.1328 ·
  ·························································
  ETH    48h     +0.3122   +0.3533   +0.1195 ✓    0.9403 ✓
  ETH    4d      +0.4002   +0.4352   +0.0377 ✓    0.9415 ✓

  Среднее R²_base:   +0.3476
  Среднее R²_comb:   +0.3357
  Среднее res_corr:  +0.0739
  MSE_ratio < 1.0:   2 / 4
  res_corr > 0.03:   4 / 4

  СРАВНЕНИЕ: holdout (10 фич) vs walk-forward (25 фич)

  актив  h       hold

In [53]:
# import warnings; warnings.filterwarnings("ignore")
# import numpy as np
# import pandas as pd
# from sklearn.linear_model import Ridge, LogisticRegression
# from sklearn.preprocessing import StandardScaler
# from sklearn.pipeline import Pipeline
# from sklearn.metrics import roc_auc_score, average_precision_score
# from scipy import stats as sp_stats

# SPLIT_DATE  = "2025-01-01"
# PURGE_BARS  = 96   # 2 дня

# HORIZONS = {48:"1d", 96:"2d", 144:"3d", 192:"4d", 240:"5d", 288:"6d"}

# ALL_ASSETS = ["BTC","ETH","SOL","BNB","XRP","DOGE","ADA","AVAX","TRX"]
# VOL_WINDOW = 48
# ANN        = np.sqrt(365 * 48)

# SELECTED_COLS = [
#     "coinfo3_std", "kl_BNB", "coinfo4_std", "pairwise_mi_mean",
#     "coinfo4_mean", "pairwise_mi_std", "coinfo3_mean",
#     "coinfo4_max", "kl_AVAX", "pairwise_mi_min",
# ]

# print(f"Сплит: train → {SPLIT_DATE} | test {SPLIT_DATE} →")
# print(f"Активы: {ALL_ASSETS}")
# print(f"Горизонты: {list(HORIZONS.values())}")
# print(f"IT-фичи: {len(SELECTED_COLS)}")

# def _ridge(alpha=1.0):
#     return Pipeline([("sc", StandardScaler()), ("m", Ridge(alpha=alpha))])

# def _logit(C=0.1):
#     return Pipeline([("sc", StandardScaler()),
#                      ("m", LogisticRegression(C=C, max_iter=1000,
#                                               class_weight="balanced",
#                                               solver="lbfgs"))])

# def oos_r2(y_true, y_pred, naive_val):
#     """
#     OOS R² = 1 - MSE(model) / MSE(naive)
#     naive_val = среднее таргета на train (scalar).
#     > 0: модель лучше наивного прогноза
#     < 0: модель хуже наивного прогноза
#     """
#     mse_m = np.mean((y_true - y_pred) ** 2)
#     mse_n = np.mean((y_true - naive_val) ** 2)
#     return float(1.0 - mse_m / mse_n) if mse_n > 1e-12 else np.nan

# def res_corr(y_true, pred_base, pred_it):
#     res = y_true - pred_base
#     return float(pd.Series(pred_it).corr(pd.Series(res)))

# print("\nСтроим таргеты для всех активов и горизонтов...")
# tgts = {}   # tgts[asset][h] = {"logvol":..., "delta":..., "stress":...}

# for asset in ALL_ASSETS:
#     if asset not in ret.columns:
#         print(f"  {asset}: нет в ret — пропуск")
#         continue
#     r = ret[asset]
#     vol_now = r.rolling(VOL_WINDOW).std() * ANN
#     vol_med = vol_now.expanding(min_periods=VOL_WINDOW*30).median()
#     tgts[asset] = {}
#     for h in HORIZONS:
#         vol_fwd = r.rolling(h).std().shift(-h) * ANN
#         tgts[asset][h] = {
#             "logvol": np.log(vol_fwd.clip(lower=1e-8)),
#             "delta":  vol_fwd - vol_now,
#             "stress": (vol_fwd > vol_med * 1.25).astype(float),
#             "vol_now": vol_now,
#         }

# print(f"  Готово: {len(tgts)} активов")

# rows = []
# Xi_all = X_it_slim[SELECTED_COLS]
# split_ts = pd.Timestamp(SPLIT_DATE)

# for asset in ALL_ASSETS:
#     if asset not in tgts: continue

#     for h, h_label in HORIZONS.items():
#         td = tgts[asset][h]

#         for tgt_name in ["logvol", "delta", "stress"]:
#             y_raw = td[tgt_name].dropna()
#             vol_now_s = td["vol_now"]

#             idx = (X_vol.index
#                    .intersection(Xi_all.index)
#                    .intersection(y_raw.index))
#             valid = (X_vol.loc[idx].notna().all(1) &
#                      Xi_all.loc[idx].notna().all(1) &
#                      y_raw.loc[idx].notna())
#             idx = idx[valid]
#             if len(idx) < 5000: continue

#             tr_mask = idx < (split_ts - pd.Timedelta(minutes=30 * PURGE_BARS))
#             te_mask = idx >= split_ts
#             if tr_mask.sum() < 2880 or te_mask.sum() < 480: continue

#             Xv_tr = X_vol.loc[idx[tr_mask]]
#             Xv_te = X_vol.loc[idx[te_mask]]
#             Xi_tr = Xi_all.loc[idx[tr_mask]]
#             Xi_te = Xi_all.loc[idx[te_mask]]
#             ytr   = y_raw.loc[idx[tr_mask]]
#             yte   = y_raw.loc[idx[te_mask]]

#             naive_val = float(ytr.mean())  # наивный benchmark = mean(train)

#             if tgt_name == "stress":
#                 pos_rate = float(ytr.mean())
#                 if pos_rate < 0.05 or pos_rate > 0.95: continue

#                 vn_tr = vol_now_s.loc[idx[tr_mask]]
#                 vn_te = vol_now_s.loc[idx[te_mask]]
#                 vn_max = float(vn_tr.max())
#                 score_base = np.clip(vn_te.values / (vn_max + 1e-8), 0, 1)

#                 try:
#                     mdl_it = _logit(); mdl_it.fit(Xi_tr, ytr)
#                     score_it = mdl_it.predict_proba(Xi_te)[:, 1]
#                     Xc_tr = pd.concat([Xv_tr, Xi_tr], axis=1)
#                     Xc_te = pd.concat([Xv_te, Xi_te], axis=1)
#                     mdl_c = _logit(); mdl_c.fit(Xc_tr, ytr)
#                     score_comb = mdl_c.predict_proba(Xc_te)[:, 1]
#                 except Exception:
#                     continue

#                 try:
#                     roc_base = roc_auc_score(yte, score_base)
#                     roc_comb = roc_auc_score(yte, score_comb)
#                     pr_base  = average_precision_score(yte, score_base)
#                     pr_comb  = average_precision_score(yte, score_comb)
#                 except Exception:
#                     continue

#                 rows.append({
#                     "asset": asset, "horizon": h_label, "h": h,
#                     "target": "stress",
#                     "pos_rate": round(pos_rate, 3),
#                     "metric_base": round(pr_base, 4),
#                     "metric_model": round(pr_comb, 4),
#                     "delta_metric": round(pr_comb - pr_base, 4),
#                     "roc_base": round(roc_base, 4),
#                     "roc_comb": round(roc_comb, 4),
#                     "r2_base": np.nan,
#                     "r2_it":   np.nan,
#                     "res_corr": np.nan,
#                     "mse_ratio": np.nan,
#                     "n_train": int(tr_mask.sum()),
#                     "n_test":  int(te_mask.sum()),
#                 })
#                 continue

#             corr_k = Xv_tr.corrwith(ytr).abs().replace([np.inf,-np.inf],np.nan).dropna()
#             top3   = corr_k.nlargest(3).index.tolist()
#             mdl_b  = _ridge(); mdl_b.fit(Xv_tr[top3], ytr)
#             pred_b_tr = mdl_b.predict(Xv_tr[top3])
#             pred_b_te = mdl_b.predict(Xv_te[top3])

#             res_tr_arr = ytr.values - pred_b_tr
#             mdl_it = _ridge(); mdl_it.fit(Xi_tr, res_tr_arr)
#             pred_it_te = mdl_it.predict(Xi_te)
#             pred_comb_te = pred_b_te + pred_it_te

#             Xc_tr = pd.concat([Xv_tr, Xi_tr], axis=1)
#             Xc_te = pd.concat([Xv_te, Xi_te], axis=1)
#             mdl_c = _ridge(); mdl_c.fit(Xc_tr, ytr)
#             pred_c_te = mdl_c.predict(Xc_te)

#             r2_b  = oos_r2(yte.values, pred_b_te,   naive_val)
#             r2_it = oos_r2(yte.values, pred_comb_te, naive_val)
#             r2_c  = oos_r2(yte.values, pred_c_te,    naive_val)
#             rc    = res_corr(yte.values, pred_b_te, pred_it_te)

#             mse_b = float(np.mean((yte.values - pred_b_te)**2))
#             mse_c = float(np.mean((yte.values - pred_c_te)**2))
#             mse_r = mse_c / mse_b if mse_b > 1e-12 else np.nan

#             rows.append({
#                 "asset": asset, "horizon": h_label, "h": h,
#                 "target": tgt_name,
#                 "pos_rate": np.nan,
#                 "metric_base": round(r2_b,  4),
#                 "metric_model": round(r2_c,  4),
#                 "delta_metric": round(r2_c - r2_b, 4),
#                 "roc_base": np.nan,
#                 "roc_comb": np.nan,
#                 "r2_base": round(r2_b,  4),
#                 "r2_it":   round(r2_it, 4),
#                 "res_corr": round(rc, 4),
#                 "mse_ratio": round(mse_r, 4),
#                 "n_train": int(tr_mask.sum()),
#                 "n_test":  int(te_mask.sum()),
#             })

# df = pd.DataFrame(rows)
# print(f"\nВсего строк: {len(df)}")

# for tgt_name in ["logvol", "delta", "stress"]:
#     sub = df[df.target == tgt_name].copy()
#     if sub.empty: continue

#     if tgt_name == "stress":
#         print(f"\n{'='*72}")
#         print(f"  STRESS CLASSIFICATION  (метрика: PR-AUC)")
#         print(f"  Baseline: vol_now percentile | Model: LogReg(vol+IT)")
#         print(f"{'='*72}")
#         print(f"\n  {'актив':<6} {'h':<4} {'pos%':>5} {'PR_base':>9} "
#               f"{'PR_comb':>9} {'ΔAUC':>7} {'ROC_comb':>10}")
#         print("  " + "-"*58)
#         prev = None
#         for _, row in sub.sort_values(["asset","h"]).iterrows():
#             if prev and prev != row["asset"]: print("  " + "·"*58)
#             prev = row["asset"]
#             mk = "↑" if row["delta_metric"] > 0.01 else "~"
#             print(f"  {row['asset']:<6} {row['horizon']:<4}"
#                   f" {row['pos_rate']:>4.0%}"
#                   f" {row['metric_base']:>9.4f}"
#                   f" {row['metric_model']:>9.4f}"
#                   f" {row['delta_metric']:>+7.4f} {mk}"
#                   f" {row['roc_comb']:>9.4f}")
#         print(f"\n  Среднее PR_base:  {sub['metric_base'].mean():.4f}")
#         print(f"  Среднее PR_comb:  {sub['metric_model'].mean():.4f}")
#         print(f"  Среднее ROC_comb: {sub['roc_comb'].mean():.4f}")
#         print(f"  Δ > 0 (comb > base): {(sub['delta_metric']>0).sum()} / {len(sub)}")

#     else:
#         label = "LOG VOL" if tgt_name == "logvol" else "DELTA VOL"
#         print(f"\n{'='*76}")
#         print(f"  {label}  (OOS R²: >0 = лучше наивного прогноза)")
#         print(f"  R²_base = Ridge(vol_k3) vs naive | R²_comb = Ridge(vol+IT) vs naive")
#         print(f"  res_corr = корреляция IT-предсказания с остатками baseline")
#         print(f"{'='*76}")
#         print(f"\n  {'актив':<6} {'h':<4} {'R²_base':>9} {'R²_comb':>9} "
#               f"{'ΔR²':>7} {'res_corr':>10} {'MSE_r':>7}")
#         print("  " + "-"*60)
#         prev = None
#         for _, row in sub.sort_values(["asset","h"]).iterrows():
#             if prev and prev != row["asset"]: print("  " + "·"*60)
#             prev = row["asset"]
#             dr2  = row["delta_metric"]
#             rc   = row["res_corr"]
#             ms   = row["mse_ratio"]
#             dr2m = "↑" if dr2 > 0.01 else ("↓" if dr2 < -0.01 else "~")
#             rcm  = "✓" if rc > 0.03 else "·"
#             msm  = "✓" if ms < 1.0  else "·"
#             print(f"  {row['asset']:<6} {row['horizon']:<4}"
#                   f" {row['r2_base']:>+9.4f}"
#                   f" {row['metric_model']:>+9.4f}"
#                   f" {dr2:>+7.4f} {dr2m}"
#                   f" {rc:>+9.4f} {rcm}"
#                   f" {ms:>6.4f} {msm}")

#         beats_base = (sub["delta_metric"] > 0).sum()
#         rc_pos     = (sub["res_corr"] > 0.03).sum()
#         mse_ok     = (sub["mse_ratio"] < 1.0).sum()
#         print(f"\n  R²_comb > R²_base:  {beats_base} / {len(sub)}")
#         print(f"  res_corr > 0.03:    {rc_pos} / {len(sub)}")
#         print(f"  MSE_ratio < 1.0:    {mse_ok} / {len(sub)}")
#         print(f"  Среднее ΔR²:       {sub['delta_metric'].mean():+.4f}")
#         print(f"  Среднее res_corr:  {sub['res_corr'].mean():+.4f}")

# print(f"\n{'='*76}")
# print(f"  ИТОГ ПО ВСЕМ ТАРГЕТАМ И АКТИВАМ")
# print(f"{'='*76}")

# reg = df[df.target.isin(["logvol","delta"])]
# print(f"\n  Регрессия (logvol + delta):")
# print(f"    R²_comb > R²_base:  {(reg['delta_metric']>0).sum()} / {len(reg)}")
# print(f"    res_corr > 0.03:    {(reg['res_corr']>0.03).sum()} / {len(reg)}")
# print(f"    MSE_ratio < 1.0:    {(reg['mse_ratio']<1.0).sum()} / {len(reg)}")
# print(f"    Среднее ΔR²:       {reg['delta_metric'].mean():+.4f}")
# print(f"    Среднее res_corr:  {reg['res_corr'].mean():+.4f}")

# st = df[df.target == "stress"]
# if not st.empty:
#     print(f"\n  Классификация (stress):")
#     print(f"    PR_comb > PR_base:   {(st['delta_metric']>0).sum()} / {len(st)}")
#     print(f"    Среднее ROC-AUC:    {st['roc_comb'].mean():.4f}")
#     print(f"    Среднее ΔPR-AUC:   {st['delta_metric'].mean():+.4f}")

# holdout_full = df.copy()
# print(f"\nholdout_full сохранён в памяти ({len(df)} строк)")

Сплит: train → 2025-01-01 | test 2025-01-01 →
Активы: ['BTC', 'ETH', 'SOL', 'BNB', 'XRP', 'DOGE', 'ADA', 'AVAX', 'TRX']
Горизонты: ['1d', '2d', '3d', '4d', '5d', '6d']
IT-фичи: 10

Строим таргеты для всех активов и горизонтов...
  Готово: 9 активов

Всего строк: 162

  LOG VOL  (OOS R²: >0 = лучше наивного прогноза)
  R²_base = Ridge(vol_k3) vs naive | R²_comb = Ridge(vol+IT) vs naive
  res_corr = корреляция IT-предсказания с остатками baseline

  актив  h      R²_base   R²_comb     ΔR²   res_corr   MSE_r
  ------------------------------------------------------------
  ADA    1d     +0.3362   +0.4171 +0.0809 ↑   +0.1666 ✓ 0.8781 ✓
  ADA    2d     +0.3047   +0.3353 +0.0306 ↑   +0.1760 ✓ 0.9560 ✓
  ADA    3d     +0.2816   +0.2760 -0.0056 ~   +0.1660 ✓ 1.0078 ·
  ADA    4d     +0.2688   +0.2341 -0.0347 ↓   +0.1476 ✓ 1.0474 ·
  ADA    5d     +0.2604   +0.2059 -0.0544 ↓   +0.1329 ✓ 1.0736 ·
  ADA    6d     +0.2665   +0.2016 -0.0649 ↓   +0.1396 ✓ 1.0885 ·
  ··································

In [54]:
# import warnings; warnings.filterwarnings("ignore")
# import numpy as np
# import pandas as pd
# from sklearn.linear_model import Ridge, RidgeCV, LogisticRegression
# from sklearn.preprocessing import StandardScaler
# from sklearn.pipeline import Pipeline
# from sklearn.metrics import roc_auc_score, average_precision_score

# SPLIT_DATE = "2025-01-01"
# PURGE_BARS = 96

# HORIZONS   = {48:"1d", 96:"2d", 144:"3d", 192:"4d", 240:"5d", 288:"6d"}
# ALL_ASSETS = ["BTC","ETH","SOL","BNB","XRP","DOGE","ADA","AVAX","TRX"]
# VOL_WINDOW = 48
# ANN        = np.sqrt(365 * 48)

# SELECTED_COLS = [
#     "coinfo3_std", "kl_BNB", "coinfo4_std", "pairwise_mi_mean",
#     "coinfo4_mean", "pairwise_mi_std", "coinfo3_mean",
#     "coinfo4_max", "kl_AVAX", "pairwise_mi_min",
# ]

# print(f"Сплит: train → {SPLIT_DATE} | test {SPLIT_DATE} →")
# print(f"IT-фичи: {len(SELECTED_COLS)}")
# print(f"Масштабирование: optimal shrinkage α на train-остатках")

# def _ridge():
#     return Pipeline([("sc", StandardScaler()),
#                      ("m", RidgeCV(alphas=[0.01, 0.1, 1, 10, 100]))])

# def _logit():
#     return Pipeline([("sc", StandardScaler()),
#                      ("m", LogisticRegression(C=0.1, max_iter=1000,
#                                               class_weight="balanced"))])

# def oos_r2(y_true, y_pred, naive_val):
#     """R² vs наивный прогноз (среднее train). >0 = лучше наивного."""
#     mse_m = np.mean((y_true - y_pred) ** 2)
#     mse_n = np.mean((y_true - naive_val) ** 2)
#     return float(1.0 - mse_m / mse_n) if mse_n > 1e-12 else np.nan

# def opt_alpha(res_train, pred_it_train):
#     """
#     Аналитический optimal shrinkage:
#     alpha* = cov(res, pred_it) / var(pred_it)
#     Ограничен [-2, 2] для стабильности.
#     """
#     cmat = np.cov(res_train, pred_it_train)
#     var_it = cmat[1, 1]
#     if var_it < 1e-12:
#         return 0.0
#     return float(np.clip(cmat[0, 1] / var_it, -2.0, 2.0))

# print("\nСтроим таргеты...")
# tgts = {}
# for asset in ALL_ASSETS:
#     if asset not in ret.columns:
#         continue
#     r       = ret[asset]
#     vol_now = r.rolling(VOL_WINDOW).std() * ANN
#     vol_med = vol_now.expanding(min_periods=VOL_WINDOW * 30).median()
#     tgts[asset] = {}
#     for h in HORIZONS:
#         vol_fwd = r.rolling(h).std().shift(-h) * ANN
#         tgts[asset][h] = {
#             "logvol":  np.log(vol_fwd.clip(lower=1e-8)),
#             "delta":   vol_fwd - vol_now,
#             "stress":  (vol_fwd > vol_med * 1.25).astype(float),
#             "vol_now": vol_now,
#         }
# print(f"  Готово: {len(tgts)} активов")

# rows      = []
# Xi_all   = X_it_slim[SELECTED_COLS]
# split_ts = pd.Timestamp(SPLIT_DATE)

# for asset in ALL_ASSETS:
#     if asset not in tgts:
#         continue
#     for h, h_label in HORIZONS.items():
#         for tgt_name in ["logvol", "delta", "stress"]:
#             y_raw     = tgts[asset][h].get(tgt_name, pd.Series(dtype=float)).dropna()
#             vol_now_s = tgts[asset][h]["vol_now"]

#             idx = (X_vol.index
#                    .intersection(Xi_all.index)
#                    .intersection(y_raw.index))
#             valid = (X_vol.loc[idx].notna().all(1) &
#                      Xi_all.loc[idx].notna().all(1) &
#                      y_raw.loc[idx].notna())
#             idx = idx[valid]
#             if len(idx) < 5000:
#                 continue

#             tr = idx < (split_ts - pd.Timedelta(minutes=30 * PURGE_BARS))
#             te = idx >= split_ts
#             if tr.sum() < 2880 or te.sum() < 480:
#                 continue

#             Xv_tr = X_vol.loc[idx[tr]];  Xv_te = X_vol.loc[idx[te]]
#             Xi_tr = Xi_all.loc[idx[tr]]; Xi_te = Xi_all.loc[idx[te]]
#             ytr   = y_raw.loc[idx[tr]];  yte   = y_raw.loc[idx[te]]
#             naive = float(ytr.mean())

#             if tgt_name == "stress":
#                 pos = float(ytr.mean())
#                 if pos < 0.05 or pos > 0.95:
#                     continue
#                 vn_max = float(vol_now_s.loc[idx[tr]].max())
#                 sc_base = np.clip(vol_now_s.loc[idx[te]].values / (vn_max + 1e-8), 0, 1)
#                 try:
#                     mdl_it = _logit(); mdl_it.fit(Xi_tr, ytr)
#                     sc_it  = mdl_it.predict_proba(Xi_te)[:, 1]
#                     Xc_tr  = pd.concat([Xv_tr, Xi_tr], axis=1)
#                     Xc_te  = pd.concat([Xv_te, Xi_te], axis=1)
#                     mdl_c  = _logit(); mdl_c.fit(Xc_tr, ytr)
#                     sc_c   = mdl_c.predict_proba(Xc_te)[:, 1]
#                     rows.append({
#                         "asset": asset, "horizon": h_label, "h": h,
#                         "target": "stress", "pos_rate": round(pos, 3),
#                         "r2_base": np.nan, "r2_comb": np.nan,
#                         "delta_r2": np.nan, "res_corr": np.nan,
#                         "mse_ratio": np.nan, "alpha_it": np.nan,
#                         "pr_base":  round(average_precision_score(yte, sc_base), 4),
#                         "pr_comb":  round(average_precision_score(yte, sc_c),    4),
#                         "delta_pr": round(average_precision_score(yte, sc_c) -
#                                           average_precision_score(yte, sc_base), 4),
#                         "roc_comb": round(roc_auc_score(yte, sc_c), 4),
#                         "n_train": int(tr.sum()), "n_test": int(te.sum()),
#                     })
#                 except Exception:
#                     pass
#                 continue

#             ck   = Xv_tr.corrwith(ytr).abs().replace([np.inf,-np.inf],np.nan).dropna()
#             top3 = ck.nlargest(3).index.tolist()
#             mb   = _ridge(); mb.fit(Xv_tr[top3], ytr)
#             pb_tr = mb.predict(Xv_tr[top3])
#             pb_te = mb.predict(Xv_te[top3])

#             res_tr = ytr.values - pb_tr
#             mit = _ridge(); mit.fit(Xi_tr, res_tr)
#             pit_tr = mit.predict(Xi_tr)
#             pit_te = mit.predict(Xi_te)

#             a = opt_alpha(res_tr, pit_tr)

#             pred_sc = pb_te + a * pit_te   # scaled

#             r2_b  = oos_r2(yte.values, pb_te,   naive)
#             r2_sc = oos_r2(yte.values, pred_sc,  naive)
#             rc    = float(pd.Series(pit_te).corr(pd.Series(yte.values - pb_te)))
#             mse_b = float(np.mean((yte.values - pb_te)**2))
#             mse_s = float(np.mean((yte.values - pred_sc)**2))

#             rows.append({
#                 "asset": asset, "horizon": h_label, "h": h,
#                 "target": tgt_name, "pos_rate": np.nan,
#                 "r2_base":  round(r2_b,  4),
#                 "r2_comb":  round(r2_sc, 4),
#                 "delta_r2": round(r2_sc - r2_b, 4),
#                 "res_corr": round(rc, 4),
#                 "mse_ratio": round(mse_s / mse_b, 4) if mse_b > 1e-12 else np.nan,
#                 "alpha_it": round(a, 3),
#                 "pr_base": np.nan, "pr_comb": np.nan,
#                 "delta_pr": np.nan, "roc_comb": np.nan,
#                 "n_train": int(tr.sum()), "n_test": int(te.sum()),
#             })

# df = pd.DataFrame(rows)
# print(f"\nВсего строк: {len(df)}")

# for tgt_name, label in [("logvol","LOG VOL"), ("delta","DELTA VOL")]:
#     sub = df[df.target == tgt_name].copy()
#     if sub.empty: continue
#     print(f"\n{'='*76}")
#     print(f"  {label}  —  pred = baseline + α·IT_residual")
#     print(f"  ΔR² > 0: IT улучшает R²  |  α показывает насколько сильно масштабируется поправка")
#     print(f"{'='*76}")
#     print(f"\n  {'актив':<6} {'h':<4} {'R²_base':>9} {'R²_comb':>9} "
#           f"{'ΔR²':>7} {'res_corr':>10} {'α':>6} {'MSE_r':>7}")
#     print("  " + "-"*65)
#     prev = None
#     for _, r in sub.sort_values(["asset","h"]).iterrows():
#         if prev and prev != r["asset"]: print("  " + "·"*65)
#         prev = r["asset"]
#         dm = "↑" if r["delta_r2"] > 0.005 else ("↓" if r["delta_r2"] < -0.005 else "~")
#         rm = "✓" if r["res_corr"] > 0.03  else "·"
#         mm = "✓" if r["mse_ratio"] < 1.0  else "·"
#         print(f"  {r['asset']:<6} {r['horizon']:<4}"
#               f" {r['r2_base']:>+9.4f}"
#               f" {r['r2_comb']:>+9.4f}"
#               f" {r['delta_r2']:>+7.4f} {dm}"
#               f" {r['res_corr']:>+9.4f} {rm}"
#               f" {r['alpha_it']:>5.2f}"
#               f" {r['mse_ratio']:>6.4f} {mm}")

#     print(f"\n  ΔR² > 0:          {(sub['delta_r2']>0).sum()} / {len(sub)}")
#     print(f"  res_corr > 0.03:  {(sub['res_corr']>0.03).sum()} / {len(sub)}")
#     print(f"  MSE < 1.0:        {(sub['mse_ratio']<1.0).sum()} / {len(sub)}")
#     print(f"  Среднее ΔR²:     {sub['delta_r2'].mean():+.4f}")
#     print(f"  Среднее res_corr:{sub['res_corr'].mean():+.4f}")
#     print(f"  Среднее α:       {sub['alpha_it'].mean():+.3f}")

# st = df[df.target == "stress"]
# if not st.empty:
#     print(f"\n{'='*70}")
#     print(f"  STRESS  (PR-AUC; baseline = vol_now percentile)")
#     print(f"{'='*70}")
#     print(f"\n  {'актив':<6} {'h':<4} {'pos%':>5} "
#           f"{'PR_base':>9} {'PR_comb':>9} {'ΔPR':>7} {'ROC_comb':>10}")
#     print("  " + "-"*56)
#     prev = None
#     for _, r in st.sort_values(["asset","h"]).iterrows():
#         if prev and prev != r["asset"]: print("  " + "·"*56)
#         prev = r["asset"]
#         mk = "↑" if r["delta_pr"] > 0.01 else "~"
#         print(f"  {r['asset']:<6} {r['horizon']:<4}"
#               f" {r['pos_rate']:>4.0%}"
#               f" {r['pr_base']:>9.4f}"
#               f" {r['pr_comb']:>9.4f}"
#               f" {r['delta_pr']:>+7.4f} {mk}"
#               f" {r['roc_comb']:>9.4f}")
#     print(f"\n  ΔPR > 0: {(st['delta_pr']>0).sum()} / {len(st)}")
#     print(f"  Среднее ROC_comb:  {st['roc_comb'].mean():.4f}")
#     print(f"  Среднее ΔPR-AUC:  {st['delta_pr'].mean():+.4f}")

# reg = df[df.target.isin(["logvol","delta"])]
# print(f"\n{'='*70}")
# print(f"  ИТОГ: среднее ΔR² по активам (logvol + delta)")
# print(f"{'='*70}")
# by_asset = reg.groupby("asset")["delta_r2"].mean().sort_values(ascending=False)
# for asset, val in by_asset.items():
#     n_pos = (reg[reg.asset==asset]["delta_r2"] > 0).sum()
#     n_tot = (reg[reg.asset==asset]).shape[0]
#     bar = "█" * max(0, int((val + 0.3) * 15))
#     print(f"  {asset:<6} {val:>+7.4f}  {bar}  ({n_pos}/{n_tot} горизонтов ↑)")

# holdout_full = df.copy()
# print(f"\nholdout_full сохранён в памяти ({len(df)} строк)")

Сплит: train → 2025-01-01 | test 2025-01-01 →
IT-фичи: 10
Масштабирование: optimal shrinkage α на train-остатках

Строим таргеты...
  Готово: 9 активов

Всего строк: 162

  LOG VOL  —  pred = baseline + α·IT_residual
  ΔR² > 0: IT улучшает R²  |  α показывает насколько сильно масштабируется поправка

  актив  h      R²_base   R²_comb     ΔR²   res_corr      α   MSE_r
  -----------------------------------------------------------------
  ADA    1d     +0.3362   +0.3412 +0.0050 ~   +0.1660 ✓  1.00 0.9924 ✓
  ADA    2d     +0.3047   +0.2995 -0.0053 ↓   +0.1753 ✓  1.00 1.0076 ·
  ADA    3d     +0.2816   +0.2641 -0.0175 ↓   +0.1649 ✓  1.00 1.0244 ·
  ADA    4d     +0.2688   +0.2337 -0.0351 ↓   +0.1468 ✓  1.00 1.0480 ·
  ADA    5d     +0.2604   +0.2116 -0.0488 ↓   +0.1322 ✓  1.00 1.0660 ·
  ADA    6d     +0.2666   +0.2108 -0.0558 ↓   +0.1389 ✓  1.00 1.0761 ·
  ·································································
  AVAX   1d     +0.1835   +0.2118 +0.0283 ↑   +0.1651 ✓  1.00 0.9653

In [56]:
# import warnings; warnings.filterwarnings("ignore")
# import numpy as np
# import pandas as pd
# from sklearn.linear_model import LogisticRegression
# from sklearn.preprocessing import StandardScaler
# from sklearn.pipeline import Pipeline
# from sklearn.metrics import (roc_auc_score, average_precision_score,
#                               precision_recall_curve, f1_score)
# import matplotlib
# matplotlib.use("Agg")
# import matplotlib.pyplot as plt

# SPLIT_DATE = "2025-01-01"
# PURGE_BARS = 96

# THRESHOLDS = [1.10, 1.25, 1.50, 2.00]   # множители медианы
# HORIZONS   = {48:"1d", 96:"2d", 192:"4d", 288:"6d"}   # 4 горизонта
# ASSETS     = ["BTC", "ETH", "SOL", "BNB", "XRP", "DOGE", "ADA", "AVAX", "TRX"]
# VOL_WINDOW = 48
# ANN        = np.sqrt(365 * 48)

# SELECTED_COLS = [
#     "coinfo3_std", "kl_BNB", "coinfo4_std", "pairwise_mi_mean",
#     "coinfo4_mean", "pairwise_mi_std", "coinfo3_mean",
#     "coinfo4_max", "kl_AVAX", "pairwise_mi_min",
# ]

# print(f"Сплит: train → {SPLIT_DATE} | test {SPLIT_DATE} →")
# print(f"Пороги: {THRESHOLDS}")
# print(f"Горизонты: {list(HORIZONS.values())}")
# print(f"Активы: {ASSETS}")

# def _logit(C=0.1):
#     return Pipeline([("sc", StandardScaler()),
#                      ("m", LogisticRegression(C=C, max_iter=2000,
#                                               class_weight="balanced",
#                                               solver="lbfgs"))])

# def lift_at_k(y_true, score, k=0.10):
#     """Lift в топ-k% предсказаний: (precision @ top-k) / (base rate)."""
#     n = len(y_true)
#     top_n = max(1, int(n * k))
#     top_idx = np.argsort(score)[::-1][:top_n]
#     prec_k = float(np.mean(y_true[top_idx]))
#     base   = float(np.mean(y_true))
#     return prec_k / base if base > 1e-8 else np.nan

# def best_f1(y_true, score):
#     """Лучший F1 по всем порогам кривой PR."""
#     prec, rec, _ = precision_recall_curve(y_true, score)
#     f1s = 2 * prec * rec / (prec + rec + 1e-10)
#     return float(np.max(f1s))

# print("\nСтроим vol и таргеты...")
# vol_data = {}  # vol_data[asset] = {"vol_now":..., "vol_med":...}
# for asset in ASSETS:
#     if asset not in ret.columns: continue
#     r       = ret[asset]
#     vol_now = r.rolling(VOL_WINDOW).std() * ANN
#     vol_med = vol_now.expanding(min_periods=VOL_WINDOW * 30).median()
#     vol_data[asset] = {"vol_now": vol_now, "vol_med": vol_med, "ret": r}
# print(f"  Готово: {len(vol_data)} активов")

# rows = []
# Xi_all   = X_it_slim[SELECTED_COLS]
# split_ts = pd.Timestamp(SPLIT_DATE)

# for asset in ASSETS:
#     if asset not in vol_data: continue
#     vd = vol_data[asset]

#     for h, h_label in HORIZONS.items():
#         vol_fwd = vd["ret"].rolling(h).std().shift(-h) * ANN

#         for thr in THRESHOLDS:
#             y_raw = (vol_fwd > vd["vol_med"] * thr).astype(float).dropna()

#             idx = (X_vol.index
#                    .intersection(Xi_all.index)
#                    .intersection(y_raw.index))
#             valid = (X_vol.loc[idx].notna().all(1) &
#                      Xi_all.loc[idx].notna().all(1) &
#                      y_raw.loc[idx].notna())
#             idx = idx[valid]
#             if len(idx) < 5000: continue

#             tr = idx < (split_ts - pd.Timedelta(minutes=30 * PURGE_BARS))
#             te = idx >= split_ts
#             if tr.sum() < 2880 or te.sum() < 480: continue

#             Xv_tr = X_vol.loc[idx[tr]];  Xv_te = X_vol.loc[idx[te]]
#             Xi_tr = Xi_all.loc[idx[tr]]; Xi_te = Xi_all.loc[idx[te]]
#             ytr   = y_raw.loc[idx[tr]];  yte   = y_raw.loc[idx[te]]

#             pos_rate_tr = float(ytr.mean())
#             pos_rate_te = float(yte.mean())

#             if pos_rate_tr < 0.02 or pos_rate_tr > 0.98: continue
#             if yte.sum() < 5: continue

#             vn_tr = vd["vol_now"].loc[idx[tr]]
#             vn_te = vd["vol_now"].loc[idx[te]]
#             vn_max = float(vn_tr.max())
#             sc_b1 = np.clip(vn_te.values / (vn_max + 1e-8), 0, 1)

#             sc_b2 = np.full(len(yte), pos_rate_tr)

#             try:
#                 m1 = _logit(); m1.fit(Xi_tr, ytr)
#                 sc_m1 = m1.predict_proba(Xi_te)[:, 1]
#             except Exception:
#                 sc_m1 = sc_b2.copy()

#             try:
#                 Xc_tr = pd.concat([Xv_tr, Xi_tr], axis=1)
#                 Xc_te = pd.concat([Xv_te, Xi_te], axis=1)
#                 m2 = _logit(); m2.fit(Xc_tr, ytr)
#                 sc_m2 = m2.predict_proba(Xc_te)[:, 1]
#             except Exception:
#                 sc_m2 = sc_b1.copy()

#             yte_np = yte.values.astype(int)

#             def safe_roc(y, s):
#                 try: return float(roc_auc_score(y, s))
#                 except: return np.nan

#             def safe_pr(y, s):
#                 try: return float(average_precision_score(y, s))
#                 except: return np.nan

#             rows.append({
#                 "asset":       asset,
#                 "horizon":     h_label,
#                 "h":           h,
#                 "threshold":   thr,
#                 "pos_tr":      round(pos_rate_tr, 3),
#                 "pos_te":      round(pos_rate_te, 3),
#                 "n_train":     int(tr.sum()),
#                 "n_test":      int(te.sum()),
#                 "pr_b1":       round(safe_pr(yte_np, sc_b1), 4),
#                 "pr_m1":       round(safe_pr(yte_np, sc_m1), 4),
#                 "pr_m2":       round(safe_pr(yte_np, sc_m2), 4),
#                 "roc_b1":      round(safe_roc(yte_np, sc_b1), 4),
#                 "roc_m1":      round(safe_roc(yte_np, sc_m1), 4),
#                 "roc_m2":      round(safe_roc(yte_np, sc_m2), 4),
#                 "lift_b1":     round(lift_at_k(yte_np, sc_b1), 3),
#                 "lift_m2":     round(lift_at_k(yte_np, sc_m2), 3),
#                 "f1_b1":       round(best_f1(yte_np, sc_b1), 4),
#                 "f1_m2":       round(best_f1(yte_np, sc_m2), 4),
#                 "delta_pr":    round(safe_pr(yte_np, sc_m2) - safe_pr(yte_np, sc_b1), 4),
#                 "delta_roc":   round(safe_roc(yte_np, sc_m2) - safe_roc(yte_np, sc_b1), 4),
#                 "delta_lift":  round(lift_at_k(yte_np, sc_m2) - lift_at_k(yte_np, sc_b1), 3),
#             })

# df = pd.DataFrame(rows)
# print(f"\nВсего строк: {len(df)}")

# print(f"\n{'='*72}")
# print(f"  АГРЕГАТ ПО ПОРОГАМ — среднее по всем активам и горизонтам")
# print(f"{'='*72}")
# print(f"\n  {'порог':>7} {'pos%':>6} {'PR_b1':>8} {'PR_m2':>8} "
#       f"{'ΔPR':>7} {'ROC_b1':>8} {'ROC_m2':>8} "
#       f"{'ΔROC':>7} {'Lift_m2':>8} {'ΔLift':>7}")
# print("  " + "-"*80)

# for thr in THRESHOLDS:
#     sub = df[df.threshold == thr]
#     if sub.empty: continue
#     beats_pr  = (sub["delta_pr"]  > 0).sum()
#     beats_roc = (sub["delta_roc"] > 0).sum()
#     print(f"  ×{thr:<6.2f}"
#           f" {sub['pos_te'].mean():>5.0%}"
#           f" {sub['pr_b1'].mean():>8.4f}"
#           f" {sub['pr_m2'].mean():>8.4f}"
#           f" {sub['delta_pr'].mean():>+7.4f}"
#           f" {sub['roc_b1'].mean():>8.4f}"
#           f" {sub['roc_m2'].mean():>8.4f}"
#           f" {sub['delta_roc'].mean():>+7.4f}"
#           f" {sub['lift_m2'].mean():>8.3f}"
#           f" {sub['delta_lift'].mean():>+7.3f}"
#           f"   ({beats_pr}/{len(sub)} PR↑, {beats_roc}/{len(sub)} ROC↑)")

# print(f"\n{'='*72}")
# print(f"  АГРЕГАТ ПО ГОРИЗОНТАМ — среднее по активам")
# print(f"{'='*72}")

# for thr in THRESHOLDS:
#     sub = df[df.threshold == thr]
#     if sub.empty: continue
#     print(f"\n  Порог ×{thr:.2f}  (pos% ≈ {sub['pos_te'].mean():.0%}):")
#     print(f"  {'h':<5} {'PR_b1':>8} {'PR_m2':>8} {'ΔPR':>7} "
#           f"{'ROC_b1':>8} {'ROC_m2':>8} {'ΔROC':>7} {'Lift_m2':>8}")
#     print("  " + "-"*65)
#     for h_label in ["1d","2d","4d","6d"]:
#         g = sub[sub.horizon == h_label]
#         if g.empty: continue
#         dpr = g["delta_pr"].mean()
#         mk  = "↑" if dpr > 0.01 else ("~" if abs(dpr) <= 0.01 else "↓")
#         print(f"  {h_label:<5}"
#               f" {g['pr_b1'].mean():>8.4f}"
#               f" {g['pr_m2'].mean():>8.4f}"
#               f" {dpr:>+7.4f} {mk}"
#               f" {g['roc_b1'].mean():>8.4f}"
#               f" {g['roc_m2'].mean():>8.4f}"
#               f" {g['delta_roc'].mean():>+7.4f}"
#               f" {g['lift_m2'].mean():>8.3f}")

# print(f"\n{'='*72}")
# print(f"  ТОП-10 результатов по ΔPR-AUC")
# print(f"{'='*72}")
# top10 = df.nlargest(10, "delta_pr")
# print(f"\n  {'актив':<6} {'h':<4} {'×thr':>5} {'pos%':>5} "
#       f"{'PR_b1':>8} {'PR_m2':>8} {'ΔPR':>8} {'ROC_m2':>9} {'Lift_m2':>9}")
# print("  " + "-"*65)
# for _, row in top10.iterrows():
#     print(f"  {row['asset']:<6} {row['horizon']:<4} "
#           f"×{row['threshold']:.2f}"
#           f" {row['pos_te']:>4.0%}"
#           f" {row['pr_b1']:>8.4f}"
#           f" {row['pr_m2']:>8.4f}"
#           f" {row['delta_pr']:>+8.4f}"
#           f" {row['roc_m2']:>9.4f}"
#           f" {row['lift_m2']:>9.3f}")

# print(f"\n{'='*72}")
# print(f"  ПО АКТИВАМ — порог ×1.25 (стандартный)")
# print(f"{'='*72}")
# sub125 = df[df.threshold == 1.25]
# print(f"\n  {'актив':<6} {'PR_b1':>8} {'PR_m2':>8} {'ΔPR':>8} "
#       f"{'ROC_b1':>8} {'ROC_m2':>8} {'Lift_m2':>9} {'h_best':>7}")
# print("  " + "-"*68)
# for asset in ASSETS:
#     g = sub125[sub125.asset == asset]
#     if g.empty: continue
#     best_h = g.loc[g["delta_pr"].idxmax(), "horizon"]
#     print(f"  {asset:<6}"
#           f" {g['pr_b1'].mean():>8.4f}"
#           f" {g['pr_m2'].mean():>8.4f}"
#           f" {g['delta_pr'].mean():>+8.4f}"
#           f" {g['roc_b1'].mean():>8.4f}"
#           f" {g['roc_m2'].mean():>8.4f}"
#           f" {g['lift_m2'].mean():>9.3f}"
#           f"  best_h={best_h}")

# print(f"\nСтроим графики...")

# fig, axes = plt.subplots(2, 2, figsize=(13, 9))
# colors = {1.10:"#2196F3", 1.25:"#4CAF50", 1.50:"#FF9800", 2.00:"#F44336"}

# ax = axes[0, 0]
# x  = np.arange(len(HORIZONS))
# w  = 0.18
# for i, thr in enumerate(THRESHOLDS):
#     sub = df[df.threshold == thr].groupby("horizon")["delta_pr"].mean()
#     vals = [sub.get(h, np.nan) for h in HORIZONS.values()]
#     ax.bar(x + (i-1.5)*w, vals, w*0.9, label=f"×{thr:.2f}",
#            color=colors[thr], alpha=0.8)
# ax.axhline(0, color="black", linewidth=0.8)
# ax.set_xticks(x); ax.set_xticklabels(list(HORIZONS.values()))
# ax.set_title("ΔPR-AUC (vol+IT minus vol_now baseline)", fontsize=10)
# ax.set_xlabel("Horizon"); ax.set_ylabel("ΔPR-AUC")
# ax.legend(fontsize=8); ax.grid(alpha=0.2, axis="y")
# ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

# ax = axes[0, 1]
# for thr in THRESHOLDS:
#     sub = df[df.threshold == thr].groupby("horizon")["roc_m2"].mean()
#     vals = [sub.get(h, np.nan) for h in HORIZONS.values()]
#     ax.plot(list(HORIZONS.values()), vals, "o-", label=f"×{thr:.2f}",
#             color=colors[thr], linewidth=1.8, markersize=6)
# ax.axhline(0.5, color="gray", linewidth=0.8, linestyle=":", alpha=0.6)
# ax.set_title("ROC-AUC (vol+IT) по горизонтам", fontsize=10)
# ax.set_xlabel("Horizon"); ax.set_ylabel("ROC-AUC")
# ax.legend(fontsize=8); ax.grid(alpha=0.2)
# ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

# ax = axes[1, 0]
# for thr in THRESHOLDS:
#     sub = df[df.threshold == thr].groupby("horizon")["lift_m2"].mean()
#     vals = [sub.get(h, np.nan) for h in HORIZONS.values()]
#     ax.plot(list(HORIZONS.values()), vals, "s-", label=f"×{thr:.2f}",
#             color=colors[thr], linewidth=1.8, markersize=6)
# ax.axhline(1.0, color="gray", linewidth=0.8, linestyle=":", alpha=0.6)
# ax.set_title("Lift @ top-10% (vol+IT)", fontsize=10)
# ax.set_xlabel("Horizon"); ax.set_ylabel("Lift")
# ax.legend(fontsize=8); ax.grid(alpha=0.2)
# ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

# ax = axes[1, 1]
# asset_delta = (df[df.threshold==1.25]
#                .groupby("asset")["delta_pr"].mean()
#                .sort_values(ascending=True))
# colors_bar = ["#E74C3C" if v < 0 else "#27AE60" for v in asset_delta.values]
# ax.barh(asset_delta.index, asset_delta.values, color=colors_bar, alpha=0.8)
# ax.axvline(0, color="black", linewidth=0.8)
# ax.set_title("ΔPR-AUC по активам (порог ×1.25)", fontsize=10)
# ax.set_xlabel("ΔPR-AUC (vol+IT minus baseline)")
# ax.grid(alpha=0.2, axis="x")
# ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

# plt.suptitle("Stress prediction: vol+IT vs vol_now baseline — разные пороги",
#              fontsize=12, y=1.01)
# plt.tight_layout()
# plt.savefig("fig_stress_thresholds.png", bbox_inches="tight", dpi=150)
# plt.savefig("fig_stress_thresholds.pdf", bbox_inches="tight", dpi=150)
# plt.show()
# print("  Saved: fig_stress_thresholds.png / .pdf")

# print(f"\n{'='*72}")
# print(f"  ИТОГ")
# print(f"{'='*72}")
# for thr in THRESHOLDS:
#     sub = df[df.threshold == thr]
#     beats = (sub["delta_pr"] > 0).sum()
#     print(f"  ×{thr:.2f}: ΔPR={sub['delta_pr'].mean():>+6.4f}  "
#           f"ROC_m2={sub['roc_m2'].mean():.4f}  "
#           f"Lift_m2={sub['lift_m2'].mean():.3f}  "
#           f"beats_baseline: {beats}/{len(sub)}")

# stress_results = df.copy()
# print(f"\nstress_results сохранён в памяти ({len(df)} строк)")

Сплит: train → 2025-01-01 | test 2025-01-01 →
Пороги: [1.1, 1.25, 1.5, 2.0]
Горизонты: ['1d', '2d', '4d', '6d']
Активы: ['BTC', 'ETH', 'SOL', 'BNB', 'XRP', 'DOGE', 'ADA', 'AVAX', 'TRX']

Строим vol и таргеты...
  Готово: 9 активов

Всего строк: 144

  АГРЕГАТ ПО ПОРОГАМ — среднее по всем активам и горизонтам

    порог   pos%    PR_b1    PR_m2     ΔPR   ROC_b1   ROC_m2    ΔROC  Lift_m2   ΔLift
  --------------------------------------------------------------------------------
  ×1.10     56%   0.7723   0.7432 -0.0291   0.7262   0.6852 -0.0410    1.649  -0.076   (13/36 PR↑, 10/36 ROC↑)
  ×1.25     43%   0.6738   0.6283 -0.0455   0.7280   0.6681 -0.0599    1.928  -0.146   (8/36 PR↑, 4/36 ROC↑)
  ×1.50     26%   0.5199   0.4409 -0.0790   0.7325   0.6406 -0.0919    2.148  -0.482   (8/36 PR↑, 4/36 ROC↑)
  ×2.00      9%   0.2730   0.1915 -0.0815   0.7148   0.5638 -0.1510    1.958  -1.061   (4/36 PR↑, 3/36 ROC↑)

  АГРЕГАТ ПО ГОРИЗОНТАМ — среднее по активам

  Порог ×1.10  (pos% ≈ 56%):
  h   

In [59]:
# import warnings; warnings.filterwarnings("ignore")
# import numpy as np
# import pandas as pd
# from sklearn.linear_model import Ridge, RidgeCV, LogisticRegression
# from sklearn.preprocessing import StandardScaler
# from sklearn.pipeline import Pipeline
# from sklearn.metrics import roc_auc_score, average_precision_score

# SPLIT_DATE = "2025-01-01"
# PURGE_BARS = 96

# HORIZONS   = {48:"1d", 96:"2d", 144:"3d", 192:"4d", 240:"5d", 288:"6d"}
# ALL_ASSETS = ["BTC","ETH","SOL","BNB","XRP","DOGE","ADA","AVAX","TRX"]
# VOL_WINDOW = 48
# ANN        = np.sqrt(365 * 48)

# SELECTED_COLS = [
#     "coinfo3_std", "kl_BNB", "coinfo4_std", "pairwise_mi_mean",
#     "coinfo4_mean", "pairwise_mi_std", "coinfo3_mean",
#     "coinfo4_max", "kl_AVAX", "pairwise_mi_min",
# ]

# print(f"Сплит: train → {SPLIT_DATE} | test {SPLIT_DATE} →")
# print(f"IT-фичи: {len(SELECTED_COLS)}")
# print(f"Масштабирование: optimal shrinkage α на train-остатках")

# def _ridge():
#     return Pipeline([("sc", StandardScaler()),
#                      ("m", RidgeCV(alphas=[0.01, 0.1, 1, 10, 100]))])

# def _logit():
#     return Pipeline([("sc", StandardScaler()),
#                      ("m", LogisticRegression(C=0.1, max_iter=1000,
#                                               class_weight="balanced"))])

# def oos_r2(y_true, y_pred, naive_val):
#     """R² vs наивный прогноз (среднее train). >0 = лучше наивного."""
#     mse_m = np.mean((y_true - y_pred) ** 2)
#     mse_n = np.mean((y_true - naive_val) ** 2)
#     return float(1.0 - mse_m / mse_n) if mse_n > 1e-12 else np.nan

# def opt_alpha(res_train, pred_it_train):
#     """
#     Аналитический optimal shrinkage:
#     alpha* = cov(res, pred_it) / var(pred_it)
#     Ограничен [-2, 2] для стабильности.
#     """
#     cmat = np.cov(res_train, pred_it_train)
#     var_it = cmat[1, 1]
#     if var_it < 1e-12:
#         return 0.0
#     return float(np.clip(cmat[0, 1] / var_it, -2.0, 2.0))

# print("\nСтроим таргеты...")
# tgts = {}
# for asset in ALL_ASSETS:
#     if asset not in ret.columns:
#         continue
#     r       = ret[asset]
#     vol_now = r.rolling(VOL_WINDOW).std() * ANN
#     vol_med = vol_now.expanding(min_periods=VOL_WINDOW * 30).median()
#     tgts[asset] = {}
#     for h in HORIZONS:
#         vol_fwd = r.rolling(h).std().shift(-h) * ANN
#         tgts[asset][h] = {
#             "logvol":  np.log(vol_fwd.clip(lower=1e-8)),
#             "delta":   vol_fwd - vol_now,
#             "stress":  (vol_fwd > vol_med * 1.25).astype(float),
#             "vol_now": vol_now,
#         }
# print(f"  Готово: {len(tgts)} активов")

# rows      = []
# Xi_all   = X_it_slim[SELECTED_COLS]
# split_ts = pd.Timestamp(SPLIT_DATE)

# for asset in ALL_ASSETS:
#     if asset not in tgts:
#         continue
#     for h, h_label in HORIZONS.items():
#         for tgt_name in ["logvol", "delta", "stress"]:
#             y_raw     = tgts[asset][h].get(tgt_name, pd.Series(dtype=float)).dropna()
#             vol_now_s = tgts[asset][h]["vol_now"]

#             idx = (X_vol.index
#                    .intersection(Xi_all.index)
#                    .intersection(y_raw.index))
#             valid = (X_vol.loc[idx].notna().all(1) &
#                      Xi_all.loc[idx].notna().all(1) &
#                      y_raw.loc[idx].notna())
#             idx = idx[valid]
#             if len(idx) < 5000:
#                 continue

#             tr = idx < (split_ts - pd.Timedelta(minutes=30 * PURGE_BARS))
#             te = idx >= split_ts
#             if tr.sum() < 2880 or te.sum() < 480:
#                 continue

#             Xv_tr = X_vol.loc[idx[tr]];  Xv_te = X_vol.loc[idx[te]]
#             Xi_tr = Xi_all.loc[idx[tr]]; Xi_te = Xi_all.loc[idx[te]]
#             ytr   = y_raw.loc[idx[tr]];  yte   = y_raw.loc[idx[te]]
#             naive = float(ytr.mean())

#             if tgt_name == "stress":
#                 pos = float(ytr.mean())
#                 if pos < 0.05 or pos > 0.95:
#                     continue
#                 vn_max = float(vol_now_s.loc[idx[tr]].max())
#                 sc_base = np.clip(vol_now_s.loc[idx[te]].values / (vn_max + 1e-8), 0, 1)
#                 try:
#                     mdl_it = _logit(); mdl_it.fit(Xi_tr, ytr)
#                     sc_it  = mdl_it.predict_proba(Xi_te)[:, 1]
#                     Xc_tr  = pd.concat([Xv_tr, Xi_tr], axis=1)
#                     Xc_te  = pd.concat([Xv_te, Xi_te], axis=1)
#                     mdl_c  = _logit(); mdl_c.fit(Xc_tr, ytr)
#                     sc_c   = mdl_c.predict_proba(Xc_te)[:, 1]
#                     rows.append({
#                         "asset": asset, "horizon": h_label, "h": h,
#                         "target": "stress", "pos_rate": round(pos, 3),
#                         "r2_base": np.nan, "r2_comb": np.nan,
#                         "delta_r2": np.nan, "res_corr": np.nan,
#                         "mse_ratio": np.nan, "alpha_it": np.nan,
#                         "pr_base":  round(average_precision_score(yte, sc_base), 4),
#                         "pr_comb":  round(average_precision_score(yte, sc_c),    4),
#                         "delta_pr": round(average_precision_score(yte, sc_c) -
#                                           average_precision_score(yte, sc_base), 4),
#                         "roc_comb": round(roc_auc_score(yte, sc_c), 4),
#                         "n_train": int(tr.sum()), "n_test": int(te.sum()),
#                     })
#                 except Exception:
#                     pass
#                 continue

#             ck   = Xv_tr.corrwith(ytr).abs().replace([np.inf,-np.inf],np.nan).dropna()
#             top3 = ck.nlargest(3).index.tolist()
#             mb   = _ridge(); mb.fit(Xv_tr[top3], ytr)
#             pb_tr = mb.predict(Xv_tr[top3])
#             pb_te = mb.predict(Xv_te[top3])

#             res_tr = ytr.values - pb_tr
#             mit = _ridge(); mit.fit(Xi_tr, res_tr)
#             pit_tr = mit.predict(Xi_tr)
#             pit_te = mit.predict(Xi_te)

#             a = opt_alpha(res_tr, pit_tr)

#             pred_sc = pb_te + a * pit_te   # scaled

#             r2_b  = oos_r2(yte.values, pb_te,   naive)
#             r2_sc = oos_r2(yte.values, pred_sc,  naive)
#             rc    = float(pd.Series(pit_te).corr(pd.Series(yte.values - pb_te)))
#             mse_b = float(np.mean((yte.values - pb_te)**2))
#             mse_s = float(np.mean((yte.values - pred_sc)**2))

#             rows.append({
#                 "asset": asset, "horizon": h_label, "h": h,
#                 "target": tgt_name, "pos_rate": np.nan,
#                 "r2_base":  round(r2_b,  4),
#                 "r2_comb":  round(r2_sc, 4),
#                 "delta_r2": round(r2_sc - r2_b, 4),
#                 "res_corr": round(rc, 4),
#                 "mse_ratio": round(mse_s / mse_b, 4) if mse_b > 1e-12 else np.nan,
#                 "alpha_it": round(a, 3),
#                 "pr_base": np.nan, "pr_comb": np.nan,
#                 "delta_pr": np.nan, "roc_comb": np.nan,
#                 "n_train": int(tr.sum()), "n_test": int(te.sum()),
#             })

# df = pd.DataFrame(rows)
# print(f"\nВсего строк: {len(df)}")

# for tgt_name, label in [("logvol","LOG VOL"), ("delta","DELTA VOL")]:
#     sub = df[df.target == tgt_name].copy()
#     if sub.empty: continue
#     print(f"\n{'='*76}")
#     print(f"  {label}  —  pred = baseline + α·IT_residual")
#     print(f"  ΔR² > 0: IT улучшает R²  |  α показывает насколько сильно масштабируется поправка")
#     print(f"{'='*76}")
#     print(f"\n  {'актив':<6} {'h':<4} {'R²_base':>9} {'R²_comb':>9} "
#           f"{'ΔR²':>7} {'res_corr':>10} {'α':>6} {'MSE_r':>7}")
#     print("  " + "-"*65)
#     prev = None
#     for _, r in sub.sort_values(["asset","h"]).iterrows():
#         if prev and prev != r["asset"]: print("  " + "·"*65)
#         prev = r["asset"]
#         dm = "↑" if r["delta_r2"] > 0.005 else ("↓" if r["delta_r2"] < -0.005 else "~")
#         rm = "✓" if r["res_corr"] > 0.03  else "·"
#         mm = "✓" if r["mse_ratio"] < 1.0  else "·"
#         print(f"  {r['asset']:<6} {r['horizon']:<4}"
#               f" {r['r2_base']:>+9.4f}"
#               f" {r['r2_comb']:>+9.4f}"
#               f" {r['delta_r2']:>+7.4f} {dm}"
#               f" {r['res_corr']:>+9.4f} {rm}"
#               f" {r['alpha_it']:>5.2f}"
#               f" {r['mse_ratio']:>6.4f} {mm}")

#     print(f"\n  ΔR² > 0:          {(sub['delta_r2']>0).sum()} / {len(sub)}")
#     print(f"  res_corr > 0.03:  {(sub['res_corr']>0.03).sum()} / {len(sub)}")
#     print(f"  MSE < 1.0:        {(sub['mse_ratio']<1.0).sum()} / {len(sub)}")
#     print(f"  Среднее ΔR²:     {sub['delta_r2'].mean():+.4f}")
#     print(f"  Среднее res_corr:{sub['res_corr'].mean():+.4f}")
#     print(f"  Среднее α:       {sub['alpha_it'].mean():+.3f}")

# st = df[df.target == "stress"]
# if not st.empty:
#     print(f"\n{'='*70}")
#     print(f"  STRESS  (PR-AUC; baseline = vol_now percentile)")
#     print(f"{'='*70}")
#     print(f"\n  {'актив':<6} {'h':<4} {'pos%':>5} "
#           f"{'PR_base':>9} {'PR_comb':>9} {'ΔPR':>7} {'ROC_comb':>10}")
#     print("  " + "-"*56)
#     prev = None
#     for _, r in st.sort_values(["asset","h"]).iterrows():
#         if prev and prev != r["asset"]: print("  " + "·"*56)
#         prev = r["asset"]
#         mk = "↑" if r["delta_pr"] > 0.01 else "~"
#         print(f"  {r['asset']:<6} {r['horizon']:<4}"
#               f" {r['pos_rate']:>4.0%}"
#               f" {r['pr_base']:>9.4f}"
#               f" {r['pr_comb']:>9.4f}"
#               f" {r['delta_pr']:>+7.4f} {mk}"
#               f" {r['roc_comb']:>9.4f}")
#     print(f"\n  ΔPR > 0: {(st['delta_pr']>0).sum()} / {len(st)}")
#     print(f"  Среднее ROC_comb:  {st['roc_comb'].mean():.4f}")
#     print(f"  Среднее ΔPR-AUC:  {st['delta_pr'].mean():+.4f}")

# reg = df[df.target.isin(["logvol","delta"])]
# print(f"\n{'='*70}")
# print(f"  ИТОГ: среднее ΔR² по активам (logvol + delta)")
# print(f"{'='*70}")
# by_asset = reg.groupby("asset")["delta_r2"].mean().sort_values(ascending=False)
# for asset, val in by_asset.items():
#     n_pos = (reg[reg.asset==asset]["delta_r2"] > 0).sum()
#     n_tot = (reg[reg.asset==asset]).shape[0]
#     bar = "█" * max(0, int((val + 0.3) * 15))
#     print(f"  {asset:<6} {val:>+7.4f}  {bar}  ({n_pos}/{n_tot} горизонтов ↑)")

# holdout_full = df.copy()
# print(f"\nholdout_full сохранён в памяти ({len(df)} строк)")

Сплит: train → 2025-01-01 | test 2025-01-01 →
IT-фичи: 10
Масштабирование: optimal shrinkage α на train-остатках

Строим таргеты...
  Готово: 9 активов

Всего строк: 162

  LOG VOL  —  pred = baseline + α·IT_residual
  ΔR² > 0: IT улучшает R²  |  α показывает насколько сильно масштабируется поправка

  актив  h      R²_base   R²_comb     ΔR²   res_corr      α   MSE_r
  -----------------------------------------------------------------
  ADA    1d     +0.3362   +0.3412 +0.0050 ~   +0.1660 ✓  1.00 0.9924 ✓
  ADA    2d     +0.3047   +0.2995 -0.0053 ↓   +0.1753 ✓  1.00 1.0076 ·
  ADA    3d     +0.2816   +0.2641 -0.0175 ↓   +0.1649 ✓  1.00 1.0244 ·
  ADA    4d     +0.2688   +0.2337 -0.0351 ↓   +0.1468 ✓  1.00 1.0480 ·
  ADA    5d     +0.2604   +0.2116 -0.0488 ↓   +0.1322 ✓  1.00 1.0660 ·
  ADA    6d     +0.2666   +0.2108 -0.0558 ↓   +0.1389 ✓  1.00 1.0761 ·
  ·································································
  AVAX   1d     +0.1835   +0.2118 +0.0283 ↑   +0.1651 ✓  1.00 0.9653

In [61]:
!pip install gudhi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 9.9 MB/s  0:00:006m0:00:0100:01


In [62]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from itertools import combinations
from sklearn.metrics import mutual_info_score
import gudhi
from gudhi.representations import Landscape
from tqdm.auto import tqdm
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

TICKER_MAP = {
    "BTC-USD":"BTC", "ETH-USD":"ETH", "BNB-USD":"BNB", "XRP-USD":"XRP",
    "SOL-USD":"SOL", "TRX-USD":"TRX", "DOGE-USD":"DOGE","ADA-USD":"ADA",
    "ZEC-USD":"ZEC", "BCH-USD":"BCH", "LINK-USD":"LINK","LEO-USD":"LEO",
    "XLM-USD":"XLM", "LTC-USD":"LTC", "XMR-USD":"XMR", "AVAX-USD":"AVAX",
    "HBAR-USD":"HBAR","DAI-USD":"DAI", "DOT-USD":"DOT", "UNI-USD":"UNI",
    "TON-USD":"TON", "CRO-USD":"CRO", "NEAR-USD":"NEAR","ICP-USD":"ICP",
    "AAVE-USD":"AAVE","ETC-USD":"ETC", "PI-USD":"PI",  "WLD-USD":"WLD",
    "MATIC-USD":"MATIC","ATOM-USD":"ATOM","FIL-USD":"FIL","ALGO-USD":"ALGO",
    "VET-USD":"VET",
}

ASSETS_NB = [v for v in TICKER_MAP.values() if v in ret.columns]
print(f"Активы из ноутбука доступные в ret: {len(ASSETS_NB)}")
print(f"  {ASSETS_NB}")

WINDOW = 192   # ~4 дня
STEP   = 48    # 1 день
BINS   = 20

R       = ret[ASSETS_NB].fillna(0).replace([np.inf,-np.inf], 0).values
idx_all = ret.index
n_total = len(R)
t_range = list(range(0, n_total - WINDOW, STEP))
times   = [idx_all[t + WINDOW] for t in t_range]
print(f"Окна: {len(t_range)} | {times[0]} → {times[-1]}")

def mutual_information(x, y, bins=BINS):
    x = np.asarray(x); y = np.asarray(y)
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]; y = y[mask]
    if len(x) < 20: return 0.0
    c_xy = np.histogram2d(x, y, bins=bins)[0]
    return float(mutual_info_score(None, None, contingency=c_xy))

def build_mi_complex_gudhi(X_window, bins=BINS):
    """Cell 16."""
    dim = X_window.shape[1]
    MI = np.zeros((dim, dim))
    for i, j in combinations(range(dim), 2):
        mi = mutual_information(X_window[:, i], X_window[:, j], bins=bins)
        MI[i, j] = MI[j, i] = mi
    triu = MI[np.triu_indices_from(MI, 1)]
    MI_max = triu.max() if len(triu) > 0 else 1.0
    st = gudhi.SimplexTree()
    for i in range(dim):
        st.insert([i], filtration=0.0)
    edge_w = np.zeros((dim, dim))
    for i, j in combinations(range(dim), 2):
        w = MI_max - MI[i, j]
        edge_w[i, j] = edge_w[j, i] = w
        st.insert([i, j], filtration=w)
    for i, j, k in combinations(range(dim), 3):
        w = max(edge_w[i, j], edge_w[i, k], edge_w[j, k])
        st.insert([i, j, k], filtration=w)
    st.initialize_filtration()
    return st, MI, MI_max

def diag_for_dim(diag, dim):
    """Cell 29."""
    out = []
    for d, (b, dd) in diag:
        if d == dim:
            out.append((float(b), float(dd)))
    return out

def diagram_statistics(diag_bd, max_filtration):
    """Cell 30."""
    if len(diag_bd) == 0: return 0.0, 0.0
    lifetimes = []
    for b, d in diag_bd:
        lt = (d - b) if np.isfinite(d) else (max_filtration - b)
        if lt > 0: lifetimes.append(lt)
    if not lifetimes: return 0.0, 0.0
    lts = np.array(lifetimes, dtype=float)
    T = float(lts.sum())
    p = lts / T
    E = float(-(p * np.log(p)).sum())
    return T, E

def landscape_norms(diag_bd, max_filtration, resolution=200):
    """Cell 15."""
    if len(diag_bd) == 0: return 0.0, 0.0
    pts = [[b, d if np.isfinite(d) else max_filtration] for b, d in diag_bd]
    if not pts: return 0.0, 0.0
    arr = np.array(pts, dtype=float)
    try:
        L = Landscape(num_landscapes=1, resolution=resolution)
        X = L.fit_transform([arr])[0]
        return float(np.sum(np.abs(X))), float(np.sqrt(np.sum(X**2)))
    except Exception:
        return 0.0, 0.0

def H_gaussian(S):
    """Cell 26."""
    if isinstance(S, np.ndarray):
        n = S.shape[0]
        det = np.linalg.det(S)
        if det <= 0: return 0.0
        return 0.5 * np.log((2 * np.pi * np.e) ** n * det)
    else:
        if S <= 0: return 0.0
        return 0.5 * np.log(2 * np.pi * np.e * S)

def I(simplex, A=None):
    """Cell 26 — оригинальный код (A[0,0] для маргиналей)."""
    d = len(simplex)
    H_marginal_sum = d * H_gaussian(A[0, 0])
    H_joint = H_gaussian(A[np.ix_(simplex, simplex)])
    return H_marginal_sum - H_joint

def II(simplex, A=None):
    """Cell 26."""
    d = len(simplex)
    H_joint = H_gaussian(A[np.ix_(simplex, simplex)])
    H_residual_sum = 0
    for idx in combinations(simplex, d - 1):
        H_residual_sum += H_gaussian(A[np.ix_(list(idx), list(idx))])
    return -(d - 1) * H_joint + H_residual_sum

def C(simplex, A=None):
    """Cell 26."""
    d = len(simplex)
    coeffs = np.zeros(2**d - 1)
    H_i    = np.zeros(2**d - 1)
    i = 0
    for l in range(1, d + 1):
        sign = 0 - (-1) ** l
        for idx in combinations(simplex, l):
            coeffs[i] = sign
            idx_list = list(idx)
            if len(idx_list) == 1:
                H_i[i] = H_gaussian(A[idx_list[0], idx_list[0]])
            else:
                H_i[i] = H_gaussian(A[np.ix_(idx_list, idx_list)])
            i += 1
    return float(coeffs @ H_i)

def build_gaussian_info_complex(X_window, triple_mode="I",
                                 filtration="sublevel", max_dim=2, eps=1e-8):
    """Cell 25."""
    dim = X_window.shape[1]
    A = np.cov(X_window, rowvar=False) + eps * np.eye(dim)
    triple_fn = {"I": I, "II": II, "C": C}[triple_mode]
    sgn = 1.0 if filtration == "sublevel" else -1.0
    st = gudhi.SimplexTree()
    for i in range(dim):
        st.insert([i], filtration=0.0)
    for i, j in combinations(range(dim), 2):
        st.insert([i, j], filtration=sgn * I([i, j], A=A))
    if max_dim >= 2:
        for i, j, k in combinations(range(dim), 3):
            st.insert([i, j, k], filtration=sgn * triple_fn([i, j, k], A=A))
    st.initialize_filtration()
    return st, A

print("\n" + "="*60)
print(f"  ЭКСПЕРИМЕНТ A — MI-комплекс")
print(f"  window={WINDOW}, step={STEP}, bins={BINS}")
print("="*60)

tp_H0_A=[];  tp_H1_A=[]
ent_H0_A=[]; ent_H1_A=[]
L1_H0_A=[]; L2_H0_A=[]
L1_H1_A=[]; L2_H1_A=[]

for t in tqdm(t_range, desc="Exp A"):
    wnd = R[t:t + WINDOW]
    try:
        st, MI, MI_max = build_mi_complex_gudhi(wnd)
        diag = st.persistence()
        d0 = diag_for_dim(diag, 0)
        d1 = diag_for_dim(diag, 1)
        T0, E0 = diagram_statistics(d0, MI_max)
        T1, E1 = diagram_statistics(d1, MI_max)
        l1_0, l2_0 = landscape_norms(d0, MI_max)
        l1_1, l2_1 = landscape_norms(d1, MI_max)
    except Exception:
        T0=E0=T1=E1=l1_0=l2_0=l1_1=l2_1=0.0

    tp_H0_A.append(T0);  tp_H1_A.append(T1)
    ent_H0_A.append(E0); ent_H1_A.append(E1)
    L1_H0_A.append(l1_0); L2_H0_A.append(l2_0)
    L1_H1_A.append(l1_1); L2_H1_A.append(l2_1)

df_A = pd.DataFrame({
    "T_H0":tp_H0_A,  "E_H0":ent_H0_A,
    "T_H1":tp_H1_A,  "E_H1":ent_H1_A,
    "L1_H0":L1_H0_A, "L2_H0":L2_H0_A,
    "L1_H1":L1_H1_A, "L2_H1":L2_H1_A,
}, index=pd.DatetimeIndex(times))

print(f"\n  H0 T: mean={df_A.T_H0.mean():.4f}  E: mean={df_A.E_H0.mean():.4f}")
print(f"  H1 T: mean={df_A.T_H1.mean():.4f}  nonzero={(df_A.T_H1>0).sum()}/{len(df_A)}")

print("\n" + "="*60)
print("  ШАГ 1: сканируем global_max_filtration (mode=I)")
print("="*60)

global_max = 0.0
for t in tqdm(t_range, desc="scan max_filt"):
    wnd = R[t:t + WINDOW]
    try:
        st, A = build_gaussian_info_complex(wnd, triple_mode="I",
                                             filtration="sublevel", max_dim=2)
        diag = st.persistence()
        fds  = [d for (_, (b, d)) in diag if np.isfinite(d)]
        if fds: global_max = max(global_max, max(fds))
    except Exception:
        pass

global_max_filtration = global_max
print(f"  global_max_filtration = {global_max_filtration:.6f}")

print("\n" + "="*60)
print("  ШАГ 2: ЭКСПЕРИМЕНТ Б — режимы I, II, C")
print("="*60)

stats_B = {}
for mode in ["I", "II", "C"]:
    tp_H0_m=[]; tp_H1_m=[]
    ent_H0_m=[]; ent_H1_m=[]
    L1_H0_m=[]; L2_H0_m=[]
    L1_H1_m=[]; L2_H1_m=[]

    for t in tqdm(t_range, desc=f"Exp B mode={mode}"):
        wnd = R[t:t + WINDOW]
        try:
            st, A = build_gaussian_info_complex(wnd, triple_mode=mode,
                                                 filtration="sublevel", max_dim=2)
            diag = st.persistence()
            d0 = diag_for_dim(diag, 0)
            d1 = diag_for_dim(diag, 1)
            T0, E0 = diagram_statistics(d0, global_max_filtration)
            T1, E1 = diagram_statistics(d1, global_max_filtration)
            l1_0, l2_0 = landscape_norms(d0, global_max_filtration)
            l1_1, l2_1 = landscape_norms(d1, global_max_filtration)
        except Exception:
            T0=E0=T1=E1=l1_0=l2_0=l1_1=l2_1=0.0

        tp_H0_m.append(T0);  tp_H1_m.append(T1)
        ent_H0_m.append(E0); ent_H1_m.append(E1)
        L1_H0_m.append(l1_0); L2_H0_m.append(l2_0)
        L1_H1_m.append(l1_1); L2_H1_m.append(l2_1)

    stats_B[mode] = {
        "times": times,
        "tp_H0": tp_H0_m,  "tp_H1": tp_H1_m,
        "ent_H0":ent_H0_m, "ent_H1":ent_H1_m,
        "L1_H0": L1_H0_m,  "L2_H0": L2_H0_m,
        "L1_H1": L1_H1_m,  "L2_H1": L2_H1_m,
    }
    print(f"  Mode {mode}: H0_T={np.nanmean(tp_H0_m):.4f}  "
          f"H1_T={np.nanmean(tp_H1_m):.4f}  "
          f"H1_nz={(np.array(tp_H1_m)>0).sum()}/{len(tp_H1_m)}")

print("\n" + "="*60)
print("  КОРРЕЛЯЦИЯ TDA-СТАТИСТИК С VOL BTC")
print("="*60)

if "BTC" in ret.columns:
    vol_btc = ret["BTC"].rolling(48).std() * np.sqrt(365 * 48)
    vol_r   = vol_btc.reindex(pd.DatetimeIndex(times), method="nearest")

    print("\n  Эксп. A:")
    for col in ["T_H0","E_H0","T_H1","E_H1","L1_H0","L1_H1"]:
        c = df_A[col].corr(vol_r)
        print(f"    {col:<8}: {c:>+.4f}  {'█'*max(0,int(abs(c)*20))}")

    print("\n  Эксп. Б:")
    for mode in ["I","II","C"]:
        print(f"    Mode {mode}:")
        for col in ["tp_H0","ent_H0","tp_H1","ent_H1"]:
            s = pd.Series(stats_B[mode][col], index=pd.DatetimeIndex(times))
            c = s.corr(vol_r)
            print(f"      {col:<10}: {c:>+.4f}  {'█'*max(0,int(abs(c)*20))}")

print("\nСтроим графики...")

btc_cum = (ret["BTC"] + 1).cumprod() if "BTC" in ret.columns else None
btc_r   = btc_cum.reindex(pd.DatetimeIndex(times), method="nearest") if btc_cum is not None else None

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
if btc_r is not None:
    axes[0].plot(pd.DatetimeIndex(times), btc_r.values, color="#1f77b4", lw=0.8)
    axes[0].set_title("BTC cumulative return")
    axes[0].spines["top"].set_visible(False); axes[0].spines["right"].set_visible(False)
axes[1].plot(pd.DatetimeIndex(times), tp_H0_A, label="H0", color="#2ca02c", lw=0.9)
axes[1].plot(pd.DatetimeIndex(times), tp_H1_A, label="H1", color="#d62728", lw=0.9)
axes[1].set_title("Exp A: Total persistence (T)"); axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.2); axes[1].spines["top"].set_visible(False); axes[1].spines["right"].set_visible(False)
axes[2].plot(pd.DatetimeIndex(times), ent_H0_A, label="H0", color="#2ca02c", lw=0.9, ls="--")
axes[2].plot(pd.DatetimeIndex(times), ent_H1_A, label="H1", color="#d62728", lw=0.9, ls="--")
axes[2].set_title("Exp A: Persistence entropy (E)"); axes[2].legend(fontsize=9)
axes[2].grid(alpha=0.2); axes[2].spines["top"].set_visible(False); axes[2].spines["right"].set_visible(False)
plt.suptitle(f"Эксперимент A ({len(ASSETS_NB)} активов, окно {WINDOW} баров)", fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig("fig_tda_A.png", bbox_inches="tight", dpi=150)
plt.savefig("fig_tda_A.pdf", bbox_inches="tight", dpi=150)
print("  Saved: fig_tda_A.png")

fig, axes = plt.subplots(5, 1, figsize=(14, 14), sharex=True)
colors = {"I":"#1f77b4","II":"#ff7f0e","C":"#2ca02c"}
if btc_r is not None:
    axes[0].plot(pd.DatetimeIndex(times), btc_r.values, color="#1f77b4", lw=0.8)
    axes[0].set_title("BTC cumulative return")
    axes[0].spines["top"].set_visible(False); axes[0].spines["right"].set_visible(False)
for mode in ["I","II","C"]:
    ti = pd.DatetimeIndex(stats_B[mode]["times"])
    axes[1].plot(ti, stats_B[mode]["L1_H0"], label=mode, color=colors[mode], lw=0.9, alpha=0.85)
    axes[2].plot(ti, stats_B[mode]["L2_H0"], label=mode, color=colors[mode], lw=0.9, alpha=0.85)
    axes[3].plot(ti, stats_B[mode]["L1_H1"], label=mode, color=colors[mode], lw=0.9, alpha=0.85)
    axes[4].plot(ti, stats_B[mode]["L2_H1"], label=mode, color=colors[mode], lw=0.9, alpha=0.85)
for ax, title in zip(axes[1:], ["L1(H0)","L2(H0)","L1(H1)","L2(H1)"]):
    ax.set_title(f"Exp Б: {title}"); ax.legend(fontsize=9)
    ax.grid(alpha=0.2); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
plt.suptitle("Эксперимент Б: Гауссовский IT-комплекс (I, II, C sublevel)", fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig("fig_tda_B.png", bbox_inches="tight", dpi=150)
plt.savefig("fig_tda_B.pdf", bbox_inches="tight", dpi=150)
print("  Saved: fig_tda_B.png")

tda_A     = df_A.copy()
tda_B     = stats_B.copy()
tda_times = pd.DatetimeIndex(times)
print(f"\ntda_A {tda_A.shape}, tda_B (I/II/C), tda_times — сохранены в памяти")

Активы из ноутбука доступные в ret: 9
  ['BTC', 'ETH', 'BNB', 'XRP', 'SOL', 'TRX', 'DOGE', 'ADA', 'AVAX']
Окна: 970 | 2023-01-05 00:00:00 → 2025-08-31 05:00:00

  ЭКСПЕРИМЕНТ A — MI-комплекс
  window=192, step=48, bins=20


Exp A: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 970/970 [00:05<00:00, 175.23it/s]



  H0 T: mean=2.0000  E: mean=1.6218
  H1 T: mean=0.0058  nonzero=291/970

  ШАГ 1: сканируем global_max_filtration (mode=I)


scan max_filt: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 970/970 [00:00<00:00, 1067.95it/s]


  global_max_filtration = 2.693593

  ШАГ 2: ЭКСПЕРИМЕНТ Б — режимы I, II, C


...

  Mode I: H0_T=4.3732  H1_T=0.6291  H1_nz=187/970


Exp B mode=II: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 970/970 [00:02<00:00, 324.83it/s]


  Mode II: H0_T=4.1232  H1_T=20.1445  H1_nz=970/970


Exp B mode=C: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 970/970 [00:03<00:00, 290.42it/s]


  Mode C: H0_T=4.0511  H1_T=12.3210  H1_nz=954/970

  КОРРЕЛЯЦИЯ TDA-СТАТИСТИК С VOL BTC

  Эксп. A:
    T_H0    : -0.0411  
    E_H0    : -0.1109  ██
    T_H1    : +0.0193  
    E_H1    : +0.0179  
    L1_H0   : +0.1068  ██
    L1_H1   : +0.0047  

  Эксп. Б:
    Mode I:
      tp_H0     : -0.1052  ██
      ent_H0    : +0.2332  ████
      tp_H1     : +0.2252  ████
      ent_H1    : +0.2782  █████
    Mode II:
      tp_H0     : -0.0836  █
      ent_H0    : +0.2478  ████
      tp_H1     : -0.1809  ███
      ent_H1    : -0.1437  ██
    Mode C:
      tp_H0     : -0.1717  ███
      ent_H0    : +0.2329  ████
      tp_H1     : -0.2099  ████
      ent_H1    : -0.2102  ████

Строим графики...
  Saved: fig_tda_A.png
  Saved: fig_tda_B.png

tda_A (970, 8), tda_B (I/II/C), tda_times — сохранены в памяти


In [105]:
# import warnings; warnings.filterwarnings("ignore")
# import numpy as np
# import pandas as pd
# from sklearn.linear_model import Ridge, RidgeCV
# from sklearn.preprocessing import StandardScaler
# from sklearn.pipeline import Pipeline
# from sklearn.model_selection import KFold, cross_val_score
# import matplotlib
# matplotlib.use("Agg")
# import matplotlib.pyplot as plt
 
# SPLIT_DATE = "2025-01-01"
# PURGE_BARS = 96
# HORIZONS   = {96:"2d", 192:"4d"}       # фокус на средних горизонтах
# ASSETS     = ["BTC","ETH","DOGE","XRP","SOL","BNB"]
# VOL_WINDOW = 48
# ANN        = np.sqrt(365 * 48)
# RNG        = np.random.default_rng(42)
# N_PERM     = 10
 
# def _ridge():
#     return Pipeline([("sc", StandardScaler()),
#                      ("m", RidgeCV(alphas=[0.01,0.1,1,10,100]))])
 
# def oos_r2(y_true, y_pred, naive_val):
#     mse_m = np.mean((y_true - y_pred)**2)
#     mse_n = np.mean((y_true - naive_val)**2)
#     return float(1.0 - mse_m / mse_n) if mse_n > 1e-12 else np.nan
 
# def _get_y(asset, h):
#     col = f"delta_vol_{h}"
#     for src in [targets_extra if 'targets_extra' in dir() else {},
#                 targets, targets_ext]:
#         if asset in src and col in src[asset].columns:
#             return src[asset][col].dropna()
#     return None
 
# try: _ = targets_extra
# except NameError: targets_extra = {}
 
# print("Собираем все фичи...")
 
# tda_frames = [tda_A.add_prefix("A_")]
# for mode in ["I","II","C"]:
#     d = tda_B[mode]
#     idx_tda = pd.DatetimeIndex(d["times"])
#     df_m = pd.DataFrame({
#         f"B{mode}_tp_H0":  d["tp_H0"],  f"B{mode}_ent_H0": d["ent_H0"],
#         f"B{mode}_tp_H1":  d["tp_H1"],  f"B{mode}_ent_H1": d["ent_H1"],
#         f"B{mode}_L1_H0":  d["L1_H0"],  f"B{mode}_L1_H1":  d["L1_H1"],
#     }, index=idx_tda)
#     tda_frames.append(df_m)
 
# X_tda_raw = pd.concat(tda_frames, axis=1).sort_index()
# X_tda_ri  = X_tda_raw.reindex(ret.index, method="ffill")
 
# IT_SLIM = ["coinfo3_std","kl_BNB","coinfo4_std","pairwise_mi_mean",
#            "coinfo4_mean","pairwise_mi_std","coinfo3_mean",
#            "coinfo4_max","kl_AVAX","pairwise_mi_min"]
 
# IT_REMOVED = ["pairwise_cov_std","pairwise_cov_max","pairwise_cov_min",
#               "pairwise_cov_mean","pairwise_mi_max","kl_BTC","kl_TRX",
#               "kl_XRP","kl_SOL","coinfo4_min","kl_DOGE","kl_ETH",
#               "coinfo3_max","kl_ADA","coinfo3_min","pairwise_mi_min",
#               "kl_AVAX","coinfo4_max"]
 
# IT_ALL_25 = IT_SLIM + [c for c in IT_REMOVED if c in X_it_slim.columns
#                         and c not in IT_SLIM]
# IT_ALL_25 = [c for c in IT_ALL_25 if c in X_it_slim.columns]
 
# TDA_COLS = list(X_tda_ri.columns)
 
# print(f"IT-фич всего:  {len(IT_ALL_25)}")
# print(f"IT slim (10):  {len(IT_SLIM)}")
# print(f"TDA-фич:       {len(TDA_COLS)}")
 
# print("\n" + "="*65)
# print("  1. КОРРЕЛЯЦИЯ ВСЕХ ФИЧ С VOL_BTC НА TEST-ПЕРИОДЕ")
# print("="*65)
 
# split_ts  = pd.Timestamp(SPLIT_DATE)
# vol_btc   = ret["BTC"].rolling(VOL_WINDOW).std() * ANN
# vol_test  = vol_btc[vol_btc.index >= split_ts]
 
# it_corrs = {}
# for col in IT_ALL_25:
#     if col not in X_it_slim.columns: continue
#     s = X_it_slim[col].reindex(vol_test.index, method="ffill")
#     it_corrs[col] = float(s.corr(vol_test))
 
# tda_corrs = {}
# for col in TDA_COLS:
#     s = X_tda_ri[col].reindex(vol_test.index, method="ffill")
#     tda_corrs[col] = float(s.corr(vol_test))
 
# print(f"\n  IT-фичи (топ-10 по |corr| с vol_BTC test):")
# it_sorted = sorted(it_corrs.items(), key=lambda x: abs(x[1]), reverse=True)
# for feat, c in it_sorted[:10]:
#     tag = " [slim]" if feat in IT_SLIM else " [removed]"
#     bar = "█" * max(0, int(abs(c)*20))
#     print(f"    {feat:<28} {c:>+.4f}  {bar}{tag}")
 
# print(f"\n  TDA-фичи (топ-10 по |corr|):")
# tda_sorted = sorted(tda_corrs.items(), key=lambda x: abs(x[1]), reverse=True)
# for feat, c in tda_sorted[:10]:
#     bar = "█" * max(0, int(abs(c)*20))
#     print(f"    {feat:<28} {c:>+.4f}  {bar}")
 
# print("\n" + "="*65)
# print("  2. PERMUTATION IMPORTANCE НА TRAIN (BTC+ETH, h=4d, logvol)")
# print("="*65)
 
# perm_results = {}
# for pilot_asset in ["BTC","ETH"]:
#     r = ret[pilot_asset]
#     vol_now = r.rolling(VOL_WINDOW).std() * ANN
#     vol_fwd = r.rolling(192).std().shift(-192) * ANN
#     y_logvol = np.log(vol_fwd.clip(lower=1e-8)).dropna()
 
#     idx = (X_vol.index.intersection(X_it_slim.index)
#            .intersection(X_tda_ri.index).intersection(y_logvol.index))
#     valid = (X_vol.loc[idx].notna().all(1) &
#              X_it_slim.loc[idx].notna().all(1) &
#              X_tda_ri.loc[idx].notna().all(1) &
#              y_logvol.loc[idx].notna())
#     idx = idx[valid]
#     tr   = idx < (split_ts - pd.Timedelta(minutes=30*PURGE_BARS))
#     idx_tr = idx[tr]
 
#     Xv_tr = X_vol.loc[idx_tr]
#     ck    = Xv_tr.corrwith(y_logvol.loc[idx_tr]).abs().dropna()
#     top3  = ck.nlargest(3).index.tolist()
#     mdl_b = _ridge(); mdl_b.fit(Xv_tr[top3], y_logvol.loc[idx_tr])
#     res_tr = y_logvol.loc[idx_tr].values - mdl_b.predict(Xv_tr[top3])
 
#     Xi_all_tr = pd.concat([
#         X_it_slim.loc[idx_tr],
#         X_tda_ri.loc[idx_tr]
#     ], axis=1)
#     Xi_arr  = Xi_all_tr.values
#     res_arr = res_tr
#     feat_names = list(Xi_all_tr.columns)
 
#     kf = KFold(n_splits=5, shuffle=False)
#     base_cv = float(np.mean(cross_val_score(_ridge(), Xi_arr, res_arr,
#                                              cv=kf, scoring="r2")))
 
#     perm_scores_asset = {}
#     for fi, feat in enumerate(feat_names):
#         deltas = []
#         for _ in range(N_PERM):
#             Xp = Xi_arr.copy()
#             Xp[:, fi] = RNG.permutation(Xp[:, fi])
#             sc = float(np.mean(cross_val_score(_ridge(), Xp, res_arr,
#                                                cv=kf, scoring="r2")))
#             deltas.append(sc - base_cv)
#         perm_scores_asset[feat] = float(np.mean(deltas))
 
#     perm_results[pilot_asset] = perm_scores_asset
#     print(f"  {pilot_asset} baseline CV R²={base_cv:.4f}")
 
# all_feats = list(perm_results["BTC"].keys())
# avg_perm  = {f: np.mean([perm_results[a].get(f,0) for a in ["BTC","ETH"]])
#              for f in all_feats}
 
# def categorize(feat):
#     if feat in IT_SLIM:    return "IT_slim"
#     if feat in IT_ALL_25:  return "IT_removed"
#     if feat.startswith("A_"): return "TDA_A"
#     if feat.startswith("BI_"): return "TDA_BI"
#     if feat.startswith("BII_"): return "TDA_BII"
#     if feat.startswith("BC_"): return "TDA_BC"
#     return "other"
 
# cat_colors = {
#     "IT_slim":"#2196F3","IT_removed":"#90CAF9",
#     "TDA_A":"#4CAF50","TDA_BI":"#FF9800",
#     "TDA_BII":"#9C27B0","TDA_BC":"#F44336","other":"#9E9E9E"
# }
 
# print(f"\n  Топ-15 полезных фич (Δ<0 = помогает):")
# sorted_perm = sorted(avg_perm.items(), key=lambda x: x[1])
# for feat, delta in sorted_perm[:15]:
#     cat = categorize(feat)
#     print(f"    {feat:<28} {delta:>+.4f}  [{cat}]")
 
# print(f"\n  Топ-10 вредных фич (Δ>0 = вредит):")
# for feat, delta in sorted(avg_perm.items(), key=lambda x: -x[1])[:10]:
#     cat = categorize(feat)
#     print(f"    {feat:<28} {delta:>+.4f}  [{cat}]")
 
# best_tda = [f for f in TDA_COLS if avg_perm.get(f,0) < -0.002]
# neutral_tda = [f for f in TDA_COLS if -0.002 <= avg_perm.get(f,0) <= 0.002]
# bad_tda  = [f for f in TDA_COLS if avg_perm.get(f,0) > 0.002]
# print(f"\n  TDA-фичи: {len(best_tda)} полезных, "
#       f"{len(neutral_tda)} нейтральных, {len(bad_tda)} вредных")
# print(f"  Лучшие TDA: {best_tda[:8]}")
 
# print("\n" + "="*65)
# print("  3. СРАВНЕНИЕ КОНФИГУРАЦИЙ НА HOLDOUT")
# print("="*65)
 
# CONFIGS = {
#     "slim_IT":    IT_SLIM,
#     "best_IT":    [f for f,d in sorted_perm[:15] if f in IT_ALL_25],
#     "slim+TDA":   IT_SLIM + best_tda[:6],
#     "best+TDA":   [f for f,d in sorted_perm[:12]
#                    if f in IT_ALL_25 or f in TDA_COLS],
# }
# CONFIGS = {k: list(dict.fromkeys(v)) for k,v in CONFIGS.items()}
 
# for name, cols in CONFIGS.items():
#     print(f"  {name}: {len(cols)} фич")
 
# config_rows = []
# for asset in ASSETS:
#     r = ret[asset] if asset in ret.columns else None
#     if r is None: continue
#     vol_now = r.rolling(VOL_WINDOW).std() * ANN
#     vol_fwd_h = {}
#     for h in HORIZONS:
#         vol_fwd_h[h] = r.rolling(h).std().shift(-h) * ANN
 
#     for h, h_label in HORIZONS.items():
#         y_raw = np.log(vol_fwd_h[h].clip(lower=1e-8)).dropna()
 
#         for cfg_name, cfg_cols in CONFIGS.items():
#             it_cols_used  = [c for c in cfg_cols if c in X_it_slim.columns]
#             tda_cols_used = [c for c in cfg_cols if c in X_tda_ri.columns]
 
#             Xi_cfg = pd.concat(
#                 [X_it_slim[it_cols_used]] +
#                 ([X_tda_ri[tda_cols_used]] if tda_cols_used else []),
#                 axis=1
#             ) if it_cols_used or tda_cols_used else X_it_slim[IT_SLIM]
 
#             idx = (X_vol.index.intersection(Xi_cfg.index).intersection(y_raw.index))
#             valid = (X_vol.loc[idx].notna().all(1) &
#                      Xi_cfg.loc[idx].notna().all(1) &
#                      y_raw.loc[idx].notna())
#             idx = idx[valid]
#             if len(idx) < 1000: continue
 
#             tr = idx < (split_ts - pd.Timedelta(minutes=30*PURGE_BARS))
#             te = idx >= split_ts
#             if tr.sum() < 480 or te.sum() < 100: continue
 
#             Xv_tr = X_vol.loc[idx[tr]]; Xv_te = X_vol.loc[idx[te]]
#             Xi_tr = Xi_cfg.loc[idx[tr]]; Xi_te = Xi_cfg.loc[idx[te]]
#             ytr   = y_raw.loc[idx[tr]];  yte   = y_raw.loc[idx[te]]
#             naive = float(ytr.mean())
 
#             ck   = Xv_tr.corrwith(ytr).abs().dropna()
#             top3 = ck.nlargest(3).index.tolist()
#             mb   = _ridge(); mb.fit(Xv_tr[top3], ytr)
#             pb_tr = mb.predict(Xv_tr[top3])
#             pb_te = mb.predict(Xv_te[top3])
 
#             res_tr = ytr.values - pb_tr
#             mc = _ridge(); mc.fit(Xi_tr, res_tr)
#             pc_te = mc.predict(Xi_te)
#             pred  = pb_te + pc_te
 
#             r2_b = oos_r2(yte.values, pb_te, naive)
#             r2_c = oos_r2(yte.values, pred,  naive)
#             rc   = float(pd.Series(pc_te).corr(pd.Series(yte.values - pb_te)))
 
#             config_rows.append({
#                 "asset":   asset, "horizon": h_label, "h": h,
#                 "config":  cfg_name,
#                 "r2_base": round(r2_b, 4),
#                 "r2_cfg":  round(r2_c, 4),
#                 "delta_r2":round(r2_c - r2_b, 4),
#                 "res_corr":round(rc, 4),
#             })
 
# cfg_df = pd.DataFrame(config_rows)
 
# print(f"\n  Среднее ΔR² (logvol) по конфигурациям:")
# print(f"  {'config':<15} {'mean_ΔR²':>10} {'ΔR²>0':>8} {'mean_rc':>9}")
# print("  " + "-"*45)
# for cfg in CONFIGS.keys():
#     sub = cfg_df[cfg_df.config == cfg]
#     if sub.empty: continue
#     pos = (sub["delta_r2"] > 0).sum()
#     print(f"  {cfg:<15} {sub['delta_r2'].mean():>+10.4f} "
#           f"{pos:>5}/{len(sub)} {sub['res_corr'].mean():>+9.4f}")
 
# best_cfg = cfg_df.groupby("config")["delta_r2"].mean().idxmax()
# print(f"\n  Лучшая конфигурация: '{best_cfg}'")
# sub_best = cfg_df[cfg_df.config == best_cfg]
# print(f"  {'актив':<6} {'h':<4} {'R²_base':>9} {'R²_best':>9} {'ΔR²':>8}")
# print("  " + "-"*40)
# prev = None
# for _, row in sub_best.sort_values(["asset","h"]).iterrows():
#     if prev and prev != row["asset"]: print("  " + "·"*40)
#     prev = row["asset"]
#     mark = "↑" if row["delta_r2"] > 0.005 else ("↓" if row["delta_r2"] < -0.005 else "~")
#     print(f"  {row['asset']:<6} {row['horizon']:<4}"
#           f" {row['r2_base']:>+9.4f} {row['r2_cfg']:>+9.4f}"
#           f" {row['delta_r2']:>+8.4f} {mark}")
 
# print("\nСтроим графики...")
 
# fig, axes = plt.subplots(1, 3, figsize=(18, 7))
 
# ax = axes[0]
# feats_sorted = [f for f,_ in sorted_perm]
# vals_sorted  = [avg_perm[f] for f in feats_sorted]
# colors_bar   = [cat_colors.get(categorize(f), "#9E9E9E") for f in feats_sorted]
# y_pos = np.arange(len(feats_sorted))
# ax.barh(y_pos, vals_sorted, color=colors_bar, alpha=0.85, height=0.8)
# ax.axvline(0, color="black", lw=0.8)
# ax.axvline(-0.002, color="green",  lw=0.8, ls=":", alpha=0.6)
# ax.axvline(+0.002, color="red",    lw=0.8, ls=":", alpha=0.6)
# ax.set_yticks(y_pos)
# ax.set_yticklabels(feats_sorted, fontsize=7)
# ax.set_xlabel("Δ CV R² при перемешивании\n(< 0 = фича полезна)")
# ax.set_title("Permutation importance\n(BTC+ETH, h=4d, logvol)", fontsize=10)
# ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
# from matplotlib.patches import Patch
# legend_elems = [Patch(facecolor=c, label=k, alpha=0.85)
#                 for k,c in cat_colors.items() if k != "other"]
# ax.legend(handles=legend_elems, fontsize=7, loc="lower right")
 
# ax = axes[1]
# cfg_order = list(CONFIGS.keys())
# asset_order = ASSETS
# x = np.arange(len(asset_order))
# w = 0.18
# cfg_colors = ["#90CAF9","#2196F3","#4CAF50","#FF9800"]
# for ci, cfg in enumerate(cfg_order):
#     sub = cfg_df[cfg_df.config == cfg]
#     vals = []
#     for asset in asset_order:
#         v = sub[sub.asset==asset]["delta_r2"].mean()
#         vals.append(v if not np.isnan(v) else 0)
#     ax.bar(x + (ci-1.5)*w, vals, w*0.9,
#            label=cfg, color=cfg_colors[ci], alpha=0.85)
# ax.axhline(0, color="black", lw=0.8)
# ax.set_xticks(x); ax.set_xticklabels(asset_order)
# ax.set_ylabel("Среднее ΔR² (все горизонты)")
# ax.set_title("ΔR² по конфигурациям × активам\n(logvol, holdout 2025)", fontsize=10)
# ax.legend(fontsize=8); ax.grid(alpha=0.2, axis="y")
# ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
 
# ax = axes[2]
# cat_corrs = {}
# for feat, c in {**it_corrs, **tda_corrs}.items():
#     cat = categorize(feat)
#     if cat not in cat_corrs: cat_corrs[cat] = []
#     cat_corrs[cat].append(c)
 
# cats_to_plot = ["IT_slim","IT_removed","TDA_A","TDA_BI","TDA_BII","TDA_BC"]
# data_plot    = [cat_corrs.get(c,[0]) for c in cats_to_plot]
# bp = ax.boxplot(data_plot, labels=cats_to_plot, patch_artist=True,
#                 medianprops=dict(color="black",lw=1.5),
#                 flierprops=dict(marker=".",markersize=4,alpha=0.5))
# for patch, cat in zip(bp["boxes"], cats_to_plot):
#     patch.set_facecolor(cat_colors.get(cat,"#9E9E9E"))
#     patch.set_alpha(0.75)
# ax.axhline(0, color="black", lw=0.5)
# ax.set_ylabel("|corr| с vol_BTC (test period)")
# ax.set_title("Корреляция фич с vol_BTC\nпо категориям", fontsize=10)
# ax.set_xticklabels(cats_to_plot, rotation=20, ha="right", fontsize=8)
# ax.grid(alpha=0.2, axis="y")
# ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
 
# plt.suptitle("Анализ полезности фич: IT + TDA комплексы",
#              fontsize=13, y=1.01)
# plt.tight_layout()
# plt.savefig("fig_feature_analysis.png", bbox_inches="tight", dpi=150)
# plt.savefig("fig_feature_analysis.pdf", bbox_inches="tight", dpi=150)
# plt.show()
# print("  Saved: fig_feature_analysis.png / .pdf")
 
# print(f"\n{'='*65}")
# print(f"  ИТОГ: лучший набор фич")
# print(f"{'='*65}")
 
# print(f"\n  Конфигурация '{best_cfg}' ({len(CONFIGS[best_cfg])} фич):")
# for f in CONFIGS[best_cfg]:
#     cat = categorize(f)
#     imp = avg_perm.get(f, 0)
#     print(f"    {f:<28} [{cat}]  perm={imp:>+.4f}")
 
# print(f"\n  Сравнение всех конфигураций:")
# for cfg in cfg_order:
#     sub = cfg_df[cfg_df.config == cfg]
#     print(f"    {cfg:<15}: ΔR²={sub['delta_r2'].mean():>+.4f}  "
#           f"rc={sub['res_corr'].mean():>+.4f}  "
#           f"↑{(sub['delta_r2']>0).sum()}/{len(sub)}")
 
# feat_analysis = {"perm": avg_perm, "cfg_df": cfg_df,
#                  "it_corrs": it_corrs, "tda_corrs": tda_corrs,
#                  "best_cfg": best_cfg, "best_cols": CONFIGS[best_cfg]}
# print(f"\nfeat_analysis сохранён в памяти")

Собираем все фичи...
IT-фич всего:  25
IT slim (10):  10
TDA-фич:       26

  1. КОРРЕЛЯЦИЯ ВСЕХ ФИЧ С VOL_BTC НА TEST-ПЕРИОДЕ

  IT-фичи (топ-10 по |corr| с vol_BTC test):
    pairwise_cov_min             +0.6662  █████████████ [removed]
    pairwise_cov_mean            +0.6416  ████████████ [removed]
    pairwise_cov_std             +0.6001  ████████████ [removed]
    pairwise_cov_max             +0.5999  ███████████ [removed]
    coinfo4_min                  +0.4073  ████████ [removed]
    pairwise_mi_mean             +0.4029  ████████ [slim]
    coinfo3_min                  +0.3937  ███████ [removed]
    coinfo3_mean                 +0.3849  ███████ [slim]
    coinfo4_mean                 +0.3849  ███████ [slim]
    pairwise_mi_min              +0.3779  ███████ [slim]

  TDA-фичи (топ-10 по |corr|):
    BI_ent_H1                    +0.3196  ██████
    BC_ent_H1                    -0.3150  ██████
    BI_L1_H1                     +0.3006  ██████
    BII_tp_H1                    -0.29

In [67]:
# import warnings; warnings.filterwarnings("ignore")
# import numpy as np
# import pandas as pd
# from sklearn.linear_model import Ridge, RidgeCV
# from sklearn.preprocessing import StandardScaler
# from sklearn.pipeline import Pipeline
# from scipy import stats as sp_stats
# import matplotlib
# matplotlib.use("Agg")
# import matplotlib.pyplot as plt

# SPLIT_DATE = "2025-01-01"
# PURGE_BARS = 96
# HORIZONS   = {48:"1d", 96:"2d", 192:"4d", 288:"6d"}
# ASSETS     = ["BTC","ETH","SOL","BNB","XRP","DOGE","ADA","AVAX","TRX"]
# VOL_WINDOW = 48
# ANN        = np.sqrt(365 * 48)

# IT_SLIM = ["coinfo3_std","kl_BNB","coinfo4_std","pairwise_mi_mean",
#            "coinfo4_mean","pairwise_mi_std","coinfo3_mean",
#            "coinfo4_max","kl_AVAX","pairwise_mi_min"]

# TDA_BEST = ["BI_tp_H0","BI_ent_H1","BI_L1_H0",
#             "BII_tp_H0","BII_ent_H0","BC_tp_H0"]

# def _ridge():
#     return Pipeline([("sc", StandardScaler()),
#                      ("m", RidgeCV(alphas=[0.01,0.1,1,10,100]))])

# def oos_r2(y_true, y_pred, naive_val):
#     mse_m = np.mean((y_true - y_pred)**2)
#     mse_n = np.mean((y_true - naive_val)**2)
#     return float(1.0 - mse_m / mse_n) if mse_n > 1e-12 else np.nan

# def res_corr(res, pred_it):
#     return float(pd.Series(pred_it).corr(pd.Series(res)))

# print("Строим X_tda...")
# tda_frames = [tda_A.add_prefix("A_")]
# for mode in ["I","II","C"]:
#     d = tda_B[mode]
#     df_m = pd.DataFrame({
#         f"B{mode}_tp_H0":  d["tp_H0"],  f"B{mode}_ent_H0": d["ent_H0"],
#         f"B{mode}_tp_H1":  d["tp_H1"],  f"B{mode}_ent_H1": d["ent_H1"],
#         f"B{mode}_L1_H0":  d["L1_H0"],  f"B{mode}_L1_H1":  d["L1_H1"],
#     }, index=pd.DatetimeIndex(d["times"]))
#     tda_frames.append(df_m)

# X_tda_raw = pd.concat(tda_frames, axis=1).sort_index()
# X_tda_ri  = X_tda_raw[TDA_BEST].reindex(ret.index, method="ffill")

# TDA_BEST = [c for c in TDA_BEST if c in X_tda_ri.columns]
# print(f"TDA фичи: {TDA_BEST}")

# split_ts = pd.Timestamp(SPLIT_DATE)
# rows = []

# for asset in ASSETS:
#     if asset not in ret.columns: continue
#     r = ret[asset]
#     vol_now = r.rolling(VOL_WINDOW).std() * ANN

#     for h, h_label in HORIZONS.items():
#         vol_fwd = r.rolling(h).std().shift(-h) * ANN
#         y_raw   = np.log(vol_fwd.clip(lower=1e-8)).dropna()

#         Xi_it  = X_it_slim[IT_SLIM]
#         Xi_tda = X_tda_ri

#         idx = (X_vol.index.intersection(Xi_it.index)
#                .intersection(Xi_tda.index).intersection(y_raw.index))
#         valid = (X_vol.loc[idx].notna().all(1) &
#                  Xi_it.loc[idx].notna().all(1) &
#                  Xi_tda.loc[idx].notna().all(1) &
#                  y_raw.loc[idx].notna())
#         idx = idx[valid]
#         if len(idx) < 1000: continue

#         tr = idx < (split_ts - pd.Timedelta(minutes=30*PURGE_BARS))
#         te = idx >= split_ts
#         if tr.sum() < 480 or te.sum() < 100: continue

#         Xv_tr  = X_vol.loc[idx[tr]];  Xv_te  = X_vol.loc[idx[te]]
#         Xit_tr = Xi_it.loc[idx[tr]];  Xit_te = Xi_it.loc[idx[te]]
#         Xtd_tr = Xi_tda.loc[idx[tr]]; Xtd_te = Xi_tda.loc[idx[te]]
#         ytr    = y_raw.loc[idx[tr]];  yte    = y_raw.loc[idx[te]]
#         naive  = float(ytr.mean())

#         ck   = Xv_tr.corrwith(ytr).abs().replace([np.inf,-np.inf],np.nan).dropna()
#         top3 = ck.nlargest(3).index.tolist()
#         mb   = _ridge(); mb.fit(Xv_tr[top3], ytr)
#         pb_tr = mb.predict(Xv_tr[top3])
#         pb_te = mb.predict(Xv_te[top3])
#         res_tr = ytr.values - pb_tr
#         res_te = yte.values - pb_te

#         mit = _ridge(); mit.fit(Xit_tr, res_tr)
#         pit_te = mit.predict(Xit_te)
#         pred_it = pb_te + pit_te

#         Xc_tr = pd.concat([Xit_tr, Xtd_tr], axis=1)
#         Xc_te = pd.concat([Xit_te, Xtd_te], axis=1)
#         mct = _ridge(); mct.fit(Xc_tr, res_tr)
#         pct_te = mct.predict(Xc_te)
#         pred_itd = pb_te + pct_te

#         r2_b   = oos_r2(yte.values, pb_te,    naive)
#         r2_it  = oos_r2(yte.values, pred_it,  naive)
#         r2_itd = oos_r2(yte.values, pred_itd, naive)

#         rc_it  = res_corr(res_te, pit_te)
#         rc_itd = res_corr(res_te, pct_te)

#         mse_b   = np.mean((yte.values - pb_te)**2)
#         mse_it  = np.mean((yte.values - pred_it)**2)
#         mse_itd = np.mean((yte.values - pred_itd)**2)

#         d_it  = (yte.values - pb_te)**2 - (yte.values - pred_it)**2
#         d_itd = (yte.values - pb_te)**2 - (yte.values - pred_itd)**2

#         def dm(d):
#             n = len(d); m = d.mean(); s = d.std(ddof=1)
#             t = m/(s/np.sqrt(n)) if s > 1e-10 else 0
#             return float(t), float(sp_stats.t.sf(t, df=n-1))

#         t_it,  p_it  = dm(d_it)
#         t_itd, p_itd = dm(d_itd)

#         rows.append({
#             "asset": asset, "horizon": h_label, "h": h,
#             "r2_base": round(r2_b,   4),
#             "r2_it":   round(r2_it,  4),
#             "r2_itd":  round(r2_itd, 4),
#             "dR2_it":  round(r2_it  - r2_b, 4),
#             "dR2_itd": round(r2_itd - r2_b, 4),
#             "dR2_tda": round(r2_itd - r2_it, 4),
#             "rc_it":   round(rc_it,  4),
#             "rc_itd":  round(rc_itd, 4),
#             "mse_r_it":  round(mse_it  / mse_b, 4) if mse_b > 0 else np.nan,
#             "mse_r_itd": round(mse_itd / mse_b, 4) if mse_b > 0 else np.nan,
#             "t_it":  round(t_it,  3),
#             "p_it":  round(p_it,  4),
#             "t_itd": round(t_itd, 3),
#             "p_itd": round(p_itd, 4),
#             "sig_it":  "***" if p_it<0.01 else ("**" if p_it<0.05 else ("*" if p_it<0.10 else "")),
#             "sig_itd": "***" if p_itd<0.01 else ("**" if p_itd<0.05 else ("*" if p_itd<0.10 else "")),
#             "n_train": int(tr.sum()),
#             "n_test":  int(te.sum()),
#         })

# df = pd.DataFrame(rows)
# print(f"\nВсего строк: {len(df)}")

# print("\n" + "="*78)
# print("  ФИНАЛЬНОЕ СРАВНЕНИЕ: Baseline → slim_IT → slim+TDA")
# print("  Таргет: logvol | Метрики: R², ΔR², res_corr, MSE_ratio, DM p-value")
# print("="*78)

# print(f"\n  {'актив':<6} {'h':<4} "
#       f"{'R²_base':>8} {'R²_IT':>7} {'R²_ITD':>7} "
#       f"{'ΔIT':>7} {'ΔTDA':>7} "
#       f"{'rc_IT':>7} {'rc_ITD':>7} "
#       f"{'MSE_IT':>7} {'MSE_ITD':>8} "
#       f"{'p_IT':>7} {'p_ITD':>8}")
# print("  " + "-"*100)

# prev = None
# for _, row in df.sort_values(["asset","h"]).iterrows():
#     if prev and prev != row["asset"]: print("  " + "·"*100)
#     prev = row["asset"]
#     it_m  = "↑" if row["dR2_it"]  > 0.005 else ("↓" if row["dR2_it"]  < -0.005 else "~")
#     tda_m = "↑" if row["dR2_tda"] > 0.005 else ("↓" if row["dR2_tda"] < -0.005 else "~")
#     print(f"  {row['asset']:<6} {row['horizon']:<4}"
#           f" {row['r2_base']:>+8.4f}"
#           f" {row['r2_it']:>+7.4f}"
#           f" {row['r2_itd']:>+7.4f}"
#           f" {row['dR2_it']:>+7.4f}{it_m}"
#           f" {row['dR2_tda']:>+7.4f}{tda_m}"
#           f" {row['rc_it']:>+7.4f}"
#           f" {row['rc_itd']:>+7.4f}"
#           f" {row['mse_r_it']:>7.4f}"
#           f" {row['mse_r_itd']:>8.4f}"
#           f" {row['p_it']:>7.4f}{row['sig_it']}"
#           f" {row['p_itd']:>7.4f}{row['sig_itd']}")

# print(f"\n{'='*78}")
# print(f"  СВОДКА")
# print(f"{'='*78}")

# n = len(df)
# print(f"""
#   Конфигурация         ΔR²(mean)  ΔR²>0    rc(mean)  MSE<1.0   p<0.10
#   ─────────────────────────────────────────────────────────────────────
#   Baseline → slim_IT   {df['dR2_it'].mean():>+8.4f}   {(df['dR2_it']>0).sum():>2}/{n}   {df['rc_it'].mean():>+7.4f}   {(df['mse_r_it']<1).sum():>2}/{n}   {(df['p_it']<0.10).sum():>2}/{n}
#   Baseline → slim+TDA  {df['dR2_itd'].mean():>+8.4f}   {(df['dR2_itd']>0).sum():>2}/{n}   {df['rc_itd'].mean():>+7.4f}   {(df['mse_r_itd']<1).sum():>2}/{n}   {(df['p_itd']<0.10).sum():>2}/{n}
#   IT → TDA incremental {df['dR2_tda'].mean():>+8.4f}   {(df['dR2_tda']>0).sum():>2}/{n}
# """)

# print(f"  Топ-5 по ΔR² (slim+TDA):")
# for _, row in df.nlargest(5,"dR2_itd").iterrows():
#     print(f"    {row['asset']} h={row['horizon']}: "
#           f"R²_base={row['r2_base']:+.4f} → R²_ITD={row['r2_itd']:+.4f} "
#           f"(ΔR²={row['dR2_itd']:+.4f}, p={row['p_itd']:.4f}{row['sig_itd']})")

# print(f"\n  По активам — среднее ΔR² (slim+TDA):")
# by_asset = df.groupby("asset")[["dR2_it","dR2_itd","rc_it","rc_itd"]].mean()
# for asset, row in by_asset.sort_values("dR2_itd",ascending=False).iterrows():
#     bar_it  = "█" * max(0, int((row["dR2_it"]  + 0.15) * 15))
#     bar_itd = "█" * max(0, int((row["dR2_itd"] + 0.15) * 15))
#     winner = "slim+TDA ↑" if row["dR2_itd"] > row["dR2_it"] + 0.005 else (
#              "slim_IT ↑"  if row["dR2_it"]  > row["dR2_itd"] + 0.005 else "~")
#     print(f"    {asset:<6}: IT={row['dR2_it']:>+.4f}  ITD={row['dR2_itd']:>+.4f}  "
#           f"rc_IT={row['rc_it']:>+.4f}  rc_ITD={row['rc_itd']:>+.4f}  {winner}")

# print("\nСтроим финальные графики...")

# fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# COLORS = {"Baseline":"#9E9E9E","slim_IT":"#2196F3","slim+TDA":"#4CAF50"}
# asset_order = df.groupby("asset")["dR2_itd"].mean().sort_values(ascending=False).index.tolist()

# ax = axes[0,0]
# x  = np.arange(len(asset_order)); w = 0.35
# it_vals  = [df[df.asset==a]["dR2_it"].mean()  for a in asset_order]
# itd_vals = [df[df.asset==a]["dR2_itd"].mean() for a in asset_order]
# ax.bar(x - w/2, it_vals,  w, label="slim_IT",   color="#2196F3", alpha=0.8)
# ax.bar(x + w/2, itd_vals, w, label="slim+TDA",  color="#4CAF50", alpha=0.8)
# ax.axhline(0, color="black", lw=0.8)
# ax.set_xticks(x); ax.set_xticklabels(asset_order, fontsize=9)
# ax.set_ylabel("Среднее ΔR² (vs baseline)")
# ax.set_title("ΔR² по активам: IT vs IT+TDA\n(logvol, все горизонты)", fontsize=10)
# ax.legend(fontsize=9); ax.grid(alpha=0.2, axis="y")
# ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

# ax = axes[0,1]
# h_order = ["1d","2d","4d","6d"]
# r2_base_h = [df[df.horizon==h]["r2_base"].mean() for h in h_order]
# r2_it_h   = [df[df.horizon==h]["r2_it"].mean()   for h in h_order]
# r2_itd_h  = [df[df.horizon==h]["r2_itd"].mean()  for h in h_order]
# ax.plot(h_order, r2_base_h, "o-", color="#9E9E9E", lw=2, label="Baseline", ms=7)
# ax.plot(h_order, r2_it_h,  "s-", color="#2196F3", lw=2, label="slim_IT",  ms=7)
# ax.plot(h_order, r2_itd_h, "^-", color="#4CAF50", lw=2, label="slim+TDA", ms=7)
# ax.axhline(0, color="black", lw=0.5, ls=":")
# ax.set_ylabel("OOS R² (vs naive mean)"); ax.set_xlabel("Горизонт")
# ax.set_title("OOS R² по горизонтам\n(среднее по 9 активам)", fontsize=10)
# ax.legend(fontsize=9); ax.grid(alpha=0.2)
# ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

# ax = axes[1,0]
# rc_it_vals  = [df[df.asset==a]["rc_it"].mean()  for a in asset_order]
# rc_itd_vals = [df[df.asset==a]["rc_itd"].mean() for a in asset_order]
# ax.bar(x - w/2, rc_it_vals,  w, label="rc slim_IT",  color="#2196F3", alpha=0.8)
# ax.bar(x + w/2, rc_itd_vals, w, label="rc slim+TDA", color="#4CAF50", alpha=0.8)
# ax.axhline(0,    color="black", lw=0.8)
# ax.axhline(0.03, color="red",   lw=0.8, ls=":", alpha=0.6, label="threshold 0.03")
# ax.set_xticks(x); ax.set_xticklabels(asset_order, fontsize=9)
# ax.set_ylabel("Residual correlation")
# ax.set_title("Residual corr: IT vs IT+TDA\n(остатки baseline)", fontsize=10)
# ax.legend(fontsize=9); ax.grid(alpha=0.2, axis="y")
# ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

# ax = axes[1,1]
# pivot = df.pivot_table(index="asset", columns="horizon",
#                        values="mse_r_itd", aggfunc="mean")
# pivot = pivot.reindex(asset_order)[h_order]
# im = ax.imshow(pivot.values, cmap="RdYlGn_r", vmin=0.85, vmax=1.15, aspect="auto")
# ax.set_xticks(range(len(h_order)));   ax.set_xticklabels(h_order)
# ax.set_yticks(range(len(asset_order))); ax.set_yticklabels(asset_order)
# for i in range(len(asset_order)):
#     for j in range(len(h_order)):
#         val = pivot.values[i,j]
#         if np.isfinite(val):
#             col = "white" if abs(val-1)>0.07 else "black"
#             mark = "✓" if val < 1.0 else ""
#             ax.text(j, i, f"{val:.3f}{mark}", ha="center", va="center",
#                     fontsize=8, color=col)
# plt.colorbar(im, ax=ax, label="MSE_ratio (slim+TDA / baseline)\n<1 = лучше baseline")
# ax.set_title("MSE_ratio heatmap (slim+TDA)\nзелёный = лучше baseline", fontsize=10)

# plt.suptitle("Финальное сравнение: Baseline vs slim_IT vs slim+TDA\n(logvol, holdout 2025-01-01 →)",
#              fontsize=12, y=1.01)
# plt.tight_layout()
# plt.savefig("fig_final_comparison.png", bbox_inches="tight", dpi=150)
# plt.savefig("fig_final_comparison.pdf", bbox_inches="tight", dpi=150)
# plt.show()
# print("  Saved: fig_final_comparison.png / .pdf")

# final_comparison = df.copy()
# print(f"\nfinal_comparison сохранён в памяти ({len(df)} строк)")

Строим X_tda...
TDA фичи: ['BI_tp_H0', 'BI_ent_H1', 'BI_L1_H0', 'BII_tp_H0', 'BII_ent_H0', 'BC_tp_H0']

Всего строк: 36

  ФИНАЛЬНОЕ СРАВНЕНИЕ: Baseline → slim_IT → slim+TDA
  Таргет: logvol | Метрики: R², ΔR², res_corr, MSE_ratio, DM p-value

  актив  h     R²_base   R²_IT  R²_ITD     ΔIT    ΔTDA   rc_IT  rc_ITD  MSE_IT  MSE_ITD    p_IT    p_ITD
  ----------------------------------------------------------------------------------------------------
  ADA    1d    +0.3362 +0.3412 +0.3548 +0.0051↑ +0.0135↑ +0.1660 +0.1265  0.9924   0.9720  0.0016***  0.0000***
  ADA    2d    +0.3047 +0.2995 +0.3236 -0.0052↓ +0.0241↑ +0.1753 +0.1523  1.0075   0.9729  0.9833  0.0000***
  ADA    4d    +0.2688 +0.2338 +0.2651 -0.0350↓ +0.0313↑ +0.1468 +0.1454  1.0479   1.0050  1.0000  0.8985
  ADA    6d    +0.2666 +0.2108 +0.2272 -0.0557↓ +0.0163↑ +0.1389 +0.1292  1.0760   1.0537  1.0000  1.0000
  ····································································································
  AVAX   1d 

In [68]:
# import numpy as np
# import pandas as pd

# df = final_comparison.copy()

# for h_label in ["1d","2d","4d","6d"]:
#     sub = df[df.horizon == h_label].sort_values("asset")
#     if sub.empty: continue

#     print(f"\n{'='*68}")
#     print(f"  Горизонт h={h_label}  (logvol, holdout 2025-01-01 →)")
#     print(f"{'='*68}")
#     print(f"  {'актив':<6} {'R²_base':>9} {'R²_IT':>9} {'R²_ITD':>9} "
#           f"{'ΔIT':>8} {'ΔTDA':>8}  лучшая")
#     print("  " + "-"*62)

#     for _, row in sub.iterrows():
#         configs = {
#             "base": row["r2_base"],
#             "IT":   row["r2_it"],
#             "ITD":  row["r2_itd"],
#         }
#         best = max(configs, key=lambda k: configs[k])
#         best_label = {"base":"baseline","IT":"slim_IT","ITD":"slim+TDA"}[best]

#         it_m  = "↑" if row["dR2_it"]  > 0.005 else ("↓" if row["dR2_it"]  < -0.005 else "~")
#         tda_m = "↑" if row["dR2_tda"] > 0.005 else ("↓" if row["dR2_tda"] < -0.005 else "~")

#         print(f"  {row['asset']:<6}"
#               f" {row['r2_base']:>+9.4f}"
#               f" {row['r2_it']:>+9.4f}"
#               f" {row['r2_itd']:>+9.4f}"
#               f" {row['dR2_it']:>+7.4f}{it_m}"
#               f" {row['dR2_tda']:>+7.4f}{tda_m}"
#               f"  → {best_label}")

#     print(f"  {'─'*62}")
#     print(f"  {'mean':<6}"
#           f" {sub['r2_base'].mean():>+9.4f}"
#           f" {sub['r2_it'].mean():>+9.4f}"
#           f" {sub['r2_itd'].mean():>+9.4f}"
#           f" {sub['dR2_it'].mean():>+7.4f}"
#           f" {sub['dR2_tda'].mean():>+7.4f}")
#     wins_it  = (sub["dR2_it"]  > 0.005).sum()
#     wins_tda = (sub["dR2_tda"] > 0.005).sum()
#     print(f"  slim_IT лучше baseline: {wins_it}/{len(sub)}  |  "
#           f"slim+TDA лучше IT: {wins_tda}/{len(sub)}")

# print(f"\n{'='*68}")
# print(f"  ИТОГ — все горизонты вместе ({len(df)} задач)")
# print(f"{'='*68}")
# print(f"\n  {'актив':<6} {'R²_base':>9} {'R²_IT':>9} {'R²_ITD':>9} "
#       f"{'ΔIT':>8} {'ΔTDA':>8}")
# print("  " + "-"*58)

# for asset in df.groupby("asset")["r2_itd"].mean().sort_values(ascending=False).index:
#     g = df[df.asset == asset]
#     print(f"  {asset:<6}"
#           f" {g['r2_base'].mean():>+9.4f}"
#           f" {g['r2_it'].mean():>+9.4f}"
#           f" {g['r2_itd'].mean():>+9.4f}"
#           f" {g['dR2_it'].mean():>+8.4f}"
#           f" {g['dR2_tda'].mean():>+8.4f}")

# print(f"  {'─'*58}")
# print(f"  {'mean':<6}"
#       f" {df['r2_base'].mean():>+9.4f}"
#       f" {df['r2_it'].mean():>+9.4f}"
#       f" {df['r2_itd'].mean():>+9.4f}"
#       f" {df['dR2_it'].mean():>+8.4f}"
#       f" {df['dR2_tda'].mean():>+8.4f}")

# n = len(df)
# print(f"\n  slim_IT  > baseline: {(df['dR2_it']>0.005).sum()}/{n} задач")
# print(f"  slim+TDA > baseline: {(df['dR2_itd']>0.005).sum()}/{n} задач")
# print(f"  slim+TDA > slim_IT:  {(df['dR2_tda']>0.005).sum()}/{n} задач")


  Горизонт h=1d  (logvol, holdout 2025-01-01 →)
  актив    R²_base     R²_IT    R²_ITD      ΔIT     ΔTDA  лучшая
  --------------------------------------------------------------
  ADA      +0.3362   +0.3412   +0.3548 +0.0051↑ +0.0135↑  → slim+TDA
  AVAX     +0.1835   +0.2118   +0.2425 +0.0283↑ +0.0307↑  → slim+TDA
  BNB      +0.3518   +0.3446   +0.3391 -0.0072↓ -0.0055↓  → baseline
  BTC      +0.1960   +0.1549   +0.1560 -0.0411↓ +0.0011~  → baseline
  DOGE     +0.2485   +0.3139   +0.3330 +0.0654↑ +0.0191↑  → slim+TDA
  ETH      +0.3996   +0.4467   +0.4583 +0.0471↑ +0.0116↑  → slim+TDA
  SOL      +0.3084   +0.2950   +0.3019 -0.0134↓ +0.0070↑  → baseline
  TRX      +0.2356   +0.1015   +0.1582 -0.1341↓ +0.0567↑  → baseline
  XRP      +0.3278   +0.3134   +0.3345 -0.0144↓ +0.0211↑  → slim+TDA
  ──────────────────────────────────────────────────────────────
  mean     +0.2875   +0.2803   +0.2976 -0.0071 +0.0173
  slim_IT лучше baseline: 4/9  |  slim+TDA лучше IT: 7/9

  Горизонт h=2d  (logv

In [73]:
# import warnings; warnings.filterwarnings("ignore")
# import numpy as np
# import pandas as pd
# from sklearn.linear_model import Ridge, RidgeCV
# from sklearn.preprocessing import StandardScaler
# from sklearn.pipeline import Pipeline
# from scipy import stats as sp_stats
# import matplotlib
# matplotlib.use("Agg")
# import matplotlib.pyplot as plt

# SPLIT_DATE = "2025-07-01"
# PURGE_BARS = 96
# HORIZONS   = {48:"1d", 96:"2d", 192:"4d", 288:"6d"}
# ASSETS     = ["BTC","ETH","SOL","BNB","XRP","DOGE","ADA","AVAX","TRX"]
# VOL_WINDOW = 48
# ANN        = np.sqrt(365 * 48)

# IT_SLIM = ["coinfo3_std","kl_BNB","coinfo4_std","pairwise_mi_mean",
#            "coinfo4_mean","pairwise_mi_std","coinfo3_mean",
#            "coinfo4_max","kl_AVAX","pairwise_mi_min"]

# TDA_BEST = ["BI_tp_H0","BI_ent_H1","BI_L1_H0",
#             "BII_tp_H0","BII_ent_H0","BC_tp_H0"]

# def _ridge():
#     return Pipeline([("sc", StandardScaler()),
#                      ("m", RidgeCV(alphas=[0.01,0.1,1,10,100]))])

# def oos_r2(y_true, y_pred, naive_val):
#     mse_m = np.mean((y_true - y_pred)**2)
#     mse_n = np.mean((y_true - naive_val)**2)
#     return float(1.0 - mse_m / mse_n) if mse_n > 1e-12 else np.nan

# def res_corr(res, pred_it):
#     return float(pd.Series(pred_it).corr(pd.Series(res)))

# print("Строим X_tda...")
# tda_frames = [tda_A.add_prefix("A_")]
# for mode in ["I","II","C"]:
#     d = tda_B[mode]
#     df_m = pd.DataFrame({
#         f"B{mode}_tp_H0":  d["tp_H0"],  f"B{mode}_ent_H0": d["ent_H0"],
#         f"B{mode}_tp_H1":  d["tp_H1"],  f"B{mode}_ent_H1": d["ent_H1"],
#         f"B{mode}_L1_H0":  d["L1_H0"],  f"B{mode}_L1_H1":  d["L1_H1"],
#     }, index=pd.DatetimeIndex(d["times"]))
#     tda_frames.append(df_m)

# X_tda_raw = pd.concat(tda_frames, axis=1).sort_index()
# X_tda_ri  = X_tda_raw[TDA_BEST].reindex(ret.index, method="ffill")

# TDA_BEST = [c for c in TDA_BEST if c in X_tda_ri.columns]
# print(f"TDA фичи: {TDA_BEST}")

# split_ts = pd.Timestamp(SPLIT_DATE)
# rows = []

# for asset in ASSETS:
#     if asset not in ret.columns: continue
#     r = ret[asset]
#     vol_now = r.rolling(VOL_WINDOW).std() * ANN

#     for h, h_label in HORIZONS.items():
#         vol_fwd = r.rolling(h).std().shift(-h) * ANN
#         y_raw   = np.log(vol_fwd.clip(lower=1e-8)).dropna()

#         Xi_it  = X_it_slim[IT_SLIM]
#         Xi_tda = X_tda_ri

#         idx = (X_vol.index.intersection(Xi_it.index)
#                .intersection(Xi_tda.index).intersection(y_raw.index))
#         valid = (X_vol.loc[idx].notna().all(1) &
#                  Xi_it.loc[idx].notna().all(1) &
#                  Xi_tda.loc[idx].notna().all(1) &
#                  y_raw.loc[idx].notna())
#         idx = idx[valid]
#         if len(idx) < 1000: continue

#         tr = idx < (split_ts - pd.Timedelta(minutes=30*PURGE_BARS))
#         te = idx >= split_ts
#         if tr.sum() < 480 or te.sum() < 100: continue

#         Xv_tr  = X_vol.loc[idx[tr]];  Xv_te  = X_vol.loc[idx[te]]
#         Xit_tr = Xi_it.loc[idx[tr]];  Xit_te = Xi_it.loc[idx[te]]
#         Xtd_tr = Xi_tda.loc[idx[tr]]; Xtd_te = Xi_tda.loc[idx[te]]
#         ytr    = y_raw.loc[idx[tr]];  yte    = y_raw.loc[idx[te]]
#         naive  = float(ytr.mean())

#         ck   = Xv_tr.corrwith(ytr).abs().replace([np.inf,-np.inf],np.nan).dropna()
#         top3 = ck.nlargest(3).index.tolist()
#         mb   = _ridge(); mb.fit(Xv_tr[top3], ytr)
#         pb_tr = mb.predict(Xv_tr[top3])
#         pb_te = mb.predict(Xv_te[top3])
#         res_tr = ytr.values - pb_tr
#         res_te = yte.values - pb_te

#         mit = _ridge(); mit.fit(Xit_tr, res_tr)
#         pit_te = mit.predict(Xit_te)
#         pred_it = pb_te + pit_te

#         mtda = _ridge(); mtda.fit(Xtd_tr, res_tr)
#         ptda_te = mtda.predict(Xtd_te)
#         pred_tda = pb_te + ptda_te

#         Xc_tr = pd.concat([Xit_tr, Xtd_tr], axis=1)
#         Xc_te = pd.concat([Xit_te, Xtd_te], axis=1)
#         mct = _ridge(); mct.fit(Xc_tr, res_tr)
#         pct_te = mct.predict(Xc_te)
#         pred_itd = pb_te + pct_te

#         r2_b   = oos_r2(yte.values, pb_te,    naive)
#         r2_it  = oos_r2(yte.values, pred_it,  naive)
#         r2_tda = oos_r2(yte.values, pred_tda, naive)
#         r2_itd = oos_r2(yte.values, pred_itd, naive)

#         rc_it  = res_corr(res_te, pit_te)
#         rc_tda = res_corr(res_te, ptda_te)
#         rc_itd = res_corr(res_te, pct_te)

#         mse_b   = np.mean((yte.values - pb_te)**2)
#         mse_it  = np.mean((yte.values - pred_it)**2)
#         mse_tda = np.mean((yte.values - pred_tda)**2)
#         mse_itd = np.mean((yte.values - pred_itd)**2)

#         def dm(d):
#             n = len(d); m = d.mean(); s = d.std(ddof=1)
#             t = m/(s/np.sqrt(n)) if s > 1e-10 else 0
#             return float(t), float(sp_stats.t.sf(t, df=n-1))

#         d_it  = (yte.values-pb_te)**2 - (yte.values-pred_it)**2
#         d_tda = (yte.values-pb_te)**2 - (yte.values-pred_tda)**2
#         d_itd = (yte.values-pb_te)**2 - (yte.values-pred_itd)**2
#         t_it,  p_it  = dm(d_it)
#         t_tda, p_tda = dm(d_tda)
#         t_itd, p_itd = dm(d_itd)

#         def sig(p): return "***" if p<0.01 else ("**" if p<0.05 else ("*" if p<0.10 else ""))

#         rows.append({
#             "asset": asset, "horizon": h_label, "h": h,
#             "r2_base": round(r2_b,   4),
#             "r2_it":   round(r2_it,  4),
#             "r2_tda":  round(r2_tda, 4),
#             "r2_itd":  round(r2_itd, 4),
#             "dR2_it":  round(r2_it  - r2_b, 4),
#             "dR2_tda": round(r2_tda - r2_b, 4),
#             "dR2_itd": round(r2_itd - r2_b, 4),
#             "dR2_tda_vs_it": round(r2_itd - r2_it, 4),
#             "rc_it":   round(rc_it,  4),
#             "rc_tda":  round(rc_tda, 4),
#             "rc_itd":  round(rc_itd, 4),
#             "mse_r_it":  round(mse_it  / mse_b, 4) if mse_b > 0 else np.nan,
#             "mse_r_tda": round(mse_tda / mse_b, 4) if mse_b > 0 else np.nan,
#             "mse_r_itd": round(mse_itd / mse_b, 4) if mse_b > 0 else np.nan,
#             "p_it":  round(p_it,  4), "sig_it":  sig(p_it),
#             "p_tda": round(p_tda, 4), "sig_tda": sig(p_tda),
#             "p_itd": round(p_itd, 4), "sig_itd": sig(p_itd),
#             "n_train": int(tr.sum()), "n_test": int(te.sum()),
#         })

# df = pd.DataFrame(rows)
# print(f"\nВсего строк: {len(df)}")

# print("\n" + "="*90)
# print("  ФИНАЛЬНОЕ СРАВНЕНИЕ — целевая переменная: logvol = log(realized_vol_fwd_h)")
# print("  R² = 1 - MSE(модель) / MSE(naive=mean_train_logvol)")
# print("  A=Baseline(vol)  B=+IT  C=+TDA  D=+IT+TDA")
# print("="*90)

# print(f"\n  {'актив':<6} {'h':<4} "
#       f"{'R²_A(base)':>11} {'R²_B(+IT)':>11} {'R²_C(+TDA)':>12} {'R²_D(+IT+TDA)':>14} "
#       f"{'ΔB':>7} {'ΔC':>7} {'ΔD':>7}  лучшая")
# print("  " + "-"*95)

# prev = None
# for _, row in df.sort_values(["asset","h"]).iterrows():
#     if prev and prev != row["asset"]: print("  " + "·"*95)
#     prev = row["asset"]
#     vals = {"A":row["r2_base"],"B":row["r2_it"],"C":row["r2_tda"],"D":row["r2_itd"]}
#     best = max(vals, key=lambda k: vals[k])
#     bm = "↑" if row["dR2_it"]>0.005 else ("↓" if row["dR2_it"]<-0.005 else "~")
#     cm = "↑" if row["dR2_tda"]>0.005 else ("↓" if row["dR2_tda"]<-0.005 else "~")
#     dm = "↑" if row["dR2_itd"]>0.005 else ("↓" if row["dR2_itd"]<-0.005 else "~")
#     print(f"  {row['asset']:<6} {row['horizon']:<4}"
#           f" {row['r2_base']:>+11.4f}"
#           f" {row['r2_it']:>+11.4f}"
#           f" {row['r2_tda']:>+12.4f}"
#           f" {row['r2_itd']:>+14.4f}"
#           f" {row['dR2_it']:>+6.4f}{bm}"
#           f" {row['dR2_tda']:>+6.4f}{cm}"
#           f" {row['dR2_itd']:>+6.4f}{dm}"
#           f"  {best}")

# print(f"\n{'='*70}")
# print(f"  СВОДКА")
# print(f"{'='*70}")

# n = len(df)
# print(f"""
#   Конфиг  Описание              ΔR²(mean)  ΔR²>0   rc(mean)  MSE<1
#   ─────────────────────────────────────────────────────────────────
#   B (+IT)  Baseline + IT        {df['dR2_it'].mean():>+8.4f}   {(df['dR2_it']>0).sum():>2}/{n}  {df['rc_it'].mean():>+7.4f}   {(df['mse_r_it']<1).sum():>2}/{n}
#   C (+TDA) Baseline + TDA       {df['dR2_tda'].mean():>+8.4f}   {(df['dR2_tda']>0).sum():>2}/{n}  {df['rc_tda'].mean():>+7.4f}   {(df['mse_r_tda']<1).sum():>2}/{n}
#   D (+ALL) Baseline + IT + TDA  {df['dR2_itd'].mean():>+8.4f}   {(df['dR2_itd']>0).sum():>2}/{n}  {df['rc_itd'].mean():>+7.4f}   {(df['mse_r_itd']<1).sum():>2}/{n}
# """)

# print(f"  Топ-5 по ΔR² (конфиг D = +IT+TDA):")
# for _, row in df.nlargest(5,"dR2_itd").iterrows():
#     print(f"    {row['asset']} h={row['horizon']}: "
#           f"R²_A={row['r2_base']:+.4f} → "
#           f"R²_B={row['r2_it']:+.4f} → "
#           f"R²_C={row['r2_tda']:+.4f} → "
#           f"R²_D={row['r2_itd']:+.4f} "
#           f"(ΔD={row['dR2_itd']:+.4f}, p={row['p_itd']:.4f}{row['sig_itd']})")

# print(f"\n  По активам — среднее R² всех конфигураций:")
# print(f"  {'актив':<6} {'R²_A(base)':>11} {'R²_B(+IT)':>11} "
#       f"{'R²_C(+TDA)':>12} {'R²_D(+ALL)':>12}  лучшая")
# print("  " + "-"*60)
# for asset in df.groupby("asset")["r2_itd"].mean().sort_values(ascending=False).index:
#     g = df[df.asset == asset]
#     ra = g["r2_base"].mean(); rb = g["r2_it"].mean()
#     rc_ = g["r2_tda"].mean(); rd = g["r2_itd"].mean()
#     best = max({"A":ra,"B":rb,"C":rc_,"D":rd}, key=lambda k: {"A":ra,"B":rb,"C":rc_,"D":rd}[k])
#     print(f"  {asset:<6} {ra:>+11.4f} {rb:>+11.4f} {rc_:>+12.4f} {rd:>+12.4f}  → {best}")

# print("\nСтроим финальные графики...")

# fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# asset_order = df.groupby("asset")["r2_itd"].mean().sort_values(ascending=False).index.tolist()
# h_order = ["1d","2d","4d","6d"]
# w = 0.2; x = np.arange(len(asset_order))

# ax = axes[0,0]
# for ci, (col, label, color) in enumerate([
#     ("r2_base","A: Baseline","#9E9E9E"),
#     ("r2_it",  "B: +IT",    "#2196F3"),
#     ("r2_tda", "C: +TDA",   "#FF9800"),
#     ("r2_itd", "D: +IT+TDA","#4CAF50"),
# ]):
#     vals = [df[df.asset==a][col].mean() for a in asset_order]
#     ax.bar(x+(ci-1.5)*w, vals, w*0.9, label=label, color=color, alpha=0.85)
# ax.axhline(0, color="black", lw=0.8)
# ax.set_xticks(x); ax.set_xticklabels(asset_order, fontsize=9)
# ax.set_ylabel("OOS R² (vs naive mean logvol)")
# ax.set_title("OOS R² по активам — 4 конфигурации\n(logvol, все горизонты)", fontsize=10)
# ax.legend(fontsize=8); ax.grid(alpha=0.2, axis="y")
# ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

# ax = axes[0,1]
# for col, label, color, ls in [
#     ("r2_base","A: Baseline","#9E9E9E","-"),
#     ("r2_it",  "B: +IT",    "#2196F3","-"),
#     ("r2_tda", "C: +TDA",   "#FF9800","--"),
#     ("r2_itd", "D: +IT+TDA","#4CAF50","-"),
# ]:
#     vals = [df[df.horizon==h][col].mean() for h in h_order]
#     ax.plot(h_order, vals, "o"+ls, color=color, lw=2, label=label, ms=7)
# ax.axhline(0, color="black", lw=0.5, ls=":")
# ax.set_ylabel("OOS R²"); ax.set_xlabel("Горизонт предсказания")
# ax.set_title("OOS R² по горизонтам\n(среднее по 9 активам)", fontsize=10)
# ax.legend(fontsize=8); ax.grid(alpha=0.2)
# ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

# ax = axes[1,0]
# x2 = np.arange(len(asset_order)); w2 = 0.28
# for ci, (col, label, color) in enumerate([
#     ("dR2_it",  "B: +IT",    "#2196F3"),
#     ("dR2_tda", "C: +TDA",   "#FF9800"),
#     ("dR2_itd", "D: +IT+TDA","#4CAF50"),
# ]):
#     vals = [df[df.asset==a][col].mean() for a in asset_order]
#     ax.bar(x2+(ci-1)*w2, vals, w2*0.9, label=label, color=color, alpha=0.85)
# ax.axhline(0, color="black", lw=0.8)
# ax.set_xticks(x2); ax.set_xticklabels(asset_order, fontsize=9)
# ax.set_ylabel("ΔR² относительно baseline A")
# ax.set_title("Инкрементальный ΔR² над baseline\n(logvol, все горизонты)", fontsize=10)
# ax.legend(fontsize=8); ax.grid(alpha=0.2, axis="y")
# ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

# ax = axes[1,1]
# pivot = df.pivot_table(index="asset", columns="horizon",
#                        values="mse_r_itd", aggfunc="mean")
# pivot = pivot.reindex(asset_order)[h_order]
# im = ax.imshow(pivot.values, cmap="RdYlGn_r", vmin=0.85, vmax=1.15, aspect="auto")
# ax.set_xticks(range(len(h_order)));    ax.set_xticklabels(h_order)
# ax.set_yticks(range(len(asset_order))); ax.set_yticklabels(asset_order)
# for i in range(len(asset_order)):
#     for j in range(len(h_order)):
#         val = pivot.values[i,j]
#         if np.isfinite(val):
#             col = "white" if abs(val-1)>0.07 else "black"
#             mark = "✓" if val < 1.0 else ""
#             ax.text(j, i, f"{val:.3f}{mark}", ha="center", va="center",
#                     fontsize=8, color=col)
# plt.colorbar(im, ax=ax, label="MSE_ratio (D: +IT+TDA / baseline)\n<1.0 = лучше ✓")
# ax.set_title("MSE_ratio конфиг D (+IT+TDA)\nзелёный <1 = лучше baseline", fontsize=10)

# plt.suptitle(
#     "Финальное сравнение: A=Baseline / B=+IT / C=+TDA / D=+IT+TDA\n"
#     "Цель: logvol = log(realized_vol_fwd)  |  holdout 2025-01-01 →",
#     fontsize=12, y=1.01)
# plt.tight_layout()
# plt.savefig("fig_final_comparison.png", bbox_inches="tight", dpi=150)
# plt.savefig("fig_final_comparison.pdf", bbox_inches="tight", dpi=150)
# plt.show()
# print("  Saved: fig_final_comparison.png / .pdf")

# final_comparison = df.copy()
# print(f"\nfinal_comparison сохранён в памяти ({len(df)} строк)")

Строим X_tda...
TDA фичи: ['BI_tp_H0', 'BI_ent_H1', 'BI_L1_H0', 'BII_tp_H0', 'BII_ent_H0', 'BC_tp_H0']

Всего строк: 36

  ФИНАЛЬНОЕ СРАВНЕНИЕ — целевая переменная: logvol = log(realized_vol_fwd_h)
  R² = 1 - MSE(модель) / MSE(naive=mean_train_logvol)
  A=Baseline(vol)  B=+IT  C=+TDA  D=+IT+TDA

  актив  h     R²_A(base)   R²_B(+IT)   R²_C(+TDA)  R²_D(+IT+TDA)      ΔB      ΔC      ΔD  лучшая
  -----------------------------------------------------------------------------------------------
  ADA    1d       +0.2670     +0.2523      +0.3067        +0.2982 -0.0147↓ +0.0397↑ +0.0312↑  C
  ADA    2d       +0.2413     +0.2165      +0.2940        +0.2668 -0.0248↓ +0.0527↑ +0.0255↑  C
  ADA    4d       +0.1948     +0.1208      +0.2548        +0.1489 -0.0739↓ +0.0600↑ -0.0459↓  C
  ADA    6d       +0.1330     +0.0360      +0.1957        +0.0616 -0.0970↓ +0.0627↑ -0.0714↓  C
  ·······························································································
  AVAX   1d       +0.1260

# ВОТ ОНО

In [110]:
# import warnings; warnings.filterwarnings("ignore")
# import numpy as np
# import pandas as pd
# from sklearn.linear_model import Ridge, RidgeCV
# from sklearn.preprocessing import StandardScaler
# from sklearn.pipeline import Pipeline
# from scipy import stats as sp_stats
# import matplotlib
# matplotlib.use("Agg")
# import matplotlib.pyplot as plt

# SPLIT_DATE = "2025-07-01"
# PURGE_BARS = 96
# HORIZONS   = {48:"1d", 96:"2d", 192:"4d", 288:"6d", 336:"7d", 672:"14d"}
# ASSETS     = ["BTC","ETH","SOL","BNB","XRP","DOGE","ADA","AVAX"]
# VOL_WINDOW = 48
# ANN        = np.sqrt(365 * 48)

# IT_SLIM = ["coinfo3_std","kl_BNB","coinfo4_std","pairwise_mi_mean",
#            "coinfo4_mean","pairwise_mi_std","coinfo3_mean",
#            "coinfo4_max","kl_AVAX","pairwise_mi_min"]

# TDA_BEST = ["BI_tp_H0","BI_ent_H1","BI_L1_H0",
#             "BII_tp_H0","BII_ent_H0","BC_tp_H0"]

# def _ridge():
#     return Pipeline([("sc", StandardScaler()),
#                      ("m", RidgeCV(alphas=[0.01,0.1,1,10,100]))])

# def oos_r2(y_true, y_pred, naive_val):
#     mse_m = np.mean((y_true - y_pred)**2)
#     mse_n = np.mean((y_true - naive_val)**2)
#     return float(1.0 - mse_m / mse_n) if mse_n > 1e-12 else np.nan

# def res_corr(res, pred_it):
#     return float(pd.Series(pred_it).corr(pd.Series(res)))

# print("Строим X_tda...")
# tda_frames = [tda_A.add_prefix("A_")]
# for mode in ["I","II","C"]:
#     d = tda_B[mode]
#     df_m = pd.DataFrame({
#         f"B{mode}_tp_H0":  d["tp_H0"],  f"B{mode}_ent_H0": d["ent_H0"],
#         f"B{mode}_tp_H1":  d["tp_H1"],  f"B{mode}_ent_H1": d["ent_H1"],
#         f"B{mode}_L1_H0":  d["L1_H0"],  f"B{mode}_L1_H1":  d["L1_H1"],
#     }, index=pd.DatetimeIndex(d["times"]))
#     tda_frames.append(df_m)

# X_tda_raw = pd.concat(tda_frames, axis=1).sort_index()
# X_tda_ri  = X_tda_raw[TDA_BEST].reindex(ret.index, method="ffill")

# TDA_BEST = [c for c in TDA_BEST if c in X_tda_ri.columns]
# print(f"TDA фичи: {TDA_BEST}")

# split_ts = pd.Timestamp(SPLIT_DATE)
# rows = []

# for asset in ASSETS:
#     if asset not in ret.columns: continue
#     r = ret[asset]
#     vol_now = r.rolling(VOL_WINDOW).std() * ANN

#     for h, h_label in HORIZONS.items():
#         vol_fwd = r.rolling(h).std().shift(-h) * ANN
#         y_raw   = np.log(vol_fwd.clip(lower=1e-8)).dropna()

#         Xi_it  = X_it_slim[IT_SLIM]
#         Xi_tda = X_tda_ri

#         idx = (X_vol.index.intersection(Xi_it.index)
#                .intersection(Xi_tda.index).intersection(y_raw.index))
#         valid = (X_vol.loc[idx].notna().all(1) &
#                  Xi_it.loc[idx].notna().all(1) &
#                  Xi_tda.loc[idx].notna().all(1) &
#                  y_raw.loc[idx].notna())
#         idx = idx[valid]
#         if len(idx) < 1000: continue

#         tr = idx < (split_ts - pd.Timedelta(minutes=30*PURGE_BARS))
#         te = idx >= split_ts
#         if tr.sum() < 480 or te.sum() < 100: continue

#         Xv_tr  = X_vol.loc[idx[tr]];  Xv_te  = X_vol.loc[idx[te]]
#         Xit_tr = Xi_it.loc[idx[tr]];  Xit_te = Xi_it.loc[idx[te]]
#         Xtd_tr = Xi_tda.loc[idx[tr]]; Xtd_te = Xi_tda.loc[idx[te]]
#         ytr    = y_raw.loc[idx[tr]];  yte    = y_raw.loc[idx[te]]
#         naive  = float(ytr.mean())

#         ck   = Xv_tr.corrwith(ytr).abs().replace([np.inf,-np.inf],np.nan).dropna()
#         top3 = ck.nlargest(3).index.tolist()
#         mb   = _ridge(); mb.fit(Xv_tr[top3], ytr)
#         pb_tr = mb.predict(Xv_tr[top3])
#         pb_te = mb.predict(Xv_te[top3])
#         res_tr = ytr.values - pb_tr
#         res_te = yte.values - pb_te

#         mit = _ridge(); mit.fit(Xit_tr, res_tr)
#         pit_te = mit.predict(Xit_te)
#         pred_it = pb_te + pit_te

#         mtda = _ridge(); mtda.fit(Xtd_tr, res_tr)
#         ptda_te = mtda.predict(Xtd_te)
#         pred_tda = pb_te + ptda_te

#         Xc_tr = pd.concat([Xit_tr, Xtd_tr], axis=1)
#         Xc_te = pd.concat([Xit_te, Xtd_te], axis=1)
#         mct = _ridge(); mct.fit(Xc_tr, res_tr)
#         pct_te = mct.predict(Xc_te)
#         pred_itd = pb_te + pct_te

#         r2_b   = oos_r2(yte.values, pb_te,    naive)
#         r2_it  = oos_r2(yte.values, pred_it,  naive)
#         r2_tda = oos_r2(yte.values, pred_tda, naive)
#         r2_itd = oos_r2(yte.values, pred_itd, naive)

#         rc_it  = res_corr(res_te, pit_te)
#         rc_tda = res_corr(res_te, ptda_te)
#         rc_itd = res_corr(res_te, pct_te)

#         mse_b   = np.mean((yte.values - pb_te)**2)
#         mse_it  = np.mean((yte.values - pred_it)**2)
#         mse_tda = np.mean((yte.values - pred_tda)**2)
#         mse_itd = np.mean((yte.values - pred_itd)**2)

#         def dm(d):
#             n = len(d); m = d.mean(); s = d.std(ddof=1)
#             t = m/(s/np.sqrt(n)) if s > 1e-10 else 0
#             return float(t), float(sp_stats.t.sf(t, df=n-1))

#         d_it  = (yte.values-pb_te)**2 - (yte.values-pred_it)**2
#         d_tda = (yte.values-pb_te)**2 - (yte.values-pred_tda)**2
#         d_itd = (yte.values-pb_te)**2 - (yte.values-pred_itd)**2
#         t_it,  p_it  = dm(d_it)
#         t_tda, p_tda = dm(d_tda)
#         t_itd, p_itd = dm(d_itd)

#         def sig(p): return "***" if p<0.01 else ("**" if p<0.05 else ("*" if p<0.10 else ""))

#         rows.append({
#             "asset": asset, "horizon": h_label, "h": h,
#             "r2_base": round(r2_b,   4),
#             "r2_it":   round(r2_it,  4),
#             "r2_tda":  round(r2_tda, 4),
#             "r2_itd":  round(r2_itd, 4),
#             "dR2_it":  round(r2_it  - r2_b, 4),
#             "dR2_tda": round(r2_tda - r2_b, 4),
#             "dR2_itd": round(r2_itd - r2_b, 4),
#             "dR2_tda_vs_it": round(r2_itd - r2_it, 4),
#             "rc_it":   round(rc_it,  4),
#             "rc_tda":  round(rc_tda, 4),
#             "rc_itd":  round(rc_itd, 4),
#             "mse_r_it":  round(mse_it  / mse_b, 4) if mse_b > 0 else np.nan,
#             "mse_r_tda": round(mse_tda / mse_b, 4) if mse_b > 0 else np.nan,
#             "mse_r_itd": round(mse_itd / mse_b, 4) if mse_b > 0 else np.nan,
#             "p_it":  round(p_it,  4), "sig_it":  sig(p_it),
#             "p_tda": round(p_tda, 4), "sig_tda": sig(p_tda),
#             "p_itd": round(p_itd, 4), "sig_itd": sig(p_itd),
#             "n_train": int(tr.sum()), "n_test": int(te.sum()),
#         })

# df = pd.DataFrame(rows)
# print(f"\nВсего строк: {len(df)}")

# print("\n" + "="*90)
# print("  ФИНАЛЬНОЕ СРАВНЕНИЕ — целевая переменная: logvol = log(realized_vol_fwd_h)")
# print("  R² = 1 - MSE(модель) / MSE(naive=mean_train_logvol)")
# print("  A=Baseline(vol)  B=+IT  C=+TDA  D=+IT+TDA")
# print("="*90)

# print(f"\n  {'актив':<6} {'h':<4} "
#       f"{'R²_A(base)':>11} {'R²_B(+IT)':>11} {'R²_C(+TDA)':>12} {'R²_D(+IT+TDA)':>14} "
#       f"{'ΔB':>7} {'ΔC':>7} {'ΔD':>7}  лучшая")
# print("  " + "-"*95)

# prev = None
# for _, row in df.sort_values(["asset","h"]).iterrows():
#     if prev and prev != row["asset"]: print("  " + "·"*95)
#     prev = row["asset"]
#     vals = {"A":row["r2_base"],"B":row["r2_it"],"C":row["r2_tda"],"D":row["r2_itd"]}
#     best = max(vals, key=lambda k: vals[k])
#     bm = "↑" if row["dR2_it"]>0.005 else ("↓" if row["dR2_it"]<-0.005 else "~")
#     cm = "↑" if row["dR2_tda"]>0.005 else ("↓" if row["dR2_tda"]<-0.005 else "~")
#     dm = "↑" if row["dR2_itd"]>0.005 else ("↓" if row["dR2_itd"]<-0.005 else "~")
#     print(f"  {row['asset']:<6} {row['horizon']:<4}"
#           f" {row['r2_base']:>+11.4f}"
#           f" {row['r2_it']:>+11.4f}"
#           f" {row['r2_tda']:>+12.4f}"
#           f" {row['r2_itd']:>+14.4f}"
#           f" {row['dR2_it']:>+6.4f}{bm}"
#           f" {row['dR2_tda']:>+6.4f}{cm}"
#           f" {row['dR2_itd']:>+6.4f}{dm}"
#           f"  {best}")

# print(f"\n{'='*70}")
# print(f"  СВОДКА")
# print(f"{'='*70}")

# n = len(df)
# print(f"""
#   Конфиг  Описание              ΔR²(mean)  ΔR²>0   rc(mean)  MSE<1
#   ─────────────────────────────────────────────────────────────────
#   B (+IT)  Baseline + IT        {df['dR2_it'].mean():>+8.4f}   {(df['dR2_it']>0).sum():>2}/{n}  {df['rc_it'].mean():>+7.4f}   {(df['mse_r_it']<1).sum():>2}/{n}
#   C (+TDA) Baseline + TDA       {df['dR2_tda'].mean():>+8.4f}   {(df['dR2_tda']>0).sum():>2}/{n}  {df['rc_tda'].mean():>+7.4f}   {(df['mse_r_tda']<1).sum():>2}/{n}
#   D (+ALL) Baseline + IT + TDA  {df['dR2_itd'].mean():>+8.4f}   {(df['dR2_itd']>0).sum():>2}/{n}  {df['rc_itd'].mean():>+7.4f}   {(df['mse_r_itd']<1).sum():>2}/{n}
# """)

# print(f"  Топ-5 по ΔR² (конфиг D = +IT+TDA):")
# for _, row in df.nlargest(5,"dR2_itd").iterrows():
#     print(f"    {row['asset']} h={row['horizon']}: "
#           f"R²_A={row['r2_base']:+.4f} → "
#           f"R²_B={row['r2_it']:+.4f} → "
#           f"R²_C={row['r2_tda']:+.4f} → "
#           f"R²_D={row['r2_itd']:+.4f} "
#           f"(ΔD={row['dR2_itd']:+.4f}, p={row['p_itd']:.4f}{row['sig_itd']})")

# print(f"\n  По активам — среднее R² всех конфигураций:")
# print(f"  {'актив':<6} {'R²_A(base)':>11} {'R²_B(+IT)':>11} "
#       f"{'R²_C(+TDA)':>12} {'R²_D(+ALL)':>12}  лучшая")
# print("  " + "-"*60)
# for asset in df.groupby("asset")["r2_itd"].mean().sort_values(ascending=False).index:
#     g = df[df.asset == asset]
#     ra = g["r2_base"].mean(); rb = g["r2_it"].mean()
#     rc_ = g["r2_tda"].mean(); rd = g["r2_itd"].mean()
#     best = max({"A":ra,"B":rb,"C":rc_,"D":rd}, key=lambda k: {"A":ra,"B":rb,"C":rc_,"D":rd}[k])
#     print(f"  {asset:<6} {ra:>+11.4f} {rb:>+11.4f} {rc_:>+12.4f} {rd:>+12.4f}  → {best}")

# print("\nСтроим финальные графики...")

# fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# asset_order = df.groupby("asset")["r2_itd"].mean().sort_values(ascending=False).index.tolist()
# h_order = ["1d","2d","4d","6d","7d","14d"]
# w = 0.2; x = np.arange(len(asset_order))

# ax = axes[0,0]
# for ci, (col, label, color) in enumerate([
#     ("r2_base","A: Baseline","#9E9E9E"),
#     ("r2_it",  "B: +IT",    "#2196F3"),
#     ("r2_tda", "C: +TDA",   "#FF9800"),
#     ("r2_itd", "D: +IT+TDA","#4CAF50"),
# ]):
#     vals = [df[df.asset==a][col].mean() for a in asset_order]
#     ax.bar(x+(ci-1.5)*w, vals, w*0.9, label=label, color=color, alpha=0.85)
# ax.axhline(0, color="black", lw=0.8)
# ax.set_xticks(x); ax.set_xticklabels(asset_order, fontsize=9)
# ax.set_ylabel("OOS R² (vs naive mean logvol)")
# ax.set_title("OOS R² по активам — 4 конфигурации\n(logvol, все горизонты)", fontsize=10)
# ax.legend(fontsize=8); ax.grid(alpha=0.2, axis="y")
# ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

# ax = axes[0,1]
# for col, label, color, ls in [
#     ("r2_base","A: Baseline","#9E9E9E","-"),
#     ("r2_it",  "B: +IT",    "#2196F3","-"),
#     ("r2_tda", "C: +TDA",   "#FF9800","--"),
#     ("r2_itd", "D: +IT+TDA","#4CAF50","-"),
# ]:
#     vals = [df[df.horizon==h][col].mean() for h in h_order]
#     ax.plot(h_order, vals, "o"+ls, color=color, lw=2, label=label, ms=7)
# ax.axhline(0, color="black", lw=0.5, ls=":")
# ax.set_ylabel("OOS R²"); ax.set_xlabel("Горизонт предсказания")
# ax.set_title("OOS R² по горизонтам\n(среднее по 9 активам)", fontsize=10)
# ax.legend(fontsize=8); ax.grid(alpha=0.2)
# ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

# ax = axes[1,0]
# x2 = np.arange(len(asset_order)); w2 = 0.28
# for ci, (col, label, color) in enumerate([
#     ("dR2_it",  "B: +IT",    "#2196F3"),
#     ("dR2_tda", "C: +TDA",   "#FF9800"),
#     ("dR2_itd", "D: +IT+TDA","#4CAF50"),
# ]):
#     vals = [df[df.asset==a][col].mean() for a in asset_order]
#     ax.bar(x2+(ci-1)*w2, vals, w2*0.9, label=label, color=color, alpha=0.85)
# ax.axhline(0, color="black", lw=0.8)
# ax.set_xticks(x2); ax.set_xticklabels(asset_order, fontsize=9)
# ax.set_ylabel("ΔR² относительно baseline A")
# ax.set_title("Инкрементальный ΔR² над baseline\n(logvol, все горизонты)", fontsize=10)
# ax.legend(fontsize=8); ax.grid(alpha=0.2, axis="y")
# ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

# ax = axes[1,1]
# pivot = df.pivot_table(index="asset", columns="horizon",
#                        values="mse_r_itd", aggfunc="mean")
# pivot = pivot.reindex(asset_order)[h_order]
# im = ax.imshow(pivot.values, cmap="RdYlGn_r", vmin=0.85, vmax=1.15, aspect="auto")
# ax.set_xticks(range(len(h_order)));    ax.set_xticklabels(h_order)
# ax.set_yticks(range(len(asset_order))); ax.set_yticklabels(asset_order)
# for i in range(len(asset_order)):
#     for j in range(len(h_order)):
#         val = pivot.values[i,j]
#         if np.isfinite(val):
#             col = "white" if abs(val-1)>0.07 else "black"
#             mark = "✓" if val < 1.0 else ""
#             ax.text(j, i, f"{val:.3f}{mark}", ha="center", va="center",
#                     fontsize=8, color=col)
# plt.colorbar(im, ax=ax, label="MSE_ratio (D: +IT+TDA / baseline)\n<1.0 = лучше ✓")
# ax.set_title("MSE_ratio конфиг D (+IT+TDA)\nзелёный <1 = лучше baseline", fontsize=10)

# plt.suptitle(
#     "Финальное сравнение: A=Baseline / B=+IT / C=+TDA / D=+IT+TDA\n"
#     "Цель: logvol = log(realized_vol_fwd)  |  holdout 2025-01-01 →",
#     fontsize=12, y=1.01)
# plt.tight_layout()
# plt.savefig("fig_final_comparison.png", bbox_inches="tight", dpi=150)
# plt.savefig("fig_final_comparison.pdf", bbox_inches="tight", dpi=150)
# plt.show()
# print("  Saved: fig_final_comparison.png / .pdf")

# final_comparison = df.copy()
# print(f"\nfinal_comparison сохранён в памяти ({len(df)} строк)")

Строим X_tda...
TDA фичи: ['BI_tp_H0', 'BI_ent_H1', 'BI_L1_H0', 'BII_tp_H0', 'BII_ent_H0', 'BC_tp_H0']

Всего строк: 48

  ФИНАЛЬНОЕ СРАВНЕНИЕ — целевая переменная: logvol = log(realized_vol_fwd_h)
  R² = 1 - MSE(модель) / MSE(naive=mean_train_logvol)
  A=Baseline(vol)  B=+IT  C=+TDA  D=+IT+TDA

  актив  h     R²_A(base)   R²_B(+IT)   R²_C(+TDA)  R²_D(+IT+TDA)      ΔB      ΔC      ΔD  лучшая
  -----------------------------------------------------------------------------------------------
  ADA    1d       +0.2670     +0.2523      +0.3067        +0.2982 -0.0147↓ +0.0397↑ +0.0312↑  C
  ADA    2d       +0.2413     +0.2165      +0.2940        +0.2668 -0.0248↓ +0.0527↑ +0.0255↑  C
  ADA    4d       +0.1948     +0.1208      +0.2548        +0.1489 -0.0739↓ +0.0600↑ -0.0459↓  C
  ADA    6d       +0.1330     +0.0360      +0.1957        +0.0616 -0.0970↓ +0.0627↑ -0.0714↓  C
  ADA    7d       +0.1210     +0.0048      +0.1891        +0.0320 -0.1162↓ +0.0681↑ -0.0890↓  C
  ADA    14d      -0.0990  

In [113]:
# print("\nBuilding final plots...")

# fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# asset_order = df.groupby("asset")["r2_itd"].mean().sort_values(ascending=False).index.tolist()
# h_order = ["1d","2d","4d","6d","7d","14d"]
# w = 0.2
# x = np.arange(len(asset_order))

# ax = axes[0, 0]
# for ci, (col, label, color) in enumerate([
#     ("r2_base", "A: Baseline", "#9E9E9E"),
#     ("r2_it",   "B: +IT",      "#2196F3"),
#     ("r2_tda",  "C: +TDA",     "#FF9800"),
#     ("r2_itd",  "D: +IT+TDA",  "#4CAF50"),
# ]):
#     vals = [df[df.asset == a][col].mean() for a in asset_order]
#     ax.bar(x + (ci - 1.5) * w, vals, w * 0.9, label=label, color=color, alpha=0.85)

# ax.axhline(0, color="black", lw=0.8)
# ax.set_xticks(x)
# ax.set_xticklabels(asset_order, fontsize=9)
# ax.set_ylabel("OOS R² (vs naive mean logvol)")
# ax.set_title("OOS R² by asset — 4 configurations\n(logvol, all horizons)", fontsize=10)
# ax.legend(fontsize=8)
# ax.grid(alpha=0.2, axis="y")
# ax.spines["top"].set_visible(False)
# ax.spines["right"].set_visible(False)

# ax = axes[0, 1]
# for col, label, color, ls in [
#     ("r2_base", "A: Baseline", "#9E9E9E", "-"),
#     ("r2_it",   "B: +IT",      "#2196F3", "-"),
#     ("r2_tda",  "C: +TDA",     "#FF9800", "--"),
#     ("r2_itd",  "D: +IT+TDA",  "#4CAF50", "-"),
# ]:
#     vals = [df[df.horizon == h][col].mean() for h in h_order]
#     ax.plot(h_order, vals, "o" + ls, color=color, lw=2, label=label, ms=7)

# ax.axhline(0, color="black", lw=0.5, ls=":")
# ax.set_ylabel("OOS R²")
# ax.set_xlabel("Forecast horizon")
# ax.set_title("OOS R² by horizon\n(mean over assets)", fontsize=10)
# ax.legend(fontsize=8)
# ax.grid(alpha=0.2)
# ax.spines["top"].set_visible(False)
# ax.spines["right"].set_visible(False)

# ax = axes[1, 0]
# x2 = np.arange(len(asset_order))
# w2 = 0.28
# for ci, (col, label, color) in enumerate([
#     ("dR2_it",  "B: +IT",      "#2196F3"),
#     ("dR2_tda", "C: +TDA",     "#FF9800"),
#     ("dR2_itd", "D: +IT+TDA",  "#4CAF50"),
# ]):
#     vals = [df[df.asset == a][col].mean() for a in asset_order]
#     ax.bar(x2 + (ci - 1) * w2, vals, w2 * 0.9, label=label, color=color, alpha=0.85)

# ax.axhline(0, color="black", lw=0.8)
# ax.set_xticks(x2)
# ax.set_xticklabels(asset_order, fontsize=9)
# ax.set_ylabel("ΔR² relative to baseline A")
# ax.set_title("Incremental ΔR² over baseline\n(logvol, all horizons)", fontsize=10)
# ax.legend(fontsize=8)
# ax.grid(alpha=0.2, axis="y")
# ax.spines["top"].set_visible(False)
# ax.spines["right"].set_visible(False)

# ax = axes[1, 1]
# pivot = df.pivot_table(index="asset", columns="horizon", values="mse_r_itd", aggfunc="mean")
# pivot = pivot.reindex(asset_order)[h_order]

# im = ax.imshow(pivot.values, cmap="RdYlGn_r", vmin=0.85, vmax=1.15, aspect="auto")
# ax.set_xticks(range(len(h_order)))
# ax.set_xticklabels(h_order)
# ax.set_yticks(range(len(asset_order)))
# ax.set_yticklabels(asset_order)

# for i in range(len(asset_order)):
#     for j in range(len(h_order)):
#         val = pivot.values[i, j]
#         if np.isfinite(val):
#             txt_color = "white" if abs(val - 1) > 0.07 else "black"
#             mark = "✓" if val < 1.0 else ""
#             ax.text(j, i, f"{val:.3f}{mark}", ha="center", va="center",
#                     fontsize=8, color=txt_color)

# cbar = plt.colorbar(im, ax=ax)
# cbar.set_label("MSE ratio (D: +IT+TDA / baseline)\n< 1.0 = better ✓")

# ax.set_title("MSE ratio for configuration D (+IT+TDA)\ngreen < 1 = better than baseline", fontsize=10)

# plt.suptitle(
#     "Final comparison: A=Baseline / B=+IT / C=+TDA / D=+IT+TDA\n"
#     "Target: logvol = log(realized_vol_fwd) | holdout 2025-07-01 →",
#     fontsize=12, y=1.01
# )

# plt.tight_layout()
# plt.savefig("fig_final_comparison.png", bbox_inches="tight", dpi=150)
# plt.savefig("fig_final_comparison.pdf", bbox_inches="tight", dpi=150)
# plt.show()

# print("Saved: fig_final_comparison.png / .pdf")


Building final plots...
Saved: fig_final_comparison.png / .pdf


In [94]:
# X_it_slim["nmi_lag1_mean"] = nmi_mean.reindex(X_it_slim.index, method="ffill")
# print("nmi_lag1_mean добавлена:", "nmi_lag1_mean" in X_it_slim.columns)
# """
# Финальное сравнение: Baseline vs slim_IT vs slim+TDA
# =====================================================
# Честное сравнение трёх конфигураций:
#   A) Baseline:  Ridge_vol_k3 (только vol-фичи)
#   B) slim_IT:   Baseline + 10 IT-фич
#   C) slim+TDA:  Baseline + 10 IT + 6 TDA-фич (лучшая конфигурация)

# Метрики для каждой пары (A→B и A→C):
#   1. OOS R²  — насколько лучше наивного прогноза
#   2. ΔR²     — инкрементальный вклад IT и TDA
#   3. res_corr — корреляция с остатками baseline
#   4. MSE_ratio — отношение MSE к baseline
#   5. Hit rate — % тестовых баров где combined < baseline по MSE

# 9 активов × 4 горизонта × taргет logvol
# """

# import warnings; warnings.filterwarnings("ignore")
# import numpy as np
# import pandas as pd
# from sklearn.linear_model import Ridge, RidgeCV
# from sklearn.preprocessing import StandardScaler
# from sklearn.pipeline import Pipeline
# from scipy import stats as sp_stats
# import matplotlib
# matplotlib.use("Agg")
# import matplotlib.pyplot as plt

# SPLIT_DATE = "2025-07-01"
# PURGE_BARS = 96
# HORIZONS   = {48:"1d", 96:"2d", 192:"4d", 288:"6d", 336:"7d", 672:"14d"}
# ASSETS     = ["BTC","ETH","SOL","BNB","XRP","DOGE","ADA","AVAX","TRX"]
# VOL_WINDOW = 48
# ANN        = np.sqrt(365 * 48)

# IT_SLIM = ["coinfo3_std","kl_BNB","coinfo4_std","pairwise_mi_mean",
#            "coinfo4_mean","pairwise_mi_std","coinfo3_mean",
#            "coinfo4_max","kl_AVAX","pairwise_mi_min",
#            "nmi_lag1_mean"]   # добавлена по результатам permutation importance

# TDA_BEST = ["BI_tp_H0","BI_ent_H1","BI_L1_H0",
#             "BII_tp_H0","BII_ent_H0","BC_tp_H0"]

# def _ridge():
#     return Pipeline([("sc", StandardScaler()),
#                      ("m", RidgeCV(alphas=[0.01,0.1,1,10,100]))])

# def oos_r2(y_true, y_pred, naive_val):
#     mse_m = np.mean((y_true - y_pred)**2)
#     mse_n = np.mean((y_true - naive_val)**2)
#     return float(1.0 - mse_m / mse_n) if mse_n > 1e-12 else np.nan

# def res_corr(res, pred_it):
#     return float(pd.Series(pred_it).corr(pd.Series(res)))

# print("Строим X_tda...")
# tda_frames = [tda_A.add_prefix("A_")]
# for mode in ["I","II","C"]:
#     d = tda_B[mode]
#     df_m = pd.DataFrame({
#         f"B{mode}_tp_H0":  d["tp_H0"],  f"B{mode}_ent_H0": d["ent_H0"],
#         f"B{mode}_tp_H1":  d["tp_H1"],  f"B{mode}_ent_H1": d["ent_H1"],
#         f"B{mode}_L1_H0":  d["L1_H0"],  f"B{mode}_L1_H1":  d["L1_H1"],
#     }, index=pd.DatetimeIndex(d["times"]))
#     tda_frames.append(df_m)

# X_tda_raw = pd.concat(tda_frames, axis=1).sort_index()
# X_tda_ri  = X_tda_raw[TDA_BEST].reindex(ret.index, method="ffill")

# TDA_BEST = [c for c in TDA_BEST if c in X_tda_ri.columns]
# print(f"TDA фичи: {TDA_BEST}")

# split_ts = pd.Timestamp(SPLIT_DATE)
# rows = []

# for asset in ASSETS:
#     if asset not in ret.columns: continue
#     r = ret[asset]
#     vol_now = r.rolling(VOL_WINDOW).std() * ANN

#     for h, h_label in HORIZONS.items():
#         vol_fwd = r.rolling(h).std().shift(-h) * ANN
#         y_raw   = np.log(vol_fwd.clip(lower=1e-8)).dropna()

#         Xi_it  = X_it_slim[IT_SLIM]
#         Xi_tda = X_tda_ri

#         idx = (X_vol.index.intersection(Xi_it.index)
#                .intersection(Xi_tda.index).intersection(y_raw.index))
#         valid = (X_vol.loc[idx].notna().all(1) &
#                  Xi_it.loc[idx].notna().all(1) &
#                  Xi_tda.loc[idx].notna().all(1) &
#                  y_raw.loc[idx].notna())
#         idx = idx[valid]
#         if len(idx) < 1000: continue

#         tr = idx < (split_ts - pd.Timedelta(minutes=30*PURGE_BARS))
#         te = idx >= split_ts
#         if tr.sum() < 480 or te.sum() < 100: continue

#         Xv_tr  = X_vol.loc[idx[tr]];  Xv_te  = X_vol.loc[idx[te]]
#         Xit_tr = Xi_it.loc[idx[tr]];  Xit_te = Xi_it.loc[idx[te]]
#         Xtd_tr = Xi_tda.loc[idx[tr]]; Xtd_te = Xi_tda.loc[idx[te]]
#         ytr    = y_raw.loc[idx[tr]];  yte    = y_raw.loc[idx[te]]
#         naive  = float(ytr.mean())

#         ck   = Xv_tr.corrwith(ytr).abs().replace([np.inf,-np.inf],np.nan).dropna()
#         top3 = ck.nlargest(3).index.tolist()
#         mb   = _ridge(); mb.fit(Xv_tr[top3], ytr)
#         pb_tr = mb.predict(Xv_tr[top3])
#         pb_te = mb.predict(Xv_te[top3])
#         res_tr = ytr.values - pb_tr
#         res_te = yte.values - pb_te

#         mit = _ridge(); mit.fit(Xit_tr, res_tr)
#         pit_te = mit.predict(Xit_te)
#         pred_it = pb_te + pit_te

#         mtda = _ridge(); mtda.fit(Xtd_tr, res_tr)
#         ptda_te = mtda.predict(Xtd_te)
#         pred_tda = pb_te + ptda_te

#         Xc_tr = pd.concat([Xit_tr, Xtd_tr], axis=1)
#         Xc_te = pd.concat([Xit_te, Xtd_te], axis=1)
#         mct = _ridge(); mct.fit(Xc_tr, res_tr)
#         pct_te = mct.predict(Xc_te)
#         pred_itd = pb_te + pct_te

#         r2_b   = oos_r2(yte.values, pb_te,    naive)
#         r2_it  = oos_r2(yte.values, pred_it,  naive)
#         r2_tda = oos_r2(yte.values, pred_tda, naive)
#         r2_itd = oos_r2(yte.values, pred_itd, naive)

#         rc_it  = res_corr(res_te, pit_te)
#         rc_tda = res_corr(res_te, ptda_te)
#         rc_itd = res_corr(res_te, pct_te)

#         mse_b   = np.mean((yte.values - pb_te)**2)
#         mse_it  = np.mean((yte.values - pred_it)**2)
#         mse_tda = np.mean((yte.values - pred_tda)**2)
#         mse_itd = np.mean((yte.values - pred_itd)**2)

#         def dm(d):
#             n = len(d); m = d.mean(); s = d.std(ddof=1)
#             t = m/(s/np.sqrt(n)) if s > 1e-10 else 0
#             return float(t), float(sp_stats.t.sf(t, df=n-1))

#         d_it  = (yte.values-pb_te)**2 - (yte.values-pred_it)**2
#         d_tda = (yte.values-pb_te)**2 - (yte.values-pred_tda)**2
#         d_itd = (yte.values-pb_te)**2 - (yte.values-pred_itd)**2
#         t_it,  p_it  = dm(d_it)
#         t_tda, p_tda = dm(d_tda)
#         t_itd, p_itd = dm(d_itd)

#         def sig(p): return "***" if p<0.01 else ("**" if p<0.05 else ("*" if p<0.10 else ""))

#         rows.append({
#             "asset": asset, "horizon": h_label, "h": h,
#             "r2_base": round(r2_b,   4),
#             "r2_it":   round(r2_it,  4),
#             "r2_tda":  round(r2_tda, 4),
#             "r2_itd":  round(r2_itd, 4),
#             "dR2_it":  round(r2_it  - r2_b, 4),
#             "dR2_tda": round(r2_tda - r2_b, 4),
#             "dR2_itd": round(r2_itd - r2_b, 4),
#             "dR2_tda_vs_it": round(r2_itd - r2_it, 4),
#             "rc_it":   round(rc_it,  4),
#             "rc_tda":  round(rc_tda, 4),
#             "rc_itd":  round(rc_itd, 4),
#             "mse_r_it":  round(mse_it  / mse_b, 4) if mse_b > 0 else np.nan,
#             "mse_r_tda": round(mse_tda / mse_b, 4) if mse_b > 0 else np.nan,
#             "mse_r_itd": round(mse_itd / mse_b, 4) if mse_b > 0 else np.nan,
#             "p_it":  round(p_it,  4), "sig_it":  sig(p_it),
#             "p_tda": round(p_tda, 4), "sig_tda": sig(p_tda),
#             "p_itd": round(p_itd, 4), "sig_itd": sig(p_itd),
#             "n_train": int(tr.sum()), "n_test": int(te.sum()),
#         })

# df = pd.DataFrame(rows)
# print(f"\nВсего строк: {len(df)}")

# print("\n" + "="*90)
# print("  ФИНАЛЬНОЕ СРАВНЕНИЕ — целевая переменная: logvol = log(realized_vol_fwd_h)")
# print("  R² = 1 - MSE(модель) / MSE(naive=mean_train_logvol)")
# print("  A=Baseline(vol)  B=+IT  C=+TDA  D=+IT+TDA")
# print("="*90)

# print(f"\n  {'актив':<6} {'h':<4} "
#       f"{'R²_A(base)':>11} {'R²_B(+IT)':>11} {'R²_C(+TDA)':>12} {'R²_D(+IT+TDA)':>14} "
#       f"{'ΔB':>7} {'ΔC':>7} {'ΔD':>7}  лучшая")
# print("  " + "-"*95)

# prev = None
# for _, row in df.sort_values(["asset","h"]).iterrows():
#     if prev and prev != row["asset"]: print("  " + "·"*95)
#     prev = row["asset"]
#     vals = {"A":row["r2_base"],"B":row["r2_it"],"C":row["r2_tda"],"D":row["r2_itd"]}
#     best = max(vals, key=lambda k: vals[k])
#     bm = "↑" if row["dR2_it"]>0.005 else ("↓" if row["dR2_it"]<-0.005 else "~")
#     cm = "↑" if row["dR2_tda"]>0.005 else ("↓" if row["dR2_tda"]<-0.005 else "~")
#     dm = "↑" if row["dR2_itd"]>0.005 else ("↓" if row["dR2_itd"]<-0.005 else "~")
#     print(f"  {row['asset']:<6} {row['horizon']:<4}"
#           f" {row['r2_base']:>+11.4f}"
#           f" {row['r2_it']:>+11.4f}"
#           f" {row['r2_tda']:>+12.4f}"
#           f" {row['r2_itd']:>+14.4f}"
#           f" {row['dR2_it']:>+6.4f}{bm}"
#           f" {row['dR2_tda']:>+6.4f}{cm}"
#           f" {row['dR2_itd']:>+6.4f}{dm}"
#           f"  {best}")

# print(f"\n{'='*70}")
# print(f"  СВОДКА")
# print(f"{'='*70}")

# n = len(df)
# print(f"""
#   Конфиг  Описание              ΔR²(mean)  ΔR²>0   rc(mean)  MSE<1
#   ─────────────────────────────────────────────────────────────────
#   B (+IT)  Baseline + IT        {df['dR2_it'].mean():>+8.4f}   {(df['dR2_it']>0).sum():>2}/{n}  {df['rc_it'].mean():>+7.4f}   {(df['mse_r_it']<1).sum():>2}/{n}
#   C (+TDA) Baseline + TDA       {df['dR2_tda'].mean():>+8.4f}   {(df['dR2_tda']>0).sum():>2}/{n}  {df['rc_tda'].mean():>+7.4f}   {(df['mse_r_tda']<1).sum():>2}/{n}
#   D (+ALL) Baseline + IT + TDA  {df['dR2_itd'].mean():>+8.4f}   {(df['dR2_itd']>0).sum():>2}/{n}  {df['rc_itd'].mean():>+7.4f}   {(df['mse_r_itd']<1).sum():>2}/{n}
# """)

# print(f"  Топ-5 по ΔR² (конфиг D = +IT+TDA):")
# for _, row in df.nlargest(5,"dR2_itd").iterrows():
#     print(f"    {row['asset']} h={row['horizon']}: "
#           f"R²_A={row['r2_base']:+.4f} → "
#           f"R²_B={row['r2_it']:+.4f} → "
#           f"R²_C={row['r2_tda']:+.4f} → "
#           f"R²_D={row['r2_itd']:+.4f} "
#           f"(ΔD={row['dR2_itd']:+.4f}, p={row['p_itd']:.4f}{row['sig_itd']})")

# print(f"\n  По активам — среднее R² всех конфигураций:")
# print(f"  {'актив':<6} {'R²_A(base)':>11} {'R²_B(+IT)':>11} "
#       f"{'R²_C(+TDA)':>12} {'R²_D(+ALL)':>12}  лучшая")
# print("  " + "-"*60)
# for asset in df.groupby("asset")["r2_itd"].mean().sort_values(ascending=False).index:
#     g = df[df.asset == asset]
#     ra = g["r2_base"].mean(); rb = g["r2_it"].mean()
#     rc_ = g["r2_tda"].mean(); rd = g["r2_itd"].mean()
#     best = max({"A":ra,"B":rb,"C":rc_,"D":rd}, key=lambda k: {"A":ra,"B":rb,"C":rc_,"D":rd}[k])
#     print(f"  {asset:<6} {ra:>+11.4f} {rb:>+11.4f} {rc_:>+12.4f} {rd:>+12.4f}  → {best}")

# print("\nСтроим финальные графики...")

# fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# asset_order = df.groupby("asset")["r2_itd"].mean().sort_values(ascending=False).index.tolist()
# h_order = ["1d","2d","4d","6d","7d","14d"]
# w = 0.2; x = np.arange(len(asset_order))

# ax = axes[0,0]
# for ci, (col, label, color) in enumerate([
#     ("r2_base","A: Baseline","#9E9E9E"),
#     ("r2_it",  "B: +IT",    "#2196F3"),
#     ("r2_tda", "C: +TDA",   "#FF9800"),
#     ("r2_itd", "D: +IT+TDA","#4CAF50"),
# ]):
#     vals = [df[df.asset==a][col].mean() for a in asset_order]
#     ax.bar(x+(ci-1.5)*w, vals, w*0.9, label=label, color=color, alpha=0.85)
# ax.axhline(0, color="black", lw=0.8)
# ax.set_xticks(x); ax.set_xticklabels(asset_order, fontsize=9)
# ax.set_ylabel("OOS R² (vs naive mean logvol)")
# ax.set_title("OOS R² по активам — 4 конфигурации\n(logvol, все горизонты)", fontsize=10)
# ax.legend(fontsize=8); ax.grid(alpha=0.2, axis="y")
# ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

# ax = axes[0,1]
# for col, label, color, ls in [
#     ("r2_base","A: Baseline","#9E9E9E","-"),
#     ("r2_it",  "B: +IT",    "#2196F3","-"),
#     ("r2_tda", "C: +TDA",   "#FF9800","--"),
#     ("r2_itd", "D: +IT+TDA","#4CAF50","-"),
# ]:
#     vals = [df[df.horizon==h][col].mean() for h in h_order]
#     ax.plot(h_order, vals, "o"+ls, color=color, lw=2, label=label, ms=7)
# ax.axhline(0, color="black", lw=0.5, ls=":")
# ax.set_ylabel("OOS R²"); ax.set_xlabel("Горизонт предсказания")
# ax.set_title("OOS R² по горизонтам\n(среднее по 9 активам)", fontsize=10)
# ax.legend(fontsize=8); ax.grid(alpha=0.2)
# ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

# ax = axes[1,0]
# x2 = np.arange(len(asset_order)); w2 = 0.28
# for ci, (col, label, color) in enumerate([
#     ("dR2_it",  "B: +IT",    "#2196F3"),
#     ("dR2_tda", "C: +TDA",   "#FF9800"),
#     ("dR2_itd", "D: +IT+TDA","#4CAF50"),
# ]):
#     vals = [df[df.asset==a][col].mean() for a in asset_order]
#     ax.bar(x2+(ci-1)*w2, vals, w2*0.9, label=label, color=color, alpha=0.85)
# ax.axhline(0, color="black", lw=0.8)
# ax.set_xticks(x2); ax.set_xticklabels(asset_order, fontsize=9)
# ax.set_ylabel("ΔR² относительно baseline A")
# ax.set_title("Инкрементальный ΔR² над baseline\n(logvol, все горизонты)", fontsize=10)
# ax.legend(fontsize=8); ax.grid(alpha=0.2, axis="y")
# ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

# ax = axes[1,1]
# pivot = df.pivot_table(index="asset", columns="horizon",
#                        values="mse_r_itd", aggfunc="mean")
# pivot = pivot.reindex(asset_order)[h_order]
# im = ax.imshow(pivot.values, cmap="RdYlGn_r", vmin=0.85, vmax=1.15, aspect="auto")
# ax.set_xticks(range(len(h_order)));    ax.set_xticklabels(h_order)
# ax.set_yticks(range(len(asset_order))); ax.set_yticklabels(asset_order)
# for i in range(len(asset_order)):
#     for j in range(len(h_order)):
#         val = pivot.values[i,j]
#         if np.isfinite(val):
#             col = "white" if abs(val-1)>0.07 else "black"
#             mark = "✓" if val < 1.0 else ""
#             ax.text(j, i, f"{val:.3f}{mark}", ha="center", va="center",
#                     fontsize=8, color=col)
# plt.colorbar(im, ax=ax, label="MSE_ratio (D: +IT+TDA / baseline)\n<1.0 = лучше ✓")
# ax.set_title("MSE_ratio конфиг D (+IT+TDA)\nзелёный <1 = лучше baseline", fontsize=10)

# plt.suptitle(
#     "Финальное сравнение: A=Baseline / B=+IT / C=+TDA / D=+IT+TDA\n"
#     "Цель: logvol = log(realized_vol_fwd)  |  holdout 2025-01-01 →",
#     fontsize=12, y=1.01)
# plt.tight_layout()
# plt.savefig("fig_final_comparison.png", bbox_inches="tight", dpi=150)
# plt.savefig("fig_final_comparison.pdf", bbox_inches="tight", dpi=150)
# plt.show()
# print("  Saved: fig_final_comparison.png / .pdf")

# final_comparison = df.copy()
# print(f"\nfinal_comparison сохранён в памяти ({len(df)} строк)")

nmi_lag1_mean добавлена: True
Строим X_tda...
TDA фичи: ['BI_tp_H0', 'BI_ent_H1', 'BI_L1_H0', 'BII_tp_H0', 'BII_ent_H0', 'BC_tp_H0']

Всего строк: 54

  ФИНАЛЬНОЕ СРАВНЕНИЕ — целевая переменная: logvol = log(realized_vol_fwd_h)
  R² = 1 - MSE(модель) / MSE(naive=mean_train_logvol)
  A=Baseline(vol)  B=+IT  C=+TDA  D=+IT+TDA

  актив  h     R²_A(base)   R²_B(+IT)   R²_C(+TDA)  R²_D(+IT+TDA)      ΔB      ΔC      ΔD  лучшая
  -----------------------------------------------------------------------------------------------
  ADA    1d       +0.2670     +0.2477      +0.3067        +0.2904 -0.0193↓ +0.0397↑ +0.0235↑  C
  ADA    2d       +0.2413     +0.2131      +0.2940        +0.2604 -0.0283↓ +0.0527↑ +0.0191↑  C
  ADA    4d       +0.1948     +0.1213      +0.2548        +0.1476 -0.0734↓ +0.0600↑ -0.0472↓  C
  ADA    6d       +0.1330     +0.0459      +0.1957        +0.0698 -0.0871↓ +0.0627↑ -0.0632↓  C
  ADA    7d       +0.1210     +0.0266      +0.1891        +0.0527 -0.0944↓ +0.0681↑ -0.0683↓ 

In [80]:
# import warnings; warnings.filterwarnings("ignore")
# import numpy as np
# import pandas as pd
# from itertools import combinations
# from sklearn.linear_model import LogisticRegression
# from sklearn.preprocessing import StandardScaler
# from sklearn.pipeline import Pipeline
# from sklearn.model_selection import KFold, cross_val_score
# from sklearn.metrics import roc_auc_score, average_precision_score
# from sklearn.feature_selection import mutual_info_classif
# import matplotlib
# matplotlib.use("Agg")
# import matplotlib.pyplot as plt

# SPLIT_DATE = "2025-04-01"
# PURGE_BARS = 96
# QUANTILES  = [0.60, 0.70, 0.80, 0.90]   # пороги стресса
# HORIZONS   = {48:"1d", 96:"2d", 192:"4d"}
# ASSETS     = ["BTC","ETH","SOL","BNB","XRP","DOGE","ADA","AVAX","TRX"]
# VOL_WINDOW = 48
# ANN        = np.sqrt(365 * 48)
# RNG        = np.random.default_rng(42)
# N_PERM     = 8

# def _logit(C=0.1):
#     return Pipeline([("sc", StandardScaler()),
#                      ("m", LogisticRegression(C=C, max_iter=2000,
#                                               class_weight="balanced",
#                                               solver="lbfgs"))])

# def lift_at_k(y_true, score, k=0.10):
#     n = len(y_true); top_n = max(1, int(n*k))
#     top_idx = np.argsort(score)[::-1][:top_n]
#     base = float(np.mean(y_true))
#     return float(np.mean(y_true[top_idx])) / base if base > 1e-8 else np.nan

# def safe_roc(y, s):
#     try: return float(roc_auc_score(y, s))
#     except: return np.nan

# def safe_pr(y, s):
#     try: return float(average_precision_score(y, s))
#     except: return np.nan

# print("Строим X_TDA...")
# tda_frames = [tda_A.add_prefix("A_")]
# for mode in ["I","II","C"]:
#     d = tda_B[mode]
#     df_m = pd.DataFrame({
#         f"B{mode}_tp_H0":  d["tp_H0"],  f"B{mode}_ent_H0": d["ent_H0"],
#         f"B{mode}_tp_H1":  d["tp_H1"],  f"B{mode}_ent_H1": d["ent_H1"],
#         f"B{mode}_L1_H0":  d["L1_H0"],  f"B{mode}_L1_H1":  d["L1_H1"],
#     }, index=pd.DatetimeIndex(d["times"]))
#     tda_frames.append(df_m)

# X_tda_raw = pd.concat(tda_frames, axis=1).sort_index()
# X_tda_ri  = X_tda_raw.reindex(ret.index, method="ffill")
# print(f"X_tda: {X_tda_ri.shape[1]} фич, reindexed на {X_tda_ri.shape[0]} баров")

# print("\n" + "="*60)
# print("  ОТБОР ФИЧЕЙ — mutual info на train (BTC h=2d Q70)")
# print("="*60)

# split_ts = pd.Timestamp(SPLIT_DATE)

# pilot_asset, pilot_h, pilot_q = "BTC", 96, 0.70
# r_pilot    = ret[pilot_asset]
# vol_now_p  = r_pilot.rolling(VOL_WINDOW).std() * ANN
# vol_fwd_p  = r_pilot.rolling(pilot_h).std().shift(-pilot_h) * ANN
# vol_q_p    = vol_now_p.expanding(min_periods=VOL_WINDOW*10).quantile(pilot_q)
# y_pilot    = (vol_fwd_p > vol_q_p).astype(float).dropna()

# idx_p = (X_vol.index.intersection(X_it_slim.index)
#          .intersection(X_tda_ri.index).intersection(y_pilot.index))
# valid_p = (X_vol.loc[idx_p].notna().all(1) &
#            X_it_slim.loc[idx_p].notna().all(1) &
#            X_tda_ri.loc[idx_p].notna().all(1) &
#            y_pilot.loc[idx_p].notna())
# idx_p = idx_p[valid_p]
# tr_p = idx_p < (split_ts - pd.Timedelta(minutes=30*PURGE_BARS))

# Xi_tr_p = X_it_slim.loc[idx_p[tr_p]]
# y_tr_p  = y_pilot.loc[idx_p[tr_p]]
# mi_it   = mutual_info_classif(Xi_tr_p.values, y_tr_p.values, random_state=42)
# it_mi_df = pd.Series(mi_it, index=Xi_tr_p.columns).sort_values(ascending=False)

# print(f"\n  IT-фичи по MI с таргетом (топ-15):")
# for feat, val in it_mi_df.head(15).items():
#     print(f"    {feat:<28} {val:.4f}")

# Xt_tr_p = X_tda_ri.loc[idx_p[tr_p]]
# mi_tda  = mutual_info_classif(Xt_tr_p.values, y_tr_p.values, random_state=42)
# tda_mi_df = pd.Series(mi_tda, index=Xt_tr_p.columns).sort_values(ascending=False)

# print(f"\n  TDA-фичи по MI с таргетом (топ-10):")
# for feat, val in tda_mi_df.head(10).items():
#     print(f"    {feat:<28} {val:.4f}")

# IT_BEST  = it_mi_df.head(8).index.tolist()
# TDA_BEST = tda_mi_df.head(6).index.tolist()

# print(f"\n  Выбраны IT-фичи ({len(IT_BEST)}): {IT_BEST}")
# print(f"  Выбраны TDA-фичи ({len(TDA_BEST)}): {TDA_BEST}")

# print("\n" + "="*60)
# print("  STRESS PREDICTION: 4 конфигурации × 4 порога × 9 активов")
# print("  A=vol_now  B=+IT  C=+TDA  D=+IT+TDA")
# print("="*60)

# rows = []

# for asset in ASSETS:
#     if asset not in ret.columns: continue
#     r       = ret[asset]
#     vol_now = r.rolling(VOL_WINDOW).std() * ANN

#     for h, h_label in HORIZONS.items():
#         vol_fwd = r.rolling(h).std().shift(-h) * ANN

#         for q in QUANTILES:
#             vol_q  = vol_now.expanding(min_periods=VOL_WINDOW*10).quantile(q)
#             y_raw  = (vol_fwd > vol_q).astype(float).dropna()

#             idx = (X_vol.index.intersection(X_it_slim.index)
#                    .intersection(X_tda_ri.index).intersection(y_raw.index))
#             valid = (X_vol.loc[idx].notna().all(1) &
#                      X_it_slim.loc[idx].notna().all(1) &
#                      X_tda_ri.loc[idx].notna().all(1) &
#                      y_raw.loc[idx].notna())
#             idx = idx[valid]
#             if len(idx) < 1000: continue

#             tr = idx < (split_ts - pd.Timedelta(minutes=30*PURGE_BARS))
#             te = idx >= split_ts
#             if tr.sum() < 480 or te.sum() < 100: continue

#             ytr = y_raw.loc[idx[tr]]; yte = y_raw.loc[idx[te]]
#             pos_tr = float(ytr.mean()); pos_te = float(yte.mean())
#             if pos_tr < 0.03 or pos_tr > 0.97: continue
#             if yte.sum() < 5: continue

#             Xv_tr  = X_vol.loc[idx[tr]];              Xv_te  = X_vol.loc[idx[te]]
#             Xit_tr = X_it_slim[IT_BEST].loc[idx[tr]]; Xit_te = X_it_slim[IT_BEST].loc[idx[te]]
#             Xtd_tr = X_tda_ri[TDA_BEST].loc[idx[tr]]; Xtd_te = X_tda_ri[TDA_BEST].loc[idx[te]]

#             vn_max  = float(vol_now.loc[idx[tr]].max())
#             sc_base = np.clip(vol_now.loc[idx[te]].values / (vn_max+1e-8), 0, 1)

#             try:
#                 m_it = _logit(); m_it.fit(Xit_tr, ytr)
#                 sc_it = m_it.predict_proba(Xit_te)[:,1]
#             except: sc_it = sc_base.copy()

#             try:
#                 m_tda = _logit(); m_tda.fit(Xtd_tr, ytr)
#                 sc_tda = m_tda.predict_proba(Xtd_te)[:,1]
#             except: sc_tda = sc_base.copy()

#             try:
#                 Xc_tr = pd.concat([Xit_tr, Xtd_tr], axis=1)
#                 Xc_te = pd.concat([Xit_te, Xtd_te], axis=1)
#                 m_all = _logit(); m_all.fit(Xc_tr, ytr)
#                 sc_all = m_all.predict_proba(Xc_te)[:,1]
#             except: sc_all = sc_base.copy()

#             try:
#                 Xfull_tr = pd.concat([Xv_tr, Xit_tr, Xtd_tr], axis=1)
#                 Xfull_te = pd.concat([Xv_te, Xit_te, Xtd_te], axis=1)
#                 m_full = _logit(); m_full.fit(Xfull_tr, ytr)
#                 sc_full = m_full.predict_proba(Xfull_te)[:,1]
#             except: sc_full = sc_base.copy()

#             yte_np = yte.values.astype(int)
#             rows.append({
#                 "asset":  asset, "horizon": h_label, "h": h,
#                 "q":      q,
#                 "pos_tr": round(pos_tr, 3),
#                 "pos_te": round(pos_te, 3),
#                 "pr_A":  round(safe_pr(yte_np, sc_base), 4),
#                 "pr_B":  round(safe_pr(yte_np, sc_it),   4),
#                 "pr_C":  round(safe_pr(yte_np, sc_tda),  4),
#                 "pr_D":  round(safe_pr(yte_np, sc_all),  4),
#                 "pr_E":  round(safe_pr(yte_np, sc_full), 4),
#                 "roc_A": round(safe_roc(yte_np, sc_base), 4),
#                 "roc_B": round(safe_roc(yte_np, sc_it),   4),
#                 "roc_C": round(safe_roc(yte_np, sc_tda),  4),
#                 "roc_D": round(safe_roc(yte_np, sc_all),  4),
#                 "roc_E": round(safe_roc(yte_np, sc_full), 4),
#                 "lift_A": round(lift_at_k(yte_np, sc_base), 3),
#                 "lift_D": round(lift_at_k(yte_np, sc_all),  3),
#                 "lift_E": round(lift_at_k(yte_np, sc_full), 3),
#                 "dpr_B":  round(safe_pr(yte_np,sc_it)  -safe_pr(yte_np,sc_base), 4),
#                 "dpr_C":  round(safe_pr(yte_np,sc_tda) -safe_pr(yte_np,sc_base), 4),
#                 "dpr_D":  round(safe_pr(yte_np,sc_all) -safe_pr(yte_np,sc_base), 4),
#                 "dpr_E":  round(safe_pr(yte_np,sc_full)-safe_pr(yte_np,sc_base), 4),
#                 "n_train": int(tr.sum()), "n_test": int(te.sum()),
#             })

# df = pd.DataFrame(rows)
# print(f"\nВсего строк: {len(df)}")

# print(f"\n{'='*72}")
# print(f"  АГРЕГАТ ПО ПОРОГАМ — среднее по активам и горизонтам")
# print(f"  A=vol_now | B=+IT | C=+TDA | D=IT+TDA | E=vol+IT+TDA")
# print(f"{'='*72}")
# print(f"\n  {'q':>5} {'pos%':>5} "
#       f"{'PR_A':>7} {'PR_B':>7} {'PR_C':>7} {'PR_D':>7} {'PR_E':>7} "
#       f"{'ΔB':>7} {'ΔC':>7} {'ΔD':>7} {'ΔE':>7} "
#       f"{'ROC_D':>7} {'Lift_D':>8}")
# print("  " + "-"*95)

# for q in QUANTILES:
#     sub = df[df.q == q]
#     if sub.empty: continue
#     beats = {k: (sub[f"dpr_{k}"]>0).sum() for k in "BCDE"}
#     print(f"  Q{q:.0%}"
#           f" {sub['pos_te'].mean():>4.0%}"
#           f" {sub['pr_A'].mean():>7.4f}"
#           f" {sub['pr_B'].mean():>7.4f}"
#           f" {sub['pr_C'].mean():>7.4f}"
#           f" {sub['pr_D'].mean():>7.4f}"
#           f" {sub['pr_E'].mean():>7.4f}"
#           f" {sub['dpr_B'].mean():>+7.4f}"
#           f" {sub['dpr_C'].mean():>+7.4f}"
#           f" {sub['dpr_D'].mean():>+7.4f}"
#           f" {sub['dpr_E'].mean():>+7.4f}"
#           f" {sub['roc_D'].mean():>7.4f}"
#           f" {sub['lift_D'].mean():>8.3f}"
#           f"  ({beats['D']}/{len(sub)} PR↑)")

# print(f"\n{'='*72}")
# print(f"  ПО АКТИВАМ — порог Q80")
# print(f"{'='*72}")
# sub80 = df[df.q == 0.80]
# print(f"\n  {'актив':<6} {'pos%':>5} "
#       f"{'PR_A':>7} {'PR_C':>7} {'PR_D':>7} {'PR_E':>7} "
#       f"{'ΔC':>7} {'ΔD':>7} {'ΔE':>7} {'ROC_E':>8} лучшая")
# print("  " + "-"*75)
# for asset in ASSETS:
#     g = sub80[sub80.asset==asset]
#     if g.empty: continue
#     means = {k: g[f"pr_{k}"].mean() for k in "ACDE"}
#     best  = max(means, key=means.get)
#     print(f"  {asset:<6}"
#           f" {g['pos_te'].mean():>4.0%}"
#           f" {g['pr_A'].mean():>7.4f}"
#           f" {g['pr_C'].mean():>7.4f}"
#           f" {g['pr_D'].mean():>7.4f}"
#           f" {g['pr_E'].mean():>7.4f}"
#           f" {g['dpr_C'].mean():>+7.4f}"
#           f" {g['dpr_D'].mean():>+7.4f}"
#           f" {g['dpr_E'].mean():>+7.4f}"
#           f" {g['roc_E'].mean():>8.4f}"
#           f"  → {best}")

# print(f"\n{'='*72}")
# print(f"  ТОП-10 по ΔPR (конфиг E = vol+IT+TDA)")
# print(f"{'='*72}")
# top10 = df.nlargest(10, "dpr_E")
# print(f"\n  {'актив':<6} {'h':<4} {'Q':>5} {'pos%':>5} "
#       f"{'PR_A':>7} {'PR_E':>7} {'ΔE':>8} {'ROC_E':>8} {'Lift_E':>8}")
# print("  " + "-"*65)
# for _, row in top10.iterrows():
#     print(f"  {row['asset']:<6} {row['horizon']:<4} Q{row['q']:.0%}"
#           f" {row['pos_te']:>4.0%}"
#           f" {row['pr_A']:>7.4f}"
#           f" {row['pr_E']:>7.4f}"
#           f" {row['dpr_E']:>+8.4f}"
#           f" {row['roc_E']:>8.4f}"
#           f" {row['lift_E']:>8.3f}")

# print("\nСтроим графики...")
# fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# q_labels = [f"Q{q:.0%}" for q in QUANTILES]
# colors = {"A":"#9E9E9E","B":"#2196F3","C":"#FF9800","D":"#4CAF50","E":"#9C27B0"}

# ax = axes[0,0]
# x = np.arange(len(QUANTILES)); w = 0.18
# for ci, k in enumerate(["B","C","D","E"]):
#     vals = [df[df.q==q][f"dpr_{k}"].mean() for q in QUANTILES]
#     ax.bar(x+(ci-1.5)*w, vals, w*0.9,
#            label={"B":"+IT","C":"+TDA","D":"IT+TDA","E":"vol+IT+TDA"}[k],
#            color=colors[k], alpha=0.85)
# ax.axhline(0, color="black", lw=0.8)
# ax.set_xticks(x); ax.set_xticklabels(q_labels)
# ax.set_ylabel("ΔPR-AUC vs baseline (vol_now)")
# ax.set_title("ΔPR-AUC по порогам стресса\n(среднее по всем активам и горизонтам)", fontsize=10)
# ax.legend(fontsize=8); ax.grid(alpha=0.2, axis="y")
# ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

# ax = axes[0,1]
# for k in ["A","B","C","D","E"]:
#     vals = [df[df.q==q][f"pr_{k}"].mean() for q in QUANTILES]
#     label = {"A":"Baseline","B":"+IT","C":"+TDA","D":"IT+TDA","E":"vol+IT+TDA"}[k]
#     ls = "--" if k=="C" else "-"
#     ax.plot(q_labels, vals, "o"+ls, color=colors[k], lw=2, label=label, ms=7)
# ax.set_ylabel("PR-AUC"); ax.set_xlabel("Порог стресса")
# ax.set_title("PR-AUC по порогам\n(среднее по активам)", fontsize=10)
# ax.legend(fontsize=8); ax.grid(alpha=0.2)
# ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

# ax = axes[1,0]
# asset_order = sub80.groupby("asset")["dpr_E"].mean().sort_values(ascending=False).index
# x3 = np.arange(len(asset_order)); w3 = 0.28
# for ci, k in enumerate(["C","D","E"]):
#     vals = [sub80[sub80.asset==a][f"dpr_{k}"].mean() for a in asset_order]
#     label = {"+TDA":"+TDA","IT+TDA":"IT+TDA","vol+IT+TDA":"vol+IT+TDA"}.get(k,k)
#     ax.bar(x3+(ci-1)*w3, vals, w3*0.9,
#            label={"C":"+TDA","D":"IT+TDA","E":"vol+IT+TDA"}[k],
#            color=colors[k], alpha=0.85)
# ax.axhline(0, color="black", lw=0.8)
# ax.set_xticks(x3); ax.set_xticklabels(list(asset_order), fontsize=9)
# ax.set_ylabel("ΔPR-AUC vs baseline")
# ax.set_title("ΔPR-AUC по активам (Q80)\n(среднее по горизонтам)", fontsize=10)
# ax.legend(fontsize=8); ax.grid(alpha=0.2, axis="y")
# ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

# ax = axes[1,1]
# pivot_roc = df.pivot_table(index="asset", columns="q",
#                             values="roc_E", aggfunc="mean")
# pivot_roc = pivot_roc.reindex(ASSETS)[QUANTILES]
# im = ax.imshow(pivot_roc.values, cmap="RdYlGn", vmin=0.45, vmax=0.85, aspect="auto")
# ax.set_xticks(range(len(QUANTILES))); ax.set_xticklabels(q_labels)
# ax.set_yticks(range(len(ASSETS)));    ax.set_yticklabels(ASSETS)
# for i in range(len(ASSETS)):
#     for j in range(len(QUANTILES)):
#         val = pivot_roc.values[i,j]
#         if np.isfinite(val):
#             col = "white" if val > 0.70 or val < 0.52 else "black"
#             ax.text(j, i, f"{val:.3f}", ha="center", va="center",
#                     fontsize=8, color=col)
# plt.colorbar(im, ax=ax, label="ROC-AUC (конфиг E: vol+IT+TDA)")
# ax.set_title("ROC-AUC heatmap (vol+IT+TDA)\nпо активам × порогам", fontsize=10)

# plt.suptitle(
#     "Stress prediction: A=vol_now / B=+IT / C=+TDA / D=IT+TDA / E=vol+IT+TDA\n"
#     f"Holdout {SPLIT_DATE} →  |  Метрика: PR-AUC  |  Горизонты: 1d, 2d, 4d",
#     fontsize=12, y=1.01)
# plt.tight_layout()
# plt.savefig("fig_stress_it_tda.png", bbox_inches="tight", dpi=150)
# plt.savefig("fig_stress_it_tda.pdf", bbox_inches="tight", dpi=150)
# plt.show()
# print("  Saved: fig_stress_it_tda.png / .pdf")

# stress_results = df.copy()
# print(f"\nstress_results сохранён в памяти ({len(df)} строк)")
# print(f"IT_BEST:  {IT_BEST}")
# print(f"TDA_BEST: {TDA_BEST}")

Строим X_TDA...
X_tda: 26 фич, reindexed на 46743 баров

  ОТБОР ФИЧЕЙ — mutual info на train (BTC h=2d Q70)

  IT-фичи по MI с таргетом (топ-15):
    coinfo4_min                  0.6362
    pairwise_mi_std              0.6359
    coinfo4_std                  0.6356
    kl_AVAX                      0.6352
    kl_ETH                       0.6351
    coinfo3_max                  0.6351
    pairwise_mi_mean             0.6351
    coinfo4_max                  0.6350
    kl_ADA                       0.6348
    kl_BTC                       0.6348
    kl_BNB                       0.6347
    coinfo3_std                  0.6345
    coinfo3_mean                 0.6345
    coinfo4_mean                 0.6345
    kl_XRP                       0.6342

  TDA-фичи по MI с таргетом (топ-10):
    BII_tp_H1                    0.5160
    BI_tp_H0                     0.5140
    BII_ent_H1                   0.5139
    A_L1_H0                      0.5135
    BII_tp_H0                    0.5134
    BC_tp_H0  

In [82]:
# import warnings; warnings.filterwarnings("ignore")
# import numpy as np
# import pandas as pd
# from itertools import combinations
# from sklearn.metrics import mutual_info_score
# import gudhi
# from gudhi.representations import Landscape
# from tqdm.auto import tqdm
# import matplotlib
# matplotlib.use("Agg")
# import matplotlib.pyplot as plt

# TICKER_MAP_FULL = {
#     "BTC-USD":"BTC","ETH-USD":"ETH","BNB-USD":"BNB","XRP-USD":"XRP",
#     "SOL-USD":"SOL","TRX-USD":"TRX","DOGE-USD":"DOGE","ADA-USD":"ADA",
#     "ZEC-USD":"ZEC","BCH-USD":"BCH","LINK-USD":"LINK","LEO-USD":"LEO",
#     "XLM-USD":"XLM","LTC-USD":"LTC","XMR-USD":"XMR","AVAX-USD":"AVAX",
#     "HBAR-USD":"HBAR","DAI-USD":"DAI","DOT-USD":"DOT","UNI-USD":"UNI",
#     "TON-USD":"TON","CRO-USD":"CRO","NEAR-USD":"NEAR","ICP-USD":"ICP",
#     "AAVE-USD":"AAVE","ETC-USD":"ETC","PI-USD":"PI","WLD-USD":"WLD",
#     "MATIC-USD":"MATIC","ATOM-USD":"ATOM","FIL-USD":"FIL",
#     "ALGO-USD":"ALGO","VET-USD":"VET",
# }

# ASSETS_WIDE = [v for v in TICKER_MAP_FULL.values() if v in ret.columns]
# ASSETS_9    = ["BTC","ETH","SOL","BNB","XRP","DOGE","ADA","AVAX","TRX"]

# print(f"Активов в ret: {len(ret.columns)}")
# print(f"Активов wide (из ноутбука ∩ ret): {len(ASSETS_WIDE)}")
# print(f"  {ASSETS_WIDE}")

# WINDOW = 192; STEP = 48; BINS = 20
# WINDOW_LABEL = f"{WINDOW} баров (~{WINDOW//48}d)"

# R_wide  = ret[ASSETS_WIDE].fillna(0).replace([np.inf,-np.inf],0).values
# R_9     = ret[ASSETS_9].fillna(0).replace([np.inf,-np.inf],0).values
# idx_all = ret.index
# n_total = len(R_wide)
# t_range = list(range(0, n_total - WINDOW, STEP))
# times   = [idx_all[t + WINDOW] for t in t_range]

# print(f"\nОкна: {len(t_range)} | окно={WINDOW_LABEL} | шаг={STEP} баров")
# print(f"Период: {times[0]} → {times[-1]}")

# def mutual_information(x, y, bins=BINS):
#     x=np.asarray(x); y=np.asarray(y)
#     mask=np.isfinite(x)&np.isfinite(y); x=x[mask]; y=y[mask]
#     if len(x)<20: return 0.0
#     c=np.histogram2d(x,y,bins=bins)[0]
#     return float(mutual_info_score(None,None,contingency=c))

# def build_mi_complex(X_window, bins=BINS):
#     dim=X_window.shape[1]; MI=np.zeros((dim,dim))
#     for i,j in combinations(range(dim),2):
#         mi=mutual_information(X_window[:,i],X_window[:,j],bins=bins)
#         MI[i,j]=MI[j,i]=mi
#     triu=MI[np.triu_indices_from(MI,1)]
#     MI_max=triu.max() if len(triu)>0 else 1.0
#     st=gudhi.SimplexTree()
#     for i in range(dim): st.insert([i],filtration=0.0)
#     edge_w=np.zeros((dim,dim))
#     for i,j in combinations(range(dim),2):
#         w=MI_max-MI[i,j]; edge_w[i,j]=edge_w[j,i]=w
#         st.insert([i,j],filtration=w)
#     for i,j,k in combinations(range(dim),3):
#         st.insert([i,j,k],filtration=max(edge_w[i,j],edge_w[i,k],edge_w[j,k]))
#     st.initialize_filtration()
#     return st, MI, MI_max

# def diag_for_dim(diag, dim):
#     return [(float(b),float(d)) for dd,(b,d) in diag if dd==dim]

# def diagram_statistics(diag_bd, max_filtration):
#     if not diag_bd: return 0.0,0.0
#     lts=[((d-b) if np.isfinite(d) else (max_filtration-b)) for b,d in diag_bd if ((d-b) if np.isfinite(d) else (max_filtration-b))>0]
#     if not lts: return 0.0,0.0
#     lts=np.array(lts); T=float(lts.sum()); p=lts/T
#     return T, float(-(p*np.log(p)).sum())

# def landscape_norms(diag_bd, max_filtration, resolution=100):
#     if not diag_bd: return 0.0,0.0
#     pts=[[b,d if np.isfinite(d) else max_filtration] for b,d in diag_bd]
#     arr=np.array(pts,dtype=float)
#     try:
#         L=Landscape(num_landscapes=1,resolution=resolution)
#         X=L.fit_transform([arr])[0]
#         return float(np.sum(np.abs(X))), float(np.sqrt(np.sum(X**2)))
#     except: return 0.0,0.0

# def H_gaussian(S):
#     if isinstance(S,np.ndarray):
#         n=S.shape[0]; det=np.linalg.det(S)
#         return 0.5*np.log((2*np.pi*np.e)**n*det) if det>0 else 0.0
#     return 0.5*np.log(2*np.pi*np.e*S) if S>0 else 0.0

# def I_gauss(simplex, A):
#     d=len(simplex); Hm=d*H_gaussian(A[0,0])
#     return Hm-H_gaussian(A[np.ix_(simplex,simplex)])

# def build_gauss_complex(X_window, triple_mode="I", eps=1e-8):
#     dim=X_window.shape[1]
#     A=np.cov(X_window,rowvar=False)+eps*np.eye(dim)
#     st=gudhi.SimplexTree()
#     for i in range(dim): st.insert([i],filtration=0.0)
#     for i,j in combinations(range(dim),2):
#         st.insert([i,j],filtration=I_gauss([i,j],A))
#     for i,j,k in combinations(range(dim),3):
#         st.insert([i,j,k],filtration=I_gauss([i,j,k],A))
#     st.initialize_filtration()
#     fds=[d for(_,(b,d)) in st.persistence() if np.isfinite(d)]
#     return st, A, max(fds) if fds else 1.0

# def run_exp_A(R, label):
#     T0s,E0s,T1s,E1s,L1_0s,L1_1s=[],[],[],[],[],[]
#     for t in tqdm(t_range, desc=f"Exp A [{label}]"):
#         wnd=R[t:t+WINDOW]
#         try:
#             st,MI,MI_max=build_mi_complex(wnd)
#             diag=st.persistence()
#             d0=diag_for_dim(diag,0); d1=diag_for_dim(diag,1)
#             T0,E0=diagram_statistics(d0,MI_max)
#             T1,E1=diagram_statistics(d1,MI_max)
#             l0,_=landscape_norms(d0,MI_max); l1,_=landscape_norms(d1,MI_max)
#         except: T0=E0=T1=E1=l0=l1=0.0
#         T0s.append(T0);E0s.append(E0);T1s.append(T1)
#         E1s.append(E1);L1_0s.append(l0);L1_1s.append(l1)
#     return pd.DataFrame({"T_H0":T0s,"E_H0":E0s,"T_H1":T1s,
#                           "E_H1":E1s,"L1_H0":L1_0s,"L1_H1":L1_1s},
#                          index=pd.DatetimeIndex(times))

# def run_exp_B(R, label):
#     T0s,E0s,T1s,E1s,L1_0s,L1_1s=[],[],[],[],[],[]
#     gmax=0.0
#     for t in tqdm(t_range[:min(50,len(t_range))], desc=f"scan [{label}]"):
#         try:
#             st,A,mf=build_gauss_complex(R[t:t+WINDOW])
#             gmax=max(gmax,mf)
#         except: pass
#     print(f"  global_max={gmax:.4f}")
#     for t in tqdm(t_range, desc=f"Exp B [{label}]"):
#         wnd=R[t:t+WINDOW]
#         try:
#             st,A,_=build_gauss_complex(wnd)
#             diag=st.persistence()
#             d0=diag_for_dim(diag,0); d1=diag_for_dim(diag,1)
#             T0,E0=diagram_statistics(d0,gmax)
#             T1,E1=diagram_statistics(d1,gmax)
#             l0,_=landscape_norms(d0,gmax); l1,_=landscape_norms(d1,gmax)
#         except: T0=E0=T1=E1=l0=l1=0.0
#         T0s.append(T0);E0s.append(E0);T1s.append(T1)
#         E1s.append(E1);L1_0s.append(l0);L1_1s.append(l1)
#     return pd.DataFrame({"BI_tp_H0":T0s,"BI_ent_H0":E0s,"BI_tp_H1":T1s,
#                           "BI_ent_H1":E1s,"BI_L1_H0":L1_0s,"BI_L1_H1":L1_1s},
#                          index=pd.DatetimeIndex(times))

# print("\n" + "="*60)
# print(f"  СТРОИМ TDA НА ШИРОКОМ ЮНИВЕРСЕ ({len(ASSETS_WIDE)} активов)")
# print("="*60)

# df_A_wide = run_exp_A(R_wide, f"{len(ASSETS_WIDE)}assets")
# df_B_wide = run_exp_B(R_wide, f"{len(ASSETS_WIDE)}assets")

# X_tda_wide = pd.concat([
#     df_A_wide.add_prefix("W_A_"),
#     df_B_wide.add_prefix("W_")
# ], axis=1).reindex(ret.index, method="ffill")

# print(f"\nX_tda_wide: {X_tda_wide.shape[1]} фич")

# print("\n" + "="*60)
# print("  СРАВНЕНИЕ TDA-СИГНАЛА: 9 активов vs широкий юниверс")
# print("="*60)

# if "BTC" in ret.columns:
#     vol_btc = ret["BTC"].rolling(48).std() * np.sqrt(365*48)
#     vol_r   = vol_btc.reindex(pd.DatetimeIndex(times), method="nearest")

#     print("\n  Корреляция с vol_BTC:")
#     print(f"  {'фича':<25} {'9-asset':>8} {'wide':>8}  Δ")
#     print("  " + "-"*45)

#     for col_9, col_w in [
#         ("T_H1",    "W_A_T_H1"),
#         ("E_H1",    "W_A_E_H1"),
#         ("T_H0",    "W_A_T_H0"),
#         ("E_H0",    "W_A_E_H0"),
#         ("BI_tp_H1","W_BI_tp_H1"),
#         ("BI_ent_H1","W_BI_ent_H1"),
#     ]:
#         if col_9 in tda_A.columns:
#             s9 = tda_A[col_9].reindex(vol_r.index, method="nearest")
#         elif col_9 in tda_B.get("I",{}).keys() if isinstance(tda_B,dict) else []:
#             s9 = pd.Series(tda_B["I"][col_9.replace("BI_","").lower().replace("tp","tp")],
#                            index=pd.DatetimeIndex(tda_B["I"]["times"]))
#             s9 = s9.reindex(vol_r.index, method="nearest")
#         else:
#             continue

#         if col_w not in X_tda_wide.columns: continue
#         sw = X_tda_wide[col_w].reindex(vol_r.index, method="nearest")

#         c9 = float(s9.corr(vol_r)); cw = float(sw.corr(vol_r))
#         delta = cw - c9
#         mark = "↑" if abs(cw)>abs(c9)+0.02 else ("↓" if abs(c9)>abs(cw)+0.02 else "~")
#         print(f"  {col_9:<25} {c9:>+8.4f}  {cw:>+8.4f}  {delta:>+6.4f} {mark}")

# from sklearn.linear_model import LogisticRegression
# from sklearn.preprocessing import StandardScaler
# from sklearn.pipeline import Pipeline
# from sklearn.metrics import roc_auc_score, average_precision_score
# from sklearn.feature_selection import mutual_info_classif

# split_ts = pd.Timestamp("2025-04-01")
# PURGE_BARS = 96
# HORIZONS   = {48:"1d", 96:"2d", 192:"4d"}
# ASSETS_PRED= ["BTC","ETH","DOGE","XRP","SOL","BNB"]
# QUANTILES  = [0.70, 0.80]

# def _logit():
#     return Pipeline([("sc",StandardScaler()),
#                      ("m",LogisticRegression(C=0.1,max_iter=2000,
#                                              class_weight="balanced"))])

# Xi_it_slim = X_it_slim[["coinfo3_std","kl_BNB","coinfo4_std",
#                           "pairwise_mi_mean","coinfo4_mean","pairwise_mi_std",
#                           "coinfo3_mean","coinfo4_max","kl_AVAX","pairwise_mi_min"]]

# r_btc=ret["BTC"]; vn=r_btc.rolling(48).std()*np.sqrt(365*48)
# vf=r_btc.rolling(96).std().shift(-96)*np.sqrt(365*48)
# vq=vn.expanding(min_periods=480).quantile(0.70)
# y_sel=(vf>vq).astype(float).dropna()
# idx_sel=X_tda_wide.index.intersection(y_sel.index)
# valid_sel=X_tda_wide.loc[idx_sel].notna().all(1)&y_sel.loc[idx_sel].notna()
# idx_sel=idx_sel[valid_sel]
# tr_sel=idx_sel<(split_ts-pd.Timedelta(minutes=30*96))
# mi_w=mutual_info_classif(X_tda_wide.loc[idx_sel[tr_sel]].values,
#                           y_sel.loc[idx_sel[tr_sel]].values,random_state=42)
# tda_wide_mi=pd.Series(mi_w,index=X_tda_wide.columns).sort_values(ascending=False)
# TDA_WIDE_BEST=tda_wide_mi.head(6).index.tolist()
# print(f"\n  Лучшие TDA_wide фичи: {TDA_WIDE_BEST}")

# print(f"\n  Stress prediction — narrow (9) vs wide ({len(ASSETS_WIDE)}) TDA:")
# print(f"  {'актив':<6} {'h':<4} {'Q':>5}  PR_A  PR_narrow  PR_wide  Δnarrow  Δwide  лучшая")
# print("  " + "-"*65)

# comp_rows=[]
# for asset in ASSETS_PRED:
#     if asset not in ret.columns: continue
#     r=ret[asset]; vn_a=r.rolling(48).std()*np.sqrt(365*48)
#     for h,hl in HORIZONS.items():
#         vf_a=r.rolling(h).std().shift(-h)*np.sqrt(365*48)
#         for q in QUANTILES:
#             vq_a=vn_a.expanding(min_periods=480).quantile(q)
#             y_raw=(vf_a>vq_a).astype(float).dropna()

#             idx=(X_it_slim.index.intersection(X_tda_ri.index)
#                  .intersection(X_tda_wide.index).intersection(y_raw.index))
#             valid=(X_it_slim.loc[idx].notna().all(1)&
#                    X_tda_ri.loc[idx].notna().all(1)&
#                    X_tda_wide.loc[idx].notna().all(1)&
#                    y_raw.loc[idx].notna())
#             idx=idx[valid]
#             if len(idx)<500: continue
#             tr=idx<(split_ts-pd.Timedelta(minutes=30*96))
#             te=idx>=split_ts
#             if tr.sum()<200 or te.sum()<50: continue

#             ytr=y_raw.loc[idx[tr]]; yte=y_raw.loc[idx[te]]
#             if ytr.mean()<0.03 or ytr.mean()>0.97: continue
#             if yte.sum()<3: continue

#             vna=vn_a.loc[idx[tr]]; vna_te=vn_a.loc[idx[te]]
#             sc_base=np.clip(vna_te.values/(float(vna.max())+1e-8),0,1)

#             Xit_tr=Xi_it_slim.loc[idx[tr]]; Xit_te=Xi_it_slim.loc[idx[te]]
#             Xn_tr=X_tda_ri[["BI_tp_H0","BI_ent_H1","BI_L1_H0",
#                               "BII_tp_H0","BII_ent_H0","BC_tp_H0"]].loc[idx[tr]]
#             Xn_te=X_tda_ri[["BI_tp_H0","BI_ent_H1","BI_L1_H0",
#                               "BII_tp_H0","BII_ent_H0","BC_tp_H0"]].loc[idx[te]]
#             Xw_tr=X_tda_wide[TDA_WIDE_BEST].loc[idx[tr]]
#             Xw_te=X_tda_wide[TDA_WIDE_BEST].loc[idx[te]]

#             try:
#                 mn=_logit()
#                 mn.fit(pd.concat([Xit_tr,Xn_tr],axis=1),ytr)
#                 sc_n=mn.predict_proba(pd.concat([Xit_te,Xn_te],axis=1))[:,1]

#                 mw=_logit()
#                 mw.fit(pd.concat([Xit_tr,Xw_tr],axis=1),ytr)
#                 sc_w=mw.predict_proba(pd.concat([Xit_te,Xw_te],axis=1))[:,1]
#             except: continue

#             yte_np=yte.values.astype(int)
#             try:
#                 pr_a=average_precision_score(yte_np,sc_base)
#                 pr_n=average_precision_score(yte_np,sc_n)
#                 pr_w=average_precision_score(yte_np,sc_w)
#             except: continue

#             best={"A":pr_a,"narrow":pr_n,"wide":pr_w}
#             best_k=max(best,key=best.get)
#             dn=pr_n-pr_a; dw=pr_w-pr_a
#             print(f"  {asset:<6} {hl:<4} Q{q:.0%}"
#                   f"  {pr_a:.4f}  {pr_n:.4f}    {pr_w:.4f}"
#                   f"  {dn:>+7.4f}  {dw:>+7.4f}  {best_k}")
#             comp_rows.append({"asset":asset,"h":hl,"q":q,
#                                "pr_A":pr_a,"pr_narrow":pr_n,"pr_wide":pr_w,
#                                "d_narrow":dn,"d_wide":dw})

# if comp_rows:
#     cdf=pd.DataFrame(comp_rows)
#     print(f"\n  Среднее ΔPR (narrow TDA): {cdf['d_narrow'].mean():>+.4f}")
#     print(f"  Среднее ΔPR (wide TDA):   {cdf['d_wide'].mean():>+.4f}")
#     print(f"  Wide лучше narrow: {(cdf['d_wide']>cdf['d_narrow']).sum()}/{len(cdf)}")

# tda_A_wide = df_A_wide.copy()
# tda_B_wide = df_B_wide.copy()
# X_tda_wide_final = X_tda_wide.copy()
# print(f"\ntda_A_wide, tda_B_wide, X_tda_wide_final — сохранены в памяти")

Активов в ret: 9
Активов wide (из ноутбука ∩ ret): 9
  ['BTC', 'ETH', 'BNB', 'XRP', 'SOL', 'TRX', 'DOGE', 'ADA', 'AVAX']

Окна: 970 | окно=192 баров (~4d) | шаг=48 баров
Период: 2023-01-05 00:00:00 → 2025-08-31 05:00:00

  СТРОИМ TDA НА ШИРОКОМ ЮНИВЕРСЕ (9 активов)


scan [9assets]: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:00<00:00, 1064.14it/s]


  global_max=1.2325


Exp B [9assets]: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 970/970 [00:01<00:00, 770.39it/s]



X_tda_wide: 12 фич

  СРАВНЕНИЕ TDA-СИГНАЛА: 9 активов vs широкий юниверс

  Корреляция с vol_BTC:
  фича                       9-asset     wide  Δ
  ---------------------------------------------
  T_H1                       +0.0193   +0.0193  +0.0000 ~
  E_H1                       +0.0179   +0.0179  +0.0000 ~
  T_H0                       -0.0411   -0.0411  +0.0000 ~
  E_H0                       -0.1109   -0.1109  +0.0000 ~

  Лучшие TDA_wide фичи: ['W_A_E_H0', 'W_A_L1_H0', 'W_A_T_H0', 'W_BI_tp_H0', 'W_BI_L1_H0', 'W_A_T_H1']

  Stress prediction — narrow (9) vs wide (9) TDA:
  актив  h        Q  PR_A  PR_narrow  PR_wide  Δnarrow  Δwide  лучшая
  -----------------------------------------------------------------
  BTC    1d   Q70%  0.4102  0.4290    0.3587  +0.0188  -0.0515  narrow
  BTC    1d   Q80%  0.4648  0.4205    0.3139  -0.0443  -0.1509  A
  BTC    2d   Q70%  0.4822  0.4433    0.3689  -0.0389  -0.1133  A
  BTC    2d   Q80%  0.5462  0.6138    0.5454  +0.0676  -0.0008  narrow
  BTC

In [83]:
# import warnings; warnings.filterwarnings("ignore")
# import numpy as np
# import pandas as pd
# from sklearn.feature_selection import mutual_info_classif
# from sklearn.linear_model import LogisticRegression
# from sklearn.preprocessing import StandardScaler
# from sklearn.pipeline import Pipeline

# VOL_WINDOW = 48
# ANN = np.sqrt(365 * 48)

# print("="*60)
# print("  ШАГ 1: Загружаем дневные данные широкого юниверса")
# print("="*60)

# WIDE_TICKERS = [
#     "BTC-USD","ETH-USD","BNB-USD","XRP-USD","SOL-USD","TRX-USD",
#     "DOGE-USD","ADA-USD","AVAX-USD","LINK-USD","LTC-USD","DOT-USD",
#     "UNI-USD","ATOM-USD","ETC-USD","XLM-USD","ALGO-USD","VET-USD",
#     "AAVE-USD","NEAR-USD","ICP-USD","FIL-USD","MATIC-USD","CRO-USD",
# ]

# TICKER_MAP = {t: t.replace("-USD","") for t in WIDE_TICKERS}

# try:
#     import yfinance as yf
#     print(f"  Загружаем {len(WIDE_TICKERS)} тикеров с 2023-01-01...")
#     data_wide = yf.download(WIDE_TICKERS, start="2023-01-01",
#                              end="2025-09-01", interval="1d",
#                              progress=False)["Close"]
#     data_wide.columns = [TICKER_MAP.get(c, c) for c in data_wide.columns]
#     data_wide = data_wide.ffill().dropna(how="all", axis=1)

#     ret_daily_wide = np.log(data_wide).diff().replace([np.inf,-np.inf], np.nan).fillna(0)
#     print(f"  Загружено: {ret_daily_wide.shape[1]} активов, {len(ret_daily_wide)} дней")
#     print(f"  Активы: {list(ret_daily_wide.columns)}")

#     ret_daily_wide.index = pd.to_datetime(ret_daily_wide.index).tz_localize(None)
#     ret_wide_30min = ret_daily_wide.reindex(ret.index, method="ffill")

#     new_assets = [c for c in ret_wide_30min.columns if c not in ret.columns]
#     print(f"\n  Новых активов (не в ret): {len(new_assets)}: {new_assets}")

#     if new_assets:
#         ret_wide = pd.concat([ret, ret_wide_30min[new_assets]], axis=1)
#     else:
#         ret_wide = ret.copy()
#     print(f"  ret_wide: {ret_wide.shape[1]} активов")

# except Exception as e:
#     print(f"  yfinance недоступен: {e}")
#     print(f"  Используем ret как есть")
#     ret_wide = ret.copy()
#     new_assets = []

# ASSETS_WIDE_FINAL = list(ret_wide.columns)
# print(f"\n  Итого активов для TDA: {len(ASSETS_WIDE_FINAL)}")

# print("\n" + "="*60)
# print("  ШАГ 2: Residual MI — отбор фичей поверх vol_now")
# print("  MI(фича, стресс | vol_now) вместо MI(фича, стресс)")
# print("="*60)

# SPLIT_DATE = "2025-04-01"
# split_ts   = pd.Timestamp(SPLIT_DATE)
# PURGE_BARS = 96

# pilot_asset = "BTC"
# r_p   = ret[pilot_asset]
# vn_p  = r_p.rolling(VOL_WINDOW).std() * ANN
# vf_p  = r_p.rolling(96).std().shift(-96) * ANN
# vq_p  = vn_p.expanding(min_periods=100).quantile(0.70)
# y_p   = (vf_p > vq_p).astype(float).dropna()
# vnr_p = vn_p.expanding(min_periods=100).rank(pct=True)   # percentile rank

# idx_p = (X_it_slim.index.intersection(ret.index)
#          .intersection(y_p.index).intersection(vnr_p.index))
# valid_p = (X_it_slim.loc[idx_p].notna().all(1) &
#            y_p.loc[idx_p].notna() &
#            vnr_p.loc[idx_p].notna())
# idx_p = idx_p[valid_p]
# tr_p  = idx_p < (split_ts - pd.Timedelta(minutes=30*PURGE_BARS))
# idx_tr = idx_p[tr_p]

# ytr_p   = y_p.loc[idx_tr]
# vnr_tr  = vnr_p.loc[idx_tr].values.reshape(-1,1)

# mdl_vol = Pipeline([("sc", StandardScaler()),
#                      ("m", LogisticRegression(C=1.0, max_iter=1000,
#                                                class_weight="balanced"))])
# mdl_vol.fit(vnr_tr, ytr_p)
# prob_vol = mdl_vol.predict_proba(vnr_tr)[:,1]   # P(stress | vol_now)

# res_stress = ytr_p.values - prob_vol   # непрерывный остаток

# Xi_tr_p = X_it_slim.loc[idx_tr]

# res_bin = (res_stress > np.median(res_stress)).astype(int)

# mi_it_res = mutual_info_classif(Xi_tr_p.values, res_bin, random_state=42)
# it_res_df  = pd.Series(mi_it_res, index=Xi_tr_p.columns).sort_values(ascending=False)

# print(f"\n  Conditional MI(IT-фичи | vol_now) — топ-15:")
# print(f"  {'фича':<28} {'MI':>8}  (сравните с ~0.63 из сырого MI)")
# for feat, val in it_res_df.head(15).items():
#     bar = "█" * max(0, int(val * 200))
#     print(f"  {feat:<28} {val:>8.4f}  {bar}")

# print(f"\n  Разброс MI: min={it_res_df.min():.4f}  max={it_res_df.max():.4f}  "
#       f"std={it_res_df.std():.4f}")
# print(f"  Теперь фичи реально различимы: std={it_res_df.std():.4f} "
#       f"(было ~0.001 с сырым MI)")

# tda_frames = [tda_A.add_prefix("A_")]
# for mode in ["I","II","C"]:
#     d = tda_B[mode]
#     df_m = pd.DataFrame({
#         f"B{mode}_tp_H0": d["tp_H0"], f"B{mode}_ent_H0": d["ent_H0"],
#         f"B{mode}_tp_H1": d["tp_H1"], f"B{mode}_ent_H1": d["ent_H1"],
#         f"B{mode}_L1_H0": d["L1_H0"],f"B{mode}_L1_H1":  d["L1_H1"],
#     }, index=pd.DatetimeIndex(d["times"]))
#     tda_frames.append(df_m)
# X_tda_raw = pd.concat(tda_frames, axis=1).sort_index()
# X_tda_ri  = X_tda_raw.reindex(ret.index, method="ffill")

# Xt_tr_p = X_tda_ri.loc[idx_tr]
# mi_tda_res = mutual_info_classif(Xt_tr_p.values, res_bin, random_state=42)
# tda_res_df = pd.Series(mi_tda_res, index=Xt_tr_p.columns).sort_values(ascending=False)

# print(f"\n  Conditional MI(TDA-фичи | vol_now) — топ-10:")
# for feat, val in tda_res_df.head(10).items():
#     bar = "█" * max(0, int(val * 200))
#     print(f"  {feat:<28} {val:>8.4f}  {bar}")

# IT_RES_BEST  = it_res_df.head(8).index.tolist()
# TDA_RES_BEST = tda_res_df.head(6).index.tolist()

# print(f"\n  Лучшие IT (conditional):  {IT_RES_BEST}")
# print(f"  Лучшие TDA (conditional): {TDA_RES_BEST}")

# print("\n" + "="*60)
# print("  ШАГ 3: Быстрая проверка — старый vs новый отбор фичей")
# print("  BTC/ETH/DOGE, h=2d, Q80, holdout 2025-01-01")
# print("="*60)

# from sklearn.metrics import roc_auc_score, average_precision_score

# OLD_IT  = ["coinfo4_min","pairwise_mi_std","coinfo4_std","kl_AVAX",
#            "kl_ETH","coinfo3_max","pairwise_mi_mean","coinfo4_max"]
# OLD_TDA = ["BII_tp_H1","BI_tp_H0","BII_ent_H1","A_L1_H0","BII_tp_H0","BC_tp_H0"]

# def _logit_quick():
#     return Pipeline([("sc",StandardScaler()),
#                      ("m",LogisticRegression(C=0.1,max_iter=2000,
#                                               class_weight="balanced"))])

# def eval_config(asset, h, q, it_cols, tda_cols):
#     r = ret[asset]
#     vn = r.rolling(VOL_WINDOW).std() * ANN
#     vf = r.rolling(h).std().shift(-h) * ANN
#     vq = vn.expanding(min_periods=100).quantile(q)
#     y_raw = (vf > vq).astype(float).dropna()
#     vnr = vn.expanding(min_periods=100).rank(pct=True)

#     idx = (X_it_slim.index.intersection(X_tda_ri.index)
#            .intersection(vnr.index).intersection(y_raw.index))
#     valid = (X_it_slim.loc[idx].notna().all(1) &
#              X_tda_ri.loc[idx].notna().all(1) &
#              vnr.loc[idx].notna() & y_raw.loc[idx].notna())
#     idx = idx[valid]
#     if len(idx) < 500: return None

#     tr = idx < (split_ts - pd.Timedelta(minutes=30*PURGE_BARS))
#     te = idx >= split_ts
#     if tr.sum() < 200 or te.sum() < 50: return None

#     ytr_ = y_raw.loc[idx[tr]]; yte_ = y_raw.loc[idx[te]]
#     if ytr_.mean() < 0.03 or ytr_.mean() > 0.97: return None
#     if yte_.sum() < 3: return None

#     Xv_tr = vnr.loc[idx[tr]].values.reshape(-1,1)
#     Xv_te = vnr.loc[idx[te]].values.reshape(-1,1)

#     Xit_tr = X_it_slim[it_cols].loc[idx[tr]]
#     Xit_te = X_it_slim[it_cols].loc[idx[te]]
#     Xtd_tr = X_tda_ri[tda_cols].loc[idx[tr]]
#     Xtd_te = X_tda_ri[tda_cols].loc[idx[te]]

#     def fp(Xtr, Xte):
#         try:
#             m = _logit_quick(); m.fit(Xtr, ytr_)
#             return m.predict_proba(Xte)[:,1]
#         except: return np.full(len(Xte), float(ytr_.mean()))

#     Xtr_A = pd.DataFrame(Xv_tr, columns=["vr"])
#     Xte_A = pd.DataFrame(Xv_te, columns=["vr"])
#     sc_A = fp(Xtr_A, Xte_A)
#     sc_D = fp(pd.concat([Xtr_A.reset_index(drop=True),
#                           Xit_tr.reset_index(drop=True),
#                           Xtd_tr.reset_index(drop=True)], axis=1),
#               pd.concat([Xte_A.reset_index(drop=True),
#                           Xit_te.reset_index(drop=True),
#                           Xtd_te.reset_index(drop=True)], axis=1))

#     yn = yte_.values.astype(int)
#     try:
#         roc_a = roc_auc_score(yn, sc_A)
#         roc_d = roc_auc_score(yn, sc_D)
#         pr_a  = average_precision_score(yn, sc_A)
#         pr_d  = average_precision_score(yn, sc_D)
#         return {"roc_A":roc_a,"roc_D":roc_d,"pr_A":pr_a,"pr_D":pr_d,
#                 "droc":roc_d-roc_a,"dpr":pr_d-pr_a}
#     except: return None

# print(f"\n  {'актив':<6} {'h':<4} {'Q':>5}  "
#       f"{'ROC_A':>7} {'ROC_old':>8} {'ROC_new':>8}  "
#       f"{'Δold':>7} {'Δnew':>7}  лучше?")
# print("  " + "-"*65)

# for asset in ["BTC","ETH","DOGE","XRP","SOL","BNB"]:
#     for h, hl in [(96,"2d"),(192,"4d")]:
#         for q in [0.70, 0.80]:
#             r_old = eval_config(asset, h, q, OLD_IT,     OLD_TDA)
#             r_new = eval_config(asset, h, q, IT_RES_BEST, TDA_RES_BEST)
#             if r_old is None or r_new is None: continue
#             better = "new ↑" if r_new["droc"] > r_old["droc"] + 0.005 else (
#                      "old ↑" if r_old["droc"] > r_new["droc"] + 0.005 else "~")
#             print(f"  {asset:<6} {hl:<4} Q{q:.0%}"
#                   f"  {r_old['roc_A']:>7.4f}"
#                   f"  {r_old['roc_D']:>8.4f}"
#                   f"  {r_new['roc_D']:>8.4f}"
#                   f"  {r_old['droc']:>+7.4f}"
#                   f"  {r_new['droc']:>+7.4f}"
#                   f"  {better}")

# print(f"""
# Итог:
#   IT_RES_BEST  (conditional MI): {IT_RES_BEST}
#   TDA_RES_BEST (conditional MI): {TDA_RES_BEST}

#   Используй эти списки в stress_v2.py:
#     IT_COLS  = IT_RES_BEST
#     TDA_COLS = TDA_RES_BEST
#   Или передай через per-asset adaptive selection
#   (уже реализовано в stress_v2.py — там MI считается на train каждого актива).
# """)

# it_res_ranking  = it_res_df.copy()
# tda_res_ranking = tda_res_df.copy()
# ret_wide_loaded = ret_wide.copy()
# print("it_res_ranking, tda_res_ranking, ret_wide_loaded — сохранены в памяти")

  ШАГ 1: Загружаем дневные данные широкого юниверса
  yfinance недоступен: No module named 'yfinance'
  Используем ret как есть

  Итого активов для TDA: 9

  ШАГ 2: Residual MI — отбор фичей поверх vol_now
  MI(фича, стресс | vol_now) вместо MI(фича, стресс)

  Conditional MI(IT-фичи | vol_now) — топ-15:
  фича                               MI  (сравните с ~0.63 из сырого MI)
  coinfo4_min                    0.6574  ███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
  coinfo3_max                    0.6568  ███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
  pairwise_mi_std                0.6565  ███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
  coinfo4_max                    0.6563  ████████████████████████████████████████████████████████████

In [87]:
# import warnings; warnings.filterwarnings("ignore")
# import numpy as np
# import pandas as pd
# from sklearn.linear_model import LogisticRegression
# from sklearn.preprocessing import StandardScaler
# from sklearn.pipeline import Pipeline
# from sklearn.metrics import roc_auc_score, average_precision_score
# from sklearn.feature_selection import mutual_info_classif
# import matplotlib
# matplotlib.use("Agg")
# import matplotlib.pyplot as plt

# SPLIT_DATE = "2025-07-01"
# PURGE_BARS = 96
# QUANTILES  = [0.60, 0.70, 0.80, 0.90]
# HORIZONS   = {48:"1d", 96:"2d", 192:"4d"}
# ASSETS     = ["BTC","ETH","SOL","BNB","XRP","DOGE","ADA","AVAX","TRX"]
# VOL_WINDOW = 48
# ANN        = np.sqrt(365 * 48)
# N_IT_FEATS = 6    # топ IT-фич для каждого актива
# N_TDA_FEATS= 5    # топ TDA-фич

# def _logit(C=0.1):
#     return Pipeline([("sc", StandardScaler()),
#                      ("m", LogisticRegression(C=C, max_iter=2000,
#                                               class_weight="balanced",
#                                               solver="lbfgs"))])

# def safe_roc(y, s):
#     try: return float(roc_auc_score(y, s))
#     except: return np.nan

# def safe_pr(y, s):
#     try: return float(average_precision_score(y, s))
#     except: return np.nan

# def lift_at_k(y_true, score, k=0.05):
#     n = len(y_true); top_n = max(1, int(n*k))
#     top_idx = np.argsort(score)[::-1][:top_n]
#     base = float(np.mean(y_true))
#     return float(np.mean(y_true[top_idx])) / base if base > 1e-8 else np.nan

# print("Строим X_TDA...")
# tda_frames = [tda_A.add_prefix("A_")]
# for mode in ["I","II","C"]:
#     d = tda_B[mode]
#     df_m = pd.DataFrame({
#         f"B{mode}_tp_H0":  d["tp_H0"],  f"B{mode}_ent_H0": d["ent_H0"],
#         f"B{mode}_tp_H1":  d["tp_H1"],  f"B{mode}_ent_H1": d["ent_H1"],
#         f"B{mode}_L1_H0":  d["L1_H0"],  f"B{mode}_L1_H1":  d["L1_H1"],
#     }, index=pd.DatetimeIndex(d["times"]))
#     tda_frames.append(df_m)

# X_tda_raw = pd.concat(tda_frames, axis=1).sort_index()
# X_tda_ri  = X_tda_raw.reindex(ret.index, method="ffill")

# vol_now_all = {}
# for asset in ASSETS:
#     if asset in ret.columns:
#         vn = ret[asset].rolling(VOL_WINDOW).std() * ANN
#         vn_rank = vn.expanding(min_periods=100).rank(pct=True)
#         vol_now_all[asset] = vn_rank

# split_ts = pd.Timestamp(SPLIT_DATE)

# print("\n" + "="*65)
# print("  STRESS PREDICTION v2")
# print("  A=vol_now | B=vol+IT | C=vol+TDA | D=vol+IT+TDA")
# print("  Фичи отбираются per-asset по MI на train")
# print("="*65)

# rows = []

# for asset in ASSETS:
#     if asset not in ret.columns: continue
#     r       = ret[asset]
#     vol_now = ret[asset].rolling(VOL_WINDOW).std() * ANN
#     vn_rank = vol_now_all[asset]

#     for h, h_label in HORIZONS.items():
#         vol_fwd = r.rolling(h).std().shift(-h) * ANN

#         for q in QUANTILES:
#             vol_q  = vol_now.expanding(min_periods=100).quantile(q)
#             y_raw  = (vol_fwd > vol_q).astype(float).dropna()

#             idx = (X_vol.index.intersection(X_it_slim.index)
#                    .intersection(X_tda_ri.index)
#                    .intersection(vn_rank.index)
#                    .intersection(y_raw.index))
#             valid = (X_vol.loc[idx].notna().all(1) &
#                      X_it_slim.loc[idx].notna().all(1) &
#                      X_tda_ri.loc[idx].notna().all(1) &
#                      vn_rank.loc[idx].notna() &
#                      y_raw.loc[idx].notna())
#             idx = idx[valid]
#             if len(idx) < 1000: continue

#             tr = idx < (split_ts - pd.Timedelta(minutes=30*PURGE_BARS))
#             te = idx >= split_ts
#             if tr.sum() < 480 or te.sum() < 100: continue

#             ytr = y_raw.loc[idx[tr]]; yte = y_raw.loc[idx[te]]
#             pos_tr = float(ytr.mean()); pos_te = float(yte.mean())
#             if pos_tr < 0.03 or pos_tr > 0.97: continue
#             if yte.sum() < 5: continue

#             Xvol_tr = vn_rank.loc[idx[tr]].values.reshape(-1,1)
#             Xvol_te = vn_rank.loc[idx[te]].values.reshape(-1,1)
#             Xvol_tr_df = pd.DataFrame(Xvol_tr, index=idx[tr], columns=["vol_rank"])
#             Xvol_te_df = pd.DataFrame(Xvol_te, index=idx[te], columns=["vol_rank"])

#             Xi_tr_all = X_it_slim.loc[idx[tr]]
#             Xt_tr_all = X_tda_ri.loc[idx[tr]]

#             mi_it  = mutual_info_classif(Xi_tr_all.values, ytr.values, random_state=42)
#             mi_tda = mutual_info_classif(Xt_tr_all.values, ytr.values, random_state=42)

#             it_cols  = pd.Series(mi_it,  index=Xi_tr_all.columns).nlargest(N_IT_FEATS).index.tolist()
#             tda_cols = pd.Series(mi_tda, index=Xt_tr_all.columns).nlargest(N_TDA_FEATS).index.tolist()

#             Xit_tr = X_it_slim[it_cols].loc[idx[tr]]
#             Xit_te = X_it_slim[it_cols].loc[idx[te]]
#             Xtd_tr = X_tda_ri[tda_cols].loc[idx[tr]]
#             Xtd_te = X_tda_ri[tda_cols].loc[idx[te]]

#             yte_np = yte.values.astype(int)

#             def fit_predict(Xtr, Xte):
#                 try:
#                     m = _logit(); m.fit(Xtr, ytr)
#                     return m.predict_proba(Xte)[:,1]
#                 except: return np.full(len(Xte), ytr.mean())

#             sc_A = fit_predict(Xvol_tr_df, Xvol_te_df)

#             sc_B = fit_predict(
#                 pd.concat([Xvol_tr_df, Xit_tr], axis=1),
#                 pd.concat([Xvol_te_df, Xit_te], axis=1))

#             sc_C = fit_predict(
#                 pd.concat([Xvol_tr_df, Xtd_tr], axis=1),
#                 pd.concat([Xvol_te_df, Xtd_te], axis=1))

#             sc_D = fit_predict(
#                 pd.concat([Xvol_tr_df, Xit_tr, Xtd_tr], axis=1),
#                 pd.concat([Xvol_te_df, Xit_te, Xtd_te], axis=1))

#             rows.append({
#                 "asset":  asset, "horizon": h_label, "h": h, "q": q,
#                 "pos_tr": round(pos_tr, 3), "pos_te": round(pos_te, 3),
#                 "roc_A": round(safe_roc(yte_np, sc_A), 4),
#                 "roc_B": round(safe_roc(yte_np, sc_B), 4),
#                 "roc_C": round(safe_roc(yte_np, sc_C), 4),
#                 "roc_D": round(safe_roc(yte_np, sc_D), 4),
#                 "pr_A":  round(safe_pr(yte_np, sc_A),  4),
#                 "pr_B":  round(safe_pr(yte_np, sc_B),  4),
#                 "pr_C":  round(safe_pr(yte_np, sc_C),  4),
#                 "pr_D":  round(safe_pr(yte_np, sc_D),  4),
#                 "droc_B": round(safe_roc(yte_np,sc_B)-safe_roc(yte_np,sc_A), 4),
#                 "droc_C": round(safe_roc(yte_np,sc_C)-safe_roc(yte_np,sc_A), 4),
#                 "droc_D": round(safe_roc(yte_np,sc_D)-safe_roc(yte_np,sc_A), 4),
#                 "dpr_B":  round(safe_pr(yte_np,sc_B)-safe_pr(yte_np,sc_A), 4),
#                 "dpr_C":  round(safe_pr(yte_np,sc_C)-safe_pr(yte_np,sc_A), 4),
#                 "dpr_D":  round(safe_pr(yte_np,sc_D)-safe_pr(yte_np,sc_A), 4),
#                 "lift5_A": round(lift_at_k(yte_np, sc_A, 0.05), 3),
#                 "lift5_D": round(lift_at_k(yte_np, sc_D, 0.05), 3),
#                 "n_train": int(tr.sum()), "n_test": int(te.sum()),
#             })

# df = pd.DataFrame(rows)
# print(f"Всего строк: {len(df)}")

# print(f"\n{'='*78}")
# print(f"  ROC-AUC — агрегат по порогам (среднее по активам и горизонтам)")
# print(f"  A=vol_now | B=vol+IT | C=vol+TDA | D=vol+IT+TDA")
# print(f"{'='*78}")
# print(f"\n  {'q':>5} {'pos%':>5} "
#       f"{'ROC_A':>7} {'ROC_B':>7} {'ROC_C':>7} {'ROC_D':>7} "
#       f"{'ΔROC_B':>8} {'ΔROC_C':>8} {'ΔROC_D':>8} "
#       f"{'D>A':>6} {'Lift5_D':>9}")
# print("  " + "-"*82)

# for q in QUANTILES:
#     sub = df[df.q==q]
#     if sub.empty: continue
#     beats = (sub["droc_D"] > 0.01).sum()
#     print(f"  Q{q:.0%}"
#           f" {sub['pos_te'].mean():>4.0%}"
#           f" {sub['roc_A'].mean():>7.4f}"
#           f" {sub['roc_B'].mean():>7.4f}"
#           f" {sub['roc_C'].mean():>7.4f}"
#           f" {sub['roc_D'].mean():>7.4f}"
#           f" {sub['droc_B'].mean():>+8.4f}"
#           f" {sub['droc_C'].mean():>+8.4f}"
#           f" {sub['droc_D'].mean():>+8.4f}"
#           f" {beats:>4}/{len(sub)}"
#           f" {sub['lift5_D'].mean():>9.3f}")

# print(f"\n{'='*78}")
# print(f"  ПО АКТИВАМ — порог Q80  |  ROC-AUC + PR-AUC")
# print(f"{'='*78}")
# sub80 = df[df.q==0.80]
# print(f"\n  {'актив':<6} {'pos%':>5} "
#       f"{'ROC_A':>7} {'ROC_B':>7} {'ROC_C':>7} {'ROC_D':>7} "
#       f"{'ΔROC_D':>8} "
#       f"{'PR_A':>7} {'PR_D':>7} {'ΔPR_D':>8}  лучшая")
# print("  " + "-"*85)

# for asset in ASSETS:
#     g = sub80[sub80.asset==asset]
#     if g.empty: continue
#     best_roc = max({"A":g["roc_A"].mean(),"B":g["roc_B"].mean(),
#                     "C":g["roc_C"].mean(),"D":g["roc_D"].mean()},
#                    key=lambda k: {"A":g["roc_A"].mean(),"B":g["roc_B"].mean(),
#                                    "C":g["roc_C"].mean(),"D":g["roc_D"].mean()}[k])
#     print(f"  {asset:<6}"
#           f" {g['pos_te'].mean():>4.0%}"
#           f" {g['roc_A'].mean():>7.4f}"
#           f" {g['roc_B'].mean():>7.4f}"
#           f" {g['roc_C'].mean():>7.4f}"
#           f" {g['roc_D'].mean():>7.4f}"
#           f" {g['droc_D'].mean():>+8.4f}"
#           f" {g['pr_A'].mean():>7.4f}"
#           f" {g['pr_D'].mean():>7.4f}"
#           f" {g['dpr_D'].mean():>+8.4f}"
#           f"  → {best_roc}")

# print(f"\n{'='*78}")
# print(f"  ТОП-10 по ΔROC (конфиг D = vol+IT+TDA)")
# print(f"{'='*78}")
# top10 = df.nlargest(10, "droc_D")
# print(f"\n  {'актив':<6} {'h':<4} {'Q':>5} {'pos%':>5} "
#       f"{'ROC_A':>7} {'ROC_D':>7} {'ΔROC':>8} "
#       f"{'PR_A':>7} {'PR_D':>7} {'ΔPR':>8} {'Lift5':>7}")
# print("  " + "-"*72)
# for _, row in top10.iterrows():
#     print(f"  {row['asset']:<6} {row['horizon']:<4} Q{row['q']:.0%}"
#           f" {row['pos_te']:>4.0%}"
#           f" {row['roc_A']:>7.4f}"
#           f" {row['roc_D']:>7.4f}"
#           f" {row['droc_D']:>+8.4f}"
#           f" {row['pr_A']:>7.4f}"
#           f" {row['pr_D']:>7.4f}"
#           f" {row['dpr_D']:>+8.4f}"
#           f" {row['lift5_D']:>7.3f}")

# print(f"\n{'='*78}")
# print(f"  СВОДКА")
# print(f"{'='*78}")
# n = len(df)
# for cfg, col_roc, col_pr in [
#     ("B: vol+IT",     "droc_B","dpr_B"),
#     ("C: vol+TDA",    "droc_C","dpr_C"),
#     ("D: vol+IT+TDA", "droc_D","dpr_D"),
# ]:
#     beats_roc = (df[col_roc] > 0.01).sum()
#     beats_pr  = (df[col_pr]  > 0.01).sum()
#     print(f"  {cfg:<18}: ΔROC={df[col_roc].mean():>+.4f}  "
#           f"ROC↑ {beats_roc:>2}/{n}  "
#           f"ΔPR={df[col_pr].mean():>+.4f}  PR↑ {beats_pr:>2}/{n}")

# print("\nСтроим графики...")
# fig, axes = plt.subplots(2, 2, figsize=(14, 10))
# q_labels = [f"Q{q:.0%}" for q in QUANTILES]
# colors = {"A":"#9E9E9E","B":"#2196F3","C":"#FF9800","D":"#4CAF50"}

# ax = axes[0,0]
# for k in ["A","B","C","D"]:
#     vals = [df[df.q==q][f"roc_{k}"].mean() for q in QUANTILES]
#     ls = "--" if k=="C" else "-"
#     label = {"A":"vol_now","B":"vol+IT","C":"vol+TDA","D":"vol+IT+TDA"}[k]
#     ax.plot(q_labels, vals, "o"+ls, color=colors[k], lw=2, label=label, ms=7)
# ax.axhline(0.5, color="gray", lw=0.8, ls=":", alpha=0.5)
# ax.set_ylabel("ROC-AUC"); ax.set_xlabel("Порог стресса")
# ax.set_title("ROC-AUC по порогам\n(среднее по 9 активам и 3 горизонтам)", fontsize=10)
# ax.legend(fontsize=8); ax.grid(alpha=0.2)
# ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

# ax = axes[0,1]
# x = np.arange(len(QUANTILES)); w = 0.25
# for ci, k in enumerate(["B","C","D"]):
#     vals = [df[df.q==q][f"droc_{k}"].mean() for q in QUANTILES]
#     label = {"B":"+IT","C":"+TDA","D":"+IT+TDA"}[k]
#     ax.bar(x+(ci-1)*w, vals, w*0.9, label=label, color=colors[k], alpha=0.85)
# ax.axhline(0, color="black", lw=0.8)
# ax.set_xticks(x); ax.set_xticklabels(q_labels)
# ax.set_ylabel("ΔROC-AUC vs vol_now baseline")
# ax.set_title("Инкрементальный ΔROC над vol_now\n(по порогам)", fontsize=10)
# ax.legend(fontsize=8); ax.grid(alpha=0.2, axis="y")
# ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

# ax = axes[1,0]
# asset_order = sub80.groupby("asset")["roc_D"].mean().sort_values(ascending=False).index.tolist()
# x3 = np.arange(len(asset_order)); w3 = 0.2
# for ci, k in enumerate(["A","B","C","D"]):
#     vals = [sub80[sub80.asset==a][f"roc_{k}"].mean() for a in asset_order]
#     label = {"A":"vol_now","B":"vol+IT","C":"vol+TDA","D":"vol+IT+TDA"}[k]
#     ax.bar(x3+(ci-1.5)*w3, vals, w3*0.9, label=label, color=colors[k], alpha=0.85)
# ax.axhline(0.5, color="gray", lw=0.8, ls=":", alpha=0.5)
# ax.set_xticks(x3); ax.set_xticklabels(list(asset_order), fontsize=9)
# ax.set_ylabel("ROC-AUC")
# ax.set_title("ROC-AUC по активам (Q80)\n(среднее по горизонтам)", fontsize=10)
# ax.legend(fontsize=8); ax.grid(alpha=0.2, axis="y")
# ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

# ax = axes[1,1]
# pivot = df.pivot_table(index="asset", columns="q",
#                        values="roc_D", aggfunc="mean")
# pivot = pivot.reindex(ASSETS)[QUANTILES]
# im = ax.imshow(pivot.values, cmap="RdYlGn", vmin=0.50, vmax=0.85, aspect="auto")
# ax.set_xticks(range(len(QUANTILES))); ax.set_xticklabels(q_labels)
# ax.set_yticks(range(len(ASSETS))); ax.set_yticklabels(ASSETS)
# for i in range(len(ASSETS)):
#     for j in range(len(QUANTILES)):
#         val = pivot.values[i,j]
#         if np.isfinite(val):
#             col = "white" if val > 0.72 else "black"
#             ax.text(j, i, f"{val:.3f}", ha="center", va="center",
#                     fontsize=8, color=col)
# plt.colorbar(im, ax=ax, label="ROC-AUC (vol+IT+TDA)")
# ax.set_title("ROC-AUC heatmap (vol+IT+TDA)\nпо активам × порогам", fontsize=10)

# plt.suptitle(
#     "Stress prediction v2: A=vol_now | B=vol+IT | C=vol+TDA | D=vol+IT+TDA\n"
#     f"Holdout {SPLIT_DATE} →  |  Per-asset feature selection  |  Горизонты: 1d, 2d, 4d",
#     fontsize=11, y=1.01)
# plt.tight_layout()
# plt.savefig("fig_stress_v2.png", bbox_inches="tight", dpi=150)
# plt.savefig("fig_stress_v2.pdf", bbox_inches="tight", dpi=150)
# plt.show()
# print("  Saved: fig_stress_v2.png / .pdf")

# stress_v2 = df.copy()
# print(f"\nstress_v2 сохранён в памяти ({len(df)} строк)")

Строим X_TDA...

  STRESS PREDICTION v2
  A=vol_now | B=vol+IT | C=vol+TDA | D=vol+IT+TDA
  Фичи отбираются per-asset по MI на train
Всего строк: 91

  ROC-AUC — агрегат по порогам (среднее по активам и горизонтам)
  A=vol_now | B=vol+IT | C=vol+TDA | D=vol+IT+TDA

      q  pos%   ROC_A   ROC_B   ROC_C   ROC_D   ΔROC_B   ΔROC_C   ΔROC_D    D>A   Lift5_D
  ----------------------------------------------------------------------------------
  Q60%  45%  0.6893  0.6846  0.7049  0.7094  -0.0047  +0.0156  +0.0200   15/26     1.701
  Q70%  31%  0.6863  0.6863  0.6931  0.7022  +0.0000  +0.0069  +0.0160   12/25     1.999
  Q80%  16%  0.6622  0.6872  0.6926  0.7178  +0.0251  +0.0305  +0.0557   18/24     2.623
  Q90%   5%  0.6261  0.6322  0.6227  0.6452  +0.0061  -0.0034  +0.0190    8/16     0.762

  ПО АКТИВАМ — порог Q80  |  ROC-AUC + PR-AUC

  актив   pos%   ROC_A   ROC_B   ROC_C   ROC_D   ΔROC_D    PR_A    PR_D    ΔPR_D  лучшая
  ----------------------------------------------------------------

In [101]:
# import warnings; warnings.filterwarnings("ignore")
# import numpy as np
# import pandas as pd
# from sklearn.metrics import mutual_info_score

# IT_WINDOW = 192   # W bars (~4 days)
# IT_K      = 4     # step size
# IT_BINS   = 10    # bins for histogram estimator

# ASSETS = ["BTC","ETH","SOL","BNB","XRP","DOGE","ADA","AVAX","TRX"]
# VOL_WINDOW = 48
# ANN = np.sqrt(365 * 48)

# print("Building missing IT features:")
# print("  1. Rolling Shannon entropy of the index")
# print("  2. NMI(r_t, r_{t-1}) — lag-1 normalised MI")

# n = len(ret)
# idx_all = ret.index

# def _rk(v, bins):
#     """Rank-quantise values into the specified number of bins."""
#     r = np.argsort(np.argsort(v))
#     return (r * bins // len(v)).clip(0, bins - 1).astype(np.int32)

# def _ffill(arr):
#     mask = np.isfinite(arr)
#     idx  = np.where(mask, np.arange(len(arr)), 0)
#     np.maximum.accumulate(idx, out=idx)
#     return arr[idx]

# def rolling_entropy(series, W=IT_WINDOW, k=IT_K, bins=IT_BINS):
#     """
#     Rolling Shannon entropy of the return distribution.
#     H = -sum(p_i * log(p_i))
#     Uses rank quantisation into the specified number of bins.
#     """
#     x = series.values.astype(float)
#     n = len(x)
#     starts = np.arange(0, n - W + 1, k)
#     ends   = starts + W
#     res    = np.full(n, np.nan)

#     for s, e in zip(starts, ends):
#         xw = x[s:e]
#         if not np.any(np.isfinite(xw)):
#             continue
#         bx = _rk(xw, bins)
#         counts = np.bincount(bx, minlength=bins).astype(float)
#         p = counts / counts.sum()
#         p = p[p > 0]
#         res[e - 1] = float(-np.sum(p * np.log(p)))

#     return pd.Series(_ffill(res), index=series.index)

# def rolling_nmi_lag1(series, W=IT_WINDOW, k=IT_K, bins=IT_BINS):
#     """
#     NMI between r_t and r_{t-1} (lag-1 auto-MI).
#     NMI = 1 - exp(-2 * MI)  — normalisation following supervisor notebook.
#     """
#     x = series.values.astype(float)
#     n = len(x)
#     starts = np.arange(0, n - W + 1, k)
#     ends   = starts + W
#     res    = np.full(n, np.nan)

#     for s, e in zip(starts, ends):
#         xw = x[s:e]
#         if not np.any(np.isfinite(xw)):
#             continue
#         x_now  = xw[1:]
#         x_lag1 = xw[:-1]
#         bx = _rk(x_now,  bins)
#         by = _rk(x_lag1, bins)
#         c = np.histogram2d(bx, by, bins=bins,
#                             range=[[0,bins],[0,bins]])[0]
#         mi = float(mutual_info_score(None, None, contingency=c))
#         nmi = 1.0 - np.exp(-2.0 * mi)   # normalisation
#         res[e - 1] = nmi

#     return pd.Series(_ffill(res), index=series.index)

# print("\nBuilding equal-weighted index...")
# available = [a for a in ASSETS if a in ret.columns]
# index_ret = ret[available].mean(axis=1)   # equal-weighted
# print(f"  Assets: {len(available)}, bars: {len(index_ret)}")

# print("  Computing rolling entropy of the index...")
# ent_index = rolling_entropy(index_ret)
# ent_index.name = "entropy_index"
# print(f"  entropy_index: mean={ent_index.mean():.4f}  std={ent_index.std():.4f}")

# print("  Computing lag-1 NMI of the index...")
# nmi_index = rolling_nmi_lag1(index_ret)
# nmi_index.name = "nmi_lag1_index"
# print(f"  nmi_lag1_index: mean={nmi_index.mean():.4f}  std={nmi_index.std():.4f}")

# print("\nComputing lag-1 NMI per asset...")
# nmi_per_asset = {}
# for asset in available:
#     s = rolling_nmi_lag1(ret[asset])
#     s.name = f"nmi_lag1_{asset}"
#     nmi_per_asset[asset] = s
#     print(f"  nmi_lag1_{asset}: mean={s.mean():.4f}")

# nmi_df = pd.DataFrame(nmi_per_asset)
# nmi_mean = nmi_df.mean(axis=1);  nmi_mean.name = "nmi_lag1_mean"
# nmi_std  = nmi_df.std(axis=1);   nmi_std.name  = "nmi_lag1_std"
# nmi_max  = nmi_df.max(axis=1);   nmi_max.name  = "nmi_lag1_max"

# print("\n" + "="*55)
# print("  CORRELATION OF NEW FEATURES WITH VOL_BTC (full period)")
# print("="*55)

# vol_btc = ret["BTC"].rolling(VOL_WINDOW).std() * ANN

# new_feats = {
#     "entropy_index":  ent_index,
#     "nmi_lag1_index": nmi_index,
#     "nmi_lag1_mean":  nmi_mean,
#     "nmi_lag1_std":   nmi_std,
#     "nmi_lag1_max":   nmi_max,
# }

# if "X_it_slim" in dir():
#     print(f"\n  Old slim IT features (for comparison):")
#     for col in ["coinfo3_std","kl_BNB","pairwise_mi_mean","coinfo4_std"]:
#         if col in X_it_slim.columns:
#             c = X_it_slim[col].corr(vol_btc)
#             print(f"    {col:<28} corr = {c:>+.4f}")

# print(f"\n  New features:")
# for name, s in new_feats.items():
#     c = s.corr(vol_btc)
#     if not np.isfinite(c):
#         print(f"    {name:<28} corr = NaN  (constant feature)")
#         continue
#     bar = "█" * max(0, int(abs(c) * 25))
#     sign = "+" if c >= 0 else "-"
#     print(f"    {name:<28} corr = {c:>+.4f}  {sign}{bar}")

# print("\n" + "="*55)
# print("  ADDING TO X_it_slim")
# print("="*55)

# new_df = pd.DataFrame(new_feats)

# if "X_it_slim" in dir():
#     common_idx = X_it_slim.index.intersection(new_df.index)
#     X_it_extended = pd.concat([
#         X_it_slim.loc[common_idx],
#         new_df.loc[common_idx]
#     ], axis=1)

#     print(f"  X_it_slim:     {X_it_slim.shape[1]} features")
#     print(f"  New features:  {new_df.shape[1]} features")
#     print(f"  X_it_extended: {X_it_extended.shape[1]} features")
#     print(f"  Features: {list(X_it_extended.columns)}")
# else:
#     X_it_extended = new_df.copy()
#     print(f"  X_it_slim not found in memory, saving new features only")

# nan_pct = X_it_extended.isna().mean()
# print(f"\n  NaN share for new features:")
# for col in new_feats.keys():
#     if col in X_it_extended.columns:
#         val = nan_pct.loc[col] if col in nan_pct.index else nan_pct[col]
#         val = float(val.iloc[0]) if hasattr(val, "iloc") else float(val)
#         print(f"    {col:<28} {val:.3%}")

# print(f"\nX_it_extended saved to memory ({X_it_extended.shape[1]} features)")
# print(f"entropy_index, nmi_index, nmi_per_asset — saved to memory")

# print("\n" + "="*60)
# print("  SELECTION: permutation importance on train (BTC h=4d logvol)")
# print("="*60)

# import matplotlib
# matplotlib.use("Agg")
# import matplotlib.pyplot as plt
# from sklearn.linear_model import RidgeCV
# from sklearn.preprocessing import StandardScaler
# from sklearn.pipeline import Pipeline
# from sklearn.model_selection import KFold, cross_val_score

# SPLIT_DATE = "2025-07-01"
# PURGE_BARS = 96
# RNG2 = np.random.default_rng(42)
# N_PERM = 15

# def _ridge():
#     return Pipeline([("sc", StandardScaler()),
#                      ("m", RidgeCV(alphas=[0.01,0.1,1,10,100]))])

# r_btc   = ret["BTC"]
# vol_now = r_btc.rolling(48).std() * ANN
# vol_fwd = r_btc.rolling(192).std().shift(-192) * ANN
# y_logvol = np.log(vol_fwd.clip(lower=1e-8)).dropna()

# split_ts = pd.Timestamp(SPLIT_DATE)

# idx = X_vol.index.intersection(X_it_extended.index).intersection(y_logvol.index)
# valid = (X_vol.loc[idx].notna().all(1) &
#          X_it_extended.loc[idx].notna().all(1) &
#          y_logvol.loc[idx].notna())
# idx = idx[valid]
# tr = idx < (split_ts - pd.Timedelta(minutes=30 * PURGE_BARS))
# idx_tr = idx[tr]

# Xv_tr = X_vol.loc[idx_tr]
# ytr   = y_logvol.loc[idx_tr]

# ck   = Xv_tr.corrwith(ytr).abs().dropna()
# top3 = ck.nlargest(3).index.tolist()
# mb   = _ridge(); mb.fit(Xv_tr[top3], ytr)
# res_tr = ytr.values - mb.predict(Xv_tr[top3])

# Xi_tr = X_it_extended.loc[idx_tr]
# Xi_arr = Xi_tr.values
# feat_names = list(Xi_tr.columns)

# kf = KFold(n_splits=5, shuffle=False)
# base_cv = float(np.mean(cross_val_score(_ridge(), Xi_arr, res_tr,
#                                           cv=kf, scoring="r2")))
# print(f"  Baseline CV R² (all {len(feat_names)} features): {base_cv:.4f}")

# perm_scores = {}
# print(f"  Computing importance ({N_PERM} permutations × {len(feat_names)} features)...")
# for fi, feat in enumerate(feat_names):
#     deltas = []
#     for _ in range(N_PERM):
#         Xp = Xi_arr.copy()
#         Xp[:, fi] = RNG2.permutation(Xp[:, fi])
#         sc = float(np.mean(cross_val_score(_ridge(), Xp, res_tr,
#                                             cv=kf, scoring="r2")))
#         deltas.append(sc - base_cv)
#     perm_scores[feat] = float(np.mean(deltas))

# sorted_feats = sorted(perm_scores.items(), key=lambda x: x[1])  # most useful at top

# NEW_FEATS  = list(new_feats.keys())
# SLIM_FEATS = [c for c in feat_names if c not in NEW_FEATS]

# def cat_color(f):
#     if f in NEW_FEATS: return "#E91E63"   # pink — new features
#     return "#2196F3"                       # blue — old slim features

# print(f"\n  Feature ranking (Δ < 0 = useful):")
# print(f"  {'feature':<28} {'Δ CV R²':>9}  status")
# print("  " + "-"*50)
# for feat, delta in sorted_feats:
#     is_new = " [NEW]" if feat in NEW_FEATS else ""
#     status = "✓ useful" if delta < -0.002 else ("✗ harmful" if delta > 0.002 else "~ neutral")
#     print(f"  {feat:<28} {delta:>+9.4f}  {status}{is_new}")

# good_feats = [f for f, d in sorted_feats if d <= 0.003]
# bad_feats  = [f for f, d in sorted_feats if d >  0.003]

# print(f"\n  Useful/neutral: {len(good_feats)} features")
# print(f"  Harmful:        {len(bad_feats)} features: {bad_feats}")

# new_good = [f for f in good_feats if f in NEW_FEATS]
# new_bad  = [f for f in bad_feats  if f in NEW_FEATS]
# print(f"\n  Among new features:")
# print(f"    Useful:  {new_good}")
# print(f"    Harmful: {new_bad}")

# X_it_extended = X_it_extended.loc[:, ~X_it_extended.columns.duplicated()]

# X_it_final = X_it_extended[[f for f in good_feats if f in X_it_extended.columns]].copy()
# print(f"\n  X_it_final: {X_it_final.shape[1]} features")
# print(f"  {list(X_it_final.columns)}")

# fig, axes = plt.subplots(1, 2, figsize=(14, max(6, len(feat_names) * 0.28)))

# ax = axes[0]
# feats_plot  = [f for f, _ in sorted_feats]
# deltas_plot = [perm_scores[f] for f in feats_plot]
# colors_bar  = [cat_color(f) for f in feats_plot]

# y_pos = np.arange(len(feats_plot))
# ax.barh(y_pos, deltas_plot, color=colors_bar, alpha=0.85, height=0.75)
# ax.axvline(0,      color="black", lw=0.8)
# ax.axvline(-0.002, color="green", lw=0.8, ls=":", alpha=0.7, label="usefulness threshold")
# ax.axvline(+0.002, color="red",   lw=0.8, ls=":", alpha=0.7, label="harmfulness threshold")
# ax.set_yticks(y_pos)
# ax.set_yticklabels(
#     [f + " ★" if f in NEW_FEATS else f for f in feats_plot],
#     fontsize=8
# )
# ax.set_xlabel("Δ CV R² upon permutation\n(< 0 = feature is useful)")
# ax.set_title("Permutation importance\n(BTC, h=4d, logvol residuals)", fontsize=10)
# ax.legend(fontsize=8)
# ax.grid(alpha=0.2, axis="x")
# ax.spines["top"].set_visible(False)
# ax.spines["right"].set_visible(False)

# from matplotlib.patches import Patch
# legend_elems = [
#     Patch(facecolor="#2196F3", alpha=0.85, label="Old slim IT features"),
#     Patch(facecolor="#E91E63", alpha=0.85, label="New features (entropy/NMI) ★"),
# ]
# ax.legend(handles=legend_elems, fontsize=8, loc="lower right")

# ax = axes[1]
# corr_vals = []
# corr_names = []
# corr_colors = []
# for feat in feats_plot:
#     if feat in X_it_extended.columns:
#         c = X_it_extended[feat].corr(vol_btc)
#         if np.isfinite(c):
#             corr_vals.append(c)
#             corr_names.append(feat + (" ★" if feat in NEW_FEATS else ""))
#             corr_colors.append(cat_color(feat))

# y_pos2 = np.arange(len(corr_vals))
# ax.barh(y_pos2, corr_vals, color=corr_colors, alpha=0.85, height=0.75)
# ax.axvline(0, color="black", lw=0.8)
# ax.axvline(+0.10, color="green", lw=0.8, ls=":", alpha=0.5)
# ax.axvline(-0.10, color="green", lw=0.8, ls=":", alpha=0.5)
# ax.set_yticks(y_pos2)
# ax.set_yticklabels(corr_names, fontsize=8)
# ax.set_xlabel("Correlation with vol_BTC (full period)")
# ax.set_title("Feature correlation with vol_BTC\n(|corr| > 0.10 = significant)", fontsize=10)
# ax.grid(alpha=0.2, axis="x")
# ax.spines["top"].set_visible(False)
# ax.spines["right"].set_visible(False)

# plt.suptitle(
#     "IT feature usefulness: old slim + new (entropy, NMI)\n"
#     "★ = new features",
#     fontsize=11, y=1.01
# )
# plt.tight_layout()
# plt.savefig("fig_new_it_features.png", bbox_inches="tight", dpi=150)
# plt.savefig("fig_new_it_features.pdf", bbox_inches="tight", dpi=150)
# plt.show()
# print("  Saved: fig_new_it_features.png / .pdf")

# X_it_final = X_it_extended[good_feats].copy()
# print(f"\nX_it_final saved to memory ({X_it_final.shape[1]} features)")
# print(f"Use X_it_final instead of X_it_slim in the holdout pipeline")

Building missing IT features:
  1. Rolling Shannon entropy of the index
  2. NMI(r_t, r_{t-1}) — lag-1 normalised MI

Building equal-weighted index...
  Assets: 9, bars: 46743
  Computing rolling entropy of the index...
  entropy_index: mean=2.3024  std=0.0000
  Computing lag-1 NMI of the index...
  nmi_lag1_index: mean=0.4114  std=0.0466

Computing lag-1 NMI per asset...
  nmi_lag1_BTC: mean=0.4186
  nmi_lag1_ETH: mean=0.4129
  nmi_lag1_SOL: mean=0.4072
  nmi_lag1_BNB: mean=0.4102
  nmi_lag1_XRP: mean=0.4130
  nmi_lag1_DOGE: mean=0.4094
  nmi_lag1_ADA: mean=0.4079
  nmi_lag1_AVAX: mean=0.4071
  nmi_lag1_TRX: mean=0.4168

  CORRELATION OF NEW FEATURES WITH VOL_BTC (full period)

  Old slim IT features (for comparison):
    coinfo3_std                  corr = -0.0306
    kl_BNB                       corr = +0.1849
    pairwise_mi_mean             corr = +0.3385
    coinfo4_std                  corr = -0.0318

  New features:
    entropy_index                corr = NaN  (constant feature

In [ ]:
# import warnings; warnings.filterwarnings("ignore")
# import numpy as np
# import pandas as pd
# from sklearn.metrics import mutual_info_score

# IT_WINDOW = 192   # W bars (~4 days)
# IT_K      = 4     # step size
# IT_BINS   = 10    # bins for histogram estimator

# ASSETS = ["BTC","ETH","SOL","BNB","XRP","DOGE","ADA","AVAX","TRX"]
# VOL_WINDOW = 48
# ANN = np.sqrt(365 * 48)

# print("Building missing IT features:")
# print("  1. Rolling Shannon entropy of the index")
# print("  2. NMI(r_t, r_{t-1}) — lag-1 normalised MI")

# n = len(ret)
# idx_all = ret.index

# def _rk(v, bins):
#     """Rank-quantise values into the specified number of bins."""
#     r = np.argsort(np.argsort(v))
#     return (r * bins // len(v)).clip(0, bins - 1).astype(np.int32)

# def _ffill(arr):
#     mask = np.isfinite(arr)
#     idx  = np.where(mask, np.arange(len(arr)), 0)
#     np.maximum.accumulate(idx, out=idx)
#     return arr[idx]

# def rolling_entropy(series, W=IT_WINDOW, k=IT_K, bins=IT_BINS):
#     """
#     Rolling Shannon entropy of the return distribution.
#     H = -sum(p_i * log(p_i))
#     Uses rank quantisation into the specified number of bins.
#     """
#     x = series.values.astype(float)
#     n = len(x)
#     starts = np.arange(0, n - W + 1, k)
#     ends   = starts + W
#     res    = np.full(n, np.nan)

#     for s, e in zip(starts, ends):
#         xw = x[s:e]
#         if not np.any(np.isfinite(xw)):
#             continue
#         bx = _rk(xw, bins)
#         counts = np.bincount(bx, minlength=bins).astype(float)
#         p = counts / counts.sum()
#         p = p[p > 0]
#         res[e - 1] = float(-np.sum(p * np.log(p)))

#     return pd.Series(_ffill(res), index=series.index)

# def rolling_nmi_lag1(series, W=IT_WINDOW, k=IT_K, bins=IT_BINS):
#     """
#     NMI between r_t and r_{t-1} (lag-1 auto-MI).
#     NMI = 1 - exp(-2 * MI)  — normalisation following supervisor notebook.
#     """
#     x = series.values.astype(float)
#     n = len(x)
#     starts = np.arange(0, n - W + 1, k)
#     ends   = starts + W
#     res    = np.full(n, np.nan)

#     for s, e in zip(starts, ends):
#         xw = x[s:e]
#         if not np.any(np.isfinite(xw)):
#             continue
#         x_now  = xw[1:]
#         x_lag1 = xw[:-1]
#         bx = _rk(x_now,  bins)
#         by = _rk(x_lag1, bins)
#         c = np.histogram2d(bx, by, bins=bins,
#                             range=[[0,bins],[0,bins]])[0]
#         mi = float(mutual_info_score(None, None, contingency=c))
#         nmi = 1.0 - np.exp(-2.0 * mi)   
#         res[e - 1] = nmi

#     return pd.Series(_ffill(res), index=series.index)

# print("\nBuilding equal-weighted index...")
# available = [a for a in ASSETS if a in ret.columns]
# index_ret = ret[available].mean(axis=1)   
# print(f"  Assets: {len(available)}, bars: {len(index_ret)}")

# print("  Computing rolling entropy of the index...")
# ent_index = rolling_entropy(index_ret)
# ent_index.name = "entropy_index"
# print(f"  entropy_index: mean={ent_index.mean():.4f}  std={ent_index.std():.4f}")

# print("  Computing lag-1 NMI of the index...")
# nmi_index = rolling_nmi_lag1(index_ret)
# nmi_index.name = "nmi_lag1_index"
# print(f"  nmi_lag1_index: mean={nmi_index.mean():.4f}  std={nmi_index.std():.4f}")

# print("\nComputing lag-1 NMI per asset...")
# nmi_per_asset = {}
# for asset in available:
#     s = rolling_nmi_lag1(ret[asset])
#     s.name = f"nmi_lag1_{asset}"
#     nmi_per_asset[asset] = s
#     print(f"  nmi_lag1_{asset}: mean={s.mean():.4f}")

# nmi_df = pd.DataFrame(nmi_per_asset)
# nmi_mean = nmi_df.mean(axis=1);  nmi_mean.name = "nmi_lag1_mean"
# nmi_std  = nmi_df.std(axis=1);   nmi_std.name  = "nmi_lag1_std"
# nmi_max  = nmi_df.max(axis=1);   nmi_max.name  = "nmi_lag1_max"

# print("\n" + "="*55)
# print("  CORRELATION OF NEW FEATURES WITH VOL_BTC (full period)")
# print("="*55)

# vol_btc = ret["BTC"].rolling(VOL_WINDOW).std() * ANN

# new_feats = {
#     "entropy_index":  ent_index,
#     "nmi_lag1_index": nmi_index,
#     "nmi_lag1_mean":  nmi_mean,
#     "nmi_lag1_std":   nmi_std,
#     "nmi_lag1_max":   nmi_max,
# }

# if "X_it_slim" in dir():
#     print(f"\n  Old slim IT features (for comparison):")
#     for col in ["coinfo3_std","kl_BNB","pairwise_mi_mean","coinfo4_std"]:
#         if col in X_it_slim.columns:
#             c = X_it_slim[col].corr(vol_btc)
#             print(f"    {col:<28} corr = {c:>+.4f}")

# print(f"\n  New features:")
# for name, s in new_feats.items():
#     c = s.corr(vol_btc)
#     if not np.isfinite(c):
#         print(f"    {name:<28} corr = NaN  (constant feature)")
#         continue
#     bar = "█" * max(0, int(abs(c) * 25))
#     sign = "+" if c >= 0 else "-"
#     print(f"    {name:<28} corr = {c:>+.4f}  {sign}{bar}")

# print("\n" + "="*55)
# print("  ADDING TO X_it_slim")
# print("="*55)

# new_df = pd.DataFrame(new_feats)

# if "X_it_slim" in dir():
#     common_idx = X_it_slim.index.intersection(new_df.index)
#     X_it_extended = pd.concat([
#         X_it_slim.loc[common_idx],
#         new_df.loc[common_idx]
#     ], axis=1)

#     print(f"  X_it_slim:     {X_it_slim.shape[1]} features")
#     print(f"  New features:  {new_df.shape[1]} features")
#     print(f"  X_it_extended: {X_it_extended.shape[1]} features")
#     print(f"  Features: {list(X_it_extended.columns)}")
# else:
#     X_it_extended = new_df.copy()
#     print(f"  X_it_slim not found in memory, saving new features only")

# nan_pct = X_it_extended.isna().mean()
# print(f"\n  NaN share for new features:")
# for col in new_feats.keys():
#     if col in X_it_extended.columns:
#         val = nan_pct.loc[col] if col in nan_pct.index else nan_pct[col]
#         val = float(val.iloc[0]) if hasattr(val, "iloc") else float(val)
#         print(f"    {col:<28} {val:.3%}")

# print(f"\nX_it_extended saved to memory ({X_it_extended.shape[1]} features)")
# print(f"entropy_index, nmi_index, nmi_per_asset — saved to memory")

# print("\n" + "="*60)
# print("  SELECTION: permutation importance on train (BTC h=4d logvol)")
# print("="*60)

# import matplotlib
# matplotlib.use("Agg")
# import matplotlib.pyplot as plt
# from sklearn.linear_model import RidgeCV
# from sklearn.preprocessing import StandardScaler
# from sklearn.pipeline import Pipeline
# from sklearn.model_selection import KFold, cross_val_score

# SPLIT_DATE = "2025-07-01"
# PURGE_BARS = 96
# RNG2 = np.random.default_rng(42)
# N_PERM = 15

# def _ridge():
#     return Pipeline([("sc", StandardScaler()),
#                      ("m", RidgeCV(alphas=[0.01,0.1,1,10,100]))])

# r_btc   = ret["BTC"]
# vol_now = r_btc.rolling(48).std() * ANN
# vol_fwd = r_btc.rolling(192).std().shift(-192) * ANN
# y_logvol = np.log(vol_fwd.clip(lower=1e-8)).dropna()

# split_ts = pd.Timestamp(SPLIT_DATE)

# idx = X_vol.index.intersection(X_it_extended.index).intersection(y_logvol.index)
# valid = (X_vol.loc[idx].notna().all(1) &
#          X_it_extended.loc[idx].notna().all(1) &
#          y_logvol.loc[idx].notna())
# idx = idx[valid]
# tr = idx < (split_ts - pd.Timedelta(minutes=30 * PURGE_BARS))
# idx_tr = idx[tr]

# Xv_tr = X_vol.loc[idx_tr]
# ytr   = y_logvol.loc[idx_tr]

# ck   = Xv_tr.corrwith(ytr).abs().dropna()
# top3 = ck.nlargest(3).index.tolist()
# mb   = _ridge(); mb.fit(Xv_tr[top3], ytr)
# res_tr = ytr.values - mb.predict(Xv_tr[top3])

# Xi_tr = X_it_extended.loc[idx_tr]
# Xi_arr = Xi_tr.values
# feat_names = list(Xi_tr.columns)

# kf = KFold(n_splits=5, shuffle=False)
# base_cv = float(np.mean(cross_val_score(_ridge(), Xi_arr, res_tr,
#                                           cv=kf, scoring="r2")))
# print(f"  Baseline CV R² (all {len(feat_names)} features): {base_cv:.4f}")

# perm_scores = {}
# print(f"  Computing importance ({N_PERM} permutations × {len(feat_names)} features)...")
# for fi, feat in enumerate(feat_names):
#     deltas = []
#     for _ in range(N_PERM):
#         Xp = Xi_arr.copy()
#         Xp[:, fi] = RNG2.permutation(Xp[:, fi])
#         sc = float(np.mean(cross_val_score(_ridge(), Xp, res_tr,
#                                             cv=kf, scoring="r2")))
#         deltas.append(sc - base_cv)
#     perm_scores[feat] = float(np.mean(deltas))

# sorted_feats = sorted(perm_scores.items(), key=lambda x: x[1])  # most useful at top

# NEW_FEATS  = list(new_feats.keys())
# SLIM_FEATS = [c for c in feat_names if c not in NEW_FEATS]

# def cat_color(f):
#     if f in NEW_FEATS: return "#E91E63"   # pink — new features
#     return "#2196F3"                       # blue — old slim features

# print(f"\n  Feature ranking (Δ < 0 = useful):")
# print(f"  {'feature':<28} {'Δ CV R²':>9}  status")
# print("  " + "-"*50)
# for feat, delta in sorted_feats:
#     is_new = " [NEW]" if feat in NEW_FEATS else ""
#     status = "✓ useful" if delta < -0.002 else ("✗ harmful" if delta > 0.002 else "~ neutral")
#     print(f"  {feat:<28} {delta:>+9.4f}  {status}{is_new}")

# good_feats = [f for f, d in sorted_feats if d <= 0.003]
# bad_feats  = [f for f, d in sorted_feats if d >  0.003]

# print(f"\n  Useful/neutral: {len(good_feats)} features")
# print(f"  Harmful:        {len(bad_feats)} features: {bad_feats}")

# new_good = [f for f in good_feats if f in NEW_FEATS]
# new_bad  = [f for f in bad_feats  if f in NEW_FEATS]
# print(f"\n  Among new features:")
# print(f"    Useful:  {new_good}")
# print(f"    Harmful: {new_bad}")

# X_it_extended = X_it_extended.loc[:, ~X_it_extended.columns.duplicated()]

# X_it_final = X_it_extended[[f for f in good_feats if f in X_it_extended.columns]].copy()
# print(f"\n  X_it_final: {X_it_final.shape[1]} features")
# print(f"  {list(X_it_final.columns)}")

# fig, axes = plt.subplots(1, 2, figsize=(14, max(6, len(feat_names) * 0.28)))

# ax = axes[0]
# feats_plot  = [f for f, _ in sorted_feats]
# deltas_plot = [perm_scores[f] for f in feats_plot]
# colors_bar  = [cat_color(f) for f in feats_plot]

# y_pos = np.arange(len(feats_plot))
# ax.barh(y_pos, deltas_plot, color=colors_bar, alpha=0.85, height=0.75)
# ax.axvline(0,      color="black", lw=0.8)
# ax.axvline(-0.002, color="green", lw=0.8, ls=":", alpha=0.7, label="usefulness threshold")
# ax.axvline(+0.002, color="red",   lw=0.8, ls=":", alpha=0.7, label="harmfulness threshold")
# ax.set_yticks(y_pos)
# ax.set_yticklabels(
#     [f + " ★" if f in NEW_FEATS else f for f in feats_plot],
#     fontsize=8
# )
# ax.set_xlabel("Δ CV R² upon permutation\n(< 0 = feature is useful)")
# ax.set_title("Permutation importance\n(BTC, h=4d, logvol residuals)", fontsize=10)
# ax.legend(fontsize=8)
# ax.grid(alpha=0.2, axis="x")
# ax.spines["top"].set_visible(False)
# ax.spines["right"].set_visible(False)

# from matplotlib.patches import Patch
# legend_elems = [
#     Patch(facecolor="#2196F3", alpha=0.85, label="Old slim IT features"),
#     Patch(facecolor="#E91E63", alpha=0.85, label="New features (entropy/NMI) ★"),
# ]
# ax.legend(handles=legend_elems, fontsize=8, loc="lower right")

# ax = axes[1]
# corr_vals = []
# corr_names = []
# corr_colors = []
# for feat in feats_plot:
#     if feat in X_it_extended.columns:
#         c = X_it_extended[feat].corr(vol_btc)
#         if np.isfinite(c):
#             corr_vals.append(c)
#             corr_names.append(feat + (" ★" if feat in NEW_FEATS else ""))
#             corr_colors.append(cat_color(feat))

# y_pos2 = np.arange(len(corr_vals))
# ax.barh(y_pos2, corr_vals, color=corr_colors, alpha=0.85, height=0.75)
# ax.axvline(0, color="black", lw=0.8)
# ax.axvline(+0.10, color="green", lw=0.8, ls=":", alpha=0.5)
# ax.axvline(-0.10, color="green", lw=0.8, ls=":", alpha=0.5)
# ax.set_yticks(y_pos2)
# ax.set_yticklabels(corr_names, fontsize=8)
# ax.set_xlabel("Correlation with vol_BTC (full period)")
# ax.set_title("Feature correlation with vol_BTC\n(|corr| > 0.10 = significant)", fontsize=10)
# ax.grid(alpha=0.2, axis="x")
# ax.spines["top"].set_visible(False)
# ax.spines["right"].set_visible(False)

# plt.suptitle(
#     "IT feature usefulness: old slim + new (entropy, NMI)\n"
#     "★ = new features",
#     fontsize=11, y=1.01
# )
# plt.tight_layout()
# plt.savefig("fig_new_it_features.png", bbox_inches="tight", dpi=150)
# plt.savefig("fig_new_it_features.pdf", bbox_inches="tight", dpi=150)
# plt.show()
# print("  Saved: fig_new_it_features.png / .pdf")

# X_it_final = X_it_extended[good_feats].copy()
# print(f"\nX_it_final saved to memory ({X_it_final.shape[1]} features)")
# print(f"Use X_it_final instead of X_it_slim in the holdout pipeline")

Building missing IT features:
  1. Rolling Shannon entropy of the index
  2. NMI(r_t, r_{t-1}) — lag-1 normalised MI

Building equal-weighted index...
  Assets: 9, bars: 46743
  Computing rolling entropy of the index...
  entropy_index: mean=2.3024  std=0.0000
  Computing lag-1 NMI of the index...
  nmi_lag1_index: mean=0.4114  std=0.0466

Computing lag-1 NMI per asset...
  nmi_lag1_BTC: mean=0.4186
  nmi_lag1_ETH: mean=0.4129
  nmi_lag1_SOL: mean=0.4072
  nmi_lag1_BNB: mean=0.4102
  nmi_lag1_XRP: mean=0.4130
  nmi_lag1_DOGE: mean=0.4094
  nmi_lag1_ADA: mean=0.4079
  nmi_lag1_AVAX: mean=0.4071
  nmi_lag1_TRX: mean=0.4168

  CORRELATION OF NEW FEATURES WITH VOL_BTC (full period)

  Old slim IT features (for comparison):
    coinfo3_std                  corr = -0.0306
    kl_BNB                       corr = +0.1849
    pairwise_mi_mean             corr = +0.3385
    coinfo4_std                  corr = -0.0318

  New features:
    entropy_index                corr = NaN  (constant feature

In [103]:
# import warnings; warnings.filterwarnings("ignore")
# import numpy as np
# import pandas as pd
# from sklearn.linear_model import Ridge, RidgeCV
# from sklearn.preprocessing import StandardScaler
# from sklearn.pipeline import Pipeline
# from scipy import stats as sp_stats
# import matplotlib
# matplotlib.use("Agg")
# import matplotlib.pyplot as plt

# SPLIT_DATE = "2025-01-01"
# PURGE_BARS = 96
# HORIZONS   = {48:"1d", 96:"2d", 192:"4d", 288:"6d", 336:"7d", 672:"14d"}
# ASSETS     = ["BTC","ETH","SOL","BNB","XRP","DOGE","ADA","AVAX","TRX"]
# VOL_WINDOW = 48
# ANN        = np.sqrt(365 * 48)

# IT_SLIM = ["coinfo3_std","kl_BNB","coinfo4_std","pairwise_mi_mean",
#            "coinfo4_mean","pairwise_mi_std","coinfo3_mean",
#            "coinfo4_max","kl_AVAX","pairwise_mi_min"]

# TDA_BEST = ["BI_tp_H0","BI_ent_H1","BI_L1_H0",
#             "BII_tp_H0","BII_ent_H0","BC_tp_H0"]

# def _ridge():
#     return Pipeline([("sc", StandardScaler()),
#                      ("m", RidgeCV(alphas=[0.01,0.1,1,10,100]))])

# def oos_r2(y_true, y_pred, naive_val):
#     mse_m = np.mean((y_true - y_pred)**2)
#     mse_n = np.mean((y_true - naive_val)**2)
#     return float(1.0 - mse_m / mse_n) if mse_n > 1e-12 else np.nan

# def res_corr(res, pred_it):
#     return float(pd.Series(pred_it).corr(pd.Series(res)))

# print("Building X_tda...")
# tda_frames = [tda_A.add_prefix("A_")]
# for mode in ["I","II","C"]:
#     d = tda_B[mode]
#     df_m = pd.DataFrame({
#         f"B{mode}_tp_H0":  d["tp_H0"],  f"B{mode}_ent_H0": d["ent_H0"],
#         f"B{mode}_tp_H1":  d["tp_H1"],  f"B{mode}_ent_H1": d["ent_H1"],
#         f"B{mode}_L1_H0":  d["L1_H0"],  f"B{mode}_L1_H1":  d["L1_H1"],
#     }, index=pd.DatetimeIndex(d["times"]))
#     tda_frames.append(df_m)

# X_tda_raw = pd.concat(tda_frames, axis=1).sort_index()
# X_tda_ri  = X_tda_raw[TDA_BEST].reindex(ret.index, method="ffill")

# TDA_BEST = [c for c in TDA_BEST if c in X_tda_ri.columns]
# print(f"TDA features: {TDA_BEST}")

# split_ts = pd.Timestamp(SPLIT_DATE)
# rows = []

# for asset in ASSETS:
#     if asset not in ret.columns: continue
#     r = ret[asset]
#     vol_now = r.rolling(VOL_WINDOW).std() * ANN

#     for h, h_label in HORIZONS.items():
#         vol_fwd = r.rolling(h).std().shift(-h) * ANN
#         y_raw   = np.log(vol_fwd.clip(lower=1e-8)).dropna()

#         Xi_it  = X_it_slim[IT_SLIM]
#         Xi_tda = X_tda_ri

#         idx = (X_vol.index.intersection(Xi_it.index)
#                .intersection(Xi_tda.index).intersection(y_raw.index))
#         valid = (X_vol.loc[idx].notna().all(1) &
#                  Xi_it.loc[idx].notna().all(1) &
#                  Xi_tda.loc[idx].notna().all(1) &
#                  y_raw.loc[idx].notna())
#         idx = idx[valid]
#         if len(idx) < 1000: continue

#         tr = idx < (split_ts - pd.Timedelta(minutes=30*PURGE_BARS))
#         te = idx >= split_ts
#         if tr.sum() < 480 or te.sum() < 100: continue

#         Xv_tr  = X_vol.loc[idx[tr]];  Xv_te  = X_vol.loc[idx[te]]
#         Xit_tr = Xi_it.loc[idx[tr]];  Xit_te = Xi_it.loc[idx[te]]
#         Xtd_tr = Xi_tda.loc[idx[tr]]; Xtd_te = Xi_tda.loc[idx[te]]
#         ytr    = y_raw.loc[idx[tr]];  yte    = y_raw.loc[idx[te]]
#         naive  = float(ytr.mean())

#         ck   = Xv_tr.corrwith(ytr).abs().replace([np.inf,-np.inf],np.nan).dropna()
#         top3 = ck.nlargest(3).index.tolist()
#         mb   = _ridge(); mb.fit(Xv_tr[top3], ytr)
#         pb_tr = mb.predict(Xv_tr[top3])
#         pb_te = mb.predict(Xv_te[top3])
#         res_tr = ytr.values - pb_tr
#         res_te = yte.values - pb_te

#         mit = _ridge(); mit.fit(Xit_tr, res_tr)
#         pit_te = mit.predict(Xit_te)
#         pred_it = pb_te + pit_te

#         mtda = _ridge(); mtda.fit(Xtd_tr, res_tr)
#         ptda_te = mtda.predict(Xtd_te)
#         pred_tda = pb_te + ptda_te

#         Xc_tr = pd.concat([Xit_tr, Xtd_tr], axis=1)
#         Xc_te = pd.concat([Xit_te, Xtd_te], axis=1)
#         mct = _ridge(); mct.fit(Xc_tr, res_tr)
#         pct_te = mct.predict(Xc_te)
#         pred_itd = pb_te + pct_te

#         r2_b   = oos_r2(yte.values, pb_te,    naive)
#         r2_it  = oos_r2(yte.values, pred_it,  naive)
#         r2_tda = oos_r2(yte.values, pred_tda, naive)
#         r2_itd = oos_r2(yte.values, pred_itd, naive)

#         rc_it  = res_corr(res_te, pit_te)
#         rc_tda = res_corr(res_te, ptda_te)
#         rc_itd = res_corr(res_te, pct_te)

#         mse_b   = np.mean((yte.values - pb_te)**2)
#         mse_it  = np.mean((yte.values - pred_it)**2)
#         mse_tda = np.mean((yte.values - pred_tda)**2)
#         mse_itd = np.mean((yte.values - pred_itd)**2)

#         def dm(d):
#             n = len(d); m = d.mean(); s = d.std(ddof=1)
#             t = m/(s/np.sqrt(n)) if s > 1e-10 else 0
#             return float(t), float(sp_stats.t.sf(t, df=n-1))

#         d_it  = (yte.values-pb_te)**2 - (yte.values-pred_it)**2
#         d_tda = (yte.values-pb_te)**2 - (yte.values-pred_tda)**2
#         d_itd = (yte.values-pb_te)**2 - (yte.values-pred_itd)**2
#         t_it,  p_it  = dm(d_it)
#         t_tda, p_tda = dm(d_tda)
#         t_itd, p_itd = dm(d_itd)

#         def sig(p): return "***" if p<0.01 else ("**" if p<0.05 else ("*" if p<0.10 else ""))

#         rows.append({
#             "asset": asset, "horizon": h_label, "h": h,
#             "r2_base": round(r2_b,   4),
#             "r2_it":   round(r2_it,  4),
#             "r2_tda":  round(r2_tda, 4),
#             "r2_itd":  round(r2_itd, 4),
#             "dR2_it":  round(r2_it  - r2_b, 4),
#             "dR2_tda": round(r2_tda - r2_b, 4),
#             "dR2_itd": round(r2_itd - r2_b, 4),
#             "dR2_tda_vs_it": round(r2_itd - r2_it, 4),
#             "rc_it":   round(rc_it,  4),
#             "rc_tda":  round(rc_tda, 4),
#             "rc_itd":  round(rc_itd, 4),
#             "mse_r_it":  round(mse_it  / mse_b, 4) if mse_b > 0 else np.nan,
#             "mse_r_tda": round(mse_tda / mse_b, 4) if mse_b > 0 else np.nan,
#             "mse_r_itd": round(mse_itd / mse_b, 4) if mse_b > 0 else np.nan,
#             "p_it":  round(p_it,  4), "sig_it":  sig(p_it),
#             "p_tda": round(p_tda, 4), "sig_tda": sig(p_tda),
#             "p_itd": round(p_itd, 4), "sig_itd": sig(p_itd),
#             "n_train": int(tr.sum()), "n_test": int(te.sum()),
#         })

# df = pd.DataFrame(rows)
# print(f"\nTotal rows: {len(df)}")

# print("\n" + "="*90)
# print("  FINAL COMPARISON — target: logvol = log(realized_vol_fwd_h)")
# print("  R² = 1 - MSE(model) / MSE(naive=mean_train_logvol)")
# print("  A=Baseline(vol)  B=+IT  C=+TDA  D=+IT+TDA")
# print("="*90)

# print(f"\n  {'asset':<6} {'h':<4} "
#       f"{'R²_A(base)':>11} {'R²_B(+IT)':>11} {'R²_C(+TDA)':>12} {'R²_D(+IT+TDA)':>14} "
#       f"{'ΔB':>7} {'ΔC':>7} {'ΔD':>7}  best")
# print("  " + "-"*95)

# prev = None
# for _, row in df.sort_values(["asset","h"]).iterrows():
#     if prev and prev != row["asset"]: print("  " + "·"*95)
#     prev = row["asset"]
#     vals = {"A":row["r2_base"],"B":row["r2_it"],"C":row["r2_tda"],"D":row["r2_itd"]}
#     best = max(vals, key=lambda k: vals[k])
#     bm = "↑" if row["dR2_it"]>0.005 else ("↓" if row["dR2_it"]<-0.005 else "~")
#     cm = "↑" if row["dR2_tda"]>0.005 else ("↓" if row["dR2_tda"]<-0.005 else "~")
#     dm = "↑" if row["dR2_itd"]>0.005 else ("↓" if row["dR2_itd"]<-0.005 else "~")
#     print(f"  {row['asset']:<6} {row['horizon']:<4}"
#           f" {row['r2_base']:>+11.4f}"
#           f" {row['r2_it']:>+11.4f}"
#           f" {row['r2_tda']:>+12.4f}"
#           f" {row['r2_itd']:>+14.4f}"
#           f" {row['dR2_it']:>+6.4f}{bm}"
#           f" {row['dR2_tda']:>+6.4f}{cm}"
#           f" {row['dR2_itd']:>+6.4f}{dm}"
#           f"  {best}")

# print(f"\n{'='*70}")
# print(f"  SUMMARY")
# print(f"{'='*70}")

# n = len(df)
# print(f"""
#   Config   Description           ΔR²(mean)  ΔR²>0   rc(mean)  MSE<1
#   ─────────────────────────────────────────────────────────────────
#   B (+IT)   Baseline + IT          {df['dR2_it'].mean():>+8.4f}   {(df['dR2_it']>0).sum():>2}/{n}  {df['rc_it'].mean():>+7.4f}   {(df['mse_r_it']<1).sum():>2}/{n}
#   C (+TDA)  Baseline + TDA        {df['dR2_tda'].mean():>+8.4f}   {(df['dR2_tda']>0).sum():>2}/{n}  {df['rc_tda'].mean():>+7.4f}   {(df['mse_r_tda']<1).sum():>2}/{n}
#   D (+ALL) Baseline + IT  + TDA   {df['dR2_itd'].mean():>+8.4f}   {(df['dR2_itd']>0).sum():>2}/{n}  {df['rc_itd'].mean():>+7.4f}   {(df['mse_r_itd']<1).sum():>2}/{n}
# """)

# print(f"  Top-5 by ΔR² (config D = +IT+TDA):")
# for _, row in df.nlargest(5,"dR2_itd").iterrows():
#     print(f"    {row['asset']} h={row['horizon']}: "
#           f"R²_A={row['r2_base']:+.4f} → "
#           f"R²_B={row['r2_it']:+.4f} → "
#           f"R²_C={row['r2_tda']:+.4f} → "
#           f"R²_D={row['r2_itd']:+.4f} "
#           f"(ΔD={row['dR2_itd']:+.4f}, p={row['p_itd']:.4f}{row['sig_itd']})")

# print(f"\n  By asset — mean R² across all configurations:")
# print(f"  {'asset':<6} {'R²_A(base)':>11} {'R²_B(+IT)':>11} "
#       f"{'R²_C(+TDA)':>12} {'R²_D(+ALL)':>12}  best")
# print("  " + "-"*60)
# for asset in df.groupby("asset")["r2_itd"].mean().sort_values(ascending=False).index:
#     g = df[df.asset == asset]
#     ra = g["r2_base"].mean(); rb = g["r2_it"].mean()
#     rc_ = g["r2_tda"].mean(); rd = g["r2_itd"].mean()
#     best = max({"A":ra,"B":rb,"C":rc_,"D":rd}, key=lambda k: {"A":ra,"B":rb,"C":rc_,"D":rd}[k])
#     print(f"  {asset:<6} {ra:>+11.4f} {rb:>+11.4f} {rc_:>+12.4f} {rd:>+12.4f}  → {best}")

# print("\nBuilding final plots...")

# fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# asset_order = df.groupby("asset")["r2_itd"].mean().sort_values(ascending=False).index.tolist()
# h_order = ["1d","2d","4d","6d","7d","14d"]
# w = 0.2; x = np.arange(len(asset_order))

# ax = axes[0,0]
# for ci, (col, label, color) in enumerate([
#     ("r2_base","A: Baseline","#9E9E9E"),
#     ("r2_it",  "B: +IT",    "#2196F3"),
#     ("r2_tda", "C: +TDA",   "#FF9800"),
#     ("r2_itd", "D: +IT+TDA","#4CAF50"),
# ]):
#     vals = [df[df.asset==a][col].mean() for a in asset_order]
#     ax.bar(x+(ci-1.5)*w, vals, w*0.9, label=label, color=color, alpha=0.85)
# ax.axhline(0, color="black", lw=0.8)
# ax.set_xticks(x); ax.set_xticklabels(asset_order, fontsize=9)
# ax.set_ylabel("OOS R² (vs naive mean logvol)")
# ax.set_title("OOS R² by asset — 4 configurations\n(logvol, all horizons)", fontsize=10)
# ax.legend(fontsize=8); ax.grid(alpha=0.2, axis="y")
# ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

# ax = axes[0,1]
# for col, label, color, ls in [
#     ("r2_base","A: Baseline","#9E9E9E","-"),
#     ("r2_it",  "B: +IT",    "#2196F3","-"),
#     ("r2_tda", "C: +TDA",   "#FF9800","--"),
#     ("r2_itd", "D: +IT+TDA","#4CAF50","-"),
# ]:
#     vals = [df[df.horizon==h][col].mean() for h in h_order]
#     ax.plot(h_order, vals, "o"+ls, color=color, lw=2, label=label, ms=7)
# ax.axhline(0, color="black", lw=0.5, ls=":")
# ax.set_ylabel("OOS R²"); ax.set_xlabel("Prediction horizon")
# ax.set_title("OOS R² by horizon\n(mean over 9 assets)", fontsize=10)
# ax.legend(fontsize=8); ax.grid(alpha=0.2)
# ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

# ax = axes[1,0]
# x2 = np.arange(len(asset_order)); w2 = 0.28
# for ci, (col, label, color) in enumerate([
#     ("dR2_it",  "B: +IT",    "#2196F3"),
#     ("dR2_tda", "C: +TDA",   "#FF9800"),
#     ("dR2_itd", "D: +IT+TDA","#4CAF50"),
# ]):
#     vals = [df[df.asset==a][col].mean() for a in asset_order]
#     ax.bar(x2+(ci-1)*w2, vals, w2*0.9, label=label, color=color, alpha=0.85)
# ax.axhline(0, color="black", lw=0.8)
# ax.set_xticks(x2); ax.set_xticklabels(asset_order, fontsize=9)
# ax.set_ylabel("ΔR² relative to baseline A")
# ax.set_title("Incremental ΔR² over baseline\n(logvol, all horizons)", fontsize=10)
# ax.legend(fontsize=8); ax.grid(alpha=0.2, axis="y")
# ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

# ax = axes[1,1]
# pivot = df.pivot_table(index="asset", columns="horizon",
#                        values="mse_r_itd", aggfunc="mean")
# pivot = pivot.reindex(asset_order)[h_order]
# im = ax.imshow(pivot.values, cmap="RdYlGn_r", vmin=0.85, vmax=1.15, aspect="auto")
# ax.set_xticks(range(len(h_order)));    ax.set_xticklabels(h_order)
# ax.set_yticks(range(len(asset_order))); ax.set_yticklabels(asset_order)
# for i in range(len(asset_order)):
#     for j in range(len(h_order)):
#         val = pivot.values[i,j]
#         if np.isfinite(val):
#             col = "white" if abs(val-1)>0.07 else "black"
#             mark = "✓" if val < 1.0 else ""
#             ax.text(j, i, f"{val:.3f}{mark}", ha="center", va="center",
#                     fontsize=8, color=col)
# plt.colorbar(im, ax=ax, label="MSE_ratio (D: +IT+TDA / baseline)\n<1.0 = better ✓")
# ax.set_title("MSE_ratio config D (+IT+TDA)\ngreen <1 = better than baseline", fontsize=10)

# plt.suptitle(
#     "Final comparison: A=Baseline / B=+IT / C=+TDA / D=+IT+TDA\n"
#     "Target: logvol = log(realized_vol_fwd)  |  holdout 2025-07-01 →",
#     fontsize=12, y=1.01)
# plt.tight_layout()
# plt.savefig("fig_final_comparison.png", bbox_inches="tight", dpi=150)
# plt.savefig("fig_final_comparison.pdf", bbox_inches="tight", dpi=150)
# plt.show()
# print("  Saved: fig_final_comparison.png / .pdf")

# final_comparison = df.copy()
# print(f"\nfinal_comparison saved to memory ({len(df)} rows)")

Building X_tda...
TDA features: ['BI_tp_H0', 'BI_ent_H1', 'BI_L1_H0', 'BII_tp_H0', 'BII_ent_H0', 'BC_tp_H0']

Total rows: 54

  FINAL COMPARISON — target: logvol = log(realized_vol_fwd_h)
  R² = 1 - MSE(model) / MSE(naive=mean_train_logvol)
  A=Baseline(vol)  B=+IT  C=+TDA  D=+IT+TDA

  asset  h     R²_A(base)   R²_B(+IT)   R²_C(+TDA)  R²_D(+IT+TDA)      ΔB      ΔC      ΔD  best
  -----------------------------------------------------------------------------------------------
  ADA    1d       +0.3362     +0.3412      +0.3351        +0.3548 +0.0051↑ -0.0010~ +0.0186↑  D
  ADA    2d       +0.3047     +0.2995      +0.3073        +0.3236 -0.0052↓ +0.0026~ +0.0189↑  D
  ADA    4d       +0.2688     +0.2338      +0.2718        +0.2651 -0.0350↓ +0.0030~ -0.0037~  C
  ADA    6d       +0.2666     +0.2108      +0.2540        +0.2272 -0.0557↓ -0.0126↓ -0.0394↓  A
  ADA    7d       +0.2623     +0.2027      +0.2410        +0.2076 -0.0596↓ -0.0213↓ -0.0546↓  A
  ADA    14d      +0.1629     +0.0738   

In [104]:
# fig_feature_analysis

Building X_tda...
  TDA features: 26
Building full feature matrix X_all...
  Total features: 56
  IT: 30, TDA: 26

Permutation importance on ['BTC', 'ETH'], h=4d, logvol...
  Baseline CV R² (all 56 features): -0.0698
  Running permutation (15 × 56)...
    10/56 done
    20/56 done
    30/56 done
    40/56 done
    50/56 done
  Done.

Correlation with vol_BTC...

Loading ΔR² by asset...
  Loaded from final_comparison (54 rows)

Building figure...
Saved: fig_feature_analysis_full.png / .pdf

Top-10 most useful features (Δ < 0):
  kl_BNB                        -0.0090  [IT_slim]
  BII_tp_H0                     -0.0065  [TDA_BII]
  coinfo3_std                   -0.0057  [IT_slim]
  BC_tp_H0                      -0.0052  [TDA_BC]
  coinfo4_std                   -0.0043  [IT_slim]
  kl_ETH                        -0.0025  [IT_removed]
  kl_ADA                        -0.0024  [IT_removed]
  BC_ent_H0                     -0.0024  [TDA_BC]
  coinfo4_min                   -0.0018  [IT_removed]
  

In [115]:
# import warnings; warnings.filterwarnings("ignore")
# import numpy as np
# import pandas as pd
# from sklearn.linear_model import Ridge, RidgeCV
# from sklearn.preprocessing import StandardScaler
# from sklearn.pipeline import Pipeline
# from sklearn.model_selection import KFold, cross_val_score
# import matplotlib
# matplotlib.use("Agg")
# import matplotlib.pyplot as plt

# SPLIT_DATE = "2025-01-01"
# PURGE_BARS = 96
# HORIZONS   = {96:"2d", 192:"4d"}       # фокус на средних горизонтах
# ASSETS     = ["BTC","ETH","DOGE","XRP","SOL","BNB"]
# VOL_WINDOW = 48
# ANN        = np.sqrt(365 * 48)
# RNG        = np.random.default_rng(42)
# N_PERM     = 10

# def _ridge():
#     return Pipeline([("sc", StandardScaler()),
#                      ("m", RidgeCV(alphas=[0.01,0.1,1,10,100]))])

# def oos_r2(y_true, y_pred, naive_val):
#     mse_m = np.mean((y_true - y_pred)**2)
#     mse_n = np.mean((y_true - naive_val)**2)
#     return float(1.0 - mse_m / mse_n) if mse_n > 1e-12 else np.nan

# def _get_y(asset, h):
#     col = f"delta_vol_{h}"
#     for src in [targets_extra if 'targets_extra' in dir() else {},
#                 targets, targets_ext]:
#         if asset in src and col in src[asset].columns:
#             return src[asset][col].dropna()
#     return None

# try: _ = targets_extra
# except NameError: targets_extra = {}

# print("Собираем все фичи...")

# tda_frames = [tda_A.add_prefix("A_")]
# for mode in ["I","II","C"]:
#     d = tda_B[mode]
#     idx_tda = pd.DatetimeIndex(d["times"])
#     df_m = pd.DataFrame({
#         f"B{mode}_tp_H0":  d["tp_H0"],  f"B{mode}_ent_H0": d["ent_H0"],
#         f"B{mode}_tp_H1":  d["tp_H1"],  f"B{mode}_ent_H1": d["ent_H1"],
#         f"B{mode}_L1_H0":  d["L1_H0"],  f"B{mode}_L1_H1":  d["L1_H1"],
#     }, index=idx_tda)
#     tda_frames.append(df_m)

# X_tda_raw = pd.concat(tda_frames, axis=1).sort_index()
# X_tda_ri  = X_tda_raw.reindex(ret.index, method="ffill")

# IT_SLIM = ["coinfo3_std","kl_BNB","coinfo4_std","pairwise_mi_mean",
#            "coinfo4_mean","pairwise_mi_std","coinfo3_mean",
#            "coinfo4_max","kl_AVAX","pairwise_mi_min"]

# IT_REMOVED = ["pairwise_cov_std","pairwise_cov_max","pairwise_cov_min",
#               "pairwise_cov_mean","pairwise_mi_max","kl_BTC","kl_TRX",
#               "kl_XRP","kl_SOL","coinfo4_min","kl_DOGE","kl_ETH",
#               "coinfo3_max","kl_ADA","coinfo3_min","pairwise_mi_min",
#               "kl_AVAX","coinfo4_max"]

# IT_ALL_25 = IT_SLIM + [c for c in IT_REMOVED if c in X_it_slim.columns
#                         and c not in IT_SLIM]
# IT_ALL_25 = [c for c in IT_ALL_25 if c in X_it_slim.columns]

# TDA_COLS = list(X_tda_ri.columns)

# print(f"IT-фич всего:  {len(IT_ALL_25)}")
# print(f"IT slim (10):  {len(IT_SLIM)}")
# print(f"TDA-фич:       {len(TDA_COLS)}")

# print("\n" + "="*65)
# print("  1. КОРРЕЛЯЦИЯ ВСЕХ ФИЧ С VOL_BTC НА TEST-ПЕРИОДЕ")
# print("="*65)

# split_ts  = pd.Timestamp(SPLIT_DATE)
# vol_btc   = ret["BTC"].rolling(VOL_WINDOW).std() * ANN
# vol_test  = vol_btc[vol_btc.index >= split_ts]

# it_corrs = {}
# for col in IT_ALL_25:
#     if col not in X_it_slim.columns: continue
#     s = X_it_slim[col].reindex(vol_test.index, method="ffill")
#     it_corrs[col] = float(s.corr(vol_test))

# tda_corrs = {}
# for col in TDA_COLS:
#     s = X_tda_ri[col].reindex(vol_test.index, method="ffill")
#     tda_corrs[col] = float(s.corr(vol_test))

# print(f"\n  IT-фичи (топ-10 по |corr| с vol_BTC test):")
# it_sorted = sorted(it_corrs.items(), key=lambda x: abs(x[1]), reverse=True)
# for feat, c in it_sorted[:10]:
#     tag = " [slim]" if feat in IT_SLIM else " [removed]"
#     bar = "█" * max(0, int(abs(c)*20))
#     print(f"    {feat:<28} {c:>+.4f}  {bar}{tag}")

# print(f"\n  TDA-фичи (топ-10 по |corr|):")
# tda_sorted = sorted(tda_corrs.items(), key=lambda x: abs(x[1]), reverse=True)
# for feat, c in tda_sorted[:10]:
#     bar = "█" * max(0, int(abs(c)*20))
#     print(f"    {feat:<28} {c:>+.4f}  {bar}")

# print("\n" + "="*65)
# print("  2. PERMUTATION IMPORTANCE НА TRAIN (BTC+ETH, h=4d, logvol)")
# print("="*65)

# perm_results = {}
# for pilot_asset in ["BTC","ETH"]:
#     r = ret[pilot_asset]
#     vol_now = r.rolling(VOL_WINDOW).std() * ANN
#     vol_fwd = r.rolling(192).std().shift(-192) * ANN
#     y_logvol = np.log(vol_fwd.clip(lower=1e-8)).dropna()

#     idx = (X_vol.index.intersection(X_it_slim.index)
#            .intersection(X_tda_ri.index).intersection(y_logvol.index))
#     valid = (X_vol.loc[idx].notna().all(1) &
#              X_it_slim.loc[idx].notna().all(1) &
#              X_tda_ri.loc[idx].notna().all(1) &
#              y_logvol.loc[idx].notna())
#     idx = idx[valid]
#     tr   = idx < (split_ts - pd.Timedelta(minutes=30*PURGE_BARS))
#     idx_tr = idx[tr]

#     Xv_tr = X_vol.loc[idx_tr]
#     ck    = Xv_tr.corrwith(y_logvol.loc[idx_tr]).abs().dropna()
#     top3  = ck.nlargest(3).index.tolist()
#     mdl_b = _ridge(); mdl_b.fit(Xv_tr[top3], y_logvol.loc[idx_tr])
#     res_tr = y_logvol.loc[idx_tr].values - mdl_b.predict(Xv_tr[top3])

#     Xi_all_tr = pd.concat([
#         X_it_slim.loc[idx_tr],
#         X_tda_ri.loc[idx_tr]
#     ], axis=1)
#     Xi_arr  = Xi_all_tr.values
#     res_arr = res_tr
#     feat_names = list(Xi_all_tr.columns)

#     kf = KFold(n_splits=5, shuffle=False)
#     base_cv = float(np.mean(cross_val_score(_ridge(), Xi_arr, res_arr,
#                                              cv=kf, scoring="r2")))

#     perm_scores_asset = {}
#     for fi, feat in enumerate(feat_names):
#         deltas = []
#         for _ in range(N_PERM):
#             Xp = Xi_arr.copy()
#             Xp[:, fi] = RNG.permutation(Xp[:, fi])
#             sc = float(np.mean(cross_val_score(_ridge(), Xp, res_arr,
#                                                cv=kf, scoring="r2")))
#             deltas.append(sc - base_cv)
#         perm_scores_asset[feat] = float(np.mean(deltas))

#     perm_results[pilot_asset] = perm_scores_asset
#     print(f"  {pilot_asset} baseline CV R²={base_cv:.4f}")

# all_feats = list(perm_results["BTC"].keys())
# avg_perm  = {f: np.mean([perm_results[a].get(f,0) for a in ["BTC","ETH"]])
#              for f in all_feats}

# def categorize(feat):
#     if feat in IT_SLIM:    return "IT_slim"
#     if feat in IT_ALL_25:  return "IT_removed"
#     if feat.startswith("A_"): return "TDA_A"
#     if feat.startswith("BI_"): return "TDA_BI"
#     if feat.startswith("BII_"): return "TDA_BII"
#     if feat.startswith("BC_"): return "TDA_BC"
#     return "other"

# cat_colors = {
#     "IT_slim":"#2196F3","IT_removed":"#90CAF9",
#     "TDA_A":"#4CAF50","TDA_BI":"#FF9800",
#     "TDA_BII":"#9C27B0","TDA_BC":"#F44336","other":"#9E9E9E"
# }

# print(f"\n  Топ-15 полезных фич (Δ<0 = помогает):")
# sorted_perm = sorted(avg_perm.items(), key=lambda x: x[1])
# for feat, delta in sorted_perm[:15]:
#     cat = categorize(feat)
#     print(f"    {feat:<28} {delta:>+.4f}  [{cat}]")

# print(f"\n  Топ-10 вредных фич (Δ>0 = вредит):")
# for feat, delta in sorted(avg_perm.items(), key=lambda x: -x[1])[:10]:
#     cat = categorize(feat)
#     print(f"    {feat:<28} {delta:>+.4f}  [{cat}]")

# best_tda = [f for f in TDA_COLS if avg_perm.get(f,0) < -0.002]
# neutral_tda = [f for f in TDA_COLS if -0.002 <= avg_perm.get(f,0) <= 0.002]
# bad_tda  = [f for f in TDA_COLS if avg_perm.get(f,0) > 0.002]
# print(f"\n  TDA-фичи: {len(best_tda)} полезных, "
#       f"{len(neutral_tda)} нейтральных, {len(bad_tda)} вредных")
# print(f"  Лучшие TDA: {best_tda[:8]}")

# print("\n" + "="*65)
# print("  3. СРАВНЕНИЕ КОНФИГУРАЦИЙ НА HOLDOUT")
# print("="*65)

# CONFIGS = {
#     "slim_IT":    IT_SLIM,
#     "best_IT":    [f for f,d in sorted_perm[:15] if f in IT_ALL_25],
#     "slim+TDA":   IT_SLIM + best_tda[:6],
#     "best+TDA":   [f for f,d in sorted_perm[:12]
#                    if f in IT_ALL_25 or f in TDA_COLS],
# }
# CONFIGS = {k: list(dict.fromkeys(v)) for k,v in CONFIGS.items()}

# for name, cols in CONFIGS.items():
#     print(f"  {name}: {len(cols)} фич")

# config_rows = []
# for asset in ASSETS:
#     r = ret[asset] if asset in ret.columns else None
#     if r is None: continue
#     vol_now = r.rolling(VOL_WINDOW).std() * ANN
#     vol_fwd_h = {}
#     for h in HORIZONS:
#         vol_fwd_h[h] = r.rolling(h).std().shift(-h) * ANN

#     for h, h_label in HORIZONS.items():
#         y_raw = np.log(vol_fwd_h[h].clip(lower=1e-8)).dropna()

#         for cfg_name, cfg_cols in CONFIGS.items():
#             it_cols_used  = [c for c in cfg_cols if c in X_it_slim.columns]
#             tda_cols_used = [c for c in cfg_cols if c in X_tda_ri.columns]

#             Xi_cfg = pd.concat(
#                 [X_it_slim[it_cols_used]] +
#                 ([X_tda_ri[tda_cols_used]] if tda_cols_used else []),
#                 axis=1
#             ) if it_cols_used or tda_cols_used else X_it_slim[IT_SLIM]

#             idx = (X_vol.index.intersection(Xi_cfg.index).intersection(y_raw.index))
#             valid = (X_vol.loc[idx].notna().all(1) &
#                      Xi_cfg.loc[idx].notna().all(1) &
#                      y_raw.loc[idx].notna())
#             idx = idx[valid]
#             if len(idx) < 1000: continue

#             tr = idx < (split_ts - pd.Timedelta(minutes=30*PURGE_BARS))
#             te = idx >= split_ts
#             if tr.sum() < 480 or te.sum() < 100: continue

#             Xv_tr = X_vol.loc[idx[tr]]; Xv_te = X_vol.loc[idx[te]]
#             Xi_tr = Xi_cfg.loc[idx[tr]]; Xi_te = Xi_cfg.loc[idx[te]]
#             ytr   = y_raw.loc[idx[tr]];  yte   = y_raw.loc[idx[te]]
#             naive = float(ytr.mean())

#             ck   = Xv_tr.corrwith(ytr).abs().dropna()
#             top3 = ck.nlargest(3).index.tolist()
#             mb   = _ridge(); mb.fit(Xv_tr[top3], ytr)
#             pb_tr = mb.predict(Xv_tr[top3])
#             pb_te = mb.predict(Xv_te[top3])

#             res_tr = ytr.values - pb_tr
#             mc = _ridge(); mc.fit(Xi_tr, res_tr)
#             pc_te = mc.predict(Xi_te)
#             pred  = pb_te + pc_te

#             r2_b = oos_r2(yte.values, pb_te, naive)
#             r2_c = oos_r2(yte.values, pred,  naive)
#             rc   = float(pd.Series(pc_te).corr(pd.Series(yte.values - pb_te)))

#             config_rows.append({
#                 "asset":   asset, "horizon": h_label, "h": h,
#                 "config":  cfg_name,
#                 "r2_base": round(r2_b, 4),
#                 "r2_cfg":  round(r2_c, 4),
#                 "delta_r2":round(r2_c - r2_b, 4),
#                 "res_corr":round(rc, 4),
#             })

# cfg_df = pd.DataFrame(config_rows)

# print(f"\n  Среднее ΔR² (logvol) по конфигурациям:")
# print(f"  {'config':<15} {'mean_ΔR²':>10} {'ΔR²>0':>8} {'mean_rc':>9}")
# print("  " + "-"*45)
# for cfg in CONFIGS.keys():
#     sub = cfg_df[cfg_df.config == cfg]
#     if sub.empty: continue
#     pos = (sub["delta_r2"] > 0).sum()
#     print(f"  {cfg:<15} {sub['delta_r2'].mean():>+10.4f} "
#           f"{pos:>5}/{len(sub)} {sub['res_corr'].mean():>+9.4f}")

# best_cfg = cfg_df.groupby("config")["delta_r2"].mean().idxmax()
# print(f"\n  Лучшая конфигурация: '{best_cfg}'")
# sub_best = cfg_df[cfg_df.config == best_cfg]
# print(f"  {'актив':<6} {'h':<4} {'R²_base':>9} {'R²_best':>9} {'ΔR²':>8}")
# print("  " + "-"*40)
# prev = None
# for _, row in sub_best.sort_values(["asset","h"]).iterrows():
#     if prev and prev != row["asset"]: print("  " + "·"*40)
#     prev = row["asset"]
#     mark = "↑" if row["delta_r2"] > 0.005 else ("↓" if row["delta_r2"] < -0.005 else "~")
#     print(f"  {row['asset']:<6} {row['horizon']:<4}"
#           f" {row['r2_base']:>+9.4f} {row['r2_cfg']:>+9.4f}"
#           f" {row['delta_r2']:>+8.4f} {mark}")

# print("\nСтроим графики...")

# fig, axes = plt.subplots(1, 3, figsize=(18, 7))

# ax = axes[0]
# feats_sorted = [f for f,_ in sorted_perm]
# vals_sorted  = [avg_perm[f] for f in feats_sorted]
# colors_bar   = [cat_colors.get(categorize(f), "#9E9E9E") for f in feats_sorted]
# y_pos = np.arange(len(feats_sorted))
# ax.barh(y_pos, vals_sorted, color=colors_bar, alpha=0.85, height=0.8)
# ax.axvline(0, color="black", lw=0.8)
# ax.axvline(-0.002, color="green",  lw=0.8, ls=":", alpha=0.6)
# ax.axvline(+0.002, color="red",    lw=0.8, ls=":", alpha=0.6)
# ax.set_yticks(y_pos)
# ax.set_yticklabels(feats_sorted, fontsize=7)
# ax.set_xlabel("Δ CV R² при перемешивании\n(< 0 = фича полезна)")
# ax.set_title("Permutation importance\n(BTC+ETH, h=4d, logvol)", fontsize=10)
# ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
# from matplotlib.patches import Patch
# legend_elems = [Patch(facecolor=c, label=k, alpha=0.85)
#                 for k,c in cat_colors.items() if k != "other"]
# ax.legend(handles=legend_elems, fontsize=7, loc="lower right")

# ax = axes[1]
# cfg_order = list(CONFIGS.keys())
# asset_order = ASSETS
# x = np.arange(len(asset_order))
# w = 0.18
# cfg_colors = ["#90CAF9","#2196F3","#4CAF50","#FF9800"]
# for ci, cfg in enumerate(cfg_order):
#     sub = cfg_df[cfg_df.config == cfg]
#     vals = []
#     for asset in asset_order:
#         v = sub[sub.asset==asset]["delta_r2"].mean()
#         vals.append(v if not np.isnan(v) else 0)
#     ax.bar(x + (ci-1.5)*w, vals, w*0.9,
#            label=cfg, color=cfg_colors[ci], alpha=0.85)
# ax.axhline(0, color="black", lw=0.8)
# ax.set_xticks(x); ax.set_xticklabels(asset_order)
# ax.set_ylabel("Среднее ΔR² (все горизонты)")
# ax.set_title("ΔR² по конфигурациям × активам\n(logvol, holdout 2025)", fontsize=10)
# ax.legend(fontsize=8); ax.grid(alpha=0.2, axis="y")
# ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

# ax = axes[2]
# cat_corrs = {}
# for feat, c in {**it_corrs, **tda_corrs}.items():
#     cat = categorize(feat)
#     if cat not in cat_corrs: cat_corrs[cat] = []
#     cat_corrs[cat].append(c)

# cats_to_plot = ["IT_slim","IT_removed","TDA_A","TDA_BI","TDA_BII","TDA_BC"]
# data_plot    = [cat_corrs.get(c,[0]) for c in cats_to_plot]
# bp = ax.boxplot(data_plot, labels=cats_to_plot, patch_artist=True,
#                 medianprops=dict(color="black",lw=1.5),
#                 flierprops=dict(marker=".",markersize=4,alpha=0.5))
# for patch, cat in zip(bp["boxes"], cats_to_plot):
#     patch.set_facecolor(cat_colors.get(cat,"#9E9E9E"))
#     patch.set_alpha(0.75)
# ax.axhline(0, color="black", lw=0.5)
# ax.set_ylabel("|corr| с vol_BTC (test period)")
# ax.set_title("Корреляция фич с vol_BTC\nпо категориям", fontsize=10)
# ax.set_xticklabels(cats_to_plot, rotation=20, ha="right", fontsize=8)
# ax.grid(alpha=0.2, axis="y")
# ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

# plt.suptitle("Анализ полезности фич: IT + TDA комплексы",
#              fontsize=13, y=1.01)
# plt.tight_layout()
# plt.savefig("fig_feature_analysis.png", bbox_inches="tight", dpi=150)
# plt.savefig("fig_feature_analysis.pdf", bbox_inches="tight", dpi=150)
# plt.show()
# print("  Saved: fig_feature_analysis.png / .pdf")

# print(f"\n{'='*65}")
# print(f"  ИТОГ: лучший набор фич")
# print(f"{'='*65}")

# print(f"\n  Конфигурация '{best_cfg}' ({len(CONFIGS[best_cfg])} фич):")
# for f in CONFIGS[best_cfg]:
#     cat = categorize(f)
#     imp = avg_perm.get(f, 0)
#     print(f"    {f:<28} [{cat}]  perm={imp:>+.4f}")

# print(f"\n  Сравнение всех конфигураций:")
# for cfg in cfg_order:
#     sub = cfg_df[cfg_df.config == cfg]
#     print(f"    {cfg:<15}: ΔR²={sub['delta_r2'].mean():>+.4f}  "
#           f"rc={sub['res_corr'].mean():>+.4f}  "
#           f"↑{(sub['delta_r2']>0).sum()}/{len(sub)}")

# feat_analysis = {"perm": avg_perm, "cfg_df": cfg_df,
#                  "it_corrs": it_corrs, "tda_corrs": tda_corrs,
#                  "best_cfg": best_cfg, "best_cols": CONFIGS[best_cfg]}
# print(f"\nfeat_analysis сохранён в памяти")

Собираем все фичи...
IT-фич всего:  25
IT slim (10):  10
TDA-фич:       26

  1. КОРРЕЛЯЦИЯ ВСЕХ ФИЧ С VOL_BTC НА TEST-ПЕРИОДЕ

  IT-фичи (топ-10 по |corr| с vol_BTC test):
    pairwise_cov_min             +0.6662  █████████████ [removed]
    pairwise_cov_mean            +0.6416  ████████████ [removed]
    pairwise_cov_std             +0.6001  ████████████ [removed]
    pairwise_cov_max             +0.5999  ███████████ [removed]
    coinfo4_min                  +0.4073  ████████ [removed]
    pairwise_mi_mean             +0.4029  ████████ [slim]
    coinfo3_min                  +0.3937  ███████ [removed]
    coinfo3_mean                 +0.3849  ███████ [slim]
    coinfo4_mean                 +0.3849  ███████ [slim]
    pairwise_mi_min              +0.3779  ███████ [slim]

  TDA-фичи (топ-10 по |corr|):
    BI_ent_H1                    +0.3196  ██████
    BC_ent_H1                    -0.3150  ██████
    BI_L1_H1                     +0.3006  ██████
    BII_tp_H1                    -0.29

In [114]:
# import warnings; warnings.filterwarnings("ignore")
# import numpy as np
# import pandas as pd
# from sklearn.linear_model import LogisticRegression
# from sklearn.preprocessing import StandardScaler
# from sklearn.pipeline import Pipeline
# from sklearn.metrics import roc_auc_score, average_precision_score
# from sklearn.feature_selection import mutual_info_classif
# import matplotlib
# matplotlib.use("Agg")
# import matplotlib.pyplot as plt

# SPLIT_DATE = "2025-07-01"
# PURGE_BARS = 96
# QUANTILES  = [0.60, 0.70, 0.80, 0.90]
# HORIZONS   = {48:"1d", 96:"2d", 192:"4d"}
# ASSETS     = ["BTC","ETH","SOL","BNB","XRP","DOGE","ADA","AVAX","TRX"]
# VOL_WINDOW = 48
# ANN        = np.sqrt(365 * 48)
# N_IT_FEATS = 6    # top IT features for each asset
# N_TDA_FEATS= 5    # top TDA features

# def _logit(C=0.1):
#     return Pipeline([("sc", StandardScaler()),
#                      ("m", LogisticRegression(C=C, max_iter=2000,
#                                               class_weight="balanced",
#                                               solver="lbfgs"))])

# def safe_roc(y, s):
#     try:
#         return float(roc_auc_score(y, s))
#     except:
#         return np.nan

# def safe_pr(y, s):
#     try:
#         return float(average_precision_score(y, s))
#     except:
#         return np.nan

# def lift_at_k(y_true, score, k=0.05):
#     n = len(y_true)
#     top_n = max(1, int(n * k))
#     top_idx = np.argsort(score)[::-1][:top_n]
#     base = float(np.mean(y_true))
#     return float(np.mean(y_true[top_idx])) / base if base > 1e-8 else np.nan

# print("Building X_TDA...")
# tda_frames = [tda_A.add_prefix("A_")]
# for mode in ["I", "II", "C"]:
#     d = tda_B[mode]
#     df_m = pd.DataFrame({
#         f"B{mode}_tp_H0":  d["tp_H0"],
#         f"B{mode}_ent_H0": d["ent_H0"],
#         f"B{mode}_tp_H1":  d["tp_H1"],
#         f"B{mode}_ent_H1": d["ent_H1"],
#         f"B{mode}_L1_H0":  d["L1_H0"],
#         f"B{mode}_L1_H1":  d["L1_H1"],
#     }, index=pd.DatetimeIndex(d["times"]))
#     tda_frames.append(df_m)

# X_tda_raw = pd.concat(tda_frames, axis=1).sort_index()
# X_tda_ri  = X_tda_raw.reindex(ret.index, method="ffill")

# vol_now_all = {}
# for asset in ASSETS:
#     if asset in ret.columns:
#         vn = ret[asset].rolling(VOL_WINDOW).std() * ANN
#         vn_rank = vn.expanding(min_periods=100).rank(pct=True)
#         vol_now_all[asset] = vn_rank

# split_ts = pd.Timestamp(SPLIT_DATE)

# print("\n" + "=" * 65)
# print("  STRESS PREDICTION v2")
# print("  A=vol_now | B=vol+IT | C=vol+TDA | D=vol+IT+TDA")
# print("  Features are selected per asset using MI on the training set")
# print("=" * 65)

# rows = []

# for asset in ASSETS:
#     if asset not in ret.columns:
#         continue

#     r       = ret[asset]
#     vol_now = ret[asset].rolling(VOL_WINDOW).std() * ANN
#     vn_rank = vol_now_all[asset]

#     for h, h_label in HORIZONS.items():
#         vol_fwd = r.rolling(h).std().shift(-h) * ANN

#         for q in QUANTILES:
#             vol_q = vol_now.expanding(min_periods=100).quantile(q)
#             y_raw = (vol_fwd > vol_q).astype(float).dropna()

#             idx = (X_vol.index.intersection(X_it_slim.index)
#                    .intersection(X_tda_ri.index)
#                    .intersection(vn_rank.index)
#                    .intersection(y_raw.index))

#             valid = (X_vol.loc[idx].notna().all(1) &
#                      X_it_slim.loc[idx].notna().all(1) &
#                      X_tda_ri.loc[idx].notna().all(1) &
#                      vn_rank.loc[idx].notna() &
#                      y_raw.loc[idx].notna())
#             idx = idx[valid]

#             if len(idx) < 1000:
#                 continue

#             tr = idx < (split_ts - pd.Timedelta(minutes=30 * PURGE_BARS))
#             te = idx >= split_ts

#             if tr.sum() < 480 or te.sum() < 100:
#                 continue

#             ytr = y_raw.loc[idx[tr]]
#             yte = y_raw.loc[idx[te]]
#             pos_tr = float(ytr.mean())
#             pos_te = float(yte.mean())

#             if pos_tr < 0.03 or pos_tr > 0.97:
#                 continue
#             if yte.sum() < 5:
#                 continue

#             Xvol_tr = vn_rank.loc[idx[tr]].values.reshape(-1, 1)
#             Xvol_te = vn_rank.loc[idx[te]].values.reshape(-1, 1)
#             Xvol_tr_df = pd.DataFrame(Xvol_tr, index=idx[tr], columns=["vol_rank"])
#             Xvol_te_df = pd.DataFrame(Xvol_te, index=idx[te], columns=["vol_rank"])

#             Xi_tr_all = X_it_slim.loc[idx[tr]]
#             Xt_tr_all = X_tda_ri.loc[idx[tr]]

#             mi_it  = mutual_info_classif(Xi_tr_all.values, ytr.values, random_state=42)
#             mi_tda = mutual_info_classif(Xt_tr_all.values, ytr.values, random_state=42)

#             it_cols  = pd.Series(mi_it,  index=Xi_tr_all.columns).nlargest(N_IT_FEATS).index.tolist()
#             tda_cols = pd.Series(mi_tda, index=Xt_tr_all.columns).nlargest(N_TDA_FEATS).index.tolist()

#             Xit_tr = X_it_slim[it_cols].loc[idx[tr]]
#             Xit_te = X_it_slim[it_cols].loc[idx[te]]
#             Xtd_tr = X_tda_ri[tda_cols].loc[idx[tr]]
#             Xtd_te = X_tda_ri[tda_cols].loc[idx[te]]

#             yte_np = yte.values.astype(int)

#             def fit_predict(Xtr, Xte):
#                 try:
#                     m = _logit()
#                     m.fit(Xtr, ytr)
#                     return m.predict_proba(Xte)[:, 1]
#                 except:
#                     return np.full(len(Xte), ytr.mean())

#             sc_A = fit_predict(Xvol_tr_df, Xvol_te_df)

#             sc_B = fit_predict(
#                 pd.concat([Xvol_tr_df, Xit_tr], axis=1),
#                 pd.concat([Xvol_te_df, Xit_te], axis=1)
#             )

#             sc_C = fit_predict(
#                 pd.concat([Xvol_tr_df, Xtd_tr], axis=1),
#                 pd.concat([Xvol_te_df, Xtd_te], axis=1)
#             )

#             sc_D = fit_predict(
#                 pd.concat([Xvol_tr_df, Xit_tr, Xtd_tr], axis=1),
#                 pd.concat([Xvol_te_df, Xit_te, Xtd_te], axis=1)
#             )

#             rows.append({
#                 "asset":  asset,
#                 "horizon": h_label,
#                 "h": h,
#                 "q": q,
#                 "pos_tr": round(pos_tr, 3),
#                 "pos_te": round(pos_te, 3),

#                 "roc_A": round(safe_roc(yte_np, sc_A), 4),
#                 "roc_B": round(safe_roc(yte_np, sc_B), 4),
#                 "roc_C": round(safe_roc(yte_np, sc_C), 4),
#                 "roc_D": round(safe_roc(yte_np, sc_D), 4),

#                 "pr_A": round(safe_pr(yte_np, sc_A), 4),
#                 "pr_B": round(safe_pr(yte_np, sc_B), 4),
#                 "pr_C": round(safe_pr(yte_np, sc_C), 4),
#                 "pr_D": round(safe_pr(yte_np, sc_D), 4),

#                 "droc_B": round(safe_roc(yte_np, sc_B) - safe_roc(yte_np, sc_A), 4),
#                 "droc_C": round(safe_roc(yte_np, sc_C) - safe_roc(yte_np, sc_A), 4),
#                 "droc_D": round(safe_roc(yte_np, sc_D) - safe_roc(yte_np, sc_A), 4),

#                 "dpr_B": round(safe_pr(yte_np, sc_B) - safe_pr(yte_np, sc_A), 4),
#                 "dpr_C": round(safe_pr(yte_np, sc_C) - safe_pr(yte_np, sc_A), 4),
#                 "dpr_D": round(safe_pr(yte_np, sc_D) - safe_pr(yte_np, sc_A), 4),

#                 "lift5_A": round(lift_at_k(yte_np, sc_A, 0.05), 3),
#                 "lift5_D": round(lift_at_k(yte_np, sc_D, 0.05), 3),

#                 "n_train": int(tr.sum()),
#                 "n_test": int(te.sum()),
#             })

# df = pd.DataFrame(rows)
# print(f"Total rows: {len(df)}")

# print(f"\n{'=' * 78}")
# print(f"  ROC-AUC — aggregate by threshold (mean over assets and horizons)")
# print(f"  A=vol_now | B=vol+IT | C=vol+TDA | D=vol+IT+TDA")
# print(f"{'=' * 78}")
# print(f"\n  {'q':>5} {'pos%':>5} "
#       f"{'ROC_A':>7} {'ROC_B':>7} {'ROC_C':>7} {'ROC_D':>7} "
#       f"{'ΔROC_B':>8} {'ΔROC_C':>8} {'ΔROC_D':>8} "
#       f"{'D>A':>6} {'Lift5_D':>9}")
# print("  " + "-" * 82)

# for q in QUANTILES:
#     sub = df[df.q == q]
#     if sub.empty:
#         continue
#     beats = (sub["droc_D"] > 0.01).sum()
#     print(f"  Q{q:.0%}"
#           f" {sub['pos_te'].mean():>4.0%}"
#           f" {sub['roc_A'].mean():>7.4f}"
#           f" {sub['roc_B'].mean():>7.4f}"
#           f" {sub['roc_C'].mean():>7.4f}"
#           f" {sub['roc_D'].mean():>7.4f}"
#           f" {sub['droc_B'].mean():>+8.4f}"
#           f" {sub['droc_C'].mean():>+8.4f}"
#           f" {sub['droc_D'].mean():>+8.4f}"
#           f" {beats:>4}/{len(sub)}"
#           f" {sub['lift5_D'].mean():>9.3f}")

# print(f"\n{'=' * 78}")
# print(f"  BY ASSET — threshold Q80  |  ROC-AUC + PR-AUC")
# print(f"{'=' * 78}")
# sub80 = df[df.q == 0.80]

# print(f"\n  {'asset':<6} {'pos%':>5} "
#       f"{'ROC_A':>7} {'ROC_B':>7} {'ROC_C':>7} {'ROC_D':>7} "
#       f"{'ΔROC_D':>8} "
#       f"{'PR_A':>7} {'PR_D':>7} {'ΔPR_D':>8}  best")
# print("  " + "-" * 85)

# for asset in ASSETS:
#     g = sub80[sub80.asset == asset]
#     if g.empty:
#         continue

#     best_roc = max(
#         {"A": g["roc_A"].mean(),
#          "B": g["roc_B"].mean(),
#          "C": g["roc_C"].mean(),
#          "D": g["roc_D"].mean()},
#         key=lambda k: {"A": g["roc_A"].mean(),
#                        "B": g["roc_B"].mean(),
#                        "C": g["roc_C"].mean(),
#                        "D": g["roc_D"].mean()}[k]
#     )

#     print(f"  {asset:<6}"
#           f" {g['pos_te'].mean():>4.0%}"
#           f" {g['roc_A'].mean():>7.4f}"
#           f" {g['roc_B'].mean():>7.4f}"
#           f" {g['roc_C'].mean():>7.4f}"
#           f" {g['roc_D'].mean():>7.4f}"
#           f" {g['droc_D'].mean():>+8.4f}"
#           f" {g['pr_A'].mean():>7.4f}"
#           f" {g['pr_D'].mean():>7.4f}"
#           f" {g['dpr_D'].mean():>+8.4f}"
#           f"  → {best_roc}")

# print(f"\n{'=' * 78}")
# print(f"  TOP 10 by ΔROC (configuration D = vol+IT+TDA)")
# print(f"{'=' * 78}")
# top10 = df.nlargest(10, "droc_D")

# print(f"\n  {'asset':<6} {'h':<4} {'Q':>5} {'pos%':>5} "
#       f"{'ROC_A':>7} {'ROC_D':>7} {'ΔROC':>8} "
#       f"{'PR_A':>7} {'PR_D':>7} {'ΔPR':>8} {'Lift5':>7}")
# print("  " + "-" * 72)

# for _, row in top10.iterrows():
#     print(f"  {row['asset']:<6} {row['horizon']:<4} Q{row['q']:.0%}"
#           f" {row['pos_te']:>4.0%}"
#           f" {row['roc_A']:>7.4f}"
#           f" {row['roc_D']:>7.4f}"
#           f" {row['droc_D']:>+8.4f}"
#           f" {row['pr_A']:>7.4f}"
#           f" {row['pr_D']:>7.4f}"
#           f" {row['dpr_D']:>+8.4f}"
#           f" {row['lift5_D']:>7.3f}")

# print(f"\n{'=' * 78}")
# print(f"  SUMMARY")
# print(f"{'=' * 78}")
# n = len(df)

# for cfg, col_roc, col_pr in [
#     ("B: vol+IT",     "droc_B", "dpr_B"),
#     ("C: vol+TDA",    "droc_C", "dpr_C"),
#     ("D: vol+IT+TDA", "droc_D", "dpr_D"),
# ]:
#     beats_roc = (df[col_roc] > 0.01).sum()
#     beats_pr  = (df[col_pr]  > 0.01).sum()
#     print(f"  {cfg:<18}: ΔROC={df[col_roc].mean():>+.4f}  "
#           f"ROC↑ {beats_roc:>2}/{n}  "
#           f"ΔPR={df[col_pr].mean():>+.4f}  PR↑ {beats_pr:>2}/{n}")

# print("\nBuilding plots...")
# fig, axes = plt.subplots(2, 2, figsize=(14, 10))
# q_labels = [f"Q{q:.0%}" for q in QUANTILES]
# colors = {"A": "#9E9E9E", "B": "#2196F3", "C": "#FF9800", "D": "#4CAF50"}

# ax = axes[0, 0]
# for k in ["A", "B", "C", "D"]:
#     vals = [df[df.q == q][f"roc_{k}"].mean() for q in QUANTILES]
#     ls = "--" if k == "C" else "-"
#     label = {"A": "vol_now", "B": "vol+IT", "C": "vol+TDA", "D": "vol+IT+TDA"}[k]
#     ax.plot(q_labels, vals, "o" + ls, color=colors[k], lw=2, label=label, ms=7)
# ax.axhline(0.5, color="gray", lw=0.8, ls=":", alpha=0.5)
# ax.set_ylabel("ROC-AUC")
# ax.set_xlabel("Stress threshold")
# ax.set_title("ROC-AUC by threshold\n(mean over 9 assets and 3 horizons)", fontsize=10)
# ax.legend(fontsize=8)
# ax.grid(alpha=0.2)
# ax.spines["top"].set_visible(False)
# ax.spines["right"].set_visible(False)

# ax = axes[0, 1]
# x = np.arange(len(QUANTILES))
# w = 0.25
# for ci, k in enumerate(["B", "C", "D"]):
#     vals = [df[df.q == q][f"droc_{k}"].mean() for q in QUANTILES]
#     label = {"B": "+IT", "C": "+TDA", "D": "+IT+TDA"}[k]
#     ax.bar(x + (ci - 1) * w, vals, w * 0.9, label=label, color=colors[k], alpha=0.85)
# ax.axhline(0, color="black", lw=0.8)
# ax.set_xticks(x)
# ax.set_xticklabels(q_labels)
# ax.set_ylabel("ΔROC-AUC vs vol_now baseline")
# ax.set_title("Incremental ΔROC over vol_now\n(by threshold)", fontsize=10)
# ax.legend(fontsize=8)
# ax.grid(alpha=0.2, axis="y")
# ax.spines["top"].set_visible(False)
# ax.spines["right"].set_visible(False)

# ax = axes[1, 0]
# asset_order = sub80.groupby("asset")["roc_D"].mean().sort_values(ascending=False).index.tolist()
# x3 = np.arange(len(asset_order))
# w3 = 0.2
# for ci, k in enumerate(["A", "B", "C", "D"]):
#     vals = [sub80[sub80.asset == a][f"roc_{k}"].mean() for a in asset_order]
#     label = {"A": "vol_now", "B": "vol+IT", "C": "vol+TDA", "D": "vol+IT+TDA"}[k]
#     ax.bar(x3 + (ci - 1.5) * w3, vals, w3 * 0.9, label=label, color=colors[k], alpha=0.85)
# ax.axhline(0.5, color="gray", lw=0.8, ls=":", alpha=0.5)
# ax.set_xticks(x3)
# ax.set_xticklabels(list(asset_order), fontsize=9)
# ax.set_ylabel("ROC-AUC")
# ax.set_title("ROC-AUC by asset (Q80)\n(mean over horizons)", fontsize=10)
# ax.legend(fontsize=8)
# ax.grid(alpha=0.2, axis="y")
# ax.spines["top"].set_visible(False)
# ax.spines["right"].set_visible(False)

# ax = axes[1, 1]
# pivot = df.pivot_table(index="asset", columns="q", values="roc_D", aggfunc="mean")
# pivot = pivot.reindex(ASSETS)[QUANTILES]
# im = ax.imshow(pivot.values, cmap="RdYlGn", vmin=0.50, vmax=0.85, aspect="auto")
# ax.set_xticks(range(len(QUANTILES)))
# ax.set_xticklabels(q_labels)
# ax.set_yticks(range(len(ASSETS)))
# ax.set_yticklabels(ASSETS)

# for i in range(len(ASSETS)):
#     for j in range(len(QUANTILES)):
#         val = pivot.values[i, j]
#         if np.isfinite(val):
#             col = "white" if val > 0.72 else "black"
#             ax.text(j, i, f"{val:.3f}", ha="center", va="center",
#                     fontsize=8, color=col)

# plt.colorbar(im, ax=ax, label="ROC-AUC (vol+IT+TDA)")
# ax.set_title("ROC-AUC heatmap (vol+IT+TDA)\nby asset × threshold", fontsize=10)

# plt.suptitle(
#     "Stress prediction v2: A=vol_now | B=vol+IT | C=vol+TDA | D=vol+IT+TDA\n"
#     f"Holdout {SPLIT_DATE} →  |  Per-asset feature selection  |  Horizons: 1d, 2d, 4d",
#     fontsize=11, y=1.01
# )
# plt.tight_layout()
# plt.savefig("fig_stress_v2.png", bbox_inches="tight", dpi=150)
# plt.savefig("fig_stress_v2.pdf", bbox_inches="tight", dpi=150)
# plt.show()
# print("  Saved: fig_stress_v2.png / .pdf")

# stress_v2 = df.copy()
# print(f"\nstress_v2 saved in memory ({len(df)} rows)")

Building X_TDA...

  STRESS PREDICTION v2
  A=vol_now | B=vol+IT | C=vol+TDA | D=vol+IT+TDA
  Features are selected per asset using MI on the training set
Total rows: 91

  ROC-AUC — aggregate by threshold (mean over assets and horizons)
  A=vol_now | B=vol+IT | C=vol+TDA | D=vol+IT+TDA

      q  pos%   ROC_A   ROC_B   ROC_C   ROC_D   ΔROC_B   ΔROC_C   ΔROC_D    D>A   Lift5_D
  ----------------------------------------------------------------------------------
  Q60%  45%  0.6893  0.6875  0.7049  0.7154  -0.0019  +0.0156  +0.0260   19/26     1.678
  Q70%  31%  0.6863  0.6899  0.6931  0.7060  +0.0037  +0.0069  +0.0197   13/25     2.072
  Q80%  16%  0.6622  0.6757  0.6926  0.7114  +0.0135  +0.0305  +0.0492   21/24     2.458
  Q90%   5%  0.6261  0.6307  0.6227  0.6436  +0.0046  -0.0034  +0.0175    8/16     0.939

  BY ASSET — threshold Q80  |  ROC-AUC + PR-AUC

  asset   pos%   ROC_A   ROC_B   ROC_C   ROC_D   ΔROC_D    PR_A    PR_D    ΔPR_D  best
  -----------------------------------------

# ДЛЯ ФИЧЕЙ РЕЛЕВАНТНО

In [ ]:
# import warnings; warnings.filterwarnings("ignore")
# import numpy as np
# import pandas as pd
# from sklearn.linear_model import Ridge, RidgeCV
# from sklearn.preprocessing import StandardScaler
# from sklearn.pipeline import Pipeline
# from sklearn.model_selection import KFold, cross_val_score
# import matplotlib
# matplotlib.use("Agg")
# import matplotlib.pyplot as plt

# SPLIT_DATE = "2025-01-01"
# PURGE_BARS = 96
# HORIZONS   = {96:"2d", 192:"4d"}      
# ASSETS     = ["BTC","ETH","DOGE","XRP","SOL","BNB"]
# VOL_WINDOW = 48
# ANN        = np.sqrt(365 * 48)
# RNG        = np.random.default_rng(42)
# N_PERM     = 10

# def _ridge():
#     return Pipeline([("sc", StandardScaler()),
#                      ("m", RidgeCV(alphas=[0.01,0.1,1,10,100]))])

# def oos_r2(y_true, y_pred, naive_val):
#     mse_m = np.mean((y_true - y_pred)**2)
#     mse_n = np.mean((y_true - naive_val)**2)
#     return float(1.0 - mse_m / mse_n) if mse_n > 1e-12 else np.nan

# def _get_y(asset, h):
#     col = f"delta_vol_{h}"
#     for src in [targets_extra if 'targets_extra' in dir() else {},
#                 targets, targets_ext]:
#         if asset in src and col in src[asset].columns:
#             return src[asset][col].dropna()
#     return None

# try: _ = targets_extra
# except NameError: targets_extra = {}

# print("Собираем все фичи...")

# tda_frames = [tda_A.add_prefix("A_")]
# for mode in ["I","II","C"]:
#     d = tda_B[mode]
#     idx_tda = pd.DatetimeIndex(d["times"])
#     df_m = pd.DataFrame({
#         f"B{mode}_tp_H0":  d["tp_H0"],  f"B{mode}_ent_H0": d["ent_H0"],
#         f"B{mode}_tp_H1":  d["tp_H1"],  f"B{mode}_ent_H1": d["ent_H1"],
#         f"B{mode}_L1_H0":  d["L1_H0"],  f"B{mode}_L1_H1":  d["L1_H1"],
#     }, index=idx_tda)
#     tda_frames.append(df_m)

# X_tda_raw = pd.concat(tda_frames, axis=1).sort_index()
# X_tda_ri  = X_tda_raw.reindex(ret.index, method="ffill")

# IT_SLIM = ["coinfo3_std","kl_BNB","coinfo4_std","pairwise_mi_mean",
#            "coinfo4_mean","pairwise_mi_std","coinfo3_mean",
#            "coinfo4_max","kl_AVAX","pairwise_mi_min"]

# IT_REMOVED = ["pairwise_cov_std","pairwise_cov_max","pairwise_cov_min",
#               "pairwise_cov_mean","pairwise_mi_max","kl_BTC","kl_TRX",
#               "kl_XRP","kl_SOL","coinfo4_min","kl_DOGE","kl_ETH",
#               "coinfo3_max","kl_ADA","coinfo3_min","pairwise_mi_min",
#               "kl_AVAX","coinfo4_max"]

# IT_ALL_25 = IT_SLIM + [c for c in IT_REMOVED if c in X_it_slim.columns
#                         and c not in IT_SLIM]
# IT_ALL_25 = [c for c in IT_ALL_25 if c in X_it_slim.columns]

# TDA_COLS = list(X_tda_ri.columns)

# print(f"IT-фич всего:  {len(IT_ALL_25)}")
# print(f"IT slim (10):  {len(IT_SLIM)}")
# print(f"TDA-фич:       {len(TDA_COLS)}")

# print("\n" + "="*65)
# print("  1. КОРРЕЛЯЦИЯ ВСЕХ ФИЧ С VOL_BTC НА TEST-ПЕРИОДЕ")
# print("="*65)

# split_ts  = pd.Timestamp(SPLIT_DATE)
# vol_btc   = ret["BTC"].rolling(VOL_WINDOW).std() * ANN
# vol_test  = vol_btc[vol_btc.index >= split_ts]

# it_corrs = {}
# for col in IT_ALL_25:
#     if col not in X_it_slim.columns: continue
#     s = X_it_slim[col].reindex(vol_test.index, method="ffill")
#     it_corrs[col] = float(s.corr(vol_test))

# tda_corrs = {}
# for col in TDA_COLS:
#     s = X_tda_ri[col].reindex(vol_test.index, method="ffill")
#     tda_corrs[col] = float(s.corr(vol_test))

# print(f"\n  IT-фичи (топ-10 по |corr| с vol_BTC test):")
# it_sorted = sorted(it_corrs.items(), key=lambda x: abs(x[1]), reverse=True)
# for feat, c in it_sorted[:10]:
#     tag = " [slim]" if feat in IT_SLIM else " [removed]"
#     bar = "█" * max(0, int(abs(c)*20))
#     print(f"    {feat:<28} {c:>+.4f}  {bar}{tag}")

# print(f"\n  TDA-фичи (топ-10 по |corr|):")
# tda_sorted = sorted(tda_corrs.items(), key=lambda x: abs(x[1]), reverse=True)
# for feat, c in tda_sorted[:10]:
#     bar = "█" * max(0, int(abs(c)*20))
#     print(f"    {feat:<28} {c:>+.4f}  {bar}")

# print("\n" + "="*65)
# print("  2. PERMUTATION IMPORTANCE НА TRAIN (BTC+ETH, h=4d, logvol)")
# print("="*65)

# perm_results = {}
# for pilot_asset in ["BTC","ETH"]:
#     r = ret[pilot_asset]
#     vol_now = r.rolling(VOL_WINDOW).std() * ANN
#     vol_fwd = r.rolling(192).std().shift(-192) * ANN
#     y_logvol = np.log(vol_fwd.clip(lower=1e-8)).dropna()

#     idx = (X_vol.index.intersection(X_it_slim.index)
#            .intersection(X_tda_ri.index).intersection(y_logvol.index))
#     valid = (X_vol.loc[idx].notna().all(1) &
#              X_it_slim.loc[idx].notna().all(1) &
#              X_tda_ri.loc[idx].notna().all(1) &
#              y_logvol.loc[idx].notna())
#     idx = idx[valid]
#     tr   = idx < (split_ts - pd.Timedelta(minutes=30*PURGE_BARS))
#     idx_tr = idx[tr]

#     Xv_tr = X_vol.loc[idx_tr]
#     ck    = Xv_tr.corrwith(y_logvol.loc[idx_tr]).abs().dropna()
#     top3  = ck.nlargest(3).index.tolist()
#     mdl_b = _ridge(); mdl_b.fit(Xv_tr[top3], y_logvol.loc[idx_tr])
#     res_tr = y_logvol.loc[idx_tr].values - mdl_b.predict(Xv_tr[top3])

#     Xi_all_tr = pd.concat([
#         X_it_slim.loc[idx_tr],
#         X_tda_ri.loc[idx_tr]
#     ], axis=1)
#     Xi_arr  = Xi_all_tr.values
#     res_arr = res_tr
#     feat_names = list(Xi_all_tr.columns)

#     kf = KFold(n_splits=5, shuffle=False)
#     base_cv = float(np.mean(cross_val_score(_ridge(), Xi_arr, res_arr,
#                                              cv=kf, scoring="r2")))

#     perm_scores_asset = {}
#     for fi, feat in enumerate(feat_names):
#         deltas = []
#         for _ in range(N_PERM):
#             Xp = Xi_arr.copy()
#             Xp[:, fi] = RNG.permutation(Xp[:, fi])
#             sc = float(np.mean(cross_val_score(_ridge(), Xp, res_arr,
#                                                cv=kf, scoring="r2")))
#             deltas.append(sc - base_cv)
#         perm_scores_asset[feat] = float(np.mean(deltas))

#     perm_results[pilot_asset] = perm_scores_asset
#     print(f"  {pilot_asset} baseline CV R²={base_cv:.4f}")

# all_feats = list(perm_results["BTC"].keys())
# avg_perm  = {f: np.mean([perm_results[a].get(f,0) for a in ["BTC","ETH"]])
#              for f in all_feats}

# def categorize(feat):
#     if feat in IT_SLIM:    return "IT_slim"
#     if feat in IT_ALL_25:  return "IT_removed"
#     if feat.startswith("A_"): return "TDA_A"
#     if feat.startswith("BI_"): return "TDA_BI"
#     if feat.startswith("BII_"): return "TDA_BII"
#     if feat.startswith("BC_"): return "TDA_BC"
#     return "other"

# cat_colors = {
#     "IT_slim":"#2196F3","IT_removed":"#90CAF9",
#     "TDA_A":"#4CAF50","TDA_BI":"#FF9800",
#     "TDA_BII":"#9C27B0","TDA_BC":"#F44336","other":"#9E9E9E"
# }

# print(f"\n  Топ-15 полезных фич (Δ<0 = помогает):")
# sorted_perm = sorted(avg_perm.items(), key=lambda x: x[1])
# for feat, delta in sorted_perm[:15]:
#     cat = categorize(feat)
#     print(f"    {feat:<28} {delta:>+.4f}  [{cat}]")

# print(f"\n  Топ-10 вредных фич (Δ>0 = вредит):")
# for feat, delta in sorted(avg_perm.items(), key=lambda x: -x[1])[:10]:
#     cat = categorize(feat)
#     print(f"    {feat:<28} {delta:>+.4f}  [{cat}]")

# best_tda = [f for f in TDA_COLS if avg_perm.get(f,0) < -0.002]
# neutral_tda = [f for f in TDA_COLS if -0.002 <= avg_perm.get(f,0) <= 0.002]
# bad_tda  = [f for f in TDA_COLS if avg_perm.get(f,0) > 0.002]
# print(f"\n  TDA-фичи: {len(best_tda)} полезных, "
#       f"{len(neutral_tda)} нейтральных, {len(bad_tda)} вредных")
# print(f"  Лучшие TDA: {best_tda[:8]}")

# print("\n" + "="*65)
# print("  3. СРАВНЕНИЕ КОНФИГУРАЦИЙ НА HOLDOUT")
# print("="*65)

# CONFIGS = {
#     "slim_IT":    IT_SLIM,
#     "best_IT":    [f for f,d in sorted_perm[:15] if f in IT_ALL_25],
#     "slim+TDA":   IT_SLIM + best_tda[:6],
#     "best+TDA":   [f for f,d in sorted_perm[:12]
#                    if f in IT_ALL_25 or f in TDA_COLS],
# }
# CONFIGS = {k: list(dict.fromkeys(v)) for k,v in CONFIGS.items()}

# for name, cols in CONFIGS.items():
#     print(f"  {name}: {len(cols)} фич")

# config_rows = []
# for asset in ASSETS:
#     r = ret[asset] if asset in ret.columns else None
#     if r is None: continue
#     vol_now = r.rolling(VOL_WINDOW).std() * ANN
#     vol_fwd_h = {}
#     for h in HORIZONS:
#         vol_fwd_h[h] = r.rolling(h).std().shift(-h) * ANN

#     for h, h_label in HORIZONS.items():
#         y_raw = np.log(vol_fwd_h[h].clip(lower=1e-8)).dropna()

#         for cfg_name, cfg_cols in CONFIGS.items():
#             it_cols_used  = [c for c in cfg_cols if c in X_it_slim.columns]
#             tda_cols_used = [c for c in cfg_cols if c in X_tda_ri.columns]

#             Xi_cfg = pd.concat(
#                 [X_it_slim[it_cols_used]] +
#                 ([X_tda_ri[tda_cols_used]] if tda_cols_used else []),
#                 axis=1
#             ) if it_cols_used or tda_cols_used else X_it_slim[IT_SLIM]

#             idx = (X_vol.index.intersection(Xi_cfg.index).intersection(y_raw.index))
#             valid = (X_vol.loc[idx].notna().all(1) &
#                      Xi_cfg.loc[idx].notna().all(1) &
#                      y_raw.loc[idx].notna())
#             idx = idx[valid]
#             if len(idx) < 1000: continue

#             tr = idx < (split_ts - pd.Timedelta(minutes=30*PURGE_BARS))
#             te = idx >= split_ts
#             if tr.sum() < 480 or te.sum() < 100: continue

#             Xv_tr = X_vol.loc[idx[tr]]; Xv_te = X_vol.loc[idx[te]]
#             Xi_tr = Xi_cfg.loc[idx[tr]]; Xi_te = Xi_cfg.loc[idx[te]]
#             ytr   = y_raw.loc[idx[tr]];  yte   = y_raw.loc[idx[te]]
#             naive = float(ytr.mean())

#             ck   = Xv_tr.corrwith(ytr).abs().dropna()
#             top3 = ck.nlargest(3).index.tolist()
#             mb   = _ridge(); mb.fit(Xv_tr[top3], ytr)
#             pb_tr = mb.predict(Xv_tr[top3])
#             pb_te = mb.predict(Xv_te[top3])

#             res_tr = ytr.values - pb_tr
#             mc = _ridge(); mc.fit(Xi_tr, res_tr)
#             pc_te = mc.predict(Xi_te)
#             pred  = pb_te + pc_te

#             r2_b = oos_r2(yte.values, pb_te, naive)
#             r2_c = oos_r2(yte.values, pred,  naive)
#             rc   = float(pd.Series(pc_te).corr(pd.Series(yte.values - pb_te)))

#             config_rows.append({
#                 "asset":   asset, "horizon": h_label, "h": h,
#                 "config":  cfg_name,
#                 "r2_base": round(r2_b, 4),
#                 "r2_cfg":  round(r2_c, 4),
#                 "delta_r2":round(r2_c - r2_b, 4),
#                 "res_corr":round(rc, 4),
#             })

# cfg_df = pd.DataFrame(config_rows)

# print(f"\n  Среднее ΔR² (logvol) по конфигурациям:")
# print(f"  {'config':<15} {'mean_ΔR²':>10} {'ΔR²>0':>8} {'mean_rc':>9}")
# print("  " + "-"*45)
# for cfg in CONFIGS.keys():
#     sub = cfg_df[cfg_df.config == cfg]
#     if sub.empty: continue
#     pos = (sub["delta_r2"] > 0).sum()
#     print(f"  {cfg:<15} {sub['delta_r2'].mean():>+10.4f} "
#           f"{pos:>5}/{len(sub)} {sub['res_corr'].mean():>+9.4f}")

# best_cfg = cfg_df.groupby("config")["delta_r2"].mean().idxmax()
# print(f"\n  Лучшая конфигурация: '{best_cfg}'")
# sub_best = cfg_df[cfg_df.config == best_cfg]
# print(f"  {'актив':<6} {'h':<4} {'R²_base':>9} {'R²_best':>9} {'ΔR²':>8}")
# print("  " + "-"*40)
# prev = None
# for _, row in sub_best.sort_values(["asset","h"]).iterrows():
#     if prev and prev != row["asset"]: print("  " + "·"*40)
#     prev = row["asset"]
#     mark = "↑" if row["delta_r2"] > 0.005 else ("↓" if row["delta_r2"] < -0.005 else "~")
#     print(f"  {row['asset']:<6} {row['horizon']:<4}"
#           f" {row['r2_base']:>+9.4f} {row['r2_cfg']:>+9.4f}"
#           f" {row['delta_r2']:>+8.4f} {mark}")

# print("\nСтроим графики...")

# fig, axes = plt.subplots(1, 3, figsize=(18, 7))

# ax = axes[0]
# feats_sorted = [f for f,_ in sorted_perm]
# vals_sorted  = [avg_perm[f] for f in feats_sorted]
# colors_bar   = [cat_colors.get(categorize(f), "#9E9E9E") for f in feats_sorted]
# y_pos = np.arange(len(feats_sorted))
# ax.barh(y_pos, vals_sorted, color=colors_bar, alpha=0.85, height=0.8)
# ax.axvline(0, color="black", lw=0.8)
# ax.axvline(-0.002, color="green",  lw=0.8, ls=":", alpha=0.6)
# ax.axvline(+0.002, color="red",    lw=0.8, ls=":", alpha=0.6)
# ax.set_yticks(y_pos)
# ax.set_yticklabels(feats_sorted, fontsize=7)
# ax.set_xlabel("Δ CV R² при перемешивании\n(< 0 = фича полезна)")
# ax.set_title("Permutation importance\n(BTC+ETH, h=4d, logvol)", fontsize=10)
# ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
# from matplotlib.patches import Patch
# legend_elems = [Patch(facecolor=c, label=k, alpha=0.85)
#                 for k,c in cat_colors.items() if k != "other"]
# ax.legend(handles=legend_elems, fontsize=7, loc="lower right")

# ax = axes[1]
# cfg_order = list(CONFIGS.keys())
# asset_order = ASSETS
# x = np.arange(len(asset_order))
# w = 0.18
# cfg_colors = ["#90CAF9","#2196F3","#4CAF50","#FF9800"]
# for ci, cfg in enumerate(cfg_order):
#     sub = cfg_df[cfg_df.config == cfg]
#     vals = []
#     for asset in asset_order:
#         v = sub[sub.asset==asset]["delta_r2"].mean()
#         vals.append(v if not np.isnan(v) else 0)
#     ax.bar(x + (ci-1.5)*w, vals, w*0.9,
#            label=cfg, color=cfg_colors[ci], alpha=0.85)
# ax.axhline(0, color="black", lw=0.8)
# ax.set_xticks(x); ax.set_xticklabels(asset_order)
# ax.set_ylabel("Среднее ΔR² (все горизонты)")
# ax.set_title("ΔR² по конфигурациям × активам\n(logvol, holdout 2025)", fontsize=10)
# ax.legend(fontsize=8); ax.grid(alpha=0.2, axis="y")
# ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

# ax = axes[2]
# cat_corrs = {}
# for feat, c in {**it_corrs, **tda_corrs}.items():
#     cat = categorize(feat)
#     if cat not in cat_corrs: cat_corrs[cat] = []
#     cat_corrs[cat].append(c)

# cats_to_plot = ["IT_slim","IT_removed","TDA_A","TDA_BI","TDA_BII","TDA_BC"]
# data_plot    = [cat_corrs.get(c,[0]) for c in cats_to_plot]
# bp = ax.boxplot(data_plot, labels=cats_to_plot, patch_artist=True,
#                 medianprops=dict(color="black",lw=1.5),
#                 flierprops=dict(marker=".",markersize=4,alpha=0.5))
# for patch, cat in zip(bp["boxes"], cats_to_plot):
#     patch.set_facecolor(cat_colors.get(cat,"#9E9E9E"))
#     patch.set_alpha(0.75)
# ax.axhline(0, color="black", lw=0.5)
# ax.set_ylabel("|corr| с vol_BTC (test period)")
# ax.set_title("Корреляция фич с vol_BTC\nпо категориям", fontsize=10)
# ax.set_xticklabels(cats_to_plot, rotation=20, ha="right", fontsize=8)
# ax.grid(alpha=0.2, axis="y")
# ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

# plt.suptitle("Анализ полезности фич: IT + TDA комплексы",
#              fontsize=13, y=1.01)
# plt.tight_layout()
# plt.savefig("fig_feature_analysis.png", bbox_inches="tight", dpi=150)
# plt.savefig("fig_feature_analysis.pdf", bbox_inches="tight", dpi=150)
# plt.show()
# print("  Saved: fig_feature_analysis.png / .pdf")

# print(f"\n{'='*65}")
# print(f"  ИТОГ: лучший набор фич")
# print(f"{'='*65}")

# print(f"\n  Конфигурация '{best_cfg}' ({len(CONFIGS[best_cfg])} фич):")
# for f in CONFIGS[best_cfg]:
#     cat = categorize(f)
#     imp = avg_perm.get(f, 0)
#     print(f"    {f:<28} [{cat}]  perm={imp:>+.4f}")

# print(f"\n  Сравнение всех конфигураций:")
# for cfg in cfg_order:
#     sub = cfg_df[cfg_df.config == cfg]
#     print(f"    {cfg:<15}: ΔR²={sub['delta_r2'].mean():>+.4f}  "
#           f"rc={sub['res_corr'].mean():>+.4f}  "
#           f"↑{(sub['delta_r2']>0).sum()}/{len(sub)}")

# feat_analysis = {"perm": avg_perm, "cfg_df": cfg_df,
#                  "it_corrs": it_corrs, "tda_corrs": tda_corrs,
#                  "best_cfg": best_cfg, "best_cols": CONFIGS[best_cfg]}
# print(f"\nfeat_analysis сохранён в памяти")

In [196]:
print("\n" + "=" * 65)
print("  2. PERMUTATION IMPORTANCE ON TRAIN (BTC+ETH, h=4d, logvol)")
print("=" * 65)

perm_results = {}
for pilot_asset in ["BTC", "ETH"]:
    r = ret[pilot_asset]
    vol_fwd = r.rolling(192).std().shift(-192) * ANN
    y_logvol = np.log(vol_fwd.clip(lower=1e-8)).dropna()

    idx = (X_vol.index
           .intersection(X_it_aug.index)
           .intersection(X_tda_full.index)
           .intersection(y_logvol.index))

    valid = (X_vol.loc[idx].notna().all(1) &
             X_it_aug.loc[idx].notna().all(1) &
             X_tda_full.loc[idx].notna().all(1) &
             y_logvol.loc[idx].notna())
    idx = idx[valid]

    tr = idx < (split_ts - pd.Timedelta(minutes=30 * PURGE_BARS))
    idx_tr = idx[tr]

    Xv_tr = X_vol.loc[idx_tr]
    ck = Xv_tr.corrwith(y_logvol.loc[idx_tr]).abs().dropna()
    top3 = ck.nlargest(3).index.tolist()

    mdl_b = _ridge()
    mdl_b.fit(Xv_tr[top3], y_logvol.loc[idx_tr])
    res_tr = y_logvol.loc[idx_tr].values - mdl_b.predict(Xv_tr[top3])

    Xi_all_tr = pd.concat([
        X_it_aug.loc[idx_tr],
        X_tda_full.loc[idx_tr],
    ], axis=1)

    Xi_arr = Xi_all_tr.values
    feat_names = list(Xi_all_tr.columns)

    kf = KFold(n_splits=5, shuffle=False)
    base_cv = float(np.mean(
        cross_val_score(_ridge(), Xi_arr, res_tr, cv=kf, scoring="r2")
    ))

    perm_scores_asset = {}
    for fi, feat in enumerate(feat_names):
        deltas = []
        for _ in range(N_PERM):
            Xp = Xi_arr.copy()
            Xp[:, fi] = RNG.permutation(Xp[:, fi])
            sc = float(np.mean(
                cross_val_score(_ridge(), Xp, res_tr, cv=kf, scoring="r2")
            ))
            deltas.append(sc - base_cv)
        perm_scores_asset[feat] = float(np.mean(deltas))

    perm_results[pilot_asset] = perm_scores_asset
    print(f"  {pilot_asset} baseline CV R² = {base_cv:.4f}")

all_feats = list(perm_results["BTC"].keys())
avg_perm = {
    f: np.mean([perm_results[a].get(f, 0.0) for a in ["BTC", "ETH"]])
    for f in all_feats
}

def categorize(feat):
    if feat in IT_ALL:
        return "IT"
    if feat in TDA_COLS:
        return "TDA"
    return "other"

cat_colors = {
    "IT":    "#2196F3",
    "TDA":   "#FF9800",
    "other": "#9E9E9E",
}

print("\n" + "=" * 65)
print("  3. HOLDOUT CONFIGURATION COMPARISON")
print("=" * 65)

CONFIGS = {
    "slim_IT": IT_SLIM,
    "best_IT": [f for f, d in sorted_perm[:20] if f in IT_ALL],
    "slim+TDA": IT_SLIM + best_tda[:6],
    "best+TDA": [f for f, d in sorted_perm[:16] if f in IT_ALL or f in TDA_COLS],
}

CONFIGS = {k: list(dict.fromkeys(v)) for k, v in CONFIGS.items()}

for name, cols in CONFIGS.items():
    print(f"  {name}: {len(cols)} features")

config_rows = []

for asset in ASSETS:
    r = ret[asset] if asset in ret.columns else None
    if r is None:
        continue

    vol_fwd_h = {}
    for h in HORIZONS:
        vol_fwd_h[h] = r.rolling(h).std().shift(-h) * ANN

    for h, h_label in HORIZONS.items():
        y_raw = np.log(vol_fwd_h[h].clip(lower=1e-8)).dropna()

        for cfg_name, cfg_cols in CONFIGS.items():
            it_cols_used = [c for c in cfg_cols if c in X_it_aug.columns]
            tda_cols_used = [c for c in cfg_cols if c in X_tda_full.columns]

            if it_cols_used or tda_cols_used:
                Xi_cfg = pd.concat(
                    [X_it_aug[it_cols_used]] +
                    ([X_tda_full[tda_cols_used]] if tda_cols_used else []),
                    axis=1
                )
            else:
                Xi_cfg = X_it_aug[IT_SLIM]

            idx = X_vol.index.intersection(Xi_cfg.index).intersection(y_raw.index)
            valid = (X_vol.loc[idx].notna().all(1) &
                     Xi_cfg.loc[idx].notna().all(1) &
                     y_raw.loc[idx].notna())
            idx = idx[valid]

            if len(idx) < 1000:
                continue

            tr = idx < (split_ts - pd.Timedelta(minutes=30 * PURGE_BARS))
            te = idx >= split_ts

            if tr.sum() < 480 or te.sum() < 100:
                continue

            Xv_tr = X_vol.loc[idx[tr]]
            Xv_te = X_vol.loc[idx[te]]
            Xi_tr = Xi_cfg.loc[idx[tr]]
            Xi_te = Xi_cfg.loc[idx[te]]
            ytr = y_raw.loc[idx[tr]]
            yte = y_raw.loc[idx[te]]
            naive = float(ytr.mean())

            ck = Xv_tr.corrwith(ytr).abs().dropna()
            top3 = ck.nlargest(3).index.tolist()

            mb = _ridge()
            mb.fit(Xv_tr[top3], ytr)
            pb_tr = mb.predict(Xv_tr[top3])
            pb_te = mb.predict(Xv_te[top3])

            res_tr = ytr.values - pb_tr

            mc = _ridge()
            mc.fit(Xi_tr, res_tr)
            pc_te = mc.predict(Xi_te)
            pred = pb_te + pc_te

            r2_b = oos_r2(yte.values, pb_te, naive)
            r2_c = oos_r2(yte.values, pred, naive)
            rc = float(pd.Series(pc_te).corr(pd.Series(yte.values - pb_te)))

            config_rows.append({
                "asset": asset,
                "horizon": h_label,
                "h": h,
                "config": cfg_name,
                "r2_base": round(r2_b, 4),
                "r2_cfg": round(r2_c, 4),
                "delta_r2": round(r2_c - r2_b, 4),
                "res_corr": round(rc, 4),
            })

cfg_df = pd.DataFrame(config_rows)

print(f"\n  Mean ΔR² (logvol) by configuration:")
print(f"  {'config':<15} {'mean_ΔR²':>10} {'ΔR²>0':>8} {'mean_rc':>9}")
print("  " + "-" * 45)

for cfg in CONFIGS.keys():
    sub = cfg_df[cfg_df.config == cfg]
    if sub.empty:
        continue
    pos = (sub["delta_r2"] > 0).sum()
    print(f"  {cfg:<15} {sub['delta_r2'].mean():>+10.4f} "
          f"{pos:>5}/{len(sub)} {sub['res_corr'].mean():>+9.4f}")

best_cfg = cfg_df.groupby("config")["delta_r2"].mean().idxmax()
print(f"\n  Best configuration: '{best_cfg}'")

sub_best = cfg_df[cfg_df.config == best_cfg]
print(f"  {'asset':<6} {'h':<4} {'R²_base':>9} {'R²_best':>9} {'ΔR²':>8}")
print("  " + "-" * 40)

prev = None
for _, row in sub_best.sort_values(["asset", "h"]).iterrows():
    if prev and prev != row["asset"]:
        print("  " + "·" * 40)
    prev = row["asset"]
    mark = "↑" if row["delta_r2"] > 0.005 else ("↓" if row["delta_r2"] < -0.005 else "~")
    print(f"  {row['asset']:<6} {row['horizon']:<4}"
          f" {row['r2_base']:>+9.4f} {row['r2_cfg']:>+9.4f}"
          f" {row['delta_r2']:>+8.4f} {mark}")

print("\nBuilding plots...")

fig, axes = plt.subplots(1, 3, figsize=(18, 7))

ax = axes[0]
feats_sorted = [f for f, _ in sorted_perm]
vals_sorted = [avg_perm[f] for f in feats_sorted]
colors_bar = [cat_colors.get(categorize(f), "#9E9E9E") for f in feats_sorted]
y_pos = np.arange(len(feats_sorted))

ax.barh(y_pos, vals_sorted, color=colors_bar, alpha=0.85, height=0.8)
ax.axvline(0, color="black", lw=0.8)
ax.axvline(-0.002, color="green", lw=0.8, ls=":", alpha=0.6)
ax.axvline(+0.002, color="red", lw=0.8, ls=":", alpha=0.6)
ax.set_yticks(y_pos)
ax.set_yticklabels(feats_sorted, fontsize=7)
ax.set_xlabel("Δ CV R² under permutation\n(< 0 = feature is useful)")
ax.set_title("Permutation importance\n(BTC+ETH, h=4d, logvol)", fontsize=10)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

from matplotlib.patches import Patch

legend_elems = [
    Patch(facecolor=cat_colors["IT"], label="IT features", alpha=0.85),
    Patch(facecolor=cat_colors["TDA"], label="TDA features", alpha=0.85),
]

ax.legend(handles=legend_elems, fontsize=8, loc="lower right", frameon=True)

ax = axes[1]
cfg_order = list(CONFIGS.keys())
asset_order = ASSETS
x = np.arange(len(asset_order))
w = 0.18
cfg_colors = ["#90CAF9", "#2196F3", "#4CAF50", "#FF9800"]

for ci, cfg in enumerate(cfg_order):
    sub = cfg_df[cfg_df.config == cfg]
    vals = []
    for asset in asset_order:
        v = sub[sub.asset == asset]["delta_r2"].mean()
        vals.append(v if not np.isnan(v) else 0.0)

    ax.bar(
        x + (ci - 1.5) * w,
        vals,
        w * 0.9,
        label=cfg,
        color=cfg_colors[ci],
        alpha=0.85
    )

ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels(asset_order)
ax.set_ylabel("Mean ΔR² (all horizons)")
ax.set_title("ΔR² by configuration × asset\n(logvol, holdout 2025)", fontsize=10)
ax.legend(fontsize=8)
ax.grid(alpha=0.2, axis="y")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax = axes[2]
cat_corrs = {}
for feat, c in {**it_corrs, **tda_corrs}.items():
    cat = categorize(feat)
    cat_corrs.setdefault(cat, []).append(c)

cats_to_plot = [
    "IT_slim", "IT_base_other", "IT_extra",
    "TDA_A", "TDA_BI", "TDA_BII", "TDA_BC", "TDA_extra"
]
data_plot = [cat_corrs.get(c, [0]) for c in cats_to_plot]

bp = ax.boxplot(
    data_plot,
    labels=cats_to_plot,
    patch_artist=True,
    medianprops=dict(color="black", lw=1.5),
    flierprops=dict(marker=".", markersize=4, alpha=0.5)
)

for patch, cat in zip(bp["boxes"], cats_to_plot):
    patch.set_facecolor(cat_colors.get(cat, "#9E9E9E"))
    patch.set_alpha(0.75)

ax.axhline(0, color="black", lw=0.5)
ax.set_ylabel("corr(feature, vol_BTC) on test period")
ax.set_title("Feature correlation with vol_BTC\nby category", fontsize=10)
ax.set_xticklabels(cats_to_plot, rotation=20, ha="right", fontsize=8)
ax.grid(alpha=0.2, axis="y")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.suptitle("Feature utility analysis: IT + TDA with alternative MI estimators",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("fig_feature_analysis_extended.png", bbox_inches="tight", dpi=150)
plt.savefig("fig_feature_analysis_extended.pdf", bbox_inches="tight", dpi=150)
plt.show()
print("  Saved: fig_feature_analysis_extended.png / .pdf")

print(f"\n{'=' * 65}")
print("  FINAL: best feature set")
print(f"{'=' * 65}")

print(f"\n  Configuration '{best_cfg}' ({len(CONFIGS[best_cfg])} features):")
for f in CONFIGS[best_cfg]:
    cat = categorize(f)
    imp = avg_perm.get(f, 0.0)
    print(f"    {f:<36} [{cat}]  perm={imp:>+.4f}")

print(f"\n  Comparison of all configurations:")
for cfg in cfg_order:
    sub = cfg_df[cfg_df.config == cfg]
    print(f"    {cfg:<15}: ΔR²={sub['delta_r2'].mean():>+.4f}  "
          f"rc={sub['res_corr'].mean():>+.4f}  "
          f"↑{(sub['delta_r2'] > 0).sum()}/{len(sub)}")

feat_analysis = {
    "perm": avg_perm,
    "cfg_df": cfg_df,
    "it_corrs": it_corrs,
    "tda_corrs": tda_corrs,
    "best_cfg": best_cfg,
    "best_cols": CONFIGS[best_cfg],
    "X_it_aug": X_it_aug,
    "X_tda_full": X_tda_full,
    "extra_it_cols": extra_it_cols,
    "extra_tda_cols": extra_tda_cols,
}
print(f"\nfeat_analysis saved in memory")


  2. PERMUTATION IMPORTANCE ON TRAIN (BTC+ETH, h=4d, logvol)
  BTC baseline CV R² = -1.0890
  ETH baseline CV R² = -1.4195

  3. HOLDOUT CONFIGURATION COMPARISON
  slim_IT: 10 features
  best_IT: 10 features
  slim+TDA: 16 features
  best+TDA: 16 features

  Mean ΔR² (logvol) by configuration:
  config            mean_ΔR²    ΔR²>0   mean_rc
  ---------------------------------------------
  slim_IT            -0.0210     4/12   +0.1162
  best_IT            -0.0385     6/12   +0.0333
  slim+TDA           -0.0036     5/12   +0.0870
  best+TDA           -0.0251     5/12   +0.0660

  Best configuration: 'slim+TDA'
  asset  h      R²_base   R²_best      ΔR²
  ----------------------------------------
  BNB    2d     +0.3104   +0.2663  -0.0441 ↓
  BNB    4d     +0.2152   +0.1233  -0.0919 ↓
  ········································
  BTC    2d     +0.1300   +0.0053  -0.1247 ↓
  BTC    4d     -0.1246   -0.2650  -0.1404 ↓
  ········································
  DOGE   2d     +0.2001   +0.3

# New win

In [164]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy import stats as sp_stats

from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

SPLIT_DATE = "2025-07-01"
PURGE_BARS = 96

HORIZONS = {
    48:  "1d",
    96:  "2d",
    192: "4d",
    288: "6d",
    336: "7d",
    672: "14d",
}

ASSETS = [a for a in ["BTC","ETH","SOL","BNB","XRP","DOGE","ADA","AVAX"] if a in ret.columns]

VOL_WINDOW = 48
ANN = np.sqrt(365 * 48)

PERM_USEFUL_THRESHOLD = -0.002

TOP_IT = 10
TOP_TDA = 6

def _ridge():
    return Pipeline([
        ("sc", StandardScaler()),
        ("m", RidgeCV(alphas=[0.01, 0.1, 1, 10, 100]))
    ])

def oos_r2(y_true, y_pred, naive_val):
    mse_m = np.mean((y_true - y_pred) ** 2)
    mse_n = np.mean((y_true - naive_val) ** 2)
    return float(1.0 - mse_m / mse_n) if mse_n > 1e-12 else np.nan

def res_corr(residuals_true, residuals_pred):
    return float(pd.Series(residuals_pred).corr(pd.Series(residuals_true)))

def hit_rate_vs_baseline(y_true, pred_base, pred_model):
    e_base = (y_true - pred_base) ** 2
    e_mod  = (y_true - pred_model) ** 2
    return float(np.mean(e_mod < e_base))

def dm_test(loss_diff):
    n = len(loss_diff)
    m = loss_diff.mean()
    s = loss_diff.std(ddof=1)
    t = m / (s / np.sqrt(n)) if s > 1e-12 else 0.0
    p = float(sp_stats.t.sf(abs(t), df=n - 1) * 2) if n > 1 else np.nan
    return float(t), float(p)

def sig(p):
    if not np.isfinite(p):
        return ""
    if p < 0.01:
        return "***"
    if p < 0.05:
        return "**"
    if p < 0.10:
        return "*"
    return ""

if "feat_analysis" in dir():
    avg_perm = feat_analysis["perm"]

    if "X_it_aug" in feat_analysis:
        X_it_all = feat_analysis["X_it_aug"].copy()
    else:
        X_it_all = X_it_slim.copy()

    if "X_tda_full" in feat_analysis:
        X_tda_all = feat_analysis["X_tda_full"].copy()
    else:
        raise ValueError("feat_analysis exists, but X_tda_full is missing inside it.")
else:
    if "avg_perm" not in dir():
        raise ValueError("avg_perm / feat_analysis['perm'] not found in memory.")
    if "X_it_aug" not in dir():
        raise ValueError("X_it_aug not found in memory.")
    if "X_tda_full" not in dir():
        raise ValueError("X_tda_full not found in memory.")

    X_it_all = X_it_aug.copy()
    X_tda_all = X_tda_full.copy()

perm_series = pd.Series(avg_perm).sort_values()

print("=" * 80)
print("NEW HOLDOUT EXPERIMENT WITH THE BEST FEATURES FROM PERMUTATION IMPORTANCE")
print("=" * 80)

it_candidates = [
    f for f in perm_series.index
    if (f in X_it_all.columns) and (perm_series[f] < PERM_USEFUL_THRESHOLD)
]

tda_candidates = [
    f for f in perm_series.index
    if (f in X_tda_all.columns) and (perm_series[f] < PERM_USEFUL_THRESHOLD)
]

BEST_IT = it_candidates[:TOP_IT]
BEST_TDA = tda_candidates[:TOP_TDA]

print("\nSelected IT features:")
for f in BEST_IT:
    print(f"  {f:<36} perm={perm_series[f]:+.4f}")

print("\nSelected TDA features:")
for f in BEST_TDA:
    print(f"  {f:<36} perm={perm_series[f]:+.4f}")

if len(BEST_IT) == 0:
    raise ValueError("No useful IT features survived the threshold.")
if len(BEST_TDA) == 0:
    raise ValueError("No useful TDA features survived the threshold.")

print(f"\nIT count:  {len(BEST_IT)}")
print(f"TDA count: {len(BEST_TDA)}")

split_ts = pd.Timestamp(SPLIT_DATE)
rows = []

for asset in ASSETS:
    r = ret[asset]

    for h, h_label in HORIZONS.items():
        vol_fwd = r.rolling(h).std().shift(-h) * ANN
        y_raw = np.log(vol_fwd.clip(lower=1e-8)).dropna()

        Xi_it = X_it_all[BEST_IT].copy()
        Xi_tda = X_tda_all[BEST_TDA].copy()

        idx = (
            X_vol.index
            .intersection(Xi_it.index)
            .intersection(Xi_tda.index)
            .intersection(y_raw.index)
        )

        valid = (
            X_vol.loc[idx].notna().all(1)
            & Xi_it.loc[idx].notna().all(1)
            & Xi_tda.loc[idx].notna().all(1)
            & y_raw.loc[idx].notna()
        )
        idx = idx[valid]

        if len(idx) < 1000:
            continue

        tr = idx < (split_ts - pd.Timedelta(minutes=30 * PURGE_BARS))
        te = idx >= split_ts

        if tr.sum() < 480 or te.sum() < 100:
            continue

        Xv_tr = X_vol.loc[idx[tr]]
        Xv_te = X_vol.loc[idx[te]]

        Xit_tr = Xi_it.loc[idx[tr]]
        Xit_te = Xi_it.loc[idx[te]]

        Xtd_tr = Xi_tda.loc[idx[tr]]
        Xtd_te = Xi_tda.loc[idx[te]]

        ytr = y_raw.loc[idx[tr]]
        yte = y_raw.loc[idx[te]]

        naive = float(ytr.mean())

        ck = Xv_tr.corrwith(ytr).abs().replace([np.inf, -np.inf], np.nan).dropna()
        top3 = ck.nlargest(3).index.tolist()

        mb = _ridge()
        mb.fit(Xv_tr[top3], ytr)

        pb_tr = mb.predict(Xv_tr[top3])
        pb_te = mb.predict(Xv_te[top3])

        res_tr = ytr.values - pb_tr
        res_te = yte.values - pb_te

        mit = _ridge()
        mit.fit(Xit_tr, res_tr)
        pit_te = mit.predict(Xit_te)
        pred_it = pb_te + pit_te

        mtda = _ridge()
        mtda.fit(Xtd_tr, res_tr)
        ptda_te = mtda.predict(Xtd_te)
        pred_tda = pb_te + ptda_te

        Xc_tr = pd.concat([Xit_tr, Xtd_tr], axis=1)
        Xc_te = pd.concat([Xit_te, Xtd_te], axis=1)

        mct = _ridge()
        mct.fit(Xc_tr, res_tr)
        pct_te = mct.predict(Xc_te)
        pred_all = pb_te + pct_te

        r2_A = oos_r2(yte.values, pb_te, naive)
        r2_B = oos_r2(yte.values, pred_it, naive)
        r2_C = oos_r2(yte.values, pred_tda, naive)
        r2_D = oos_r2(yte.values, pred_all, naive)

        rc_B = res_corr(res_te, pit_te)
        rc_C = res_corr(res_te, ptda_te)
        rc_D = res_corr(res_te, pct_te)

        mse_A = np.mean((yte.values - pb_te) ** 2)
        mse_B = np.mean((yte.values - pred_it) ** 2)
        mse_C = np.mean((yte.values - pred_tda) ** 2)
        mse_D = np.mean((yte.values - pred_all) ** 2)

        hr_B = hit_rate_vs_baseline(yte.values, pb_te, pred_it)
        hr_C = hit_rate_vs_baseline(yte.values, pb_te, pred_tda)
        hr_D = hit_rate_vs_baseline(yte.values, pb_te, pred_all)

        dB = (yte.values - pb_te) ** 2 - (yte.values - pred_it) ** 2
        dC = (yte.values - pb_te) ** 2 - (yte.values - pred_tda) ** 2
        dD = (yte.values - pb_te) ** 2 - (yte.values - pred_all) ** 2

        _, p_B = dm_test(dB)
        _, p_C = dm_test(dC)
        _, p_D = dm_test(dD)

        rows.append({
            "asset": asset,
            "horizon": h_label,
            "h": h,

            "r2_A": round(r2_A, 4),
            "r2_B": round(r2_B, 4),
            "r2_C": round(r2_C, 4),
            "r2_D": round(r2_D, 4),

            "dR2_B": round(r2_B - r2_A, 4),
            "dR2_C": round(r2_C - r2_A, 4),
            "dR2_D": round(r2_D - r2_A, 4),

            "rc_B": round(rc_B, 4),
            "rc_C": round(rc_C, 4),
            "rc_D": round(rc_D, 4),

            "mse_ratio_B": round(mse_B / mse_A, 4) if mse_A > 0 else np.nan,
            "mse_ratio_C": round(mse_C / mse_A, 4) if mse_A > 0 else np.nan,
            "mse_ratio_D": round(mse_D / mse_A, 4) if mse_A > 0 else np.nan,

            "hit_B": round(hr_B, 4),
            "hit_C": round(hr_C, 4),
            "hit_D": round(hr_D, 4),

            "p_B": round(p_B, 4),
            "p_C": round(p_C, 4),
            "p_D": round(p_D, 4),

            "sig_B": sig(p_B),
            "sig_C": sig(p_C),
            "sig_D": sig(p_D),

            "n_train": int(tr.sum()),
            "n_test": int(te.sum()),
        })

df_best = pd.DataFrame(rows)
print(f"\nRows collected: {len(df_best)}")

if df_best.empty:
    raise ValueError("The result dataframe is empty. Check intersections, NaNs, and sample sizes.")

print("\n" + "=" * 100)
print("FINAL COMPARISON WITH AUTOMATICALLY SELECTED BEST FEATURES")
print("Target: logvol = log(realized_vol_fwd_h)")
print("A = Baseline(vol) | B = +best IT | C = +best TDA | D = +best IT + best TDA")
print("=" * 100)

print(
    f"\n  {'asset':<6} {'h':<4}"
    f" {'R²_A':>9} {'R²_B':>9} {'R²_C':>9} {'R²_D':>9}"
    f" {'ΔB':>8} {'ΔC':>8} {'ΔD':>8}  best"
)
print("  " + "-" * 90)

prev = None
for _, row in df_best.sort_values(["asset", "h"]).iterrows():
    if prev and prev != row["asset"]:
        print("  " + "·" * 90)
    prev = row["asset"]

    vals = {"A": row["r2_A"], "B": row["r2_B"], "C": row["r2_C"], "D": row["r2_D"]}
    best = max(vals, key=vals.get)

    mb = "↑" if row["dR2_B"] > 0.005 else ("↓" if row["dR2_B"] < -0.005 else "~")
    mc = "↑" if row["dR2_C"] > 0.005 else ("↓" if row["dR2_C"] < -0.005 else "~")
    md = "↑" if row["dR2_D"] > 0.005 else ("↓" if row["dR2_D"] < -0.005 else "~")

    print(
        f"  {row['asset']:<6} {row['horizon']:<4}"
        f" {row['r2_A']:>+9.4f}"
        f" {row['r2_B']:>+9.4f}"
        f" {row['r2_C']:>+9.4f}"
        f" {row['r2_D']:>+9.4f}"
        f" {row['dR2_B']:>+7.4f}{mb}"
        f" {row['dR2_C']:>+7.4f}{mc}"
        f" {row['dR2_D']:>+7.4f}{md}"
        f"  {best}"
    )

print("\n" + "=" * 75)
print("SUMMARY")
print("=" * 75)

n = len(df_best)
print(f"""
  Config         mean ΔR²   ΔR²>0    mean rc   MSE<1    hit rate
  ----------------------------------------------------------------
  B (+best IT)   {df_best['dR2_B'].mean():>+8.4f}   {(df_best['dR2_B']>0).sum():>2}/{n}   {df_best['rc_B'].mean():>+7.4f}   {(df_best['mse_ratio_B']<1).sum():>2}/{n}   {df_best['hit_B'].mean():.3f}
  C (+best TDA)  {df_best['dR2_C'].mean():>+8.4f}   {(df_best['dR2_C']>0).sum():>2}/{n}   {df_best['rc_C'].mean():>+7.4f}   {(df_best['mse_ratio_C']<1).sum():>2}/{n}   {df_best['hit_C'].mean():.3f}
  D (+both)      {df_best['dR2_D'].mean():>+8.4f}   {(df_best['dR2_D']>0).sum():>2}/{n}   {df_best['rc_D'].mean():>+7.4f}   {(df_best['mse_ratio_D']<1).sum():>2}/{n}   {df_best['hit_D'].mean():.3f}
""")

print("Top-5 tasks by ΔR² for configuration D:")
for _, row in df_best.nlargest(5, "dR2_D").iterrows():
    print(
        f"  {row['asset']} h={row['horizon']}: "
        f"R²_A={row['r2_A']:+.4f} -> "
        f"R²_B={row['r2_B']:+.4f} -> "
        f"R²_C={row['r2_C']:+.4f} -> "
        f"R²_D={row['r2_D']:+.4f} "
        f"(ΔD={row['dR2_D']:+.4f}, p={row['p_D']:.4f}{row['sig_D']})"
    )

asset_summary = (
    df_best.groupby("asset", as_index=False)
    .agg(
        r2_A=("r2_A", "mean"),
        r2_B=("r2_B", "mean"),
        r2_C=("r2_C", "mean"),
        r2_D=("r2_D", "mean"),

        dR2_B=("dR2_B", "mean"),
        dR2_C=("dR2_C", "mean"),
        dR2_D=("dR2_D", "mean"),

        rc_B=("rc_B", "mean"),
        rc_C=("rc_C", "mean"),
        rc_D=("rc_D", "mean"),

        mse_ratio_B=("mse_ratio_B", "mean"),
        mse_ratio_C=("mse_ratio_C", "mean"),
        mse_ratio_D=("mse_ratio_D", "mean"),

        hit_B=("hit_B", "mean"),
        hit_C=("hit_C", "mean"),
        hit_D=("hit_D", "mean"),
    )
)

def choose_best_config(row):
    vals = {
        "A": row["r2_A"],
        "B": row["r2_B"],
        "C": row["r2_C"],
        "D": row["r2_D"],
    }
    return max(vals, key=vals.get)

def choose_best_addon(row):
    vals = {
        "B": row["dR2_B"],
        "C": row["dR2_C"],
        "D": row["dR2_D"],
    }
    return max(vals, key=vals.get)

asset_summary["best_config"] = asset_summary.apply(choose_best_config, axis=1)
asset_summary["best_addon"] = asset_summary.apply(choose_best_addon, axis=1)

asset_summary = asset_summary.sort_values(
    ["r2_D", "r2_C", "r2_B", "r2_A"],
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 90)
print("ASSET-LEVEL AGGREGATION (MEAN OVER HORIZONS)")
print("=" * 90)

print(
    f"\n  {'asset':<6}"
    f" {'R²_A':>10} {'R²_B':>10} {'R²_C':>10} {'R²_D':>10}"
    f" {'ΔB':>9} {'ΔC':>9} {'ΔD':>9}"
    f" {'best_cfg':>10} {'best_addon':>12}"
)

print("  " + "-" * 100)

for _, row in asset_summary.iterrows():
    print(
        f"  {row['asset']:<6}"
        f" {row['r2_A']:>+10.4f}"
        f" {row['r2_B']:>+10.4f}"
        f" {row['r2_C']:>+10.4f}"
        f" {row['r2_D']:>+10.4f}"
        f" {row['dR2_B']:>+9.4f}"
        f" {row['dR2_C']:>+9.4f}"
        f" {row['dR2_D']:>+9.4f}"
        f" {row['best_config']:>10}"
        f" {row['best_addon']:>12}"
    )

print("\nBest configuration counts across assets:")
print(asset_summary["best_config"].value_counts().sort_index())

print("\nBest incremental addon counts across assets:")
print(asset_summary["best_addon"].value_counts().sort_index())

print("\nBuilding final plots...")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

asset_order = (
    df_best.groupby("asset")["r2_D"]
    .mean()
    .sort_values(ascending=False)
    .index
    .tolist()
)

h_order = ["1d", "2d", "4d", "6d", "7d", "14d"]
x = np.arange(len(asset_order))
w = 0.2

ax = axes[0, 0]
for ci, (col, label, color) in enumerate([
    ("r2_A", "A: Baseline",   "#9E9E9E"),
    ("r2_B", "B: +best IT",   "#2196F3"),
    ("r2_C", "C: +best TDA",  "#FF9800"),
    ("r2_D", "D: +best both", "#4CAF50"),
]):
    vals = [df_best[df_best.asset == a][col].mean() for a in asset_order]
    ax.bar(x + (ci - 1.5) * w, vals, w * 0.9, label=label, color=color, alpha=0.85)

ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels(asset_order, fontsize=9)
ax.set_ylabel("OOS R² (vs naive mean logvol)")
ax.set_title("OOS R² by asset — 4 configurations\n(logvol, all horizons)", fontsize=10)
ax.legend(fontsize=8)
ax.grid(alpha=0.2, axis="y")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax = axes[0, 1]
for col, label, color, ls in [
    ("r2_A", "A: Baseline",   "#9E9E9E", "-"),
    ("r2_B", "B: +best IT",   "#2196F3", "-"),
    ("r2_C", "C: +best TDA",  "#FF9800", "--"),
    ("r2_D", "D: +best both", "#4CAF50", "-"),
]:
    vals = [df_best[df_best.horizon == h][col].mean() for h in h_order]
    ax.plot(h_order, vals, "o" + ls, color=color, lw=2, label=label, ms=7)

ax.axhline(0, color="black", lw=0.5, ls=":")
ax.set_ylabel("OOS R²")
ax.set_xlabel("Forecast horizon")
ax.set_title("OOS R² by horizon\n(mean over assets)", fontsize=10)
ax.legend(fontsize=8)
ax.grid(alpha=0.2)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax = axes[1, 0]
x2 = np.arange(len(asset_order))
w2 = 0.28

for ci, (col, label, color) in enumerate([
    ("dR2_B", "B: +best IT",   "#2196F3"),
    ("dR2_C", "C: +best TDA",  "#FF9800"),
    ("dR2_D", "D: +best both", "#4CAF50"),
]):
    vals = [df_best[df_best.asset == a][col].mean() for a in asset_order]
    ax.bar(x2 + (ci - 1) * w2, vals, w2 * 0.9, label=label, color=color, alpha=0.85)

ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(x2)
ax.set_xticklabels(asset_order, fontsize=9)
ax.set_ylabel("ΔR² relative to baseline A")
ax.set_title("Incremental ΔR² over baseline\n(logvol, all horizons)", fontsize=10)
ax.legend(fontsize=8)
ax.grid(alpha=0.2, axis="y")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax = axes[1, 1]
pivot = df_best.pivot_table(
    index="asset",
    columns="horizon",
    values="mse_ratio_D",
    aggfunc="mean"
)
pivot = pivot.reindex(asset_order)[h_order]

im = ax.imshow(pivot.values, cmap="RdYlGn_r", vmin=0.85, vmax=1.15, aspect="auto")
ax.set_xticks(range(len(h_order)))
ax.set_xticklabels(h_order)
ax.set_yticks(range(len(asset_order)))
ax.set_yticklabels(asset_order)

for i in range(len(asset_order)):
    for j in range(len(h_order)):
        val = pivot.values[i, j]
        if np.isfinite(val):
            col = "white" if abs(val - 1) > 0.07 else "black"
            mark = "✓" if val < 1.0 else ""
            ax.text(j, i, f"{val:.3f}{mark}", ha="center", va="center", fontsize=8, color=col)

plt.colorbar(im, ax=ax, label="MSE ratio (D / baseline)\n<1.0 = better ✓")
ax.set_title("MSE ratio for configuration D\n(green < 1 = better than baseline)", fontsize=10)

plt.suptitle(
    "Final comparison with automatically selected best features\n"
    f"Target: logvol = log(realized_vol_fwd) | holdout {SPLIT_DATE} ->",
    fontsize=12,
    y=1.01
)
plt.tight_layout()
plt.savefig("fig_final_comparison_best_features.png", bbox_inches="tight", dpi=150)
plt.savefig("fig_final_comparison_best_features.pdf", bbox_inches="tight", dpi=150)
plt.show()

print("Saved: fig_final_comparison_best_features.png / .pdf")

final_comparison_best = df_best.copy()
best_feature_sets = {
    "BEST_IT": BEST_IT,
    "BEST_TDA": BEST_TDA,
}
print(f"\nfinal_comparison_best saved in memory ({len(df_best)} rows)")
print("best_feature_sets saved in memory")

NEW HOLDOUT EXPERIMENT WITH THE BEST FEATURES FROM PERMUTATION IMPORTANCE

Selected IT features:
  kl_BTC                               perm=-0.0811
  coinfo4_std                          perm=-0.0297
  kl_ADA                               perm=-0.0293
  nmi_lag1_mean                        perm=-0.0283
  coinfo3_std                          perm=-0.0216
  pairwise_mi_mean                     perm=-0.0064

Selected TDA features:
  BI_L1_H0                             perm=-0.1617
  BC_ent_H1                            perm=-0.0687
  BII_tp_H1                            perm=-0.0671
  BII_L1_H1                            perm=-0.0454
  A_T_H0                               perm=-0.0196
  A_E_H0                               perm=-0.0115

IT count:  6
TDA count: 6

Rows collected: 48

FINAL COMPARISON WITH AUTOMATICALLY SELECTED BEST FEATURES
Target: logvol = log(realized_vol_fwd_h)
A = Baseline(vol) | B = +best IT | C = +best TDA | D = +best IT + best TDA

  asset  h         R²_A      R²

In [178]:
!pip install arch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 990.7/990.7 kB 107.1 MB/s  0:00:00

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [184]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy import stats as sp_stats

from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

try:
    from arch import arch_model
    HAS_ARCH = True
except ImportError:
    print("arch package not found. Install with: pip install arch")
    HAS_ARCH = False

SPLIT_DATE = "2025-07-01"
PURGE_BARS = 96

HORIZONS = {
    48:  "1d",
    96:  "2d",
    192: "4d",
    288: "6d",
    336: "7d",
    672: "14d",
}

ASSETS = [a for a in ["BTC","ETH","SOL","BNB","XRP","DOGE","ADA","AVAX"] if a in ret.columns]

VOL_WINDOW = 48
ANN = np.sqrt(365 * 48)

PERM_USEFUL_THRESHOLD = -0.002
TOP_IT  = 10
TOP_TDA = 6

GARCH_P = 1
GARCH_Q = 1

def _ridge():
    return Pipeline([
        ("sc", StandardScaler()),
        ("m", RidgeCV(alphas=[0.01, 0.1, 1, 10, 100]))
    ])

def oos_r2(y_true, y_pred, naive_val):
    mse_m = np.mean((y_true - y_pred) ** 2)
    mse_n = np.mean((y_true - naive_val) ** 2)
    return float(1.0 - mse_m / mse_n) if mse_n > 1e-12 else np.nan

def res_corr(residuals_true, residuals_pred):
    return float(pd.Series(residuals_pred).corr(pd.Series(residuals_true)))

def hit_rate_vs_baseline(y_true, pred_base, pred_model):
    e_base = (y_true - pred_base) ** 2
    e_mod  = (y_true - pred_model) ** 2
    return float(np.mean(e_mod < e_base))

def dm_test(loss_diff):
    n = len(loss_diff)
    m = loss_diff.mean()
    s = loss_diff.std(ddof=1)
    t = m / (s / np.sqrt(n)) if s > 1e-12 else 0.0
    p = float(sp_stats.t.sf(abs(t), df=n - 1) * 2) if n > 1 else np.nan
    return float(t), float(p)

def sig(p):
    if not np.isfinite(p): return ""
    if p < 0.01: return "***"
    if p < 0.05: return "**"
    if p < 0.10: return "*"
    return ""

def build_har_features(ret_series):
    rv_d = ret_series.rolling(48).std()   * ANN
    rv_w = ret_series.rolling(240).std()  * ANN
    rv_m = ret_series.rolling(1056).std() * ANN

    return pd.DataFrame({
        "har_d": np.log(rv_d.clip(lower=1e-8)),
        "har_w": np.log(rv_w.clip(lower=1e-8)),
        "har_m": np.log(rv_m.clip(lower=1e-8)),
    })

def fit_garch_forecast(r_train, r_test, h, ann_factor):
    """
    Обучает GARCH(1,1) на r_train, предсказывает h-шаговую
    условную волатильность для каждой точки r_test.

    Возвращает log(annualized_vol_forecast) для каждой точки теста.
    Если GARCH не сходится — возвращает NaN.
    """
    if not HAS_ARCH:
        return np.full(len(r_test), np.nan)

    try:
        scale = 100.0
        r_scaled = r_train.values * scale

        am = arch_model(r_scaled, vol="Garch", p=GARCH_P, q=GARCH_Q,
                        mean="Zero", dist="normal")
        res = am.fit(disp="off", show_warning=False)

        fc = res.forecast(horizon=h, reindex=False)
        sigma_h = np.sqrt(fc.variance.iloc[-1].mean()) / scale * ann_factor

        log_sigma = float(np.log(max(sigma_h, 1e-8)))
        return np.full(len(r_test), log_sigma)

    except Exception:
        return np.full(len(r_test), np.nan)

def build_garch_forecasts(ret_series, split_ts, h, ann_factor,
                           purge_bars=PURGE_BARS):
    """
    Строит GARCH прогнозы для test-периода.
    Train = всё до split_ts - purge_bars.
    Возвращает Series с log(vol_forecast) на test-периоде.
    """
    cutoff = split_ts - pd.Timedelta(minutes=30 * purge_bars)
    r_train = ret_series[ret_series.index < cutoff].dropna()
    r_test  = ret_series[ret_series.index >= split_ts].dropna()

    if len(r_train) < 200 or len(r_test) == 0:
        return pd.Series(np.nan, index=r_test.index)

    preds = fit_garch_forecast(r_train, r_test, h, ann_factor)
    return pd.Series(preds, index=r_test.index)

print("Building GARCH(1,1) forecasts for all assets and horizons...")
split_ts_global = pd.Timestamp(SPLIT_DATE)

garch_forecasts = {}   # {(asset, h): Series on test index}
for asset in ASSETS:
    for h in HORIZONS:
        key = (asset, h)
        garch_forecasts[key] = build_garch_forecasts(
            ret[asset], split_ts_global, h, ANN
        )
        n_valid = garch_forecasts[key].notna().sum()
        print(f"  {asset} h={HORIZONS[h]}: {n_valid} test points")

print("  GARCH forecasts done.")

if "window_it_features" in dir() and 96 in window_it_features:
    X_it_all, BEST_IT_96 = window_it_features[96]
    X_it_all = X_it_all.copy()
    print("Loaded IT features with W=96 from window_it_features[96]")
else:
    raise ValueError(
        "window_it_features[96] not found. "
        "Run window sensitivity script first."
    )

if "feat_analysis" in dir():
    avg_perm  = feat_analysis["perm"]
    if "X_tda_full" not in feat_analysis:
        raise ValueError("X_tda_full missing in feat_analysis.")
    X_tda_all = feat_analysis["X_tda_full"].copy()
else:
    if "avg_perm" not in dir():
        raise ValueError("avg_perm not found in memory.")
    if "X_tda_full" not in dir():
        raise ValueError("X_tda_full not found in memory.")
    X_tda_all = X_tda_full.copy()

perm_series = pd.Series(avg_perm).sort_values()

print("\n" + "=" * 80)
print("HOLDOUT EXPERIMENT (W=96): HAR-RV / vol-baseline / +IT / +TDA / +both")
print("=" * 80)

BEST_IT = [f for f in BEST_IT_96 if f in X_it_all.columns]

tda_candidates = [
    f for f in perm_series.index
    if (f in X_tda_all.columns) and (perm_series[f] < PERM_USEFUL_THRESHOLD)
]
BEST_TDA = tda_candidates[:TOP_TDA]

print("\nSelected IT features (W=96):")
for f in BEST_IT:
    print(f"  {f:<36} perm={perm_series.get(f, float('nan')):+.4f}")

print("\nSelected TDA features:")
for f in BEST_TDA:
    print(f"  {f:<36} perm={perm_series[f]:+.4f}")

if len(BEST_IT) == 0:
    raise ValueError("No IT features found for W=96.")
if len(BEST_TDA) == 0:
    raise ValueError("No useful TDA features survived the threshold.")

print(f"\nIT count:  {len(BEST_IT)}")
print(f"TDA count: {len(BEST_TDA)}")

split_ts = pd.Timestamp(SPLIT_DATE)
rows = []

for asset in ASSETS:
    r = ret[asset]

    X_har = build_har_features(r)

    for h, h_label in HORIZONS.items():
        vol_fwd = r.rolling(h).std().shift(-h) * ANN
        y_raw   = np.log(vol_fwd.clip(lower=1e-8)).dropna()

        Xi_it  = X_it_all[BEST_IT].copy()
        Xi_tda = X_tda_all[BEST_TDA].copy()

        idx = (
            X_vol.index
            .intersection(Xi_it.index)
            .intersection(Xi_tda.index)
            .intersection(X_har.index)
            .intersection(y_raw.index)
        )

        valid = (
            X_vol.loc[idx].notna().all(1)
            & Xi_it.loc[idx].notna().all(1)
            & Xi_tda.loc[idx].notna().all(1)
            & X_har.loc[idx].notna().all(1)
            & y_raw.loc[idx].notna()
        )
        idx = idx[valid]

        if len(idx) < 1000:
            continue

        tr = idx < (split_ts - pd.Timedelta(minutes=30 * PURGE_BARS))
        te = idx >= split_ts

        if tr.sum() < 480 or te.sum() < 100:
            continue

        Xv_tr  = X_vol.loc[idx[tr]];  Xv_te  = X_vol.loc[idx[te]]
        Xit_tr = Xi_it.loc[idx[tr]];  Xit_te = Xi_it.loc[idx[te]]
        Xtd_tr = Xi_tda.loc[idx[tr]]; Xtd_te = Xi_tda.loc[idx[te]]
        Xh_tr  = X_har.loc[idx[tr]];  Xh_te  = X_har.loc[idx[te]]

        ytr   = y_raw.loc[idx[tr]]
        yte   = y_raw.loc[idx[te]]
        naive = float(ytr.mean())

        m_har = _ridge()
        m_har.fit(Xh_tr, ytr)
        ph_te  = m_har.predict(Xh_te)
        r2_HAR = oos_r2(yte.values, ph_te, naive)
        mse_HAR = np.mean((yte.values - ph_te) ** 2)

        garch_ser = garch_forecasts.get((asset, h), pd.Series(dtype=float))
        garch_te  = garch_ser.reindex(idx[te]).values

        garch_valid = np.isfinite(garch_te)
        if garch_valid.sum() > 10:
            yte_g  = yte.values[garch_valid]
            pg_te  = garch_te[garch_valid]
            naive_g = float(ytr.mean())
            r2_GARCH  = oos_r2(yte_g, pg_te, naive_g)
            mse_GARCH = np.mean((yte_g - pg_te) ** 2)
            garch_te_full = np.where(garch_valid, garch_te, naive)
        else:
            r2_GARCH      = np.nan
            mse_GARCH     = np.nan
            garch_te_full = np.full(len(yte), naive)

        ck   = Xv_tr.corrwith(ytr).abs().replace([np.inf, -np.inf], np.nan).dropna()
        top3 = ck.nlargest(3).index.tolist()

        mb = _ridge()
        mb.fit(Xv_tr[top3], ytr)
        pb_tr = mb.predict(Xv_tr[top3])
        pb_te = mb.predict(Xv_te[top3])

        res_tr = ytr.values - pb_tr
        res_te = yte.values - pb_te

        mit = _ridge()
        mit.fit(Xit_tr, res_tr)
        pit_te  = mit.predict(Xit_te)
        pred_it = pb_te + pit_te

        mtda = _ridge()
        mtda.fit(Xtd_tr, res_tr)
        ptda_te  = mtda.predict(Xtd_te)
        pred_tda = pb_te + ptda_te

        Xc_tr = pd.concat([Xit_tr, Xtd_tr], axis=1)
        Xc_te = pd.concat([Xit_te, Xtd_te], axis=1)

        mct = _ridge()
        mct.fit(Xc_tr, res_tr)
        pct_te   = mct.predict(Xc_te)
        pred_all = pb_te + pct_te

        r2_A = oos_r2(yte.values, pb_te,    naive)
        r2_B = oos_r2(yte.values, pred_it,  naive)
        r2_C = oos_r2(yte.values, pred_tda, naive)
        r2_D = oos_r2(yte.values, pred_all, naive)

        rc_B = res_corr(res_te, pit_te)
        rc_C = res_corr(res_te, ptda_te)
        rc_D = res_corr(res_te, pct_te)

        mse_A = np.mean((yte.values - pb_te) ** 2)
        mse_B = np.mean((yte.values - pred_it) ** 2)
        mse_C = np.mean((yte.values - pred_tda) ** 2)
        mse_D = np.mean((yte.values - pred_all) ** 2)

        hr_B = hit_rate_vs_baseline(yte.values, pb_te, pred_it)
        hr_C = hit_rate_vs_baseline(yte.values, pb_te, pred_tda)
        hr_D = hit_rate_vs_baseline(yte.values, pb_te, pred_all)

        dB = (yte.values - pb_te)**2 - (yte.values - pred_it)**2
        dC = (yte.values - pb_te)**2 - (yte.values - pred_tda)**2
        dD = (yte.values - pb_te)**2 - (yte.values - pred_all)**2
        _, p_B = dm_test(dB)
        _, p_C = dm_test(dC)
        _, p_D = dm_test(dD)

        dD_vs_HAR = (yte.values - ph_te)**2 - (yte.values - pred_all)**2
        _, p_D_vs_HAR = dm_test(dD_vs_HAR)

        dD_vs_GARCH = (yte.values - garch_te_full)**2 - (yte.values - pred_all)**2
        _, p_D_vs_GARCH = dm_test(dD_vs_GARCH)

        rows.append({
            "asset":   asset,
            "horizon": h_label,
            "h":       h,

            "r2_HAR":   round(r2_HAR,   4),
            "r2_GARCH": round(r2_GARCH, 4) if np.isfinite(r2_GARCH) else np.nan,
            "r2_A":     round(r2_A,     4),
            "r2_B":     round(r2_B,     4),
            "r2_C":     round(r2_C,     4),
            "r2_D":     round(r2_D,     4),

            "dR2_B": round(r2_B - r2_A, 4),
            "dR2_C": round(r2_C - r2_A, 4),
            "dR2_D": round(r2_D - r2_A, 4),

            "dR2_D_vs_HAR":   round(r2_D - r2_HAR,   4),
            "dR2_D_vs_GARCH": round(r2_D - r2_GARCH, 4) if np.isfinite(r2_GARCH) else np.nan,

            "rc_B": round(rc_B, 4),
            "rc_C": round(rc_C, 4),
            "rc_D": round(rc_D, 4),

            "mse_ratio_B": round(mse_B / mse_A, 4) if mse_A > 0 else np.nan,
            "mse_ratio_C": round(mse_C / mse_A, 4) if mse_A > 0 else np.nan,
            "mse_ratio_D": round(mse_D / mse_A, 4) if mse_A > 0 else np.nan,

            "mse_ratio_D_vs_HAR":   round(mse_D / mse_HAR,   4) if mse_HAR   > 0 else np.nan,
            "mse_ratio_D_vs_GARCH": round(mse_D / mse_GARCH, 4) if (np.isfinite(r2_GARCH) and mse_GARCH > 0) else np.nan,

            "hit_B": round(hr_B, 4),
            "hit_C": round(hr_C, 4),
            "hit_D": round(hr_D, 4),

            "p_B": round(p_B, 4),
            "p_C": round(p_C, 4),
            "p_D": round(p_D, 4),
            "p_D_vs_HAR":   round(p_D_vs_HAR,   4),
            "p_D_vs_GARCH": round(p_D_vs_GARCH, 4),

            "sig_B":          sig(p_B),
            "sig_C":          sig(p_C),
            "sig_D":          sig(p_D),
            "sig_D_vs_HAR":   sig(p_D_vs_HAR),
            "sig_D_vs_GARCH": sig(p_D_vs_GARCH),

            "n_train": int(tr.sum()),
            "n_test":  int(te.sum()),
        })

df_best = pd.DataFrame(rows)
print(f"\nRows collected: {len(df_best)}")

if df_best.empty:
    raise ValueError("The result dataframe is empty.")

print("\n" + "=" * 115)
print("FINAL COMPARISON WITH AUTOMATICALLY SELECTED BEST FEATURES")
print("Target: logvol = log(realized_vol_fwd_h)")
print("HAR = HAR-RV | GARCH = GARCH(1,1) | A = vol-baseline | B = +IT | C = +TDA | D = +IT+TDA")
print("=" * 115)

print(
    f"\n  {'asset':<6} {'h':<4}"
    f" {'R²_HAR':>9} {'R²_GARCH':>10} {'R²_A':>9}"
    f" {'R²_B':>9} {'R²_C':>9} {'R²_D':>9}"
    f" {'ΔD/HAR':>9} {'ΔD/GARCH':>10}  best"
)
print("  " + "-" * 115)

prev = None
for _, row in df_best.sort_values(["asset", "h"]).iterrows():
    if prev and prev != row["asset"]:
        print("  " + "·" * 115)
    prev = row["asset"]

    vals = {
        "HAR":   row["r2_HAR"],
        "GARCH": row["r2_GARCH"] if np.isfinite(row["r2_GARCH"]) else -999,
        "A":     row["r2_A"],
        "B":     row["r2_B"],
        "C":     row["r2_C"],
        "D":     row["r2_D"],
    }
    best = max(vals, key=vals.get)

    dh = row["dR2_D_vs_HAR"]
    dg = row["dR2_D_vs_GARCH"]
    mh = "↑" if dh >  0.005 else ("↓" if dh < -0.005 else "~")
    mg = "↑" if (np.isfinite(dg) and dg >  0.005) else \
         ("↓" if (np.isfinite(dg) and dg < -0.005) else "~")

    garch_str = f"{row['r2_GARCH']:>+10.4f}" if np.isfinite(row["r2_GARCH"]) else f"{'N/A':>10}"
    dg_str    = f"{dg:>+9.4f}{mg}" if np.isfinite(dg) else f"{'N/A':>10}"

    print(
        f"  {row['asset']:<6} {row['horizon']:<4}"
        f" {row['r2_HAR']:>+9.4f}"
        f" {garch_str}"
        f" {row['r2_A']:>+9.4f}"
        f" {row['r2_B']:>+9.4f}"
        f" {row['r2_C']:>+9.4f}"
        f" {row['r2_D']:>+9.4f}"
        f" {row['dR2_D_vs_HAR']:>+8.4f}{mh}"
        f" {dg_str}"
        f"  {best}"
    )

print("\n" + "=" * 75)
print("SUMMARY")
print("=" * 75)

n = len(df_best)
n_garch = df_best["r2_GARCH"].notna().sum()

print(f"""
  Config              mean R²    mean ΔR² vs A   ΔR²>0
  -------------------------------------------------------
  HAR-RV              {df_best['r2_HAR'].mean():>+8.4f}    —
  GARCH(1,1)          {df_best['r2_GARCH'].mean():>+8.4f}    —         (N={n_garch}/{n} valid)
  A  (vol baseline)   {df_best['r2_A'].mean():>+8.4f}    —
  B  (+best IT)       {df_best['r2_B'].mean():>+8.4f}   {df_best['dR2_B'].mean():>+8.4f}   {(df_best['dR2_B']>0).sum():>2}/{n}
  C  (+best TDA)      {df_best['r2_C'].mean():>+8.4f}   {df_best['dR2_C'].mean():>+8.4f}   {(df_best['dR2_C']>0).sum():>2}/{n}
  D  (+both)          {df_best['r2_D'].mean():>+8.4f}   {df_best['dR2_D'].mean():>+8.4f}   {(df_best['dR2_D']>0).sum():>2}/{n}

  D vs HAR-RV:    mean ΔR²={df_best['dR2_D_vs_HAR'].mean():>+.4f}   D beats HAR:   {(df_best['dR2_D_vs_HAR']>0).sum()}/{n}
  D vs GARCH(1,1): mean ΔR²={df_best['dR2_D_vs_GARCH'].mean():>+.4f}   D beats GARCH: {(df_best['dR2_D_vs_GARCH']>0).sum()}/{n_garch}

  DM test (D vs HAR-RV):
    p<0.01: {(df_best['p_D_vs_HAR']<0.01).sum()}/{n}   p<0.05: {(df_best['p_D_vs_HAR']<0.05).sum()}/{n}   p<0.10: {(df_best['p_D_vs_HAR']<0.10).sum()}/{n}
  DM test (D vs GARCH):
    p<0.01: {(df_best['p_D_vs_GARCH']<0.01).sum()}/{n}   p<0.05: {(df_best['p_D_vs_GARCH']<0.05).sum()}/{n}   p<0.10: {(df_best['p_D_vs_GARCH']<0.10).sum()}/{n}
""")

print("Top-5 tasks by ΔR² (D vs HAR-RV):")
for _, row in df_best.nlargest(5, "dR2_D_vs_HAR").iterrows():
    print(
        f"  {row['asset']} h={row['horizon']}: "
        f"R²_HAR={row['r2_HAR']:+.4f}  "
        f"R²_GARCH={row['r2_GARCH']:+.4f}  "
        f"R²_D={row['r2_D']:+.4f}  "
        f"Δ_HAR={row['dR2_D_vs_HAR']:+.4f}  "
        f"p={row['p_D_vs_HAR']:.4f}{row['sig_D_vs_HAR']}"
    )

asset_summary = (
    df_best.groupby("asset", as_index=False)
    .agg(
        r2_HAR=("r2_HAR",   "mean"),
        r2_GARCH=("r2_GARCH","mean"),
        r2_A=("r2_A", "mean"),
        r2_B=("r2_B", "mean"),
        r2_C=("r2_C", "mean"),
        r2_D=("r2_D", "mean"),
        dR2_B=("dR2_B", "mean"),
        dR2_C=("dR2_C", "mean"),
        dR2_D=("dR2_D", "mean"),
        dR2_D_vs_HAR=("dR2_D_vs_HAR",   "mean"),
        dR2_D_vs_GARCH=("dR2_D_vs_GARCH","mean"),
        rc_B=("rc_B", "mean"),
        rc_C=("rc_C", "mean"),
        rc_D=("rc_D", "mean"),
        mse_ratio_B=("mse_ratio_B", "mean"),
        mse_ratio_C=("mse_ratio_C", "mean"),
        mse_ratio_D=("mse_ratio_D", "mean"),
        hit_B=("hit_B", "mean"),
        hit_C=("hit_C", "mean"),
        hit_D=("hit_D", "mean"),
    )
)

def choose_best_config(row):
    vals = {
        "HAR":   row["r2_HAR"],
        "GARCH": row["r2_GARCH"] if np.isfinite(row["r2_GARCH"]) else -999,
        "A":     row["r2_A"],
        "B":     row["r2_B"],
        "C":     row["r2_C"],
        "D":     row["r2_D"],
    }
    return max(vals, key=vals.get)

def choose_best_addon(row):
    vals = {"B": row["dR2_B"], "C": row["dR2_C"], "D": row["dR2_D"]}
    return max(vals, key=vals.get)

asset_summary["best_config"] = asset_summary.apply(choose_best_config, axis=1)
asset_summary["best_addon"]  = asset_summary.apply(choose_best_addon,  axis=1)
asset_summary = asset_summary.sort_values(
    ["r2_D","r2_C","r2_B","r2_A"], ascending=False
).reset_index(drop=True)

print("\n" + "=" * 115)
print("ASSET-LEVEL AGGREGATION (MEAN OVER HORIZONS)")
print("=" * 115)

print(
    f"\n  {'asset':<6}"
    f" {'R²_HAR':>9} {'R²_GARCH':>10} {'R²_A':>9}"
    f" {'R²_B':>9} {'R²_C':>9} {'R²_D':>9}"
    f" {'ΔD/HAR':>9} {'ΔD/GARCH':>10}"
    f" {'best_cfg':>10}"
)
print("  " + "-" * 115)

for _, row in asset_summary.iterrows():
    garch_str = f"{row['r2_GARCH']:>+10.4f}" if np.isfinite(row["r2_GARCH"]) else f"{'N/A':>10}"
    dg_str    = f"{row['dR2_D_vs_GARCH']:>+10.4f}" if np.isfinite(row["dR2_D_vs_GARCH"]) else f"{'N/A':>10}"
    print(
        f"  {row['asset']:<6}"
        f" {row['r2_HAR']:>+9.4f}"
        f" {garch_str}"
        f" {row['r2_A']:>+9.4f}"
        f" {row['r2_B']:>+9.4f}"
        f" {row['r2_C']:>+9.4f}"
        f" {row['r2_D']:>+9.4f}"
        f" {row['dR2_D_vs_HAR']:>+9.4f}"
        f" {dg_str}"
        f" {row['best_config']:>10}"
    )

print("\nBest configuration counts across assets:")
print(asset_summary["best_config"].value_counts().sort_index())

print("\nBuilding final plots...")

fig, axes = plt.subplots(2, 3, figsize=(21, 11))

asset_order = (
    df_best.groupby("asset")["r2_D"]
    .mean()
    .sort_values(ascending=False)
    .index.tolist()
)
h_order = ["1d","2d","4d","6d","7d","14d"]
x = np.arange(len(asset_order))
w = 0.14

ax = axes[0, 0]
for ci, (col, label, color) in enumerate([
    ("r2_HAR",   "HAR-RV",       "#795548"),
    ("r2_GARCH", "GARCH(1,1)",   "#9C27B0"),
    ("r2_A",     "A: Baseline",  "#9E9E9E"),
    ("r2_B",     "B: +IT",       "#2196F3"),
    ("r2_C",     "C: +TDA",      "#FF9800"),
    ("r2_D",     "D: +both",     "#4CAF50"),
]):
    vals = [df_best[df_best.asset == a][col].mean() for a in asset_order]
    ax.bar(x + (ci - 2.5) * w, vals, w * 0.9, label=label, color=color, alpha=0.85)

ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels(asset_order, fontsize=8)
ax.set_ylabel("OOS R²")
ax.set_title("OOS R² by asset — 6 configurations\n(logvol, all horizons)", fontsize=10)
ax.legend(fontsize=7)
ax.grid(alpha=0.2, axis="y")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax = axes[0, 1]
for col, label, color, ls in [
    ("r2_HAR",   "HAR-RV",      "#795548", ":"),
    ("r2_GARCH", "GARCH(1,1)",  "#9C27B0", ":"),
    ("r2_A",     "A: Baseline", "#9E9E9E", "-"),
    ("r2_B",     "B: +IT",      "#2196F3", "-"),
    ("r2_C",     "C: +TDA",     "#FF9800", "--"),
    ("r2_D",     "D: +both",    "#4CAF50", "-"),
]:
    vals = [df_best[df_best.horizon == h][col].mean() for h in h_order]
    ax.plot(h_order, vals, "o" + ls, color=color, lw=2, label=label, ms=6)

ax.axhline(0, color="black", lw=0.5, ls=":")
ax.set_ylabel("OOS R²")
ax.set_xlabel("Forecast horizon")
ax.set_title("OOS R² by horizon\n(mean over assets)", fontsize=10)
ax.legend(fontsize=7)
ax.grid(alpha=0.2)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax = axes[0, 2]
vals_har = [df_best[df_best.asset==a]["dR2_D_vs_HAR"].mean() for a in asset_order]
colors_bar = ["#4CAF50" if v >= 0 else "#F44336" for v in vals_har]
ax.bar(np.arange(len(asset_order)), vals_har, color=colors_bar, alpha=0.85)
ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(np.arange(len(asset_order)))
ax.set_xticklabels(asset_order, fontsize=9)
ax.set_ylabel("ΔR² (D minus HAR-RV)")
ax.set_title("Configuration D vs HAR-RV\n(mean over horizons)", fontsize=10)
ax.grid(alpha=0.2, axis="y")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax = axes[1, 0]
x2 = np.arange(len(asset_order))
w2 = 0.28
for ci, (col, label, color) in enumerate([
    ("dR2_B", "B: +IT",   "#2196F3"),
    ("dR2_C", "C: +TDA",  "#FF9800"),
    ("dR2_D", "D: +both", "#4CAF50"),
]):
    vals = [df_best[df_best.asset==a][col].mean() for a in asset_order]
    ax.bar(x2 + (ci-1)*w2, vals, w2*0.9, label=label, color=color, alpha=0.85)

ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(x2)
ax.set_xticklabels(asset_order, fontsize=9)
ax.set_ylabel("ΔR² relative to baseline A")
ax.set_title("Incremental ΔR² over vol baseline\n(logvol, all horizons)", fontsize=10)
ax.legend(fontsize=8)
ax.grid(alpha=0.2, axis="y")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax = axes[1, 1]
pivot = df_best.pivot_table(
    index="asset", columns="horizon",
    values="mse_ratio_D", aggfunc="mean"
).reindex(asset_order)[h_order]

im = ax.imshow(pivot.values, cmap="RdYlGn_r", vmin=0.85, vmax=1.15, aspect="auto")
ax.set_xticks(range(len(h_order)));    ax.set_xticklabels(h_order)
ax.set_yticks(range(len(asset_order))); ax.set_yticklabels(asset_order)
for i in range(len(asset_order)):
    for j in range(len(h_order)):
        val = pivot.values[i, j]
        if np.isfinite(val):
            col = "white" if abs(val-1) > 0.07 else "black"
            mark = "✓" if val < 1.0 else ""
            ax.text(j, i, f"{val:.3f}{mark}", ha="center", va="center",
                    fontsize=8, color=col)
plt.colorbar(im, ax=ax, label="MSE ratio (D / baseline A)\n<1.0 = better ✓")
ax.set_title("MSE ratio: D vs vol baseline\n(green < 1 = better)", fontsize=10)

ax = axes[1, 2]
pivot2 = df_best.pivot_table(
    index="asset", columns="horizon",
    values="mse_ratio_D_vs_HAR", aggfunc="mean"
).reindex(asset_order)[h_order]

im2 = ax.imshow(pivot2.values, cmap="RdYlGn_r", vmin=0.85, vmax=1.15, aspect="auto")
ax.set_xticks(range(len(h_order)));    ax.set_xticklabels(h_order)
ax.set_yticks(range(len(asset_order))); ax.set_yticklabels(asset_order)
for i in range(len(asset_order)):
    for j in range(len(h_order)):
        val = pivot2.values[i, j]
        if np.isfinite(val):
            col = "white" if abs(val-1) > 0.07 else "black"
            mark = "✓" if val < 1.0 else ""
            ax.text(j, i, f"{val:.3f}{mark}", ha="center", va="center",
                    fontsize=8, color=col)
plt.colorbar(im2, ax=ax, label="MSE ratio (D / HAR-RV)\n<1.0 = better ✓")
ax.set_title("MSE ratio: D vs HAR-RV\n(green < 1 = better)", fontsize=10)

plt.suptitle(
    "Final comparison (IT W=96): HAR-RV / vol-baseline / +IT / +TDA / +both\n"
    f"Target: logvol = log(realized_vol_fwd) | holdout {SPLIT_DATE} →",
    fontsize=12, y=1.01
)
plt.tight_layout()
plt.savefig("fig_final_comparison_W96.png", bbox_inches="tight", dpi=150)
plt.savefig("fig_final_comparison_W96.pdf", bbox_inches="tight", dpi=150)
plt.show()
print("Saved: fig_final_comparison_W96.png / .pdf")

final_comparison_W96 = df_best.copy()
best_feature_sets_W96 = {"BEST_IT": BEST_IT, "BEST_TDA": BEST_TDA}
print(f"\nfinal_comparison_W96 saved in memory ({len(df_best)} rows)")
print("best_feature_sets_W96 saved in memory")

Building GARCH(1,1) forecasts for all assets and horizons...
  BTC h=1d: 2977 test points
  BTC h=2d: 2977 test points
  BTC h=4d: 2977 test points
  BTC h=6d: 2977 test points
  BTC h=7d: 2977 test points
  BTC h=14d: 2977 test points
  ETH h=1d: 2977 test points
  ETH h=2d: 2977 test points
  ETH h=4d: 2977 test points
  ETH h=6d: 2977 test points
  ETH h=7d: 2977 test points
  ETH h=14d: 2977 test points
  SOL h=1d: 2977 test points
  SOL h=2d: 2977 test points
  SOL h=4d: 2977 test points
  SOL h=6d: 2977 test points
  SOL h=7d: 2977 test points
  SOL h=14d: 2977 test points
  BNB h=1d: 2977 test points
  BNB h=2d: 2977 test points
  BNB h=4d: 2977 test points
  BNB h=6d: 2977 test points
  BNB h=7d: 2977 test points
  BNB h=14d: 2977 test points
  XRP h=1d: 2977 test points
  XRP h=2d: 2977 test points
  XRP h=4d: 2977 test points
  XRP h=6d: 2977 test points
  XRP h=7d: 2977 test points
  XRP h=14d: 2977 test points
  DOGE h=1d: 2977 test points
  DOGE h=2d: 2977 test points
  DO

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import mutual_info_score

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

WINDOWS_TO_TEST = [96, 144, 192, 384]
IT_STEP  = 4
IT_BINS  = 10

SPLIT_DATE = "2025-07-01"
PURGE_BARS = 96

QUANTILES = [0.60, 0.70, 0.80, 0.90]
HORIZONS  = {48: "1d", 96: "2d", 192: "4d"}

ASSETS = [a for a in ["BTC","ETH","SOL","BNB","XRP","DOGE","ADA","AVAX"]
          if a in ret.columns]

VOL_WINDOW = 48
ANN        = np.sqrt(365 * 48)

N_IT_FEATS  = 6
N_TDA_FEATS = 5

PERM_USEFUL_THRESHOLD = -0.002
TOP_IT = 10

def _logit(C=0.1):
    return Pipeline([
        ("sc", StandardScaler()),
        ("m", LogisticRegression(C=C, max_iter=2000,
                                  class_weight="balanced",
                                  solver="lbfgs"))
    ])

def safe_roc(y, s):
    try:    return float(roc_auc_score(y, s))
    except: return np.nan

def safe_pr(y, s):
    try:    return float(average_precision_score(y, s))
    except: return np.nan

def fit_predict_logit(Xtr, ytr, Xte):
    try:
        m = _logit(); m.fit(Xtr, ytr)
        return m.predict_proba(Xte)[:, 1]
    except:
        return np.full(len(Xte), float(np.mean(ytr)))

def _rank_to_bins(x, bins=10):
    x = np.asarray(x, dtype=float)
    ranks = pd.Series(x).rank(method="average").values
    scaled = np.floor((ranks - 1) / max(1, len(x)) * bins).astype(int)
    return np.clip(scaled, 0, bins - 1)

def _drop_finite(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    return x[mask], y[mask]

def mi_hist_rank(x, y, bins=10):
    x, y = _drop_finite(x, y)
    if len(x) < max(20, bins): return np.nan
    bx = _rank_to_bins(x, bins)
    by = _rank_to_bins(y, bins)
    c  = np.histogram2d(bx, by, bins=bins,
                        range=[[0,bins],[0,bins]])[0]
    return float(mutual_info_score(None, None, contingency=c))

def _entropy(x, bins=10):
    x = x[np.isfinite(x)]
    if len(x) < 5: return np.nan
    bx = _rank_to_bins(x, bins)
    counts = np.bincount(bx, minlength=bins).astype(float)
    p = counts / counts.sum(); p = p[p > 0]
    return float(-np.sum(p * np.log(p)))

def _joint_entropy(xs, bins=10):
    if any(len(x) < 5 for x in xs): return np.nan
    multi  = np.stack([_rank_to_bins(x, bins) for x in xs], axis=1)
    codes  = multi @ (bins ** np.arange(len(xs)))
    counts = np.bincount(codes, minlength=bins**len(xs)).astype(float)
    p = counts / counts.sum(); p = p[p > 0]
    return float(-np.sum(p * np.log(p)))

def build_it_features_for_window(ret_df, assets, window, step, bins):
    avail  = [a for a in assets if a in ret_df.columns]
    idx    = ret_df.index
    n      = len(idx)
    starts = np.arange(0, n - window + 1, step)

    pairs = [(avail[i], avail[j])
              for i in range(len(avail))
              for j in range(i+1, len(avail))]
    triples = [(avail[i], avail[j], avail[k])
                for i in range(len(avail))
                for j in range(i+1, len(avail))
                for k in range(j+1, len(avail))]
    quads = [(avail[i], avail[j], avail[k], avail[l])
              for i in range(len(avail))
              for j in range(i+1, len(avail))
              for k in range(j+1, len(avail))
              for l in range(k+1, len(avail))]

    cols = (
        ["pairwise_mi_mean","pairwise_mi_std",
         "pairwise_mi_max", "pairwise_mi_min"] +
        [f"kl_{a}" for a in avail] +
        ["coinfo3_mean","coinfo3_std","coinfo3_max","coinfo3_min"] +
        ["coinfo4_mean","coinfo4_std","coinfo4_max","coinfo4_min"]
    )
    out = pd.DataFrame(np.nan, index=idx, columns=cols)

    print(f"    Building IT W={window}: {len(starts)} windows...")

    for s in starts:
        e   = s + window
        sub = ret_df.iloc[s:e]

        mi_arr = np.array(
            [v for v in [mi_hist_rank(sub[a].values, sub[b].values, bins)
                         for a, b in pairs] if np.isfinite(v)], dtype=float)
        if len(mi_arr):
            out.at[idx[e-1], "pairwise_mi_mean"] = mi_arr.mean()
            out.at[idx[e-1], "pairwise_mi_std"]  = mi_arr.std(ddof=0)
            out.at[idx[e-1], "pairwise_mi_max"]  = mi_arr.max()
            out.at[idx[e-1], "pairwise_mi_min"]  = mi_arr.min()

        if s >= window:
            prev = ret_df.iloc[s-window:s]
            for a in avail:
                xc = sub[a].values; xp = prev[a].values
                if len(xc) < 5 or len(xp) < 5: continue
                pc = (np.bincount(_rank_to_bins(xc, bins),
                      minlength=bins).astype(float) + 1e-8)
                pp = (np.bincount(_rank_to_bins(xp, bins),
                      minlength=bins).astype(float) + 1e-8)
                pc /= pc.sum(); pp /= pp.sum()
                out.at[idx[e-1], f"kl_{a}"] = float(np.sum(pc * np.log(pc/pp)))

        ci3_vals = []
        for (a, b, c) in triples:
            xa, xb, xc_ = sub[a].values, sub[b].values, sub[c].values
            vs = [_entropy(xa), _entropy(xb), _entropy(xc_),
                  _joint_entropy([xa,xb]), _joint_entropy([xa,xc_]),
                  _joint_entropy([xb,xc_]), _joint_entropy([xa,xb,xc_])]
            if all(v is not None and np.isfinite(v) for v in vs):
                h_a,h_b,h_c,h_ab,h_ac,h_bc,h_abc = vs
                ci3_vals.append(h_a+h_b+h_c - h_ab-h_ac-h_bc + h_abc)
        if ci3_vals:
            c3 = np.array(ci3_vals)
            out.at[idx[e-1],"coinfo3_mean"] = c3.mean()
            out.at[idx[e-1],"coinfo3_std"]  = c3.std(ddof=0)
            out.at[idx[e-1],"coinfo3_max"]  = c3.max()
            out.at[idx[e-1],"coinfo3_min"]  = c3.min()

        ci4_vals = []
        for (a,b,c,d) in quads:
            xs_ = [sub[x].values for x in [a,b,c,d]]
            h1 = [_entropy(x) for x in xs_]
            h2 = [_joint_entropy([xs_[i],xs_[j]])
                  for i in range(4) for j in range(i+1,4)]
            h3 = [_joint_entropy([xs_[i],xs_[j],xs_[k]])
                  for i in range(4) for j in range(i+1,4)
                  for k in range(j+1,4)]
            h4 = _joint_entropy(xs_)
            if all(v is not None and np.isfinite(v) for v in h1+h2+h3+[h4]):
                ci4_vals.append(sum(h1)-sum(h2)+sum(h3)-h4)
        if ci4_vals:
            c4 = np.array(ci4_vals)
            out.at[idx[e-1],"coinfo4_mean"] = c4.mean()
            out.at[idx[e-1],"coinfo4_std"]  = c4.std(ddof=0)
            out.at[idx[e-1],"coinfo4_max"]  = c4.max()
            out.at[idx[e-1],"coinfo4_min"]  = c4.min()

    return out.ffill()

print("Loading TDA features...")

if "feat_analysis" in dir() and "X_tda_full" in feat_analysis:
    X_tda_all = feat_analysis["X_tda_full"].copy()
elif "X_tda_full" in dir():
    X_tda_all = X_tda_full.copy()
else:
    tda_frames = [tda_A.add_prefix("A_")]
    for mode in ["I","II","C"]:
        d = tda_B[mode]
        df_m = pd.DataFrame({
            f"B{mode}_tp_H0":  d["tp_H0"], f"B{mode}_ent_H0": d["ent_H0"],
            f"B{mode}_tp_H1":  d["tp_H1"], f"B{mode}_ent_H1": d["ent_H1"],
            f"B{mode}_L1_H0":  d["L1_H0"], f"B{mode}_L1_H1":  d["L1_H1"],
        }, index=pd.DatetimeIndex(d["times"]))
        tda_frames.append(df_m)
    X_tda_all = (pd.concat(tda_frames, axis=1)
                   .sort_index()
                   .reindex(ret.index, method="ffill"))

print(f"  TDA shape: {X_tda_all.shape}")

if "best_feature_sets" in dir():
    BEST_TDA_GLOBAL = [c for c in best_feature_sets.get("BEST_TDA", [])
                       if c in X_tda_all.columns]
else:
    BEST_TDA_GLOBAL = [c for c in
                       ["BI_L1_H0","BC_ent_H1","BII_tp_H1",
                        "BII_L1_H1","A_T_H0","A_E_H0"]
                       if c in X_tda_all.columns]

print(f"  TDA pool: {BEST_TDA_GLOBAL}")

vol_now_rank = {}
for asset in ASSETS:
    vn = ret[asset].rolling(VOL_WINDOW).std() * ANN
    vol_now_rank[asset] = vn.expanding(min_periods=100).rank(pct=True)

def run_stress_for_window(X_it_w, best_it_global, window_label):
    split_ts = pd.Timestamp(SPLIT_DATE)
    rows = []

    for asset in ASSETS:
        r       = ret[asset]
        vol_now = r.rolling(VOL_WINDOW).std() * ANN
        vn_rank = vol_now_rank[asset]

        Xi_it = X_it_w[best_it_global].copy()
        Xi_td = X_tda_all[BEST_TDA_GLOBAL].copy()

        for h, h_label in HORIZONS.items():
            vol_fwd = r.rolling(h).std().shift(-h) * ANN

            for q in QUANTILES:
                vol_q = vol_now.expanding(min_periods=100).quantile(q)
                y_raw = (vol_fwd > vol_q).astype(float).dropna()

                idx = (Xi_it.index
                       .intersection(Xi_td.index)
                       .intersection(vn_rank.index)
                       .intersection(y_raw.index))
                valid = (Xi_it.loc[idx].notna().all(1)
                         & Xi_td.loc[idx].notna().all(1)
                         & vn_rank.loc[idx].notna()
                         & y_raw.loc[idx].notna())
                idx = idx[valid]
                if len(idx) < 1000: continue

                tr = idx < (split_ts - pd.Timedelta(minutes=30*PURGE_BARS))
                te = idx >= split_ts
                if tr.sum() < 480 or te.sum() < 100: continue

                ytr = y_raw.loc[idx[tr]]; yte = y_raw.loc[idx[te]]
                pos_tr = float(ytr.mean()); pos_te = float(yte.mean())
                if pos_tr < 0.03 or pos_tr > 0.97: continue
                if yte.sum() < 5: continue

                Xvol_tr = pd.DataFrame(
                    vn_rank.loc[idx[tr]].values.reshape(-1,1),
                    index=idx[tr], columns=["vol_rank"])
                Xvol_te = pd.DataFrame(
                    vn_rank.loc[idx[te]].values.reshape(-1,1),
                    index=idx[te], columns=["vol_rank"])

                Xi_tr_all = Xi_it.loc[idx[tr]]
                Xi_te_all = Xi_it.loc[idx[te]]
                Xt_tr_all = Xi_td.loc[idx[tr]]
                Xt_te_all = Xi_td.loc[idx[te]]

                mi_it = mutual_info_classif(
                    Xi_tr_all.values, ytr.values, random_state=42)
                mi_td = mutual_info_classif(
                    Xt_tr_all.values, ytr.values, random_state=42)

                it_cols = (pd.Series(mi_it, index=Xi_tr_all.columns)
                             .nlargest(min(N_IT_FEATS, Xi_tr_all.shape[1]))
                             .index.tolist())
                td_cols = (pd.Series(mi_td, index=Xt_tr_all.columns)
                             .nlargest(min(N_TDA_FEATS, Xt_tr_all.shape[1]))
                             .index.tolist())

                Xit_tr = Xi_tr_all[it_cols]; Xit_te = Xi_te_all[it_cols]
                Xtd_tr = Xt_tr_all[td_cols]; Xtd_te = Xt_te_all[td_cols]

                yte_np = yte.values.astype(int)

                sc_A = fit_predict_logit(Xvol_tr, ytr, Xvol_te)
                sc_B = fit_predict_logit(
                    pd.concat([Xvol_tr, Xit_tr], axis=1), ytr,
                    pd.concat([Xvol_te, Xit_te], axis=1))
                sc_C = fit_predict_logit(
                    pd.concat([Xvol_tr, Xtd_tr], axis=1), ytr,
                    pd.concat([Xvol_te, Xtd_te], axis=1))
                sc_D = fit_predict_logit(
                    pd.concat([Xvol_tr, Xit_tr, Xtd_tr], axis=1), ytr,
                    pd.concat([Xvol_te, Xit_te, Xtd_te], axis=1))

                roc_A = safe_roc(yte_np, sc_A)
                roc_B = safe_roc(yte_np, sc_B)
                roc_C = safe_roc(yte_np, sc_C)
                roc_D = safe_roc(yte_np, sc_D)

                rows.append({
                    "window":   window_label,
                    "asset":    asset,
                    "horizon":  h_label,
                    "h":        h,
                    "q":        q,
                    "pos_tr":   round(pos_tr, 3),
                    "pos_te":   round(pos_te, 3),
                    "roc_A":    round(roc_A, 4),
                    "roc_B":    round(roc_B, 4),
                    "roc_C":    round(roc_C, 4),
                    "roc_D":    round(roc_D, 4),
                    "pr_A":     round(safe_pr(yte_np, sc_A), 4),
                    "pr_B":     round(safe_pr(yte_np, sc_B), 4),
                    "pr_C":     round(safe_pr(yte_np, sc_C), 4),
                    "pr_D":     round(safe_pr(yte_np, sc_D), 4),
                    "droc_B":   round(roc_B - roc_A, 4),
                    "droc_C":   round(roc_C - roc_A, 4),
                    "droc_D":   round(roc_D - roc_A, 4),
                    "dpr_B":    round(safe_pr(yte_np,sc_B)-safe_pr(yte_np,sc_A), 4),
                    "dpr_C":    round(safe_pr(yte_np,sc_C)-safe_pr(yte_np,sc_A), 4),
                    "dpr_D":    round(safe_pr(yte_np,sc_D)-safe_pr(yte_np,sc_A), 4),
                    "n_train":  int(tr.sum()),
                    "n_test":   int(te.sum()),
                })

    return pd.DataFrame(rows)

print("\n" + "=" * 70)
print("WINDOW SENSITIVITY — STRESS EVENT CLASSIFICATION")
print(f"Windows: {WINDOWS_TO_TEST} bars")
print("=" * 70)

if "feat_analysis" in dir():
    avg_perm_src = feat_analysis["perm"]
else:
    avg_perm_src = avg_perm

perm_series = pd.Series(avg_perm_src).sort_values()

all_stress_results = []

for w in WINDOWS_TO_TEST:
    w_label = f"W={w}"
    print(f"\n{'='*60}")
    print(f"  Window: {w} bars (~{w/48:.1f} days)")
    print(f"{'='*60}")

    X_it_w = build_it_features_for_window(
        ret, ASSETS, w, IT_STEP, IT_BINS)

    best_it_w = [
        f for f in perm_series.index
        if (f in X_it_w.columns)
        and (perm_series[f] < PERM_USEFUL_THRESHOLD)
    ][:TOP_IT]

    if len(best_it_w) == 0:
        best_it_w = [f for f in perm_series.index
                     if f in X_it_w.columns][:6]
        print(f"  WARNING: fallback to top-6 by perm score")

    print(f"  IT pool ({len(best_it_w)}): {best_it_w}")

    df_s = run_stress_for_window(X_it_w, best_it_w, w_label)
    all_stress_results.append(df_s)

    n = len(df_s)
    if n > 0:
        print(f"\n  Results ({n} tasks):")
        for q in QUANTILES:
            sub = df_s[df_s.q == q]
            if sub.empty: continue
            print(f"    Q{q:.0%}: "
                  f"ΔROC_B={sub['droc_B'].mean():>+.4f}  "
                  f"ΔROC_C={sub['droc_C'].mean():>+.4f}  "
                  f"ΔROC_D={sub['droc_D'].mean():>+.4f}  "
                  f"D>A: {(sub['droc_D']>0.01).sum()}/{len(sub)}")
    else:
        print("  WARNING: no rows collected.")

df_stress_window = pd.concat(all_stress_results, ignore_index=True)
print(f"\nTotal rows: {len(df_stress_window)}")

print("\n" + "=" * 80)
print("STRESS WINDOW SENSITIVITY SUMMARY")
print("=" * 80)

print(f"\n  {'Window':<10} {'days':>5}"
      f" {'ΔROC_B':>8} {'ΔROC_C':>8} {'ΔROC_D':>8}"
      f" {'ΔPR_B':>8} {'ΔPR_C':>8} {'ΔPR_D':>8}"
      f" {'D>A(ROC)':>10}")
print("  " + "-" * 80)

for w in WINDOWS_TO_TEST:
    wl  = f"W={w}"
    sub = df_stress_window[df_stress_window.window == wl]
    if sub.empty: continue
    n = len(sub)
    beats = (sub["droc_D"] > 0.01).sum()
    print(f"  {wl:<10} {w/48:>5.1f}"
          f" {sub['droc_B'].mean():>+8.4f}"
          f" {sub['droc_C'].mean():>+8.4f}"
          f" {sub['droc_D'].mean():>+8.4f}"
          f" {sub['dpr_B'].mean():>+8.4f}"
          f" {sub['dpr_C'].mean():>+8.4f}"
          f" {sub['dpr_D'].mean():>+8.4f}"
          f" {beats:>6}/{n}")

print(f"\n  By threshold (ΔROC_D, mean over assets and horizons):")
print(f"  {'Window':<10}", end="")
for q in QUANTILES:
    print(f"  {'Q'+f'{q:.0%}':>8}", end="")
print()
print("  " + "-" * (10 + 10*len(QUANTILES)))

for w in WINDOWS_TO_TEST:
    wl  = f"W={w}"
    sub = df_stress_window[df_stress_window.window == wl]
    print(f"  {wl:<10}", end="")
    for q in QUANTILES:
        val = sub[sub.q==q]["droc_D"].mean() if not sub.empty else np.nan
        print(f"  {val:>+8.4f}", end="")
    print()

print(f"\n  Asset-level ΔROC_D at Q80:")
print(f"  {'Asset':<6}", end="")
for w in WINDOWS_TO_TEST:
    print(f"  {'W='+str(w):>10}", end="")
print()
print("  " + "-" * (6 + 12*len(WINDOWS_TO_TEST)))

for asset in ASSETS:
    print(f"  {asset:<6}", end="")
    for w in WINDOWS_TO_TEST:
        wl  = f"W={w}"
        sub = df_stress_window[
            (df_stress_window.window==wl) &
            (df_stress_window.asset==asset) &
            (df_stress_window.q==0.80)]
        val = sub["droc_D"].mean() if not sub.empty else np.nan
        if np.isfinite(val):
            print(f"  {val:>+10.4f}", end="")
        else:
            print(f"  {'N/A':>10}", end="")
    print()

print(f"\n  TDA invariance check (ΔROC_C should be identical):")
print(f"  {'Window':<10} {'ΔROC_C':>8} {'ΔPR_C':>8}")
for w in WINDOWS_TO_TEST:
    wl  = f"W={w}"
    sub = df_stress_window[df_stress_window.window == wl]
    print(f"  {wl:<10} {sub['droc_C'].mean():>+8.4f}"
          f" {sub['dpr_C'].mean():>+8.4f}")

print("\nBuilding stress window sensitivity plots...")

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
w_labels = [f"W={w}" for w in WINDOWS_TO_TEST]
w_days   = [f"{w/48:.1f}d" for w in WINDOWS_TO_TEST]
colors   = {"B": "#2196F3", "C": "#FF9800", "D": "#4CAF50"}

ax = axes[0]
for cfg in ["B","C","D"]:
    vals = [df_stress_window[df_stress_window.window==wl][f"droc_{cfg}"].mean()
            for wl in w_labels]
    ax.plot(w_days, vals, "o-", color=colors[cfg],
            lw=2, ms=8, label=f"Config {cfg}")
ax.axhline(0, color="black", lw=0.8, ls=":")
ax.set_xlabel("IT window size")
ax.set_ylabel("Mean ΔROC-AUC vs vol_now baseline")
ax.set_title("Mean ΔROC by IT window\n(all assets, horizons, thresholds)", fontsize=10)
ax.legend(fontsize=9)
ax.grid(alpha=0.25)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax = axes[1]
q_labels = [f"Q{q:.0%}" for q in QUANTILES]
x = np.arange(len(QUANTILES))
w_bar = 0.2
bar_colors = ["#1565C0","#42A5F5","#4CAF50","#FF9800"]
for wi, w in enumerate(WINDOWS_TO_TEST):
    wl   = f"W={w}"
    vals = [df_stress_window[
                (df_stress_window.window==wl) &
                (df_stress_window.q==q)]["droc_D"].mean()
            for q in QUANTILES]
    ax.bar(x + (wi-1.5)*w_bar, vals, w_bar*0.9,
           label=f"W={w} (~{w/48:.0f}d)",
           color=bar_colors[wi], alpha=0.85)
ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels(q_labels)
ax.set_xlabel("Stress threshold")
ax.set_ylabel("ΔROC-AUC (D vs baseline)")
ax.set_title("ΔROC_D by threshold × window\n(mean over assets and horizons)", fontsize=10)
ax.legend(fontsize=8)
ax.grid(alpha=0.2, axis="y")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax = axes[2]
heat = np.full((len(ASSETS), len(WINDOWS_TO_TEST)), np.nan)
for j, w in enumerate(WINDOWS_TO_TEST):
    wl = f"W={w}"
    for i, asset in enumerate(ASSETS):
        sub = df_stress_window[
            (df_stress_window.window==wl) &
            (df_stress_window.asset==asset) &
            (df_stress_window.q==0.80)]
        if not sub.empty:
            heat[i, j] = sub["droc_D"].mean()

im = ax.imshow(heat, cmap="RdYlGn", vmin=-0.10, vmax=0.15, aspect="auto")
ax.set_xticks(range(len(WINDOWS_TO_TEST)))
ax.set_xticklabels(w_days)
ax.set_yticks(range(len(ASSETS)))
ax.set_yticklabels(ASSETS)
ax.set_xlabel("IT window size")
ax.set_title("ΔROC_D at Q80 by asset × window\n(green = better than baseline)", fontsize=10)

for i in range(len(ASSETS)):
    for j in range(len(WINDOWS_TO_TEST)):
        val = heat[i, j]
        if np.isfinite(val):
            col = "white" if abs(val) > 0.07 else "black"
            ax.text(j, i, f"{val:+.3f}", ha="center", va="center",
                    fontsize=8, color=col)

plt.colorbar(im, ax=ax, label="ΔROC-AUC_D vs vol_now baseline")

plt.suptitle(
    f"Stress classification window sensitivity: "
    f"IT W ∈ {{{', '.join(str(w) for w in WINDOWS_TO_TEST)}}} bars\n"
    f"TDA features unchanged (W=192). Holdout {SPLIT_DATE} →",
    fontsize=12, y=1.01
)
plt.tight_layout()
plt.savefig("fig_stress_window_sensitivity.png", bbox_inches="tight", dpi=150)
plt.savefig("fig_stress_window_sensitivity.pdf", bbox_inches="tight", dpi=150)
plt.show()
print("Saved: fig_stress_window_sensitivity.png / .pdf")

stress_window_sensitivity_results = df_stress_window.copy()
print(f"\nstress_window_sensitivity_results saved ({len(df_stress_window)} rows)")

In [186]:
print("\n" + "=" * 70)
print("WINDOW SENSITIVITY — STRESS EVENT CLASSIFICATION")
print(f"Windows: {WINDOWS_TO_TEST} bars")
print("=" * 70)

if "window_it_features" not in dir():
    raise ValueError(
        "window_it_features not found. "
        "Run volatility window sensitivity script first."
    )

if "feat_analysis" in dir():
    avg_perm_src = feat_analysis["perm"]
else:
    avg_perm_src = avg_perm

perm_series = pd.Series(avg_perm_src).sort_values()

all_stress_results = []

for w in WINDOWS_TO_TEST:
    w_label = f"W={w}"
    print(f"\n{'='*60}")
    print(f"  Window: {w} bars (~{w/48:.1f} days)")
    print(f"{'='*60}")

    if w not in window_it_features:
        print(f"  WARNING: W={w} not found in window_it_features, skipping.")
        continue

    X_it_w, best_it_w = window_it_features[w]
    X_it_w = X_it_w.copy()

    print(f"  IT pool ({len(best_it_w)}): {best_it_w}")

    df_s = run_stress_for_window(X_it_w, best_it_w, w_label)
    all_stress_results.append(df_s)

    n = len(df_s)
    if n > 0:
        print(f"\n  Results ({n} tasks):")
        for q in QUANTILES:
            sub = df_s[df_s.q == q]
            if sub.empty: continue
            print(f"    Q{q:.0%}: "
                  f"ΔROC_B={sub['droc_B'].mean():>+.4f}  "
                  f"ΔROC_C={sub['droc_C'].mean():>+.4f}  "
                  f"ΔROC_D={sub['droc_D'].mean():>+.4f}  "
                  f"D>A: {(sub['droc_D']>0.01).sum()}/{len(sub)}")
    else:
        print("  WARNING: no rows collected.")

df_stress_window = pd.concat(all_stress_results, ignore_index=True)
print(f"\nTotal rows: {len(df_stress_window)}")


WINDOW SENSITIVITY — STRESS EVENT CLASSIFICATION
Windows: [96, 144, 192, 384] bars

  Window: 96 bars (~2.0 days)
  IT pool (5): ['kl_BTC', 'coinfo4_std', 'kl_ADA', 'coinfo3_std', 'pairwise_mi_mean']

  Results (80 tasks):
    Q60%: ΔROC_B=+0.0112  ΔROC_C=+0.0268  ΔROC_D=+0.0344  D>A: 18/23
    Q70%: ΔROC_B=+0.0176  ΔROC_C=+0.0148  ΔROC_D=+0.0280  D>A: 17/22
    Q80%: ΔROC_B=+0.0411  ΔROC_C=+0.0229  ΔROC_D=+0.0459  D>A: 15/21
    Q90%: ΔROC_B=+0.0371  ΔROC_C=-0.0058  ΔROC_D=+0.0120  D>A: 5/14

  Window: 144 bars (~3.0 days)
  IT pool (5): ['kl_BTC', 'coinfo4_std', 'kl_ADA', 'coinfo3_std', 'pairwise_mi_mean']

  Results (80 tasks):
    Q60%: ΔROC_B=+0.0074  ΔROC_C=+0.0284  ΔROC_D=+0.0302  D>A: 18/23
    Q70%: ΔROC_B=+0.0132  ΔROC_C=+0.0176  ΔROC_D=+0.0249  D>A: 18/22
    Q80%: ΔROC_B=+0.0324  ΔROC_C=+0.0157  ΔROC_D=+0.0348  D>A: 14/21
    Q90%: ΔROC_B=+0.0347  ΔROC_C=+0.0062  ΔROC_D=+0.0251  D>A: 6/14

  Window: 192 bars (~4.0 days)
  IT pool (5): ['kl_BTC', 'coinfo4_std', 'kl_ADA', 'c

In [191]:
df_sw = stress_window_sensitivity_results.copy()
WINDOWS_TO_TEST = sorted(
    df_sw["window"].str.replace("W=","").astype(int).unique().tolist()
)
QUANTILES_list = sorted(df_sw["q"].unique().tolist())
ASSETS_order   = ["BTC","ETH","SOL","BNB","XRP","DOGE","ADA","AVAX"]
HORIZONS_order = ["1d","2d","4d"]

for w in WINDOWS_TO_TEST:
    wl  = f"W={w}"
    sub_w = df_sw[df_sw.window == wl]
    n   = len(sub_w)

    print("\n" + "=" * 100)
    print(f"WINDOW {wl}  (~{w/48:.1f} days) — {n} tasks")
    print("A = vol_now | B = +IT | C = +TDA | D = +IT+TDA")
    print("=" * 100)

    print(f"\n  By threshold (mean over assets and horizons):")
    print(f"  {'Q':>5} {'pos%':>5}"
          f" {'ROC_A':>7} {'ROC_B':>7} {'ROC_C':>7} {'ROC_D':>7}"
          f" {'ΔROC_B':>8} {'ΔROC_C':>8} {'ΔROC_D':>8}"
          f" {'D>A':>6} {'ΔPR_D':>8}")
    print("  " + "-" * 85)

    for q in QUANTILES_list:
        sub = sub_w[sub_w.q == q]
        if sub.empty: continue
        beats = (sub["droc_D"] > 0.01).sum()
        print(f"  Q{q:.0%}"
              f" {sub['pos_te'].mean():>4.0%}"
              f" {sub['roc_A'].mean():>7.4f}"
              f" {sub['roc_B'].mean():>7.4f}"
              f" {sub['roc_C'].mean():>7.4f}"
              f" {sub['roc_D'].mean():>7.4f}"
              f" {sub['droc_B'].mean():>+8.4f}"
              f" {sub['droc_C'].mean():>+8.4f}"
              f" {sub['droc_D'].mean():>+8.4f}"
              f" {beats:>4}/{len(sub)}"
              f" {sub['dpr_D'].mean():>+8.4f}")

    print(f"\n  By asset at Q80 (mean over horizons):")
    print(f"  {'asset':<6} {'pos%':>5}"
          f" {'ROC_A':>7} {'ROC_B':>7} {'ROC_C':>7} {'ROC_D':>7}"
          f" {'ΔROC_B':>8} {'ΔROC_C':>8} {'ΔROC_D':>8}"
          f" {'ΔPR_D':>8}  best")
    print("  " + "-" * 90)

    sub80 = sub_w[sub_w.q == 0.80]
    for asset in ASSETS_order:
        g = sub80[sub80.asset == asset]
        if g.empty: continue
        vals = {"A": g["roc_A"].mean(), "B": g["roc_B"].mean(),
                "C": g["roc_C"].mean(), "D": g["roc_D"].mean()}
        best = max(vals, key=vals.get)
        print(f"  {asset:<6}"
              f" {g['pos_te'].mean():>4.0%}"
              f" {g['roc_A'].mean():>7.4f}"
              f" {g['roc_B'].mean():>7.4f}"
              f" {g['roc_C'].mean():>7.4f}"
              f" {g['roc_D'].mean():>7.4f}"
              f" {g['droc_B'].mean():>+8.4f}"
              f" {g['droc_C'].mean():>+8.4f}"
              f" {g['droc_D'].mean():>+8.4f}"
              f" {g['dpr_D'].mean():>+8.4f}  -> {best}")

    print(f"\n  By horizon at Q80 (mean over assets):")
    print(f"  {'h':<4}"
          f" {'ROC_A':>7} {'ROC_B':>7} {'ROC_C':>7} {'ROC_D':>7}"
          f" {'ΔROC_B':>8} {'ΔROC_C':>8} {'ΔROC_D':>8}")
    print("  " + "-" * 68)

    for h_label in HORIZONS_order:
        g = sub80[sub80.horizon == h_label]
        if g.empty: continue
        print(f"  {h_label:<4}"
              f" {g['roc_A'].mean():>7.4f}"
              f" {g['roc_B'].mean():>7.4f}"
              f" {g['roc_C'].mean():>7.4f}"
              f" {g['roc_D'].mean():>7.4f}"
              f" {g['droc_B'].mean():>+8.4f}"
              f" {g['droc_C'].mean():>+8.4f}"
              f" {g['droc_D'].mean():>+8.4f}")

    print(f"\n  Overall ({n} tasks):"
          f"  ΔROC_B={sub_w['droc_B'].mean():>+.4f}"
          f"  ΔROC_C={sub_w['droc_C'].mean():>+.4f}"
          f"  ΔROC_D={sub_w['droc_D'].mean():>+.4f}"
          f"  ΔPR_D={sub_w['dpr_D'].mean():>+.4f}"
          f"  D>A: {(sub_w['droc_D']>0.01).sum()}/{n}")

print("\n" + "=" * 80)
print("CROSS-WINDOW SUMMARY")
print("=" * 80)

print(f"\n  {'Window':<10} {'days':>5}"
      f" {'ΔROC_B':>8} {'ΔROC_C':>8} {'ΔROC_D':>8}"
      f" {'ΔPR_B':>8} {'ΔPR_C':>8} {'ΔPR_D':>8}"
      f" {'D>A':>8}")
print("  " + "-" * 78)

for w in WINDOWS_TO_TEST:
    wl  = f"W={w}"
    sub = df_sw[df_sw.window == wl]
    if sub.empty: continue
    n = len(sub)
    beats = (sub["droc_D"] > 0.01).sum()
    print(f"  {wl:<10} {w/48:>5.1f}"
          f" {sub['droc_B'].mean():>+8.4f}"
          f" {sub['droc_C'].mean():>+8.4f}"
          f" {sub['droc_D'].mean():>+8.4f}"
          f" {sub['dpr_B'].mean():>+8.4f}"
          f" {sub['dpr_C'].mean():>+8.4f}"
          f" {sub['dpr_D'].mean():>+8.4f}"
          f" {beats:>5}/{n}")

print(f"\n  Asset-level ΔROC_D at Q80:")
print(f"  {'Asset':<6}", end="")
for w in WINDOWS_TO_TEST:
    print(f"  {'W='+str(w):>10}", end="")
print()
print("  " + "-" * (6 + 12*len(WINDOWS_TO_TEST)))

for asset in ASSETS_order:
    print(f"  {asset:<6}", end="")
    for w in WINDOWS_TO_TEST:
        wl  = f"W={w}"
        sub = df_sw[(df_sw.window==wl) &
                    (df_sw.asset==asset) &
                    (df_sw.q==0.80)]
        val = sub["droc_D"].mean() if not sub.empty else float("nan")
        if val == val:
            print(f"  {val:>+10.4f}", end="")
        else:
            print(f"  {'N/A':>10}", end="")
    print()

print(f"\n  TDA invariance check (ΔROC_C fixed W=192):")
print(f"  {'Window':<10} {'ΔROC_C':>8} {'ΔPR_C':>8}")
for w in WINDOWS_TO_TEST:
    wl  = f"W={w}"
    sub = df_sw[df_sw.window == wl]
    if sub.empty: continue
    print(f"  {wl:<10} {sub['droc_C'].mean():>+8.4f}"
          f" {sub['dpr_C'].mean():>+8.4f}")


WINDOW W=96  (~2.0 days) — 80 tasks
A = vol_now | B = +IT | C = +TDA | D = +IT+TDA

  By threshold (mean over assets and horizons):
      Q  pos%   ROC_A   ROC_B   ROC_C   ROC_D   ΔROC_B   ΔROC_C   ΔROC_D    D>A    ΔPR_D
  -------------------------------------------------------------------------------------
  Q60%  46%  0.6840  0.6952  0.7108  0.7183  +0.0112  +0.0268  +0.0344   18/23  +0.0470
  Q70%  32%  0.6832  0.7008  0.6980  0.7111  +0.0176  +0.0148  +0.0280   17/22  +0.0677
  Q80%  17%  0.6608  0.7019  0.6837  0.7067  +0.0411  +0.0229  +0.0459   15/21  +0.1080
  Q90%   5%  0.6250  0.6622  0.6193  0.6370  +0.0371  -0.0058  +0.0120    5/14  +0.0015

  By asset at Q80 (mean over horizons):
  asset   pos%   ROC_A   ROC_B   ROC_C   ROC_D   ΔROC_B   ΔROC_C   ΔROC_D    ΔPR_D  best
  ------------------------------------------------------------------------------------------
  BTC      1%  0.3506  0.2805  0.3121  0.2711  -0.0702  -0.0385  -0.0796  -0.0006  -> A
  ETH     28%  0.5895  0.60

In [192]:
df_sw = stress_window_sensitivity_results.copy()
WINDOWS_TO_TEST = sorted(
    df_sw["window"].str.replace("W=","").astype(int).unique().tolist()
)
QUANTILES_list = sorted(df_sw["q"].unique().tolist())
ASSETS_order   = ["BTC","ETH","SOL","BNB","XRP","DOGE","ADA","AVAX"]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
w_labels = [f"W={w}" for w in WINDOWS_TO_TEST]
w_days   = [f"{w/48:.1f}d" for w in WINDOWS_TO_TEST]
colors   = {"B": "#2196F3", "C": "#FF9800", "D": "#4CAF50"}

ax = axes[0]
for cfg in ["B", "C", "D"]:
    vals = [df_sw[df_sw.window == wl][f"droc_{cfg}"].mean()
            for wl in w_labels]
    ax.plot(w_days, vals, "o-", color=colors[cfg],
            lw=2, ms=8, label=f"Config {cfg}")
ax.axhline(0, color="black", lw=0.8, ls=":")
ax.set_xlabel("IT window size")
ax.set_ylabel("Mean ΔROC-AUC vs vol\\_now baseline")
ax.set_title("Mean ΔROC by IT window\n(all assets, horizons, thresholds)",
             fontsize=10)
ax.legend(fontsize=9)
ax.grid(alpha=0.25)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax = axes[1]
q_labels  = [f"Q{q:.0%}" for q in QUANTILES_list]
x         = np.arange(len(QUANTILES_list))
w_bar     = 0.2
bar_colors = ["#1565C0", "#42A5F5", "#4CAF50", "#FF9800"]

for wi, w in enumerate(WINDOWS_TO_TEST):
    wl   = f"W={w}"
    vals = [df_sw[(df_sw.window == wl) & (df_sw.q == q)]["droc_D"].mean()
            for q in QUANTILES_list]
    ax.bar(x + (wi - 1.5) * w_bar, vals, w_bar * 0.9,
           label=f"W={w} (~{w/48:.0f}d)",
           color=bar_colors[wi], alpha=0.85)

ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels(q_labels)
ax.set_xlabel("Stress threshold")
ax.set_ylabel("ΔROC-AUC (D vs baseline)")
ax.set_title("ΔROC\\_D by threshold × window\n"
             "(mean over assets and horizons)", fontsize=10)
ax.legend(fontsize=8)
ax.grid(alpha=0.2, axis="y")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax = axes[2]
heat = np.full((len(ASSETS_order), len(WINDOWS_TO_TEST)), np.nan)
for j, w in enumerate(WINDOWS_TO_TEST):
    wl = f"W={w}"
    for i, asset in enumerate(ASSETS_order):
        sub = df_sw[(df_sw.window == wl) &
                    (df_sw.asset  == asset) &
                    (df_sw.q      == 0.80)]
        if not sub.empty:
            heat[i, j] = sub["droc_D"].mean()

im = ax.imshow(heat, cmap="RdYlGn", vmin=-0.12, vmax=0.15, aspect="auto")
ax.set_xticks(range(len(WINDOWS_TO_TEST)))
ax.set_xticklabels(w_days)
ax.set_yticks(range(len(ASSETS_order)))
ax.set_yticklabels(ASSETS_order)
ax.set_xlabel("IT window size")
ax.set_title("ΔROC\\_D at Q80 by asset × window\n"
             "(green = better than baseline)", fontsize=10)

for i in range(len(ASSETS_order)):
    for j in range(len(WINDOWS_TO_TEST)):
        val = heat[i, j]
        if np.isfinite(val):
            col  = "white" if abs(val) > 0.07 else "black"
            ax.text(j, i, f"{val:+.3f}", ha="center", va="center",
                    fontsize=8, color=col)

plt.colorbar(im, ax=ax,
             label="ΔROC-AUC\\_D vs vol\\_now baseline")

plt.suptitle(
    f"Stress classification window sensitivity: "
    f"IT W ∈ {{{', '.join(str(w) for w in WINDOWS_TO_TEST)}}} bars\n"
    f"TDA features unchanged (W=192). Holdout {SPLIT_DATE} →",
    fontsize=12, y=1.01
)
plt.tight_layout()
plt.savefig("fig_stress_window_sensitivity.png", bbox_inches="tight", dpi=150)
plt.savefig("fig_stress_window_sensitivity.pdf", bbox_inches="tight", dpi=150)
plt.show()
print("Saved: fig_stress_window_sensitivity.png / .pdf")

Saved: fig_stress_window_sensitivity.png / .pdf


In [188]:
for name in ["stress_window_sensitivity_results", "df_stress_window", 
             "all_stress_results", "window_it_features"]:
    print(f"{name}: {'✓' if name in dir() else '✗'}")

stress_window_sensitivity_results: ✗
df_stress_window: ✓
all_stress_results: ✓
window_it_features: ✓


In [189]:
stress_window_sensitivity_results = df_stress_window.copy()
print(f"stress_window_sensitivity_results saved ({len(df_stress_window)} rows)")

stress_window_sensitivity_results saved (320 rows)


In [190]:
stress_window_sensitivity_results

,window,asset,horizon,h,q,pos_tr,pos_te,roc_A,roc_B,roc_C,...,pr_C,pr_D,droc_B,droc_C,droc_D,dpr_B,dpr_C,dpr_D,n_train,n_test
0,W=96,BTC,1d,48,0.6,0.448,0.119,0.6135,0.6004,0.6329,...,0.1535,0.1530,-0.0131,0.0194,0.0116,-0.0019,0.0106,0.0102,43478,2977
1,W=96,BTC,1d,48,0.7,0.333,0.054,0.5827,0.5555,0.6011,...,0.0642,0.0613,-0.0272,0.0184,-0.0005,-0.0034,0.0028,-0.0000,43478,2977
2,W=96,BTC,1d,48,0.8,0.227,0.008,0.3506,0.2805,0.3121,...,0.0059,0.0057,-0.0702,-0.0385,-0.0796,-0.0005,-0.0004,-0.0006,43478,2977
3,W=96,BTC,2d,96,0.6,0.475,0.058,0.4081,0.4011,0.4290,...,0.0509,0.0460,-0.0070,0.0209,0.0113,-0.0043,0.0016,-0.0033,43478,2977
4,W=96,ETH,1d,48,0.6,0.517,0.640,0.6574,0.6465,0.6660,...,0.7274,0.7385,-0.0110,0.0086,0.0079,-0.0079,0.0017,0.0128,43478,2977
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,W=384,AVAX,2d,96,0.7,0.346,0.184,0.6130,0.6536,0.5722,...,0.2273,0.3322,0.0407,-0.0408,0.0153,0.0398,-0.0187,0.0861,42903,2977
316,W=384,AVAX,2d,96,0.8,0.211,0.075,0.6472,0.7220,0.6613,...,0.1507,0.1602,0.0748,0.0140,0.0340,0.0449,-0.0129,-0.0033,42903,2977
317,W=384,AVAX,4d,192,0.6,0.517,0.371,0.6850,0.7297,0.7297,...,0.6020,0.6146,0.0448,0.0447,0.0634,0.0655,0.0900,0.1027,42903,2977
318,W=384,AVAX,4d,192,0.7,0.352,0.245,0.6593,0.7009,0.6560,...,0.3542,0.4122,0.0416,-0.0032,0.0409,0.0372,0.0323,0.0903,42903,2977


In [119]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

old_rows = [
    ("ADA","1d",  0.2670,  0.2523,  0.3067,  0.2982),
    ("ADA","2d",  0.2413,  0.2165,  0.2940,  0.2668),
    ("ADA","4d",  0.1948,  0.1208,  0.2548,  0.1489),
    ("ADA","6d",  0.1330,  0.0360,  0.1957,  0.0616),
    ("ADA","7d",  0.1210,  0.0048,  0.1891,  0.0320),
    ("ADA","14d", -0.0990, -0.5945,  0.0657, -0.4020),

    ("AVAX","1d",  0.1260,  0.0734,  0.1560,  0.1200),
    ("AVAX","2d",  0.1056, -0.0078,  0.1448,  0.0649),
    ("AVAX","4d",  0.1883, -0.0036,  0.2274,  0.0537),
    ("AVAX","6d",  0.0701, -0.0685,  0.1045, -0.0276),
    ("AVAX","7d",  0.0419, -0.0664,  0.0856, -0.0148),
    ("AVAX","14d", -0.5613, -1.5173, -0.4966, -1.2211),

    ("BNB","1d",  0.3898,  0.3507,  0.4061,  0.3666),
    ("BNB","2d",  0.3814,  0.3232,  0.3986,  0.3422),
    ("BNB","4d",  0.3750,  0.2912,  0.3801,  0.2993),
    ("BNB","6d",  0.3524,  0.2285,  0.3478,  0.2348),
    ("BNB","7d",  0.3428,  0.1913,  0.3343,  0.2024),
    ("BNB","14d", 0.1255, -0.8769,  0.1369, -0.8856),

    ("BTC","1d",  0.3555,  0.3274,  0.3543,  0.3252),
    ("BTC","2d",  0.3453,  0.3056,  0.3431,  0.3086),
    ("BTC","4d",  0.3952,  0.3477,  0.3926,  0.3694),
    ("BTC","6d",  0.4248,  0.3915,  0.4349,  0.4361),
    ("BTC","7d",  0.4664,  0.4462,  0.4815,  0.4962),
    ("BTC","14d", 0.2806,  0.2824,  0.3259,  0.4183),

    ("DOGE","1d",  0.1887,  0.1982,  0.2268,  0.2713),
    ("DOGE","2d",  0.2228,  0.2388,  0.2771,  0.3363),
    ("DOGE","4d",  0.2263,  0.2257,  0.2890,  0.3211),
    ("DOGE","6d",  0.1637,  0.1805,  0.2222,  0.2612),
    ("DOGE","7d",  0.1713,  0.2170,  0.2361,  0.3022),
    ("DOGE","14d", -0.0064, 0.0727,  0.1403,  0.2128),

    ("ETH","1d",  0.1864,  0.2211,  0.2005,  0.2662),
    ("ETH","2d",  0.2248,  0.2558,  0.2311,  0.3096),
    ("ETH","4d",  0.2535,  0.2352,  0.2300,  0.2540),
    ("ETH","6d",  0.2426,  0.2630,  0.2084,  0.2561),
    ("ETH","7d",  0.2724,  0.3207,  0.2348,  0.3097),
    ("ETH","14d", 0.1574,  0.2398,  0.1866,  0.2865),

    ("SOL","1d",  0.2840,  0.2522,  0.2735,  0.2482),
    ("SOL","2d",  0.2923,  0.2386,  0.2725,  0.2349),
    ("SOL","4d",  0.3340,  0.2751,  0.2914,  0.2573),
    ("SOL","6d",  0.3272,  0.2905,  0.2512,  0.2543),
    ("SOL","7d",  0.3448,  0.3156,  0.2556,  0.2813),
    ("SOL","14d", 0.3746,  0.1921,  0.1266,  0.1734),

    ("XRP","1d",  0.1001,  0.0789,  0.1249,  0.1148),
    ("XRP","2d",  0.0358,  0.0298,  0.0795,  0.0777),
    ("XRP","4d", -0.0082,  0.0007,  0.0564,  0.0185),
    ("XRP","6d", -0.0475, -0.0402,  0.0159, -0.0432),
    ("XRP","7d", -0.0936, -0.0814, -0.0269, -0.0813),
    ("XRP","14d",-0.2833, -0.4040, -0.1395, -0.2821),
]

new_rows = [
    ("ADA","1d",  0.2670,  0.2627,  0.3490,  0.3477),
    ("ADA","2d",  0.2413,  0.2377,  0.3515,  0.3568),
    ("ADA","4d",  0.1948,  0.1633,  0.3151,  0.3163),
    ("ADA","6d",  0.1330,  0.0759,  0.2573,  0.2405),
    ("ADA","7d",  0.1210,  0.0510,  0.2543,  0.2194),
    ("ADA","14d", -0.0990, -0.3930,  0.1013,  0.0318),

    ("AVAX","1d",  0.1260,  0.1032,  0.1651,  0.1389),
    ("AVAX","2d",  0.1056,  0.0853,  0.1696,  0.1365),
    ("AVAX","4d",  0.1883,  0.1486,  0.2247,  0.1632),
    ("AVAX","6d",  0.0701,  0.0207,  0.1226,  0.0730),
    ("AVAX","7d",  0.0419, -0.0111,  0.0936,  0.0388),
    ("AVAX","14d", -0.5613, -1.2264, -0.5189, -0.7516),

    ("BNB","1d",  0.3898,  0.3981,  0.3883,  0.4130),
    ("BNB","2d",  0.3814,  0.4046,  0.3624,  0.4173),
    ("BNB","4d",  0.3750,  0.3907,  0.3010,  0.3643),
    ("BNB","6d",  0.3524,  0.3469,  0.2511,  0.3121),
    ("BNB","7d",  0.3428,  0.3181,  0.2339,  0.2869),
    ("BNB","14d", 0.1255, -0.5463, -0.4452, -0.6457),

    ("BTC","1d",  0.3555,  0.3069,  0.2920,  0.3037),
    ("BTC","2d",  0.3453,  0.2945,  0.2805,  0.3172),
    ("BTC","4d",  0.3952,  0.3511,  0.3619,  0.3789),
    ("BTC","6d",  0.4248,  0.3936,  0.4057,  0.4196),
    ("BTC","7d",  0.4664,  0.4395,  0.4509,  0.4487),
    ("BTC","14d", 0.2806,  0.2897,  0.4717,  0.4211),

    ("DOGE","1d",  0.1887,  0.2262,  0.3054,  0.3055),
    ("DOGE","2d",  0.2228,  0.2926,  0.3905,  0.4061),
    ("DOGE","4d",  0.2263,  0.2990,  0.4261,  0.4243),
    ("DOGE","6d",  0.1637,  0.2374,  0.3269,  0.3319),
    ("DOGE","7d",  0.1713,  0.2516,  0.3026,  0.3069),
    ("DOGE","14d", -0.0064, 0.0435,  0.0546,  0.0727),

    ("ETH","1d",  0.1864,  0.2464,  0.3114,  0.3227),
    ("ETH","2d",  0.2248,  0.3170,  0.4025,  0.4250),
    ("ETH","4d",  0.2535,  0.3384,  0.4406,  0.4517),
    ("ETH","6d",  0.2426,  0.3368,  0.3996,  0.4333),
    ("ETH","7d",  0.2724,  0.3736,  0.4186,  0.4630),
    ("ETH","14d", 0.1574,  0.4060,  0.3915,  0.5955),

    ("SOL","1d",  0.2840,  0.2764,  0.2600,  0.2593),
    ("SOL","2d",  0.2923,  0.2983,  0.2568,  0.2620),
    ("SOL","4d",  0.3340,  0.3396,  0.2912,  0.2737),
    ("SOL","6d",  0.3272,  0.3499,  0.3206,  0.3244),
    ("SOL","7d",  0.3448,  0.3775,  0.3474,  0.3638),
    ("SOL","14d", 0.3746,  0.4071,  0.4528,  0.4325),

    ("TRX","1d",  0.2400,  0.2091,  0.0315,  0.0677),
    ("TRX","2d",  0.1761,  0.1441, -0.0887, -0.0579),
    ("TRX","4d",  0.1571,  0.1643, -0.2356, -0.1576),
    ("TRX","6d",  0.1586,  0.1605, -0.3715, -0.2386),
    ("TRX","7d",  0.1443,  0.1259, -0.4563, -0.3022),
    ("TRX","14d", 0.0296, -0.2716, -0.7550, -0.8481),

    ("XRP","1d",  0.1001,  0.0980,  0.1841,  0.1536),
    ("XRP","2d",  0.0358,  0.0226,  0.1645,  0.1162),
    ("XRP","4d", -0.0082, -0.0461,  0.1255,  0.0603),
    ("XRP","6d", -0.0475, -0.0936,  0.0753, -0.0070),
    ("XRP","7d", -0.0936, -0.1472,  0.0205, -0.0741),
    ("XRP","14d",-0.2833, -0.5143, -0.3071, -0.5270),
]

cols = ["asset", "horizon", "r2_A", "r2_B", "r2_C", "r2_D"]
old_df = pd.DataFrame(old_rows, columns=cols)
new_df = pd.DataFrame(new_rows, columns=cols)

def add_deltas(df):
    df = df.copy()
    df["dB"] = df["r2_B"] - df["r2_A"]
    df["dC"] = df["r2_C"] - df["r2_A"]
    df["dD"] = df["r2_D"] - df["r2_A"]
    return df

old_df = add_deltas(old_df)
new_df = add_deltas(new_df)

common = old_df.merge(
    new_df,
    on=["asset", "horizon"],
    suffixes=("_old", "_new")
)

def print_summary(label, df):
    print(f"\n{label}")
    for cfg in ["B", "C", "D"]:
        col = f"d{cfg}"
        mean_val = df[col].mean()
        pos = (df[col] > 0).sum()
        total = len(df)
        print(f"  {cfg}: mean ΔR² = {mean_val:+.4f}   positive = {pos}/{total}")

def print_summary_merged(label, df, suffix):
    print(f"\n{label}")
    for cfg in ["B", "C", "D"]:
        col = f"d{cfg}{suffix}"
        mean_val = df[col].mean()
        pos = (df[col] > 0).sum()
        total = len(df)
        print(f"  {cfg}: mean ΔR² = {mean_val:+.4f}   positive = {pos}/{total}")

print("=" * 72)
print("SUMMARY: OLD VS NEW VOLATILITY RESULTS")
print("=" * 72)

print_summary("OLD (reported)", old_df)
print_summary("NEW (reported, includes TRX)", new_df)

print_summary_merged("OLD on common 48 tasks", common, "_old")
print_summary_merged("NEW on common 48 tasks", common, "_new")

print("\nFair comparison on common 48 tasks:")
for cfg in ["B", "C", "D"]:
    old_mean = common[f"d{cfg}_old"].mean()
    new_mean = common[f"d{cfg}_new"].mean()
    print(f"  {cfg}: {old_mean:+.4f}  ->  {new_mean:+.4f}   change = {new_mean - old_mean:+.4f}")

plt.rcParams["figure.dpi"] = 140

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

ax = axes[0, 0]
cfgs = ["B", "C", "D"]
x = np.arange(len(cfgs))
w = 0.34

old_means = [common[f"d{c}_old"].mean() for c in cfgs]
new_means = [common[f"d{c}_new"].mean() for c in cfgs]

ax.bar(x - w/2, old_means, width=w, label="Old results", alpha=0.85)
ax.bar(x + w/2, new_means, width=w, label="New results", alpha=0.85)
ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels(["+IT", "+TDA", "+IT+TDA"])
ax.set_ylabel("Mean ΔR² over baseline")
ax.set_title("Mean ΔR² by configuration\n(common 48 tasks only)")
ax.legend()
ax.grid(alpha=0.2, axis="y")
for i, v in enumerate(old_means):
    ax.text(i - w/2, v + (0.003 if v >= 0 else -0.012), f"{v:+.3f}", ha="center", fontsize=9)
for i, v in enumerate(new_means):
    ax.text(i + w/2, v + (0.003 if v >= 0 else -0.012), f"{v:+.3f}", ha="center", fontsize=9)

ax = axes[0, 1]
asset_order = ["BTC", "ETH", "SOL", "BNB", "XRP", "DOGE", "ADA", "AVAX"]
asset_cmp = common.groupby("asset")[["dD_old", "dD_new"]].mean().reindex(asset_order)

x = np.arange(len(asset_order))
ax.bar(x - w/2, asset_cmp["dD_old"], width=w, label="Old ΔR²_D", alpha=0.85)
ax.bar(x + w/2, asset_cmp["dD_new"], width=w, label="New ΔR²_D", alpha=0.85)
ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels(asset_order)
ax.set_ylabel("Mean ΔR²_D")
ax.set_title("Mean ΔR² of combined model (D) by asset")
ax.legend()
ax.grid(alpha=0.2, axis="y")

ax = axes[1, 0]
xvals = common["dD_old"].values
yvals = common["dD_new"].values

ax.scatter(xvals, yvals, alpha=0.8)
mn = min(xvals.min(), yvals.min()) - 0.02
mx = max(xvals.max(), yvals.max()) + 0.02
ax.plot([mn, mx], [mn, mx], "--", color="gray", lw=1.0)
ax.axhline(0, color="black", lw=0.7)
ax.axvline(0, color="black", lw=0.7)
ax.set_xlim(mn, mx)
ax.set_ylim(mn, mx)
ax.set_xlabel("Old ΔR²_D")
ax.set_ylabel("New ΔR²_D")
ax.set_title("Task-level comparison of combined model (D)")
ax.grid(alpha=0.2)

common["delta_gain_D"] = common["dD_new"] - common["dD_old"]
top_gain = common.nlargest(5, "delta_gain_D")
for _, row in top_gain.iterrows():
    ax.annotate(
        f"{row['asset']}-{row['horizon']}",
        (row["dD_old"], row["dD_new"]),
        fontsize=8,
        xytext=(4, 4),
        textcoords="offset points"
    )

ax = axes[1, 1]
h_order = ["1d", "2d", "4d", "6d", "7d", "14d"]

heat = (
    common.assign(delta_gain_D=common["dD_new"] - common["dD_old"])
    .pivot(index="asset", columns="horizon", values="delta_gain_D")
    .reindex(index=asset_order, columns=h_order)
)

im = ax.imshow(heat.values, aspect="auto", cmap="RdYlGn", vmin=-0.20, vmax=0.20)
ax.set_xticks(np.arange(len(h_order)))
ax.set_xticklabels(h_order)
ax.set_yticks(np.arange(len(asset_order)))
ax.set_yticklabels(asset_order)
ax.set_title("Improvement in combined model D\n(new ΔR²_D minus old ΔR²_D)")

for i in range(len(asset_order)):
    for j in range(len(h_order)):
        val = heat.iloc[i, j]
        if pd.notna(val):
            txt_color = "white" if abs(val) > 0.10 else "black"
            ax.text(j, i, f"{val:+.2f}", ha="center", va="center", fontsize=8, color=txt_color)

cbar = plt.colorbar(im, ax=ax)
cbar.set_label("Δ improvement")

plt.suptitle("Old vs new volatility results", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("compare_old_vs_new_vol_results.png", bbox_inches="tight")
plt.savefig("compare_old_vs_new_vol_results.pdf", bbox_inches="tight")
plt.show()

print("\nSaved: compare_old_vs_new_vol_results.png / .pdf")

SUMMARY: OLD VS NEW VOLATILITY RESULTS

OLD (reported)
  B: mean ΔR² = -0.0866   positive = 14/48
  C: mean ΔR² = +0.0229   positive = 34/48
  D: mean ΔR² = -0.0436   positive = 23/48

NEW (reported, includes TRX)
  B: mean ΔR² = -0.0326   positive = 23/54
  C: mean ΔR² = -0.0038   positive = 32/54
  D: mean ΔR² = -0.0101   positive = 31/54

OLD on common 48 tasks
  B: mean ΔR² = -0.0866   positive = 14/48
  C: mean ΔR² = +0.0229   positive = 34/48
  D: mean ΔR² = -0.0436   positive = 23/48

NEW on common 48 tasks
  B: mean ΔR² = -0.0289   positive = 21/48
  C: mean ΔR² = +0.0536   positive = 32/48
  D: mean ΔR² = +0.0395   positive = 31/48

Fair comparison on common 48 tasks:
  B: -0.0866  ->  -0.0289   change = +0.0577
  C: +0.0229  ->  +0.0536   change = +0.0307
  D: -0.0436  ->  +0.0395   change = +0.0831

Saved: compare_old_vs_new_vol_results.png / .pdf


In [174]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.feature_selection import mutual_info_classif

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

SPLIT_DATE = "2025-07-01"
PURGE_BARS = 96

QUANTILES = [0.60, 0.70, 0.80, 0.90]
HORIZONS = {
    48:  "1d",
    96:  "2d",
    192: "4d",
}

ASSETS = [a for a in ["BTC","ETH","SOL","BNB","XRP","DOGE","ADA","AVAX"] if a in ret.columns]

VOL_WINDOW = 48
ANN = np.sqrt(365 * 48)

N_IT_FEATS  = 6
N_TDA_FEATS = 5

def _logit(C=0.1):
    return Pipeline([
        ("sc", StandardScaler()),
        ("m", LogisticRegression(
            C=C,
            max_iter=2000,
            class_weight="balanced",
            solver="lbfgs"
        ))
    ])

def safe_roc(y, s):
    try:
        return float(roc_auc_score(y, s))
    except Exception:
        return np.nan

def safe_pr(y, s):
    try:
        return float(average_precision_score(y, s))
    except Exception:
        return np.nan

def lift_at_k(y_true, score, k=0.05):
    n = len(y_true)
    top_n = max(1, int(n * k))
    top_idx = np.argsort(score)[::-1][:top_n]
    base = float(np.mean(y_true))
    return float(np.mean(y_true[top_idx])) / base if base > 1e-8 else np.nan

def fit_predict_logit(Xtr, ytr, Xte):
    try:
        m = _logit()
        m.fit(Xtr, ytr)
        return m.predict_proba(Xte)[:, 1]
    except Exception:
        return np.full(len(Xte), float(np.mean(ytr)))

print("=" * 80)
print("STRESS PREDICTION WITH THE NEW BEST FEATURES")
print("=" * 80)

if "feat_analysis" in dir() and ("X_it_aug" in feat_analysis):
    X_it_all = feat_analysis["X_it_aug"].copy()
elif "X_it_aug" in dir():
    X_it_all = X_it_aug.copy()
elif "X_it_extended" in dir():
    X_it_all = X_it_extended.copy()
elif "X_it_slim" in dir():
    X_it_all = X_it_slim.copy()
else:
    raise ValueError("No IT feature matrix found in memory (X_it_aug / X_it_extended / X_it_slim).")

if "feat_analysis" in dir() and ("X_tda_full" in feat_analysis):
    X_tda_all = feat_analysis["X_tda_full"].copy()
elif "X_tda_full" in dir():
    X_tda_all = X_tda_full.copy()
else:
    print("Building X_tda_all from tda_A and tda_B...")
    tda_frames = [tda_A.add_prefix("A_")]
    for mode in ["I", "II", "C"]:
        d = tda_B[mode]
        df_m = pd.DataFrame({
            f"B{mode}_tp_H0":  d["tp_H0"],
            f"B{mode}_ent_H0": d["ent_H0"],
            f"B{mode}_tp_H1":  d["tp_H1"],
            f"B{mode}_ent_H1": d["ent_H1"],
            f"B{mode}_L1_H0":  d["L1_H0"],
            f"B{mode}_L1_H1":  d["L1_H1"],
        }, index=pd.DatetimeIndex(d["times"]))
        tda_frames.append(df_m)

    X_tda_raw = pd.concat(tda_frames, axis=1).sort_index()
    X_tda_all = X_tda_raw.reindex(ret.index, method="ffill")

if "best_feature_sets" in dir():
    BEST_IT_GLOBAL  = list(best_feature_sets.get("BEST_IT", []))
    BEST_TDA_GLOBAL = list(best_feature_sets.get("BEST_TDA", []))
else:
    BEST_IT_GLOBAL = [
        "kl_BTC",
        "coinfo4_std",
        "kl_ADA",
        "nmi_lag1_mean",
        "coinfo3_std",
        "pairwise_mi_mean",
    ]
    BEST_TDA_GLOBAL = [
        "BI_L1_H0",
        "BC_ent_H1",
        "BII_tp_H1",
        "BII_L1_H1",
        "A_T_H0",
        "A_E_H0",
    ]

BEST_IT_GLOBAL  = [c for c in BEST_IT_GLOBAL if c in X_it_all.columns]
BEST_TDA_GLOBAL = [c for c in BEST_TDA_GLOBAL if c in X_tda_all.columns]

if len(BEST_IT_GLOBAL) == 0:
    raise ValueError("No new best IT features were found in X_it_all.")
if len(BEST_TDA_GLOBAL) == 0:
    raise ValueError("No new best TDA features were found in X_tda_all.")

print("\nGlobal IT candidate pool:")
for c in BEST_IT_GLOBAL:
    print(f"  {c}")

print("\nGlobal TDA candidate pool:")
for c in BEST_TDA_GLOBAL:
    print(f"  {c}")

print(f"\nIT pool size:  {len(BEST_IT_GLOBAL)}")
print(f"TDA pool size: {len(BEST_TDA_GLOBAL)}")

vol_now_rank = {}
for asset in ASSETS:
    vn = ret[asset].rolling(VOL_WINDOW).std() * ANN
    vn_rank = vn.expanding(min_periods=100).rank(pct=True)
    vol_now_rank[asset] = vn_rank

split_ts = pd.Timestamp(SPLIT_DATE)

print("\n" + "=" * 72)
print("STRESS PREDICTION")
print("A = vol_now | B = vol_now + best IT | C = vol_now + best TDA | D = vol_now + best IT + best TDA")
print("Feature selection is adaptive per asset on the training set")
print("=" * 72)

rows = []

for asset in ASSETS:
    r = ret[asset]
    vol_now = r.rolling(VOL_WINDOW).std() * ANN
    vn_rank = vol_now_rank[asset]

    for h, h_label in HORIZONS.items():
        vol_fwd = r.rolling(h).std().shift(-h) * ANN

        for q in QUANTILES:
            vol_q = vol_now.expanding(min_periods=100).quantile(q)
            y_raw = (vol_fwd > vol_q).astype(float).dropna()

            Xi_it = X_it_all[BEST_IT_GLOBAL].copy()
            Xi_td = X_tda_all[BEST_TDA_GLOBAL].copy()

            idx = (
                Xi_it.index
                .intersection(Xi_td.index)
                .intersection(vn_rank.index)
                .intersection(y_raw.index)
            )

            valid = (
                Xi_it.loc[idx].notna().all(1)
                & Xi_td.loc[idx].notna().all(1)
                & vn_rank.loc[idx].notna()
                & y_raw.loc[idx].notna()
            )
            idx = idx[valid]

            if len(idx) < 1000:
                continue

            tr = idx < (split_ts - pd.Timedelta(minutes=30 * PURGE_BARS))
            te = idx >= split_ts

            if tr.sum() < 480 or te.sum() < 100:
                continue

            ytr = y_raw.loc[idx[tr]]
            yte = y_raw.loc[idx[te]]

            pos_tr = float(ytr.mean())
            pos_te = float(yte.mean())

            if pos_tr < 0.03 or pos_tr > 0.97:
                continue
            if yte.sum() < 5:
                continue

            Xvol_tr_df = pd.DataFrame(
                vn_rank.loc[idx[tr]].values.reshape(-1, 1),
                index=idx[tr],
                columns=["vol_rank"]
            )
            Xvol_te_df = pd.DataFrame(
                vn_rank.loc[idx[te]].values.reshape(-1, 1),
                index=idx[te],
                columns=["vol_rank"]
            )

            Xi_tr_all = Xi_it.loc[idx[tr]]
            Xi_te_all = Xi_it.loc[idx[te]]

            Xt_tr_all = Xi_td.loc[idx[tr]]
            Xt_te_all = Xi_td.loc[idx[te]]

            mi_it = mutual_info_classif(Xi_tr_all.values, ytr.values, random_state=42)
            mi_td = mutual_info_classif(Xt_tr_all.values, ytr.values, random_state=42)

            it_cols = (
                pd.Series(mi_it, index=Xi_tr_all.columns)
                .nlargest(min(N_IT_FEATS, Xi_tr_all.shape[1]))
                .index.tolist()
            )
            td_cols = (
                pd.Series(mi_td, index=Xt_tr_all.columns)
                .nlargest(min(N_TDA_FEATS, Xt_tr_all.shape[1]))
                .index.tolist()
            )

            Xit_tr = Xi_tr_all[it_cols]
            Xit_te = Xi_te_all[it_cols]

            Xtd_tr = Xt_tr_all[td_cols]
            Xtd_te = Xt_te_all[td_cols]

            yte_np = yte.values.astype(int)

            sc_A = fit_predict_logit(Xvol_tr_df, ytr, Xvol_te_df)

            sc_B = fit_predict_logit(
                pd.concat([Xvol_tr_df, Xit_tr], axis=1),
                ytr,
                pd.concat([Xvol_te_df, Xit_te], axis=1)
            )

            sc_C = fit_predict_logit(
                pd.concat([Xvol_tr_df, Xtd_tr], axis=1),
                ytr,
                pd.concat([Xvol_te_df, Xtd_te], axis=1)
            )

            sc_D = fit_predict_logit(
                pd.concat([Xvol_tr_df, Xit_tr, Xtd_tr], axis=1),
                ytr,
                pd.concat([Xvol_te_df, Xit_te, Xtd_te], axis=1)
            )

            rows.append({
                "asset": asset,
                "horizon": h_label,
                "h": h,
                "q": q,

                "pos_tr": round(pos_tr, 3),
                "pos_te": round(pos_te, 3),

                "it_cols": ", ".join(it_cols),
                "tda_cols": ", ".join(td_cols),

                "roc_A": round(safe_roc(yte_np, sc_A), 4),
                "roc_B": round(safe_roc(yte_np, sc_B), 4),
                "roc_C": round(safe_roc(yte_np, sc_C), 4),
                "roc_D": round(safe_roc(yte_np, sc_D), 4),

                "pr_A": round(safe_pr(yte_np, sc_A), 4),
                "pr_B": round(safe_pr(yte_np, sc_B), 4),
                "pr_C": round(safe_pr(yte_np, sc_C), 4),
                "pr_D": round(safe_pr(yte_np, sc_D), 4),

                "droc_B": round(safe_roc(yte_np, sc_B) - safe_roc(yte_np, sc_A), 4),
                "droc_C": round(safe_roc(yte_np, sc_C) - safe_roc(yte_np, sc_A), 4),
                "droc_D": round(safe_roc(yte_np, sc_D) - safe_roc(yte_np, sc_A), 4),

                "dpr_B": round(safe_pr(yte_np, sc_B) - safe_pr(yte_np, sc_A), 4),
                "dpr_C": round(safe_pr(yte_np, sc_C) - safe_pr(yte_np, sc_A), 4),
                "dpr_D": round(safe_pr(yte_np, sc_D) - safe_pr(yte_np, sc_A), 4),

                "lift5_A": round(lift_at_k(yte_np, sc_A, 0.05), 3),
                "lift5_D": round(lift_at_k(yte_np, sc_D, 0.05), 3),

                "n_train": int(tr.sum()),
                "n_test": int(te.sum()),
            })

df_new_stress = pd.DataFrame(rows)
print(f"\nTotal rows: {len(df_new_stress)}")

if df_new_stress.empty:
    raise ValueError("The stress result dataframe is empty. Check intersections / NaNs / class balance.")

print("\n" + "=" * 80)
print("ROC-AUC — AGGREGATE BY THRESHOLD (MEAN OVER ASSETS AND HORIZONS)")
print("=" * 80)

print(
    f"\n  {'q':>5} {'pos%':>5}"
    f" {'ROC_A':>7} {'ROC_B':>7} {'ROC_C':>7} {'ROC_D':>7}"
    f" {'ΔROC_B':>8} {'ΔROC_C':>8} {'ΔROC_D':>8}"
    f" {'D>A':>6} {'Lift5_D':>9}"
)
print("  " + "-" * 84)

for q in QUANTILES:
    sub = df_new_stress[df_new_stress.q == q]
    if sub.empty:
        continue
    beats = (sub["droc_D"] > 0.01).sum()

    print(
        f"  Q{q:.0%}"
        f" {sub['pos_te'].mean():>4.0%}"
        f" {sub['roc_A'].mean():>7.4f}"
        f" {sub['roc_B'].mean():>7.4f}"
        f" {sub['roc_C'].mean():>7.4f}"
        f" {sub['roc_D'].mean():>7.4f}"
        f" {sub['droc_B'].mean():>+8.4f}"
        f" {sub['droc_C'].mean():>+8.4f}"
        f" {sub['droc_D'].mean():>+8.4f}"
        f" {beats:>4}/{len(sub)}"
        f" {sub['lift5_D'].mean():>9.3f}"
    )

print("\n" + "=" * 80)
print("BY ASSET — THRESHOLD Q80 | ROC-AUC + PR-AUC")
print("=" * 80)

sub80 = df_new_stress[df_new_stress.q == 0.80]

print(
    f"\n  {'asset':<6} {'pos%':>5}"
    f" {'ROC_A':>7} {'ROC_B':>7} {'ROC_C':>7} {'ROC_D':>7}"
    f" {'ΔROC_D':>8}"
    f" {'PR_A':>7} {'PR_D':>7} {'ΔPR_D':>8}  best"
)
print("  " + "-" * 88)

for asset in ASSETS:
    g = sub80[sub80.asset == asset]
    if g.empty:
        continue

    vals = {
        "A": g["roc_A"].mean(),
        "B": g["roc_B"].mean(),
        "C": g["roc_C"].mean(),
        "D": g["roc_D"].mean(),
    }
    best = max(vals, key=vals.get)

    print(
        f"  {asset:<6}"
        f" {g['pos_te'].mean():>4.0%}"
        f" {g['roc_A'].mean():>7.4f}"
        f" {g['roc_B'].mean():>7.4f}"
        f" {g['roc_C'].mean():>7.4f}"
        f" {g['roc_D'].mean():>7.4f}"
        f" {g['droc_D'].mean():>+8.4f}"
        f" {g['pr_A'].mean():>7.4f}"
        f" {g['pr_D'].mean():>7.4f}"
        f" {g['dpr_D'].mean():>+8.4f}"
        f"  -> {best}"
    )

print("\n" + "=" * 80)
print("TOP 10 BY ΔROC (CONFIGURATION D = vol_now + best IT + best TDA)")
print("=" * 80)

top10 = df_new_stress.nlargest(10, "droc_D")

print(
    f"\n  {'asset':<6} {'h':<4} {'Q':>5} {'pos%':>5}"
    f" {'ROC_A':>7} {'ROC_D':>7} {'ΔROC':>8}"
    f" {'PR_A':>7} {'PR_D':>7} {'ΔPR':>8} {'Lift5':>7}"
)
print("  " + "-" * 76)

for _, row in top10.iterrows():
    print(
        f"  {row['asset']:<6} {row['horizon']:<4} Q{row['q']:.0%}"
        f" {row['pos_te']:>4.0%}"
        f" {row['roc_A']:>7.4f}"
        f" {row['roc_D']:>7.4f}"
        f" {row['droc_D']:>+8.4f}"
        f" {row['pr_A']:>7.4f}"
        f" {row['pr_D']:>7.4f}"
        f" {row['dpr_D']:>+8.4f}"
        f" {row['lift5_D']:>7.3f}"
    )

print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)

n = len(df_new_stress)

for cfg, col_roc, col_pr in [
    ("B: vol_now + best IT",     "droc_B", "dpr_B"),
    ("C: vol_now + best TDA",    "droc_C", "dpr_C"),
    ("D: vol_now + best IT+TDA", "droc_D", "dpr_D"),
]:
    beats_roc = (df_new_stress[col_roc] > 0.01).sum()
    beats_pr  = (df_new_stress[col_pr]  > 0.01).sum()
    print(
        f"  {cfg:<24}: ΔROC={df_new_stress[col_roc].mean():>+.4f}  "
        f"ROC↑ {beats_roc:>2}/{n}  "
        f"ΔPR={df_new_stress[col_pr].mean():>+.4f}  "
        f"PR↑ {beats_pr:>2}/{n}"
    )

print("\nBuilding plots...")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
q_labels = [f"Q{q:.0%}" for q in QUANTILES]
colors = {
    "A": "#9E9E9E",
    "B": "#2196F3",
    "C": "#FF9800",
    "D": "#4CAF50",
}

ax = axes[0, 0]
for k in ["A", "B", "C", "D"]:
    vals = [df_new_stress[df_new_stress.q == q][f"roc_{k}"].mean() for q in QUANTILES]
    ls = "--" if k == "C" else "-"
    label = {
        "A": "vol_now",
        "B": "vol_now + best IT",
        "C": "vol_now + best TDA",
        "D": "vol_now + best IT+TDA",
    }[k]
    ax.plot(q_labels, vals, "o" + ls, color=colors[k], lw=2, label=label, ms=7)

ax.axhline(0.5, color="gray", lw=0.8, ls=":", alpha=0.5)
ax.set_ylabel("ROC-AUC")
ax.set_xlabel("Stress threshold")
ax.set_title("ROC-AUC by threshold\n(mean over assets and horizons)", fontsize=10)
ax.legend(fontsize=8)
ax.grid(alpha=0.2)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax = axes[0, 1]
x = np.arange(len(QUANTILES))
w = 0.25

for ci, k in enumerate(["B", "C", "D"]):
    vals = [df_new_stress[df_new_stress.q == q][f"droc_{k}"].mean() for q in QUANTILES]
    label = {
        "B": "+best IT",
        "C": "+best TDA",
        "D": "+best IT+TDA",
    }[k]
    ax.bar(x + (ci - 1) * w, vals, w * 0.9, label=label, color=colors[k], alpha=0.85)

ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels(q_labels)
ax.set_xlabel("Stress threshold")
ax.set_ylabel("ΔROC-AUC vs vol_now baseline")
ax.set_title("Incremental ΔROC over vol_now\n(by threshold)", fontsize=10)
ax.legend(
    fontsize=6,
    frameon=True,
    handlelength=1.0,
    handletextpad=0.4,
    borderpad=0.3,
    labelspacing=0.3
)
ax.grid(alpha=0.2, axis="y")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax = axes[1, 0]
asset_order = sub80.groupby("asset")["roc_D"].mean().sort_values(ascending=False).index.tolist()
x3 = np.arange(len(asset_order))
w3 = 0.2

for ci, k in enumerate(["A", "B", "C", "D"]):
    vals = [sub80[sub80.asset == a][f"roc_{k}"].mean() for a in asset_order]
    label = {
        "A": "vol_now",
        "B": "vol_now + IT",
        "C": "vol_now + TDA",
        "D": "vol_now + IT + TDA",
    }[k]
    ax.bar(x3 + (ci - 1.5) * w3, vals, w3 * 0.9, label=label, color=colors[k], alpha=0.85)

ax.axhline(0.5, color="gray", lw=0.8, ls=":", alpha=0.5)
ax.set_xticks(x3)
ax.set_xticklabels(asset_order, fontsize=9)
ax.set_ylabel("ROC-AUC")
ax.set_title("ROC-AUC by asset (Q80)\n(mean over horizons)", fontsize=10)
ax.legend(
    fontsize=8,
    loc="upper right",
    frameon=True,
    handlelength=0.7,
    handleheight=0.5,
    handletextpad=0.25,
    borderpad=0.2,
    labelspacing=0.15,
    borderaxespad=0.2,
    columnspacing=0.4
)
ax.grid(alpha=0.2, axis="y")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax = axes[1, 1]
pivot = df_new_stress.pivot_table(index="asset", columns="q", values="roc_D", aggfunc="mean")
pivot = pivot.reindex(ASSETS)[QUANTILES]

im = ax.imshow(pivot.values, cmap="RdYlGn", vmin=0.50, vmax=0.85, aspect="auto")
ax.set_xticks(range(len(QUANTILES)))
ax.set_xticklabels(q_labels)
ax.set_yticks(range(len(ASSETS)))
ax.set_yticklabels(ASSETS)

for i in range(len(ASSETS)):
    for j in range(len(QUANTILES)):
        val = pivot.values[i, j]
        if np.isfinite(val):
            text_color = "white" if val > 0.72 else "black"
            ax.text(j, i, f"{val:.3f}", ha="center", va="center", fontsize=8, color=text_color)

plt.colorbar(im, ax=ax, label="ROC-AUC (vol_now + best IT + best TDA)")
ax.set_title("ROC-AUC heatmap for configuration D\n(by asset × threshold)", fontsize=10)

plt.suptitle(
    "Stress prediction with the new best features\n"
    f"A=vol_now | B=vol_now+best IT | C=vol_now+best TDA | D=vol_now+best IT+TDA\n"
    f"Holdout {SPLIT_DATE} -> | Adaptive per-asset selection | Horizons: 1d, 2d, 4d",
    fontsize=11,
    y=1.01
)

plt.tight_layout()
plt.savefig("fig_stress_best_features.png", bbox_inches="tight", dpi=150)
plt.savefig("fig_stress_best_features.pdf", bbox_inches="tight", dpi=150)
plt.show()

print("Saved: fig_stress_best_features.png / .pdf")

stress_best_features = df_new_stress.copy()
stress_best_feature_sets = {
    "BEST_IT_GLOBAL": BEST_IT_GLOBAL,
    "BEST_TDA_GLOBAL": BEST_TDA_GLOBAL,
}

print(f"\nstress_best_features saved in memory ({len(df_new_stress)} rows)")
print("stress_best_feature_sets saved in memory")

STRESS PREDICTION WITH THE NEW BEST FEATURES

Global IT candidate pool:
  kl_BTC
  coinfo4_std
  kl_ADA
  nmi_lag1_mean
  coinfo3_std
  pairwise_mi_mean

Global TDA candidate pool:
  BI_L1_H0
  BC_ent_H1
  BII_tp_H1
  BII_L1_H1
  A_T_H0
  A_E_H0

IT pool size:  6
TDA pool size: 6

STRESS PREDICTION
A = vol_now | B = vol_now + best IT | C = vol_now + best TDA | D = vol_now + best IT + best TDA
Feature selection is adaptive per asset on the training set

Total rows: 80

ROC-AUC — AGGREGATE BY THRESHOLD (MEAN OVER ASSETS AND HORIZONS)

      q  pos%   ROC_A   ROC_B   ROC_C   ROC_D   ΔROC_B   ΔROC_C   ΔROC_D    D>A   Lift5_D
  ------------------------------------------------------------------------------------
  Q60%  46%  0.6840  0.6863  0.7103  0.7117  +0.0023  +0.0263  +0.0278   15/23     1.598
  Q70%  32%  0.6832  0.6848  0.7002  0.7030  +0.0016  +0.0171  +0.0199   17/22     1.846
  Q80%  17%  0.6608  0.6917  0.6722  0.7011  +0.0309  +0.0115  +0.0403   14/21     2.760
  Q90%   5%  0.62

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy import stats as sp_stats

from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

SPLIT_DATE = "2025-07-01"
PURGE_BARS = 96

HORIZONS = {
    48:  "1d",
    96:  "2d",
    192: "4d",
    288: "6d",
    336: "7d",
    672: "14d",
}

ASSETS = [a for a in ["BTC","ETH","SOL","BNB","XRP","DOGE","ADA","AVAX","TRX"] if a in ret.columns]

VOL_WINDOW = 48
ANN = np.sqrt(365 * 48)

PERM_USEFUL_THRESHOLD = -0.002

TOP_IT = 10
TOP_TDA = 6

def _ridge():
    return Pipeline([
        ("sc", StandardScaler()),
        ("m", RidgeCV(alphas=[0.01, 0.1, 1, 10, 100]))
    ])

def oos_r2(y_true, y_pred, naive_val):
    mse_m = np.mean((y_true - y_pred) ** 2)
    mse_n = np.mean((y_true - naive_val) ** 2)
    return float(1.0 - mse_m / mse_n) if mse_n > 1e-12 else np.nan

def res_corr(residuals_true, residuals_pred):
    return float(pd.Series(residuals_pred).corr(pd.Series(residuals_true)))

def hit_rate_vs_baseline(y_true, pred_base, pred_model):
    e_base = (y_true - pred_base) ** 2
    e_mod  = (y_true - pred_model) ** 2
    return float(np.mean(e_mod < e_base))

def dm_test(loss_diff):
    n = len(loss_diff)
    m = loss_diff.mean()
    s = loss_diff.std(ddof=1)
    t = m / (s / np.sqrt(n)) if s > 1e-12 else 0.0
    p = float(sp_stats.t.sf(abs(t), df=n - 1) * 2) if n > 1 else np.nan
    return float(t), float(p)

def sig(p):
    if not np.isfinite(p):
        return ""
    if p < 0.01:
        return "***"
    if p < 0.05:
        return "**"
    if p < 0.10:
        return "*"
    return ""

if "feat_analysis" in dir():
    avg_perm = feat_analysis["perm"]

    if "X_it_aug" in feat_analysis:
        X_it_all = feat_analysis["X_it_aug"].copy()
    else:
        X_it_all = X_it_slim.copy()

    if "X_tda_full" in feat_analysis:
        X_tda_all = feat_analysis["X_tda_full"].copy()
    else:
        raise ValueError("feat_analysis exists, but X_tda_full is missing inside it.")
else:
    if "avg_perm" not in dir():
        raise ValueError("avg_perm / feat_analysis['perm'] not found in memory.")
    if "X_it_aug" not in dir():
        raise ValueError("X_it_aug not found in memory.")
    if "X_tda_full" not in dir():
        raise ValueError("X_tda_full not found in memory.")

    X_it_all = X_it_aug.copy()
    X_tda_all = X_tda_full.copy()

perm_series = pd.Series(avg_perm).sort_values()

print("=" * 80)
print("NEW HOLDOUT EXPERIMENT WITH THE BEST FEATURES FROM PERMUTATION IMPORTANCE")
print("=" * 80)

it_candidates = [
    f for f in perm_series.index
    if (f in X_it_all.columns) and (perm_series[f] < PERM_USEFUL_THRESHOLD)
]

tda_candidates = [
    f for f in perm_series.index
    if (f in X_tda_all.columns) and (perm_series[f] < PERM_USEFUL_THRESHOLD)
]

BEST_IT = it_candidates[:TOP_IT]
BEST_TDA = tda_candidates[:TOP_TDA]

print("\nSelected IT features:")
for f in BEST_IT:
    print(f"  {f:<36} perm={perm_series[f]:+.4f}")

print("\nSelected TDA features:")
for f in BEST_TDA:
    print(f"  {f:<36} perm={perm_series[f]:+.4f}")

if len(BEST_IT) == 0:
    raise ValueError("No useful IT features survived the threshold.")
if len(BEST_TDA) == 0:
    raise ValueError("No useful TDA features survived the threshold.")

print(f"\nIT count:  {len(BEST_IT)}")
print(f"TDA count: {len(BEST_TDA)}")

split_ts = pd.Timestamp(SPLIT_DATE)
rows = []

for asset in ASSETS:
    r = ret[asset]

    for h, h_label in HORIZONS.items():
        vol_fwd = r.rolling(h).std().shift(-h) * ANN
        y_raw = np.log(vol_fwd.clip(lower=1e-8)).dropna()

        Xi_it = X_it_all[BEST_IT].copy()
        Xi_tda = X_tda_all[BEST_TDA].copy()

        idx = (
            X_vol.index
            .intersection(Xi_it.index)
            .intersection(Xi_tda.index)
            .intersection(y_raw.index)
        )

        valid = (
            X_vol.loc[idx].notna().all(1)
            & Xi_it.loc[idx].notna().all(1)
            & Xi_tda.loc[idx].notna().all(1)
            & y_raw.loc[idx].notna()
        )
        idx = idx[valid]

        if len(idx) < 1000:
            continue

        tr = idx < (split_ts - pd.Timedelta(minutes=30 * PURGE_BARS))
        te = idx >= split_ts

        if tr.sum() < 480 or te.sum() < 100:
            continue

        Xv_tr = X_vol.loc[idx[tr]]
        Xv_te = X_vol.loc[idx[te]]

        Xit_tr = Xi_it.loc[idx[tr]]
        Xit_te = Xi_it.loc[idx[te]]

        Xtd_tr = Xi_tda.loc[idx[tr]]
        Xtd_te = Xi_tda.loc[idx[te]]

        ytr = y_raw.loc[idx[tr]]
        yte = y_raw.loc[idx[te]]

        naive = float(ytr.mean())

        ck = Xv_tr.corrwith(ytr).abs().replace([np.inf, -np.inf], np.nan).dropna()
        top3 = ck.nlargest(3).index.tolist()

        mb = _ridge()
        mb.fit(Xv_tr[top3], ytr)

        pb_tr = mb.predict(Xv_tr[top3])
        pb_te = mb.predict(Xv_te[top3])

        res_tr = ytr.values - pb_tr
        res_te = yte.values - pb_te

        mit = _ridge()
        mit.fit(Xit_tr, res_tr)
        pit_te = mit.predict(Xit_te)
        pred_it = pb_te + pit_te

        mtda = _ridge()
        mtda.fit(Xtd_tr, res_tr)
        ptda_te = mtda.predict(Xtd_te)
        pred_tda = pb_te + ptda_te

        Xc_tr = pd.concat([Xit_tr, Xtd_tr], axis=1)
        Xc_te = pd.concat([Xit_te, Xtd_te], axis=1)

        mct = _ridge()
        mct.fit(Xc_tr, res_tr)
        pct_te = mct.predict(Xc_te)
        pred_all = pb_te + pct_te

        r2_A = oos_r2(yte.values, pb_te, naive)
        r2_B = oos_r2(yte.values, pred_it, naive)
        r2_C = oos_r2(yte.values, pred_tda, naive)
        r2_D = oos_r2(yte.values, pred_all, naive)

        rc_B = res_corr(res_te, pit_te)
        rc_C = res_corr(res_te, ptda_te)
        rc_D = res_corr(res_te, pct_te)

        mse_A = np.mean((yte.values - pb_te) ** 2)
        mse_B = np.mean((yte.values - pred_it) ** 2)
        mse_C = np.mean((yte.values - pred_tda) ** 2)
        mse_D = np.mean((yte.values - pred_all) ** 2)

        hr_B = hit_rate_vs_baseline(yte.values, pb_te, pred_it)
        hr_C = hit_rate_vs_baseline(yte.values, pb_te, pred_tda)
        hr_D = hit_rate_vs_baseline(yte.values, pb_te, pred_all)

        dB = (yte.values - pb_te) ** 2 - (yte.values - pred_it) ** 2
        dC = (yte.values - pb_te) ** 2 - (yte.values - pred_tda) ** 2
        dD = (yte.values - pb_te) ** 2 - (yte.values - pred_all) ** 2

        _, p_B = dm_test(dB)
        _, p_C = dm_test(dC)
        _, p_D = dm_test(dD)

        rows.append({
            "asset": asset,
            "horizon": h_label,
            "h": h,

            "r2_A": round(r2_A, 4),
            "r2_B": round(r2_B, 4),
            "r2_C": round(r2_C, 4),
            "r2_D": round(r2_D, 4),

            "dR2_B": round(r2_B - r2_A, 4),
            "dR2_C": round(r2_C - r2_A, 4),
            "dR2_D": round(r2_D - r2_A, 4),

            "rc_B": round(rc_B, 4),
            "rc_C": round(rc_C, 4),
            "rc_D": round(rc_D, 4),

            "mse_ratio_B": round(mse_B / mse_A, 4) if mse_A > 0 else np.nan,
            "mse_ratio_C": round(mse_C / mse_A, 4) if mse_A > 0 else np.nan,
            "mse_ratio_D": round(mse_D / mse_A, 4) if mse_A > 0 else np.nan,

            "hit_B": round(hr_B, 4),
            "hit_C": round(hr_C, 4),
            "hit_D": round(hr_D, 4),

            "p_B": round(p_B, 4),
            "p_C": round(p_C, 4),
            "p_D": round(p_D, 4),

            "sig_B": sig(p_B),
            "sig_C": sig(p_C),
            "sig_D": sig(p_D),

            "n_train": int(tr.sum()),
            "n_test": int(te.sum()),
        })

df_best = pd.DataFrame(rows)
print(f"\nRows collected: {len(df_best)}")

if df_best.empty:
    raise ValueError("The result dataframe is empty. Check intersections, NaNs, and sample sizes.")

print("\n" + "=" * 100)
print("FINAL COMPARISON WITH AUTOMATICALLY SELECTED BEST FEATURES")
print("Target: logvol = log(realized_vol_fwd_h)")
print("A = Baseline(vol) | B = +best IT | C = +best TDA | D = +best IT + best TDA")
print("=" * 100)

print(
    f"\n  {'asset':<6} {'h':<4}"
    f" {'R²_A':>9} {'R²_B':>9} {'R²_C':>9} {'R²_D':>9}"
    f" {'ΔB':>8} {'ΔC':>8} {'ΔD':>8}  best"
)
print("  " + "-" * 90)

prev = None
for _, row in df_best.sort_values(["asset", "h"]).iterrows():
    if prev and prev != row["asset"]:
        print("  " + "·" * 90)
    prev = row["asset"]

    vals = {"A": row["r2_A"], "B": row["r2_B"], "C": row["r2_C"], "D": row["r2_D"]}
    best = max(vals, key=vals.get)

    mb = "↑" if row["dR2_B"] > 0.005 else ("↓" if row["dR2_B"] < -0.005 else "~")
    mc = "↑" if row["dR2_C"] > 0.005 else ("↓" if row["dR2_C"] < -0.005 else "~")
    md = "↑" if row["dR2_D"] > 0.005 else ("↓" if row["dR2_D"] < -0.005 else "~")

    print(
        f"  {row['asset']:<6} {row['horizon']:<4}"
        f" {row['r2_A']:>+9.4f}"
        f" {row['r2_B']:>+9.4f}"
        f" {row['r2_C']:>+9.4f}"
        f" {row['r2_D']:>+9.4f}"
        f" {row['dR2_B']:>+7.4f}{mb}"
        f" {row['dR2_C']:>+7.4f}{mc}"
        f" {row['dR2_D']:>+7.4f}{md}"
        f"  {best}"
    )

print("\n" + "=" * 75)
print("SUMMARY")
print("=" * 75)

n = len(df_best)
print(f"""
  Config         mean ΔR²   ΔR²>0    mean rc   MSE<1    hit rate
  ----------------------------------------------------------------
  B (+best IT)   {df_best['dR2_B'].mean():>+8.4f}   {(df_best['dR2_B']>0).sum():>2}/{n}   {df_best['rc_B'].mean():>+7.4f}   {(df_best['mse_ratio_B']<1).sum():>2}/{n}   {df_best['hit_B'].mean():.3f}
  C (+best TDA)  {df_best['dR2_C'].mean():>+8.4f}   {(df_best['dR2_C']>0).sum():>2}/{n}   {df_best['rc_C'].mean():>+7.4f}   {(df_best['mse_ratio_C']<1).sum():>2}/{n}   {df_best['hit_C'].mean():.3f}
  D (+both)      {df_best['dR2_D'].mean():>+8.4f}   {(df_best['dR2_D']>0).sum():>2}/{n}   {df_best['rc_D'].mean():>+7.4f}   {(df_best['mse_ratio_D']<1).sum():>2}/{n}   {df_best['hit_D'].mean():.3f}
""")

print("Top-5 tasks by ΔR² for configuration D:")
for _, row in df_best.nlargest(5, "dR2_D").iterrows():
    print(
        f"  {row['asset']} h={row['horizon']}: "
        f"R²_A={row['r2_A']:+.4f} -> "
        f"R²_B={row['r2_B']:+.4f} -> "
        f"R²_C={row['r2_C']:+.4f} -> "
        f"R²_D={row['r2_D']:+.4f} "
        f"(ΔD={row['dR2_D']:+.4f}, p={row['p_D']:.4f}{row['sig_D']})"
    )

asset_summary = (
    df_best.groupby("asset", as_index=False)
    .agg(
        r2_A=("r2_A", "mean"),
        r2_B=("r2_B", "mean"),
        r2_C=("r2_C", "mean"),
        r2_D=("r2_D", "mean"),

        dR2_B=("dR2_B", "mean"),
        dR2_C=("dR2_C", "mean"),
        dR2_D=("dR2_D", "mean"),

        rc_B=("rc_B", "mean"),
        rc_C=("rc_C", "mean"),
        rc_D=("rc_D", "mean"),

        mse_ratio_B=("mse_ratio_B", "mean"),
        mse_ratio_C=("mse_ratio_C", "mean"),
        mse_ratio_D=("mse_ratio_D", "mean"),

        hit_B=("hit_B", "mean"),
        hit_C=("hit_C", "mean"),
        hit_D=("hit_D", "mean"),
    )
)

def choose_best_config(row):
    vals = {
        "A": row["r2_A"],
        "B": row["r2_B"],
        "C": row["r2_C"],
        "D": row["r2_D"],
    }
    return max(vals, key=vals.get)

def choose_best_addon(row):
    vals = {
        "B": row["dR2_B"],
        "C": row["dR2_C"],
        "D": row["dR2_D"],
    }
    return max(vals, key=vals.get)

asset_summary["best_config"] = asset_summary.apply(choose_best_config, axis=1)
asset_summary["best_addon"] = asset_summary.apply(choose_best_addon, axis=1)

asset_summary = asset_summary.sort_values(
    ["r2_D", "r2_C", "r2_B", "r2_A"],
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 90)
print("ASSET-LEVEL AGGREGATION (MEAN OVER HORIZONS)")
print("=" * 90)

print(
    f"\n  {'asset':<6}"
    f" {'R²_A':>10} {'R²_B':>10} {'R²_C':>10} {'R²_D':>10}"
    f" {'ΔB':>9} {'ΔC':>9} {'ΔD':>9}"
    f" {'best_cfg':>10} {'best_addon':>12}"
)

print("  " + "-" * 100)

for _, row in asset_summary.iterrows():
    print(
        f"  {row['asset']:<6}"
        f" {row['r2_A']:>+10.4f}"
        f" {row['r2_B']:>+10.4f}"
        f" {row['r2_C']:>+10.4f}"
        f" {row['r2_D']:>+10.4f}"
        f" {row['dR2_B']:>+9.4f}"
        f" {row['dR2_C']:>+9.4f}"
        f" {row['dR2_D']:>+9.4f}"
        f" {row['best_config']:>10}"
        f" {row['best_addon']:>12}"
    )

print("\nBest configuration counts across assets:")
print(asset_summary["best_config"].value_counts().sort_index())

print("\nBest incremental addon counts across assets:")
print(asset_summary["best_addon"].value_counts().sort_index())

print("\nBuilding final plots...")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

asset_order = (
    df_best.groupby("asset")["r2_D"]
    .mean()
    .sort_values(ascending=False)
    .index
    .tolist()
)

h_order = ["1d", "2d", "4d", "6d", "7d", "14d"]
x = np.arange(len(asset_order))
w = 0.2

ax = axes[0, 0]
for ci, (col, label, color) in enumerate([
    ("r2_A", "A: Baseline",   "#9E9E9E"),
    ("r2_B", "B: +best IT",   "#2196F3"),
    ("r2_C", "C: +best TDA",  "#FF9800"),
    ("r2_D", "D: +best both", "#4CAF50"),
]):
    vals = [df_best[df_best.asset == a][col].mean() for a in asset_order]
    ax.bar(x + (ci - 1.5) * w, vals, w * 0.9, label=label, color=color, alpha=0.85)

ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels(asset_order, fontsize=9)
ax.set_ylabel("OOS R² (vs naive mean logvol)")
ax.set_title("OOS R² by asset — 4 configurations\n(logvol, all horizons)", fontsize=10)
ax.legend(fontsize=8)
ax.grid(alpha=0.2, axis="y")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax = axes[0, 1]
for col, label, color, ls in [
    ("r2_A", "A: Baseline",   "#9E9E9E", "-"),
    ("r2_B", "B: +best IT",   "#2196F3", "-"),
    ("r2_C", "C: +best TDA",  "#FF9800", "--"),
    ("r2_D", "D: +best both", "#4CAF50", "-"),
]:
    vals = [df_best[df_best.horizon == h][col].mean() for h in h_order]
    ax.plot(h_order, vals, "o" + ls, color=color, lw=2, label=label, ms=7)

ax.axhline(0, color="black", lw=0.5, ls=":")
ax.set_ylabel("OOS R²")
ax.set_xlabel("Forecast horizon")
ax.set_title("OOS R² by horizon\n(mean over assets)", fontsize=10)
ax.legend(fontsize=8)
ax.grid(alpha=0.2)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax = axes[1, 0]
x2 = np.arange(len(asset_order))
w2 = 0.28

for ci, (col, label, color) in enumerate([
    ("dR2_B", "B: +best IT",   "#2196F3"),
    ("dR2_C", "C: +best TDA",  "#FF9800"),
    ("dR2_D", "D: +best both", "#4CAF50"),
]):
    vals = [df_best[df_best.asset == a][col].mean() for a in asset_order]
    ax.bar(x2 + (ci - 1) * w2, vals, w2 * 0.9, label=label, color=color, alpha=0.85)

ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(x2)
ax.set_xticklabels(asset_order, fontsize=9)
ax.set_ylabel("ΔR² relative to baseline A")
ax.set_title("Incremental ΔR² over baseline\n(logvol, all horizons)", fontsize=10)
ax.legend(fontsize=8)
ax.grid(alpha=0.2, axis="y")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax = axes[1, 1]
pivot = df_best.pivot_table(
    index="asset",
    columns="horizon",
    values="mse_ratio_D",
    aggfunc="mean"
)
pivot = pivot.reindex(asset_order)[h_order]

im = ax.imshow(pivot.values, cmap="RdYlGn_r", vmin=0.85, vmax=1.15, aspect="auto")
ax.set_xticks(range(len(h_order)))
ax.set_xticklabels(h_order)
ax.set_yticks(range(len(asset_order)))
ax.set_yticklabels(asset_order)

for i in range(len(asset_order)):
    for j in range(len(h_order)):
        val = pivot.values[i, j]
        if np.isfinite(val):
            col = "white" if abs(val - 1) > 0.07 else "black"
            mark = "✓" if val < 1.0 else ""
            ax.text(j, i, f"{val:.3f}{mark}", ha="center", va="center", fontsize=8, color=col)

plt.colorbar(im, ax=ax, label="MSE ratio (D / baseline)\n<1.0 = better ✓")
ax.set_title("MSE ratio for configuration D\n(green < 1 = better than baseline)", fontsize=10)

plt.suptitle(
    "Final comparison with automatically selected best features\n"
    f"Target: logvol = log(realized_vol_fwd) | holdout {SPLIT_DATE} ->",
    fontsize=12,
    y=1.01
)
plt.tight_layout()
plt.savefig("fig_final_comparison_best_features.png", bbox_inches="tight", dpi=150)
plt.savefig("fig_final_comparison_best_features.pdf", bbox_inches="tight", dpi=150)
plt.show()

print("Saved: fig_final_comparison_best_features_96.png / .pdf")

final_comparison_best = df_best.copy()
best_feature_sets = {
    "BEST_IT": BEST_IT,
    "BEST_TDA": BEST_TDA,
}
print(f"\nfinal_comparison_best saved in memory ({len(df_best)} rows)")
print("best_feature_sets saved in memory")

In [126]:
print("feat_analysis" in dir())
print("X_it_aug" in dir())
print("X_tda_full" in dir())
print("avg_perm" in dir())

True
True
True
True


In [133]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from itertools import combinations
from sklearn.metrics import mutual_info_score
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

SPLIT_DATE = "2025-07-01"
PURGE_BARS = 96
WINDOWS    = [96, 144, 192, 288, 384]
STEP       = 4
BINS       = 10
HORIZONS   = {96: "2d", 192: "4d", 336: "7d"}
ASSETS     = ["BTC", "ETH", "SOL", "BNB", "XRP", "DOGE", "ADA", "AVAX", "TRX"]
VOL_WINDOW = 48
ANN        = np.sqrt(365 * 48)

BEST_IT  = ["kl_BTC", "coinfo4_std", "kl_ADA", "coinfo3_std", "pairwise_mi_mean"]
BEST_TDA = ["BI_L1_H0", "BII_tp_H1", "A_T_H0", "BI_ent_H1"]

def _ridge():
    return Pipeline([("sc", StandardScaler()),
                     ("m", RidgeCV(alphas=[0.01, 0.1, 1, 10, 100]))])

def oos_r2(y_true, y_pred, naive):
    mse_m = np.mean((y_true - y_pred) ** 2)
    mse_n = np.mean((y_true - naive) ** 2)
    return float(1.0 - mse_m / mse_n) if mse_n > 1e-12 else np.nan

def _rk(v, bins=BINS):
    r = np.argsort(np.argsort(v))
    return (r * bins // len(v)).clip(0, bins - 1).astype(np.int32)

def _mi_hist(x, y, bins=BINS):
    bx = _rk(x, bins); by = _rk(y, bins)
    c = np.histogram2d(bx, by, bins=bins,
                       range=[[0, bins], [0, bins]])[0]
    return float(mutual_info_score(None, None, contingency=c))

def build_it_features(ret_data, assets, W, k=STEP, bins=BINS):
    avail = [a for a in assets if a in ret_data.columns]
    R = ret_data[avail].fillna(0).values
    n = len(R)
    idx_all = ret_data.index
    starts  = np.arange(0, n - W + 1, k)
    ends    = starts + W
    times   = [idx_all[e - 1] for e in ends]
    pairs   = list(combinations(range(len(avail)), 2))
    triples = list(combinations(range(len(avail)), 3))

    rows = []
    for si, (s, e) in enumerate(zip(starts, ends)):
        wnd = R[s:e]
        mi_vals = [_mi_hist(wnd[:, i], wnd[:, j], bins) for i, j in pairs]
        kl_vals = {}
        for ai, asset in enumerate(avail):
            if s >= W:
                x_c = wnd[:, ai]
                x_p = R[s - W:s, ai]
                bc = _rk(x_c, bins); bp = _rk(x_p, bins)
                pc = np.bincount(bc, minlength=bins).astype(float) + 1e-8
                pp = np.bincount(bp, minlength=bins).astype(float) + 1e-8
                pc /= pc.sum(); pp /= pp.sum()
                kl_vals[f"kl_{asset}"] = float(np.sum(pc * np.log(pc / pp)))
            else:
                kl_vals[f"kl_{asset}"] = 0.0
        ci3 = []
        for i, j, k_ in triples:
            mij  = _mi_hist(wnd[:, i], wnd[:, j], bins)
            mjk  = _mi_hist(wnd[:, j], wnd[:, k_], bins)
            mijk = _mi_hist(wnd[:, i], wnd[:, j] + wnd[:, k_], bins)
            ci3.append(mij + mjk - mijk)
        ci4 = []
        quads = list(combinations(range(len(avail)), 4))
        for qi, (i, j, k_, l) in enumerate(quads[:30]):
            ci4.append(_mi_hist(wnd[:, i], wnd[:, j], bins) -
                       _mi_hist(wnd[:, k_], wnd[:, l], bins))
        row = {
            "pairwise_mi_mean": float(np.mean(mi_vals)),
            "pairwise_mi_std":  float(np.std(mi_vals)),
            "coinfo3_mean": float(np.mean(ci3)) if ci3 else 0.0,
            "coinfo3_std":  float(np.std(ci3))  if ci3 else 0.0,
            "coinfo4_std":  float(np.std(ci4))  if ci4 else 0.0,
        }
        row.update(kl_vals)
        rows.append(row)
    return pd.DataFrame(rows, index=pd.DatetimeIndex(times))

def build_tda_gauss_fast(ret_data, assets, W, k=STEP):
    try:
        import gudhi
    except ImportError:
        print("  GUDHI not available — TDA rebuild skipped")
        return None

    avail = [a for a in assets if a in ret_data.columns]
    R = ret_data[avail].fillna(0).values
    n = len(R); dim = len(avail)
    idx_all = ret_data.index
    starts  = np.arange(0, n - W + 1, k)
    ends    = starts + W
    times   = [idx_all[e - 1] for e in ends]
    eps     = 1e-8

    def H_gauss(sub):
        d = sub.shape[0]
        det = np.linalg.det(sub)
        return 0.5 * np.log((2*np.pi*np.e)**d * det) if det > 1e-10 else 0.0

    tp_H0=[]; ent_H0=[]; tp_H1=[]; ent_H1=[]; L1_H0=[]
    T_H0_list=[]; E_H0_list=[]

    for s, e in zip(starts, ends):
        wnd = R[s:e]
        A = np.cov(wnd, rowvar=False) + eps*np.eye(dim)
        st = gudhi.SimplexTree()
        for i in range(dim): st.insert([i], filtration=0.0)
        for i, j in combinations(range(dim), 2):
            w = H_gauss(A[[i],:][:,[i]]) + H_gauss(A[[j],:][:,[j]]) - H_gauss(A[np.ix_([i,j],[i,j])])
            st.insert([i,j], filtration=abs(w))
        for i, j, k_ in combinations(range(dim), 3):
            w = H_gauss(A[[i],:][:,[i]]) + H_gauss(A[[j],:][:,[j]]) + H_gauss(A[[k_],:][:,[k_]]) - H_gauss(A[np.ix_([i,j,k_],[i,j,k_])])
            st.insert([i,j,k_], filtration=abs(w))
        st.initialize_filtration()
        diag = st.persistence()

        def diag_stats(dd):
            pts = [(float(b), float(d) if np.isfinite(d) else 1.0)
                   for ddd,(b,d) in diag if ddd==dd]
            if not pts: return 0., 0.
            lts = np.array([d-b for b,d in pts]); lts=lts[lts>0]
            if not len(lts): return 0., 0.
            T=lts.sum(); p=lts/T
            return float(T), float(-(p*np.log(p)).sum())

        t0,e0 = diag_stats(0); t1,e1 = diag_stats(1)
        tp_H0.append(t0); ent_H0.append(e0)
        tp_H1.append(t1); ent_H1.append(e1)
        T_H0_list.append(t0); E_H0_list.append(e0)
        pts0 = [(float(b), float(d) if np.isfinite(d) else 1.0)
                for ddd,(b,d) in diag if ddd==0 and np.isfinite(d)]
        L1_H0.append(sum(d-b for b,d in pts0) if pts0 else 0.0)

    return pd.DataFrame({
        "BI_tp_H0":  tp_H0, "BI_ent_H0": ent_H0,
        "BI_tp_H1":  tp_H1, "BI_ent_H1": ent_H1,
        "BI_L1_H0":  L1_H0,
        "A_T_H0":    T_H0_list, "A_E_H0": E_H0_list,
        "BII_tp_H1": tp_H1,
    }, index=pd.DatetimeIndex(times))

def evaluate_window(W, X_it_w, X_tda_w):
    split_ts = pd.Timestamp(SPLIT_DATE)
    rows = []
    it_cols  = [c for c in BEST_IT  if c in X_it_w.columns]
    tda_cols = [c for c in BEST_TDA if c in X_tda_w.columns]
    if not it_cols or not tda_cols:
        print(f"  W={W}: missing columns — IT:{it_cols}, TDA:{tda_cols}")
        return pd.DataFrame()

    X_it_ri  = X_it_w[it_cols].reindex(ret.index, method="ffill")
    X_tda_ri = X_tda_w[tda_cols].reindex(ret.index, method="ffill")

    for asset in ASSETS:
        if asset not in ret.columns: continue
        r = ret[asset]
        for h, hl in HORIZONS.items():
            vol_fwd = r.rolling(h).std().shift(-h) * ANN
            y_raw   = np.log(vol_fwd.clip(lower=1e-8)).dropna()
            idx = (X_vol.index.intersection(X_it_ri.index)
                   .intersection(X_tda_ri.index).intersection(y_raw.index))
            valid = (X_vol.loc[idx].notna().all(1) &
                     X_it_ri.loc[idx].notna().all(1) &
                     X_tda_ri.loc[idx].notna().all(1) &
                     y_raw.loc[idx].notna())
            idx = idx[valid]
            if len(idx) < 500: continue
            tr = idx < (split_ts - pd.Timedelta(minutes=30*PURGE_BARS))
            te = idx >= split_ts
            if tr.sum() < 240 or te.sum() < 50: continue

            ytr = y_raw.loc[idx[tr]]; yte = y_raw.loc[idx[te]]
            naive = float(ytr.mean())
            Xv_tr = X_vol.loc[idx[tr]]; Xv_te = X_vol.loc[idx[te]]
            ck = Xv_tr.corrwith(ytr).abs().dropna()
            top3 = ck.nlargest(3).index.tolist()
            mb = _ridge(); mb.fit(Xv_tr[top3], ytr)
            pb_te  = mb.predict(Xv_te[top3])
            res_tr = ytr.values - mb.predict(Xv_tr[top3])

            Xc_tr = pd.concat([X_it_ri.loc[idx[tr]], X_tda_ri.loc[idx[tr]]], axis=1)
            Xc_te = pd.concat([X_it_ri.loc[idx[te]], X_tda_ri.loc[idx[te]]], axis=1)
            mct = _ridge(); mct.fit(Xc_tr, res_tr)
            pred_D = pb_te + mct.predict(Xc_te)

            rows.append({"W": W, "asset": asset, "horizon": hl,
                         "r2_A": oos_r2(yte.values, pb_te,   naive),
                         "r2_D": oos_r2(yte.values, pred_D,  naive),
                         "dR2":  oos_r2(yte.values, pred_D,  naive) -
                                 oos_r2(yte.values, pb_te,   naive)})
    return pd.DataFrame(rows)

print("=" * 65)
print("  WINDOW SENSITIVITY — VOLATILITY PREDICTION")
print(f"  Windows: {WINDOWS} bars  |  Step: {STEP} bars")
print("=" * 65)

all_results = []

for W in WINDOWS:
    print(f"\nW = {W} bars ({W//48}d)...")
    X_it_w = build_it_features(ret, ASSETS, W)
    print(f"  IT: {X_it_w.shape[1]} features, {len(X_it_w)} windows")

    if W == 192 and "tda_A" in dir() and "tda_B" in dir():
        frames = [tda_A.add_prefix("A_")]
        for mode in ["I","II","C"]:
            d = tda_B[mode]
            fm = pd.DataFrame({
                f"B{mode}_tp_H0": d["tp_H0"], f"B{mode}_ent_H0": d["ent_H0"],
                f"B{mode}_tp_H1": d["tp_H1"], f"B{mode}_ent_H1": d["ent_H1"],
                f"B{mode}_L1_H0": d["L1_H0"], f"B{mode}_L1_H1": d["L1_H1"],
            }, index=pd.DatetimeIndex(d["times"]))
            frames.append(fm)
        X_tda_w = pd.concat(frames, axis=1).sort_index()
        print(f"  TDA (precomputed): {X_tda_w.shape[1]} features")
    else:
        X_tda_w = build_tda_gauss_fast(ret, ASSETS, W)
        if X_tda_w is None:
            continue
        print(f"  TDA (rebuilt): {X_tda_w.shape[1]} features, {len(X_tda_w)} windows")

    df_w = evaluate_window(W, X_it_w, X_tda_w)
    if not df_w.empty:
        all_results.append(df_w)
        print(f"  mean ΔR²={df_w['dR2'].mean():+.4f}  "
              f"pos={( df_w['dR2']>0).sum()}/{len(df_w)}")

if all_results:
    df_all = pd.concat(all_results, ignore_index=True)

    print("\n" + "="*65)
    print("  SUMMARY — mean ΔR²(D) by window")
    print("="*65)
    print(f"  {'Window':<12} {'mean ΔR²':>10} {'ΔR²>0':>8}")
    print("  " + "-"*32)
    for W in WINDOWS:
        s = df_all[df_all.W==W]
        if s.empty: continue
        print(f"  W={W} ({W//48}d){' ':2} {s['dR2'].mean():>+10.4f}  "
              f"{(s['dR2']>0).sum():>3}/{len(s)}")

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    ax = axes[0]
    for hl in HORIZONS.values():
        xs, ys = [], []
        for W in WINDOWS:
            s = df_all[(df_all.W==W)&(df_all.horizon==hl)]
            if not s.empty: xs.append(W); ys.append(s["dR2"].mean())
        ax.plot(xs, ys, "o-", lw=2, ms=7, label=f"h={hl}")
    ax.axhline(0, color="black", lw=0.8, ls=":")
    ax.axvline(192, color="gray", lw=1.2, ls="--", alpha=0.6, label="W=192 (thesis)")
    ax.set_xlabel("Feature window W (bars)")
    ax.set_ylabel("Mean ΔR² (config D vs baseline)")
    ax.set_title("Window sensitivity — volatility prediction", fontsize=10)
    ax.legend(fontsize=8); ax.grid(alpha=0.2)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

    ax = axes[1]
    heat = df_all.pivot_table(index="asset", columns="W", values="dR2", aggfunc="mean")
    heat = heat.reindex(columns=WINDOWS)
    im = ax.imshow(heat.values, cmap="RdYlGn", vmin=-0.15, vmax=0.15, aspect="auto")
    ax.set_xticks(range(len(WINDOWS)))
    ax.set_xticklabels([f"W={w}" for w in WINDOWS], fontsize=8)
    ax.set_yticks(range(len(heat.index)))
    ax.set_yticklabels(heat.index, fontsize=8)
    for i in range(len(heat.index)):
        for j in range(len(WINDOWS)):
            val = heat.values[i,j]
            if np.isfinite(val):
                c = "white" if abs(val)>0.08 else "black"
                ax.text(j, i, f"{val:+.2f}", ha="center", va="center", fontsize=7, color=c)
    plt.colorbar(im, ax=ax, label="Mean ΔR²(D) vs baseline")
    ax.set_title("ΔR² by asset × window", fontsize=10)

    plt.suptitle(
        "Window sensitivity: volatility prediction\n"
        f"Config D = vol + best IT + best TDA | holdout {SPLIT_DATE}",
        fontsize=11, y=1.01)
    plt.tight_layout()
    plt.savefig("fig_window_sensitivity_vol.png", bbox_inches="tight", dpi=150)
    plt.savefig("fig_window_sensitivity_vol.pdf", bbox_inches="tight", dpi=150)
    plt.show()
    print("\nSaved: fig_window_sensitivity_vol.png / .pdf")

    window_sensitivity_vol = df_all.copy()
    print(f"window_sensitivity_vol saved to memory ({len(df_all)} rows)")

  WINDOW SENSITIVITY — VOLATILITY PREDICTION
  Windows: [96, 144, 192, 288, 384] bars  |  Step: 4 bars

W = 96 bars (2d)...
  IT: 14 features, 11662 windows
  TDA (rebuilt): 8 features, 11662 windows
  mean ΔR²=+0.0438  pos=21/27

W = 144 bars (3d)...
  IT: 14 features, 11650 windows
  TDA (rebuilt): 8 features, 11650 windows
  mean ΔR²=+0.0295  pos=19/27

W = 192 bars (4d)...
  IT: 14 features, 11638 windows
  TDA (precomputed): 26 features
  mean ΔR²=+0.0077  pos=16/27

W = 288 bars (6d)...
  IT: 14 features, 11614 windows
  TDA (rebuilt): 8 features, 11614 windows
  mean ΔR²=-0.0512  pos=11/27

W = 384 bars (8d)...
  IT: 14 features, 11590 windows
  TDA (rebuilt): 8 features, 11590 windows
  mean ΔR²=-0.0418  pos=15/27

  SUMMARY — mean ΔR²(D) by window
  Window         mean ΔR²    ΔR²>0
  --------------------------------
  W=96 (2d)      +0.0438   21/27
  W=144 (3d)      +0.0295   19/27
  W=192 (4d)      +0.0077   16/27
  W=288 (6d)      -0.0512   11/27
  W=384 (8d)      -0.0418   

In [137]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from itertools import combinations

from sklearn.metrics import mutual_info_score, roc_auc_score, average_precision_score
from sklearn.feature_selection import mutual_info_regression, mutual_info_classif
from sklearn.linear_model import RidgeCV, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

SELECTION_DATE = "2025-01-01"
FINAL_DATE = "2025-07-01"
PURGE_BARS = 96

WINDOWS = [96, 144, 192]
STEP = 4
BINS = 10

ASSETS = [a for a in ["BTC","ETH","SOL","BNB","XRP","DOGE","ADA","AVAX"] if a in ret.columns]

VOL_HORIZONS = {
    48: "1d",
    96: "2d",
    192: "4d",
    288: "6d",
    336: "7d",
    672: "14d",
}

STRESS_HORIZONS = {
    48: "1d",
    96: "2d",
    192: "4d",
}

STRESS_QUANTILES = [0.60, 0.70, 0.80, 0.90]

VOL_WINDOW = 48
ANN = np.sqrt(365 * 48)

TOP_IT = 6
TOP_TDA = 6
PERM_THRESHOLD = -0.002

def _ridge():
    return Pipeline([
        ("sc", StandardScaler()),
        ("m", RidgeCV(alphas=[0.01, 0.1, 1, 10, 100]))
    ])

def _logit():
    return Pipeline([
        ("sc", StandardScaler()),
        ("m", LogisticRegression(
            C=0.1,
            max_iter=2000,
            class_weight="balanced",
            solver="lbfgs"
        ))
    ])

def oos_r2(y_true, y_pred, naive):
    mse_m = np.mean((y_true - y_pred) ** 2)
    mse_n = np.mean((y_true - naive) ** 2)
    return float(1 - mse_m / mse_n) if mse_n > 1e-12 else np.nan

In [139]:
# def _rank_bins(x, bins=10):
#     x = np.asarray(x, dtype=float)
#     r = pd.Series(x).rank(method="average").values
#     b = np.floor((r - 1) / max(1, len(x)) * bins).astype(int)
#     return np.clip(b, 0, bins - 1)

# def mi_hist_rank(x, y, bins=10):
#     x = np.asarray(x, dtype=float)
#     y = np.asarray(y, dtype=float)
#     m = np.isfinite(x) & np.isfinite(y)
#     x, y = x[m], y[m]
#     if len(x) < max(20, bins):
#         return np.nan
#     bx = _rank_bins(x, bins)
#     by = _rank_bins(y, bins)
#     c = np.histogram2d(bx, by, bins=bins, range=[[0, bins], [0, bins]])[0]
#     return float(mutual_info_score(None, None, contingency=c))

# def build_it_features_for_window(ret_df, assets, W, step=4, bins=10):
#     avail = [a for a in assets if a in ret_df.columns]
#     R = ret_df[avail].replace([np.inf, -np.inf], np.nan).fillna(0).values
#     idx = ret_df.index
#     n = len(R)

#     pairs = list(combinations(range(len(avail)), 2))
#     triples = list(combinations(range(len(avail)), 3))
#     quads = list(combinations(range(len(avail)), 4))

#     rows = []
#     times = []

#     for s in range(0, n - W + 1, step):
#         e = s + W
#         wnd = R[s:e]
#         row = {}

#         for ai, asset in enumerate(avail):
#             if s >= W:
#                 x_cur = wnd[:, ai]
#                 x_prev = R[s-W:s, ai]

#                 bc = _rank_bins(x_cur, bins)
#                 bp = _rank_bins(x_prev, bins)

#                 pc = np.bincount(bc, minlength=bins).astype(float) + 1e-8
#                 pp = np.bincount(bp, minlength=bins).astype(float) + 1e-8
#                 pc /= pc.sum()
#                 pp /= pp.sum()

#                 row[f"kl_{asset}"] = float(np.sum(pc * np.log(pc / pp)))
#             else:
#                 row[f"kl_{asset}"] = 0.0

#         mi_vals = []
#         for i, j in pairs:
#             mi = mi_hist_rank(wnd[:, i], wnd[:, j], bins=bins)
#             if np.isfinite(mi):
#                 mi_vals.append(mi)

#         row["pairwise_mi_mean"] = float(np.mean(mi_vals)) if mi_vals else np.nan
#         row["pairwise_mi_std"] = float(np.std(mi_vals)) if mi_vals else np.nan
#         row["pairwise_mi_max"] = float(np.max(mi_vals)) if mi_vals else np.nan
#         row["pairwise_mi_min"] = float(np.min(mi_vals)) if mi_vals else np.nan

#         ci3_vals = []
#         for i, j, k in triples:
#             mij = mi_hist_rank(wnd[:, i], wnd[:, j], bins=bins)
#             mjk = mi_hist_rank(wnd[:, j], wnd[:, k], bins=bins)
#             mijk = mi_hist_rank(wnd[:, i], wnd[:, j] + wnd[:, k], bins=bins)
#             if np.isfinite(mij) and np.isfinite(mjk) and np.isfinite(mijk):
#                 ci3_vals.append(mij + mjk - mijk)

#         row["coinfo3_mean"] = float(np.mean(ci3_vals)) if ci3_vals else np.nan
#         row["coinfo3_std"] = float(np.std(ci3_vals)) if ci3_vals else np.nan

#         ci4_vals = []
#         for i, j, k, l in quads[:50]:
#             a = mi_hist_rank(wnd[:, i], wnd[:, j], bins=bins)
#             b = mi_hist_rank(wnd[:, k], wnd[:, l], bins=bins)
#             if np.isfinite(a) and np.isfinite(b):
#                 ci4_vals.append(a - b)

#         row["coinfo4_mean"] = float(np.mean(ci4_vals)) if ci4_vals else np.nan
#         row["coinfo4_std"] = float(np.std(ci4_vals)) if ci4_vals else np.nan

#         rows.append(row)
#         times.append(idx[e - 1])

#     return pd.DataFrame(rows, index=pd.DatetimeIndex(times)).ffill()

# def build_tda_features_for_window(ret_df, assets, W, step=4):
#     try:
#         import gudhi
#         from gudhi.representations import Landscape
#     except ImportError:
#         raise ImportError("Install gudhi first.")

#     avail = [a for a in assets if a in ret_df.columns]
#     R = ret_df[avail].replace([np.inf, -np.inf], np.nan).fillna(0).values
#     idx = ret_df.index
#     n = len(R)
#     dim = len(avail)

#     def H_gauss(S):
#         S = np.asarray(S, dtype=float)
#         if S.ndim == 0:
#             return 0.5 * np.log(2 * np.pi * np.e * max(float(S), 1e-12))
#         d = S.shape[0]
#         det = np.linalg.det(S)
#         if det <= 1e-12:
#             return 0.0
#         return float(0.5 * np.log((2 * np.pi * np.e) ** d * det))

#     def diag_stats(diag, hom_dim):
#         lifetimes = []
#         for d, (b, death) in diag:
#             if d != hom_dim:
#                 continue
#             if np.isfinite(b) and np.isfinite(death) and death > b:
#                 lifetimes.append(death - b)
#         if not lifetimes:
#             return 0.0, 0.0
#         lts = np.asarray(lifetimes)
#         T = float(lts.sum())
#         p = lts / T
#         E = float(-(p * np.log(p + 1e-16)).sum())
#         return T, E

#     def landscape_l1(intervals):
#         arr = np.asarray(intervals, dtype=float)
#         if arr.size == 0:
#             return 0.0
#         arr = arr[np.isfinite(arr).all(axis=1)]
#         arr = arr[arr[:, 1] > arr[:, 0]]
#         if len(arr) == 0:
#             return 0.0
#         L = Landscape(num_landscapes=1, resolution=200)
#         vec = L.fit_transform([arr])[0]
#         return float(np.sum(np.abs(vec)))

#     rows = []
#     times = []

#     for s in range(0, n - W + 1, step):
#         e = s + W
#         wnd = R[s:e]
#         A = np.cov(wnd, rowvar=False) + 1e-8 * np.eye(dim)

#         st = gudhi.SimplexTree()

#         for i in range(dim):
#             st.insert([i], filtration=0.0)

#         for i, j in combinations(range(dim), 2):
#             H_i = H_gauss(A[np.ix_([i], [i])])
#             H_j = H_gauss(A[np.ix_([j], [j])])
#             H_ij = H_gauss(A[np.ix_([i, j], [i, j])])
#             w = abs(H_i + H_j - H_ij)
#             st.insert([i, j], filtration=w)

#         for i, j, k in combinations(range(dim), 3):
#             H_i = H_gauss(A[np.ix_([i], [i])])
#             H_j = H_gauss(A[np.ix_([j], [j])])
#             H_k = H_gauss(A[np.ix_([k], [k])])
#             H_ijk = H_gauss(A[np.ix_([i, j, k], [i, j, k])])
#             w = abs(H_i + H_j + H_k - H_ijk)
#             st.insert([i, j, k], filtration=w)

#         st.initialize_filtration()
#         diag = st.persistence()

#         T0, E0 = diag_stats(diag, 0)
#         T1, E1 = diag_stats(diag, 1)

#         d0 = st.persistence_intervals_in_dimension(0)
#         d1 = st.persistence_intervals_in_dimension(1)

#         row = {
#             "BI_tp_H0": T0,
#             "BI_ent_H0": E0,
#             "BI_tp_H1": T1,
#             "BI_ent_H1": E1,
#             "BI_L1_H0": landscape_l1(d0),
#             "BI_L1_H1": landscape_l1(d1),

#             "A_T_H0": T0,
#             "A_E_H0": E0,
#             "BII_tp_H1": T1,
#             "BII_L1_H1": landscape_l1(d1),
#             "BC_ent_H1": E1,
#         }

#         rows.append(row)
#         times.append(idx[e - 1])

#     return pd.DataFrame(rows, index=pd.DatetimeIndex(times)).ffill()

In [ ]:
# feature_windows = {}

# for W in WINDOWS:
#     print(f"\nBuilding features for W={W} bars ({W/48:.1f} days)")

#     X_it_w = build_it_features_for_window(
#         ret_df=ret,
#         assets=ASSETS,
#         W=W,
#         step=STEP,
#         bins=BINS
#     )

#     X_tda_w = build_tda_features_for_window(
#         ret_df=ret,
#         assets=ASSETS,
#         W=W,
#         step=STEP
#     )

#     feature_windows[W] = {
#         "X_it": X_it_w.reindex(ret.index, method="ffill"),
#         "X_tda": X_tda_w.reindex(ret.index, method="ffill"),
#     }

#     print(f"  IT:  {X_it_w.shape}")
#     print(f"  TDA: {X_tda_w.shape}")


Building features for W=96 bars (2.0 days)


In [152]:
# def select_features_for_window(W, X_it, X_tda):
#     split_ts = pd.Timestamp(SELECTION_DATE)

#     perm_scores = {}

#     for pilot_asset in ["BTC", "ETH"]:
#         if pilot_asset not in ret.columns:
#             continue

#         r = ret[pilot_asset]
#         vol_fwd = r.rolling(192).std().shift(-192) * ANN
#         y = np.log(vol_fwd.clip(lower=1e-8)).dropna()

#         idx = (
#             X_vol.index
#             .intersection(X_it.index)
#             .intersection(X_tda.index)
#             .intersection(y.index)
#         )

#         valid = (
#             X_vol.loc[idx].notna().all(1)
#             & X_it.loc[idx].notna().all(1)
#             & X_tda.loc[idx].notna().all(1)
#             & y.loc[idx].notna()
#         )
#         idx = idx[valid]

#         tr = idx < (split_ts - pd.Timedelta(minutes=30 * PURGE_BARS))
#         idx_tr = idx[tr]

#         Xv_tr = X_vol.loc[idx_tr]
#         ytr = y.loc[idx_tr]

#         ck = Xv_tr.corrwith(ytr).abs().dropna()
#         top3 = ck.nlargest(3).index.tolist()

#         mb = _ridge()
#         mb.fit(Xv_tr[top3], ytr)
#         res_tr = ytr.values - mb.predict(Xv_tr[top3])

#         X_all = pd.concat([X_it.loc[idx_tr], X_tda.loc[idx_tr]], axis=1)
#         X_all = X_all.loc[:, ~X_all.columns.duplicated()]

#         mi = mutual_info_regression(
#             X_all.values,
#             res_tr,
#             random_state=42,
#             n_neighbors=5
#         )

#         for feat, val in zip(X_all.columns, mi):
#             perm_scores.setdefault(feat, []).append(float(val))

#     score = pd.Series({k: np.mean(v) for k, v in perm_scores.items()}).sort_values(ascending=False)

#     it_cols = [c for c in score.index if c in X_it.columns][:TOP_IT]
#     tda_cols = [c for c in score.index if c in X_tda.columns][:TOP_TDA]

#     return it_cols, tda_cols, score

# selected_by_window = {}

# for W in WINDOWS:
#     X_it = feature_windows[W]["X_it"]
#     X_tda = feature_windows[W]["X_tda"]

#     it_cols, tda_cols, score = select_features_for_window(W, X_it, X_tda)

#     selected_by_window[W] = {
#         "BEST_IT": it_cols,
#         "BEST_TDA": tda_cols,
#         "score": score,
#     }

#     print(f"\nW={W}")
#     print("  BEST_IT:")
#     for c in it_cols:
#         print(f"    {c}")
#     print("  BEST_TDA:")
#     for c in tda_cols:
#         print(f"    {c}")


W=96
  BEST_IT:
    pairwise_mi_mean
    coinfo3_mean
    coinfo4_mean
    coinfo4_std
    pairwise_mi_max
    pairwise_mi_std
  BEST_TDA:
    BI_tp_H1
    BII_tp_H1
    BII_L1_H1
    BI_L1_H1
    A_T_H0
    BI_tp_H0

W=144
  BEST_IT:
    pairwise_mi_mean
    coinfo4_mean
    coinfo3_mean
    coinfo4_std
    pairwise_mi_min
    pairwise_mi_max
  BEST_TDA:
    BI_tp_H1
    BII_tp_H1
    BI_L1_H1
    BII_L1_H1
    BI_tp_H0
    A_T_H0

W=192
  BEST_IT:
    pairwise_mi_mean
    coinfo4_mean
    coinfo3_mean
    pairwise_mi_min
    pairwise_mi_std
    pairwise_mi_max
  BEST_TDA:
    BII_tp_H1
    BI_tp_H1
    BII_L1_H1
    BI_L1_H1
    BI_tp_H0
    A_T_H0


In [ ]:
# def evaluate_vol_window(W, X_it, X_tda, it_cols, tda_cols):
#     split_ts = pd.Timestamp(FINAL_DATE)
#     rows = []

#     Xi_it = X_it[it_cols].copy()
#     Xi_tda = X_tda[tda_cols].copy()

#     for asset in ASSETS:
#         r = ret[asset]

#         for h, h_label in VOL_HORIZONS.items():
#             vol_fwd = r.rolling(h).std().shift(-h) * ANN
#             y = np.log(vol_fwd.clip(lower=1e-8)).dropna()

#             idx = (
#                 X_vol.index
#                 .intersection(Xi_it.index)
#                 .intersection(Xi_tda.index)
#                 .intersection(y.index)
#             )

#             valid = (
#                 X_vol.loc[idx].notna().all(1)
#                 & Xi_it.loc[idx].notna().all(1)
#                 & Xi_tda.loc[idx].notna().all(1)
#                 & y.loc[idx].notna()
#             )
#             idx = idx[valid]

#             if len(idx) < 1000:
#                 continue

#             tr = idx < (split_ts - pd.Timedelta(minutes=30 * PURGE_BARS))
#             te = idx >= split_ts

#             if tr.sum() < 480 or te.sum() < 100:
#                 continue

#             Xv_tr = X_vol.loc[idx[tr]]
#             Xv_te = X_vol.loc[idx[te]]
#             ytr = y.loc[idx[tr]]
#             yte = y.loc[idx[te]]

#             naive = float(ytr.mean())

#             ck = Xv_tr.corrwith(ytr).abs().dropna()
#             top3 = ck.nlargest(3).index.tolist()

#             mb = _ridge()
#             mb.fit(Xv_tr[top3], ytr)

#             pb_tr = mb.predict(Xv_tr[top3])
#             pb_te = mb.predict(Xv_te[top3])
#             res_tr = ytr.values - pb_tr

#             Xit_tr = Xi_it.loc[idx[tr]]
#             Xit_te = Xi_it.loc[idx[te]]
#             Xtd_tr = Xi_tda.loc[idx[tr]]
#             Xtd_te = Xi_tda.loc[idx[te]]

#             models = {}

#             for name, Xtr, Xte in [
#                 ("B", Xit_tr, Xit_te),
#                 ("C", Xtd_tr, Xtd_te),
#                 ("D", pd.concat([Xit_tr, Xtd_tr], axis=1),
#                       pd.concat([Xit_te, Xtd_te], axis=1)),
#             ]:
#                 m = _ridge()
#                 m.fit(Xtr, res_tr)
#                 pred = pb_te + m.predict(Xte)
#                 models[name] = pred

#             r2_A = oos_r2(yte.values, pb_te, naive)

#             rows.append({
#                 "W": W,
#                 "asset": asset,
#                 "horizon": h_label,
#                 "h": h,
#                 "r2_A": r2_A,
#                 "r2_B": oos_r2(yte.values, models["B"], naive),
#                 "r2_C": oos_r2(yte.values, models["C"], naive),
#                 "r2_D": oos_r2(yte.values, models["D"], naive),
#             })

#     out = pd.DataFrame(rows)
#     for k in ["B", "C", "D"]:
#         out[f"dR2_{k}"] = out[f"r2_{k}"] - out["r2_A"]
#     return out

# vol_window_results = []

# for W in WINDOWS:
#     X_it = feature_windows[W]["X_it"]
#     X_tda = feature_windows[W]["X_tda"]
#     it_cols = selected_by_window[W]["BEST_IT"]
#     tda_cols = selected_by_window[W]["BEST_TDA"]

#     df_w = evaluate_vol_window(W, X_it, X_tda, it_cols, tda_cols)
#     vol_window_results.append(df_w)

#     print(f"\nW={W}")
#     for k in ["B", "C", "D"]:
#         print(
#             f"  {k}: mean ΔR²={df_w[f'dR2_{k}'].mean():+.4f} "
#             f"positive={(df_w[f'dR2_{k}'] > 0).sum()}/{len(df_w)}"
#         )

# vol_window_results = pd.concat(vol_window_results, ignore_index=True)

In [149]:
# vol_window_results

,W,asset,horizon,h,r2_A,r2_B,r2_C,r2_D,dR2_B,dR2_C,dR2_D
0,96,BTC,1d,48,0.35547054,0.29325107,0.37506535,0.34732598,-0.06221948,0.01959481,-0.00814457
1,96,BTC,2d,96,0.34529145,0.21434970,0.37775184,0.31096833,-0.13094176,0.03246039,-0.03432313
2,96,BTC,4d,192,0.39515498,0.24701214,0.46112757,0.35741184,-0.14814283,0.06597260,-0.03774313
3,96,BTC,6d,288,0.42477446,0.32209033,0.49794255,0.43261805,-0.10268413,0.07316809,0.00784359
4,96,BTC,7d,336,0.46637969,0.38889426,0.53612355,0.49085614,-0.07748543,0.06974386,0.02447645
...,...,...,...,...,...,...,...,...,...,...,...
139,192,AVAX,2d,96,0.10555100,0.17165407,0.05732509,0.18045297,0.06610307,-0.04822591,0.07490197
140,192,AVAX,4d,192,0.18828921,0.20068499,0.15492998,0.20841880,0.01239578,-0.03335923,0.02012960
141,192,AVAX,6d,288,0.07014602,0.07512751,0.00258428,0.00955443,0.00498148,-0.06756174,-0.06059160
142,192,AVAX,7d,336,0.04192284,0.06720201,-0.07190967,-0.05522002,0.02527917,-0.11383251,-0.09714286


In [150]:
# vol_window_results[vol_window_results['W'] == 96]

,W,asset,horizon,h,r2_A,r2_B,r2_C,r2_D,dR2_B,dR2_C,dR2_D
0,96,BTC,1d,48,0.35547054,0.29325107,0.37506535,0.34732598,-0.06221948,0.01959481,-0.00814457
1,96,BTC,2d,96,0.34529145,0.21434970,0.37775184,0.31096833,-0.13094176,0.03246039,-0.03432313
2,96,BTC,4d,192,0.39515498,0.24701214,0.46112757,0.35741184,-0.14814283,0.06597260,-0.03774313
3,96,BTC,6d,288,0.42477446,0.32209033,0.49794255,0.43261805,-0.10268413,0.07316809,0.00784359
4,96,BTC,7d,336,0.46637969,0.38889426,0.53612355,0.49085614,-0.07748543,0.06974386,0.02447645
5,96,BTC,14d,672,0.28058987,0.28829747,0.38122081,0.38695526,0.00770759,0.10063094,0.10636538
6,96,ETH,1d,48,0.18640125,0.27357313,0.14441340,0.24875227,0.08717189,-0.04198784,0.06235103
7,96,ETH,2d,96,0.22484595,0.32269573,0.18585423,0.30157976,0.09784978,-0.03899172,0.07673381
8,96,ETH,4d,192,0.25352843,0.36100339,0.23413093,0.36490018,0.10747496,-0.01939750,0.11137175
9,96,ETH,6d,288,0.24255103,0.37070107,0.22144783,0.35646164,0.12815004,-0.02110321,0.11391061


In [ ]:
# def safe_roc(y, s):
#     try:
#         return float(roc_auc_score(y, s))
#     except Exception:
#         return np.nan

# def safe_pr(y, s):
#     try:
#         return float(average_precision_score(y, s))
#     except Exception:
#         return np.nan

# def fit_logit_score(Xtr, ytr, Xte):
#     try:
#         m = _logit()
#         m.fit(Xtr, ytr)
#         return m.predict_proba(Xte)[:, 1]
#     except Exception:
#         return np.full(len(Xte), float(np.mean(ytr)))

# def evaluate_stress_window(W, X_it, X_tda, it_cols, tda_cols):
#     split_ts = pd.Timestamp(FINAL_DATE)
#     rows = []

#     Xi_it = X_it[it_cols].copy()
#     Xi_tda = X_tda[tda_cols].copy()

#     for asset in ASSETS:
#         r = ret[asset]
#         vol_now = r.rolling(VOL_WINDOW).std() * ANN
#         vol_rank = vol_now.expanding(min_periods=100).rank(pct=True)

#         for h, h_label in STRESS_HORIZONS.items():
#             vol_fwd = r.rolling(h).std().shift(-h) * ANN

#             for q in STRESS_QUANTILES:
#                 vol_q = vol_now.expanding(min_periods=100).quantile(q)
#                 y = (vol_fwd > vol_q).astype(float).dropna()

#                 idx = (
#                     Xi_it.index
#                     .intersection(Xi_tda.index)
#                     .intersection(vol_rank.index)
#                     .intersection(y.index)
#                 )

#                 valid = (
#                     Xi_it.loc[idx].notna().all(1)
#                     & Xi_tda.loc[idx].notna().all(1)
#                     & vol_rank.loc[idx].notna()
#                     & y.loc[idx].notna()
#                 )
#                 idx = idx[valid]

#                 if len(idx) < 1000:
#                     continue

#                 tr = idx < (split_ts - pd.Timedelta(minutes=30 * PURGE_BARS))
#                 te = idx >= split_ts

#                 if tr.sum() < 480 or te.sum() < 100:
#                     continue

#                 ytr = y.loc[idx[tr]].astype(int)
#                 yte = y.loc[idx[te]].astype(int)

#                 if ytr.mean() < 0.03 or ytr.mean() > 0.97:
#                     continue
#                 if yte.sum() < 5:
#                     continue

#                 Xvol_tr = pd.DataFrame({"vol_rank": vol_rank.loc[idx[tr]]}, index=idx[tr])
#                 Xvol_te = pd.DataFrame({"vol_rank": vol_rank.loc[idx[te]]}, index=idx[te])

#                 Xit_tr = Xi_it.loc[idx[tr]]
#                 Xit_te = Xi_it.loc[idx[te]]
#                 Xtd_tr = Xi_tda.loc[idx[tr]]
#                 Xtd_te = Xi_tda.loc[idx[te]]

#                 sc_A = fit_logit_score(Xvol_tr, ytr, Xvol_te)
#                 sc_B = fit_logit_score(pd.concat([Xvol_tr, Xit_tr], axis=1), ytr,
#                                         pd.concat([Xvol_te, Xit_te], axis=1))
#                 sc_C = fit_logit_score(pd.concat([Xvol_tr, Xtd_tr], axis=1), ytr,
#                                         pd.concat([Xvol_te, Xtd_te], axis=1))
#                 sc_D = fit_logit_score(pd.concat([Xvol_tr, Xit_tr, Xtd_tr], axis=1), ytr,
#                                         pd.concat([Xvol_te, Xit_te, Xtd_te], axis=1))

#                 rows.append({
#                     "W": W,
#                     "asset": asset,
#                     "horizon": h_label,
#                     "q": q,
#                     "pos_te": float(yte.mean()),
#                     "roc_A": safe_roc(yte, sc_A),
#                     "roc_B": safe_roc(yte, sc_B),
#                     "roc_C": safe_roc(yte, sc_C),
#                     "roc_D": safe_roc(yte, sc_D),
#                     "pr_A": safe_pr(yte, sc_A),
#                     "pr_B": safe_pr(yte, sc_B),
#                     "pr_C": safe_pr(yte, sc_C),
#                     "pr_D": safe_pr(yte, sc_D),
#                 })

#     out = pd.DataFrame(rows)
#     for k in ["B", "C", "D"]:
#         out[f"droc_{k}"] = out[f"roc_{k}"] - out["roc_A"]
#         out[f"dpr_{k}"] = out[f"pr_{k}"] - out["pr_A"]
#     return out

# stress_window_results = []

# for W in WINDOWS:
#     X_it = feature_windows[W]["X_it"]
#     X_tda = feature_windows[W]["X_tda"]
#     it_cols = selected_by_window[W]["BEST_IT"]
#     tda_cols = selected_by_window[W]["BEST_TDA"]

#     df_w = evaluate_stress_window(W, X_it, X_tda, it_cols, tda_cols)
#     stress_window_results.append(df_w)

#     print(f"\nW={W}")
#     for k in ["B", "C", "D"]:
#         print(
#             f"  {k}: mean ΔROC={df_w[f'droc_{k}'].mean():+.4f} "
#             f"ROC↑={(df_w[f'droc_{k}'] > 0.01).sum()}/{len(df_w)} "
#             f"mean ΔPR={df_w[f'dpr_{k}'].mean():+.4f}"
#         )

# stress_window_results = pd.concat(stress_window_results, ignore_index=True)

In [146]:
# stress_window_results

,W,asset,horizon,q,pos_te,roc_A,roc_B,roc_C,roc_D,pr_A,pr_B,pr_C,pr_D,droc_B,dpr_B,droc_C,dpr_C,droc_D,dpr_D
0,96,BTC,1d,0.6,0.11891166,0.61348975,0.60685354,0.61890146,0.61262711,0.14280794,0.14142638,0.13667183,0.13615380,-0.00663621,-0.00138157,0.00541171,-0.00613611,-0.00086264,-0.00665414
1,96,BTC,1d,0.7,0.05408129,0.58272604,0.56648345,0.56513137,0.56547104,0.06135840,0.05869914,0.05606816,0.05653959,-0.01624259,-0.00265927,-0.01759467,-0.00529024,-0.01725499,-0.00481882
2,96,BTC,1d,0.8,0.00806181,0.35063213,0.30535331,0.50163675,0.49873010,0.00623391,0.00573983,0.00820496,0.00799495,-0.04527881,-0.00049408,0.15100463,0.00197106,0.14809798,0.00176104
3,96,BTC,2d,0.6,0.05844810,0.40807263,0.40522060,0.43948602,0.41459069,0.04934264,0.04559961,0.05283227,0.04580201,-0.00285203,-0.00374303,0.03141339,0.00348963,0.00651806,-0.00354063
4,96,ETH,1d,0.6,0.63990595,0.65744555,0.66772829,0.64359600,0.64872096,0.72568673,0.74251146,0.70623606,0.71231695,0.01028274,0.01682473,-0.01384955,-0.01945067,-0.00872459,-0.01336977
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
235,192,AVAX,2d,0.7,0.18374202,0.61298365,0.61931298,0.65800814,0.66791177,0.24607801,0.23481919,0.25991112,0.27985321,0.00632932,-0.01125882,0.04502449,0.01383311,0.05492812,0.03377520
236,192,AVAX,2d,0.8,0.07490763,0.64723468,0.74808432,0.71899007,0.81212325,0.16355231,0.14800808,0.16492491,0.19017946,0.10084964,-0.01554424,0.07175539,0.00137259,0.16488858,0.02662714
237,192,AVAX,4d,0.6,0.37050722,0.68496562,0.69710434,0.73926257,0.74598625,0.51192314,0.59260296,0.56044112,0.66732190,0.01213872,0.08067982,0.05429695,0.04851798,0.06102064,0.15539877
238,192,AVAX,4d,0.7,0.24454148,0.65929058,0.72772514,0.70784329,0.75311860,0.32187595,0.41388090,0.37020007,0.44572767,0.06843457,0.09200495,0.04855271,0.04832412,0.09382803,0.12385172


In [145]:
# import matplotlib.pyplot as plt

# fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ax = axes[0]
# for k, label in [("B", "+IT"), ("C", "+TDA"), ("D", "+IT+TDA")]:
#     s = vol_window_results.groupby("W")[f"dR2_{k}"].mean()
#     ax.plot(s.index, s.values, "o-", label=label)

# ax.axhline(0, color="black", lw=0.8)
# ax.set_title("Volatility prediction: mean ΔR² by window")
# ax.set_xlabel("Feature window W")
# ax.set_ylabel("Mean ΔR² vs baseline")
# ax.legend()
# ax.grid(alpha=0.2)

# ax = axes[1]
# for k, label in [("B", "+IT"), ("C", "+TDA"), ("D", "+IT+TDA")]:
#     s = stress_window_results.groupby("W")[f"droc_{k}"].mean()
#     ax.plot(s.index, s.values, "o-", label=label)

# ax.axhline(0, color="black", lw=0.8)
# ax.set_title("Stress prediction: mean ΔROC by window")
# ax.set_xlabel("Feature window W")
# ax.set_ylabel("Mean ΔROC vs baseline")
# ax.legend()
# ax.grid(alpha=0.2)

# plt.tight_layout()
# plt.savefig("fig_window_selection_summary.png", bbox_inches="tight", dpi=150)
# plt.show()

# 96

In [157]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from itertools import combinations
from sklearn.metrics import mutual_info_score
import gudhi
from gudhi.representations import Landscape

FEATURE_WINDOW = 96
FEATURE_STEP = 4
TDA_STEP = 48
BINS = 10

ASSETS_FEATURE = [a for a in ["BTC","ETH","SOL","BNB","XRP","DOGE","ADA","AVAX","TRX"] if a in ret.columns]

print("=" * 80)
print(f"BUILDING W={FEATURE_WINDOW} FEATURES")
print("=" * 80)

def _rank_bins(x, bins=BINS):
    x = np.asarray(x, dtype=float)
    ranks = pd.Series(x).rank(method="average").values
    scaled = np.floor((ranks - 1) / max(1, len(x)) * bins).astype(int)
    return np.clip(scaled, 0, bins - 1)

def mi_hist_rank(x, y, bins=BINS):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    x, y = x[mask], y[mask]
    if len(x) < max(20, bins):
        return 0.0
    bx = _rank_bins(x, bins)
    by = _rank_bins(y, bins)
    c = np.histogram2d(bx, by, bins=bins, range=[[0, bins], [0, bins]])[0]
    return float(mutual_info_score(None, None, contingency=c))

def kl_hist_rank(x_now, x_prev, bins=BINS):
    bx = _rank_bins(x_now, bins)
    bp = _rank_bins(x_prev, bins)
    p = np.bincount(bx, minlength=bins).astype(float) + 1e-8
    q = np.bincount(bp, minlength=bins).astype(float) + 1e-8
    p /= p.sum()
    q /= q.sum()
    return float(np.sum(p * np.log(p / q)))

def build_it_full_for_window(ret_df, assets, window=96, step=4, bins=10):
    avail = [a for a in assets if a in ret_df.columns]
    R = ret_df[avail].replace([np.inf, -np.inf], np.nan).fillna(0.0).values
    idx = ret_df.index
    n = len(R)

    pairs = list(combinations(range(len(avail)), 2))
    triples = list(combinations(range(len(avail)), 3))
    quads = list(combinations(range(len(avail)), 4))

    rows = []
    times = []

    for s in range(0, n - window + 1, step):
        e = s + window
        wnd = R[s:e]

        mi_vals = []
        cov_vals = []

        for i, j in pairs:
            mi_vals.append(mi_hist_rank(wnd[:, i], wnd[:, j], bins=bins))
            cov_vals.append(float(np.cov(wnd[:, i], wnd[:, j])[0, 1]))

        coinfo3_vals = []
        for i, j, k in triples:
            mij = mi_hist_rank(wnd[:, i], wnd[:, j], bins=bins)
            mik = mi_hist_rank(wnd[:, i], wnd[:, k], bins=bins)
            mjk = mi_hist_rank(wnd[:, j], wnd[:, k], bins=bins)
            mijk_proxy = mi_hist_rank(wnd[:, i], wnd[:, j] + wnd[:, k], bins=bins)
            coinfo3_vals.append(mij + mik + mjk - mijk_proxy)

        coinfo4_vals = []
        for i, j, k, l in quads[:50]:
            a = mi_hist_rank(wnd[:, i], wnd[:, j], bins=bins)
            b = mi_hist_rank(wnd[:, k], wnd[:, l], bins=bins)
            coinfo4_vals.append(a - b)

        row = {
            "pairwise_mi_mean": float(np.mean(mi_vals)),
            "pairwise_mi_std":  float(np.std(mi_vals)),
            "pairwise_mi_min":  float(np.min(mi_vals)),
            "pairwise_mi_max":  float(np.max(mi_vals)),

            "pairwise_cov_mean": float(np.mean(cov_vals)),
            "pairwise_cov_std":  float(np.std(cov_vals)),
            "pairwise_cov_min":  float(np.min(cov_vals)),
            "pairwise_cov_max":  float(np.max(cov_vals)),

            "coinfo3_mean": float(np.mean(coinfo3_vals)) if coinfo3_vals else 0.0,
            "coinfo3_std":  float(np.std(coinfo3_vals)) if coinfo3_vals else 0.0,
            "coinfo3_min":  float(np.min(coinfo3_vals)) if coinfo3_vals else 0.0,
            "coinfo3_max":  float(np.max(coinfo3_vals)) if coinfo3_vals else 0.0,

            "coinfo4_mean": float(np.mean(coinfo4_vals)) if coinfo4_vals else 0.0,
            "coinfo4_std":  float(np.std(coinfo4_vals)) if coinfo4_vals else 0.0,
            "coinfo4_min":  float(np.min(coinfo4_vals)) if coinfo4_vals else 0.0,
            "coinfo4_max":  float(np.max(coinfo4_vals)) if coinfo4_vals else 0.0,
        }

        for ai, asset in enumerate(avail):
            if s >= window:
                row[f"kl_{asset}"] = kl_hist_rank(R[s:e, ai], R[s-window:s, ai], bins=bins)
            else:
                row[f"kl_{asset}"] = 0.0

        rows.append(row)
        times.append(idx[e - 1])

    return pd.DataFrame(rows, index=pd.DatetimeIndex(times)).sort_index()

X_it_full_W96 = build_it_full_for_window(
    ret_df=ret,
    assets=ASSETS_FEATURE,
    window=FEATURE_WINDOW,
    step=FEATURE_STEP,
    bins=BINS,
)

X_it_full_W96 = X_it_full_W96.reindex(ret.index).ffill()

print(f"X_it_full_W96: {X_it_full_W96.shape}")

def diag_for_dim(diag, dim):
    return [(float(b), float(d)) for dd, (b, d) in diag if dd == dim]

def diagram_statistics(diag_bd, max_filtration):
    lifetimes = []
    for b, d in diag_bd:
        lt = (d - b) if np.isfinite(d) else (max_filtration - b)
        if lt > 0:
            lifetimes.append(lt)

    if not lifetimes:
        return 0.0, 0.0

    lts = np.asarray(lifetimes, dtype=float)
    T = float(lts.sum())
    p = lts / T
    E = float(-(p * np.log(p)).sum())
    return T, E

def landscape_norms(diag_bd, max_filtration, resolution=200):
    if len(diag_bd) == 0:
        return 0.0, 0.0

    pts = [[b, d if np.isfinite(d) else max_filtration] for b, d in diag_bd]
    arr = np.asarray(pts, dtype=float)
    arr = arr[np.isfinite(arr).all(axis=1)]
    arr = arr[arr[:, 1] > arr[:, 0]]

    if len(arr) == 0:
        return 0.0, 0.0

    try:
        L = Landscape(num_landscapes=1, resolution=resolution)
        vec = L.fit_transform([arr])[0]
        return float(np.sum(np.abs(vec))), float(np.sqrt(np.sum(vec ** 2)))
    except Exception:
        return 0.0, 0.0

def H_gaussian(S):
    S = np.asarray(S)
    if S.ndim == 0:
        return 0.5 * np.log(2 * np.pi * np.e * float(S)) if float(S) > 0 else 0.0

    n = S.shape[0]
    det = np.linalg.det(S)
    if det <= 0 or not np.isfinite(det):
        return 0.0
    return float(0.5 * np.log((2 * np.pi * np.e) ** n * det))

def I(simplex, A=None):
    d = len(simplex)
    H_marginal_sum = d * H_gaussian(A[0, 0])
    H_joint = H_gaussian(A[np.ix_(simplex, simplex)])
    return H_marginal_sum - H_joint

def II(simplex, A=None):
    d = len(simplex)
    H_joint = H_gaussian(A[np.ix_(simplex, simplex)])
    H_residual_sum = 0.0
    for idx in combinations(simplex, d - 1):
        H_residual_sum += H_gaussian(A[np.ix_(list(idx), list(idx))])
    return -(d - 1) * H_joint + H_residual_sum

def C(simplex, A=None):
    d = len(simplex)
    coeffs = []
    vals = []
    for l in range(1, d + 1):
        sign = 0 - (-1) ** l
        for idx in combinations(simplex, l):
            idx_list = list(idx)
            coeffs.append(sign)
            if len(idx_list) == 1:
                vals.append(H_gaussian(A[idx_list[0], idx_list[0]]))
            else:
                vals.append(H_gaussian(A[np.ix_(idx_list, idx_list)]))
    return float(np.asarray(coeffs) @ np.asarray(vals))

def build_mi_complex_gudhi(X_window, bins=BINS):
    dim = X_window.shape[1]
    MI = np.zeros((dim, dim))

    for i, j in combinations(range(dim), 2):
        mi = mi_hist_rank(X_window[:, i], X_window[:, j], bins=bins)
        MI[i, j] = MI[j, i] = mi

    triu = MI[np.triu_indices_from(MI, 1)]
    MI_max = float(np.nanmax(triu)) if len(triu) else 1.0

    st = gudhi.SimplexTree()

    for i in range(dim):
        st.insert([i], filtration=0.0)

    edge_w = np.zeros((dim, dim))
    for i, j in combinations(range(dim), 2):
        w = MI_max - MI[i, j]
        edge_w[i, j] = edge_w[j, i] = w
        st.insert([i, j], filtration=float(w))

    for i, j, k in combinations(range(dim), 3):
        w = max(edge_w[i, j], edge_w[i, k], edge_w[j, k])
        st.insert([i, j, k], filtration=float(w))

    st.make_filtration_non_decreasing()
    return st, MI, MI_max

def build_gaussian_info_complex(X_window, triple_mode="I", eps=1e-8):
    dim = X_window.shape[1]
    A = np.cov(X_window, rowvar=False) + eps * np.eye(dim)
    triple_fn = {"I": I, "II": II, "C": C}[triple_mode]

    st = gudhi.SimplexTree()

    for i in range(dim):
        st.insert([i], filtration=0.0)

    for i, j in combinations(range(dim), 2):
        st.insert([i, j], filtration=float(I([i, j], A=A)))

    for i, j, k in combinations(range(dim), 3):
        st.insert([i, j, k], filtration=float(triple_fn([i, j, k], A=A)))

    st.make_filtration_non_decreasing()
    return st, A

def build_tda_A_B_for_window(ret_df, assets, window=96, step=48, bins=10):
    avail = [a for a in assets if a in ret_df.columns]
    R = ret_df[avail].replace([np.inf, -np.inf], np.nan).fillna(0.0).values
    idx = ret_df.index
    n = len(R)

    starts = list(range(0, n - window, step))
    times = [idx[t + window] for t in starts]

    print(f"TDA windows: {len(starts)} | {times[0]} -> {times[-1]}")

    rows_A = []

    for t in starts:
        wnd = R[t:t + window]

        try:
            st, MI, MI_max = build_mi_complex_gudhi(wnd, bins=bins)
            diag = st.persistence()

            d0 = diag_for_dim(diag, 0)
            d1 = diag_for_dim(diag, 1)

            T0, E0 = diagram_statistics(d0, MI_max)
            T1, E1 = diagram_statistics(d1, MI_max)

            L1_0, L2_0 = landscape_norms(d0, MI_max)
            L1_1, L2_1 = landscape_norms(d1, MI_max)

        except Exception:
            T0 = E0 = T1 = E1 = L1_0 = L2_0 = L1_1 = L2_1 = 0.0

        rows_A.append({
            "T_H0": T0,
            "E_H0": E0,
            "T_H1": T1,
            "E_H1": E1,
            "L1_H0": L1_0,
            "L2_H0": L2_0,
            "L1_H1": L1_1,
            "L2_H1": L2_1,
        })

    tda_A_w = pd.DataFrame(rows_A, index=pd.DatetimeIndex(times))

    global_max = 0.0

    for t in starts:
        wnd = R[t:t + window]
        try:
            st, A = build_gaussian_info_complex(wnd, triple_mode="I")
            diag = st.persistence()
            finite_deaths = [d for _, (_, d) in diag if np.isfinite(d)]
            if finite_deaths:
                global_max = max(global_max, max(finite_deaths))
        except Exception:
            pass

    global_max = float(global_max) if global_max > 0 else 1.0

    tda_B_w = {}

    for mode in ["I", "II", "C"]:
        rows = []

        for t in starts:
            wnd = R[t:t + window]

            try:
                st, A = build_gaussian_info_complex(wnd, triple_mode=mode)
                diag = st.persistence()

                d0 = diag_for_dim(diag, 0)
                d1 = diag_for_dim(diag, 1)

                T0, E0 = diagram_statistics(d0, global_max)
                T1, E1 = diagram_statistics(d1, global_max)

                L1_0, L2_0 = landscape_norms(d0, global_max)
                L1_1, L2_1 = landscape_norms(d1, global_max)

            except Exception:
                T0 = E0 = T1 = E1 = L1_0 = L2_0 = L1_1 = L2_1 = 0.0

            rows.append({
                "tp_H0": T0,
                "ent_H0": E0,
                "tp_H1": T1,
                "ent_H1": E1,
                "L1_H0": L1_0,
                "L2_H0": L2_0,
                "L1_H1": L1_1,
                "L2_H1": L2_1,
            })

        df_mode = pd.DataFrame(rows, index=pd.DatetimeIndex(times))

        tda_B_w[mode] = {
            "times": list(pd.DatetimeIndex(times)),
            "tp_H0": df_mode["tp_H0"].values,
            "ent_H0": df_mode["ent_H0"].values,
            "tp_H1": df_mode["tp_H1"].values,
            "ent_H1": df_mode["ent_H1"].values,
            "L1_H0": df_mode["L1_H0"].values,
            "L2_H0": df_mode["L2_H0"].values,
            "L1_H1": df_mode["L1_H1"].values,
            "L2_H1": df_mode["L2_H1"].values,
        }

    return tda_A_w, tda_B_w

tda_A_W96, tda_B_W96 = build_tda_A_B_for_window(
    ret_df=ret,
    assets=ASSETS_FEATURE,
    window=FEATURE_WINDOW,
    step=TDA_STEP,
    bins=BINS,
)

print(f"tda_A_W96: {tda_A_W96.shape}")
print("tda_B_W96 modes:", list(tda_B_W96.keys()))

BUILDING W=96 FEATURES
X_it_full_W96: (46743, 25)
TDA windows: 972 | 2023-01-03 00:00:00 -> 2025-08-31 05:00:00
tda_A_W96: (972, 8)
tda_B_W96 modes: ['I', 'II', 'C']


In [158]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mutual_info_score
from sklearn.feature_selection import mutual_info_regression

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

FEATURE_WINDOW = 96

SPLIT_DATE = "2025-01-01"
PURGE_BARS = 96

HORIZONS = {
    96:  "2d",
    192: "4d",
}

ASSETS = ["BTC", "ETH", "DOGE", "XRP", "SOL", "BNB"]

VOL_WINDOW = 48
ANN = np.sqrt(365 * 48)

RNG = np.random.default_rng(42)
N_PERM = 10

IT_WINDOW = FEATURE_WINDOW
IT_STEP = 4

INDEX_WINDOW_BARS = 365 * 48
INDEX_LAG_BARS = 48
INDEX_FEATURE_STEP = 48
ENTROPY_BINS = 32

MI_ESTIMATORS = {
    "hist_rank": {"kind": "hist_rank", "bins": 10},
    "gaussian": {"kind": "gaussian"},
    "knn": {"kind": "knn", "n_neighbors": 5, "random_state": 42},
}

EXTRA_MI_FEATURES = ["gaussian", "knn"]
INDEX_NMI_ESTIMATOR = "hist_rank"
INDEX_FEATURE_TARGETS = {"index": None}
INDEX_WEIGHTS = None
EXTRA_TDA_FROM_MI = []

def _ridge():
    return Pipeline([
        ("sc", StandardScaler()),
        ("m", RidgeCV(alphas=[0.01, 0.1, 1, 10, 100]))
    ])

def oos_r2(y_true, y_pred, naive_val):
    mse_m = np.mean((y_true - y_pred) ** 2)
    mse_n = np.mean((y_true - naive_val) ** 2)
    return float(1.0 - mse_m / mse_n) if mse_n > 1e-12 else np.nan

def _drop_finite_pairs(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    return x[mask], y[mask]

def _rank_to_bins(x, bins=10):
    x = np.asarray(x, dtype=float)
    ranks = pd.Series(x).rank(method="average").values
    scaled = np.floor((ranks - 1) / max(1, len(x)) * bins).astype(int)
    return np.clip(scaled, 0, bins - 1)

def mi_hist_rank(x, y, bins=10):
    x, y = _drop_finite_pairs(x, y)
    if len(x) < max(20, bins):
        return np.nan
    bx = _rank_to_bins(x, bins=bins)
    by = _rank_to_bins(y, bins=bins)
    c = np.histogram2d(bx, by, bins=bins, range=[[0, bins], [0, bins]])[0]
    return float(mutual_info_score(None, None, contingency=c))

def mi_gaussian(x, y):
    x, y = _drop_finite_pairs(x, y)
    if len(x) < 10:
        return np.nan
    r = np.corrcoef(x, y)[0, 1]
    if not np.isfinite(r):
        return np.nan
    r = np.clip(r, -0.999999, 0.999999)
    return float(-0.5 * np.log(1.0 - r * r))

def mi_knn_sym(x, y, n_neighbors=5, random_state=42):
    x, y = _drop_finite_pairs(x, y)
    if len(x) < max(20, n_neighbors + 2):
        return np.nan

    x2 = x.reshape(-1, 1)
    y2 = y.reshape(-1, 1)

    mi_xy = mutual_info_regression(
        x2, y,
        discrete_features=False,
        n_neighbors=n_neighbors,
        random_state=random_state
    )[0]

    mi_yx = mutual_info_regression(
        y2, x,
        discrete_features=False,
        n_neighbors=n_neighbors,
        random_state=random_state
    )[0]

    return float(0.5 * (mi_xy + mi_yx))

def estimate_mi(x, y, estimator_name="hist_rank"):
    cfg = MI_ESTIMATORS[estimator_name]
    kind = cfg["kind"]

    if kind == "hist_rank":
        return mi_hist_rank(x, y, bins=cfg.get("bins", 10))

    if kind == "gaussian":
        return mi_gaussian(x, y)

    if kind == "knn":
        return mi_knn_sym(
            x, y,
            n_neighbors=cfg.get("n_neighbors", 5),
            random_state=cfg.get("random_state", 42)
        )

    raise ValueError(f"Unknown MI estimator: {estimator_name}")

def nmi_from_mi(mi):
    if not np.isfinite(mi):
        return np.nan
    return float(1.0 - np.exp(-2.0 * mi))

def build_weighted_index_returns(ret_df, assets, weights=None):
    avail = [a for a in assets if a in ret_df.columns]

    if weights is None:
        w = pd.Series(1.0 / len(avail), index=avail)
    else:
        w = pd.Series(weights, dtype=float).reindex(avail).fillna(0.0)
        w = w / w.sum()

    idx = ret_df[avail].mul(w, axis=1).sum(axis=1)
    idx.name = "weighted_index_ret"
    return idx

def _rolling_hist_entropy_raw(series, window, step, bins=32):
    x = series.astype(float).values
    n = len(x)
    res = np.full(n, np.nan)

    for s in np.arange(0, n - window + 1, step):
        e = s + window
        xw = x[s:e]
        xw = xw[np.isfinite(xw)]

        if len(xw) < 100:
            continue

        counts, _ = np.histogram(xw, bins=bins)
        p = counts.astype(float)

        if p.sum() <= 0:
            continue

        p /= p.sum()
        p = p[p > 0]
        res[e - 1] = -np.sum(p * np.log(p))

    return pd.Series(res, index=series.index).ffill()

def _rolling_nmi_lag(series, window, lag_bars, step, estimator_name):
    x = series.astype(float).values
    n = len(x)
    res = np.full(n, np.nan)

    for s in np.arange(0, n - window + 1, step):
        e = s + window
        xw = x[s:e]

        if len(xw) <= lag_bars + 5:
            continue

        x_now = xw[lag_bars:]
        x_lag = xw[:-lag_bars]

        mi = estimate_mi(x_now, x_lag, estimator_name=estimator_name)
        res[e - 1] = nmi_from_mi(mi)

    return pd.Series(res, index=series.index).ffill()

def build_index_feature_block(ret_df, assets, targets_dict, weights=None):
    frames = []
    weighted_index = build_weighted_index_returns(ret_df, assets, weights=weights)

    for target_name, target_spec in targets_dict.items():
        if target_spec is None:
            s = weighted_index.copy()
            base_name = target_name
        else:
            if target_spec not in ret_df.columns:
                continue
            s = ret_df[target_spec].copy()
            base_name = target_name

        ent = _rolling_hist_entropy_raw(
            s,
            window=INDEX_WINDOW_BARS,
            step=INDEX_FEATURE_STEP,
            bins=ENTROPY_BINS
        )
        ent.name = f"entropy_{base_name}_1y"

        nmi = _rolling_nmi_lag(
            s,
            window=INDEX_WINDOW_BARS,
            lag_bars=INDEX_LAG_BARS,
            step=INDEX_FEATURE_STEP,
            estimator_name=INDEX_NMI_ESTIMATOR
        )
        nmi.name = f"nmi_{base_name}_1d_{INDEX_NMI_ESTIMATOR}"

        frames.append(ent.to_frame())
        frames.append(nmi.to_frame())

    return pd.concat(frames, axis=1).reindex(ret_df.index).ffill() if frames else pd.DataFrame(index=ret_df.index)

def build_pairwise_mi_aggregates(ret_df, assets, estimator_name, window=96, step=4, prefix=None):
    avail = [a for a in assets if a in ret_df.columns]
    idx = ret_df.index
    n = len(idx)

    if prefix is None:
        prefix = estimator_name

    out = pd.DataFrame(index=idx, columns=[
        f"{prefix}_pairwise_mi_mean",
        f"{prefix}_pairwise_mi_std",
        f"{prefix}_pairwise_mi_max",
        f"{prefix}_pairwise_mi_min",
    ], dtype=float)

    pairs = [(avail[i], avail[j]) for i in range(len(avail)) for j in range(i + 1, len(avail))]

    for s in np.arange(0, n - window + 1, step):
        e = s + window
        sub = ret_df.iloc[s:e]
        vals = []

        for a, b in pairs:
            mi = estimate_mi(sub[a].values, sub[b].values, estimator_name=estimator_name)
            if np.isfinite(mi):
                vals.append(mi)

        if vals:
            vals = np.asarray(vals, dtype=float)
            out.iloc[e - 1] = [vals.mean(), vals.std(ddof=0), vals.max(), vals.min()]

    return out.ffill()

print("=" * 80)
print("FEATURE ANALYSIS W=96")
print("=" * 80)

tda_frames = [tda_A_W96.add_prefix("A_")]

for mode in ["I", "II", "C"]:
    d = tda_B_W96[mode]
    idx_tda = pd.DatetimeIndex(d["times"])

    df_m = pd.DataFrame({
        f"B{mode}_tp_H0":  d["tp_H0"],
        f"B{mode}_ent_H0": d["ent_H0"],
        f"B{mode}_tp_H1":  d["tp_H1"],
        f"B{mode}_ent_H1": d["ent_H1"],
        f"B{mode}_L1_H0":  d["L1_H0"],
        f"B{mode}_L1_H1":  d["L1_H1"],
    }, index=idx_tda)

    tda_frames.append(df_m)

X_tda_raw_W96 = pd.concat(tda_frames, axis=1).sort_index()
X_tda_full_W96 = X_tda_raw_W96.reindex(ret.index, method="ffill")

X_it_base_W96 = X_it_full_W96.copy()

PREDEFINED_IT_SLIM = [
    "coinfo3_std", "kl_BNB", "coinfo4_std", "pairwise_mi_mean",
    "coinfo4_mean", "pairwise_mi_std", "coinfo3_mean",
    "coinfo4_max", "kl_AVAX", "pairwise_mi_min"
]

IT_SLIM_W96 = [c for c in PREDEFINED_IT_SLIM if c in X_it_base_W96.columns]
IT_OTHER_BASE_W96 = [c for c in X_it_base_W96.columns if c not in IT_SLIM_W96]

extra_it_frames = []
extra_it_cols_W96 = []

for est_name in EXTRA_MI_FEATURES:
    print(f"Building extra standalone MI features W96: {est_name}")
    df_extra = build_pairwise_mi_aggregates(
        ret_df=ret,
        assets=ASSETS,
        estimator_name=est_name,
        window=IT_WINDOW,
        step=IT_STEP,
        prefix=est_name,
    )
    extra_it_frames.append(df_extra)
    extra_it_cols_W96.extend(list(df_extra.columns))

print("Building index / entropy / NMI features...")
df_index_extra = build_index_feature_block(
    ret_df=ret,
    assets=ASSETS,
    targets_dict=INDEX_FEATURE_TARGETS,
    weights=INDEX_WEIGHTS,
)

if len(df_index_extra.columns) > 0:
    extra_it_frames.append(df_index_extra)
    extra_it_cols_W96.extend(list(df_index_extra.columns))

X_it_aug_W96 = pd.concat([X_it_base_W96] + extra_it_frames, axis=1)
X_it_aug_W96 = X_it_aug_W96.reindex(ret.index).ffill()
X_it_aug_W96 = X_it_aug_W96.loc[:, ~X_it_aug_W96.columns.duplicated()]

X_tda_full_W96 = X_tda_full_W96.loc[:, ~X_tda_full_W96.columns.duplicated()]

IT_ALL_W96 = list(X_it_aug_W96.columns)
TDA_COLS_W96 = list(X_tda_full_W96.columns)

print(f"Base IT W96:      {len(X_it_base_W96.columns)}")
print(f"IT slim W96:      {len(IT_SLIM_W96)}")
print(f"Extra IT W96:     {len(extra_it_cols_W96)}")
print(f"Total IT W96:     {len(IT_ALL_W96)}")
print(f"Total TDA W96:    {len(TDA_COLS_W96)}")

split_ts = pd.Timestamp(SPLIT_DATE)
vol_btc = ret["BTC"].rolling(VOL_WINDOW).std() * ANN
vol_test = vol_btc[vol_btc.index >= split_ts]

it_corrs_W96 = {}
for col in IT_ALL_W96:
    s = X_it_aug_W96[col].reindex(vol_test.index, method="ffill")
    it_corrs_W96[col] = float(s.corr(vol_test))

tda_corrs_W96 = {}
for col in TDA_COLS_W96:
    s = X_tda_full_W96[col].reindex(vol_test.index, method="ffill")
    tda_corrs_W96[col] = float(s.corr(vol_test))

print("\n" + "=" * 65)
print("PERMUTATION IMPORTANCE W=96 ON TRAIN")
print("=" * 65)

perm_results_W96 = {}

for pilot_asset in ["BTC", "ETH"]:
    r = ret[pilot_asset]
    vol_fwd = r.rolling(192).std().shift(-192) * ANN
    y_logvol = np.log(vol_fwd.clip(lower=1e-8)).dropna()

    idx = (
        X_vol.index
        .intersection(X_it_aug_W96.index)
        .intersection(X_tda_full_W96.index)
        .intersection(y_logvol.index)
    )

    valid = (
        X_vol.loc[idx].notna().all(1)
        & X_it_aug_W96.loc[idx].notna().all(1)
        & X_tda_full_W96.loc[idx].notna().all(1)
        & y_logvol.loc[idx].notna()
    )

    idx = idx[valid]
    tr = idx < (split_ts - pd.Timedelta(minutes=30 * PURGE_BARS))
    idx_tr = idx[tr]

    Xv_tr = X_vol.loc[idx_tr]
    ytr = y_logvol.loc[idx_tr]

    ck = Xv_tr.corrwith(ytr).abs().dropna()
    top3 = ck.nlargest(3).index.tolist()

    mdl_b = _ridge()
    mdl_b.fit(Xv_tr[top3], ytr)

    res_tr = ytr.values - mdl_b.predict(Xv_tr[top3])

    Xi_all_tr = pd.concat([
        X_it_aug_W96.loc[idx_tr],
        X_tda_full_W96.loc[idx_tr],
    ], axis=1)

    Xi_all_tr = Xi_all_tr.loc[:, ~Xi_all_tr.columns.duplicated()]

    Xi_arr = Xi_all_tr.values
    feat_names = list(Xi_all_tr.columns)

    kf = KFold(n_splits=5, shuffle=False)

    base_cv = float(np.mean(
        cross_val_score(_ridge(), Xi_arr, res_tr, cv=kf, scoring="r2")
    ))

    perm_scores_asset = {}

    for fi, feat in enumerate(feat_names):
        deltas = []

        for _ in range(N_PERM):
            Xp = Xi_arr.copy()
            Xp[:, fi] = RNG.permutation(Xp[:, fi])

            sc = float(np.mean(
                cross_val_score(_ridge(), Xp, res_tr, cv=kf, scoring="r2")
            ))

            deltas.append(sc - base_cv)

        perm_scores_asset[feat] = float(np.mean(deltas))

    perm_results_W96[pilot_asset] = perm_scores_asset
    print(f"{pilot_asset} baseline CV R² = {base_cv:.4f}")

all_feats_W96 = list(perm_results_W96["BTC"].keys())

avg_perm_W96 = {
    f: np.mean([perm_results_W96[a].get(f, 0.0) for a in ["BTC", "ETH"]])
    for f in all_feats_W96
}

sorted_perm_W96 = sorted(avg_perm_W96.items(), key=lambda x: x[1])

def categorize_W96(feat):
    if feat in IT_SLIM_W96:
        return "IT_slim"
    if feat in extra_it_cols_W96:
        return "IT_extra"
    if feat in IT_OTHER_BASE_W96:
        return "IT_base_other"
    if feat.startswith("A_"):
        return "TDA_A"
    if feat.startswith("BI_"):
        return "TDA_BI"
    if feat.startswith("BII_"):
        return "TDA_BII"
    if feat.startswith("BC_"):
        return "TDA_BC"
    return "other"

print("\nTop-15 useful W96 features:")
for feat, delta in sorted_perm_W96[:15]:
    print(f"  {feat:<36} {delta:+.4f} [{categorize_W96(feat)}]")

print("\nTop-10 harmful W96 features:")
for feat, delta in sorted(avg_perm_W96.items(), key=lambda x: -x[1])[:10]:
    print(f"  {feat:<36} {delta:+.4f} [{categorize_W96(feat)}]")

best_tda_W96 = [f for f in TDA_COLS_W96 if avg_perm_W96.get(f, 0) < -0.002]
best_it_W96 = [f for f in IT_ALL_W96 if avg_perm_W96.get(f, 0) < -0.002]

print(f"\nBest TDA W96: {best_tda_W96[:8]}")
print(f"Best IT W96:  {best_it_W96[:12]}")

CONFIGS_W96 = {
    "slim_IT": IT_SLIM_W96,
    "best_IT": [f for f, d in sorted_perm_W96[:20] if f in IT_ALL_W96],
    "slim+TDA": IT_SLIM_W96 + best_tda_W96[:6],
    "best+TDA": [f for f, d in sorted_perm_W96[:16] if f in IT_ALL_W96 or f in TDA_COLS_W96],
}

CONFIGS_W96 = {k: list(dict.fromkeys(v)) for k, v in CONFIGS_W96.items()}

config_rows = []

for asset in ASSETS:
    if asset not in ret.columns:
        continue

    r = ret[asset]

    for h, h_label in HORIZONS.items():
        vol_fwd = r.rolling(h).std().shift(-h) * ANN
        y_raw = np.log(vol_fwd.clip(lower=1e-8)).dropna()

        for cfg_name, cfg_cols in CONFIGS_W96.items():
            it_cols_used = [c for c in cfg_cols if c in X_it_aug_W96.columns]
            tda_cols_used = [c for c in cfg_cols if c in X_tda_full_W96.columns]

            Xi_cfg = pd.concat(
                [X_it_aug_W96[it_cols_used]] +
                ([X_tda_full_W96[tda_cols_used]] if tda_cols_used else []),
                axis=1,
            )

            Xi_cfg = Xi_cfg.loc[:, ~Xi_cfg.columns.duplicated()]

            idx = X_vol.index.intersection(Xi_cfg.index).intersection(y_raw.index)

            valid = (
                X_vol.loc[idx].notna().all(1)
                & Xi_cfg.loc[idx].notna().all(1)
                & y_raw.loc[idx].notna()
            )

            idx = idx[valid]

            if len(idx) < 1000:
                continue

            tr = idx < (split_ts - pd.Timedelta(minutes=30 * PURGE_BARS))
            te = idx >= split_ts

            if tr.sum() < 480 or te.sum() < 100:
                continue

            Xv_tr = X_vol.loc[idx[tr]]
            Xv_te = X_vol.loc[idx[te]]
            Xi_tr = Xi_cfg.loc[idx[tr]]
            Xi_te = Xi_cfg.loc[idx[te]]
            ytr = y_raw.loc[idx[tr]]
            yte = y_raw.loc[idx[te]]

            naive = float(ytr.mean())

            ck = Xv_tr.corrwith(ytr).abs().dropna()
            top3 = ck.nlargest(3).index.tolist()

            mb = _ridge()
            mb.fit(Xv_tr[top3], ytr)

            pb_tr = mb.predict(Xv_tr[top3])
            pb_te = mb.predict(Xv_te[top3])

            res_tr = ytr.values - pb_tr

            mc = _ridge()
            mc.fit(Xi_tr, res_tr)

            pred = pb_te + mc.predict(Xi_te)

            r2_b = oos_r2(yte.values, pb_te, naive)
            r2_c = oos_r2(yte.values, pred, naive)

            rc = float(pd.Series(mc.predict(Xi_te)).corr(pd.Series(yte.values - pb_te)))

            config_rows.append({
                "asset": asset,
                "horizon": h_label,
                "h": h,
                "config": cfg_name,
                "r2_base": round(r2_b, 4),
                "r2_cfg": round(r2_c, 4),
                "delta_r2": round(r2_c - r2_b, 4),
                "res_corr": round(rc, 4),
            })

cfg_df_W96 = pd.DataFrame(config_rows)

print("\nMean ΔR² W96 by config:")
for cfg in CONFIGS_W96:
    sub = cfg_df_W96[cfg_df_W96.config == cfg]
    print(
        f"  {cfg:<15}: ΔR²={sub['delta_r2'].mean():+.4f} "
        f"↑{(sub['delta_r2'] > 0).sum()}/{len(sub)} "
        f"rc={sub['res_corr'].mean():+.4f}"
    )

best_cfg_W96 = cfg_df_W96.groupby("config")["delta_r2"].mean().idxmax()
print(f"\nBest config W96: {best_cfg_W96}")

feat_analysis_W96 = {
    "perm": avg_perm_W96,
    "cfg_df": cfg_df_W96,
    "it_corrs": it_corrs_W96,
    "tda_corrs": tda_corrs_W96,
    "best_cfg": best_cfg_W96,
    "best_cols": CONFIGS_W96[best_cfg_W96],
    "X_it_aug": X_it_aug_W96,
    "X_tda_full": X_tda_full_W96,
    "extra_it_cols": extra_it_cols_W96,
    "feature_window": FEATURE_WINDOW,
}

print("\nfeat_analysis_W96 saved in memory")

FEATURE ANALYSIS W=96
Building extra standalone MI features W96: gaussian
Building extra standalone MI features W96: knn
Building index / entropy / NMI features...
Base IT W96:      25
IT slim W96:      10
Extra IT W96:     10
Total IT W96:     35
Total TDA W96:    26

PERMUTATION IMPORTANCE W=96 ON TRAIN
BTC baseline CV R² = -0.3335
ETH baseline CV R² = -0.5296

Top-15 useful W96 features:
  pairwise_mi_mean                     -0.0367 [IT_slim]
  knn_pairwise_mi_mean                 -0.0263 [IT_extra]
  entropy_index_1y                     -0.0239 [IT_extra]
  kl_AVAX                              -0.0232 [IT_slim]
  coinfo3_mean                         -0.0148 [IT_slim]
  A_E_H1                               -0.0144 [TDA_A]
  gaussian_pairwise_mi_std             -0.0123 [IT_extra]
  BI_tp_H1                             -0.0107 [TDA_BI]
  BI_L1_H1                             -0.0061 [TDA_BI]
  BC_tp_H1                             -0.0047 [TDA_BC]
  knn_pairwise_mi_max                 

In [159]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy import stats as sp_stats

from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

SPLIT_DATE = "2025-07-01"
PURGE_BARS = 96

HORIZONS = {
    48:  "1d",
    96:  "2d",
    192: "4d",
    288: "6d",
    336: "7d",
    672: "14d",
}

ASSETS = [a for a in ["BTC","ETH","SOL","BNB","XRP","DOGE","ADA","AVAX"] if a in ret.columns]

VOL_WINDOW = 48
ANN = np.sqrt(365 * 48)

PERM_USEFUL_THRESHOLD = -0.002
TOP_IT = 10
TOP_TDA = 6

def _ridge():
    return Pipeline([
        ("sc", StandardScaler()),
        ("m", RidgeCV(alphas=[0.01, 0.1, 1, 10, 100]))
    ])

def oos_r2(y_true, y_pred, naive_val):
    mse_m = np.mean((y_true - y_pred) ** 2)
    mse_n = np.mean((y_true - naive_val) ** 2)
    return float(1.0 - mse_m / mse_n) if mse_n > 1e-12 else np.nan

def res_corr(residuals_true, residuals_pred):
    return float(pd.Series(residuals_pred).corr(pd.Series(residuals_true)))

def hit_rate_vs_baseline(y_true, pred_base, pred_model):
    e_base = (y_true - pred_base) ** 2
    e_mod = (y_true - pred_model) ** 2
    return float(np.mean(e_mod < e_base))

def dm_test(loss_diff):
    n = len(loss_diff)
    m = loss_diff.mean()
    s = loss_diff.std(ddof=1)
    t = m / (s / np.sqrt(n)) if s > 1e-12 else 0.0
    p = float(sp_stats.t.sf(abs(t), df=n - 1) * 2) if n > 1 else np.nan
    return float(t), float(p)

def sig(p):
    if not np.isfinite(p):
        return ""
    if p < 0.01:
        return "***"
    if p < 0.05:
        return "**"
    if p < 0.10:
        return "*"
    return ""

avg_perm = feat_analysis_W96["perm"]
X_it_all = feat_analysis_W96["X_it_aug"].copy()
X_tda_all = feat_analysis_W96["X_tda_full"].copy()

perm_series = pd.Series(avg_perm).sort_values()

it_candidates = [
    f for f in perm_series.index
    if (f in X_it_all.columns) and (perm_series[f] < PERM_USEFUL_THRESHOLD)
]

tda_candidates = [
    f for f in perm_series.index
    if (f in X_tda_all.columns) and (perm_series[f] < PERM_USEFUL_THRESHOLD)
]

BEST_IT_W96 = it_candidates[:TOP_IT]
BEST_TDA_W96 = tda_candidates[:TOP_TDA]

print("=" * 80)
print("FINAL HOLDOUT EXPERIMENT — W=96")
print("=" * 80)

print("\nSelected IT features:")
for f in BEST_IT_W96:
    print(f"  {f:<36} perm={perm_series[f]:+.4f}")

print("\nSelected TDA features:")
for f in BEST_TDA_W96:
    print(f"  {f:<36} perm={perm_series[f]:+.4f}")

print(f"\nIT count:  {len(BEST_IT_W96)}")
print(f"TDA count: {len(BEST_TDA_W96)}")

split_ts = pd.Timestamp(SPLIT_DATE)
rows = []

for asset in ASSETS:
    r = ret[asset]

    for h, h_label in HORIZONS.items():
        vol_fwd = r.rolling(h).std().shift(-h) * ANN
        y_raw = np.log(vol_fwd.clip(lower=1e-8)).dropna()

        Xi_it = X_it_all[BEST_IT_W96].copy()
        Xi_tda = X_tda_all[BEST_TDA_W96].copy()

        idx = (
            X_vol.index
            .intersection(Xi_it.index)
            .intersection(Xi_tda.index)
            .intersection(y_raw.index)
        )

        valid = (
            X_vol.loc[idx].notna().all(1)
            & Xi_it.loc[idx].notna().all(1)
            & Xi_tda.loc[idx].notna().all(1)
            & y_raw.loc[idx].notna()
        )

        idx = idx[valid]

        if len(idx) < 1000:
            continue

        tr = idx < (split_ts - pd.Timedelta(minutes=30 * PURGE_BARS))
        te = idx >= split_ts

        if tr.sum() < 480 or te.sum() < 100:
            continue

        Xv_tr = X_vol.loc[idx[tr]]
        Xv_te = X_vol.loc[idx[te]]

        Xit_tr = Xi_it.loc[idx[tr]]
        Xit_te = Xi_it.loc[idx[te]]

        Xtd_tr = Xi_tda.loc[idx[tr]]
        Xtd_te = Xi_tda.loc[idx[te]]

        ytr = y_raw.loc[idx[tr]]
        yte = y_raw.loc[idx[te]]

        naive = float(ytr.mean())

        ck = Xv_tr.corrwith(ytr).abs().replace([np.inf, -np.inf], np.nan).dropna()
        top3 = ck.nlargest(3).index.tolist()

        mb = _ridge()
        mb.fit(Xv_tr[top3], ytr)

        pb_tr = mb.predict(Xv_tr[top3])
        pb_te = mb.predict(Xv_te[top3])

        res_tr = ytr.values - pb_tr
        res_te = yte.values - pb_te

        mit = _ridge()
        mit.fit(Xit_tr, res_tr)
        pit_te = mit.predict(Xit_te)
        pred_it = pb_te + pit_te

        mtda = _ridge()
        mtda.fit(Xtd_tr, res_tr)
        ptda_te = mtda.predict(Xtd_te)
        pred_tda = pb_te + ptda_te

        Xc_tr = pd.concat([Xit_tr, Xtd_tr], axis=1)
        Xc_te = pd.concat([Xit_te, Xtd_te], axis=1)

        mct = _ridge()
        mct.fit(Xc_tr, res_tr)
        pct_te = mct.predict(Xc_te)
        pred_all = pb_te + pct_te

        r2_A = oos_r2(yte.values, pb_te, naive)
        r2_B = oos_r2(yte.values, pred_it, naive)
        r2_C = oos_r2(yte.values, pred_tda, naive)
        r2_D = oos_r2(yte.values, pred_all, naive)

        mse_A = np.mean((yte.values - pb_te) ** 2)
        mse_B = np.mean((yte.values - pred_it) ** 2)
        mse_C = np.mean((yte.values - pred_tda) ** 2)
        mse_D = np.mean((yte.values - pred_all) ** 2)

        dB = (yte.values - pb_te) ** 2 - (yte.values - pred_it) ** 2
        dC = (yte.values - pb_te) ** 2 - (yte.values - pred_tda) ** 2
        dD = (yte.values - pb_te) ** 2 - (yte.values - pred_all) ** 2

        _, p_B = dm_test(dB)
        _, p_C = dm_test(dC)
        _, p_D = dm_test(dD)

        rows.append({
            "asset": asset,
            "horizon": h_label,
            "h": h,

            "r2_A": round(r2_A, 4),
            "r2_B": round(r2_B, 4),
            "r2_C": round(r2_C, 4),
            "r2_D": round(r2_D, 4),

            "dR2_B": round(r2_B - r2_A, 4),
            "dR2_C": round(r2_C - r2_A, 4),
            "dR2_D": round(r2_D - r2_A, 4),

            "rc_B": round(res_corr(res_te, pit_te), 4),
            "rc_C": round(res_corr(res_te, ptda_te), 4),
            "rc_D": round(res_corr(res_te, pct_te), 4),

            "mse_ratio_B": round(mse_B / mse_A, 4) if mse_A > 0 else np.nan,
            "mse_ratio_C": round(mse_C / mse_A, 4) if mse_A > 0 else np.nan,
            "mse_ratio_D": round(mse_D / mse_A, 4) if mse_A > 0 else np.nan,

            "hit_B": round(hit_rate_vs_baseline(yte.values, pb_te, pred_it), 4),
            "hit_C": round(hit_rate_vs_baseline(yte.values, pb_te, pred_tda), 4),
            "hit_D": round(hit_rate_vs_baseline(yte.values, pb_te, pred_all), 4),

            "p_B": round(p_B, 4),
            "p_C": round(p_C, 4),
            "p_D": round(p_D, 4),

            "sig_B": sig(p_B),
            "sig_C": sig(p_C),
            "sig_D": sig(p_D),

            "n_train": int(tr.sum()),
            "n_test": int(te.sum()),
        })

df_best_W96 = pd.DataFrame(rows)

print(f"\nRows collected: {len(df_best_W96)}")

print("\n" + "=" * 100)
print("FINAL COMPARISON — W=96")
print("A = Baseline(vol) | B = +best IT | C = +best TDA | D = +best IT + best TDA")
print("=" * 100)

print(
    f"\n  {'asset':<6} {'h':<4}"
    f" {'R²_A':>9} {'R²_B':>9} {'R²_C':>9} {'R²_D':>9}"
    f" {'ΔB':>8} {'ΔC':>8} {'ΔD':>8}  best"
)
print("  " + "-" * 90)

prev = None

for _, row in df_best_W96.sort_values(["asset", "h"]).iterrows():
    if prev and prev != row["asset"]:
        print("  " + "·" * 90)

    prev = row["asset"]

    vals = {"A": row["r2_A"], "B": row["r2_B"], "C": row["r2_C"], "D": row["r2_D"]}
    best = max(vals, key=vals.get)

    mb = "↑" if row["dR2_B"] > 0.005 else ("↓" if row["dR2_B"] < -0.005 else "~")
    mc = "↑" if row["dR2_C"] > 0.005 else ("↓" if row["dR2_C"] < -0.005 else "~")
    md = "↑" if row["dR2_D"] > 0.005 else ("↓" if row["dR2_D"] < -0.005 else "~")

    print(
        f"  {row['asset']:<6} {row['horizon']:<4}"
        f" {row['r2_A']:>+9.4f}"
        f" {row['r2_B']:>+9.4f}"
        f" {row['r2_C']:>+9.4f}"
        f" {row['r2_D']:>+9.4f}"
        f" {row['dR2_B']:>+7.4f}{mb}"
        f" {row['dR2_C']:>+7.4f}{mc}"
        f" {row['dR2_D']:>+7.4f}{md}"
        f"  {best}"
    )

print("\n" + "=" * 75)
print("SUMMARY — W=96")
print("=" * 75)

n = len(df_best_W96)

print(f"""
  Config         mean ΔR²   ΔR²>0    mean rc   MSE<1    hit rate
  ----------------------------------------------------------------
  B (+best IT)   {df_best_W96['dR2_B'].mean():>+8.4f}   {(df_best_W96['dR2_B']>0).sum():>2}/{n}   {df_best_W96['rc_B'].mean():>+7.4f}   {(df_best_W96['mse_ratio_B']<1).sum():>2}/{n}   {df_best_W96['hit_B'].mean():.3f}
  C (+best TDA)  {df_best_W96['dR2_C'].mean():>+8.4f}   {(df_best_W96['dR2_C']>0).sum():>2}/{n}   {df_best_W96['rc_C'].mean():>+7.4f}   {(df_best_W96['mse_ratio_C']<1).sum():>2}/{n}   {df_best_W96['hit_C'].mean():.3f}
  D (+both)      {df_best_W96['dR2_D'].mean():>+8.4f}   {(df_best_W96['dR2_D']>0).sum():>2}/{n}   {df_best_W96['rc_D'].mean():>+7.4f}   {(df_best_W96['mse_ratio_D']<1).sum():>2}/{n}   {df_best_W96['hit_D'].mean():.3f}
""")

asset_summary_W96 = (
    df_best_W96.groupby("asset", as_index=False)
    .agg(
        r2_A=("r2_A", "mean"),
        r2_B=("r2_B", "mean"),
        r2_C=("r2_C", "mean"),
        r2_D=("r2_D", "mean"),
        dR2_B=("dR2_B", "mean"),
        dR2_C=("dR2_C", "mean"),
        dR2_D=("dR2_D", "mean"),
        mse_ratio_D=("mse_ratio_D", "mean"),
    )
)

asset_summary_W96["best_config"] = asset_summary_W96.apply(
    lambda row: max(
        {"A": row["r2_A"], "B": row["r2_B"], "C": row["r2_C"], "D": row["r2_D"]},
        key={"A": row["r2_A"], "B": row["r2_B"], "C": row["r2_C"], "D": row["r2_D"]}.get,
    ),
    axis=1,
)

print("\nASSET SUMMARY — W=96")
print(asset_summary_W96.sort_values("r2_D", ascending=False).to_string(index=False))

final_comparison_best_W96 = df_best_W96.copy()
asset_summary_best_W96 = asset_summary_W96.copy()

best_feature_sets_W96 = {
    "FEATURE_WINDOW": 96,
    "BEST_IT": BEST_IT_W96,
    "BEST_TDA": BEST_TDA_W96,
}

print("\nSaved to memory:")
print("  final_comparison_best_W96")
print("  asset_summary_best_W96")
print("  best_feature_sets_W96")

FINAL HOLDOUT EXPERIMENT — W=96

Selected IT features:
  pairwise_mi_mean                     perm=-0.0367
  knn_pairwise_mi_mean                 perm=-0.0263
  entropy_index_1y                     perm=-0.0239
  kl_AVAX                              perm=-0.0232
  coinfo3_mean                         perm=-0.0148
  gaussian_pairwise_mi_std             perm=-0.0123
  knn_pairwise_mi_max                  perm=-0.0022

Selected TDA features:
  A_E_H1                               perm=-0.0144
  BI_tp_H1                             perm=-0.0107
  BI_L1_H1                             perm=-0.0061
  BC_tp_H1                             perm=-0.0047

IT count:  7
TDA count: 4

Rows collected: 48

FINAL COMPARISON — W=96
A = Baseline(vol) | B = +best IT | C = +best TDA | D = +best IT + best TDA

  asset  h         R²_A      R²_B      R²_C      R²_D       ΔB       ΔC       ΔD  best
  ------------------------------------------------------------------------------------------
  ADA    1d     +0.19

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy import stats as sp_stats

from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mutual_info_score

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

WINDOWS_TO_TEST = [96, 144, 192, 384] 
IT_STEP   = 4
IT_BINS   = 10

SPLIT_DATE = "2025-07-01"
PURGE_BARS = 96

HORIZONS = {
    48:  "1d",
    96:  "2d",
    192: "4d",
    288: "6d",
    336: "7d",
    672: "14d",
}

ASSETS = [a for a in ["BTC","ETH","SOL","BNB","XRP","DOGE","ADA","AVAX"]
          if a in ret.columns]

VOL_WINDOW = 48
ANN = np.sqrt(365 * 48)

PERM_USEFUL_THRESHOLD = -0.002
TOP_IT  = 10
TOP_TDA = 6

def _ridge():
    return Pipeline([
        ("sc", StandardScaler()),
        ("m", RidgeCV(alphas=[0.01, 0.1, 1, 10, 100]))
    ])

def oos_r2(y_true, y_pred, naive_val):
    mse_m = np.mean((y_true - y_pred) ** 2)
    mse_n = np.mean((y_true - naive_val) ** 2)
    return float(1.0 - mse_m / mse_n) if mse_n > 1e-12 else np.nan

def dm_test(loss_diff):
    n = len(loss_diff)
    m = loss_diff.mean()
    s = loss_diff.std(ddof=1)
    t = m / (s / np.sqrt(n)) if s > 1e-12 else 0.0
    p = float(sp_stats.t.sf(abs(t), df=n - 1) * 2) if n > 1 else np.nan
    return float(t), float(p)

def sig(p):
    if not np.isfinite(p): return ""
    if p < 0.01: return "***"
    if p < 0.05: return "**"
    if p < 0.10: return "*"
    return ""

def _rank_to_bins(x, bins=10):
    x = np.asarray(x, dtype=float)
    ranks = pd.Series(x).rank(method="average").values
    scaled = np.floor((ranks - 1) / max(1, len(x)) * bins).astype(int)
    return np.clip(scaled, 0, bins - 1)

def _drop_finite(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    return x[mask], y[mask]

def mi_hist_rank(x, y, bins=10):
    x, y = _drop_finite(x, y)
    if len(x) < max(20, bins): return np.nan
    bx = _rank_to_bins(x, bins)
    by = _rank_to_bins(y, bins)
    c = np.histogram2d(bx, by, bins=bins,
                       range=[[0, bins], [0, bins]])[0]
    return float(mutual_info_score(None, None, contingency=c))

def _entropy(x, bins=10):
    x = x[np.isfinite(x)]
    if len(x) < 5: return np.nan
    bx = _rank_to_bins(x, bins)
    counts = np.bincount(bx, minlength=bins).astype(float)
    p = counts / counts.sum()
    p = p[p > 0]
    return float(-np.sum(p * np.log(p)))

def _joint_entropy(xs, bins=10):
    if any(len(x) < 5 for x in xs): return np.nan
    multi = np.stack([_rank_to_bins(x, bins) for x in xs], axis=1)
    codes = multi @ (bins ** np.arange(len(xs)))
    counts = np.bincount(codes,
                         minlength=bins**len(xs)).astype(float)
    p = counts / counts.sum()
    p = p[p > 0]
    return float(-np.sum(p * np.log(p)))

def build_it_features_for_window(ret_df, assets, window, step, bins):
    """
    Строит базовые IT фичи (pairwise MI, KL, coinfo3, coinfo4)
    для заданного размера окна.
    Возвращает DataFrame с теми же колонками что и X_it_aug,
    но вычисленными с другим window.
    """
    avail  = [a for a in assets if a in ret_df.columns]
    idx    = ret_df.index
    n      = len(idx)
    starts = np.arange(0, n - window + 1, step)

    pairs   = [(avail[i], avail[j])
                for i in range(len(avail))
                for j in range(i + 1, len(avail))]
    triples = [(avail[i], avail[j], avail[k])
                for i in range(len(avail))
                for j in range(i + 1, len(avail))
                for k in range(j + 1, len(avail))]
    quads   = [(avail[i], avail[j], avail[k], avail[l])
                for i in range(len(avail))
                for j in range(i + 1, len(avail))
                for k in range(j + 1, len(avail))
                for l in range(k + 1, len(avail))]

    cols = (
        ["pairwise_mi_mean", "pairwise_mi_std",
         "pairwise_mi_max",  "pairwise_mi_min"] +
        [f"kl_{a}" for a in avail] +
        ["coinfo3_mean", "coinfo3_std",
         "coinfo3_max",  "coinfo3_min"] +
        ["coinfo4_mean", "coinfo4_std",
         "coinfo4_max",  "coinfo4_min"]
    )
    out = pd.DataFrame(np.nan, index=idx, columns=cols)

    print(f"    Building IT W={window}: {len(starts)} windows...")

    for s in starts:
        e   = s + window
        sub = ret_df.iloc[s:e]

        mi_arr = np.array(
            [v for v in [mi_hist_rank(sub[a].values,
                                      sub[b].values, bins)
                         for a, b in pairs]
             if np.isfinite(v)], dtype=float)
        if len(mi_arr):
            out.at[idx[e-1], "pairwise_mi_mean"] = mi_arr.mean()
            out.at[idx[e-1], "pairwise_mi_std"]  = mi_arr.std(ddof=0)
            out.at[idx[e-1], "pairwise_mi_max"]  = mi_arr.max()
            out.at[idx[e-1], "pairwise_mi_min"]  = mi_arr.min()

        if s >= window:
            prev = ret_df.iloc[s - window:s]
            for a in avail:
                xc = sub[a].values; xp = prev[a].values
                if len(xc) < 5 or len(xp) < 5: continue
                pc = (np.bincount(_rank_to_bins(xc, bins),
                                  minlength=bins).astype(float) + 1e-8)
                pp = (np.bincount(_rank_to_bins(xp, bins),
                                  minlength=bins).astype(float) + 1e-8)
                pc /= pc.sum(); pp /= pp.sum()
                out.at[idx[e-1], f"kl_{a}"] = float(
                    np.sum(pc * np.log(pc / pp)))

        ci3_vals = []
        for (a, b, c) in triples:
            xa = sub[a].values
            xb = sub[b].values
            xc_ = sub[c].values
            vs = [_entropy(xa), _entropy(xb), _entropy(xc_),
                  _joint_entropy([xa, xb]),
                  _joint_entropy([xa, xc_]),
                  _joint_entropy([xb, xc_]),
                  _joint_entropy([xa, xb, xc_])]
            if all(v is not None and np.isfinite(v) for v in vs):
                h_a,h_b,h_c,h_ab,h_ac,h_bc,h_abc = vs
                ci3_vals.append(
                    h_a+h_b+h_c - h_ab-h_ac-h_bc + h_abc)
        if ci3_vals:
            c3 = np.array(ci3_vals)
            out.at[idx[e-1], "coinfo3_mean"] = c3.mean()
            out.at[idx[e-1], "coinfo3_std"]  = c3.std(ddof=0)
            out.at[idx[e-1], "coinfo3_max"]  = c3.max()
            out.at[idx[e-1], "coinfo3_min"]  = c3.min()

        ci4_vals = []
        for (a, b, c, d) in quads:
            xs_ = [sub[x].values for x in [a, b, c, d]]
            h1 = [_entropy(x) for x in xs_]
            h2 = [_joint_entropy([xs_[i], xs_[j]])
                  for i in range(4)
                  for j in range(i+1, 4)]
            h3 = [_joint_entropy([xs_[i], xs_[j], xs_[k]])
                  for i in range(4)
                  for j in range(i+1, 4)
                  for k in range(j+1, 4)]
            h4 = _joint_entropy(xs_)
            if all(v is not None and np.isfinite(v)
                   for v in h1+h2+h3+[h4]):
                ci4_vals.append(sum(h1)-sum(h2)+sum(h3)-h4)
        if ci4_vals:
            c4 = np.array(ci4_vals)
            out.at[idx[e-1], "coinfo4_mean"] = c4.mean()
            out.at[idx[e-1], "coinfo4_std"]  = c4.std(ddof=0)
            out.at[idx[e-1], "coinfo4_max"]  = c4.max()
            out.at[idx[e-1], "coinfo4_min"]  = c4.min()

    return out.ffill()

print("Loading TDA features...")

if "feat_analysis" in dir() and "X_tda_full" in feat_analysis:
    X_tda_all = feat_analysis["X_tda_full"].copy()
elif "X_tda_full" in dir():
    X_tda_all = X_tda_full.copy()
else:
    tda_frames = [tda_A.add_prefix("A_")]
    for mode in ["I", "II", "C"]:
        d = tda_B[mode]
        df_m = pd.DataFrame({
            f"B{mode}_tp_H0":  d["tp_H0"],
            f"B{mode}_ent_H0": d["ent_H0"],
            f"B{mode}_tp_H1":  d["tp_H1"],
            f"B{mode}_ent_H1": d["ent_H1"],
            f"B{mode}_L1_H0":  d["L1_H0"],
            f"B{mode}_L1_H1":  d["L1_H1"],
        }, index=pd.DatetimeIndex(d["times"]))
        tda_frames.append(df_m)
    X_tda_all = (pd.concat(tda_frames, axis=1)
                   .sort_index()
                   .reindex(ret.index, method="ffill"))

print(f"  TDA shape: {X_tda_all.shape}")

if "best_feature_sets" in dir():
    BEST_TDA = list(best_feature_sets.get("BEST_TDA", []))
else:
    BEST_TDA = ["BI_L1_H0", "BC_ent_H1", "BII_tp_H1",
                "BII_L1_H1", "A_T_H0", "A_E_H0"]

BEST_TDA = [c for c in BEST_TDA if c in X_tda_all.columns]
print(f"  Using {len(BEST_TDA)} TDA features: {BEST_TDA}")

def run_holdout_for_window(X_it_w, best_it, best_tda, window_label):
    """
    Запускает A/B/C/D сравнение для одного набора IT фич.
    Возвращает DataFrame с результатами.
    """
    split_ts = pd.Timestamp(SPLIT_DATE)
    rows = []

    for asset in ASSETS:
        r = ret[asset]
        Xi_it  = X_it_w[best_it].copy()
        Xi_tda = X_tda_all[best_tda].copy()

        for h, h_label in HORIZONS.items():
            vol_fwd = r.rolling(h).std().shift(-h) * ANN
            y_raw   = np.log(vol_fwd.clip(lower=1e-8)).dropna()

            idx = (X_vol.index
                   .intersection(Xi_it.index)
                   .intersection(Xi_tda.index)
                   .intersection(y_raw.index))
            valid = (X_vol.loc[idx].notna().all(1)
                     & Xi_it.loc[idx].notna().all(1)
                     & Xi_tda.loc[idx].notna().all(1)
                     & y_raw.loc[idx].notna())
            idx = idx[valid]

            if len(idx) < 1000: continue

            tr = idx < (split_ts - pd.Timedelta(minutes=30 * PURGE_BARS))
            te = idx >= split_ts
            if tr.sum() < 480 or te.sum() < 100: continue

            Xv_tr  = X_vol.loc[idx[tr]]; Xv_te  = X_vol.loc[idx[te]]
            Xit_tr = Xi_it.loc[idx[tr]]; Xit_te = Xi_it.loc[idx[te]]
            Xtd_tr = Xi_tda.loc[idx[tr]]; Xtd_te = Xi_tda.loc[idx[te]]
            ytr = y_raw.loc[idx[tr]]; yte = y_raw.loc[idx[te]]
            naive = float(ytr.mean())

            ck   = (Xv_tr.corrwith(ytr).abs()
                    .replace([np.inf, -np.inf], np.nan).dropna())
            top3 = ck.nlargest(3).index.tolist()
            mb = _ridge(); mb.fit(Xv_tr[top3], ytr)
            pb_tr = mb.predict(Xv_tr[top3])
            pb_te = mb.predict(Xv_te[top3])
            res_tr = ytr.values - pb_tr
            res_te = yte.values - pb_te

            mit = _ridge(); mit.fit(Xit_tr, res_tr)
            pit_te  = mit.predict(Xit_te)
            pred_it = pb_te + pit_te

            mtda = _ridge(); mtda.fit(Xtd_tr, res_tr)
            ptda_te  = mtda.predict(Xtd_te)
            pred_tda = pb_te + ptda_te

            Xc_tr = pd.concat([Xit_tr, Xtd_tr], axis=1)
            Xc_te = pd.concat([Xit_te, Xtd_te], axis=1)
            mct = _ridge(); mct.fit(Xc_tr, res_tr)
            pct_te   = mct.predict(Xc_te)
            pred_all = pb_te + pct_te

            r2_A = oos_r2(yte.values, pb_te,    naive)
            r2_B = oos_r2(yte.values, pred_it,  naive)
            r2_C = oos_r2(yte.values, pred_tda, naive)
            r2_D = oos_r2(yte.values, pred_all, naive)

            dD = ((yte.values - pb_te)**2
                  - (yte.values - pred_all)**2)
            _, p_D = dm_test(dD)

            rows.append({
                "window":  window_label,
                "asset":   asset,
                "horizon": h_label,
                "h":       h,
                "r2_A":   round(r2_A, 4),
                "r2_B":   round(r2_B, 4),
                "r2_C":   round(r2_C, 4),
                "r2_D":   round(r2_D, 4),
                "dR2_B":  round(r2_B - r2_A, 4),
                "dR2_C":  round(r2_C - r2_A, 4),
                "dR2_D":  round(r2_D - r2_A, 4),
                "p_D":    round(p_D, 4),
                "sig_D":  sig(p_D),
                "n_train": int(tr.sum()),
                "n_test":  int(te.sum()),
            })

    return pd.DataFrame(rows)

print("\n" + "=" * 70)
print("WINDOW SENSITIVITY ANALYSIS")
print(f"Windows: {WINDOWS_TO_TEST} bars")
print("=" * 70)

all_results = []
window_it_features = {}  

for w in WINDOWS_TO_TEST:
    w_label = f"W={w}"
    w_days  = w / 48
    print(f"\n{'='*60}")
    print(f"  Window: {w} bars (~{w_days:.1f} days)")
    print(f"{'='*60}")

    X_it_w = build_it_features_for_window(
        ret, ASSETS, w, IT_STEP, IT_BINS)
    print(f"  IT features built: {X_it_w.shape}")

    if "feat_analysis" in dir():
        avg_perm_src = feat_analysis["perm"]
    else:
        avg_perm_src = avg_perm

    perm_series = pd.Series(avg_perm_src).sort_values()
    best_it_w = [
        f for f in perm_series.index
        if (f in X_it_w.columns)
        and (perm_series[f] < PERM_USEFUL_THRESHOLD)
    ][:TOP_IT]

    if len(best_it_w) == 0:
        best_it_w = [
            f for f in perm_series.index
            if f in X_it_w.columns
        ][:6]
        print(f"  WARNING: no features passed threshold, "
              f"using top-{len(best_it_w)} by perm score")

    print(f"  Selected IT features ({len(best_it_w)}): {best_it_w}")

    window_it_features[w] = (X_it_w, best_it_w)

    df_w = run_holdout_for_window(
        X_it_w, best_it_w, BEST_TDA, w_label)
    all_results.append(df_w)

    n = len(df_w)
    if n > 0:
        print(f"\n  Results ({n} tasks):")
        print(f"    B (+IT):  ΔR²={df_w['dR2_B'].mean():>+.4f}  "
              f"pos={(df_w['dR2_B']>0).sum():>2}/{n}")
        print(f"    C (+TDA): ΔR²={df_w['dR2_C'].mean():>+.4f}  "
              f"pos={(df_w['dR2_C']>0).sum():>2}/{n}")
        print(f"    D (+all): ΔR²={df_w['dR2_D'].mean():>+.4f}  "
              f"pos={(df_w['dR2_D']>0).sum():>2}/{n}")
    else:
        print("  WARNING: no rows collected for this window.")

df_window = pd.concat(all_results, ignore_index=True)
print(f"\nTotal rows: {len(df_window)}")

print("\n" + "=" * 70)
print("WINDOW SENSITIVITY SUMMARY")
print("=" * 70)

print(f"\n  {'Window':<10} {'bars':>6} {'days':>5}"
      f" {'ΔR²_B':>8} {'ΔR²_C':>8} {'ΔR²_D':>8}"
      f" {'B>0':>6} {'C>0':>6} {'D>0':>6}")
print("  " + "-" * 65)

for w in WINDOWS_TO_TEST:
    w_label = f"W={w}"
    sub = df_window[df_window.window == w_label]
    if sub.empty: continue
    n = len(sub)
    print(f"  {w_label:<10} {w:>6} {w/48:>5.1f}"
          f" {sub['dR2_B'].mean():>+8.4f}"
          f" {sub['dR2_C'].mean():>+8.4f}"
          f" {sub['dR2_D'].mean():>+8.4f}"
          f" {(sub['dR2_B']>0).sum():>4}/{n}"
          f" {(sub['dR2_C']>0).sum():>4}/{n}"
          f" {(sub['dR2_D']>0).sum():>4}/{n}")

print(f"\n  Asset-level ΔR²_D by window:")
print(f"  {'Asset':<6}", end="")
for w in WINDOWS_TO_TEST:
    print(f"  {'W='+str(w):>10}", end="")
print()
print("  " + "-" * (6 + 12 * len(WINDOWS_TO_TEST)))

for asset in ASSETS:
    print(f"  {asset:<6}", end="")
    for w in WINDOWS_TO_TEST:
        sub = df_window[(df_window.window == f"W={w}")
                        & (df_window.asset == asset)]
        val = sub["dR2_D"].mean() if not sub.empty else np.nan
        if np.isfinite(val):
            print(f"  {val:>+10.4f}", end="")
        else:
            print(f"  {'N/A':>10}", end="")
    print()

print("\nBuilding window sensitivity plots...")

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
w_labels  = [f"W={w}" for w in WINDOWS_TO_TEST]
w_days    = [f"{w/48:.1f}d" for w in WINDOWS_TO_TEST]
colors    = {"B": "#2196F3", "C": "#FF9800", "D": "#4CAF50"}

ax = axes[0]
for cfg in ["B", "C", "D"]:
    vals = [df_window[df_window.window == wl][f"dR2_{cfg}"].mean()
            for wl in w_labels]
    ax.plot(w_days, vals, "o-", color=colors[cfg],
            lw=2, ms=8, label=f"Config {cfg}")
ax.axhline(0, color="black", lw=0.8, ls=":")
ax.set_xlabel("IT window size")
ax.set_ylabel("Mean ΔR² vs baseline A")
ax.set_title("Mean ΔR² by IT window\n(all assets, all horizons)", fontsize=10)
ax.legend(fontsize=9)
ax.grid(alpha=0.25)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax = axes[1]
for cfg in ["B", "C", "D"]:
    fracs = []
    for wl in w_labels:
        sub = df_window[df_window.window == wl]
        n   = len(sub)
        fracs.append((sub[f"dR2_{cfg}"] > 0).sum() / n if n > 0 else np.nan)
    ax.plot(w_days, fracs, "o-", color=colors[cfg],
            lw=2, ms=8, label=f"Config {cfg}")
ax.axhline(0.5, color="gray", lw=0.8, ls=":", alpha=0.5)
ax.set_xlabel("IT window size")
ax.set_ylabel("Fraction of tasks with ΔR² > 0")
ax.set_title("Hit rate by IT window\n(fraction of tasks improved)", fontsize=10)
ax.legend(fontsize=9)
ax.grid(alpha=0.25)
ax.set_ylim(0, 1)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax = axes[2]
heat_data = np.full((len(ASSETS), len(WINDOWS_TO_TEST)), np.nan)
for j, w in enumerate(WINDOWS_TO_TEST):
    wl = f"W={w}"
    for i, asset in enumerate(ASSETS):
        sub = df_window[(df_window.window == wl)
                        & (df_window.asset == asset)]
        if not sub.empty:
            heat_data[i, j] = sub["dR2_D"].mean()

im = ax.imshow(heat_data, cmap="RdYlGn", vmin=-0.15, vmax=0.15,
               aspect="auto")
ax.set_xticks(range(len(WINDOWS_TO_TEST)))
ax.set_xticklabels(w_days)
ax.set_yticks(range(len(ASSETS)))
ax.set_yticklabels(ASSETS)
ax.set_xlabel("IT window size")
ax.set_title("ΔR²_D by asset × window\n(green = better than baseline)", fontsize=10)

for i in range(len(ASSETS)):
    for j in range(len(WINDOWS_TO_TEST)):
        val = heat_data[i, j]
        if np.isfinite(val):
            col = "white" if abs(val) > 0.08 else "black"
            ax.text(j, i, f"{val:+.3f}", ha="center", va="center",
                    fontsize=8, color=col)

plt.colorbar(im, ax=ax, label="ΔR²_D vs baseline A")

plt.suptitle(
    f"Window sensitivity analysis: IT features computed with "
    f"W ∈ {{{', '.join(str(w) for w in WINDOWS_TO_TEST)}}} bars\n"
    f"TDA features unchanged (W=192). Holdout {SPLIT_DATE} →",
    fontsize=12, y=1.01
)
plt.tight_layout()
plt.savefig("fig_window_sensitivity.png", bbox_inches="tight", dpi=150)
plt.savefig("fig_window_sensitivity.pdf", bbox_inches="tight", dpi=150)
plt.show()
print("Saved: fig_window_sensitivity.png / .pdf")

window_sensitivity_results = df_window.copy()
print(f"\nwindow_sensitivity_results saved in memory ({len(df_window)} rows)")

Loading TDA features...
  TDA shape: (46743, 26)
  Using 6 TDA features: ['BI_L1_H0', 'BC_ent_H1', 'BII_tp_H1', 'BII_L1_H1', 'A_T_H0', 'A_E_H0']

WINDOW SENSITIVITY ANALYSIS
Windows: [96, 144, 192, 384] bars

  Window: 96 bars (~2.0 days)
    Building IT W=96: 11662 windows...


In [ ]:
df_window = window_sensitivity_results.copy()
WINDOWS_TO_TEST = sorted(df_window["window"].str.replace("W=","").astype(int).unique().tolist())

print("\n" + "=" * 70)
print("WINDOW SENSITIVITY SUMMARY")
print("=" * 70)

print(f"\n  {'Window':<10} {'bars':>6} {'days':>5}"
      f" {'ΔR²_B':>8} {'ΔR²_C':>8} {'ΔR²_D':>8}"
      f" {'B>0':>6} {'C>0':>6} {'D>0':>6}")
print("  " + "-" * 65)

for w in WINDOWS_TO_TEST:
    w_label = f"W={w}"
    sub = df_window[df_window.window == w_label]
    if sub.empty: continue
    n = len(sub)
    print(f"  {w_label:<10} {w:>6} {w/48:>5.1f}"
          f" {sub['dR2_B'].mean():>+8.4f}"
          f" {sub['dR2_C'].mean():>+8.4f}"
          f" {sub['dR2_D'].mean():>+8.4f}"
          f" {(sub['dR2_B']>0).sum():>4}/{n}"
          f" {(sub['dR2_C']>0).sum():>4}/{n}"
          f" {(sub['dR2_D']>0).sum():>4}/{n}")

print(f"\n  Asset-level ΔR²_D by window:")
ASSETS_order = df_window["asset"].unique().tolist()
print(f"  {'Asset':<6}", end="")
for w in WINDOWS_TO_TEST:
    print(f"  {'W='+str(w):>10}", end="")
print()
print("  " + "-" * (6 + 12 * len(WINDOWS_TO_TEST)))

for asset in ASSETS_order:
    print(f"  {asset:<6}", end="")
    for w in WINDOWS_TO_TEST:
        sub = df_window[(df_window.window == f"W={w}")
                        & (df_window.asset == asset)]
        val = sub["dR2_D"].mean() if not sub.empty else float("nan")
        if val == val: 
            print(f"  {val:>+10.4f}", end="")
        else:
            print(f"  {'N/A':>10}", end="")
    print()

print(f"\n  Asset-level ΔR²_C by window:")
print(f"  {'Asset':<6}", end="")
for w in WINDOWS_TO_TEST:
    print(f"  {'W='+str(w):>10}", end="")
print()
print("  " + "-" * (6 + 12 * len(WINDOWS_TO_TEST)))

for asset in ASSETS_order:
    print(f"  {asset:<6}", end="")
    for w in WINDOWS_TO_TEST:
        sub = df_window[(df_window.window == f"W={w}")
                        & (df_window.asset == asset)]
        val = sub["dR2_C"].mean() if not sub.empty else float("nan")
        if val == val:
            print(f"  {val:>+10.4f}", end="")
        else:
            print(f"  {'N/A':>10}", end="")
    print()

print(f"\n  Horizon-level ΔR²_D (mean over assets) by window:")
HORIZONS_order = ["1d","2d","4d","6d","7d","14d"]
print(f"  {'Horizon':<8}", end="")
for w in WINDOWS_TO_TEST:
    print(f"  {'W='+str(w):>10}", end="")
print()
print("  " + "-" * (8 + 12 * len(WINDOWS_TO_TEST)))

for h_label in HORIZONS_order:
    sub_h = df_window[df_window.horizon == h_label]
    if sub_h.empty: continue
    print(f"  {h_label:<8}", end="")
    for w in WINDOWS_TO_TEST:
        sub = sub_h[sub_h.window == f"W={w}"]
        val = sub["dR2_D"].mean() if not sub.empty else float("nan")
        if val == val:
            print(f"  {val:>+10.4f}", end="")
        else:
            print(f"  {'N/A':>10}", end="")
    print()

print(f"\n  Total rows in df_window: {len(df_window)}")
print(f"  Windows found: {[f'W={w}' for w in WINDOWS_TO_TEST]}")
print(f"  Assets found:  {ASSETS_order}")


WINDOW SENSITIVITY SUMMARY

  Window       bars  days    ΔR²_B    ΔR²_C    ΔR²_D    B>0    C>0    D>0
  -----------------------------------------------------------------
  W=96           96   2.0  +0.0204  +0.0536  +0.0850   35/48   32/48   33/48
  W=144         144   3.0  +0.0031  +0.0536  +0.0619   30/48   32/48   30/48
  W=192         192   4.0  -0.0178  +0.0536  +0.0459   25/48   32/48   31/48
  W=384         384   8.0  -0.0401  +0.0536  +0.0146   26/48   32/48   26/48

  Asset-level ΔR²_D by window:
  Asset         W=96       W=144       W=192       W=384
  ------------------------------------------------------
  BTC        -0.0618     -0.0578     -0.0543     -0.0207
  ETH        +0.2298     +0.2219     +0.2094     +0.1478
  SOL        -0.0274     -0.0416     -0.0487     -0.0274
  BNB        -0.0718     -0.0882     -0.1031     -0.1491
  XRP        +0.0908     +0.0744     +0.0728     +0.0836
  DOGE       +0.1922     +0.1605     +0.1452     +0.1153
  ADA        +0.2107     +0.1887 

In [183]:
df_window = window_sensitivity_results.copy()
WINDOWS_TO_TEST = sorted(
    df_window["window"].str.replace("W=","").astype(int).unique().tolist()
)
HORIZONS_order = ["1d","2d","4d","6d","7d","14d"]
ASSETS_order   = ["BTC","ETH","SOL","BNB","XRP","DOGE","ADA","AVAX"]

def sig(p):
    if not (p == p): return ""
    if p < 0.01: return "***"
    if p < 0.05: return "**"
    if p < 0.10: return "*"
    return ""

for w in WINDOWS_TO_TEST:
    w_label = f"W={w}"
    sub_w   = df_window[df_window.window == w_label]
    n       = len(sub_w)

    print("\n" + "=" * 100)
    print(f"WINDOW {w_label}  (~{w/48:.1f} days) — {n} tasks")
    print("A = vol baseline | B = +IT | C = +TDA | D = +IT+TDA")
    print("=" * 100)

    print(
        f"\n  {'asset':<6} {'h':<4}"
        f" {'R²_A':>9} {'R²_B':>9} {'R²_C':>9} {'R²_D':>9}"
        f" {'ΔB':>8} {'ΔC':>8} {'ΔD':>8}  sig_D  best"
    )
    print("  " + "-" * 95)

    prev = None
    for _, row in sub_w.sort_values(["asset","h"]).iterrows():
        if prev and prev != row["asset"]:
            print("  " + "·" * 95)
        prev = row["asset"]

        vals = {
            "A": row["r2_A"], "B": row["r2_B"],
            "C": row["r2_C"], "D": row["r2_D"],
        }
        best = max(vals, key=vals.get)

        mb = "↑" if row["dR2_B"] >  0.005 else ("↓" if row["dR2_B"] < -0.005 else "~")
        mc = "↑" if row["dR2_C"] >  0.005 else ("↓" if row["dR2_C"] < -0.005 else "~")
        md = "↑" if row["dR2_D"] >  0.005 else ("↓" if row["dR2_D"] < -0.005 else "~")

        print(
            f"  {row['asset']:<6} {row['horizon']:<4}"
            f" {row['r2_A']:>+9.4f}"
            f" {row['r2_B']:>+9.4f}"
            f" {row['r2_C']:>+9.4f}"
            f" {row['r2_D']:>+9.4f}"
            f" {row['dR2_B']:>+7.4f}{mb}"
            f" {row['dR2_C']:>+7.4f}{mc}"
            f" {row['dR2_D']:>+7.4f}{md}"
            f"  {row['sig_D']:>4}   {best}"
        )

    print(f"\n  Asset summary (mean over horizons):")
    print(
        f"  {'asset':<6}"
        f" {'R²_A':>9} {'R²_B':>9} {'R²_C':>9} {'R²_D':>9}"
        f" {'ΔB':>8} {'ΔC':>8} {'ΔD':>8}  best"
    )
    print("  " + "-" * 80)

    for asset in ASSETS_order:
        g = sub_w[sub_w.asset == asset]
        if g.empty: continue
        vals = {
            "A": g["r2_A"].mean(), "B": g["r2_B"].mean(),
            "C": g["r2_C"].mean(), "D": g["r2_D"].mean(),
        }
        best = max(vals, key=vals.get)
        print(
            f"  {asset:<6}"
            f" {g['r2_A'].mean():>+9.4f}"
            f" {g['r2_B'].mean():>+9.4f}"
            f" {g['r2_C'].mean():>+9.4f}"
            f" {g['r2_D'].mean():>+9.4f}"
            f" {g['dR2_B'].mean():>+8.4f}"
            f" {g['dR2_C'].mean():>+8.4f}"
            f" {g['dR2_D'].mean():>+8.4f}"
            f"  {best}"
        )

    print(f"\n  Horizon summary (mean over assets):")
    print(
        f"  {'h':<4}"
        f" {'R²_A':>9} {'R²_B':>9} {'R²_C':>9} {'R²_D':>9}"
        f" {'ΔB':>8} {'ΔC':>8} {'ΔD':>8}"
    )
    print("  " + "-" * 72)

    for h_label in HORIZONS_order:
        g = sub_w[sub_w.horizon == h_label]
        if g.empty: continue
        print(
            f"  {h_label:<4}"
            f" {g['r2_A'].mean():>+9.4f}"
            f" {g['r2_B'].mean():>+9.4f}"
            f" {g['r2_C'].mean():>+9.4f}"
            f" {g['r2_D'].mean():>+9.4f}"
            f" {g['dR2_B'].mean():>+8.4f}"
            f" {g['dR2_C'].mean():>+8.4f}"
            f" {g['dR2_D'].mean():>+8.4f}"
        )

    print(f"\n  Overall: "
          f"ΔR²_B={sub_w['dR2_B'].mean():>+.4f} "
          f"({(sub_w['dR2_B']>0).sum()}/{n})  "
          f"ΔR²_C={sub_w['dR2_C'].mean():>+.4f} "
          f"({(sub_w['dR2_C']>0).sum()}/{n})  "
          f"ΔR²_D={sub_w['dR2_D'].mean():>+.4f} "
          f"({(sub_w['dR2_D']>0).sum()}/{n})")


WINDOW W=96  (~2.0 days) — 48 tasks
A = vol baseline | B = +IT | C = +TDA | D = +IT+TDA

  asset  h         R²_A      R²_B      R²_C      R²_D       ΔB       ΔC       ΔD  sig_D  best
  -----------------------------------------------------------------------------------------------
  ADA    1d     +0.2670   +0.3056   +0.3490   +0.3989 +0.0387↑ +0.0820↑ +0.1320↑   ***   D
  ADA    2d     +0.2413   +0.2880   +0.3515   +0.4171 +0.0467↑ +0.1102↑ +0.1758↑   ***   D
  ADA    4d     +0.1948   +0.2415   +0.3151   +0.3658 +0.0467↑ +0.1203↑ +0.1710↑   ***   D
  ADA    6d     +0.1330   +0.1986   +0.2573   +0.3520 +0.0656↑ +0.1243↑ +0.2190↑   ***   D
  ADA    7d     +0.1210   +0.1872   +0.2543   +0.3592 +0.0662↑ +0.1333↑ +0.2382↑   ***   D
  ADA    14d    -0.0990   -0.1195   +0.1013   +0.2291 -0.0205↓ +0.2003↑ +0.3281↑   ***   D
  ·······························································································
  AVAX   1d     +0.1260   +0.1793   +0.1651   +0.2542 +0.0533↑ +0.0391↑ +0

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

COLORS = {
    "A": "#9E9E9E",
    "B": "#2196F3",
    "C": "#FF9800",
    "D": "#4CAF50",
}

df_vol  = final_comparison_best.copy()
h_order = ["1d", "2d", "4d", "6d", "7d", "14d"]

ax = axes[0]
for col, label, color, ls in [
    ("r2_A", "A: Baseline",  COLORS["A"], "-"),
    ("r2_B", "B: +best IT",  COLORS["B"], "-"),
    ("r2_C", "C: +best TDA", COLORS["C"], "--"),
    ("r2_D", "D: +best both",COLORS["D"], "-"),
]:
    vals = [df_vol[df_vol.horizon == h][col].mean() for h in h_order]
    ax.plot(h_order, vals, "o" + ls, color=color, lw=2,
            ms=7, label=label)

ax.axhline(0, color="black", lw=0.8, ls=":")
ax.set_xlabel("Forecast horizon")
ax.set_ylabel(r"OOS $R^2$ (mean over assets)")
ax.set_title("OOS $R^2$ by horizon\n(IT window $W=96$, mean over assets)",
             fontsize=10)
ax.legend(fontsize=9)
ax.grid(alpha=0.25, ls="--")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

df_sw = stress_window_sensitivity_results.copy()
df_s  = df_sw[df_sw.window == "W=96"].copy()
h_stress = ["1d", "2d", "4d"]

ax = axes[1]
for col, label, color, ls in [
    ("roc_A", "A: vol\_now",   COLORS["A"], "-"),
    ("roc_B", "B: +IT",        COLORS["B"], "-"),
    ("roc_C", "C: +TDA",       COLORS["C"], "--"),
    ("roc_D", "D: +IT+TDA",    COLORS["D"], "-"),
]:
    sub = df_s[df_s.q == 0.80]
    vals = [sub[sub.horizon == h][col].mean() for h in h_stress]
    ax.plot(h_stress, vals, "o" + ls, color=color, lw=2,
            ms=7, label=label)

ax.axhline(0, color="black", lw=0.8, ls=":")
ax.set_xlabel("Forecast horizon")
ax.set_ylabel("ROC-AUC (mean over assets)")
ax.set_title("ROC-AUC by horizon\n"
             "(IT window $W=96$, Q80, mean over assets)", fontsize=10)
ax.legend(fontsize=9)
ax.grid(alpha=0.25, ls="--")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.suptitle(
    "Predictive performance at the best IT window $W=96$ bars "
    "($\\approx$2 days)\nHoldout period: 2025-07-01 $\\to$ 2025-09-01",
    fontsize=11
)
plt.tight_layout()
plt.savefig("fig_w96_horizons.png", bbox_inches="tight", dpi=150)
plt.savefig("fig_w96_horizons.pdf", bbox_inches="tight", dpi=150)
plt.show()
print("Saved: fig_w96_horizons.png / .pdf")

Saved: fig_w96_horizons.png / .pdf
